# Notebook 2 — Javelin V6.1 Training and Release (Refactored)

**Refactor version:** `NB2_REFACTOR_20260920_v1.1.0_SAFE_MODES`  
**Source notebook:** `Javelin_V6_Train_and_Release.ipynb`  
**Source SHA256:** `e68dbe4c3753f6e56c7d5055172c1450f26bf0749121dfff8e94c93cc8c5273c`

This notebook is a production-preserving refactor of the validated Notebook 2 training/release workflow. The validated numerical logic and original execution order are retained while the notebook is reorganized into explicit production stages.

### Refactor principles

- Preserve the validated V4 → V5 → V6 → V6.1 → U1 mathematics and temporal data-use rules.
- Preserve immutable artifact paths, versioning, audit packages, release gates, bundle contract, reload verification, and `ACTIVE_MODEL` promotion behavior.
- Add explicit execution modes so validation cannot accidentally create a persistent model release or update `ACTIVE_MODEL`.
- Keep TEST and POST_TEST out of model/architecture tuning exactly as in the validated workflow.
- Group 55 nonempty legacy execution cells into 20 production stages with explicit contracts.
- Remove one exact duplicate diagnostic source cell and omit the empty trailing cell.
- Normalize redundant `from __future__ import annotations` lines when legacy cells are coalesced; this is syntax-only and does not alter model logic.
- Strip historical outputs/execution counts so Restart & Run All produces a fresh audit trail.

> Historical-fold implementations remain explicit rather than being generalized into one loop. Fold-specific artifact names, evidence packages, and audit paths are part of the validated evidence chain and are intentionally preserved.


## Execution contract and safety modes

Run from top to bottom in a fresh kernel. Each stage depends only on earlier stages.

**Edit only `EXECUTION_MODE` in the next cell before running.** The notebook supports three modes:

- `VALIDATE_REFACTOR` — **default and safest**. Runs the complete calculations and release-contract checks using a temporary scratch artifact root. No persistent model version is written under `model_artifacts`, and `ACTIVE_MODEL.json` is never modified.
- `TRAIN_CANDIDATE` — creates a new persistent versioned training run, artifacts, release bundle, and verification package under `model_artifacts`, but **does not promote** it to `ACTIVE_MODEL.json`.
- `TRAIN_AND_PROMOTE` — full production behavior. Creates a new versioned model candidate and, only after all gates and reload verification pass, atomically promotes it to `ACTIVE_MODEL.json`.

Notebook 3 consumes the resulting runtime contract and therefore depends on stable artifact names/schemas for the structural posterior, V5 terminal states and OU parameters, V6/V6.1 specifications, U1 uncertainty specification, architecture decision, evidence/audit artifacts, smoke-test fixture, release manifest, and active-model pointer.

For validating this refactor against the current active release, leave the mode at `VALIDATE_REFACTOR`.


## Execution mode

This is the only control you normally need to change before running Notebook 2. `VALIDATE_REFACTOR` is intentionally the default.

In [1]:
# ==================================================================================================
# NOTEBOOK 2 — EXECUTION MODE SAFETY CONTROL
# ==================================================================================================
#
# EDIT ONLY THIS VALUE FOR NORMAL USE:
#
#     VALIDATE_REFACTOR  -> transient scratch run; no persistent model version; no ACTIVE_MODEL update
#     TRAIN_CANDIDATE    -> persistent candidate + bundle; no ACTIVE_MODEL update
#     TRAIN_AND_PROMOTE  -> full production train/release/promotion workflow
#
# ==================================================================================================

from pathlib import Path
import tempfile

EXECUTION_MODE = "VALIDATE_REFACTOR"

VALID_EXECUTION_MODES = {
    "VALIDATE_REFACTOR",
    "TRAIN_CANDIDATE",
    "TRAIN_AND_PROMOTE",
}

if EXECUTION_MODE not in VALID_EXECUTION_MODES:
    raise ValueError(
        f"Invalid EXECUTION_MODE={EXECUTION_MODE!r}. "
        f"Expected one of {sorted(VALID_EXECUTION_MODES)}."
    )

IS_VALIDATION_MODE = EXECUTION_MODE == "VALIDATE_REFACTOR"
PERSISTENT_ARTIFACTS_ENABLED = EXECUTION_MODE in {
    "TRAIN_CANDIDATE",
    "TRAIN_AND_PROMOTE",
}
ACTIVE_MODEL_PROMOTION_ENABLED = EXECUTION_MODE == "TRAIN_AND_PROMOTE"

VALIDATION_SCRATCH_ROOT = (
    Path(tempfile.mkdtemp(prefix="javelin_v6_refactor_validation_"))
    if IS_VALIDATION_MODE
    else None
)

RUN_PERSISTENCE_SCOPE = (
    "TRANSIENT_SCRATCH"
    if IS_VALIDATION_MODE
    else "PERSISTENT_MODEL_ARTIFACTS"
)

EXECUTION_MODE_CONFIG = {
    "schema_version": "1.0.0",
    "execution_mode": EXECUTION_MODE,
    "persistent_artifacts_enabled": PERSISTENT_ARTIFACTS_ENABLED,
    "active_model_promotion_enabled": ACTIVE_MODEL_PROMOTION_ENABLED,
    "run_persistence_scope": RUN_PERSISTENCE_SCOPE,
    "validation_scratch_root": (
        str(VALIDATION_SCRATCH_ROOT)
        if VALIDATION_SCRATCH_ROOT is not None
        else None
    ),
}

print("=" * 120)
print("NOTEBOOK 2 — EXECUTION MODE")
print("=" * 120)
print(f"Execution mode                  : {EXECUTION_MODE}")
print(f"Persistent artifacts enabled    : {PERSISTENT_ARTIFACTS_ENABLED}")
print(f"ACTIVE_MODEL promotion enabled  : {ACTIVE_MODEL_PROMOTION_ENABLED}")
print(f"Run persistence scope           : {RUN_PERSISTENCE_SCOPE}")
if VALIDATION_SCRATCH_ROOT is not None:
    print(f"Validation scratch root         : {VALIDATION_SCRATCH_ROOT}")
print("=" * 120)


NOTEBOOK 2 — EXECUTION MODE
Execution mode                  : VALIDATE_REFACTOR
Persistent artifacts enabled    : False
ACTIVE_MODEL promotion enabled  : False
Run persistence scope           : TRANSIENT_SCRATCH
Validation scratch root         : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_


## Stage 01 — Configuration and canonical data

Resolve versioned paths, load ACTIVE_CANONICAL, validate canonical dataset identity, and establish immutable run configuration.

**Legacy source cells represented:** 1  
**Execution rule:** run only after the preceding stage has completed successfully.


In [2]:
# ==================================================================================================
# REFACTORED STAGE 01 — Configuration and canonical data
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 1
# Original source SHA256: 679fc978a8f41e9a7b57e12a8cfa28cbfcdf7017831b15178241c67adfc87e55
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 1 — CONFIGURATION + ACTIVE CANONICAL DATASET LOAD
# ==================================================================================================
#
# Purpose
# -------
# Establish the reproducible training environment for a new V6 model release.
#
# This cell:
#
#   1. locates ACTIVE_CANONICAL.json created by Notebook 1
#   2. loads the active production canonical dataset
#   3. verifies its SHA256 against the active pointer
#   4. validates the canonical schema
#   5. creates a versioned model-training run directory
#   6. freezes the source-data identity in the training manifest
#
#
# IMPORTANT
# ---------
# Notebook 2 trains ONLY from the active canonical release.
#
# It must not train directly from an incremental candidate file.
#
# ==================================================================================================

import hashlib
import json

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/Users/veesheenyuen/Documents/code/veeyuen/SAA"
)


WORLD_ATHLETICS_ROOT = (
    PROJECT_ROOT
    /
    "world_athletics_global"
)


FINAL_DATA_DIR = (
    WORLD_ATHLETICS_ROOT
    /
    "final"
)


ACTIVE_CANONICAL_PATH = (
    FINAL_DATA_DIR
    /
    "ACTIVE_CANONICAL.json"
)


PERSISTENT_MODEL_ARTIFACT_ROOT = (
    PROJECT_ROOT
    /
    "model_artifacts"
)


# Execution-mode routing:
#   VALIDATE_REFACTOR -> transient scratch root only
#   TRAIN_CANDIDATE / TRAIN_AND_PROMOTE -> persistent model_artifacts root
MODEL_ARTIFACT_ROOT = (
    VALIDATION_SCRATCH_ROOT
    if IS_VALIDATION_MODE
    else PERSISTENT_MODEL_ARTIFACT_ROOT
)


MODEL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# PART B — TRAINING RUN ID
# ==================================================================================================

TRAINING_RUN_ID = (
    datetime.now(
        timezone.utc
    )
    .strftime(
        "%Y%m%d_%H%M%S"
    )
)


MODEL_VERSION = (
    f"V6_{TRAINING_RUN_ID}"
)


TRAINING_RUN_DIR = (
    MODEL_ARTIFACT_ROOT
    /
    MODEL_VERSION
)


TRAINING_RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


TRAINING_CHECKPOINT_DIR = (
    TRAINING_RUN_DIR
    /
    "checkpoints"
)


TRAINING_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRAINING_MANIFEST_PATH = (
    TRAINING_RUN_DIR
    /
    "training_manifest.json"
)


# ==================================================================================================
# PART C — CANONICAL SCHEMA
# ==================================================================================================

EXPECTED_CANONICAL_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",
    "gender_raw",
    "implement_weight_g",
    "canonical_birth_date",
    "canonical_birth_year",
    "birth_precision",
    "age_years",
    "spec_regime",
    "record_source",
    "reconciliation_status",
    "event_id",
    "model_group",
    "spec_model_eligible",
    "core_group_eligible",
    "age_available",
    "v4_base_eligible",
]


CORE_MODEL_GROUPS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


# ==================================================================================================
# PART D — FILE HASH HELPER
# ==================================================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
):

    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ==================================================================================================
# PART E — LOAD ACTIVE CANONICAL POINTER
# ==================================================================================================

if not ACTIVE_CANONICAL_PATH.exists():

    raise FileNotFoundError(
        "ACTIVE_CANONICAL.json was not found.\n\n"
        f"Expected:\n{ACTIVE_CANONICAL_PATH}\n\n"
        "Notebook 1 production promotion must complete before "
        "Notebook 2 training begins."
    )


with ACTIVE_CANONICAL_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    ACTIVE_CANONICAL = json.load(
        f
    )


required_pointer_fields = [
    "run_id",
    "production_path",
    "production_sha256",
    "rows",
    "athletes",
    "v4_eligible",
    "max_result_date",
]


missing_pointer_fields = [
    field
    for field in required_pointer_fields
    if field not in ACTIVE_CANONICAL
]


if missing_pointer_fields:

    raise RuntimeError(
        "ACTIVE_CANONICAL.json is missing required fields:\n"
        + "\n".join(
            f"  - {field}"
            for field in missing_pointer_fields
        )
    )


# ==================================================================================================
# PART F — ACTIVE PRODUCTION DATASET
# ==================================================================================================

CANONICAL_DATASET_PATH = Path(
    ACTIVE_CANONICAL[
        "production_path"
    ]
)


EXPECTED_CANONICAL_SHA256 = (
    ACTIVE_CANONICAL[
        "production_sha256"
    ]
)


if not CANONICAL_DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Active canonical dataset does not exist:\n"
        f"{CANONICAL_DATASET_PATH}"
    )


ACTUAL_CANONICAL_SHA256 = sha256_file(
    CANONICAL_DATASET_PATH
)


HASH_PASS = bool(
    ACTUAL_CANONICAL_SHA256
    ==
    EXPECTED_CANONICAL_SHA256
)


if not HASH_PASS:

    raise RuntimeError(
        "ACTIVE CANONICAL HASH MISMATCH.\n\n"
        f"Expected:\n{EXPECTED_CANONICAL_SHA256}\n\n"
        f"Actual:\n{ACTUAL_CANONICAL_SHA256}\n\n"
        "Training is blocked because the active dataset no longer "
        "matches its release pointer."
    )


# ==================================================================================================
# PART G — LOAD CANONICAL DATA
# ==================================================================================================

CANONICAL = pd.read_parquet(
    CANONICAL_DATASET_PATH
)


CANONICAL[
    "result_date_dt"
] = pd.to_datetime(
    CANONICAL[
        "result_date_dt"
    ],
    errors="coerce",
)


CANONICAL[
    "canonical_birth_date"
] = pd.to_datetime(
    CANONICAL[
        "canonical_birth_date"
    ],
    errors="coerce",
)


CANONICAL[
    "mark_numeric_m"
] = pd.to_numeric(
    CANONICAL[
        "mark_numeric_m"
    ],
    errors="coerce",
)


CANONICAL[
    "age_years"
] = pd.to_numeric(
    CANONICAL[
        "age_years"
    ],
    errors="coerce",
)


CANONICAL[
    "implement_weight_g"
] = pd.to_numeric(
    CANONICAL[
        "implement_weight_g"
    ],
    errors="coerce",
)


# ==================================================================================================
# PART H — SCHEMA VALIDATION
# ==================================================================================================

SCHEMA_PASS = bool(
    list(
        CANONICAL.columns
    )
    ==
    EXPECTED_CANONICAL_COLUMNS
)


if not SCHEMA_PASS:

    raise RuntimeError(
        "Active canonical schema differs from the training contract.\n\n"
        f"Expected:\n{EXPECTED_CANONICAL_COLUMNS}\n\n"
        f"Actual:\n{list(CANONICAL.columns)}"
    )


# ==================================================================================================
# PART I — POINTER VS DATASET VALIDATION
# ==================================================================================================

N_ROWS = int(
    len(
        CANONICAL
    )
)


N_ATHLETES = int(
    CANONICAL[
        "aaAthleteId"
    ]
    .nunique()
)


MAX_RESULT_DATE = (
    CANONICAL[
        "result_date_dt"
    ]
    .max()
)


N_V4_ELIGIBLE = int(
    CANONICAL[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


POINTER_ROW_PASS = bool(
    N_ROWS
    ==
    int(
        ACTIVE_CANONICAL[
            "rows"
        ]
    )
)


POINTER_ATHLETE_PASS = bool(
    N_ATHLETES
    ==
    int(
        ACTIVE_CANONICAL[
            "athletes"
        ]
    )
)


POINTER_V4_PASS = bool(
    N_V4_ELIGIBLE
    ==
    int(
        ACTIVE_CANONICAL[
            "v4_eligible"
        ]
    )
)


pointer_max_date = pd.to_datetime(
    ACTIVE_CANONICAL[
        "max_result_date"
    ],
    errors="coerce",
)


POINTER_DATE_PASS = bool(
    MAX_RESULT_DATE
    ==
    pointer_max_date
)


ACTIVE_CANONICAL_PASS = bool(
    HASH_PASS
    and
    SCHEMA_PASS
    and
    POINTER_ROW_PASS
    and
    POINTER_ATHLETE_PASS
    and
    POINTER_V4_PASS
    and
    POINTER_DATE_PASS
)


if not ACTIVE_CANONICAL_PASS:

    raise RuntimeError(
        "Active canonical pointer validation failed. "
        "Training is blocked."
    )


# ==================================================================================================
# PART J — BASIC MODEL POPULATION SUMMARY
# ==================================================================================================

MODEL_GROUP_SUMMARY = (
    CANONICAL.groupby(
        "model_group",
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),

        spec_eligible=(
            "spec_model_eligible",
            lambda s:
                int(
                    s.fillna(False)
                    .astype(bool)
                    .sum()
                ),
        ),

        age_available=(
            "age_available",
            lambda s:
                int(
                    s.fillna(False)
                    .astype(bool)
                    .sum()
                ),
        ),

        v4_eligible=(
            "v4_base_eligible",
            lambda s:
                int(
                    s.fillna(False)
                    .astype(bool)
                    .sum()
                ),
        ),
    )
)


# ==================================================================================================
# PART K — V4 TRAINING POPULATION
# ==================================================================================================

V4_BASE = (
    CANONICAL.loc[
        CANONICAL[
            "v4_base_eligible"
        ]
        .fillna(False)
        .astype(bool)
    ]
    .copy()
)


V4_BASE = (
    V4_BASE.loc[
        V4_BASE[
            "model_group"
        ]
        .isin(
            CORE_MODEL_GROUPS
        )
    ]
    .copy()
)


V4_BASE = (
    V4_BASE.sort_values(
        [
            "result_date_dt",
            "aaAthleteId",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART L — TRAINING SOURCE SNAPSHOT
# ==================================================================================================
#
# Save the precise training input used by this V6 run.
#
# This makes the model release reproducible even if production data later
# advances again.
#
# ==================================================================================================

TRAINING_SOURCE_SNAPSHOT_PATH = (
    TRAINING_RUN_DIR
    /
    "canonical_training_source.parquet"
)


shutil_imported = False


try:

    import shutil

    shutil_imported = True

except Exception:

    pass


if not shutil_imported:

    raise RuntimeError(
        "Could not import shutil."
    )


shutil.copy2(
    CANONICAL_DATASET_PATH,
    TRAINING_SOURCE_SNAPSHOT_PATH,
)


TRAINING_SOURCE_SHA256 = sha256_file(
    TRAINING_SOURCE_SNAPSHOT_PATH
)


if (
    TRAINING_SOURCE_SHA256
    !=
    ACTUAL_CANONICAL_SHA256
):

    raise RuntimeError(
        "Training source snapshot hash does not match active canonical."
    )


# ==================================================================================================
# PART M — TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST = {
    "status":
        (
            "VALIDATION_INITIALIZED"
            if IS_VALIDATION_MODE
            else "INITIALIZED"
        ),

    "execution_mode":
        EXECUTION_MODE,

    "persistent_artifacts_enabled":
        bool(PERSISTENT_ARTIFACTS_ENABLED),

    "active_model_promotion_enabled":
        bool(ACTIVE_MODEL_PROMOTION_ENABLED),

    "run_persistence_scope":
        RUN_PERSISTENCE_SCOPE,

    "persistent_model_artifact_root":
        str(PERSISTENT_MODEL_ARTIFACT_ROOT),

    "artifact_root_for_this_run":
        str(MODEL_ARTIFACT_ROOT),

    "validation_scratch_root":
        (
            str(VALIDATION_SCRATCH_ROOT)
            if VALIDATION_SCRATCH_ROOT is not None
            else None
        ),

    "model_family":
        "V6",

    "model_version":
        MODEL_VERSION,

    "training_run_id":
        TRAINING_RUN_ID,

    "created_at_utc":
        datetime.now(
            timezone.utc
        )
        .isoformat(),

    "source_canonical": {
        "notebook_1_run_id":
            ACTIVE_CANONICAL[
                "run_id"
            ],

        "production_path":
            str(
                CANONICAL_DATASET_PATH
            ),

        "production_sha256":
            ACTUAL_CANONICAL_SHA256,

        "source_snapshot_path":
            str(
                TRAINING_SOURCE_SNAPSHOT_PATH
            ),

        "source_snapshot_sha256":
            TRAINING_SOURCE_SHA256,

        "rows":
            N_ROWS,

        "athletes":
            N_ATHLETES,

        "max_result_date":
            MAX_RESULT_DATE.isoformat(),

        "v4_eligible":
            N_V4_ELIGIBLE,
    },

    "core_model_groups":
        CORE_MODEL_GROUPS,

    "training_directory":
        str(
            TRAINING_RUN_DIR
        ),
}


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART N — GLOBALS FOR NOTEBOOK 2
# ==================================================================================================

NOTEBOOK_2_READY = True


# ==================================================================================================
# PART O — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — ACTIVE CANONICAL INITIALIZATION")
print("=" * 145)


print()
print("MODEL RELEASE")
print("-" * 145)

print(
    f"Training run ID                  : "
    f"{TRAINING_RUN_ID}"
)

print(
    f"Model version                    : "
    f"{MODEL_VERSION}"
)

print(
    f"Training directory              : "
    f"{TRAINING_RUN_DIR}"
)


print()
print("ACTIVE CANONICAL")
print("-" * 145)

print(
    f"Notebook 1 release run           : "
    f"{ACTIVE_CANONICAL['run_id']}"
)

print(
    f"Dataset                          : "
    f"{CANONICAL_DATASET_PATH}"
)

print(
    f"SHA256 verified                 : "
    f"{HASH_PASS}"
)

print(
    f"Rows                             : "
    f"{N_ROWS:,}"
)

print(
    f"Athletes                         : "
    f"{N_ATHLETES:,}"
)

print(
    f"V4 eligible                      : "
    f"{N_V4_ELIGIBLE:,}"
)

print(
    f"Max result date                  : "
    f"{MAX_RESULT_DATE.date()}"
)


print()
print("ACTIVE POINTER QA")
print("-" * 145)

print(
    f"Row count                        : "
    f"{POINTER_ROW_PASS}"
)

print(
    f"Athlete count                    : "
    f"{POINTER_ATHLETE_PASS}"
)

print(
    f"V4 population                    : "
    f"{POINTER_V4_PASS}"
)

print(
    f"Max result date                  : "
    f"{POINTER_DATE_PASS}"
)

print(
    f"Schema                           : "
    f"{SCHEMA_PASS}"
)


print()
print("MODEL GROUP POPULATION")
print("-" * 145)

display(
    MODEL_GROUP_SUMMARY
)


print()
print("V4 BASE")
print("-" * 145)

print(
    f"Rows                             : "
    f"{len(V4_BASE):,}"
)

print(
    f"Athletes                         : "
    f"{V4_BASE['aaAthleteId'].nunique():,}"
)

print(
    f"Date range                       : "
    f"{V4_BASE['result_date_dt'].min().date()} "
    f"→ "
    f"{V4_BASE['result_date_dt'].max().date()}"
)


print()
print("REPRODUCIBILITY")
print("-" * 145)

print(
    f"Training source snapshot         : "
    f"{TRAINING_SOURCE_SNAPSHOT_PATH}"
)

print(
    f"Training manifest                : "
    f"{TRAINING_MANIFEST_PATH}"
)


print()
print("=" * 145)
print("CELL 1 STATUS: V6 TRAINING ENVIRONMENT READY")
print("=" * 145)


# ==================================================================================================
# REFACTOR SAFETY MODE STATUS
# ==================================================================================================
print()
print("EXECUTION MODE SAFETY")
print("-" * 145)
print(f"Execution mode                  : {EXECUTION_MODE}")
print(f"Artifact root for this run      : {MODEL_ARTIFACT_ROOT}")
print(f"Persistent model version        : {PERSISTENT_ARTIFACTS_ENABLED}")
print(f"ACTIVE_MODEL promotion allowed  : {ACTIVE_MODEL_PROMOTION_ENABLED}")



V6 TRAINING — ACTIVE CANONICAL INITIALIZATION

MODEL RELEASE
-------------------------------------------------------------------------------------------------------------------------------------------------
Training run ID                  : 20260922_125751
Model version                    : V6_20260922_125751
Training directory              : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751

ACTIVE CANONICAL
-------------------------------------------------------------------------------------------------------------------------------------------------
Notebook 1 release run           : 20260919_225312
Dataset                          : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026.parquet
SHA256 verified                 : True
Rows                             : 249,233
Athletes                         : 21,901
V4 eligible                      : 192,983
Max result

,model_group,rows,athletes,spec_eligible,age_available,v4_eligible
0,M_500,98,29,98,96,0
1,M_600,617,237,617,581,0
2,M_700,13973,4209,13922,11629,11580
3,M_800,99042,9687,99041,85806,85805
4,W_400,486,191,486,387,0
5,W_500,22129,5023,22028,18846,18755
6,W_600,89184,8590,89165,76862,76843
7,NaN,23704,5279,0,20413,0



V4 BASE
-------------------------------------------------------------------------------------------------------------------------------------------------
Rows                             : 192,983
Athletes                         : 14,517
Date range                       : 1991-08-10 → 2026-09-18

REPRODUCIBILITY
-------------------------------------------------------------------------------------------------------------------------------------------------
Training source snapshot         : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/canonical_training_source.parquet
Training manifest                : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/training_manifest.json

CELL 1 STATUS: V6 TRAINING ENVIRONMENT READY

EXECUTION MODE SAFETY
------------------------------------------------------------------------------------------------------------------------------

## Stage 02 — Feature table and chronology

Build the model feature table and distinct-date chronology used by all downstream model stages.

**Legacy source cells represented:** 2  
**Execution rule:** run only after the preceding stage has completed successfully.


In [3]:
# ==================================================================================================
# REFACTORED STAGE 02 — Feature table and chronology
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 2
# Original source SHA256: 4fa3a9386bf6c631ae2c557bb5594956e01311e941aa46d6ee11a025117c3e98
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 2 — MODEL FEATURE TABLE + DISTINCT-DATE CHRONOLOGY
# ==================================================================================================
#
# Purpose
# -------
# Convert the canonical V4-eligible population into the formal modelling table.
#
#
# This cell:
#
#   1. enforces the V4/V6 modelling age domain: 13 <= age <= 35
#   2. retains all valid physical performances, including multiple same-day results
#   3. creates deterministic sex / group / youth encodings
#   4. creates deterministic athlete indexing
#   5. creates athlete chronology based on DISTINCT PRIOR DATES
#   6. guarantees no same-day leakage
#   7. creates history_span_days and days_since_previous_date explicitly
#   8. creates prior-date Rolling3 summaries for later diagnostics / V5/V6 work
#   9. saves the model-ready feature table and chronology table
#
#
# IMPORTANT CHRONOLOGY RULE
# -------------------------
#
# Suppose an athlete has three results on 2026-06-01.
#
# All three results receive EXACTLY THE SAME prior-history features.
#
# None of those three results can use another 2026-06-01 result as history.
#
#
# Model-group order is frozen:
#
#       0 = M_700
#       1 = M_800
#       2 = W_500
#       3 = W_600
#
#
# Sex order is frozen:
#
#       0 = M
#       1 = W
#
#
# Youth implement:
#
#       M_700 -> 1
#       W_500 -> 1
#       M_800 -> 0
#       W_600 -> 0
#
#
# age_scaled:
#
#       (age_years - 18) / 5
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_READY",
    "V4_BASE",
    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 Cell 1 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_READY:

    raise RuntimeError(
        "Notebook 2 initialization did not pass."
    )


# ==================================================================================================
# PART B — FROZEN MODELLING CONTRACT
# ==================================================================================================

MODEL_AGE_MIN = 13.0
MODEL_AGE_MAX = 35.0


GROUP_ORDER = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


GROUP_INDEX = {
    group:
        idx

    for idx, group
    in enumerate(
        GROUP_ORDER
    )
}


SEX_ORDER = [
    "M",
    "W",
]


SEX_INDEX = {
    sex:
        idx

    for idx, sex
    in enumerate(
        SEX_ORDER
    )
}


YOUTH_GROUPS = {
    "M_700",
    "W_500",
}


# ==================================================================================================
# PART C — CLEAN CANONICAL MODEL BASE
# ==================================================================================================

FEATURE_SOURCE = (
    V4_BASE.copy()
)


FEATURE_SOURCE[
    "aaAthleteId"
] = pd.to_numeric(
    FEATURE_SOURCE[
        "aaAthleteId"
    ],
    errors="coerce",
)


FEATURE_SOURCE[
    "result_date_dt"
] = pd.to_datetime(
    FEATURE_SOURCE[
        "result_date_dt"
    ],
    errors="coerce",
)


FEATURE_SOURCE[
    "mark_numeric_m"
] = pd.to_numeric(
    FEATURE_SOURCE[
        "mark_numeric_m"
    ],
    errors="coerce",
)


FEATURE_SOURCE[
    "age_years"
] = pd.to_numeric(
    FEATURE_SOURCE[
        "age_years"
    ],
    errors="coerce",
)


FEATURE_SOURCE[
    "model_group"
] = (
    FEATURE_SOURCE[
        "model_group"
    ]
    .astype(str)
    .str.strip()
)


N_V4_BASE_INPUT = int(
    len(
        FEATURE_SOURCE
    )
)


# ==================================================================================================
# PART D — MODEL-LEVEL EXCLUSION FLAGS
# ==================================================================================================

FEATURE_SOURCE[
    "_invalid_athlete"
] = (
    FEATURE_SOURCE[
        "aaAthleteId"
    ]
    .isna()
)


FEATURE_SOURCE[
    "_invalid_date"
] = (
    FEATURE_SOURCE[
        "result_date_dt"
    ]
    .isna()
)


FEATURE_SOURCE[
    "_invalid_mark"
] = (
    FEATURE_SOURCE[
        "mark_numeric_m"
    ]
    .isna()

    |
    ~np.isfinite(
        FEATURE_SOURCE[
            "mark_numeric_m"
        ]
    )
)


FEATURE_SOURCE[
    "_invalid_age"
] = (
    FEATURE_SOURCE[
        "age_years"
    ]
    .isna()

    |
    ~np.isfinite(
        FEATURE_SOURCE[
            "age_years"
        ]
    )
)


FEATURE_SOURCE[
    "_outside_age_domain"
] = (
    FEATURE_SOURCE[
        "age_years"
    ]
    .notna()

    &
    (
        (
            FEATURE_SOURCE[
                "age_years"
            ]
            <
            MODEL_AGE_MIN
        )

        |

        (
            FEATURE_SOURCE[
                "age_years"
            ]
            >
            MODEL_AGE_MAX
        )
    )
)


FEATURE_SOURCE[
    "_invalid_group"
] = (
    ~FEATURE_SOURCE[
        "model_group"
    ]
    .isin(
        GROUP_ORDER
    )
)


MODEL_EXCLUSION_MASK = (
    FEATURE_SOURCE[
        "_invalid_athlete"
    ]

    |
    FEATURE_SOURCE[
        "_invalid_date"
    ]

    |
    FEATURE_SOURCE[
        "_invalid_mark"
    ]

    |
    FEATURE_SOURCE[
        "_invalid_age"
    ]

    |
    FEATURE_SOURCE[
        "_outside_age_domain"
    ]

    |
    FEATURE_SOURCE[
        "_invalid_group"
    ]
)


MODEL_EXCLUSIONS = (
    FEATURE_SOURCE.loc[
        MODEL_EXCLUSION_MASK
    ]
    .copy()
)


MODEL_BASE = (
    FEATURE_SOURCE.loc[
        ~MODEL_EXCLUSION_MASK
    ]
    .copy()
)


MODEL_BASE[
    "aaAthleteId"
] = (
    MODEL_BASE[
        "aaAthleteId"
    ]
    .astype(
        "int64"
    )
)


MODEL_BASE = (
    MODEL_BASE.sort_values(
        [
            "result_date_dt",
            "aaAthleteId",
            "model_group",
            "mark_numeric_m",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART E — EXCLUSION SUMMARY
# ==================================================================================================

EXCLUSION_SUMMARY = pd.DataFrame(
    [
        {
            "reason":
                "invalid_athlete",

            "rows":
                int(
                    FEATURE_SOURCE[
                        "_invalid_athlete"
                    ]
                    .sum()
                ),
        },

        {
            "reason":
                "invalid_date",

            "rows":
                int(
                    FEATURE_SOURCE[
                        "_invalid_date"
                    ]
                    .sum()
                ),
        },

        {
            "reason":
                "invalid_mark",

            "rows":
                int(
                    FEATURE_SOURCE[
                        "_invalid_mark"
                    ]
                    .sum()
                ),
        },

        {
            "reason":
                "invalid_age",

            "rows":
                int(
                    FEATURE_SOURCE[
                        "_invalid_age"
                    ]
                    .sum()
                ),
        },

        {
            "reason":
                "age_below_13",

            "rows":
                int(
                    (
                        FEATURE_SOURCE[
                            "age_years"
                        ]
                        <
                        MODEL_AGE_MIN
                    )
                    .sum()
                ),
        },

        {
            "reason":
                "age_above_35",

            "rows":
                int(
                    (
                        FEATURE_SOURCE[
                            "age_years"
                        ]
                        >
                        MODEL_AGE_MAX
                    )
                    .sum()
                ),
        },

        {
            "reason":
                "invalid_group",

            "rows":
                int(
                    FEATURE_SOURCE[
                        "_invalid_group"
                    ]
                    .sum()
                ),
        },
    ]
)


N_MODEL_ROWS = int(
    len(
        MODEL_BASE
    )
)


N_MODEL_EXCLUDED = int(
    MODEL_EXCLUSION_MASK.sum()
)


ROW_ARITHMETIC_PASS = bool(
    N_V4_BASE_INPUT
    ==
    N_MODEL_ROWS
    +
    N_MODEL_EXCLUDED
)


# ==================================================================================================
# PART F — SEX / GROUP / YOUTH FEATURES
# ==================================================================================================

MODEL_BASE[
    "sex_group"
] = (
    MODEL_BASE[
        "model_group"
    ]
    .str[
        0
    ]
)


MODEL_BASE[
    "sex_idx"
] = (
    MODEL_BASE[
        "sex_group"
    ]
    .map(
        SEX_INDEX
    )
)


MODEL_BASE[
    "group_idx"
] = (
    MODEL_BASE[
        "model_group"
    ]
    .map(
        GROUP_INDEX
    )
)


MODEL_BASE[
    "is_youth_implement"
] = (
    MODEL_BASE[
        "model_group"
    ]
    .isin(
        YOUTH_GROUPS
    )
    .astype(
        "int8"
    )
)


MODEL_BASE[
    "age_scaled"
] = (
    (
        MODEL_BASE[
            "age_years"
        ]
        -
        18.0
    )
    /
    5.0
)


MODEL_BASE[
    "result_year"
] = (
    MODEL_BASE[
        "result_date_dt"
    ]
    .dt.year
    .astype(
        "int16"
    )
)


# ==================================================================================================
# PART G — DETERMINISTIC ATHLETE INDEX
# ==================================================================================================
#
# Sort IDs before assigning index so the mapping is reproducible.
# ==================================================================================================

ATHLETE_IDS = np.array(
    sorted(
        MODEL_BASE[
            "aaAthleteId"
        ]
        .unique()
        .tolist()
    ),
    dtype=np.int64,
)


ATHLETE_INDEX = {
    int(
        athlete_id
    ):
        idx

    for idx, athlete_id
    in enumerate(
        ATHLETE_IDS
    )
}


MODEL_BASE[
    "athlete_idx"
] = (
    MODEL_BASE[
        "aaAthleteId"
    ]
    .map(
        ATHLETE_INDEX
    )
    .astype(
        "int32"
    )
)


N_MODEL_ATHLETES = int(
    len(
        ATHLETE_IDS
    )
)


# ==================================================================================================
# PART H — BUILD DISTINCT ATHLETE-DATE TABLE
# ==================================================================================================
#
# Multiple results on a single date are intentionally collapsed ONLY for the
# purpose of calculating prior-history chronology.
#
# The original performance rows remain untouched.
#
# ==================================================================================================

ATHLETE_DATE = (
    MODEL_BASE.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_results_on_date=(
            "mark_numeric_m",
            "size",
        ),

        date_mean_mark=(
            "mark_numeric_m",
            "mean",
        ),

        date_min_mark=(
            "mark_numeric_m",
            "min",
        ),

        date_max_mark=(
            "mark_numeric_m",
            "max",
        ),

        date_age_min=(
            "age_years",
            "min",
        ),

        date_age_max=(
            "age_years",
            "max",
        ),
    )
)


ATHLETE_DATE = (
    ATHLETE_DATE.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART I — DISTINCT-DATE CHRONOLOGY
# ==================================================================================================

athlete_date_group = (
    ATHLETE_DATE.groupby(
        "aaAthleteId",
        sort=False,
        group_keys=False,
    )
)


# Number of distinct PRIOR competition dates.
ATHLETE_DATE[
    "prior_distinct_dates"
] = (
    athlete_date_group
    .cumcount()
    .astype(
        "int32"
    )
)


# First observed modelling date.
ATHLETE_DATE[
    "first_model_date"
] = (
    athlete_date_group[
        "result_date_dt"
    ]
    .transform(
        "min"
    )
)


# Span from first observed date to current date.
#
# First date -> 0 days.
ATHLETE_DATE[
    "history_span_days"
] = (
    ATHLETE_DATE[
        "result_date_dt"
    ]
    -
    ATHLETE_DATE[
        "first_model_date"
    ]
).dt.days.astype(
    "int32"
)


# Strictly previous DISTINCT date.
ATHLETE_DATE[
    "previous_result_date"
] = (
    athlete_date_group[
        "result_date_dt"
    ]
    .shift(
        1
    )
)


ATHLETE_DATE[
    "days_since_previous_date"
] = (
    ATHLETE_DATE[
        "result_date_dt"
    ]
    -
    ATHLETE_DATE[
        "previous_result_date"
    ]
).dt.days


# ==================================================================================================
# PART J — NUMBER OF PRIOR PHYSICAL RESULTS
# ==================================================================================================
#
# If current date contains 3 results, none of those 3 counts as prior history
# for another result on the same date.
#
# ==================================================================================================

ATHLETE_DATE[
    "_cumulative_results"
] = (
    athlete_date_group[
        "n_results_on_date"
    ]
    .cumsum()
)


ATHLETE_DATE[
    "prior_result_count"
] = (
    ATHLETE_DATE[
        "_cumulative_results"
    ]
    -
    ATHLETE_DATE[
        "n_results_on_date"
    ]
).astype(
    "int32"
)


# ==================================================================================================
# PART K — STRICTLY PRIOR DATE-LEVEL MARK FEATURES
# ==================================================================================================

ATHLETE_DATE[
    "prior_date_mean_mark"
] = (
    athlete_date_group[
        "date_mean_mark"
    ]
    .shift(
        1
    )
)


ATHLETE_DATE[
    "prior_rolling3_date_mean"
] = (
    athlete_date_group[
        "date_mean_mark"
    ]
    .transform(
        lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .mean()
    )
)


ATHLETE_DATE[
    "prior_rolling3_date_count"
] = (
    athlete_date_group[
        "date_mean_mark"
    ]
    .transform(
        lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .count()
    )
    .fillna(
        0
    )
    .astype(
        "int8"
    )
)


# ==================================================================================================
# PART L — MERGE CHRONOLOGY BACK TO EVERY PERFORMANCE ROW
# ==================================================================================================

CHRONOLOGY_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",
    "n_results_on_date",
    "prior_distinct_dates",
    "prior_result_count",
    "first_model_date",
    "history_span_days",
    "previous_result_date",
    "days_since_previous_date",
    "prior_date_mean_mark",
    "prior_rolling3_date_mean",
    "prior_rolling3_date_count",
]


MODEL_FEATURES = (
    MODEL_BASE.merge(
        ATHLETE_DATE[
            CHRONOLOGY_COLUMNS
        ],

        on=[
            "aaAthleteId",
            "result_date_dt",
        ],

        how="left",

        validate="many_to_one",
    )
)


MODEL_FEATURES = (
    MODEL_FEATURES.sort_values(
        [
            "result_date_dt",
            "aaAthleteId",
            "model_group",
            "mark_numeric_m",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART M — NO-SAME-DAY-LEAKAGE QA
# ==================================================================================================
#
# Every same-athlete/same-date row must have identical chronology.
# ==================================================================================================

chronology_check_columns = [
    "prior_distinct_dates",
    "prior_result_count",
    "history_span_days",
    "previous_result_date",
    "days_since_previous_date",
    "prior_date_mean_mark",
    "prior_rolling3_date_mean",
    "prior_rolling3_date_count",
]


SAME_DAY_CHRONOLOGY_VARIANTS = (
    MODEL_FEATURES.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )[
        chronology_check_columns
    ]
    .nunique(
        dropna=False
    )
)


SAME_DAY_LEAKAGE_GROUPS = (
    SAME_DAY_CHRONOLOGY_VARIANTS.loc[
        (
            SAME_DAY_CHRONOLOGY_VARIANTS
            >
            1
        )
        .any(
            axis=1
        )
    ]
)


N_SAME_DAY_LEAKAGE_GROUPS = int(
    len(
        SAME_DAY_LEAKAGE_GROUPS
    )
)


# Previous date must always be strictly earlier.
previous_date_mask = (
    MODEL_FEATURES[
        "previous_result_date"
    ]
    .notna()
)


N_NONSTRICT_PREVIOUS_DATES = int(
    (
        MODEL_FEATURES.loc[
            previous_date_mask,
            "previous_result_date"
        ]
        >=
        MODEL_FEATURES.loc[
            previous_date_mask,
            "result_date_dt"
        ]
    )
    .sum()
)


# First date must have zero prior dates/results.
FIRST_DATE_ROWS = (
    MODEL_FEATURES[
        "prior_distinct_dates"
    ]
    .eq(
        0
    )
)


N_FIRST_DATE_PRIOR_COUNT_ERRORS = int(
    (
        MODEL_FEATURES.loc[
            FIRST_DATE_ROWS,
            "prior_result_count"
        ]
        !=
        0
    )
    .sum()
)


N_FIRST_DATE_SPAN_ERRORS = int(
    (
        MODEL_FEATURES.loc[
            FIRST_DATE_ROWS,
            "history_span_days"
        ]
        !=
        0
    )
    .sum()
)


# ==================================================================================================
# PART N — FEATURE VALIDITY QA
# ==================================================================================================

N_INVALID_SEX_IDX = int(
    MODEL_FEATURES[
        "sex_idx"
    ]
    .isna()
    .sum()
)


N_INVALID_GROUP_IDX = int(
    MODEL_FEATURES[
        "group_idx"
    ]
    .isna()
    .sum()
)


N_INVALID_ATHLETE_IDX = int(
    MODEL_FEATURES[
        "athlete_idx"
    ]
    .isna()
    .sum()
)


AGE_DOMAIN_PASS = bool(
    MODEL_FEATURES[
        "age_years"
    ]
    .between(
        MODEL_AGE_MIN,
        MODEL_AGE_MAX,
        inclusive="both",
    )
    .all()
)


FEATURE_ROW_PASS = bool(
    len(
        MODEL_FEATURES
    )
    ==
    N_MODEL_ROWS
)


CELL_2_PASS = bool(
    ROW_ARITHMETIC_PASS

    and AGE_DOMAIN_PASS

    and FEATURE_ROW_PASS

    and N_INVALID_SEX_IDX == 0

    and N_INVALID_GROUP_IDX == 0

    and N_INVALID_ATHLETE_IDX == 0

    and N_SAME_DAY_LEAKAGE_GROUPS == 0

    and N_NONSTRICT_PREVIOUS_DATES == 0

    and N_FIRST_DATE_PRIOR_COUNT_ERRORS == 0

    and N_FIRST_DATE_SPAN_ERRORS == 0
)


# ==================================================================================================
# PART O — POPULATION SUMMARIES
# ==================================================================================================

MODEL_GROUP_FEATURE_SUMMARY = (
    MODEL_FEATURES.groupby(
        "model_group",
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),

        min_age=(
            "age_years",
            "min",
        ),

        max_age=(
            "age_years",
            "max",
        ),

        min_date=(
            "result_date_dt",
            "min",
        ),

        max_date=(
            "result_date_dt",
            "max",
        ),
    )
)


HISTORY_DEPTH_SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "athletes",

            "value":
                N_MODEL_ATHLETES,
        },

        {
            "metric":
                "athletes_with_2plus_dates",

            "value":
                int(
                    (
                        ATHLETE_DATE.groupby(
                            "aaAthleteId"
                        )
                        .size()
                        >= 2
                    )
                    .sum()
                ),
        },

        {
            "metric":
                "athletes_with_3plus_dates",

            "value":
                int(
                    (
                        ATHLETE_DATE.groupby(
                            "aaAthleteId"
                        )
                        .size()
                        >= 3
                    )
                    .sum()
                ),
        },

        {
            "metric":
                "athletes_with_5plus_dates",

            "value":
                int(
                    (
                        ATHLETE_DATE.groupby(
                            "aaAthleteId"
                        )
                        .size()
                        >= 5
                    )
                    .sum()
                ),
        },

        {
            "metric":
                "athletes_with_10plus_dates",

            "value":
                int(
                    (
                        ATHLETE_DATE.groupby(
                            "aaAthleteId"
                        )
                        .size()
                        >= 10
                    )
                    .sum()
                ),
        },
    ]
)


# ==================================================================================================
# PART P — SAVE FEATURE ARTIFACTS
# ==================================================================================================

MODEL_FEATURES_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_model_features.parquet"
)


ATHLETE_DATE_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_athlete_date_chronology.parquet"
)


ATHLETE_INDEX_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_athlete_index.parquet"
)


MODEL_EXCLUSIONS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_model_feature_exclusions.parquet"
)


EXCLUSION_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_model_feature_exclusion_summary.parquet"
)


MODEL_FEATURES.to_parquet(
    MODEL_FEATURES_PATH,
    index=False,
)


ATHLETE_DATE.drop(
    columns=[
        "_cumulative_results"
    ],
    errors="ignore",
).to_parquet(
    ATHLETE_DATE_PATH,
    index=False,
)


pd.DataFrame(
    {
        "aaAthleteId":
            ATHLETE_IDS,

        "athlete_idx":
            np.arange(
                len(
                    ATHLETE_IDS
                ),
                dtype=np.int32,
            ),
    }
).to_parquet(
    ATHLETE_INDEX_PATH,
    index=False,
)


MODEL_EXCLUSIONS.to_parquet(
    MODEL_EXCLUSIONS_PATH,
    index=False,
)


EXCLUSION_SUMMARY.to_parquet(
    EXCLUSION_SUMMARY_PATH,
    index=False,
)


# ==================================================================================================
# PART Q — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "feature_engineering"
] = {

    "status":
        (
            "PASSED"
            if CELL_2_PASS
            else "FAILED"
        ),

    "age_domain": {
        "minimum":
            MODEL_AGE_MIN,

        "maximum":
            MODEL_AGE_MAX,
    },

    "group_order":
        GROUP_ORDER,

    "sex_order":
        SEX_ORDER,

    "youth_groups":
        sorted(
            YOUTH_GROUPS
        ),

    "age_scaling":
        "(age_years - 18) / 5",

    "v4_base_input_rows":
        N_V4_BASE_INPUT,

    "model_rows":
        N_MODEL_ROWS,

    "excluded_rows":
        N_MODEL_EXCLUDED,

    "model_athletes":
        N_MODEL_ATHLETES,

    "distinct_athlete_dates":
        int(
            len(
                ATHLETE_DATE
            )
        ),

    "same_day_leakage_groups":
        N_SAME_DAY_LEAKAGE_GROUPS,

    "nonstrict_previous_dates":
        N_NONSTRICT_PREVIOUS_DATES,

    "model_features_path":
        str(
            MODEL_FEATURES_PATH
        ),

    "athlete_date_path":
        str(
            ATHLETE_DATE_PATH
        ),

    "athlete_index_path":
        str(
            ATHLETE_INDEX_PATH
        ),

    "model_exclusions_path":
        str(
            MODEL_EXCLUSIONS_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "FEATURE_TABLE_READY"
    if CELL_2_PASS
    else "FEATURE_TABLE_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART R — NOTEBOOK GLOBALS
# ==================================================================================================

V6_MODEL_FEATURES = (
    MODEL_FEATURES
)


V6_ATHLETE_DATE = (
    ATHLETE_DATE.drop(
        columns=[
            "_cumulative_results"
        ],
        errors="ignore",
    )
    .copy()
)


V6_ATHLETE_IDS = (
    ATHLETE_IDS
)


V6_ATHLETE_INDEX = (
    ATHLETE_INDEX
)


NOTEBOOK_2_CELL_2_PASS = (
    CELL_2_PASS
)


# ==================================================================================================
# PART S — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — MODEL FEATURE TABLE")
print("=" * 145)


print()
print("CANONICAL → MODEL POPULATION")
print("-" * 145)

print(
    f"V4-base input rows               : "
    f"{N_V4_BASE_INPUT:,}"
)

print(
    f"Model rows                       : "
    f"{N_MODEL_ROWS:,}"
)

print(
    f"Excluded rows                    : "
    f"{N_MODEL_EXCLUDED:,}"
)

print(
    f"Row arithmetic                   : "
    f"{ROW_ARITHMETIC_PASS}"
)

print(
    f"Model athletes                   : "
    f"{N_MODEL_ATHLETES:,}"
)

print(
    f"Distinct athlete dates           : "
    f"{len(ATHLETE_DATE):,}"
)

print(
    f"Model date range                 : "
    f"{MODEL_FEATURES['result_date_dt'].min().date()} "
    f"→ "
    f"{MODEL_FEATURES['result_date_dt'].max().date()}"
)

print(
    f"Model age range                  : "
    f"{MODEL_FEATURES['age_years'].min():.4f} "
    f"→ "
    f"{MODEL_FEATURES['age_years'].max():.4f}"
)


print()
print("MODEL-LEVEL EXCLUSIONS")
print("-" * 145)

display(
    EXCLUSION_SUMMARY
)


print()
print("MODEL GROUPS")
print("-" * 145)

display(
    MODEL_GROUP_FEATURE_SUMMARY
)


print()
print("HISTORY DEPTH")
print("-" * 145)

display(
    HISTORY_DEPTH_SUMMARY
)


print()
print("CHRONOLOGY QA")
print("-" * 145)

print(
    f"Same-day leakage groups          : "
    f"{N_SAME_DAY_LEAKAGE_GROUPS:,}"
)

print(
    f"Previous-date ordering errors    : "
    f"{N_NONSTRICT_PREVIOUS_DATES:,}"
)

print(
    f"First-date prior-count errors    : "
    f"{N_FIRST_DATE_PRIOR_COUNT_ERRORS:,}"
)

print(
    f"First-date history-span errors   : "
    f"{N_FIRST_DATE_SPAN_ERRORS:,}"
)


print()
print("ENCODING QA")
print("-" * 145)

print(
    f"Invalid sex indices              : "
    f"{N_INVALID_SEX_IDX:,}"
)

print(
    f"Invalid group indices            : "
    f"{N_INVALID_GROUP_IDX:,}"
)

print(
    f"Invalid athlete indices          : "
    f"{N_INVALID_ATHLETE_IDX:,}"
)

print(
    f"Age-domain pass                  : "
    f"{AGE_DOMAIN_PASS}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Model feature table              : "
    f"{MODEL_FEATURES_PATH}"
)

print(
    f"Athlete-date chronology          : "
    f"{ATHLETE_DATE_PATH}"
)

print(
    f"Athlete index                    : "
    f"{ATHLETE_INDEX_PATH}"
)

print(
    f"Exclusion audit                  : "
    f"{MODEL_EXCLUSIONS_PATH}"
)


print()
print("=" * 145)


if CELL_2_PASS:

    print(
        "CELL 2 STATUS: V6 MODEL FEATURE TABLE PASSED"
    )

else:

    print(
        "CELL 2 STATUS: FEATURE TABLE REVIEW REQUIRED"
    )


print("=" * 145)



V6 TRAINING — MODEL FEATURE TABLE

CANONICAL → MODEL POPULATION
-------------------------------------------------------------------------------------------------------------------------------------------------
V4-base input rows               : 192,983
Model rows                       : 189,128
Excluded rows                    : 3,855
Row arithmetic                   : True
Model athletes                   : 14,478
Distinct athlete dates           : 187,971
Model date range                 : 1991-08-10 → 2026-09-18
Model age range                  : 13.0000 → 35.0000

MODEL-LEVEL EXCLUSIONS
-------------------------------------------------------------------------------------------------------------------------------------------------


,reason,rows
0,invalid_athlete,0
1,invalid_date,0
2,invalid_mark,0
3,invalid_age,0
4,age_below_13,55
5,age_above_35,3800
6,invalid_group,0



MODEL GROUPS
-------------------------------------------------------------------------------------------------------------------------------------------------


,model_group,rows,athletes,min_age,max_age,min_date,max_date
0,M_700,11577,3184,13.692273,35.000000,1999-07-16,2026-09-13
1,M_800,83514,6500,13.120050,35.000000,1991-08-10,2026-09-13
2,W_500,18700,3835,13.000000,23.932045,2005-06-18,2026-09-13
3,W_600,75337,5644,13.596446,35.000000,1999-04-03,2026-09-18



HISTORY DEPTH
-------------------------------------------------------------------------------------------------------------------------------------------------


,metric,value
0,athletes,14478
1,athletes_with_2plus_dates,11583
2,athletes_with_3plus_dates,9911
3,athletes_with_5plus_dates,7794
4,athletes_with_10plus_dates,4955



CHRONOLOGY QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Same-day leakage groups          : 0
Previous-date ordering errors    : 0
First-date prior-count errors    : 0
First-date history-span errors   : 0

ENCODING QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Invalid sex indices              : 0
Invalid group indices            : 0
Invalid athlete indices          : 0
Age-domain pass                  : True

OUTPUT
-------------------------------------------------------------------------------------------------------------------------------------------------
Model feature table              : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/v6_model_features.parquet
Athlete-date chronology          : /var/folders/q5/yf8g5p8

## Stage 03 — Temporal split and frozen spline basis

Freeze temporal partitions and the V4 spline design before any model fitting.

**Legacy source cells represented:** 3  
**Execution rule:** run only after the preceding stage has completed successfully.


In [4]:
# ==================================================================================================
# REFACTORED STAGE 03 — Temporal split and frozen spline basis
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 3
# Original source SHA256: 8635d5e087ccb7db51094841a7319a52e19a46a1fa1811b2b639b128a171e5f8
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 3 — TEMPORAL SPLIT + FROZEN V4 SPLINE BASIS
# ==================================================================================================
#
# Purpose
# -------
# Freeze the historical evaluation design BEFORE any model fitting.
#
#
# TEMPORAL SPLIT
# --------------
#
# TRAIN
#       result_date <= 2025-12-31
#
# VALIDATION
#       2026-01-01 <= result_date <= 2026-04-30
#
# TEST
#       2026-05-01 <= result_date <= 2026-08-30
#
# POST_TEST
#       result_date >= 2026-08-31
#
#
# The POST_TEST period is intentionally held outside historical model selection.
# It will later be available for the final production refit after the candidate
# V6 model has passed validation and test release gates.
#
#
# SPLINE CONTRACT
# ---------------
#
# Cubic B-spline:
#
#       degree              = 3
#       basis columns       = 7
#       include_intercept   = False
#       lower bound         = 13
#       upper bound         = 35
#
# Frozen internal knots:
#
#       17.99900066394245
#       20.10171324530962
#       22.163357221571967
#       25.31742609362273
#
#
# These knots are retained from the established V4 structural-model contract.
#
# The raw basis is centered using TRAIN-ONLY means.
#
# No validation, test or post-test observation is allowed to influence
# the spline centering constants.
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from patsy import dmatrix


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_2_PASS",
    "V6_MODEL_FEATURES",
    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through Cell 2 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_2_PASS:

    raise RuntimeError(
        "Cell 2 feature-table QA did not pass."
    )


# ==================================================================================================
# PART B — FROZEN TEMPORAL SPLIT CONTRACT
# ==================================================================================================

TRAIN_END = pd.Timestamp(
    "2025-12-31"
)


VALIDATION_START = pd.Timestamp(
    "2026-01-01"
)


VALIDATION_END = pd.Timestamp(
    "2026-04-30"
)


TEST_START = pd.Timestamp(
    "2026-05-01"
)


TEST_END = pd.Timestamp(
    "2026-08-30"
)


POST_TEST_START = pd.Timestamp(
    "2026-08-31"
)


# ==================================================================================================
# PART C — FROZEN SPLINE CONTRACT
# ==================================================================================================

SPLINE_DEGREE = 3

SPLINE_DF = 7

SPLINE_LOWER_BOUND = 13.0

SPLINE_UPPER_BOUND = 35.0


SPLINE_INTERNAL_KNOTS = np.array(
    [
        17.99900066394245,
        20.10171324530962,
        22.163357221571967,
        25.31742609362273,
    ],
    dtype=float,
)


if len(
    SPLINE_INTERNAL_KNOTS
) != 4:

    raise RuntimeError(
        "Frozen spline contract should contain exactly four internal knots."
    )


# ==================================================================================================
# PART D — COPY FEATURE TABLE
# ==================================================================================================

MODEL_DATA = (
    V6_MODEL_FEATURES.copy()
)


MODEL_DATA[
    "result_date_dt"
] = pd.to_datetime(
    MODEL_DATA[
        "result_date_dt"
    ],
    errors="coerce",
)


MODEL_DATA[
    "age_years"
] = pd.to_numeric(
    MODEL_DATA[
        "age_years"
    ],
    errors="coerce",
)


N_MODEL_DATA_ROWS = int(
    len(
        MODEL_DATA
    )
)


# ==================================================================================================
# PART E — ASSIGN TEMPORAL SPLIT
# ==================================================================================================

MODEL_DATA[
    "temporal_split"
] = None


train_mask = (
    MODEL_DATA[
        "result_date_dt"
    ]
    <=
    TRAIN_END
)


validation_mask = (
    MODEL_DATA[
        "result_date_dt"
    ]
    .between(
        VALIDATION_START,
        VALIDATION_END,
        inclusive="both",
    )
)


test_mask = (
    MODEL_DATA[
        "result_date_dt"
    ]
    .between(
        TEST_START,
        TEST_END,
        inclusive="both",
    )
)


post_test_mask = (
    MODEL_DATA[
        "result_date_dt"
    ]
    >=
    POST_TEST_START
)


MODEL_DATA.loc[
    train_mask,
    "temporal_split",
] = "TRAIN"


MODEL_DATA.loc[
    validation_mask,
    "temporal_split",
] = "VALIDATION"


MODEL_DATA.loc[
    test_mask,
    "temporal_split",
] = "TEST"


MODEL_DATA.loc[
    post_test_mask,
    "temporal_split",
] = "POST_TEST"


# ==================================================================================================
# PART F — TEMPORAL-SPLIT QA
# ==================================================================================================

N_UNASSIGNED = int(
    MODEL_DATA[
        "temporal_split"
    ]
    .isna()
    .sum()
)


N_MULTI_ASSIGNED = int(
    (
        train_mask.astype(int)
        +
        validation_mask.astype(int)
        +
        test_mask.astype(int)
        +
        post_test_mask.astype(int)
        >
        1
    )
    .sum()
)


if N_UNASSIGNED > 0:

    display(
        MODEL_DATA.loc[
            MODEL_DATA[
                "temporal_split"
            ]
            .isna()
        ]
        .head(
            100
        )
    )


if N_MULTI_ASSIGNED > 0:

    raise RuntimeError(
        "Temporal split masks overlap."
    )


# ==================================================================================================
# PART G — SPLIT DATAFRAMES
# ==================================================================================================

V6_TRAIN = (
    MODEL_DATA.loc[
        MODEL_DATA[
            "temporal_split"
        ]
        .eq(
            "TRAIN"
        )
    ]
    .copy()
)


V6_VALIDATION = (
    MODEL_DATA.loc[
        MODEL_DATA[
            "temporal_split"
        ]
        .eq(
            "VALIDATION"
        )
    ]
    .copy()
)


V6_TEST = (
    MODEL_DATA.loc[
        MODEL_DATA[
            "temporal_split"
        ]
        .eq(
            "TEST"
        )
    ]
    .copy()
)


V6_POST_TEST = (
    MODEL_DATA.loc[
        MODEL_DATA[
            "temporal_split"
        ]
        .eq(
            "POST_TEST"
        )
    ]
    .copy()
)


# ==================================================================================================
# PART H — TEMPORAL BOUNDARY QA
# ==================================================================================================

TRAIN_DATE_PASS = bool(
    V6_TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    TRAIN_END
)


VALIDATION_DATE_PASS = bool(
    V6_VALIDATION[
        "result_date_dt"
    ]
    .min()
    >=
    VALIDATION_START

    and

    V6_VALIDATION[
        "result_date_dt"
    ]
    .max()
    <=
    VALIDATION_END
)


TEST_DATE_PASS = bool(
    V6_TEST[
        "result_date_dt"
    ]
    .min()
    >=
    TEST_START

    and

    V6_TEST[
        "result_date_dt"
    ]
    .max()
    <=
    TEST_END
)


POST_TEST_DATE_PASS = bool(
    V6_POST_TEST[
        "result_date_dt"
    ]
    .min()
    >=
    POST_TEST_START
)


SPLIT_ROW_ARITHMETIC_PASS = bool(
    len(
        V6_TRAIN
    )
    +
    len(
        V6_VALIDATION
    )
    +
    len(
        V6_TEST
    )
    +
    len(
        V6_POST_TEST
    )
    ==
    N_MODEL_DATA_ROWS
)


# ==================================================================================================
# PART I — BUILD RAW SPLINE BASIS
# ==================================================================================================
#
# Explicit knots are supplied rather than inferred from the current dataset.
#
# This freezes the basis geometry across future model releases.
#
# ==================================================================================================

def build_raw_spline_basis(
    age_values,
):

    age_values = np.asarray(
        age_values,
        dtype=float,
    )


    spline_formula = (
        "0 + bs("
        "age,"
        f" knots={tuple(SPLINE_INTERNAL_KNOTS.tolist())},"
        f" degree={SPLINE_DEGREE},"
        " include_intercept=False,"
        f" lower_bound={SPLINE_LOWER_BOUND},"
        f" upper_bound={SPLINE_UPPER_BOUND}"
        ")"
    )


    basis = np.asarray(
        dmatrix(
            spline_formula,
            {
                "age":
                    age_values
            },
            return_type="dataframe",
        ),
        dtype=float,
    )


    return basis


# ==================================================================================================
# PART J — TRAIN-ONLY SPLINE CENTERING
# ==================================================================================================

B_TRAIN_RAW = build_raw_spline_basis(
    V6_TRAIN[
        "age_years"
    ]
    .to_numpy()
)


if B_TRAIN_RAW.shape[
    1
] != SPLINE_DF:

    raise RuntimeError(
        "Unexpected spline basis width.\n\n"
        f"Expected: {SPLINE_DF}\n"
        f"Actual:   {B_TRAIN_RAW.shape[1]}"
    )


SPLINE_CENTER_MEANS = (
    B_TRAIN_RAW.mean(
        axis=0
    )
)


if len(
    SPLINE_CENTER_MEANS
) != SPLINE_DF:

    raise RuntimeError(
        "Spline center vector has incorrect width."
    )


# ==================================================================================================
# PART K — ADD SPLINE FEATURES TO ALL SPLITS
# ==================================================================================================

SPLINE_RAW_COLUMNS = [
    f"spline_raw_{i}"
    for i in range(
        SPLINE_DF
    )
]


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(
        SPLINE_DF
    )
]


def add_spline_features(
    df,
):

    out = (
        df.copy()
    )


    raw_basis = build_raw_spline_basis(
        out[
            "age_years"
        ]
        .to_numpy()
    )


    if raw_basis.shape[
        1
    ] != SPLINE_DF:

        raise RuntimeError(
            "Spline basis width changed unexpectedly."
        )


    centered_basis = (
        raw_basis
        -
        SPLINE_CENTER_MEANS
    )


    for idx in range(
        SPLINE_DF
    ):

        out[
            SPLINE_RAW_COLUMNS[
                idx
            ]
        ] = (
            raw_basis[
                :,
                idx
            ]
        )


        out[
            SPLINE_COLUMNS[
                idx
            ]
        ] = (
            centered_basis[
                :,
                idx
            ]
        )


    return out


V6_TRAIN = add_spline_features(
    V6_TRAIN
)


V6_VALIDATION = add_spline_features(
    V6_VALIDATION
)


V6_TEST = add_spline_features(
    V6_TEST
)


V6_POST_TEST = add_spline_features(
    V6_POST_TEST
)


# ==================================================================================================
# PART L — REASSEMBLE MASTER SPLIT TABLE
# ==================================================================================================

V6_SPLIT_DATA = pd.concat(
    [
        V6_TRAIN,
        V6_VALIDATION,
        V6_TEST,
        V6_POST_TEST,
    ],
    ignore_index=True,
)


V6_SPLIT_DATA = (
    V6_SPLIT_DATA.sort_values(
        [
            "result_date_dt",
            "aaAthleteId",
            "model_group",
            "mark_numeric_m",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART M — SPLINE QA
# ==================================================================================================

N_SPLINE_NAN = int(
    V6_SPLIT_DATA[
        SPLINE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)


N_SPLINE_INF = int(
    np.isinf(
        V6_SPLIT_DATA[
            SPLINE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )
    .sum()
)


TRAIN_CENTER_CHECK = (
    V6_TRAIN[
        SPLINE_COLUMNS
    ]
    .mean(
        axis=0
    )
    .to_numpy(
        dtype=float
    )
)


MAX_ABS_TRAIN_CENTER = float(
    np.abs(
        TRAIN_CENTER_CHECK
    )
    .max()
)


TRAIN_CENTER_PASS = bool(
    MAX_ABS_TRAIN_CENTER
    <
    1e-12
)


# ==================================================================================================
# PART N — SPLIT POPULATION SUMMARY
# ==================================================================================================

TEMPORAL_SPLIT_SUMMARY = (
    V6_SPLIT_DATA.groupby(
        "temporal_split",
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),

        first_date=(
            "result_date_dt",
            "min",
        ),

        last_date=(
            "result_date_dt",
            "max",
        ),

        mean_age=(
            "age_years",
            "mean",
        ),
    )
)


SPLIT_ORDER = {
    "TRAIN":
        0,

    "VALIDATION":
        1,

    "TEST":
        2,

    "POST_TEST":
        3,
}


TEMPORAL_SPLIT_SUMMARY[
    "_order"
] = (
    TEMPORAL_SPLIT_SUMMARY[
        "temporal_split"
    ]
    .map(
        SPLIT_ORDER
    )
)


TEMPORAL_SPLIT_SUMMARY = (
    TEMPORAL_SPLIT_SUMMARY.sort_values(
        "_order"
    )
    .drop(
        columns=[
            "_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART O — SPLIT BY MODEL GROUP
# ==================================================================================================

TEMPORAL_GROUP_SUMMARY = (
    V6_SPLIT_DATA.groupby(
        [
            "temporal_split",
            "model_group",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),
    )
)


TEMPORAL_GROUP_SUMMARY[
    "_order"
] = (
    TEMPORAL_GROUP_SUMMARY[
        "temporal_split"
    ]
    .map(
        SPLIT_ORDER
    )
)


TEMPORAL_GROUP_SUMMARY = (
    TEMPORAL_GROUP_SUMMARY.sort_values(
        [
            "_order",
            "model_group",
        ]
    )
    .drop(
        columns=[
            "_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART P — ATHLETE OVERLAP SUMMARY
# ==================================================================================================
#
# Athlete overlap between temporal periods is expected and desirable.
#
# The split is temporal, not athlete-disjoint.
#
# ==================================================================================================

TRAIN_ATHLETES = set(
    V6_TRAIN[
        "aaAthleteId"
    ]
    .unique()
)


VALIDATION_ATHLETES = set(
    V6_VALIDATION[
        "aaAthleteId"
    ]
    .unique()
)


TEST_ATHLETES = set(
    V6_TEST[
        "aaAthleteId"
    ]
    .unique()
)


POST_TEST_ATHLETES = set(
    V6_POST_TEST[
        "aaAthleteId"
    ]
    .unique()
)


ATHLETE_OVERLAP_SUMMARY = pd.DataFrame(
    [
        {
            "comparison":
                "VALIDATION seen in TRAIN",

            "athletes":
                len(
                    VALIDATION_ATHLETES
                    &
                    TRAIN_ATHLETES
                ),

            "total_target_athletes":
                len(
                    VALIDATION_ATHLETES
                ),
        },

        {
            "comparison":
                "TEST seen in TRAIN",

            "athletes":
                len(
                    TEST_ATHLETES
                    &
                    TRAIN_ATHLETES
                ),

            "total_target_athletes":
                len(
                    TEST_ATHLETES
                ),
        },

        {
            "comparison":
                "POST_TEST seen in TRAIN",

            "athletes":
                len(
                    POST_TEST_ATHLETES
                    &
                    TRAIN_ATHLETES
                ),

            "total_target_athletes":
                len(
                    POST_TEST_ATHLETES
                ),
        },
    ]
)


ATHLETE_OVERLAP_SUMMARY[
    "seen_pct"
] = (
    100.0
    *
    ATHLETE_OVERLAP_SUMMARY[
        "athletes"
    ]
    /
    ATHLETE_OVERLAP_SUMMARY[
        "total_target_athletes"
    ]
)


# ==================================================================================================
# PART Q — CELL QA
# ==================================================================================================

CELL_3_PASS = bool(

    N_UNASSIGNED == 0

    and N_MULTI_ASSIGNED == 0

    and SPLIT_ROW_ARITHMETIC_PASS

    and TRAIN_DATE_PASS

    and VALIDATION_DATE_PASS

    and TEST_DATE_PASS

    and POST_TEST_DATE_PASS

    and N_SPLINE_NAN == 0

    and N_SPLINE_INF == 0

    and TRAIN_CENTER_PASS
)


# ==================================================================================================
# PART R — SAVE ARTIFACTS
# ==================================================================================================

TRAIN_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_train.parquet"
)


VALIDATION_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_validation.parquet"
)


TEST_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_test.parquet"
)


POST_TEST_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_post_test.parquet"
)


SPLIT_DATA_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_temporal_split_full.parquet"
)


SPLINE_METADATA_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_spline_metadata.json"
)


SPLIT_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_temporal_split_summary.parquet"
)


V6_TRAIN.to_parquet(
    TRAIN_PATH,
    index=False,
)


V6_VALIDATION.to_parquet(
    VALIDATION_PATH,
    index=False,
)


V6_TEST.to_parquet(
    TEST_PATH,
    index=False,
)


V6_POST_TEST.to_parquet(
    POST_TEST_PATH,
    index=False,
)


V6_SPLIT_DATA.to_parquet(
    SPLIT_DATA_PATH,
    index=False,
)


TEMPORAL_SPLIT_SUMMARY.to_parquet(
    SPLIT_SUMMARY_PATH,
    index=False,
)


SPLINE_METADATA = {

    "degree":
        SPLINE_DEGREE,

    "df":
        SPLINE_DF,

    "include_intercept":
        False,

    "lower_bound":
        SPLINE_LOWER_BOUND,

    "upper_bound":
        SPLINE_UPPER_BOUND,

    "internal_knots":
        SPLINE_INTERNAL_KNOTS.tolist(),

    "raw_columns":
        SPLINE_RAW_COLUMNS,

    "centered_columns":
        SPLINE_COLUMNS,

    "center_means_train":
        SPLINE_CENTER_MEANS.tolist(),

    "centering_population":
        "TRAIN_ONLY",

    "train_end":
        TRAIN_END.isoformat(),
}


with SPLINE_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SPLINE_METADATA,
        f,
        indent=2,
    )


# ==================================================================================================
# PART S — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "temporal_split"
] = {

    "status":
        (
            "PASSED"
            if CELL_3_PASS
            else "FAILED"
        ),

    "train":
        {
            "start":
                str(
                    V6_TRAIN[
                        "result_date_dt"
                    ]
                    .min()
                    .date()
                ),

            "end":
                str(
                    TRAIN_END.date()
                ),

            "rows":
                int(
                    len(
                        V6_TRAIN
                    )
                ),
        },

    "validation":
        {
            "start":
                str(
                    VALIDATION_START.date()
                ),

            "end":
                str(
                    VALIDATION_END.date()
                ),

            "rows":
                int(
                    len(
                        V6_VALIDATION
                    )
                ),
        },

    "test":
        {
            "start":
                str(
                    TEST_START.date()
                ),

            "end":
                str(
                    TEST_END.date()
                ),

            "rows":
                int(
                    len(
                        V6_TEST
                    )
                ),
        },

    "post_test":
        {
            "start":
                str(
                    POST_TEST_START.date()
                ),

            "end":
                str(
                    V6_POST_TEST[
                        "result_date_dt"
                    ]
                    .max()
                    .date()
                ),

            "rows":
                int(
                    len(
                        V6_POST_TEST
                    )
                ),
        },
}


TRAINING_MANIFEST[
    "spline_basis"
] = SPLINE_METADATA


TRAINING_MANIFEST[
    "status"
] = (
    "TEMPORAL_SPLIT_AND_SPLINE_READY"
    if CELL_3_PASS
    else "TEMPORAL_SPLIT_AND_SPLINE_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART T — NOTEBOOK GLOBALS
# ==================================================================================================

V6_SPLINE_COLUMNS = (
    SPLINE_COLUMNS
)


V6_SPLINE_RAW_COLUMNS = (
    SPLINE_RAW_COLUMNS
)


V6_SPLINE_CENTER_MEANS = (
    SPLINE_CENTER_MEANS
)


V6_SPLINE_INTERNAL_KNOTS = (
    SPLINE_INTERNAL_KNOTS
)


NOTEBOOK_2_CELL_3_PASS = (
    CELL_3_PASS
)


# ==================================================================================================
# PART U — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — TEMPORAL SPLIT + FROZEN SPLINE BASIS")
print("=" * 145)


print()
print("TEMPORAL SPLIT")
print("-" * 145)

display(
    TEMPORAL_SPLIT_SUMMARY
)


print()
print("TEMPORAL SPLIT BY MODEL GROUP")
print("-" * 145)

display(
    TEMPORAL_GROUP_SUMMARY
)


print()
print("ATHLETE OVERLAP")
print("-" * 145)

display(
    ATHLETE_OVERLAP_SUMMARY
)


print()
print("SPLINE CONTRACT")
print("-" * 145)

print(
    f"Degree                           : "
    f"{SPLINE_DEGREE}"
)

print(
    f"Basis columns                    : "
    f"{SPLINE_DF}"
)

print(
    f"Lower bound                      : "
    f"{SPLINE_LOWER_BOUND}"
)

print(
    f"Upper bound                      : "
    f"{SPLINE_UPPER_BOUND}"
)

print(
    "Internal knots                   : "
    f"{SPLINE_INTERNAL_KNOTS.tolist()}"
)

print()
print(
    "TRAIN-only center means:"
)

print(
    SPLINE_CENTER_MEANS
)


print()
print("TEMPORAL QA")
print("-" * 145)

print(
    f"Unassigned rows                  : "
    f"{N_UNASSIGNED:,}"
)

print(
    f"Multiply-assigned rows           : "
    f"{N_MULTI_ASSIGNED:,}"
)

print(
    f"Split row arithmetic             : "
    f"{SPLIT_ROW_ARITHMETIC_PASS}"
)

print(
    f"Train dates                      : "
    f"{TRAIN_DATE_PASS}"
)

print(
    f"Validation dates                 : "
    f"{VALIDATION_DATE_PASS}"
)

print(
    f"Test dates                       : "
    f"{TEST_DATE_PASS}"
)

print(
    f"Post-test dates                  : "
    f"{POST_TEST_DATE_PASS}"
)


print()
print("SPLINE QA")
print("-" * 145)

print(
    f"Spline NaN values                : "
    f"{N_SPLINE_NAN:,}"
)

print(
    f"Spline infinite values           : "
    f"{N_SPLINE_INF:,}"
)

print(
    f"Max abs TRAIN centered mean      : "
    f"{MAX_ABS_TRAIN_CENTER:.3e}"
)

print(
    f"TRAIN centering pass             : "
    f"{TRAIN_CENTER_PASS}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Train dataset                    : "
    f"{TRAIN_PATH}"
)

print(
    f"Validation dataset               : "
    f"{VALIDATION_PATH}"
)

print(
    f"Test dataset                     : "
    f"{TEST_PATH}"
)

print(
    f"Post-test dataset                : "
    f"{POST_TEST_PATH}"
)

print(
    f"Spline metadata                  : "
    f"{SPLINE_METADATA_PATH}"
)


print()
print("=" * 145)


if CELL_3_PASS:

    print(
        "CELL 3 STATUS: TEMPORAL SPLIT + SPLINE BASIS PASSED"
    )

else:

    print(
        "CELL 3 STATUS: TEMPORAL/SPLINE REVIEW REQUIRED"
    )


print("=" * 145)



V6 TRAINING — TEMPORAL SPLIT + FROZEN SPLINE BASIS

TEMPORAL SPLIT
-------------------------------------------------------------------------------------------------------------------------------------------------


,temporal_split,rows,athletes,first_date,last_date,mean_age
0,TRAIN,170644,12945,1991-08-10,2025-12-27,21.758625
1,VALIDATION,4987,2798,2026-01-04,2026-04-30,20.864508
2,TEST,12865,5200,2026-05-01,2026-08-30,21.555414
3,POST_TEST,632,564,2026-08-31,2026-09-18,21.566311



TEMPORAL SPLIT BY MODEL GROUP
-------------------------------------------------------------------------------------------------------------------------------------------------


,temporal_split,model_group,rows,athletes
0,TRAIN,M_700,10401,2859
1,TRAIN,M_800,75476,5798
2,TRAIN,W_500,16968,3426
3,TRAIN,W_600,67799,5035
4,VALIDATION,M_700,283,170
5,VALIDATION,M_800,2238,1318
6,VALIDATION,W_500,465,285
7,VALIDATION,W_600,2001,1068
8,TEST,M_700,824,416
9,TEST,M_800,5567,2324



ATHLETE OVERLAP
-------------------------------------------------------------------------------------------------------------------------------------------------


,comparison,athletes,total_target_athletes,seen_pct
0,VALIDATION seen in TRAIN,2299,2798,82.165833
1,TEST seen in TRAIN,4023,5200,77.365385
2,POST_TEST seen in TRAIN,420,564,74.468085



SPLINE CONTRACT
-------------------------------------------------------------------------------------------------------------------------------------------------
Degree                           : 3
Basis columns                    : 7
Lower bound                      : 13.0
Upper bound                      : 35.0
Internal knots                   : [17.99900066394245, 20.10171324530962, 22.163357221571967, 25.31742609362273]

TRAIN-only center means:
[0.06733439 0.16285735 0.2857551  0.26490824 0.12561033 0.05900119
 0.02327416]

TEMPORAL QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Unassigned rows                  : 0
Multiply-assigned rows           : 0
Split row arithmetic             : True
Train dates                      : True
Validation dates                 : True
Test dates                       : True
Post-test dates                  : True

SPLINE QA
--------------------

## Stage 04 — Structural V4 Bayesian model

Fit the structural Bayesian trajectory model and persist its versioned posterior/specification artifacts.

**Legacy source cells represented:** 4  
**Execution rule:** run only after the preceding stage has completed successfully.


In [5]:
# ==================================================================================================
# REFACTORED STAGE 04 — Structural V4 Bayesian model
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 4
# Original source SHA256: 7d22ce795b101e22c7f8600500cad40392bc600e32da8a5b59435b45f6a8e2a9
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 4 — FIT STRUCTURAL V4 BAYESIAN MODEL
# ==================================================================================================
#
# Structural model
# ----------------
#
# y_it ~ StudentT(
#     nu,
#     mu_it,
#     sigma_group[group_i]
# )
#
#
# mu_it =
#
#     alpha_sex[sex_i]
#
#     + sex-specific age spline
#
#     + youth_implement_effect[sex_i] * youth_implement_i
#
#     + athlete_intercept[athlete_i]
#
#     + athlete_slope[athlete_i] * age_scaled_i
#
#
# age_scaled =
#
#     (age_years - 18) / 5
#
#
# PRIORS — HISTORICAL V4 CONTRACT
# --------------------------------
#
# alpha_sex
#     ~ Normal(45, 15)
#
# sigma_spline
#     ~ HalfNormal(2)
#
# spline_start
#     ~ Normal(0, 3)
#
# spline_delta
#     ~ Normal(0, sigma_spline)
#
# beta_spline
#     = spline_start + cumulative spline_delta
#
# youth_implement_effect
#     ~ Normal(0, 8)
#
# sigma_athlete_intercept
#     ~ HalfNormal(10)
#
# athlete_intercept_z
#     ~ Normal(0, 1)
#
# athlete_intercept
#     = sigma_athlete_intercept * athlete_intercept_z
#
# sigma_athlete_slope
#     ~ HalfNormal(4)
#
# athlete_slope_z
#     ~ Normal(0, 1)
#
# athlete_slope
#     = sigma_athlete_slope * athlete_slope_z
#
# sigma_group
#     ~ HalfNormal(5)
#
# nu_minus_two
#     ~ Exponential(0.1)
#
# nu
#     = nu_minus_two + 2
#
#
# FITTING
# -------
#
# ADVI iterations      = 40,000
# posterior draws      = 2,000
# random seed          = 42
#
#
# IMPORTANT
# ---------
#
# Only TRAIN is used here.
#
# VALIDATION, TEST and POST_TEST remain completely untouched.
#
# ==================================================================================================

import json
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_3_PASS",

    "V6_TRAIN",
    "V6_VALIDATION",
    "V6_TEST",
    "V6_POST_TEST",

    "V6_SPLINE_COLUMNS",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through Cell 3 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_3_PASS:

    raise RuntimeError(
        "Cell 3 temporal split / spline QA did not pass."
    )


# ==================================================================================================
# PART B — FROZEN MODEL SETTINGS
# ==================================================================================================

RANDOM_SEED = 42

ADVI_ITERATIONS = 40_000

POSTERIOR_DRAWS = 2_000


SEX_ORDER_V4 = [
    "M",
    "W",
]


GROUP_ORDER_V4 = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_SEX = len(
    SEX_ORDER_V4
)


N_GROUPS = len(
    GROUP_ORDER_V4
)


N_SPLINE = len(
    V6_SPLINE_COLUMNS
)


N_SPLINE_STEPS = (
    N_SPLINE
    -
    1
)


if N_SPLINE != 7:

    raise RuntimeError(
        f"V4 requires 7 spline columns; found {N_SPLINE}."
    )


# ==================================================================================================
# PART C — TRAIN-ONLY DATA
# ==================================================================================================

V4_TRAIN = (
    V6_TRAIN.copy()
)


V4_TRAIN = (
    V4_TRAIN.sort_values(
        [
            "result_date_dt",
            "aaAthleteId",
            "model_group",
            "mark_numeric_m",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


N_TRAIN_ROWS = int(
    len(
        V4_TRAIN
    )
)


# ==================================================================================================
# PART D — TRAIN-ONLY ATHLETE INDEX
# ==================================================================================================
#
# We deliberately do NOT reuse the all-period athlete_idx.
#
# Only athletes visible in TRAIN receive fitted random effects.
#
# This later allows:
#
#     seen athlete    -> use fitted athlete random effect
#     unseen athlete  -> integrate / predict using population distribution
#
# ==================================================================================================

V4_TRAIN_ATHLETE_IDS = np.array(
    sorted(
        V4_TRAIN[
            "aaAthleteId"
        ]
        .astype(
            "int64"
        )
        .unique()
        .tolist()
    ),
    dtype=np.int64,
)


V4_TRAIN_ATHLETE_INDEX = {
    int(
        athlete_id
    ):
        idx

    for idx, athlete_id
    in enumerate(
        V4_TRAIN_ATHLETE_IDS
    )
}


V4_TRAIN[
    "train_athlete_idx"
] = (
    V4_TRAIN[
        "aaAthleteId"
    ]
    .map(
        V4_TRAIN_ATHLETE_INDEX
    )
    .astype(
        "int32"
    )
)


N_TRAIN_ATHLETES = int(
    len(
        V4_TRAIN_ATHLETE_IDS
    )
)


# ==================================================================================================
# PART E — NUMPY MODEL ARRAYS
# ==================================================================================================

Y_TRAIN = (
    V4_TRAIN[
        "mark_numeric_m"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


SEX_IDX_TRAIN = (
    V4_TRAIN[
        "sex_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


GROUP_IDX_TRAIN = (
    V4_TRAIN[
        "group_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


YOUTH_TRAIN = (
    V4_TRAIN[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


AGE_SCALED_TRAIN = (
    V4_TRAIN[
        "age_scaled"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


ATHLETE_IDX_TRAIN = (
    V4_TRAIN[
        "train_athlete_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


B_TRAIN = (
    V4_TRAIN[
        V6_SPLINE_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)


# ==================================================================================================
# PART F — INPUT QA
# ==================================================================================================

INPUT_QA = {
    "y_finite":
        bool(
            np.isfinite(
                Y_TRAIN
            )
            .all()
        ),

    "age_scaled_finite":
        bool(
            np.isfinite(
                AGE_SCALED_TRAIN
            )
            .all()
        ),

    "spline_finite":
        bool(
            np.isfinite(
                B_TRAIN
            )
            .all()
        ),

    "sex_index_valid":
        bool(
            (
                (SEX_IDX_TRAIN >= 0)
                &
                (SEX_IDX_TRAIN < N_SEX)
            )
            .all()
        ),

    "group_index_valid":
        bool(
            (
                (GROUP_IDX_TRAIN >= 0)
                &
                (GROUP_IDX_TRAIN < N_GROUPS)
            )
            .all()
        ),

    "athlete_index_valid":
        bool(
            (
                (ATHLETE_IDX_TRAIN >= 0)
                &
                (
                    ATHLETE_IDX_TRAIN
                    <
                    N_TRAIN_ATHLETES
                )
            )
            .all()
        ),

    "spline_width":
        bool(
            B_TRAIN.shape[
                1
            ]
            ==
            N_SPLINE
        ),
}


INPUT_QA_PASS = bool(
    all(
        INPUT_QA.values()
    )
)


if not INPUT_QA_PASS:

    raise RuntimeError(
        f"V4 training input QA failed:\n{INPUT_QA}"
    )


# ==================================================================================================
# PART G — MODEL COORDINATES
# ==================================================================================================

V4_COORDS = {

    "sex":
        SEX_ORDER_V4,

    "group":
        GROUP_ORDER_V4,

    "athlete":
        V4_TRAIN_ATHLETE_IDS,

    "spline":
        np.arange(
            N_SPLINE,
            dtype=int,
        ),

    "spline_step":
        np.arange(
            N_SPLINE_STEPS,
            dtype=int,
        ),
}


# ==================================================================================================
# PART H — BUILD V4 PYMC MODEL
# ==================================================================================================

with pm.Model(
    coords=V4_COORDS
) as V4_MODEL:

    # ----------------------------------------------------------------------------------------------
    # Sex-specific baseline
    # ----------------------------------------------------------------------------------------------

    alpha_sex = pm.Normal(
        "alpha_sex",
        mu=45.0,
        sigma=15.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Smooth sex-specific spline coefficient prior
    # ----------------------------------------------------------------------------------------------

    sigma_spline = pm.HalfNormal(
        "sigma_spline",
        sigma=2.0,
        dims="sex",
    )


    spline_start = pm.Normal(
        "spline_start",
        mu=0.0,
        sigma=3.0,
        dims="sex",
    )


    spline_delta = pm.Normal(
        "spline_delta",
        mu=0.0,
        sigma=
            sigma_spline[
                :,
                None
            ],
        dims=(
            "sex",
            "spline_step",
        ),
    )


    beta_spline_tensor = pt.concatenate(
        [
            spline_start[
                :,
                None
            ],

            (
                spline_start[
                    :,
                    None
                ]
                +
                pt.cumsum(
                    spline_delta,
                    axis=1,
                )
            ),
        ],
        axis=1,
    )


    beta_spline = pm.Deterministic(
        "beta_spline",
        beta_spline_tensor,
        dims=(
            "sex",
            "spline",
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Youth implement effect
    # ----------------------------------------------------------------------------------------------

    youth_implement_effect = pm.Normal(
        "youth_implement_effect",
        mu=0.0,
        sigma=8.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random intercept — non-centered
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_intercept = pm.HalfNormal(
        "sigma_athlete_intercept",
        sigma=10.0,
    )


    athlete_intercept_z = pm.Normal(
        "athlete_intercept_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_intercept = pm.Deterministic(
        "athlete_intercept",
        (
            sigma_athlete_intercept
            *
            athlete_intercept_z
        ),
        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random age slope — non-centered
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_slope = pm.HalfNormal(
        "sigma_athlete_slope",
        sigma=4.0,
    )


    athlete_slope_z = pm.Normal(
        "athlete_slope_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_slope = pm.Deterministic(
        "athlete_slope",
        (
            sigma_athlete_slope
            *
            athlete_slope_z
        ),
        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Group-specific Student-t residual scale
    # ----------------------------------------------------------------------------------------------

    sigma_group = pm.HalfNormal(
        "sigma_group",
        sigma=5.0,
        dims="group",
    )


    # ----------------------------------------------------------------------------------------------
    # Student-t degrees of freedom
    # ----------------------------------------------------------------------------------------------

    nu_minus_two = pm.Exponential(
        "nu_minus_two",
        lam=0.1,
    )


    nu = pm.Deterministic(
        "nu",
        (
            nu_minus_two
            +
            2.0
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Structural mean
    # ----------------------------------------------------------------------------------------------

    spline_effect = pt.sum(
        (
            B_TRAIN
            *
            beta_spline[
                SEX_IDX_TRAIN
            ]
        ),
        axis=1,
    )


    mu_train = (
        alpha_sex[
            SEX_IDX_TRAIN
        ]

        +
        spline_effect

        +
        (
            youth_implement_effect[
                SEX_IDX_TRAIN
            ]
            *
            YOUTH_TRAIN
        )

        +
        athlete_intercept[
            ATHLETE_IDX_TRAIN
        ]

        +
        (
            athlete_slope[
                ATHLETE_IDX_TRAIN
            ]
            *
            AGE_SCALED_TRAIN
        )
    )


    pm.Deterministic(
        "mu_train",
        mu_train,
    )


    # ----------------------------------------------------------------------------------------------
    # Student-t likelihood
    # ----------------------------------------------------------------------------------------------

    mark_obs = pm.StudentT(
        "mark_obs",
        nu=nu,
        mu=mu_train,
        sigma=
            sigma_group[
                GROUP_IDX_TRAIN
            ],
        observed=
            Y_TRAIN,
    )


# ==================================================================================================
# PART I — MODEL STRUCTURE QA
# ==================================================================================================

EXPECTED_V4_NAMED_VARS = {
    "alpha_sex",
    "sigma_spline",
    "spline_start",
    "spline_delta",
    "beta_spline",
    "youth_implement_effect",
    "sigma_athlete_intercept",
    "athlete_intercept_z",
    "athlete_intercept",
    "sigma_athlete_slope",
    "athlete_slope_z",
    "athlete_slope",
    "sigma_group",
    "nu_minus_two",
    "nu",
    "mu_train",
    "mark_obs",
}


MODEL_VAR_NAMES = set(
    V4_MODEL.named_vars.keys()
)


MISSING_MODEL_VARS = (
    EXPECTED_V4_NAMED_VARS
    -
    MODEL_VAR_NAMES
)


if MISSING_MODEL_VARS:

    raise RuntimeError(
        "V4 PyMC model is missing expected variables:\n"
        + "\n".join(
            sorted(
                MISSING_MODEL_VARS
            )
        )
    )


# ==================================================================================================
# PART J — FIT V4 USING ADVI
# ==================================================================================================

with V4_MODEL:

    V4_APPROX = pm.fit(
        n=
            ADVI_ITERATIONS,

        method=
            "advi",

        random_seed=
            RANDOM_SEED,

        progressbar=
            True,
    )


# ==================================================================================================
# PART K — ADVI LOSS HISTORY
# ==================================================================================================

ADVI_HISTORY = np.asarray(
    V4_APPROX.hist,
    dtype=np.float64,
)


ADVI_HISTORY_DF = pd.DataFrame(
    {
        "iteration":
            np.arange(
                1,
                len(
                    ADVI_HISTORY
                )
                +
                1,
                dtype=np.int32,
            ),

        "loss":
            ADVI_HISTORY,
    }
)


N_ADVI_STEPS = int(
    len(
        ADVI_HISTORY
    )
)


ADVI_TAIL = (
    ADVI_HISTORY[
        -min(
            1000,
            len(
                ADVI_HISTORY
            ),
        ):
    ]
)


N_NONFINITE_ADVI_TAIL = int(
    (
        ~np.isfinite(
            ADVI_TAIL
        )
    )
    .sum()
)


ADVI_INITIAL_MEDIAN = float(
    np.nanmedian(
        ADVI_HISTORY[
            :min(
                500,
                len(
                    ADVI_HISTORY
                ),
            )
        ]
    )
)


ADVI_FINAL_MEDIAN = float(
    np.nanmedian(
        ADVI_HISTORY[
            -min(
                500,
                len(
                    ADVI_HISTORY
                ),
            ):
        ]
    )
)


ADVI_LOSS_CHANGE = (
    ADVI_FINAL_MEDIAN
    -
    ADVI_INITIAL_MEDIAN
)


# ==================================================================================================
# PART L — SAMPLE VARIATIONAL POSTERIOR
# ==================================================================================================

with V4_MODEL:

    V4_IDATA = V4_APPROX.sample(
        draws=
            POSTERIOR_DRAWS,

        random_seed=
            RANDOM_SEED,

        return_inferencedata=
            True,
    )


# ==================================================================================================
# PART M — POSTERIOR CONTRACT QA
# ==================================================================================================

EXPECTED_POSTERIOR_VARS = {
    "alpha_sex",
    "sigma_spline",
    "spline_start",
    "spline_delta",
    "beta_spline",
    "youth_implement_effect",
    "sigma_athlete_intercept",
    "athlete_intercept_z",
    "athlete_intercept",
    "sigma_athlete_slope",
    "athlete_slope_z",
    "athlete_slope",
    "sigma_group",
    "nu_minus_two",
    "nu",
}


POSTERIOR_VARS = set(
    V4_IDATA.posterior.data_vars
)


MISSING_POSTERIOR_VARS = (
    EXPECTED_POSTERIOR_VARS
    -
    POSTERIOR_VARS
)


N_POSTERIOR_DRAWS = int(
    V4_IDATA.posterior.sizes[
        "draw"
    ]
)


N_POSTERIOR_CHAINS = int(
    V4_IDATA.posterior.sizes[
        "chain"
    ]
)


POSTERIOR_ATHLETE_DIM = int(
    V4_IDATA.posterior[
        "athlete_intercept"
    ]
    .sizes[
        "athlete"
    ]
)


POSTERIOR_SPLINE_DIM = int(
    V4_IDATA.posterior[
        "beta_spline"
    ]
    .sizes[
        "spline"
    ]
)


POSTERIOR_SEX_DIM = int(
    V4_IDATA.posterior[
        "beta_spline"
    ]
    .sizes[
        "sex"
    ]
)


POSTERIOR_GROUP_DIM = int(
    V4_IDATA.posterior[
        "sigma_group"
    ]
    .sizes[
        "group"
    ]
)


# ==================================================================================================
# PART N — SMALL-PARAMETER FINITENESS QA
# ==================================================================================================
#
# Avoid materializing the large athlete posterior arrays again merely for QA.
#
# ==================================================================================================

SMALL_POSTERIOR_VARS = [
    "alpha_sex",
    "sigma_spline",
    "spline_start",
    "spline_delta",
    "beta_spline",
    "youth_implement_effect",
    "sigma_athlete_intercept",
    "sigma_athlete_slope",
    "sigma_group",
    "nu",
]


POSTERIOR_NONFINITE_COUNTS = {}


for var_name in SMALL_POSTERIOR_VARS:

    values = (
        V4_IDATA.posterior[
            var_name
        ]
        .to_numpy()
    )


    POSTERIOR_NONFINITE_COUNTS[
        var_name
    ] = int(
        (
            ~np.isfinite(
                values
            )
        )
        .sum()
    )


N_SMALL_POSTERIOR_NONFINITE = int(
    sum(
        POSTERIOR_NONFINITE_COUNTS.values()
    )
)


# ==================================================================================================
# PART O — POSTERIOR SUMMARY
# ==================================================================================================

SUMMARY_VARS = [
    "alpha_sex",
    "sigma_spline",
    "spline_start",
    "beta_spline",
    "youth_implement_effect",
    "sigma_athlete_intercept",
    "sigma_athlete_slope",
    "sigma_group",
    "nu",
]


V4_POSTERIOR_SUMMARY = (
    az.summary(
        V4_IDATA,

        var_names=
            SUMMARY_VARS,

        kind=
            "stats",

        hdi_prob=
            0.94,
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "parameter"
        }
    )
)


# ==================================================================================================
# PART P — FIT QA
# ==================================================================================================

ADVI_LENGTH_PASS = bool(
    N_ADVI_STEPS
    ==
    ADVI_ITERATIONS
)


ADVI_TAIL_PASS = bool(
    N_NONFINITE_ADVI_TAIL
    ==
    0
)


POSTERIOR_DRAW_PASS = bool(
    N_POSTERIOR_DRAWS
    ==
    POSTERIOR_DRAWS
)


POSTERIOR_VARIABLE_PASS = bool(
    len(
        MISSING_POSTERIOR_VARS
    )
    ==
    0
)


POSTERIOR_DIMENSION_PASS = bool(

    POSTERIOR_ATHLETE_DIM
    ==
    N_TRAIN_ATHLETES

    and

    POSTERIOR_SPLINE_DIM
    ==
    N_SPLINE

    and

    POSTERIOR_SEX_DIM
    ==
    N_SEX

    and

    POSTERIOR_GROUP_DIM
    ==
    N_GROUPS
)


POSTERIOR_FINITE_PASS = bool(
    N_SMALL_POSTERIOR_NONFINITE
    ==
    0
)


CELL_4_PASS = bool(

    INPUT_QA_PASS

    and ADVI_LENGTH_PASS

    and ADVI_TAIL_PASS

    and POSTERIOR_DRAW_PASS

    and POSTERIOR_VARIABLE_PASS

    and POSTERIOR_DIMENSION_PASS

    and POSTERIOR_FINITE_PASS
)


# ==================================================================================================
# PART Q — SAVE TRAIN ATHLETE INDEX
# ==================================================================================================

V4_TRAIN_ATHLETE_INDEX_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_train_athlete_index.parquet"
)


pd.DataFrame(
    {
        "aaAthleteId":
            V4_TRAIN_ATHLETE_IDS,

        "train_athlete_idx":
            np.arange(
                N_TRAIN_ATHLETES,
                dtype=np.int32,
            ),
    }
).to_parquet(
    V4_TRAIN_ATHLETE_INDEX_PATH,
    index=False,
)


# ==================================================================================================
# PART R — SAVE ADVI HISTORY
# ==================================================================================================

V4_ADVI_HISTORY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_advi_loss_history.parquet"
)


ADVI_HISTORY_DF.to_parquet(
    V4_ADVI_HISTORY_PATH,
    index=False,
)


# ==================================================================================================
# PART S — SAVE POSTERIOR SUMMARY
# ==================================================================================================

V4_POSTERIOR_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_posterior_summary.parquet"
)


V4_POSTERIOR_SUMMARY.to_parquet(
    V4_POSTERIOR_SUMMARY_PATH,
    index=False,
)


# ==================================================================================================
# PART T — SAVE FULL V4 POSTERIOR
# ==================================================================================================

V4_POSTERIOR_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_structural_posterior.nc"
)


V4_IDATA.to_netcdf(
    V4_POSTERIOR_PATH
)


# ==================================================================================================
# PART U — SAVE MODEL SPECIFICATION
# ==================================================================================================

V4_MODEL_SPEC_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_model_spec.json"
)


V4_MODEL_SPEC = {

    "model":
        "V4_STRUCTURAL",

    "training_rows":
        N_TRAIN_ROWS,

    "training_athletes":
        N_TRAIN_ATHLETES,

    "sex_order":
        SEX_ORDER_V4,

    "group_order":
        GROUP_ORDER_V4,

    "spline_columns":
        list(
            V6_SPLINE_COLUMNS
        ),

    "age_scaled":
        "(age_years - 18) / 5",

    "likelihood":
        "StudentT",

    "priors": {

        "alpha_sex":
            "Normal(45,15)",

        "sigma_spline":
            "HalfNormal(2)",

        "spline_start":
            "Normal(0,3)",

        "spline_delta":
            "Normal(0,sigma_spline)",

        "youth_implement_effect":
            "Normal(0,8)",

        "sigma_athlete_intercept":
            "HalfNormal(10)",

        "athlete_intercept_z":
            "Normal(0,1)",

        "sigma_athlete_slope":
            "HalfNormal(4)",

        "athlete_slope_z":
            "Normal(0,1)",

        "sigma_group":
            "HalfNormal(5)",

        "nu_minus_two":
            "Exponential(0.1)",

        "nu":
            "nu_minus_two + 2",
    },

    "inference": {

        "method":
            "ADVI",

        "iterations":
            ADVI_ITERATIONS,

        "posterior_draws":
            POSTERIOR_DRAWS,

        "random_seed":
            RANDOM_SEED,
    },
}


with V4_MODEL_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        V4_MODEL_SPEC,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART V — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "v4_structural_fit"
] = {

    "status":
        (
            "PASSED"
            if CELL_4_PASS
            else "FAILED"
        ),

    "training_rows":
        N_TRAIN_ROWS,

    "training_athletes":
        N_TRAIN_ATHLETES,

    "advi_iterations":
        ADVI_ITERATIONS,

    "posterior_draws":
        N_POSTERIOR_DRAWS,

    "posterior_chains":
        N_POSTERIOR_CHAINS,

    "random_seed":
        RANDOM_SEED,

    "advi_initial_median_loss":
        ADVI_INITIAL_MEDIAN,

    "advi_final_median_loss":
        ADVI_FINAL_MEDIAN,

    "advi_loss_change":
        ADVI_LOSS_CHANGE,

    "advi_tail_nonfinite":
        N_NONFINITE_ADVI_TAIL,

    "posterior_path":
        str(
            V4_POSTERIOR_PATH
        ),

    "posterior_summary_path":
        str(
            V4_POSTERIOR_SUMMARY_PATH
        ),

    "advi_history_path":
        str(
            V4_ADVI_HISTORY_PATH
        ),

    "train_athlete_index_path":
        str(
            V4_TRAIN_ATHLETE_INDEX_PATH
        ),

    "model_spec_path":
        str(
            V4_MODEL_SPEC_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V4_STRUCTURAL_READY"
    if CELL_4_PASS
    else "V4_STRUCTURAL_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART W — NOTEBOOK GLOBALS
# ==================================================================================================

NOTEBOOK_2_CELL_4_PASS = (
    CELL_4_PASS
)


# ==================================================================================================
# PART X — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — STRUCTURAL V4 BAYESIAN FIT")
print("=" * 145)


print()
print("TRAINING POPULATION")
print("-" * 145)

print(
    f"TRAIN rows                       : "
    f"{N_TRAIN_ROWS:,}"
)

print(
    f"TRAIN athletes                   : "
    f"{N_TRAIN_ATHLETES:,}"
)

print(
    f"Spline columns                   : "
    f"{N_SPLINE}"
)


print()
print("ADVI")
print("-" * 145)

print(
    f"Requested iterations             : "
    f"{ADVI_ITERATIONS:,}"
)

print(
    f"Completed iterations             : "
    f"{N_ADVI_STEPS:,}"
)

print(
    f"Initial median loss              : "
    f"{ADVI_INITIAL_MEDIAN:,.4f}"
)

print(
    f"Final median loss                : "
    f"{ADVI_FINAL_MEDIAN:,.4f}"
)

print(
    f"Loss change                      : "
    f"{ADVI_LOSS_CHANGE:,.4f}"
)

print(
    f"Nonfinite values in final 1,000  : "
    f"{N_NONFINITE_ADVI_TAIL:,}"
)


print()
print("POSTERIOR")
print("-" * 145)

print(
    f"Chains                           : "
    f"{N_POSTERIOR_CHAINS:,}"
)

print(
    f"Draws                            : "
    f"{N_POSTERIOR_DRAWS:,}"
)

print(
    f"Athlete dimension                : "
    f"{POSTERIOR_ATHLETE_DIM:,}"
)

print(
    f"Sex dimension                    : "
    f"{POSTERIOR_SEX_DIM:,}"
)

print(
    f"Group dimension                  : "
    f"{POSTERIOR_GROUP_DIM:,}"
)

print(
    f"Spline dimension                 : "
    f"{POSTERIOR_SPLINE_DIM:,}"
)

print(
    f"Missing posterior variables      : "
    f"{len(MISSING_POSTERIOR_VARS):,}"
)

print(
    f"Nonfinite small posterior values : "
    f"{N_SMALL_POSTERIOR_NONFINITE:,}"
)


print()
print("POSTERIOR SUMMARY")
print("-" * 145)

display(
    V4_POSTERIOR_SUMMARY
)


print()
print("QA")
print("-" * 145)

print(
    f"Input QA                         : "
    f"{INPUT_QA_PASS}"
)

print(
    f"ADVI iteration count             : "
    f"{ADVI_LENGTH_PASS}"
)

print(
    f"ADVI finite tail                 : "
    f"{ADVI_TAIL_PASS}"
)

print(
    f"Posterior draw count             : "
    f"{POSTERIOR_DRAW_PASS}"
)

print(
    f"Posterior variable contract      : "
    f"{POSTERIOR_VARIABLE_PASS}"
)

print(
    f"Posterior dimensions             : "
    f"{POSTERIOR_DIMENSION_PASS}"
)

print(
    f"Posterior finite                 : "
    f"{POSTERIOR_FINITE_PASS}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Posterior                        : "
    f"{V4_POSTERIOR_PATH}"
)

print(
    f"Posterior summary                : "
    f"{V4_POSTERIOR_SUMMARY_PATH}"
)

print(
    f"ADVI history                     : "
    f"{V4_ADVI_HISTORY_PATH}"
)

print(
    f"TRAIN athlete index              : "
    f"{V4_TRAIN_ATHLETE_INDEX_PATH}"
)

print(
    f"Model specification              : "
    f"{V4_MODEL_SPEC_PATH}"
)


print()
print("=" * 145)


if CELL_4_PASS:

    print(
        "CELL 4 STATUS: STRUCTURAL V4 FIT PASSED"
    )

else:

    print(
        "CELL 4 STATUS: STRUCTURAL V4 FIT REVIEW REQUIRED"
    )


print("=" * 145)


Output()

Finished [100%]: Average Loss = 4.6649e+05



V6 TRAINING — STRUCTURAL V4 BAYESIAN FIT

TRAINING POPULATION
-------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN rows                       : 170,644
TRAIN athletes                   : 12,945
Spline columns                   : 7

ADVI
-------------------------------------------------------------------------------------------------------------------------------------------------
Requested iterations             : 40,000
Completed iterations             : 40,000
Initial median loss              : 1,079,463.8445
Final median loss                : 466,474.0318
Loss change                      : -612,989.8127
Nonfinite values in final 1,000  : 0

POSTERIOR
-------------------------------------------------------------------------------------------------------------------------------------------------
Chains                           : 1
Draws                            : 2,000
Athlete dimens

,parameter,mean,sd,hdi_3%,hdi_97%
0,alpha_sex[M],49.498,0.015,49.468,49.526
1,alpha_sex[W],40.998,0.010,40.980,41.018
2,sigma_spline[M],3.580,0.840,2.097,5.091
3,sigma_spline[W],2.693,0.730,1.551,4.226
4,spline_start[M],9.069,0.677,7.809,10.319
5,spline_start[W],5.866,0.148,5.606,6.167
6,"beta_spline[M, 0]",9.069,0.677,7.809,10.319
7,"beta_spline[M, 1]",16.108,0.688,14.836,17.389
8,"beta_spline[M, 2]",19.686,0.688,18.386,20.977
9,"beta_spline[M, 3]",21.868,0.690,20.633,23.230



QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Input QA                         : True
ADVI iteration count             : True
ADVI finite tail                 : True
Posterior draw count             : True
Posterior variable contract      : True
Posterior dimensions             : True
Posterior finite                 : True

OUTPUT
-------------------------------------------------------------------------------------------------------------------------------------------------
Posterior                        : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/v4_structural_posterior.nc
Posterior summary                : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/checkpoints/v4_posterior_summary.parquet
ADVI history                     : /var/folders/q5/yf8g5p896_b

## Stage 05 — V4 evaluation and V5 OU estimation

Evaluate frozen V4, construct residual-state inputs, and estimate the continuous-time V5 OU process from TRAIN only.

**Legacy source cells represented:** 5, 6  
**Execution rule:** run only after the preceding stage has completed successfully.


In [6]:
# ==================================================================================================
# REFACTORED STAGE 05 — V4 evaluation and V5 OU estimation
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 5
# Original source SHA256: 07ea8a539203d21d22f7c7e520a33ab68bb9f5accc333473d96ace7e10ed6715
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 15 — STRUCTURAL V4 EVALUATION + V5 RESIDUAL-STATE INPUT
# ==================================================================================================
#
# Purpose
# -------
#
# 1. Convert the fitted V4 posterior into posterior-mean structural predictions.
#
# 2. Evaluate:
#
#       TRAIN
#       VALIDATION
#       TEST
#       POST_TEST
#
# 3. Explicitly distinguish:
#
#       SEEN athlete
#           athlete appeared in TRAIN and therefore has fitted athlete
#           random intercept / slope parameters.
#
#       UNSEEN athlete
#           athlete did not appear in TRAIN.
#           Athlete random-effect expectation is therefore zero.
#
# 4. Construct structural residuals:
#
#       residual_it
#           = observed_it - mu_V4_it
#
# 5. Construct the V5 latent-state input correctly:
#
#       ONE STATE OBSERVATION PER ATHLETE PER DISTINCT DATE
#
#    This is essential.
#
#    Multiple performances on the same day — including performances with
#    different implement groups — belong to the SAME latent athlete state.
#
# 6. Preserve the composition of same-day observations so later V5 fitting
#    still knows whether the daily state observation was based on:
#
#       M_700
#       M_800
#       W_500
#       W_600
#       or a mixture of groups.
#
#
# IMPORTANT CORRECTION
# --------------------
#
# The old implementation grouped the V5 state table by:
#
#       athlete + date + model_group
#
# which could produce multiple state rows on the same athlete-date and hence
# zero-day state transitions.
#
# This corrected version groups by:
#
#       athlete + date
#
# only.
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_4_PASS",

    "V4_IDATA",

    "V4_TRAIN",
    "V6_VALIDATION",
    "V6_TEST",
    "V6_POST_TEST",

    "V4_TRAIN_ATHLETE_INDEX",
    "V4_TRAIN_ATHLETE_IDS",

    "V6_SPLINE_COLUMNS",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through the V4 fit cell first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_4_PASS:

    raise RuntimeError(
        "The structural V4 fit did not pass."
    )


# ==================================================================================================
# PART B — POSTERIOR-MEAN HELPER
# ==================================================================================================

def posterior_mean(
    variable_name,
):

    return (
        V4_IDATA.posterior[
            variable_name
        ]
        .mean(
            dim=[
                "chain",
                "draw",
            ]
        )
        .to_numpy()
    )


# ==================================================================================================
# PART C — EXTRACT POSTERIOR-MEAN PARAMETERS
# ==================================================================================================

V4_ALPHA_MEAN = posterior_mean(
    "alpha_sex"
)


V4_BETA_SPLINE_MEAN = posterior_mean(
    "beta_spline"
)


V4_YOUTH_EFFECT_MEAN = posterior_mean(
    "youth_implement_effect"
)


V4_ATHLETE_INTERCEPT_MEAN = posterior_mean(
    "athlete_intercept"
)


V4_ATHLETE_SLOPE_MEAN = posterior_mean(
    "athlete_slope"
)


V4_SIGMA_GROUP_MEAN = posterior_mean(
    "sigma_group"
)


V4_NU_MEAN = float(
    posterior_mean(
        "nu"
    )
)


V4_SIGMA_ATHLETE_INTERCEPT_MEAN = float(
    posterior_mean(
        "sigma_athlete_intercept"
    )
)


V4_SIGMA_ATHLETE_SLOPE_MEAN = float(
    posterior_mean(
        "sigma_athlete_slope"
    )
)


# ==================================================================================================
# PART D — PARAMETER DIMENSION QA
# ==================================================================================================

PARAMETER_DIMENSIONS = {
    "alpha_sex":
        V4_ALPHA_MEAN.shape,

    "beta_spline":
        V4_BETA_SPLINE_MEAN.shape,

    "youth_implement_effect":
        V4_YOUTH_EFFECT_MEAN.shape,

    "athlete_intercept":
        V4_ATHLETE_INTERCEPT_MEAN.shape,

    "athlete_slope":
        V4_ATHLETE_SLOPE_MEAN.shape,

    "sigma_group":
        V4_SIGMA_GROUP_MEAN.shape,
}


PARAMETER_DIMENSION_PASS = bool(

    V4_ALPHA_MEAN.shape
    ==
    (2,)

    and

    V4_BETA_SPLINE_MEAN.shape
    ==
    (
        2,
        7,
    )

    and

    V4_YOUTH_EFFECT_MEAN.shape
    ==
    (2,)

    and

    V4_ATHLETE_INTERCEPT_MEAN.shape[0]
    ==
    len(
        V4_TRAIN_ATHLETE_IDS
    )

    and

    V4_ATHLETE_SLOPE_MEAN.shape[0]
    ==
    len(
        V4_TRAIN_ATHLETE_IDS
    )

    and

    V4_SIGMA_GROUP_MEAN.shape
    ==
    (4,)
)


if not PARAMETER_DIMENSION_PASS:

    raise RuntimeError(
        "Unexpected V4 posterior-mean parameter dimensions:\n"
        f"{PARAMETER_DIMENSIONS}"
    )


# ==================================================================================================
# PART E — TRAIN ATHLETE LOOKUP
# ==================================================================================================

TRAIN_ATHLETE_LOOKUP = {
    int(
        athlete_id
    ):
        int(
            athlete_idx
        )

    for athlete_id, athlete_idx
    in V4_TRAIN_ATHLETE_INDEX.items()
}


# ==================================================================================================
# PART F — STRUCTURAL PREDICTION FUNCTION
# ==================================================================================================

def make_v4_structural_predictions(
    df,
    split_name,
):

    out = (
        df.copy()
        .reset_index(
            drop=True
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Input arrays
    # ----------------------------------------------------------------------------------------------

    sex_idx = (
        out[
            "sex_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )


    group_idx = (
        out[
            "group_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )


    youth = (
        out[
            "is_youth_implement"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    age_scaled = (
        out[
            "age_scaled"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    spline_matrix = (
        out[
            V6_SPLINE_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    observed = (
        out[
            "mark_numeric_m"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    athlete_ids = (
        out[
            "aaAthleteId"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Seen / unseen athlete classification
    # ----------------------------------------------------------------------------------------------

    train_athlete_idx = np.array(
        [
            TRAIN_ATHLETE_LOOKUP.get(
                int(
                    athlete_id
                ),
                -1,
            )

            for athlete_id
            in athlete_ids
        ],
        dtype=np.int32,
    )


    seen = (
        train_athlete_idx
        >=
        0
    )


    # ----------------------------------------------------------------------------------------------
    # Population structural components
    # ----------------------------------------------------------------------------------------------

    intercept_component = (
        V4_ALPHA_MEAN[
            sex_idx
        ]
    )


    spline_component = np.sum(
        (
            spline_matrix
            *
            V4_BETA_SPLINE_MEAN[
                sex_idx
            ]
        ),
        axis=1,
    )


    youth_component = (
        V4_YOUTH_EFFECT_MEAN[
            sex_idx
        ]
        *
        youth
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random effects
    # ----------------------------------------------------------------------------------------------

    athlete_intercept_component = np.zeros(
        len(
            out
        ),
        dtype=np.float64,
    )


    athlete_slope_coefficient = np.zeros(
        len(
            out
        ),
        dtype=np.float64,
    )


    if seen.any():

        seen_indices = (
            train_athlete_idx[
                seen
            ]
        )


        athlete_intercept_component[
            seen
        ] = (
            V4_ATHLETE_INTERCEPT_MEAN[
                seen_indices
            ]
        )


        athlete_slope_coefficient[
            seen
        ] = (
            V4_ATHLETE_SLOPE_MEAN[
                seen_indices
            ]
        )


    athlete_slope_component = (
        athlete_slope_coefficient
        *
        age_scaled
    )


    # ----------------------------------------------------------------------------------------------
    # Structural posterior-mean prediction
    # ----------------------------------------------------------------------------------------------

    mu_v4 = (
        intercept_component

        +
        spline_component

        +
        youth_component

        +
        athlete_intercept_component

        +
        athlete_slope_component
    )


    # ----------------------------------------------------------------------------------------------
    # Residual
    # ----------------------------------------------------------------------------------------------

    structural_residual = (
        observed
        -
        mu_v4
    )


    # ----------------------------------------------------------------------------------------------
    # Output fields
    # ----------------------------------------------------------------------------------------------

    out[
        "prediction_split"
    ] = split_name


    out[
        "athlete_seen_in_training"
    ] = seen


    out[
        "train_athlete_idx"
    ] = train_athlete_idx


    out[
        "v4_population_intercept"
    ] = intercept_component


    out[
        "v4_spline_component"
    ] = spline_component


    out[
        "v4_youth_component"
    ] = youth_component


    out[
        "v4_athlete_intercept_component"
    ] = athlete_intercept_component


    out[
        "v4_athlete_slope_coefficient"
    ] = athlete_slope_coefficient


    out[
        "v4_athlete_slope_component"
    ] = athlete_slope_component


    out[
        "v4_structural_mean"
    ] = mu_v4


    out[
        "v4_structural_residual"
    ] = structural_residual


    out[
        "v4_abs_error"
    ] = np.abs(
        structural_residual
    )


    out[
        "v4_squared_error"
    ] = (
        structural_residual
        **
        2
    )


    out[
        "v4_sigma_group"
    ] = (
        V4_SIGMA_GROUP_MEAN[
            group_idx
        ]
    )


    out[
        "v4_nu"
    ] = V4_NU_MEAN


    return out


# ==================================================================================================
# PART G — GENERATE STRUCTURAL PREDICTIONS
# ==================================================================================================

V4_TRAIN_PRED = (
    make_v4_structural_predictions(
        V4_TRAIN,
        "TRAIN",
    )
)


V4_VALIDATION_PRED = (
    make_v4_structural_predictions(
        V6_VALIDATION,
        "VALIDATION",
    )
)


V4_TEST_PRED = (
    make_v4_structural_predictions(
        V6_TEST,
        "TEST",
    )
)


V4_POST_TEST_PRED = (
    make_v4_structural_predictions(
        V6_POST_TEST,
        "POST_TEST",
    )
)


V4_ALL_PREDICTIONS = pd.concat(
    [
        V4_TRAIN_PRED,
        V4_VALIDATION_PRED,
        V4_TEST_PRED,
        V4_POST_TEST_PRED,
    ],
    ignore_index=True,
)


# ==================================================================================================
# PART H — BASIC PREDICTION QA
# ==================================================================================================

PREDICTION_COLUMNS_TO_CHECK = [
    "v4_structural_mean",
    "v4_structural_residual",
    "v4_sigma_group",
]


N_NONFINITE_PREDICTIONS = int(
    (
        ~np.isfinite(
            V4_ALL_PREDICTIONS[
                PREDICTION_COLUMNS_TO_CHECK
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


TRAIN_ALL_SEEN_PASS = bool(
    V4_TRAIN_PRED[
        "athlete_seen_in_training"
    ]
    .all()
)


ROW_COUNT_PASS = bool(

    len(
        V4_TRAIN_PRED
    )
    ==
    len(
        V4_TRAIN
    )

    and

    len(
        V4_VALIDATION_PRED
    )
    ==
    len(
        V6_VALIDATION
    )

    and

    len(
        V4_TEST_PRED
    )
    ==
    len(
        V6_TEST
    )

    and

    len(
        V4_POST_TEST_PRED
    )
    ==
    len(
        V6_POST_TEST
    )
)


# ==================================================================================================
# PART I — SEEN / UNSEEN COUNTS
# ==================================================================================================

N_VALIDATION_SEEN_ROWS = int(
    V4_VALIDATION_PRED[
        "athlete_seen_in_training"
    ]
    .sum()
)


N_VALIDATION_UNSEEN_ROWS = int(
    (
        ~V4_VALIDATION_PRED[
            "athlete_seen_in_training"
        ]
    )
    .sum()
)


N_TEST_SEEN_ROWS = int(
    V4_TEST_PRED[
        "athlete_seen_in_training"
    ]
    .sum()
)


N_TEST_UNSEEN_ROWS = int(
    (
        ~V4_TEST_PRED[
            "athlete_seen_in_training"
        ]
    )
    .sum()
)


N_POST_TEST_SEEN_ROWS = int(
    V4_POST_TEST_PRED[
        "athlete_seen_in_training"
    ]
    .sum()
)


N_POST_TEST_UNSEEN_ROWS = int(
    (
        ~V4_POST_TEST_PRED[
            "athlete_seen_in_training"
        ]
    )
    .sum()
)


# ==================================================================================================
# PART J — METRIC HELPER
# ==================================================================================================

def calculate_metrics(
    df,
):

    residual = (
        df[
            "v4_structural_residual"
        ]
        .to_numpy(
            dtype=float
        )
    )


    if len(
        residual
    ) == 0:

        return pd.Series(
            {
                "n":
                    0,

                "athletes":
                    0,

                "MAE":
                    np.nan,

                "RMSE":
                    np.nan,

                "Bias":
                    np.nan,
            }
        )


    return pd.Series(
        {
            "n":
                int(
                    len(
                        residual
                    )
                ),

            "athletes":
                int(
                    df[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "MAE":
                float(
                    np.mean(
                        np.abs(
                            residual
                        )
                    )
                ),

            "RMSE":
                float(
                    np.sqrt(
                        np.mean(
                            residual
                            **
                            2
                        )
                    )
                ),

            "Bias":
                float(
                    np.mean(
                        residual
                    )
                ),
        }
    )


# ==================================================================================================
# PART K — OVERALL METRICS
# ==================================================================================================

V4_METRICS_BY_SPLIT = (
    V4_ALL_PREDICTIONS.groupby(
        "prediction_split",
        sort=False,
    )
    .apply(
        calculate_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART L — SEEN VS UNSEEN METRICS
# ==================================================================================================

V4_ALL_PREDICTIONS[
    "athlete_training_status"
] = np.where(
    V4_ALL_PREDICTIONS[
        "athlete_seen_in_training"
    ],
    "SEEN",
    "UNSEEN",
)


V4_METRICS_SEEN_UNSEEN = (
    V4_ALL_PREDICTIONS.groupby(
        [
            "prediction_split",
            "athlete_training_status",
        ],
        sort=False,
    )
    .apply(
        calculate_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART M — METRICS BY MODEL GROUP
# ==================================================================================================

V4_METRICS_BY_GROUP = (
    V4_ALL_PREDICTIONS.groupby(
        [
            "prediction_split",
            "model_group",
        ],
        sort=False,
    )
    .apply(
        calculate_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART N — GROUP × SEEN STATUS METRICS
# ==================================================================================================

V4_METRICS_GROUP_SEEN = (
    V4_ALL_PREDICTIONS.groupby(
        [
            "prediction_split",
            "model_group",
            "athlete_training_status",
        ],
        sort=False,
    )
    .apply(
        calculate_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART O — ATHLETE-LEVEL RESIDUAL SUMMARY
# ==================================================================================================

V4_ATHLETE_RESIDUAL_SUMMARY = (
    V4_ALL_PREDICTIONS.groupby(
        [
            "prediction_split",
            "aaAthleteId",
            "athlete_training_status",
        ],
        as_index=False,
    )
    .agg(
        n_results=(
            "mark_numeric_m",
            "size",
        ),

        mean_residual=(
            "v4_structural_residual",
            "mean",
        ),

        median_residual=(
            "v4_structural_residual",
            "median",
        ),

        mae=(
            "v4_abs_error",
            "mean",
        ),

        first_date=(
            "result_date_dt",
            "min",
        ),

        last_date=(
            "result_date_dt",
            "max",
        ),
    )
)


# ==================================================================================================
# PART P — TRAIN RESIDUAL SERIES FOR V5
# ==================================================================================================

V5_TRAIN_RESIDUALS = (
    V4_TRAIN_PRED[
        [
            "aaAthleteId",
            "result_date_dt",
            "mark_numeric_m",

            "model_group",
            "group_idx",

            "sex_group",
            "sex_idx",

            "age_years",
            "age_scaled",

            "athlete_seen_in_training",

            "v4_structural_mean",
            "v4_structural_residual",
            "v4_sigma_group",
            "v4_nu",

            "n_results_on_date",
            "prior_distinct_dates",
            "prior_result_count",
            "history_span_days",
            "days_since_previous_date",
        ]
    ]
    .copy()
)


# ==================================================================================================
# PART Q — BUILD ONE V5 STATE OBSERVATION PER ATHLETE-DATE
# ==================================================================================================
#
# The V5 latent state belongs to the ATHLETE, not to the implement group.
#
# Therefore:
#
#       one athlete + one date = one state observation
#
# ==================================================================================================

V5_DATE_SOURCE = (
    V5_TRAIN_RESIDUALS.copy()
)


STATE_GROUP_ORDER = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


# --------------------------------------------------------------------------------------------------
# Q1 — GROUP INDICATORS
# --------------------------------------------------------------------------------------------------

for group in STATE_GROUP_ORDER:

    safe_group = (
        group.replace(
            "_",
            ""
        )
    )


    V5_DATE_SOURCE[
        f"_count_{safe_group}"
    ] = (
        V5_DATE_SOURCE[
            "model_group"
        ]
        .eq(
            group
        )
        .astype(
            "int16"
        )
    )


# --------------------------------------------------------------------------------------------------
# Q2 — ROW-LEVEL V4 MEASUREMENT VARIANCE
# --------------------------------------------------------------------------------------------------

V5_DATE_SOURCE[
    "_v4_sigma_squared"
] = (
    V5_DATE_SOURCE[
        "v4_sigma_group"
    ]
    .astype(float)
    **
    2
)


# --------------------------------------------------------------------------------------------------
# Q3 — AGGREGATE TO ATHLETE-DATE
# --------------------------------------------------------------------------------------------------

V5_TRAIN_DATE_RESIDUALS = (
    V5_DATE_SOURCE.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        n_results_on_date=(
            "mark_numeric_m",
            "size",
        ),

        observed_mean_mark=(
            "mark_numeric_m",
            "mean",
        ),

        structural_mean=(
            "v4_structural_mean",
            "mean",
        ),

        residual_mean=(
            "v4_structural_residual",
            "mean",
        ),

        residual_median=(
            "v4_structural_residual",
            "median",
        ),

        residual_min=(
            "v4_structural_residual",
            "min",
        ),

        residual_max=(
            "v4_structural_residual",
            "max",
        ),

        sex_group=(
            "sex_group",
            "first",
        ),

        sex_idx=(
            "sex_idx",
            "first",
        ),

        age_years_mean=(
            "age_years",
            "mean",
        ),

        v4_sigma_squared_sum=(
            "_v4_sigma_squared",
            "sum",
        ),

        n_M700=(
            "_count_M700",
            "sum",
        ),

        n_M800=(
            "_count_M800",
            "sum",
        ),

        n_W500=(
            "_count_W500",
            "sum",
        ),

        n_W600=(
            "_count_W600",
            "sum",
        ),
    )
)


# ==================================================================================================
# PART R — EFFECTIVE MEASUREMENT VARIANCE OF DAILY MEAN
# ==================================================================================================
#
# If:
#
#       ybar = (1/n) sum_j y_j
#
# and row-level measurement errors are independent,
#
# then:
#
#       Var(ybar)
#           = sum_j sigma_j^2 / n^2
#
# ==================================================================================================

V5_TRAIN_DATE_RESIDUALS[
    "v4_effective_measurement_variance"
] = (
    V5_TRAIN_DATE_RESIDUALS[
        "v4_sigma_squared_sum"
    ]
    /
    (
        V5_TRAIN_DATE_RESIDUALS[
            "n_results_on_date"
        ]
        .astype(float)
        **
        2
    )
)


V5_TRAIN_DATE_RESIDUALS[
    "v4_effective_measurement_sigma"
] = np.sqrt(
    V5_TRAIN_DATE_RESIDUALS[
        "v4_effective_measurement_variance"
    ]
)


# ==================================================================================================
# PART S — SAME-DAY IMPLEMENT COMPOSITION
# ==================================================================================================

STATE_GROUP_COUNT_COLUMNS = [
    "n_M700",
    "n_M800",
    "n_W500",
    "n_W600",
]


V5_TRAIN_DATE_RESIDUALS[
    "n_model_groups_on_date"
] = (
    (
        V5_TRAIN_DATE_RESIDUALS[
            STATE_GROUP_COUNT_COLUMNS
        ]
        >
        0
    )
    .sum(
        axis=1
    )
    .astype(
        "int8"
    )
)


def identify_state_group(
    row,
):

    groups = []


    if row[
        "n_M700"
    ] > 0:

        groups.append(
            "M_700"
        )


    if row[
        "n_M800"
    ] > 0:

        groups.append(
            "M_800"
        )


    if row[
        "n_W500"
    ] > 0:

        groups.append(
            "W_500"
        )


    if row[
        "n_W600"
    ] > 0:

        groups.append(
            "W_600"
        )


    if len(
        groups
    ) == 1:

        return groups[
            0
        ]


    if len(
        groups
    ) > 1:

        return "MIXED"


    return "UNKNOWN"


V5_TRAIN_DATE_RESIDUALS[
    "state_observation_group"
] = (
    V5_TRAIN_DATE_RESIDUALS.apply(
        identify_state_group,
        axis=1,
    )
)


# ==================================================================================================
# PART T — SORT STATE OBSERVATIONS
# ==================================================================================================

V5_TRAIN_DATE_RESIDUALS = (
    V5_TRAIN_DATE_RESIDUALS.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART U — STATE CHRONOLOGY
# ==================================================================================================

state_group = (
    V5_TRAIN_DATE_RESIDUALS.groupby(
        "aaAthleteId",
        sort=False,
        group_keys=False,
    )
)


V5_TRAIN_DATE_RESIDUALS[
    "previous_state_date"
] = (
    state_group[
        "result_date_dt"
    ]
    .shift(
        1
    )
)


V5_TRAIN_DATE_RESIDUALS[
    "state_gap_days"
] = (
    V5_TRAIN_DATE_RESIDUALS[
        "result_date_dt"
    ]
    -
    V5_TRAIN_DATE_RESIDUALS[
        "previous_state_date"
    ]
).dt.days


V5_TRAIN_DATE_RESIDUALS[
    "state_date_index"
] = (
    state_group
    .cumcount()
    .astype(
        "int32"
    )
)


# ==================================================================================================
# PART V — STATE CHRONOLOGY QA
# ==================================================================================================

N_DUPLICATE_ATHLETE_DATES = int(
    V5_TRAIN_DATE_RESIDUALS.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ],
        keep=False,
    )
    .sum()
)


state_gap_mask = (
    V5_TRAIN_DATE_RESIDUALS[
        "state_gap_days"
    ]
    .notna()
)


N_NONPOSITIVE_STATE_GAPS = int(
    (
        V5_TRAIN_DATE_RESIDUALS.loc[
            state_gap_mask,
            "state_gap_days",
        ]
        <=
        0
    )
    .sum()
)


N_DATE_RESIDUAL_NONFINITE = int(
    (
        ~np.isfinite(
            V5_TRAIN_DATE_RESIDUALS[
                "residual_mean"
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


N_EFFECTIVE_VARIANCE_NONFINITE = int(
    (
        ~np.isfinite(
            V5_TRAIN_DATE_RESIDUALS[
                "v4_effective_measurement_variance"
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


N_NONPOSITIVE_EFFECTIVE_VARIANCE = int(
    (
        V5_TRAIN_DATE_RESIDUALS[
            "v4_effective_measurement_variance"
        ]
        <=
        0
    )
    .sum()
)


DATE_AGGREGATION_ROW_PASS = bool(
    int(
        V5_TRAIN_DATE_RESIDUALS[
            "n_results_on_date"
        ]
        .sum()
    )
    ==
    len(
        V5_TRAIN_RESIDUALS
    )
)


# ==================================================================================================
# PART W — SEX CONSISTENCY QA
# ==================================================================================================

sex_variants = (
    V5_DATE_SOURCE.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )[
        "sex_group"
    ]
    .nunique(
        dropna=False
    )
)


N_STATE_DATE_SEX_CONFLICTS = int(
    (
        sex_variants
        >
        1
    )
    .sum()
)


# ==================================================================================================
# PART X — SAME-DAY SUMMARY
# ==================================================================================================

MIXED_GROUP_STATE_DATES = (
    V5_TRAIN_DATE_RESIDUALS.loc[
        V5_TRAIN_DATE_RESIDUALS[
            "n_model_groups_on_date"
        ]
        >
        1
    ]
    .copy()
)


N_MIXED_GROUP_STATE_DATES = int(
    len(
        MIXED_GROUP_STATE_DATES
    )
)


N_MULTI_RESULT_STATE_DATES = int(
    (
        V5_TRAIN_DATE_RESIDUALS[
            "n_results_on_date"
        ]
        >
        1
    )
    .sum()
)


STATE_CHRONOLOGY_PASS = bool(

    N_DUPLICATE_ATHLETE_DATES == 0

    and

    N_NONPOSITIVE_STATE_GAPS == 0

    and

    N_DATE_RESIDUAL_NONFINITE == 0

    and

    N_EFFECTIVE_VARIANCE_NONFINITE == 0

    and

    N_NONPOSITIVE_EFFECTIVE_VARIANCE == 0

    and

    DATE_AGGREGATION_ROW_PASS

    and

    N_STATE_DATE_SEX_CONFLICTS == 0
)


# ==================================================================================================
# PART Y — V4 LIKELIHOOD PARAMETER SUMMARY
# ==================================================================================================

V4_LIKELIHOOD_PARAMETERS = pd.DataFrame(
    {
        "model_group":
            [
                "M_700",
                "M_800",
                "W_500",
                "W_600",
            ],

        "sigma_group_mean":
            V4_SIGMA_GROUP_MEAN,
    }
)


# ==================================================================================================
# PART Z — FINAL CELL QA
# ==================================================================================================

CELL_15_PASS = bool(

    PARAMETER_DIMENSION_PASS

    and

    ROW_COUNT_PASS

    and

    TRAIN_ALL_SEEN_PASS

    and

    N_NONFINITE_PREDICTIONS == 0

    and

    STATE_CHRONOLOGY_PASS
)


# ==================================================================================================
# PART AA — SAVE ARTIFACTS
# ==================================================================================================

V4_ALL_PREDICTIONS_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_structural_predictions_all.parquet"
)


V4_VALIDATION_PRED_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_validation_predictions.parquet"
)


V4_TEST_PRED_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_test_predictions.parquet"
)


V4_POST_TEST_PRED_PATH = (
    TRAINING_RUN_DIR
    /
    "v4_post_test_predictions.parquet"
)


V5_TRAIN_RESIDUALS_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_train_structural_residuals.parquet"
)


V5_TRAIN_DATE_RESIDUALS_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_train_date_residuals.parquet"
)


V4_METRICS_SPLIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_metrics_by_split.parquet"
)


V4_METRICS_SEEN_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_metrics_seen_unseen.parquet"
)


V4_METRICS_GROUP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_metrics_by_group.parquet"
)


V4_METRICS_GROUP_SEEN_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_metrics_group_seen.parquet"
)


V4_ATHLETE_RESIDUAL_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_athlete_residual_summary.parquet"
)


V4_LIKELIHOOD_PARAMETERS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v4_likelihood_parameters.parquet"
)


MIXED_STATE_DATES_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_mixed_group_state_dates.parquet"
)


V4_ALL_PREDICTIONS.to_parquet(
    V4_ALL_PREDICTIONS_PATH,
    index=False,
)


V4_VALIDATION_PRED.to_parquet(
    V4_VALIDATION_PRED_PATH,
    index=False,
)


V4_TEST_PRED.to_parquet(
    V4_TEST_PRED_PATH,
    index=False,
)


V4_POST_TEST_PRED.to_parquet(
    V4_POST_TEST_PRED_PATH,
    index=False,
)


V5_TRAIN_RESIDUALS.to_parquet(
    V5_TRAIN_RESIDUALS_PATH,
    index=False,
)


V5_TRAIN_DATE_RESIDUALS.to_parquet(
    V5_TRAIN_DATE_RESIDUALS_PATH,
    index=False,
)


V4_METRICS_BY_SPLIT.to_parquet(
    V4_METRICS_SPLIT_PATH,
    index=False,
)


V4_METRICS_SEEN_UNSEEN.to_parquet(
    V4_METRICS_SEEN_PATH,
    index=False,
)


V4_METRICS_BY_GROUP.to_parquet(
    V4_METRICS_GROUP_PATH,
    index=False,
)


V4_METRICS_GROUP_SEEN.to_parquet(
    V4_METRICS_GROUP_SEEN_PATH,
    index=False,
)


V4_ATHLETE_RESIDUAL_SUMMARY.to_parquet(
    V4_ATHLETE_RESIDUAL_SUMMARY_PATH,
    index=False,
)


V4_LIKELIHOOD_PARAMETERS.to_parquet(
    V4_LIKELIHOOD_PARAMETERS_PATH,
    index=False,
)


MIXED_GROUP_STATE_DATES.to_parquet(
    MIXED_STATE_DATES_PATH,
    index=False,
)


# ==================================================================================================
# PART AB — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "v4_structural_evaluation"
] = {

    "status":
        (
            "PASSED"
            if CELL_15_PASS
            else "FAILED"
        ),

    "posterior_mean_nu":
        V4_NU_MEAN,

    "posterior_mean_sigma_athlete_intercept":
        V4_SIGMA_ATHLETE_INTERCEPT_MEAN,

    "posterior_mean_sigma_athlete_slope":
        V4_SIGMA_ATHLETE_SLOPE_MEAN,

    "validation": {

        "rows":
            int(
                len(
                    V4_VALIDATION_PRED
                )
            ),

        "seen_rows":
            N_VALIDATION_SEEN_ROWS,

        "unseen_rows":
            N_VALIDATION_UNSEEN_ROWS,
    },

    "test": {

        "rows":
            int(
                len(
                    V4_TEST_PRED
                )
            ),

        "seen_rows":
            N_TEST_SEEN_ROWS,

        "unseen_rows":
            N_TEST_UNSEEN_ROWS,
    },

    "post_test": {

        "rows":
            int(
                len(
                    V4_POST_TEST_PRED
                )
            ),

        "seen_rows":
            N_POST_TEST_SEEN_ROWS,

        "unseen_rows":
            N_POST_TEST_UNSEEN_ROWS,
    },

    "v5_train_performance_rows":
        int(
            len(
                V5_TRAIN_RESIDUALS
            )
        ),

    "v5_train_state_rows":
        int(
            len(
                V5_TRAIN_DATE_RESIDUALS
            )
        ),

    "multi_result_state_dates":
        N_MULTI_RESULT_STATE_DATES,

    "mixed_group_state_dates":
        N_MIXED_GROUP_STATE_DATES,

    "duplicate_athlete_dates":
        N_DUPLICATE_ATHLETE_DATES,

    "nonpositive_state_gaps":
        N_NONPOSITIVE_STATE_GAPS,

    "state_chronology_pass":
        STATE_CHRONOLOGY_PASS,

    "predictions_path":
        str(
            V4_ALL_PREDICTIONS_PATH
        ),

    "v5_train_residuals_path":
        str(
            V5_TRAIN_RESIDUALS_PATH
        ),

    "v5_train_state_path":
        str(
            V5_TRAIN_DATE_RESIDUALS_PATH
        ),

    "mixed_state_dates_path":
        str(
            MIXED_STATE_DATES_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V4_EVALUATED_V5_STATE_READY"
    if CELL_15_PASS
    else "V4_EVALUATION_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART AC — NOTEBOOK GLOBALS
# ==================================================================================================

NOTEBOOK_2_CELL_15_PASS = (
    CELL_15_PASS
)


# Compatibility alias in case subsequent code expects the earlier numbering.
NOTEBOOK_2_CELL_5_PASS = (
    CELL_15_PASS
)


# ==================================================================================================
# PART AD — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — STRUCTURAL V4 EVALUATION + V5 STATE INPUT")
print("=" * 145)


print()
print("POSTERIOR-MEAN STRUCTURAL PARAMETERS")
print("-" * 145)

print(
    f"nu                               : "
    f"{V4_NU_MEAN:.6f}"
)

print(
    f"sigma athlete intercept          : "
    f"{V4_SIGMA_ATHLETE_INTERCEPT_MEAN:.6f}"
)

print(
    f"sigma athlete slope              : "
    f"{V4_SIGMA_ATHLETE_SLOPE_MEAN:.6f}"
)

print()

display(
    V4_LIKELIHOOD_PARAMETERS
)


print()
print("OVERALL STRUCTURAL V4 METRICS")
print("-" * 145)

display(
    V4_METRICS_BY_SPLIT
)


print()
print("SEEN VS UNSEEN ATHLETES")
print("-" * 145)

display(
    V4_METRICS_SEEN_UNSEEN
)


print()
print("METRICS BY MODEL GROUP")
print("-" * 145)

display(
    V4_METRICS_BY_GROUP
)


print()
print("GROUP × SEEN STATUS")
print("-" * 145)

display(
    V4_METRICS_GROUP_SEEN
)


print()
print("V5 STATE INPUT")
print("-" * 145)

print(
    f"TRAIN performance residual rows  : "
    f"{len(V5_TRAIN_RESIDUALS):,}"
)

print(
    f"Distinct athlete-date states     : "
    f"{len(V5_TRAIN_DATE_RESIDUALS):,}"
)

print(
    f"TRAIN residual athletes          : "
    f"{V5_TRAIN_DATE_RESIDUALS['aaAthleteId'].nunique():,}"
)

print(
    f"Dates with >1 performance        : "
    f"{N_MULTI_RESULT_STATE_DATES:,}"
)

print(
    f"Dates with >1 model group        : "
    f"{N_MIXED_GROUP_STATE_DATES:,}"
)


if N_MIXED_GROUP_STATE_DATES > 0:

    print()
    print("SAMPLE — MIXED IMPLEMENT STATE DATES")
    print("-" * 145)

    display(
        MIXED_GROUP_STATE_DATES[
            [
                "aaAthleteId",
                "result_date_dt",
                "n_results_on_date",
                "n_M700",
                "n_M800",
                "n_W500",
                "n_W600",
                "residual_mean",
                "v4_effective_measurement_variance",
            ]
        ]
        .head(
            30
        )
    )


print()
print("SEEN / UNSEEN ROW COUNTS")
print("-" * 145)

print(
    f"VALIDATION seen                  : "
    f"{N_VALIDATION_SEEN_ROWS:,}"
)

print(
    f"VALIDATION unseen                : "
    f"{N_VALIDATION_UNSEEN_ROWS:,}"
)

print(
    f"TEST seen                        : "
    f"{N_TEST_SEEN_ROWS:,}"
)

print(
    f"TEST unseen                      : "
    f"{N_TEST_UNSEEN_ROWS:,}"
)

print(
    f"POST_TEST seen                   : "
    f"{N_POST_TEST_SEEN_ROWS:,}"
)

print(
    f"POST_TEST unseen                 : "
    f"{N_POST_TEST_UNSEEN_ROWS:,}"
)


print()
print("STATE CHRONOLOGY QA")
print("-" * 145)

print(
    f"Duplicate athlete-date states    : "
    f"{N_DUPLICATE_ATHLETE_DATES:,}"
)

print(
    f"Nonpositive state gaps           : "
    f"{N_NONPOSITIVE_STATE_GAPS:,}"
)

print(
    f"Nonfinite residual means         : "
    f"{N_DATE_RESIDUAL_NONFINITE:,}"
)

print(
    f"Nonfinite effective variance     : "
    f"{N_EFFECTIVE_VARIANCE_NONFINITE:,}"
)

print(
    f"Nonpositive effective variance   : "
    f"{N_NONPOSITIVE_EFFECTIVE_VARIANCE:,}"
)

print(
    f"Date aggregation row arithmetic  : "
    f"{DATE_AGGREGATION_ROW_PASS}"
)

print(
    f"Sex conflicts on state date      : "
    f"{N_STATE_DATE_SEX_CONFLICTS:,}"
)

print(
    f"State chronology pass            : "
    f"{STATE_CHRONOLOGY_PASS}"
)


print()
print("OVERALL QA")
print("-" * 145)

print(
    f"Posterior parameter dimensions   : "
    f"{PARAMETER_DIMENSION_PASS}"
)

print(
    f"Prediction row counts            : "
    f"{ROW_COUNT_PASS}"
)

print(
    f"All TRAIN rows seen              : "
    f"{TRAIN_ALL_SEEN_PASS}"
)

print(
    f"Nonfinite prediction values      : "
    f"{N_NONFINITE_PREDICTIONS:,}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"All V4 predictions               : "
    f"{V4_ALL_PREDICTIONS_PATH}"
)

print(
    f"V5 TRAIN row residuals           : "
    f"{V5_TRAIN_RESIDUALS_PATH}"
)

print(
    f"V5 TRAIN state observations      : "
    f"{V5_TRAIN_DATE_RESIDUALS_PATH}"
)

print(
    f"Mixed-group state audit          : "
    f"{MIXED_STATE_DATES_PATH}"
)

print(
    f"Seen/unseen metrics              : "
    f"{V4_METRICS_SEEN_PATH}"
)


print()
print("=" * 145)


if CELL_15_PASS:

    print(
        "CELL 15 STATUS: STRUCTURAL V4 EVALUATION + V5 STATE INPUT PASSED"
    )

else:

    print(
        "CELL 15 STATUS: REVIEW REQUIRED"
    )


print("=" * 145)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 6
# Original source SHA256: d38c60b1d4b505b33f85269a52b56179b6b0a5d4526281d737c6f6ca42760d4a
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 16 — ESTIMATE V5 CONTINUOUS-TIME OU STATE MODEL
# ==================================================================================================
#
# State model
# -----------
#
# Structural residual:
#
#       y_it = z_it + epsilon_it
#
#
# Latent transient athlete state:
#
#       z_t | z_prev
#           ~ Normal(
#                 phi(dt) * z_prev,
#                 V_state * (1 - phi(dt)^2)
#             )
#
#
# where:
#
#       phi(dt) = exp(-dt / tau)
#
#
# and:
#
#       tau       = persistence time constant in days
#       V_state   = stationary latent-state variance
#
#
# Observation model
# -----------------
#
#       epsilon_it ~ Normal(0, R_it)
#
#
# with group-specific row-level measurement variances:
#
#       R_M700
#       R_M800
#       R_W500
#       R_W600
#
#
# For a same-day state observation containing n performances:
#
#       ybar = mean(residual_j)
#
# and:
#
#       R_date
#           = sum_g n_g * R_g / n^2
#
#
# This correctly handles:
#
#       • multiple same-day results
#       • youth + senior implement on the same date
#       • irregular gaps between competition dates
#
#
# Estimation
# ----------
#
# Exact Gaussian state-space likelihood evaluated by Kalman filtering.
#
# Parameters are optimized in log-space to guarantee positivity.
#
# Multiple deterministic starting points are used to check optimizer robustness.
#
#
# IMPORTANT
# ---------
#
# TRAIN ONLY.
#
# Validation, test and post-test observations do not enter parameter estimation.
#
# ==================================================================================================

import json
import math

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_15_PASS",
    "V5_TRAIN_DATE_RESIDUALS",
    "V4_SIGMA_GROUP_MEAN",
    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through Cell 15 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_15_PASS:

    raise RuntimeError(
        "Cell 15 did not pass."
    )


# ==================================================================================================
# PART B — STATE DATA
# ==================================================================================================

STATE_DATA = (
    V5_TRAIN_DATE_RESIDUALS.copy()
)


STATE_DATA[
    "result_date_dt"
] = pd.to_datetime(
    STATE_DATA[
        "result_date_dt"
    ],
    errors="coerce",
)


STATE_DATA = (
    STATE_DATA.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


N_STATE_ROWS = int(
    len(
        STATE_DATA
    )
)


N_STATE_ATHLETES = int(
    STATE_DATA[
        "aaAthleteId"
    ]
    .nunique()
)


# ==================================================================================================
# PART C — NUMPY ARRAYS
# ==================================================================================================

Y_STATE = (
    STATE_DATA[
        "residual_mean"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


GAP_DAYS = (
    STATE_DATA[
        "state_gap_days"
    ]
    .fillna(
        0.0
    )
    .to_numpy(
        dtype=np.float64
    )
)


STATE_INDEX = (
    STATE_DATA[
        "state_date_index"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


IS_FIRST_STATE = (
    STATE_INDEX
    ==
    0
)


N_RESULTS_DATE = (
    STATE_DATA[
        "n_results_on_date"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


GROUP_COUNTS = (
    STATE_DATA[
        [
            "n_M700",
            "n_M800",
            "n_W500",
            "n_W600",
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


# ==================================================================================================
# PART D — INPUT QA
# ==================================================================================================

INPUT_QA = {

    "state_residuals_finite":
        bool(
            np.isfinite(
                Y_STATE
            )
            .all()
        ),

    "state_gaps_finite":
        bool(
            np.isfinite(
                GAP_DAYS
            )
            .all()
        ),

    "strict_positive_nonfirst_gaps":
        bool(
            (
                GAP_DAYS[
                    ~IS_FIRST_STATE
                ]
                >
                0
            )
            .all()
        ),

    "first_state_gap_zero":
        bool(
            (
                GAP_DAYS[
                    IS_FIRST_STATE
                ]
                ==
                0
            )
            .all()
        ),

    "positive_result_counts":
        bool(
            (
                N_RESULTS_DATE
                >
                0
            )
            .all()
        ),

    "group_count_arithmetic":
        bool(
            np.allclose(
                GROUP_COUNTS.sum(
                    axis=1
                ),
                N_RESULTS_DATE,
            )
        ),
}


INPUT_QA_PASS = bool(
    all(
        INPUT_QA.values()
    )
)


if not INPUT_QA_PASS:

    raise RuntimeError(
        f"V5 OU input QA failed:\n{INPUT_QA}"
    )


# ==================================================================================================
# PART E — OPTIONAL NUMBA ACCELERATION
# ==================================================================================================

NUMBA_AVAILABLE = False


try:

    from numba import njit

    NUMBA_AVAILABLE = True

except Exception:

    NUMBA_AVAILABLE = False


# ==================================================================================================
# PART F — KALMAN NEGATIVE LOG-LIKELIHOOD
# ==================================================================================================
#
# Parameter vector:
#
#       theta[0] = log(tau_days)
#       theta[1] = log(V_state)
#       theta[2] = log(R_M700)
#       theta[3] = log(R_M800)
#       theta[4] = log(R_W500)
#       theta[5] = log(R_W600)
#
#
# Initial state:
#
#       z_0 ~ Normal(0, V_state)
#
# i.e. stationary OU prior.
#
# ==================================================================================================

LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


def kalman_nll_python(
    theta,
    y,
    gaps,
    is_first,
    n_results,
    group_counts,
):

    tau = math.exp(
        theta[0]
    )


    state_var = math.exp(
        theta[1]
    )


    measurement_vars = np.exp(
        np.asarray(
            theta[
                2:6
            ],
            dtype=np.float64,
        )
    )


    state_mean = 0.0
    state_posterior_var = state_var

    nll = 0.0


    for i in range(
        len(
            y
        )
    ):

        # ------------------------------------------------------------------------------------------
        # New athlete: stationary prior.
        # ------------------------------------------------------------------------------------------

        if is_first[i]:

            predicted_mean = 0.0

            predicted_var = state_var


        else:

            gap = gaps[i]


            phi = math.exp(
                -gap
                /
                tau
            )


            process_var = (
                state_var
                *
                (
                    1.0
                    -
                    phi
                    *
                    phi
                )
            )


            predicted_mean = (
                phi
                *
                state_mean
            )


            predicted_var = (
                phi
                *
                phi
                *
                state_posterior_var

                +
                process_var
            )


        # ------------------------------------------------------------------------------------------
        # Measurement variance of daily mean.
        # ------------------------------------------------------------------------------------------

        numerator = (
            group_counts[i, 0]
            *
            measurement_vars[0]

            +
            group_counts[i, 1]
            *
            measurement_vars[1]

            +
            group_counts[i, 2]
            *
            measurement_vars[2]

            +
            group_counts[i, 3]
            *
            measurement_vars[3]
        )


        observation_var = (
            numerator
            /
            (
                n_results[i]
                *
                n_results[i]
            )
        )


        innovation = (
            y[i]
            -
            predicted_mean
        )


        innovation_var = (
            predicted_var
            +
            observation_var
        )


        if (
            not np.isfinite(
                innovation_var
            )
            or
            innovation_var
            <=
            0
        ):

            return 1e100


        nll += 0.5 * (
            LOG_2PI
            +
            math.log(
                innovation_var
            )
            +
            (
                innovation
                *
                innovation
                /
                innovation_var
            )
        )


        # ------------------------------------------------------------------------------------------
        # Kalman update.
        # ------------------------------------------------------------------------------------------

        kalman_gain = (
            predicted_var
            /
            innovation_var
        )


        state_mean = (
            predicted_mean
            +
            kalman_gain
            *
            innovation
        )


        state_posterior_var = (
            (
                1.0
                -
                kalman_gain
            )
            *
            predicted_var
        )


        # Numerical floor.
        if state_posterior_var < 1e-12:

            state_posterior_var = 1e-12


    return float(
        nll
    )


# ==================================================================================================
# PART G — NUMBA VERSION
# ==================================================================================================

if NUMBA_AVAILABLE:

    @njit(
        cache=False
    )
    def kalman_nll_numba(
        theta,
        y,
        gaps,
        is_first,
        n_results,
        group_counts,
    ):

        tau = np.exp(
            theta[0]
        )


        state_var = np.exp(
            theta[1]
        )


        measurement_vars = np.empty(
            4,
            dtype=np.float64,
        )


        for g in range(
            4
        ):

            measurement_vars[g] = np.exp(
                theta[
                    2
                    +
                    g
                ]
            )


        state_mean = 0.0

        state_posterior_var = (
            state_var
        )


        nll = 0.0


        for i in range(
            y.shape[0]
        ):

            if is_first[i]:

                predicted_mean = 0.0

                predicted_var = (
                    state_var
                )


            else:

                phi = np.exp(
                    -gaps[i]
                    /
                    tau
                )


                process_var = (
                    state_var
                    *
                    (
                        1.0
                        -
                        phi
                        *
                        phi
                    )
                )


                predicted_mean = (
                    phi
                    *
                    state_mean
                )


                predicted_var = (
                    phi
                    *
                    phi
                    *
                    state_posterior_var

                    +
                    process_var
                )


            numerator = 0.0


            for g in range(
                4
            ):

                numerator += (
                    group_counts[
                        i,
                        g
                    ]
                    *
                    measurement_vars[
                        g
                    ]
                )


            observation_var = (
                numerator
                /
                (
                    n_results[i]
                    *
                    n_results[i]
                )
            )


            innovation = (
                y[i]
                -
                predicted_mean
            )


            innovation_var = (
                predicted_var
                +
                observation_var
            )


            if (
                not np.isfinite(
                    innovation_var
                )
                or
                innovation_var
                <=
                0.0
            ):

                return 1e100


            nll += 0.5 * (
                LOG_2PI
                +
                np.log(
                    innovation_var
                )
                +
                (
                    innovation
                    *
                    innovation
                    /
                    innovation_var
                )
            )


            kalman_gain = (
                predicted_var
                /
                innovation_var
            )


            state_mean = (
                predicted_mean
                +
                kalman_gain
                *
                innovation
            )


            state_posterior_var = (
                (
                    1.0
                    -
                    kalman_gain
                )
                *
                predicted_var
            )


            if state_posterior_var < 1e-12:

                state_posterior_var = 1e-12


        return nll


    # Trigger compilation now.
    _ = kalman_nll_numba(
        np.log(
            np.array(
                [
                    200.0,
                    1.0,
                    12.0,
                    10.0,
                    7.0,
                    6.0,
                ],
                dtype=np.float64,
            )
        ),
        Y_STATE[:100],
        GAP_DAYS[:100],
        IS_FIRST_STATE[:100],
        N_RESULTS_DATE[:100],
        GROUP_COUNTS[:100],
    )


# ==================================================================================================
# PART H — OBJECTIVE WRAPPER
# ==================================================================================================

def v5_objective(
    theta,
):

    theta = np.asarray(
        theta,
        dtype=np.float64,
    )


    if NUMBA_AVAILABLE:

        return float(
            kalman_nll_numba(
                theta,
                Y_STATE,
                GAP_DAYS,
                IS_FIRST_STATE,
                N_RESULTS_DATE,
                GROUP_COUNTS,
            )
        )


    return kalman_nll_python(
        theta,
        Y_STATE,
        GAP_DAYS,
        IS_FIRST_STATE,
        N_RESULTS_DATE,
        GROUP_COUNTS,
    )


# ==================================================================================================
# PART I — INITIAL MEASUREMENT VARIANCE
# ==================================================================================================
#
# Use the fitted V4 Student-t residual scales only as optimizer starting points.
#
# They are NOT fixed V5 parameters.
#
# ==================================================================================================

V4_SIGMA_START = np.asarray(
    V4_SIGMA_GROUP_MEAN,
    dtype=float,
)


V4_VARIANCE_START = (
    V4_SIGMA_START
    **
    2
)


# ==================================================================================================
# PART J — DETERMINISTIC MULTI-START GRID
# ==================================================================================================

TAU_STARTS = [
    60.0,
    200.0,
    700.0,
]


STATE_SD_STARTS = [
    0.75,
    1.50,
    3.00,
]


OPTIMIZER_STARTS = []


for tau_start in TAU_STARTS:

    for state_sd_start in STATE_SD_STARTS:

        OPTIMIZER_STARTS.append(
            np.log(
                np.concatenate(
                    [
                        np.array(
                            [
                                tau_start,
                                state_sd_start ** 2,
                            ],
                            dtype=float,
                        ),

                        V4_VARIANCE_START,
                    ]
                )
            )
        )


# Historical fitted V5 parameters are included ONLY as an additional
# robustness starting point. They are not imposed on the new fit.
HISTORICAL_V5_START = np.log(
    np.array(
        [
            205.93329,

            1.1453342328922724 ** 2,

            17.1886849087697,
            11.9485433500227,
            8.39074626105651,
            7.70223415103696,
        ],
        dtype=float,
    )
)


OPTIMIZER_STARTS.append(
    HISTORICAL_V5_START
)


# ==================================================================================================
# PART K — PARAMETER BOUNDS
# ==================================================================================================
#
# Broad numerical bounds, not substantive priors.
#
# ==================================================================================================

PARAMETER_BOUNDS = [

    # tau_days
    (
        np.log(
            5.0
        ),
        np.log(
            5000.0
        ),
    ),

    # stationary state variance
    (
        np.log(
            1e-4
        ),
        np.log(
            100.0
        ),
    ),

    # measurement variances
    (
        np.log(
            1e-4
        ),
        np.log(
            400.0
        ),
    ),

    (
        np.log(
            1e-4
        ),
        np.log(
            400.0
        ),
    ),

    (
        np.log(
            1e-4
        ),
        np.log(
            400.0
        ),
    ),

    (
        np.log(
            1e-4
        ),
        np.log(
            400.0
        ),
    ),
]


# ==================================================================================================
# PART L — MULTI-START OPTIMIZATION
# ==================================================================================================

OPTIMIZER_RESULTS = []


for start_id, x0 in enumerate(
    OPTIMIZER_STARTS,
    start=1,
):

    result = minimize(
        v5_objective,

        x0=
            x0,

        method=
            "L-BFGS-B",

        bounds=
            PARAMETER_BOUNDS,

        options={
            "maxiter":
                2000,

            "ftol":
                1e-10,

            "gtol":
                1e-6,

            "maxls":
                50,
        },
    )


    fitted = np.exp(
        result.x
    )


    OPTIMIZER_RESULTS.append(
        {
            "start_id":
                start_id,

            "success":
                bool(
                    result.success
                ),

            "status":
                int(
                    result.status
                ),

            "message":
                str(
                    result.message
                ),

            "n_iterations":
                int(
                    result.nit
                ),

            "n_function_evals":
                int(
                    result.nfev
                ),

            "negative_log_likelihood":
                float(
                    result.fun
                ),

            "tau_days":
                float(
                    fitted[0]
                ),

            "state_variance":
                float(
                    fitted[1]
                ),

            "state_sigma":
                float(
                    np.sqrt(
                        fitted[1]
                    )
                ),

            "R_M700":
                float(
                    fitted[2]
                ),

            "R_M800":
                float(
                    fitted[3]
                ),

            "R_W500":
                float(
                    fitted[4]
                ),

            "R_W600":
                float(
                    fitted[5]
                ),

            "_result_object":
                result,
        }
    )


# ==================================================================================================
# PART M — OPTIMIZER RESULT TABLE
# ==================================================================================================

OPTIMIZER_RESULTS_DF = pd.DataFrame(
    [
        {
            key:
                value

            for key, value
            in row.items()

            if key
            !=
            "_result_object"
        }

        for row
        in OPTIMIZER_RESULTS
    ]
)


OPTIMIZER_RESULTS_DF = (
    OPTIMIZER_RESULTS_DF.sort_values(
        "negative_log_likelihood"
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART N — CHOOSE BEST SOLUTION
# ==================================================================================================

BEST_ROW = (
    OPTIMIZER_RESULTS_DF.iloc[
        0
    ]
)


BEST_START_ID = int(
    BEST_ROW[
        "start_id"
    ]
)


BEST_RESULT = next(
    row[
        "_result_object"
    ]

    for row
    in OPTIMIZER_RESULTS

    if row[
        "start_id"
    ]
    ==
    BEST_START_ID
)


BEST_PARAMS = np.exp(
    BEST_RESULT.x
)


V5_TAU_DAYS = float(
    BEST_PARAMS[0]
)


V5_STATE_VARIANCE = float(
    BEST_PARAMS[1]
)


V5_STATE_SIGMA = float(
    np.sqrt(
        V5_STATE_VARIANCE
    )
)


V5_MEASUREMENT_VARIANCES = {
    "M_700":
        float(
            BEST_PARAMS[2]
        ),

    "M_800":
        float(
            BEST_PARAMS[3]
        ),

    "W_500":
        float(
            BEST_PARAMS[4]
        ),

    "W_600":
        float(
            BEST_PARAMS[5]
        ),
}


V5_MEASUREMENT_SIGMAS = {
    group:
        float(
            np.sqrt(
                variance
            )
        )

    for group, variance
    in V5_MEASUREMENT_VARIANCES.items()
}


V5_NEGATIVE_LOG_LIKELIHOOD = float(
    BEST_RESULT.fun
)


# ==================================================================================================
# PART O — IMPLIED PERSISTENCE LAW
# ==================================================================================================

PERSISTENCE_HORIZONS = np.array(
    [
        1,
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
        730,
    ],
    dtype=float,
)


V5_PERSISTENCE_TABLE = pd.DataFrame(
    {
        "horizon_days":
            PERSISTENCE_HORIZONS.astype(
                int
            )
    }
)


V5_PERSISTENCE_TABLE[
    "phi"
] = np.exp(
    -PERSISTENCE_HORIZONS
    /
    V5_TAU_DAYS
)


V5_HALF_LIFE_DAYS = float(
    V5_TAU_DAYS
    *
    np.log(
        2.0
    )
)


# ==================================================================================================
# PART P — MODEL INFORMATION CRITERIA
# ==================================================================================================

N_V5_PARAMETERS = 6


V5_AIC = float(
    2.0
    *
    N_V5_PARAMETERS

    +
    2.0
    *
    V5_NEGATIVE_LOG_LIKELIHOOD
)


V5_BIC = float(
    np.log(
        N_STATE_ROWS
    )
    *
    N_V5_PARAMETERS

    +
    2.0
    *
    V5_NEGATIVE_LOG_LIKELIHOOD
)


# ==================================================================================================
# PART Q — OPTIMIZER ROBUSTNESS
# ==================================================================================================
#
# Compare all successful fits with the best negative log likelihood.
#
# ==================================================================================================

successful_results = (
    OPTIMIZER_RESULTS_DF.loc[
        OPTIMIZER_RESULTS_DF[
            "success"
        ]
    ]
    .copy()
)


N_SUCCESSFUL_STARTS = int(
    len(
        successful_results
    )
)


if N_SUCCESSFUL_STARTS > 0:

    successful_results[
        "delta_nll"
    ] = (
        successful_results[
            "negative_log_likelihood"
        ]
        -
        V5_NEGATIVE_LOG_LIKELIHOOD
    )


    N_NEAR_OPTIMAL_STARTS = int(
        (
            successful_results[
                "delta_nll"
            ]
            <=
            1e-3
        )
        .sum()
    )


    MAX_NEAR_OPTIMAL_TAU_DIFF = float(
        successful_results.loc[
            successful_results[
                "delta_nll"
            ]
            <=
            1e-3,
            "tau_days",
        ]
        .sub(
            V5_TAU_DAYS
        )
        .abs()
        .max()
    )


else:

    N_NEAR_OPTIMAL_STARTS = 0

    MAX_NEAR_OPTIMAL_TAU_DIFF = np.nan


OPTIMIZER_ROBUST_PASS = bool(

    bool(
        BEST_RESULT.success
    )

    and

    np.isfinite(
        V5_NEGATIVE_LOG_LIKELIHOOD
    )

    and

    N_SUCCESSFUL_STARTS
    >=
    2

    and

    N_NEAR_OPTIMAL_STARTS
    >=
    2
)


# ==================================================================================================
# PART R — BOUNDARY QA
# ==================================================================================================

best_log_params = (
    BEST_RESULT.x
)


BOUNDARY_TOL = 1e-4


AT_LOWER_BOUND = []


AT_UPPER_BOUND = []


PARAMETER_NAMES = [
    "tau_days",
    "state_variance",
    "R_M700",
    "R_M800",
    "R_W500",
    "R_W600",
]


for name, value, bounds in zip(
    PARAMETER_NAMES,
    best_log_params,
    PARAMETER_BOUNDS,
):

    lower, upper = bounds


    if (
        abs(
            value
            -
            lower
        )
        <
        BOUNDARY_TOL
    ):

        AT_LOWER_BOUND.append(
            name
        )


    if (
        abs(
            value
            -
            upper
        )
        <
        BOUNDARY_TOL
    ):

        AT_UPPER_BOUND.append(
            name
        )


N_BOUNDARY_PARAMETERS = int(
    len(
        AT_LOWER_BOUND
    )
    +
    len(
        AT_UPPER_BOUND
    )
)


BOUNDARY_PASS = bool(
    N_BOUNDARY_PARAMETERS
    ==
    0
)


# ==================================================================================================
# PART S — PARAMETER TABLE
# ==================================================================================================

V5_PARAMETER_TABLE = pd.DataFrame(
    [
        {
            "parameter":
                "tau_days",

            "value":
                V5_TAU_DAYS,
        },

        {
            "parameter":
                "half_life_days",

            "value":
                V5_HALF_LIFE_DAYS,
        },

        {
            "parameter":
                "state_variance",

            "value":
                V5_STATE_VARIANCE,
        },

        {
            "parameter":
                "state_sigma",

            "value":
                V5_STATE_SIGMA,
        },

        {
            "parameter":
                "R_M700",

            "value":
                V5_MEASUREMENT_VARIANCES[
                    "M_700"
                ],
        },

        {
            "parameter":
                "R_M800",

            "value":
                V5_MEASUREMENT_VARIANCES[
                    "M_800"
                ],
        },

        {
            "parameter":
                "R_W500",

            "value":
                V5_MEASUREMENT_VARIANCES[
                    "W_500"
                ],
        },

        {
            "parameter":
                "R_W600",

            "value":
                V5_MEASUREMENT_VARIANCES[
                    "W_600"
                ],
        },

        {
            "parameter":
                "sigma_M700",

            "value":
                V5_MEASUREMENT_SIGMAS[
                    "M_700"
                ],
        },

        {
            "parameter":
                "sigma_M800",

            "value":
                V5_MEASUREMENT_SIGMAS[
                    "M_800"
                ],
        },

        {
            "parameter":
                "sigma_W500",

            "value":
                V5_MEASUREMENT_SIGMAS[
                    "W_500"
                ],
        },

        {
            "parameter":
                "sigma_W600",

            "value":
                V5_MEASUREMENT_SIGMAS[
                    "W_600"
                ],
        },
    ]
)


# ==================================================================================================
# PART T — HISTORICAL V5 COMPARISON
# ==================================================================================================
#
# Diagnostic only.
#
# No release decision is based on matching the old model.
#
# ==================================================================================================

HISTORICAL_V5_REFERENCE = pd.DataFrame(
    [
        {
            "parameter":
                "tau_days",

            "historical":
                205.93329,

            "new":
                V5_TAU_DAYS,
        },

        {
            "parameter":
                "state_sigma",

            "historical":
                1.1453342328922724,

            "new":
                V5_STATE_SIGMA,
        },

        {
            "parameter":
                "R_M700",

            "historical":
                17.1886849087697,

            "new":
                V5_MEASUREMENT_VARIANCES[
                    "M_700"
                ],
        },

        {
            "parameter":
                "R_M800",

            "historical":
                11.9485433500227,

            "new":
                V5_MEASUREMENT_VARIANCES[
                    "M_800"
                ],
        },

        {
            "parameter":
                "R_W500",

            "historical":
                8.39074626105651,

            "new":
                V5_MEASUREMENT_VARIANCES[
                    "W_500"
                ],
        },

        {
            "parameter":
                "R_W600",

            "historical":
                7.70223415103696,

            "new":
                V5_MEASUREMENT_VARIANCES[
                    "W_600"
                ],
        },
    ]
)


HISTORICAL_V5_REFERENCE[
    "difference"
] = (
    HISTORICAL_V5_REFERENCE[
        "new"
    ]
    -
    HISTORICAL_V5_REFERENCE[
        "historical"
    ]
)


HISTORICAL_V5_REFERENCE[
    "pct_difference"
] = (
    100.0
    *
    HISTORICAL_V5_REFERENCE[
        "difference"
    ]
    /
    HISTORICAL_V5_REFERENCE[
        "historical"
    ]
)


# ==================================================================================================
# PART U — FINAL QA
# ==================================================================================================

PARAMETERS_FINITE_PASS = bool(

    np.isfinite(
        V5_TAU_DAYS
    )

    and

    np.isfinite(
        V5_STATE_VARIANCE
    )

    and

    np.isfinite(
        np.array(
            list(
                V5_MEASUREMENT_VARIANCES.values()
            )
        )
    )
    .all()
)


PARAMETERS_POSITIVE_PASS = bool(

    V5_TAU_DAYS > 0

    and

    V5_STATE_VARIANCE > 0

    and

    all(
        variance > 0

        for variance
        in V5_MEASUREMENT_VARIANCES.values()
    )
)


CELL_16_PASS = bool(

    INPUT_QA_PASS

    and

    PARAMETERS_FINITE_PASS

    and

    PARAMETERS_POSITIVE_PASS

    and

    OPTIMIZER_ROBUST_PASS

    and

    BOUNDARY_PASS
)


# ==================================================================================================
# PART V — SAVE ARTIFACTS
# ==================================================================================================

V5_PARAMETER_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_ou_parameters.json"
)


V5_PARAMETER_TABLE_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_ou_parameter_table.parquet"
)


V5_OPTIMIZER_RESULTS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_ou_optimizer_robustness.parquet"
)


V5_PERSISTENCE_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_ou_persistence_law.parquet"
)


V5_HISTORICAL_COMPARISON_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_historical_parameter_comparison.parquet"
)


V5_PARAMETER_ARTIFACT = {

    "model":
        "V5_CONTINUOUS_TIME_OU",

    "state_observation":
        "mean structural residual per athlete-date",

    "tau_days":
        V5_TAU_DAYS,

    "half_life_days":
        V5_HALF_LIFE_DAYS,

    "stationary_state_variance":
        V5_STATE_VARIANCE,

    "stationary_state_sigma":
        V5_STATE_SIGMA,

    "measurement_variance": {
        group:
            variance

        for group, variance
        in V5_MEASUREMENT_VARIANCES.items()
    },

    "measurement_sigma": {
        group:
            sigma

        for group, sigma
        in V5_MEASUREMENT_SIGMAS.items()
    },

    "negative_log_likelihood":
        V5_NEGATIVE_LOG_LIKELIHOOD,

    "AIC":
        V5_AIC,

    "BIC":
        V5_BIC,

    "n_state_rows":
        N_STATE_ROWS,

    "n_state_athletes":
        N_STATE_ATHLETES,

    "n_parameters":
        N_V5_PARAMETERS,

    "optimizer": {
        "method":
            "L-BFGS-B",

        "starts":
            len(
                OPTIMIZER_STARTS
            ),

        "successful_starts":
            N_SUCCESSFUL_STARTS,

        "near_optimal_starts":
            N_NEAR_OPTIMAL_STARTS,

        "best_start_id":
            BEST_START_ID,

        "numba_accelerated":
            NUMBA_AVAILABLE,
    },
}


with V5_PARAMETER_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        V5_PARAMETER_ARTIFACT,
        f,
        indent=2,
        default=str,
    )


V5_PARAMETER_TABLE.to_parquet(
    V5_PARAMETER_TABLE_PATH,
    index=False,
)


OPTIMIZER_RESULTS_DF.to_parquet(
    V5_OPTIMIZER_RESULTS_PATH,
    index=False,
)


V5_PERSISTENCE_TABLE.to_parquet(
    V5_PERSISTENCE_PATH,
    index=False,
)


HISTORICAL_V5_REFERENCE.to_parquet(
    V5_HISTORICAL_COMPARISON_PATH,
    index=False,
)


# ==================================================================================================
# PART W — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "v5_ou_fit"
] = {

    "status":
        (
            "PASSED"
            if CELL_16_PASS
            else "FAILED"
        ),

    "state_rows":
        N_STATE_ROWS,

    "state_athletes":
        N_STATE_ATHLETES,

    "tau_days":
        V5_TAU_DAYS,

    "half_life_days":
        V5_HALF_LIFE_DAYS,

    "state_variance":
        V5_STATE_VARIANCE,

    "state_sigma":
        V5_STATE_SIGMA,

    "measurement_variances":
        V5_MEASUREMENT_VARIANCES,

    "negative_log_likelihood":
        V5_NEGATIVE_LOG_LIKELIHOOD,

    "AIC":
        V5_AIC,

    "BIC":
        V5_BIC,

    "optimizer_successful_starts":
        N_SUCCESSFUL_STARTS,

    "optimizer_near_optimal_starts":
        N_NEAR_OPTIMAL_STARTS,

    "boundary_parameters":
        (
            AT_LOWER_BOUND
            +
            AT_UPPER_BOUND
        ),

    "parameter_path":
        str(
            V5_PARAMETER_PATH
        ),

    "optimizer_results_path":
        str(
            V5_OPTIMIZER_RESULTS_PATH
        ),

    "persistence_path":
        str(
            V5_PERSISTENCE_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V5_OU_PARAMETERS_READY"
    if CELL_16_PASS
    else "V5_OU_FIT_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART X — NOTEBOOK GLOBALS
# ==================================================================================================

NOTEBOOK_2_CELL_16_PASS = (
    CELL_16_PASS
)


# ==================================================================================================
# PART Y — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — V5 CONTINUOUS-TIME OU PARAMETER ESTIMATION")
print("=" * 145)


print()
print("TRAINING STATE DATA")
print("-" * 145)

print(
    f"State observations               : "
    f"{N_STATE_ROWS:,}"
)

print(
    f"Athletes                         : "
    f"{N_STATE_ATHLETES:,}"
)

print(
    f"Numba acceleration               : "
    f"{NUMBA_AVAILABLE}"
)


print()
print("OPTIMIZER ROBUSTNESS")
print("-" * 145)

display(
    OPTIMIZER_RESULTS_DF[
        [
            "start_id",
            "success",
            "n_iterations",
            "negative_log_likelihood",
            "tau_days",
            "state_sigma",
            "R_M700",
            "R_M800",
            "R_W500",
            "R_W600",
        ]
    ]
)


print()
print(
    f"Successful starts                : "
    f"{N_SUCCESSFUL_STARTS:,} / {len(OPTIMIZER_STARTS):,}"
)

print(
    f"Near-optimal starts (ΔNLL≤1e-3) : "
    f"{N_NEAR_OPTIMAL_STARTS:,}"
)

print(
    f"Best start ID                    : "
    f"{BEST_START_ID}"
)

print(
    f"Best optimizer success           : "
    f"{BEST_RESULT.success}"
)

print(
    f"Best optimizer message           : "
    f"{BEST_RESULT.message}"
)


print()
print("V5 OU PARAMETERS")
print("-" * 145)

print(
    f"tau                              : "
    f"{V5_TAU_DAYS:.6f} days"
)

print(
    f"half-life                        : "
    f"{V5_HALF_LIFE_DAYS:.6f} days"
)

print(
    f"stationary state variance        : "
    f"{V5_STATE_VARIANCE:.6f}"
)

print(
    f"stationary state sigma           : "
    f"{V5_STATE_SIGMA:.6f} m"
)


print()

display(
    V5_PARAMETER_TABLE
)


print()
print("IMPLIED CONTINUOUS PERSISTENCE LAW")
print("-" * 145)

display(
    V5_PERSISTENCE_TABLE
)


print()
print("HISTORICAL V5 COMPARISON — DIAGNOSTIC ONLY")
print("-" * 145)

display(
    HISTORICAL_V5_REFERENCE
)


print()
print("LIKELIHOOD")
print("-" * 145)

print(
    f"Negative log likelihood          : "
    f"{V5_NEGATIVE_LOG_LIKELIHOOD:,.6f}"
)

print(
    f"AIC                              : "
    f"{V5_AIC:,.6f}"
)

print(
    f"BIC                              : "
    f"{V5_BIC:,.6f}"
)


print()
print("QA")
print("-" * 145)

print(
    f"Input QA                         : "
    f"{INPUT_QA_PASS}"
)

print(
    f"Parameters finite               : "
    f"{PARAMETERS_FINITE_PASS}"
)

print(
    f"Parameters positive             : "
    f"{PARAMETERS_POSITIVE_PASS}"
)

print(
    f"Optimizer robust                : "
    f"{OPTIMIZER_ROBUST_PASS}"
)

print(
    f"Boundary parameters             : "
    f"{N_BOUNDARY_PARAMETERS}"
)

print(
    f"Boundary pass                   : "
    f"{BOUNDARY_PASS}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"V5 parameter artifact            : "
    f"{V5_PARAMETER_PATH}"
)

print(
    f"Optimizer robustness             : "
    f"{V5_OPTIMIZER_RESULTS_PATH}"
)

print(
    f"Persistence law                  : "
    f"{V5_PERSISTENCE_PATH}"
)


print()
print("=" * 145)


if CELL_16_PASS:

    print(
        "CELL 16 STATUS: V5 OU PARAMETER ESTIMATION PASSED"
    )

else:

    print(
        "CELL 16 STATUS: V5 OU PARAMETER ESTIMATION REVIEW REQUIRED"
    )


print("=" * 145)



V6 TRAINING — STRUCTURAL V4 EVALUATION + V5 STATE INPUT

POSTERIOR-MEAN STRUCTURAL PARAMETERS
-------------------------------------------------------------------------------------------------------------------------------------------------
nu                               : 7.361654
sigma athlete intercept          : 8.821278
sigma athlete slope              : 1.925922



,model_group,sigma_group_mean
0,M_700,3.535483
1,M_800,2.949993
2,W_500,2.451988
3,W_600,2.368909



OVERALL STRUCTURAL V4 METRICS
-------------------------------------------------------------------------------------------------------------------------------------------------


,prediction_split,n,athletes,MAE,RMSE,Bias
0,TRAIN,170644.0,12945.0,2.257118,3.081942,-0.089732
1,VALIDATION,4987.0,2798.0,3.639276,5.245284,0.229901
2,TEST,12865.0,5200.0,3.735700,5.325524,0.100082
3,POST_TEST,632.0,564.0,4.675588,6.534483,0.429996



SEEN VS UNSEEN ATHLETES
-------------------------------------------------------------------------------------------------------------------------------------------------


...


IMPLIED CONTINUOUS PERSISTENCE LAW
-------------------------------------------------------------------------------------------------------------------------------------------------


,horizon_days,phi
0,1,0.989930
1,7,0.931607
2,14,0.867891
3,30,0.738141
4,60,0.544853
5,90,0.402178
6,120,0.296865
7,180,0.161748
8,365,0.024871
9,730,0.000619



HISTORICAL V5 COMPARISON — DIAGNOSTIC ONLY
-------------------------------------------------------------------------------------------------------------------------------------------------


,parameter,historical,new,difference,pct_difference
0,tau_days,205.933290,98.807800,-107.125490,-52.019511
1,state_sigma,1.145334,1.701828,0.556494,48.587870
2,R_M700,17.188685,11.039562,-6.149123,-35.774251
3,R_M800,11.948543,8.157757,-3.790786,-31.725925
4,R_W500,8.390746,4.814586,-3.576160,-42.620288
5,R_W600,7.702234,4.513135,-3.189100,-41.404864



LIKELIHOOD
-------------------------------------------------------------------------------------------------------------------------------------------------
Negative log likelihood          : 422,410.013237
AIC                              : 844,832.026473
BIC                              : 844,892.275111

QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Input QA                         : True
Parameters finite               : True
Parameters positive             : True
Optimizer robust                : True
Boundary parameters             : 0
Boundary pass                   : True

OUTPUT
-------------------------------------------------------------------------------------------------------------------------------------------------
V5 parameter artifact            : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/v5_ou_paramet

## Stage 06 — V5 filtered state recovery and rolling evaluation

Recover filtered V5 state sequences/terminal states and perform true rolling-origin V5 evaluation.

**Legacy source cells represented:** 7, 8  
**Execution rule:** run only after the preceding stage has completed successfully.


In [7]:
# ==================================================================================================
# REFACTORED STAGE 06 — V5 filtered state recovery and rolling evaluation
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 7
# Original source SHA256: ddd8badab2ff6fbdd185d3511a5a6526ae051ccb16ca97683082396c52a580de
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 17 — RECOVER FILTERED V5 TRAIN STATE SEQUENCE
# ==================================================================================================
#
# Purpose
# -------
#
# Replay the fitted continuous-time OU / Kalman model through every TRAIN
# athlete-date and recover:
#
#       prior state mean
#       prior state variance
#       OU persistence phi
#       observation variance
#       innovation
#       innovation variance
#       Kalman gain
#       posterior state mean
#       posterior state variance
#
#
# This produces the athlete-specific transient state available immediately
# AFTER each observed TRAIN date.
#
#
# Why this matters
# ----------------
#
# For forecasting a future performance at date t+h:
#
#       z_forecast(h)
#           = exp(-h / tau) * z_last
#
#
# where z_last is the athlete's filtered posterior state after the last
# observation available before the prediction date.
#
#
# CRITICAL QA
# -----------
#
# Replaying the filter must reproduce the negative log likelihood from
# Cell 16.
#
# If it does not, the filtering implementation differs from the optimizer's
# likelihood and forecasting must NOT proceed.
#
# ==================================================================================================

import json

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_16_PASS",

    "V5_TRAIN_DATE_RESIDUALS",

    "V5_TAU_DAYS",
    "V5_STATE_VARIANCE",
    "V5_STATE_SIGMA",

    "V5_MEASUREMENT_VARIANCES",
    "V5_NEGATIVE_LOG_LIKELIHOOD",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through Cell 16 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_16_PASS:

    raise RuntimeError(
        "Cell 16 V5 OU parameter estimation did not pass."
    )


# ==================================================================================================
# PART B — PREPARE STATE TABLE
# ==================================================================================================

STATE_FILTER = (
    V5_TRAIN_DATE_RESIDUALS.copy()
)


STATE_FILTER[
    "result_date_dt"
] = pd.to_datetime(
    STATE_FILTER[
        "result_date_dt"
    ],
    errors="coerce",
)


STATE_FILTER = (
    STATE_FILTER.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


N_FILTER_ROWS = int(
    len(
        STATE_FILTER
    )
)


N_FILTER_ATHLETES = int(
    STATE_FILTER[
        "aaAthleteId"
    ]
    .nunique()
)


# ==================================================================================================
# PART C — ARRAYS
# ==================================================================================================

Y = (
    STATE_FILTER[
        "residual_mean"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


GAPS = (
    STATE_FILTER[
        "state_gap_days"
    ]
    .fillna(
        0.0
    )
    .to_numpy(
        dtype=np.float64
    )
)


STATE_INDEX = (
    STATE_FILTER[
        "state_date_index"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


IS_FIRST = (
    STATE_INDEX
    ==
    0
)


N_RESULTS = (
    STATE_FILTER[
        "n_results_on_date"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


GROUP_COUNTS = (
    STATE_FILTER[
        [
            "n_M700",
            "n_M800",
            "n_W500",
            "n_W600",
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


ATHLETE_IDS_STATE = (
    STATE_FILTER[
        "aaAthleteId"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


# ==================================================================================================
# PART D — FITTED PARAMETER VECTOR
# ==================================================================================================

MEASUREMENT_VARIANCE_VECTOR = np.array(
    [
        V5_MEASUREMENT_VARIANCES[
            "M_700"
        ],

        V5_MEASUREMENT_VARIANCES[
            "M_800"
        ],

        V5_MEASUREMENT_VARIANCES[
            "W_500"
        ],

        V5_MEASUREMENT_VARIANCES[
            "W_600"
        ],
    ],
    dtype=np.float64,
)


# ==================================================================================================
# PART E — OUTPUT ARRAYS
# ==================================================================================================

OU_PHI = np.ones(
    N_FILTER_ROWS,
    dtype=np.float64,
)


PROCESS_VARIANCE = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


PRIOR_STATE_MEAN = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


PRIOR_STATE_VARIANCE = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


OBSERVATION_VARIANCE = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


INNOVATION = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


INNOVATION_VARIANCE = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


STANDARDIZED_INNOVATION = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


KALMAN_GAIN = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


POSTERIOR_STATE_MEAN = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


POSTERIOR_STATE_VARIANCE = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


NLL_CONTRIBUTION = np.zeros(
    N_FILTER_ROWS,
    dtype=np.float64,
)


# ==================================================================================================
# PART F — FILTER REPLAY
# ==================================================================================================

LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


state_mean = 0.0

state_variance = (
    V5_STATE_VARIANCE
)


previous_athlete = None


for i in range(
    N_FILTER_ROWS
):

    athlete_id = (
        ATHLETE_IDS_STATE[
            i
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # New athlete
    # ----------------------------------------------------------------------------------------------

    if (
        IS_FIRST[i]
        or
        previous_athlete is None
        or
        athlete_id
        !=
        previous_athlete
    ):

        phi = 1.0

        process_variance = 0.0

        prior_mean = 0.0

        prior_variance = (
            V5_STATE_VARIANCE
        )


    # ----------------------------------------------------------------------------------------------
    # Existing athlete — OU propagation
    # ----------------------------------------------------------------------------------------------

    else:

        gap = (
            GAPS[i]
        )


        phi = float(
            np.exp(
                -gap
                /
                V5_TAU_DAYS
            )
        )


        process_variance = (
            V5_STATE_VARIANCE
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


        prior_mean = (
            phi
            *
            state_mean
        )


        prior_variance = (
            phi
            *
            phi
            *
            state_variance

            +
            process_variance
        )


    # ----------------------------------------------------------------------------------------------
    # Effective observation variance of date-level residual mean
    # ----------------------------------------------------------------------------------------------

    observation_variance = float(
        np.dot(
            GROUP_COUNTS[
                i
            ],
            MEASUREMENT_VARIANCE_VECTOR,
        )
        /
        (
            N_RESULTS[i]
            *
            N_RESULTS[i]
        )
    )


    innovation = (
        Y[i]
        -
        prior_mean
    )


    innovation_variance = (
        prior_variance
        +
        observation_variance
    )


    if (
        not np.isfinite(
            innovation_variance
        )
        or
        innovation_variance
        <=
        0.0
    ):

        raise RuntimeError(
            f"Invalid innovation variance at row {i}: "
            f"{innovation_variance}"
        )


    standardized_innovation = (
        innovation
        /
        np.sqrt(
            innovation_variance
        )
    )


    kalman_gain = (
        prior_variance
        /
        innovation_variance
    )


    posterior_mean = (
        prior_mean
        +
        kalman_gain
        *
        innovation
    )


    posterior_variance = (
        (
            1.0
            -
            kalman_gain
        )
        *
        prior_variance
    )


    posterior_variance = max(
        posterior_variance,
        1e-12,
    )


    nll_i = 0.5 * (
        LOG_2PI
        +
        np.log(
            innovation_variance
        )
        +
        (
            innovation
            *
            innovation
            /
            innovation_variance
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Save
    # ----------------------------------------------------------------------------------------------

    OU_PHI[i] = (
        phi
    )


    PROCESS_VARIANCE[i] = (
        process_variance
    )


    PRIOR_STATE_MEAN[i] = (
        prior_mean
    )


    PRIOR_STATE_VARIANCE[i] = (
        prior_variance
    )


    OBSERVATION_VARIANCE[i] = (
        observation_variance
    )


    INNOVATION[i] = (
        innovation
    )


    INNOVATION_VARIANCE[i] = (
        innovation_variance
    )


    STANDARDIZED_INNOVATION[i] = (
        standardized_innovation
    )


    KALMAN_GAIN[i] = (
        kalman_gain
    )


    POSTERIOR_STATE_MEAN[i] = (
        posterior_mean
    )


    POSTERIOR_STATE_VARIANCE[i] = (
        posterior_variance
    )


    NLL_CONTRIBUTION[i] = (
        nll_i
    )


    # ----------------------------------------------------------------------------------------------
    # Advance filter state
    # ----------------------------------------------------------------------------------------------

    state_mean = (
        posterior_mean
    )


    state_variance = (
        posterior_variance
    )


    previous_athlete = (
        athlete_id
    )


# ==================================================================================================
# PART G — ATTACH FILTER RESULTS
# ==================================================================================================

STATE_FILTER[
    "ou_phi"
] = OU_PHI


STATE_FILTER[
    "ou_process_variance"
] = PROCESS_VARIANCE


STATE_FILTER[
    "state_prior_mean"
] = PRIOR_STATE_MEAN


STATE_FILTER[
    "state_prior_variance"
] = PRIOR_STATE_VARIANCE


STATE_FILTER[
    "state_observation_variance"
] = OBSERVATION_VARIANCE


STATE_FILTER[
    "state_innovation"
] = INNOVATION


STATE_FILTER[
    "state_innovation_variance"
] = INNOVATION_VARIANCE


STATE_FILTER[
    "state_standardized_innovation"
] = STANDARDIZED_INNOVATION


STATE_FILTER[
    "kalman_gain"
] = KALMAN_GAIN


STATE_FILTER[
    "state_posterior_mean"
] = POSTERIOR_STATE_MEAN


STATE_FILTER[
    "state_posterior_variance"
] = POSTERIOR_STATE_VARIANCE


STATE_FILTER[
    "state_posterior_sigma"
] = np.sqrt(
    POSTERIOR_STATE_VARIANCE
)


STATE_FILTER[
    "state_nll_contribution"
] = NLL_CONTRIBUTION


# ==================================================================================================
# PART H — LIKELIHOOD RECONSTRUCTION QA
# ==================================================================================================

FILTER_REPLAY_NLL = float(
    STATE_FILTER[
        "state_nll_contribution"
    ]
    .sum()
)


NLL_ABS_DIFF = float(
    abs(
        FILTER_REPLAY_NLL
        -
        V5_NEGATIVE_LOG_LIKELIHOOD
    )
)


NLL_REL_DIFF = float(
    NLL_ABS_DIFF
    /
    max(
        1.0,
        abs(
            V5_NEGATIVE_LOG_LIKELIHOOD
        ),
    )
)


NLL_RECONSTRUCTION_PASS = bool(
    NLL_ABS_DIFF
    <
    1e-4
)


# ==================================================================================================
# PART I — FIRST-STATE QA
# ==================================================================================================

first_rows = (
    STATE_FILTER[
        "state_date_index"
    ]
    .eq(
        0
    )
)


N_FIRST_PRIOR_MEAN_ERRORS = int(
    (
        np.abs(
            STATE_FILTER.loc[
                first_rows,
                "state_prior_mean",
            ]
        )
        >
        1e-12
    )
    .sum()
)


N_FIRST_PRIOR_VARIANCE_ERRORS = int(
    (
        np.abs(
            STATE_FILTER.loc[
                first_rows,
                "state_prior_variance",
            ]
            -
            V5_STATE_VARIANCE
        )
        >
        1e-10
    )
    .sum()
)


N_FIRST_PHI_ERRORS = int(
    (
        np.abs(
            STATE_FILTER.loc[
                first_rows,
                "ou_phi",
            ]
            -
            1.0
        )
        >
        1e-12
    )
    .sum()
)


# ==================================================================================================
# PART J — GENERAL FILTER QA
# ==================================================================================================

FILTER_NUMERIC_COLUMNS = [
    "state_prior_mean",
    "state_prior_variance",
    "state_observation_variance",
    "state_innovation",
    "state_innovation_variance",
    "state_standardized_innovation",
    "kalman_gain",
    "state_posterior_mean",
    "state_posterior_variance",
]


N_FILTER_NONFINITE = int(
    (
        ~np.isfinite(
            STATE_FILTER[
                FILTER_NUMERIC_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


N_NONPOSITIVE_PRIOR_VARIANCE = int(
    (
        STATE_FILTER[
            "state_prior_variance"
        ]
        <=
        0
    )
    .sum()
)


N_NONPOSITIVE_POSTERIOR_VARIANCE = int(
    (
        STATE_FILTER[
            "state_posterior_variance"
        ]
        <=
        0
    )
    .sum()
)


N_INVALID_KALMAN_GAIN = int(
    (
        (
            STATE_FILTER[
                "kalman_gain"
            ]
            <
            0
        )

        |

        (
            STATE_FILTER[
                "kalman_gain"
            ]
            >
            1
        )
    )
    .sum()
)


# ==================================================================================================
# PART K — TERMINAL TRAIN STATE PER ATHLETE
# ==================================================================================================
#
# This is the state available at the end of TRAIN history for each athlete.
#
# It will be propagated forward into validation/test forecasts.
#
# ==================================================================================================

V5_TRAIN_TERMINAL_STATE = (
    STATE_FILTER.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .groupby(
        "aaAthleteId",
        as_index=False,
        sort=False,
    )
    .tail(
        1
    )
    .copy()
    .reset_index(
        drop=True
    )
)


V5_TRAIN_TERMINAL_STATE = (
    V5_TRAIN_TERMINAL_STATE[
        [
            "aaAthleteId",
            "result_date_dt",
            "state_date_index",
            "state_posterior_mean",
            "state_posterior_variance",
            "state_posterior_sigma",
            "n_results_on_date",
            "n_model_groups_on_date",
            "state_observation_group",
        ]
    ]
    .rename(
        columns={
            "result_date_dt":
                "last_train_state_date",

            "state_posterior_mean":
                "last_train_state_mean",

            "state_posterior_variance":
                "last_train_state_variance",

            "state_posterior_sigma":
                "last_train_state_sigma",
        }
    )
)


N_TERMINAL_STATES = int(
    len(
        V5_TRAIN_TERMINAL_STATE
    )
)


TERMINAL_STATE_COUNT_PASS = bool(
    N_TERMINAL_STATES
    ==
    N_FILTER_ATHLETES
)


# ==================================================================================================
# PART L — STATE DISTRIBUTION SUMMARY
# ==================================================================================================

STATE_SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "posterior_state_mean_mean",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_mean"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "posterior_state_mean_sd",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_mean"
                    ]
                    .std()
                ),
        },

        {
            "metric":
                "posterior_state_mean_q05",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_mean"
                    ]
                    .quantile(
                        0.05
                    )
                ),
        },

        {
            "metric":
                "posterior_state_mean_q50",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_mean"
                    ]
                    .quantile(
                        0.50
                    )
                ),
        },

        {
            "metric":
                "posterior_state_mean_q95",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_mean"
                    ]
                    .quantile(
                        0.95
                    )
                ),
        },

        {
            "metric":
                "posterior_state_sigma_mean",

            "value":
                float(
                    STATE_FILTER[
                        "state_posterior_sigma"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "kalman_gain_mean",

            "value":
                float(
                    STATE_FILTER[
                        "kalman_gain"
                    ]
                    .mean()
                ),
        },
    ]
)


# ==================================================================================================
# PART M — INNOVATION DIAGNOSTICS
# ==================================================================================================

INNOVATION_SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "innovation_mean",

            "value":
                float(
                    STATE_FILTER[
                        "state_innovation"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "innovation_sd",

            "value":
                float(
                    STATE_FILTER[
                        "state_innovation"
                    ]
                    .std()
                ),
        },

        {
            "metric":
                "standardized_innovation_mean",

            "value":
                float(
                    STATE_FILTER[
                        "state_standardized_innovation"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "standardized_innovation_sd",

            "value":
                float(
                    STATE_FILTER[
                        "state_standardized_innovation"
                    ]
                    .std()
                ),
        },

        {
            "metric":
                "abs_std_innovation_gt_2_pct",

            "value":
                float(
                    100.0
                    *
                    (
                        np.abs(
                            STATE_FILTER[
                                "state_standardized_innovation"
                            ]
                        )
                        >
                        2.0
                    )
                    .mean()
                ),
        },

        {
            "metric":
                "abs_std_innovation_gt_3_pct",

            "value":
                float(
                    100.0
                    *
                    (
                        np.abs(
                            STATE_FILTER[
                                "state_standardized_innovation"
                            ]
                        )
                        >
                        3.0
                    )
                    .mean()
                ),
        },
    ]
)


# ==================================================================================================
# PART N — TERMINAL STATE SUMMARY
# ==================================================================================================

TERMINAL_STATE_SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "terminal_state_mean_mean",

            "value":
                float(
                    V5_TRAIN_TERMINAL_STATE[
                        "last_train_state_mean"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "terminal_state_mean_sd",

            "value":
                float(
                    V5_TRAIN_TERMINAL_STATE[
                        "last_train_state_mean"
                    ]
                    .std()
                ),
        },

        {
            "metric":
                "terminal_state_sigma_mean",

            "value":
                float(
                    V5_TRAIN_TERMINAL_STATE[
                        "last_train_state_sigma"
                    ]
                    .mean()
                ),
        },

        {
            "metric":
                "terminal_state_abs_mean",

            "value":
                float(
                    V5_TRAIN_TERMINAL_STATE[
                        "last_train_state_mean"
                    ]
                    .abs()
                    .mean()
                ),
        },
    ]
)


# ==================================================================================================
# PART O — FINAL CELL QA
# ==================================================================================================

CELL_17_PASS = bool(

    NLL_RECONSTRUCTION_PASS

    and

    N_FIRST_PRIOR_MEAN_ERRORS == 0

    and

    N_FIRST_PRIOR_VARIANCE_ERRORS == 0

    and

    N_FIRST_PHI_ERRORS == 0

    and

    N_FILTER_NONFINITE == 0

    and

    N_NONPOSITIVE_PRIOR_VARIANCE == 0

    and

    N_NONPOSITIVE_POSTERIOR_VARIANCE == 0

    and

    N_INVALID_KALMAN_GAIN == 0

    and

    TERMINAL_STATE_COUNT_PASS
)


# ==================================================================================================
# PART P — SAVE ARTIFACTS
# ==================================================================================================

V5_FILTERED_STATE_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_train_filtered_states.parquet"
)


V5_TERMINAL_STATE_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_train_terminal_states.parquet"
)


V5_STATE_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_filtered_state_summary.parquet"
)


V5_INNOVATION_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_innovation_summary.parquet"
)


V5_TERMINAL_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_terminal_state_summary.parquet"
)


STATE_FILTER.to_parquet(
    V5_FILTERED_STATE_PATH,
    index=False,
)


V5_TRAIN_TERMINAL_STATE.to_parquet(
    V5_TERMINAL_STATE_PATH,
    index=False,
)


STATE_SUMMARY.to_parquet(
    V5_STATE_SUMMARY_PATH,
    index=False,
)


INNOVATION_SUMMARY.to_parquet(
    V5_INNOVATION_SUMMARY_PATH,
    index=False,
)


TERMINAL_STATE_SUMMARY.to_parquet(
    V5_TERMINAL_SUMMARY_PATH,
    index=False,
)


# ==================================================================================================
# PART Q — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "v5_filtered_state_recovery"
] = {

    "status":
        (
            "PASSED"
            if CELL_17_PASS
            else "FAILED"
        ),

    "state_rows":
        N_FILTER_ROWS,

    "athletes":
        N_FILTER_ATHLETES,

    "terminal_states":
        N_TERMINAL_STATES,

    "optimizer_nll":
        V5_NEGATIVE_LOG_LIKELIHOOD,

    "replayed_nll":
        FILTER_REPLAY_NLL,

    "nll_absolute_difference":
        NLL_ABS_DIFF,

    "nll_relative_difference":
        NLL_REL_DIFF,

    "first_state_prior_mean_errors":
        N_FIRST_PRIOR_MEAN_ERRORS,

    "first_state_prior_variance_errors":
        N_FIRST_PRIOR_VARIANCE_ERRORS,

    "invalid_kalman_gain":
        N_INVALID_KALMAN_GAIN,

    "filtered_state_path":
        str(
            V5_FILTERED_STATE_PATH
        ),

    "terminal_state_path":
        str(
            V5_TERMINAL_STATE_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V5_FILTERED_STATES_READY"
    if CELL_17_PASS
    else "V5_FILTERED_STATE_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART R — NOTEBOOK GLOBALS
# ==================================================================================================

V5_TRAIN_FILTERED_STATES = (
    STATE_FILTER
)


NOTEBOOK_2_CELL_17_PASS = (
    CELL_17_PASS
)


# ==================================================================================================
# PART S — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — V5 FILTERED TRAIN STATE RECOVERY")
print("=" * 145)


print()
print("STATE SEQUENCE")
print("-" * 145)

print(
    f"Filtered state rows              : "
    f"{N_FILTER_ROWS:,}"
)

print(
    f"Athletes                         : "
    f"{N_FILTER_ATHLETES:,}"
)

print(
    f"Terminal athlete states          : "
    f"{N_TERMINAL_STATES:,}"
)


print()
print("LIKELIHOOD RECONSTRUCTION")
print("-" * 145)

print(
    f"Cell 16 optimizer NLL            : "
    f"{V5_NEGATIVE_LOG_LIKELIHOOD:,.9f}"
)

print(
    f"Filter replay NLL                : "
    f"{FILTER_REPLAY_NLL:,.9f}"
)

print(
    f"Absolute difference              : "
    f"{NLL_ABS_DIFF:.12f}"
)

print(
    f"Relative difference              : "
    f"{NLL_REL_DIFF:.3e}"
)

print(
    f"NLL reconstruction pass          : "
    f"{NLL_RECONSTRUCTION_PASS}"
)


print()
print("FILTER QA")
print("-" * 145)

print(
    f"First-state prior-mean errors    : "
    f"{N_FIRST_PRIOR_MEAN_ERRORS:,}"
)

print(
    f"First-state prior-var errors     : "
    f"{N_FIRST_PRIOR_VARIANCE_ERRORS:,}"
)

print(
    f"First-state phi errors           : "
    f"{N_FIRST_PHI_ERRORS:,}"
)

print(
    f"Nonfinite filter values          : "
    f"{N_FILTER_NONFINITE:,}"
)

print(
    f"Nonpositive prior variances      : "
    f"{N_NONPOSITIVE_PRIOR_VARIANCE:,}"
)

print(
    f"Nonpositive posterior variances  : "
    f"{N_NONPOSITIVE_POSTERIOR_VARIANCE:,}"
)

print(
    f"Invalid Kalman gains             : "
    f"{N_INVALID_KALMAN_GAIN:,}"
)

print(
    f"Terminal-state count pass        : "
    f"{TERMINAL_STATE_COUNT_PASS}"
)


print()
print("FILTERED STATE DISTRIBUTION")
print("-" * 145)

display(
    STATE_SUMMARY
)


print()
print("INNOVATION DIAGNOSTICS")
print("-" * 145)

display(
    INNOVATION_SUMMARY
)


print()
print("TERMINAL TRAIN STATES")
print("-" * 145)

display(
    TERMINAL_STATE_SUMMARY
)


print()
print("SAMPLE TERMINAL STATES")
print("-" * 145)

display(
    V5_TRAIN_TERMINAL_STATE.head(
        20
    )
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Filtered state sequence          : "
    f"{V5_FILTERED_STATE_PATH}"
)

print(
    f"Terminal TRAIN states            : "
    f"{V5_TERMINAL_STATE_PATH}"
)

print(
    f"Innovation summary               : "
    f"{V5_INNOVATION_SUMMARY_PATH}"
)


print()
print("=" * 145)


if CELL_17_PASS:

    print(
        "CELL 17 STATUS: V5 FILTERED TRAIN STATE RECOVERY PASSED"
    )

else:

    print(
        "CELL 17 STATUS: V5 FILTERED STATE REVIEW REQUIRED"
    )


print("=" * 145)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 8
# Original source SHA256: aba32a5c3b68e735c0ad808b2b0970d7f211d183a543a1a1a2a0a39df0dc7592
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# NOTEBOOK 2 — V6 TRAINING AND RELEASE
#
# CELL 18 — TRUE ROLLING-ORIGIN V5 EVALUATION
# ==================================================================================================
#
# Purpose
# -------
#
# Simulate how V5 behaves in production.
#
# For every VALIDATION / TEST / POST_TEST athlete-date:
#
#   1. use ONLY the state available BEFORE that date
#   2. propagate that state forward using the fitted OU persistence law
#   3. predict all performances on that date
#   4. score those predictions
#   5. only AFTER scoring, update the latent state using the observed residual(s)
#   6. continue to the athlete's next date
#
#
# This means:
#
#       no same-day leakage
#       no future leakage
#       validation observations may update state before later test dates
#       test observations may update state before later test dates
#
#
# This is intentional.
#
# V4/V5 PARAMETERS are frozen from TRAIN.
# Updating an athlete's latent state with newly observed performances is
# filtering, not parameter fitting.
#
#
# V5 prediction:
#
#       mu_V5(t)
#           =
#           mu_V4(t)
#           +
#           z_prior(t)
#
#
# OU propagation:
#
#       phi(dt)
#           =
#           exp(-dt / tau)
#
#
#       z_prior(t)
#           =
#           phi(dt) * z_last
#
#
#       P_prior(t)
#           =
#           phi(dt)^2 * P_last
#           +
#           V_state * (1 - phi(dt)^2)
#
#
# For an athlete not seen in TRAIN:
#
#       first future prediction:
#
#           z_prior = 0
#           P_prior = V_state
#
# After the athlete produces a result, V5 can update the latent state and
# subsequent predictions become individualized.
#
# ==================================================================================================

import json

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "NOTEBOOK_2_CELL_17_PASS",

    "V4_VALIDATION_PRED",
    "V4_TEST_PRED",
    "V4_POST_TEST_PRED",

    "V5_TRAIN_TERMINAL_STATE",

    "V5_TAU_DAYS",
    "V5_STATE_VARIANCE",
    "V5_STATE_SIGMA",
    "V5_MEASUREMENT_VARIANCES",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 2 through Cell 17 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NOTEBOOK_2_CELL_17_PASS:

    raise RuntimeError(
        "Cell 17 filtered-state recovery did not pass."
    )


# ==================================================================================================
# PART B — COMBINE FUTURE EVALUATION ROWS
# ==================================================================================================

EVAL_ROWS = pd.concat(
    [
        V4_VALIDATION_PRED.copy(),
        V4_TEST_PRED.copy(),
        V4_POST_TEST_PRED.copy(),
    ],
    ignore_index=True,
)


EVAL_ROWS[
    "result_date_dt"
] = pd.to_datetime(
    EVAL_ROWS[
        "result_date_dt"
    ],
    errors="coerce",
)


EVAL_ROWS[
    "athlete_training_status"
] = np.where(
    EVAL_ROWS[
        "athlete_seen_in_training"
    ],
    "SEEN",
    "UNSEEN",
)


EVAL_ROWS = (
    EVAL_ROWS.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
            "model_group",
            "mark_numeric_m",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


N_EVAL_ROWS = int(
    len(
        EVAL_ROWS
    )
)


N_EVAL_ATHLETES = int(
    EVAL_ROWS[
        "aaAthleteId"
    ]
    .nunique()
)


# ==================================================================================================
# PART C — MEASUREMENT VARIANCE LOOKUP
# ==================================================================================================

MEASUREMENT_VARIANCE_LOOKUP = {
    str(group):
        float(variance)

    for group, variance
    in V5_MEASUREMENT_VARIANCES.items()
}


EVAL_ROWS[
    "v5_measurement_variance"
] = (
    EVAL_ROWS[
        "model_group"
    ]
    .map(
        MEASUREMENT_VARIANCE_LOOKUP
    )
)


if EVAL_ROWS[
    "v5_measurement_variance"
].isna().any():

    bad_groups = sorted(
        EVAL_ROWS.loc[
            EVAL_ROWS[
                "v5_measurement_variance"
            ]
            .isna(),
            "model_group",
        ]
        .astype(str)
        .unique()
        .tolist()
    )


    raise RuntimeError(
        "Missing V5 measurement variance for groups:\n"
        + "\n".join(
            f"  - {group}"
            for group in bad_groups
        )
    )


# ==================================================================================================
# PART D — BUILD ONE FUTURE OBSERVATION PER ATHLETE-DATE
# ==================================================================================================

EVAL_DATE_SOURCE = (
    EVAL_ROWS.copy()
)


STATE_GROUP_ORDER = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


for group in STATE_GROUP_ORDER:

    safe_group = (
        group.replace(
            "_",
            ""
        )
    )


    EVAL_DATE_SOURCE[
        f"_count_{safe_group}"
    ] = (
        EVAL_DATE_SOURCE[
            "model_group"
        ]
        .eq(
            group
        )
        .astype(
            "int16"
        )
    )


# ==================================================================================================
# PART E — DATE-LEVEL EVALUATION TABLE
# ==================================================================================================

EVAL_DATES = (
    EVAL_DATE_SOURCE.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        prediction_split=(
            "prediction_split",
            "first",
        ),

        n_split_values=(
            "prediction_split",
            "nunique",
        ),

        athlete_seen_in_training=(
            "athlete_seen_in_training",
            "first",
        ),

        n_results_on_date=(
            "mark_numeric_m",
            "size",
        ),

        residual_mean=(
            "v4_structural_residual",
            "mean",
        ),

        residual_median=(
            "v4_structural_residual",
            "median",
        ),

        n_M700=(
            "_count_M700",
            "sum",
        ),

        n_M800=(
            "_count_M800",
            "sum",
        ),

        n_W500=(
            "_count_W500",
            "sum",
        ),

        n_W600=(
            "_count_W600",
            "sum",
        ),
    )
)


EVAL_DATES = (
    EVAL_DATES.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


N_EVAL_DATES = int(
    len(
        EVAL_DATES
    )
)


# ==================================================================================================
# PART F — DATE TABLE QA
# ==================================================================================================

N_DATE_SPLIT_CONFLICTS = int(
    (
        EVAL_DATES[
            "n_split_values"
        ]
        !=
        1
    )
    .sum()
)


N_DUPLICATE_EVAL_DATES = int(
    EVAL_DATES.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ],
        keep=False,
    )
    .sum()
)


DATE_ROW_ARITHMETIC_PASS = bool(
    int(
        EVAL_DATES[
            "n_results_on_date"
        ]
        .sum()
    )
    ==
    N_EVAL_ROWS
)


# ==================================================================================================
# PART G — TERMINAL TRAIN STATE LOOKUP
# ==================================================================================================

TRAIN_STATE_LOOKUP = {}


for _, row in V5_TRAIN_TERMINAL_STATE.iterrows():

    athlete_id = int(
        row[
            "aaAthleteId"
        ]
    )


    TRAIN_STATE_LOOKUP[
        athlete_id
    ] = {
        "date":
            pd.Timestamp(
                row[
                    "last_train_state_date"
                ]
            ),

        "mean":
            float(
                row[
                    "last_train_state_mean"
                ]
            ),

        "variance":
            float(
                row[
                    "last_train_state_variance"
                ]
            ),

        "count":
            int(
                row[
                    "state_date_index"
                ]
            )
            +
            1,
    }


# ==================================================================================================
# PART H — GROUP VARIANCE VECTOR
# ==================================================================================================

GROUP_VARIANCE_VECTOR = np.array(
    [
        V5_MEASUREMENT_VARIANCES[
            "M_700"
        ],

        V5_MEASUREMENT_VARIANCES[
            "M_800"
        ],

        V5_MEASUREMENT_VARIANCES[
            "W_500"
        ],

        V5_MEASUREMENT_VARIANCES[
            "W_600"
        ],
    ],
    dtype=np.float64,
)


# ==================================================================================================
# PART I — TRUE ROLLING-ORIGIN FILTER
# ==================================================================================================

ROLLING_DATE_RESULTS = []


for athlete_id, athlete_dates in EVAL_DATES.groupby(
    "aaAthleteId",
    sort=False,
):

    athlete_id = int(
        athlete_id
    )


    athlete_dates = (
        athlete_dates.sort_values(
            "result_date_dt",
            kind="mergesort",
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Initial state
    # ----------------------------------------------------------------------------------------------

    seed = (
        TRAIN_STATE_LOOKUP.get(
            athlete_id
        )
    )


    if seed is not None:

        state_mean = float(
            seed[
                "mean"
            ]
        )


        state_variance = float(
            seed[
                "variance"
            ]
        )


        state_date = pd.Timestamp(
            seed[
                "date"
            ]
        )


        state_observation_count = int(
            seed[
                "count"
            ]
        )


        seeded_from_train = True


    else:

        state_mean = 0.0

        state_variance = float(
            V5_STATE_VARIANCE
        )


        state_date = None

        state_observation_count = 0

        seeded_from_train = False


    # ----------------------------------------------------------------------------------------------
    # Walk forward one DATE at a time
    # ----------------------------------------------------------------------------------------------

    for _, date_row in athlete_dates.iterrows():

        prediction_date = pd.Timestamp(
            date_row[
                "result_date_dt"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # 1 — PRIOR STATE AVAILABLE BEFORE THIS DATE
        # ------------------------------------------------------------------------------------------

        if state_date is None:

            gap_days = np.nan

            phi = np.nan

            prior_mean = 0.0

            prior_variance = float(
                V5_STATE_VARIANCE
            )


            source_date = pd.NaT


        else:

            gap_days = int(
                (
                    prediction_date
                    -
                    state_date
                )
                .days
            )


            if gap_days <= 0:

                raise RuntimeError(
                    "Rolling-origin chronology failure.\n\n"
                    f"Athlete: {athlete_id}\n"
                    f"Previous state date: {state_date}\n"
                    f"Prediction date:     {prediction_date}\n"
                    f"Gap:                 {gap_days}"
                )


            phi = float(
                np.exp(
                    -gap_days
                    /
                    V5_TAU_DAYS
                )
            )


            process_variance = (
                V5_STATE_VARIANCE
                *
                (
                    1.0
                    -
                    phi
                    *
                    phi
                )
            )


            prior_mean = (
                phi
                *
                state_mean
            )


            prior_variance = (
                phi
                *
                phi
                *
                state_variance

                +
                process_variance
            )


            source_date = pd.Timestamp(
                state_date
            )


        state_history_before = int(
            state_observation_count
        )


        state_history_available = bool(
            state_history_before
            >
            0
        )


        # ------------------------------------------------------------------------------------------
        # 2 — DATE-LEVEL OBSERVATION VARIANCE
        # ------------------------------------------------------------------------------------------

        group_counts = np.array(
            [
                date_row[
                    "n_M700"
                ],

                date_row[
                    "n_M800"
                ],

                date_row[
                    "n_W500"
                ],

                date_row[
                    "n_W600"
                ],
            ],
            dtype=np.float64,
        )


        n_results = float(
            date_row[
                "n_results_on_date"
            ]
        )


        observation_variance = float(
            np.dot(
                group_counts,
                GROUP_VARIANCE_VECTOR,
            )
            /
            (
                n_results
                *
                n_results
            )
        )


        # ------------------------------------------------------------------------------------------
        # 3 — SCORE DATE BEFORE UPDATE
        # ------------------------------------------------------------------------------------------

        observed_residual_mean = float(
            date_row[
                "residual_mean"
            ]
        )


        innovation = (
            observed_residual_mean
            -
            prior_mean
        )


        innovation_variance = (
            prior_variance
            +
            observation_variance
        )


        if (
            not np.isfinite(
                innovation_variance
            )
            or
            innovation_variance
            <=
            0.0
        ):

            raise RuntimeError(
                "Invalid rolling innovation variance."
            )


        standardized_innovation = (
            innovation
            /
            np.sqrt(
                innovation_variance
            )
        )


        # ------------------------------------------------------------------------------------------
        # 4 — UPDATE ONLY AFTER SCORING
        # ------------------------------------------------------------------------------------------

        kalman_gain = (
            prior_variance
            /
            innovation_variance
        )


        posterior_mean = (
            prior_mean
            +
            kalman_gain
            *
            innovation
        )


        posterior_variance = (
            (
                1.0
                -
                kalman_gain
            )
            *
            prior_variance
        )


        posterior_variance = max(
            float(
                posterior_variance
            ),
            1e-12,
        )


        # ------------------------------------------------------------------------------------------
        # 5 — RECORD
        # ------------------------------------------------------------------------------------------

        ROLLING_DATE_RESULTS.append(
            {
                "aaAthleteId":
                    athlete_id,

                "result_date_dt":
                    prediction_date,

                "prediction_split":
                    date_row[
                        "prediction_split"
                    ],

                "athlete_seen_in_training":
                    bool(
                        date_row[
                            "athlete_seen_in_training"
                        ]
                    ),

                "seeded_from_train":
                    seeded_from_train,

                "state_source_date":
                    source_date,

                "state_gap_days":
                    gap_days,

                "state_observation_count_before":
                    state_history_before,

                "state_history_available":
                    state_history_available,

                "ou_phi":
                    phi,

                "state_prior_mean":
                    prior_mean,

                "state_prior_variance":
                    prior_variance,

                "state_prior_sigma":
                    float(
                        np.sqrt(
                            prior_variance
                        )
                    ),

                "n_results_on_date":
                    int(
                        n_results
                    ),

                "n_M700":
                    int(
                        date_row[
                            "n_M700"
                        ]
                    ),

                "n_M800":
                    int(
                        date_row[
                            "n_M800"
                        ]
                    ),

                "n_W500":
                    int(
                        date_row[
                            "n_W500"
                        ]
                    ),

                "n_W600":
                    int(
                        date_row[
                            "n_W600"
                        ]
                    ),

                "state_observation_variance":
                    observation_variance,

                "observed_residual_mean":
                    observed_residual_mean,

                "state_innovation":
                    innovation,

                "state_innovation_variance":
                    innovation_variance,

                "state_standardized_innovation":
                    standardized_innovation,

                "kalman_gain":
                    kalman_gain,

                "state_posterior_mean":
                    posterior_mean,

                "state_posterior_variance":
                    posterior_variance,

                "state_posterior_sigma":
                    float(
                        np.sqrt(
                            posterior_variance
                        )
                    ),
            }
        )


        # ------------------------------------------------------------------------------------------
        # 6 — ADVANCE STATE
        # ------------------------------------------------------------------------------------------

        state_mean = (
            posterior_mean
        )


        state_variance = (
            posterior_variance
        )


        state_date = (
            prediction_date
        )


        state_observation_count += 1


# ==================================================================================================
# PART J — ROLLING DATE TABLE
# ==================================================================================================

V5_ROLLING_DATE_STATES = pd.DataFrame(
    ROLLING_DATE_RESULTS
)


V5_ROLLING_DATE_STATES = (
    V5_ROLLING_DATE_STATES.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART K — MERGE PRIOR STATE BACK TO PERFORMANCE ROWS
# ==================================================================================================

ROLLING_MERGE_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",

    "seeded_from_train",

    "state_source_date",
    "state_gap_days",

    "state_observation_count_before",
    "state_history_available",

    "ou_phi",

    "state_prior_mean",
    "state_prior_variance",
    "state_prior_sigma",

    "state_observation_variance",

    "state_innovation",
    "state_innovation_variance",
    "state_standardized_innovation",

    "kalman_gain",

    "state_posterior_mean",
    "state_posterior_variance",
    "state_posterior_sigma",
]


V5_ROLLING_PREDICTIONS = (
    EVAL_ROWS.merge(
        V5_ROLLING_DATE_STATES[
            ROLLING_MERGE_COLUMNS
        ],

        on=[
            "aaAthleteId",
            "result_date_dt",
        ],

        how="left",

        validate="many_to_one",
    )
)


# ==================================================================================================
# PART L — V5 PERFORMANCE-LEVEL PREDICTIONS
# ==================================================================================================

V5_ROLLING_PREDICTIONS[
    "v5_state_correction"
] = (
    V5_ROLLING_PREDICTIONS[
        "state_prior_mean"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_prediction_mean"
] = (
    V5_ROLLING_PREDICTIONS[
        "v4_structural_mean"
    ]

    +

    V5_ROLLING_PREDICTIONS[
        "v5_state_correction"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_residual"
] = (
    V5_ROLLING_PREDICTIONS[
        "mark_numeric_m"
    ]

    -

    V5_ROLLING_PREDICTIONS[
        "v5_prediction_mean"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_abs_error"
] = np.abs(
    V5_ROLLING_PREDICTIONS[
        "v5_residual"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_squared_error"
] = (
    V5_ROLLING_PREDICTIONS[
        "v5_residual"
    ]
    **
    2
)


# ==================================================================================================
# PART M — PRELIMINARY V5 PREDICTION VARIANCE
# ==================================================================================================
#
# This is NOT yet the final V6 uncertainty calibration.
#
# It is simply:
#
#       latent state uncertainty
#       +
#       fitted V5 group measurement variance
#
# ==================================================================================================

V5_ROLLING_PREDICTIONS[
    "v5_prediction_variance_raw"
] = (
    V5_ROLLING_PREDICTIONS[
        "state_prior_variance"
    ]

    +

    V5_ROLLING_PREDICTIONS[
        "v5_measurement_variance"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_prediction_sigma_raw"
] = np.sqrt(
    V5_ROLLING_PREDICTIONS[
        "v5_prediction_variance_raw"
    ]
)


V5_ROLLING_PREDICTIONS[
    "v5_standardized_error_raw"
] = (
    V5_ROLLING_PREDICTIONS[
        "v5_residual"
    ]

    /

    V5_ROLLING_PREDICTIONS[
        "v5_prediction_sigma_raw"
    ]
)


# ==================================================================================================
# PART N — NO-LEAKAGE QA
# ==================================================================================================

source_date_available = (
    V5_ROLLING_PREDICTIONS[
        "state_source_date"
    ]
    .notna()
)


N_SOURCE_DATE_LEAKAGE = int(
    (
        V5_ROLLING_PREDICTIONS.loc[
            source_date_available,
            "state_source_date",
        ]
        >=
        V5_ROLLING_PREDICTIONS.loc[
            source_date_available,
            "result_date_dt",
        ]
    )
    .sum()
)


N_NONPOSITIVE_ROLLING_GAPS = int(
    (
        V5_ROLLING_PREDICTIONS.loc[
            source_date_available,
            "state_gap_days",
        ]
        <=
        0
    )
    .sum()
)


# ==================================================================================================
# PART O — SAME-DAY FORECAST CONSISTENCY
# ==================================================================================================

SAME_DAY_STATE_VARIANTS = (
    V5_ROLLING_PREDICTIONS.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )[
        [
            "state_prior_mean",
            "state_prior_variance",
            "state_observation_count_before",
        ]
    ]
    .nunique(
        dropna=False
    )
)


N_SAME_DAY_STATE_CONFLICTS = int(
    (
        SAME_DAY_STATE_VARIANTS
        >
        1
    )
    .any(
        axis=1
    )
    .sum()
)


# ==================================================================================================
# PART P — FIRST UNSEEN FORECAST QA
# ==================================================================================================

FIRST_UNSEEN_DATE_MASK = (
    ~V5_ROLLING_DATE_STATES[
        "seeded_from_train"
    ]

    &

    V5_ROLLING_DATE_STATES[
        "state_observation_count_before"
    ]
    .eq(
        0
    )
)


N_FIRST_UNSEEN_DATES = int(
    FIRST_UNSEEN_DATE_MASK.sum()
)


N_FIRST_UNSEEN_MEAN_ERRORS = int(
    (
        np.abs(
            V5_ROLLING_DATE_STATES.loc[
                FIRST_UNSEEN_DATE_MASK,
                "state_prior_mean",
            ]
        )
        >
        1e-12
    )
    .sum()
)


N_FIRST_UNSEEN_VARIANCE_ERRORS = int(
    (
        np.abs(
            V5_ROLLING_DATE_STATES.loc[
                FIRST_UNSEEN_DATE_MASK,
                "state_prior_variance",
            ]

            -

            V5_STATE_VARIANCE
        )
        >
        1e-10
    )
    .sum()
)


# ==================================================================================================
# PART Q — GENERAL NUMERICAL QA
# ==================================================================================================

ROLLING_NUMERIC_COLUMNS = [
    "state_prior_mean",
    "state_prior_variance",
    "state_posterior_mean",
    "state_posterior_variance",
    "v5_prediction_mean",
    "v5_residual",
    "v5_prediction_variance_raw",
]


N_ROLLING_NONFINITE = int(
    (
        ~np.isfinite(
            V5_ROLLING_PREDICTIONS[
                ROLLING_NUMERIC_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


ROW_COUNT_PASS = bool(
    len(
        V5_ROLLING_PREDICTIONS
    )
    ==
    N_EVAL_ROWS
)


DATE_COUNT_PASS = bool(
    len(
        V5_ROLLING_DATE_STATES
    )
    ==
    N_EVAL_DATES
)


# ==================================================================================================
# PART R — METRIC HELPER
# ==================================================================================================

def compare_v4_v5_metrics(
    df,
):

    n = int(
        len(
            df
        )
    )


    if n == 0:

        return pd.Series(
            {
                "n":
                    0,

                "athletes":
                    0,

                "V4_MAE":
                    np.nan,

                "V5_MAE":
                    np.nan,

                "V5_MAE_gain_m":
                    np.nan,

                "V4_RMSE":
                    np.nan,

                "V5_RMSE":
                    np.nan,

                "V5_RMSE_gain_m":
                    np.nan,

                "V4_Bias":
                    np.nan,

                "V5_Bias":
                    np.nan,
            }
        )


    v4_error = (
        df[
            "v4_structural_residual"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v5_error = (
        df[
            "v5_residual"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v4_mae = float(
        np.mean(
            np.abs(
                v4_error
            )
        )
    )


    v5_mae = float(
        np.mean(
            np.abs(
                v5_error
            )
        )
    )


    v4_rmse = float(
        np.sqrt(
            np.mean(
                v4_error
                **
                2
            )
        )
    )


    v5_rmse = float(
        np.sqrt(
            np.mean(
                v5_error
                **
                2
            )
        )
    )


    return pd.Series(
        {
            "n":
                n,

            "athletes":
                int(
                    df[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4_mae,

            "V5_MAE":
                v5_mae,

            "V5_MAE_gain_m":
                (
                    v4_mae
                    -
                    v5_mae
                ),

            "V4_RMSE":
                v4_rmse,

            "V5_RMSE":
                v5_rmse,

            "V5_RMSE_gain_m":
                (
                    v4_rmse
                    -
                    v5_rmse
                ),

            "V4_Bias":
                float(
                    np.mean(
                        v4_error
                    )
                ),

            "V5_Bias":
                float(
                    np.mean(
                        v5_error
                    )
                ),
        }
    )


# ==================================================================================================
# PART S — OVERALL SPLIT COMPARISON
# ==================================================================================================

V5_METRICS_BY_SPLIT = (
    V5_ROLLING_PREDICTIONS.groupby(
        "prediction_split",
        sort=False,
    )
    .apply(
        compare_v4_v5_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART T — SEEN VS UNSEEN
# ==================================================================================================

V5_METRICS_SEEN_UNSEEN = (
    V5_ROLLING_PREDICTIONS.groupby(
        [
            "prediction_split",
            "athlete_training_status",
        ],
        sort=False,
    )
    .apply(
        compare_v4_v5_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART U — MODEL GROUP COMPARISON
# ==================================================================================================

V5_METRICS_BY_GROUP = (
    V5_ROLLING_PREDICTIONS.groupby(
        [
            "prediction_split",
            "model_group",
        ],
        sort=False,
    )
    .apply(
        compare_v4_v5_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART V — HISTORY AVAILABLE VS COLD STATE
# ==================================================================================================

V5_ROLLING_PREDICTIONS[
    "v5_history_status"
] = np.where(
    V5_ROLLING_PREDICTIONS[
        "state_history_available"
    ],
    "STATE_HISTORY",
    "NO_STATE_HISTORY",
)


V5_METRICS_BY_HISTORY = (
    V5_ROLLING_PREDICTIONS.groupby(
        [
            "prediction_split",
            "v5_history_status",
        ],
        sort=False,
    )
    .apply(
        compare_v4_v5_metrics,
        include_groups=False,
    )
    .reset_index()
)


# ==================================================================================================
# PART W — ESTABLISHED ATHLETE TEST RESULT
# ==================================================================================================
#
# This is particularly important for the project objective.
#
# ==================================================================================================

TEST_SEEN = (
    V5_ROLLING_PREDICTIONS.loc[
        V5_ROLLING_PREDICTIONS[
            "prediction_split"
        ]
        .eq(
            "TEST"
        )

        &

        V5_ROLLING_PREDICTIONS[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


TEST_SEEN_METRICS = (
    compare_v4_v5_metrics(
        TEST_SEEN
    )
)


# ==================================================================================================
# PART X — RAW UNCERTAINTY DIAGNOSTICS
# ==================================================================================================
#
# Diagnostic only.
#
# V6 will calibrate uncertainty later.
#
# ==================================================================================================

RAW_UNCERTAINTY_SUMMARY = (
    V5_ROLLING_PREDICTIONS.groupby(
        "prediction_split",
        as_index=False,
    )
    .agg(
        n=(
            "aaAthleteId",
            "size",
        ),

        standardized_error_mean=(
            "v5_standardized_error_raw",
            "mean",
        ),

        standardized_error_sd=(
            "v5_standardized_error_raw",
            "std",
        ),

        raw_sigma_mean=(
            "v5_prediction_sigma_raw",
            "mean",
        ),
    )
)


# ==================================================================================================
# PART Y — CELL QA
# ==================================================================================================

CELL_18_PASS = bool(

    N_DATE_SPLIT_CONFLICTS == 0

    and

    N_DUPLICATE_EVAL_DATES == 0

    and

    DATE_ROW_ARITHMETIC_PASS

    and

    ROW_COUNT_PASS

    and

    DATE_COUNT_PASS

    and

    N_SOURCE_DATE_LEAKAGE == 0

    and

    N_NONPOSITIVE_ROLLING_GAPS == 0

    and

    N_SAME_DAY_STATE_CONFLICTS == 0

    and

    N_FIRST_UNSEEN_MEAN_ERRORS == 0

    and

    N_FIRST_UNSEEN_VARIANCE_ERRORS == 0

    and

    N_ROLLING_NONFINITE == 0
)


# ==================================================================================================
# PART Z — SAVE ARTIFACTS
# ==================================================================================================

V5_ROLLING_PREDICTIONS_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_rolling_predictions.parquet"
)


V5_ROLLING_DATE_STATES_PATH = (
    TRAINING_RUN_DIR
    /
    "v5_rolling_date_states.parquet"
)


V5_METRICS_SPLIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_metrics_by_split.parquet"
)


V5_METRICS_SEEN_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_metrics_seen_unseen.parquet"
)


V5_METRICS_GROUP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_metrics_by_group.parquet"
)


V5_METRICS_HISTORY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_metrics_by_state_history.parquet"
)


V5_RAW_UNCERTAINTY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v5_raw_uncertainty_summary.parquet"
)


V5_ROLLING_PREDICTIONS.to_parquet(
    V5_ROLLING_PREDICTIONS_PATH,
    index=False,
)


V5_ROLLING_DATE_STATES.to_parquet(
    V5_ROLLING_DATE_STATES_PATH,
    index=False,
)


V5_METRICS_BY_SPLIT.to_parquet(
    V5_METRICS_SPLIT_PATH,
    index=False,
)


V5_METRICS_SEEN_UNSEEN.to_parquet(
    V5_METRICS_SEEN_PATH,
    index=False,
)


V5_METRICS_BY_GROUP.to_parquet(
    V5_METRICS_GROUP_PATH,
    index=False,
)


V5_METRICS_BY_HISTORY.to_parquet(
    V5_METRICS_HISTORY_PATH,
    index=False,
)


RAW_UNCERTAINTY_SUMMARY.to_parquet(
    V5_RAW_UNCERTAINTY_PATH,
    index=False,
)


# ==================================================================================================
# PART AA — UPDATE TRAINING MANIFEST
# ==================================================================================================

TRAINING_MANIFEST[
    "v5_rolling_evaluation"
] = {

    "status":
        (
            "PASSED"
            if CELL_18_PASS
            else "FAILED"
        ),

    "evaluation_rows":
        N_EVAL_ROWS,

    "evaluation_athletes":
        N_EVAL_ATHLETES,

    "evaluation_dates":
        N_EVAL_DATES,

    "first_unseen_dates":
        N_FIRST_UNSEEN_DATES,

    "source_date_leakage":
        N_SOURCE_DATE_LEAKAGE,

    "same_day_state_conflicts":
        N_SAME_DAY_STATE_CONFLICTS,

    "test_seen": {
        "n":
            int(
                TEST_SEEN_METRICS[
                    "n"
                ]
            ),

        "v4_mae":
            float(
                TEST_SEEN_METRICS[
                    "V4_MAE"
                ]
            ),

        "v5_mae":
            float(
                TEST_SEEN_METRICS[
                    "V5_MAE"
                ]
            ),

        "mae_gain_m":
            float(
                TEST_SEEN_METRICS[
                    "V5_MAE_gain_m"
                ]
            ),

        "v4_rmse":
            float(
                TEST_SEEN_METRICS[
                    "V4_RMSE"
                ]
            ),

        "v5_rmse":
            float(
                TEST_SEEN_METRICS[
                    "V5_RMSE"
                ]
            ),

        "rmse_gain_m":
            float(
                TEST_SEEN_METRICS[
                    "V5_RMSE_gain_m"
                ]
            ),
    },

    "rolling_predictions_path":
        str(
            V5_ROLLING_PREDICTIONS_PATH
        ),

    "rolling_date_states_path":
        str(
            V5_ROLLING_DATE_STATES_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V5_ROLLING_EVALUATION_READY"
    if CELL_18_PASS
    else "V5_ROLLING_EVALUATION_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART AB — NOTEBOOK GLOBALS
# ==================================================================================================

NOTEBOOK_2_CELL_18_PASS = (
    CELL_18_PASS
)


# ==================================================================================================
# PART AC — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("V6 TRAINING — TRUE ROLLING-ORIGIN V5 EVALUATION")
print("=" * 145)


print()
print("EVALUATION POPULATION")
print("-" * 145)

print(
    f"Performance rows                 : "
    f"{N_EVAL_ROWS:,}"
)

print(
    f"Athletes                         : "
    f"{N_EVAL_ATHLETES:,}"
)

print(
    f"Distinct athlete-dates           : "
    f"{N_EVAL_DATES:,}"
)

print(
    f"First dates for non-TRAIN athletes: "
    f"{N_FIRST_UNSEEN_DATES:,}"
)


print()
print("V4 → V5 BY SPLIT")
print("-" * 145)

display(
    V5_METRICS_BY_SPLIT
)


print()
print("V4 → V5: SEEN VS UNSEEN")
print("-" * 145)

display(
    V5_METRICS_SEEN_UNSEEN
)


print()
print("V4 → V5 BY MODEL GROUP")
print("-" * 145)

display(
    V5_METRICS_BY_GROUP
)


print()
print("V4 → V5 BY STATE HISTORY")
print("-" * 145)

display(
    V5_METRICS_BY_HISTORY
)


print()
print("ESTABLISHED ATHLETE TEST RESULT")
print("-" * 145)

print(
    f"Rows                             : "
    f"{int(TEST_SEEN_METRICS['n']):,}"
)

print(
    f"V4 MAE                           : "
    f"{TEST_SEEN_METRICS['V4_MAE']:.6f}"
)

print(
    f"V5 MAE                           : "
    f"{TEST_SEEN_METRICS['V5_MAE']:.6f}"
)

print(
    f"MAE gain                         : "
    f"{TEST_SEEN_METRICS['V5_MAE_gain_m']:+.6f} m"
)

print(
    f"V4 RMSE                          : "
    f"{TEST_SEEN_METRICS['V4_RMSE']:.6f}"
)

print(
    f"V5 RMSE                          : "
    f"{TEST_SEEN_METRICS['V5_RMSE']:.6f}"
)

print(
    f"RMSE gain                        : "
    f"{TEST_SEEN_METRICS['V5_RMSE_gain_m']:+.6f} m"
)


print()
print("RAW UNCERTAINTY DIAGNOSTIC")
print("-" * 145)

display(
    RAW_UNCERTAINTY_SUMMARY
)


print()
print("NO-LEAKAGE QA")
print("-" * 145)

print(
    f"Date split conflicts             : "
    f"{N_DATE_SPLIT_CONFLICTS:,}"
)

print(
    f"Duplicate evaluation dates       : "
    f"{N_DUPLICATE_EVAL_DATES:,}"
)

print(
    f"Date row arithmetic              : "
    f"{DATE_ROW_ARITHMETIC_PASS}"
)

print(
    f"Prediction row count             : "
    f"{ROW_COUNT_PASS}"
)

print(
    f"Prediction date count            : "
    f"{DATE_COUNT_PASS}"
)

print(
    f"Source-date leakage              : "
    f"{N_SOURCE_DATE_LEAKAGE:,}"
)

print(
    f"Nonpositive rolling gaps         : "
    f"{N_NONPOSITIVE_ROLLING_GAPS:,}"
)

print(
    f"Same-day state conflicts         : "
    f"{N_SAME_DAY_STATE_CONFLICTS:,}"
)

print(
    f"First-unseen mean errors         : "
    f"{N_FIRST_UNSEEN_MEAN_ERRORS:,}"
)

print(
    f"First-unseen variance errors     : "
    f"{N_FIRST_UNSEEN_VARIANCE_ERRORS:,}"
)

print(
    f"Nonfinite rolling values         : "
    f"{N_ROLLING_NONFINITE:,}"
)


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Rolling V5 predictions           : "
    f"{V5_ROLLING_PREDICTIONS_PATH}"
)

print(
    f"Rolling date states              : "
    f"{V5_ROLLING_DATE_STATES_PATH}"
)

print(
    f"Seen/unseen metrics              : "
    f"{V5_METRICS_SEEN_PATH}"
)


print()
print("=" * 145)


if CELL_18_PASS:

    print(
        "CELL 18 STATUS: TRUE ROLLING-ORIGIN V5 EVALUATION PASSED"
    )

else:

    print(
        "CELL 18 STATUS: ROLLING-ORIGIN V5 REVIEW REQUIRED"
    )


print("=" * 145)



V6 TRAINING — V5 FILTERED TRAIN STATE RECOVERY

STATE SEQUENCE
-------------------------------------------------------------------------------------------------------------------------------------------------
Filtered state rows              : 169,641
Athletes                         : 12,945
Terminal athlete states          : 12,945

LIKELIHOOD RECONSTRUCTION
-------------------------------------------------------------------------------------------------------------------------------------------------
Cell 16 optimizer NLL            : 422,410.013236531
Filter replay NLL                : 422,410.013236534
Absolute difference              : 0.000000002445
Relative difference              : 5.788e-15
NLL reconstruction pass          : True

FILTER QA
-------------------------------------------------------------------------------------------------------------------------------------------------
First-state prior-mean errors    : 0
First-state prior-var errors     : 0
First-state phi er

,metric,value
0,posterior_state_mean_mean,-0.016800
1,posterior_state_mean_sd,1.111267
2,posterior_state_mean_q05,-1.816871
3,posterior_state_mean_q50,0.008414
4,posterior_state_mean_q95,1.718366
5,posterior_state_sigma_mean,1.275400
6,kalman_gain_mean,0.267889



INNOVATION DIAGNOSTICS
-------------------------------------------------------------------------------------------------------------------------------------------------


,metric,value
0,innovation_mean,-0.110829
1,innovation_sd,2.969044
2,standardized_innovation_mean,-0.035308
3,standardized_innovation_sd,0.999376
4,abs_std_innovation_gt_2_pct,4.297900
5,abs_std_innovation_gt_3_pct,0.773987



TERMINAL TRAIN STATES
-------------------------------------------------------------------------------------------------------------------------------------------------


...


V4 → V5 BY STATE HISTORY
-------------------------------------------------------------------------------------------------------------------------------------------------


,prediction_split,v5_history_status,n,athletes,V4_MAE,V5_MAE,V5_MAE_gain_m,V4_RMSE,V5_RMSE,V5_RMSE_gain_m,V4_Bias,V5_Bias
0,TEST,STATE_HISTORY,11918.0,4563.0,3.299850,2.801537,0.498312,4.536743,3.827769,0.708973,0.441244,0.215800
1,VALIDATION,STATE_HISTORY,4487.0,2443.0,3.009784,2.739015,0.270770,4.035683,3.628277,0.407406,0.266798,0.189799
2,POST_TEST,STATE_HISTORY,532.0,475.0,4.034962,3.409295,0.625666,5.638612,4.820643,0.817969,0.420011,0.072257
3,TEST,NO_STATE_HISTORY,947.0,934.0,9.220884,9.220884,0.000000,11.236639,11.236639,0.000000,-4.193439,-4.193439
4,VALIDATION,NO_STATE_HISTORY,500.0,499.0,9.288329,9.288329,0.000000,11.325081,11.325081,0.000000,-0.101222,-0.101222
5,POST_TEST,NO_STATE_HISTORY,100.0,100.0,8.083719,8.083719,0.000000,10.035780,10.035780,0.000000,0.483113,0.483113



ESTABLISHED ATHLETE TEST RESULT
-------------------------------------------------------------------------------------------------------------------------------------------------
Rows                             : 10,941
V4 MAE                           : 2.914155
V5 MAE                           : 2.550114
MAE gain                         : +0.364041 m
V4 RMSE                          : 3.802461
V5 RMSE                          : 3.371135
RMSE gain                        : +0.431326 m

RAW UNCERTAINTY DIAGNOSTIC
-------------------------------------------------------------------------------------------------------------------------------------------------


,prediction_split,n,standardized_error_mean,standardized_error_sd,raw_sigma_mean
0,POST_TEST,632,-0.018678,1.908403,2.971759
1,TEST,12865,-0.052926,1.580556,2.947553
2,VALIDATION,4987,0.029082,1.623159,2.990177



NO-LEAKAGE QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Date split conflicts             : 0
Duplicate evaluation dates       : 0
Date row arithmetic              : True
Prediction row count             : True
Prediction date count            : True
Source-date leakage              : 0
Nonpositive rolling gaps         : 0
Same-day state conflicts         : 0
First-unseen mean errors         : 0
First-unseen variance errors     : 0
Nonfinite rolling values         : 0

OUTPUT
-------------------------------------------------------------------------------------------------------------------------------------------------
Rolling V5 predictions           : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/v5_rolling_predictions.parquet
Rolling date states              : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_

## Stage 07 — Current V6 mean and uncertainty calibration

Calibrate the current V6 structural mean and its initial uncertainty law using VALIDATION only.

**Legacy source cells represented:** 9, 10  
**Execution rule:** run only after the preceding stage has completed successfully.


In [8]:
# ==================================================================================================
# REFACTORED STAGE 07 — Current V6 mean and uncertainty calibration
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 9
# Original source SHA256: 70eaf1d96710d96af44dde5ae89db90bc5d8e8d72093af186727760e2ece0213
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 19 — V6 MEAN CALIBRATION — VALIDATION ONLY
# ==================================================================================================
#
# 1. First VALIDATION date for each TRAIN-seen athlete
# 2. Estimate empirical intercept/slope persistence by horizon band
# 3. Compare constant/decay persistence laws by AICc
# 4. Compare NONE / COMMON / SEX / GROUP offsets by AICc
# 5. Freeze v6_mean_spec.json
#
# TEST is not used.
# ==================================================================================================

import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from scipy.optimize import minimize


# --------------------------------------------------------------------------------------------------
# Prerequisites
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_18_PASS",
    "V5_ROLLING_PREDICTIONS",
    "V5_MEASUREMENT_VARIANCES",
    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n" +
        "\n".join(f"  - {x}" for x in missing)
    )

if not NOTEBOOK_2_CELL_18_PASS:
    raise RuntimeError("Cell 18 did not pass.")


# --------------------------------------------------------------------------------------------------
# Validation calibration population
# --------------------------------------------------------------------------------------------------

VAL = (
    V5_ROLLING_PREDICTIONS.loc[
        V5_ROLLING_PREDICTIONS["prediction_split"].eq("VALIDATION")
    ]
    .copy()
)

VAL["result_date_dt"] = pd.to_datetime(VAL["result_date_dt"])
VAL["state_source_date"] = pd.to_datetime(VAL["state_source_date"])


VAL_DATE = (
    VAL.groupby(
        ["aaAthleteId", "result_date_dt"],
        as_index=False,
    )
    .agg(
        athlete_seen_in_training=("athlete_seen_in_training", "first"),
        sex_group=("sex_group", "first"),
        model_group=("model_group", "first"),
        n_model_groups=("model_group", "nunique"),
        n_results_on_date=("mark_numeric_m", "size"),
        actual_mark_mean=("mark_numeric_m", "mean"),

        v4_structural_mean=("v4_structural_mean", "mean"),
        athlete_intercept_component=("v4_athlete_intercept_component", "mean"),
        athlete_slope_component=("v4_athlete_slope_component", "mean"),

        state_prior_mean=("state_prior_mean", "first"),
        state_prior_variance=("state_prior_variance", "first"),
        state_source_date=("state_source_date", "first"),
        state_gap_days=("state_gap_days", "first"),
    )
    .sort_values(
        ["aaAthleteId", "result_date_dt"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


VAL_DATE["validation_date_index"] = (
    VAL_DATE.groupby("aaAthleteId", sort=False)
    .cumcount()
)


CAL = (
    VAL_DATE.loc[
        VAL_DATE["validation_date_index"].eq(0)
        &
        VAL_DATE["athlete_seen_in_training"]
        &
        VAL_DATE["state_source_date"].notna()
        &
        VAL_DATE["state_gap_days"].between(1, 365, inclusive="both")
        &
        VAL_DATE["n_model_groups"].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)


if len(CAL) != CAL["aaAthleteId"].nunique():
    raise RuntimeError("Expected one calibration row per athlete.")


# --------------------------------------------------------------------------------------------------
# Structural decomposition
# --------------------------------------------------------------------------------------------------

CAL["population_component"] = (
    CAL["v4_structural_mean"]
    - CAL["athlete_intercept_component"]
    - CAL["athlete_slope_component"]
)


CAL["actual_structural_component"] = (
    CAL["actual_mark_mean"]
    - CAL["population_component"]
    - CAL["state_prior_mean"]
)


reconstructed = (
    CAL["population_component"]
    + CAL["athlete_intercept_component"]
    + CAL["athlete_slope_component"]
    + CAL["state_prior_mean"]
)

reference = (
    CAL["v4_structural_mean"]
    + CAL["state_prior_mean"]
)

MAX_RECONSTRUCTION_ERROR = float(
    np.abs(reconstructed - reference).max()
)

RECONSTRUCTION_PASS = MAX_RECONSTRUCTION_ERROR < 1e-10


# --------------------------------------------------------------------------------------------------
# Horizon bands + component centering
# --------------------------------------------------------------------------------------------------

CAL["gap_band"] = pd.cut(
    CAL["state_gap_days"],
    bins=[0, 30, 60, 90, 120, 180, 365],
    labels=[
        "01-30",
        "31-60",
        "61-90",
        "91-120",
        "121-180",
        "181-365",
    ],
    include_lowest=True,
)


CENTERS = (
    CAL.groupby(
        ["gap_band", "sex_group"],
        observed=True,
        as_index=False,
    )
    .agg(
        intercept_center=("athlete_intercept_component", "mean"),
        slope_center=("athlete_slope_component", "mean"),
    )
)


CAL = CAL.merge(
    CENTERS,
    on=["gap_band", "sex_group"],
    how="left",
    validate="many_to_one",
)


CAL["intercept_deviation"] = (
    CAL["athlete_intercept_component"]
    - CAL["intercept_center"]
)

CAL["slope_deviation"] = (
    CAL["athlete_slope_component"]
    - CAL["slope_center"]
)


# --------------------------------------------------------------------------------------------------
# Empirical persistence by horizon band
# --------------------------------------------------------------------------------------------------

MIN_BAND_ROWS = 30
empirical_rows = []


for band, d in CAL.groupby(
    "gap_band",
    observed=True,
    sort=True,
):

    if len(d) < MIN_BAND_ROWS:

        empirical_rows.append({
            "gap_band": str(band),
            "n": len(d),
            "median_gap_days": (
                float(d["state_gap_days"].median())
                if len(d)
                else np.nan
            ),
            "rho_intercept": np.nan,
            "intercept_se": np.nan,
            "rho_slope": np.nan,
            "slope_se": np.nan,
            "status": "INSUFFICIENT_ROWS",
        })

        continue


    fit = (
        smf.ols(
            """
            actual_structural_component
            ~ intercept_deviation
            + slope_deviation
            + C(model_group)
            """,
            data=d,
        )
        .fit(cov_type="HC3")
    )


    empirical_rows.append({
        "gap_band": str(band),
        "n": len(d),
        "median_gap_days": float(d["state_gap_days"].median()),

        "rho_intercept":
            float(fit.params["intercept_deviation"]),

        "intercept_se":
            float(fit.bse["intercept_deviation"]),

        "rho_slope":
            float(fit.params["slope_deviation"]),

        "slope_se":
            float(fit.bse["slope_deviation"]),

        "status":
            "FITTED",
    })


V6_EMPIRICAL_PERSISTENCE = pd.DataFrame(empirical_rows)

FIT_BANDS = (
    V6_EMPIRICAL_PERSISTENCE.loc[
        V6_EMPIRICAL_PERSISTENCE["status"].eq("FITTED")
    ]
    .reset_index(drop=True)
)

if len(FIT_BANDS) < 4:
    raise RuntimeError("Too few persistence bands.")


H = FIT_BANDS["median_gap_days"].to_numpy(float)

OBS_I = FIT_BANDS["rho_intercept"].to_numpy(float)
OBS_S = FIT_BANDS["rho_slope"].to_numpy(float)

SE_I = FIT_BANDS["intercept_se"].to_numpy(float)
SE_S = FIT_BANDS["slope_se"].to_numpy(float)


# --------------------------------------------------------------------------------------------------
# Persistence laws
# --------------------------------------------------------------------------------------------------

def persistence_values(model, p, h):

    h = np.asarray(h, dtype=float)

    if model == "M0_CONST_CONST":
        return (
            np.full(len(h), p[0]),
            np.full(len(h), p[1]),
        )

    if model == "M1_CONST_EXP":
        return (
            np.full(len(h), p[0]),
            p[1] * np.exp(-h / p[2]),
        )

    if model == "M2_EXP_CONST":
        return (
            p[0] * np.exp(-h / p[1]),
            np.full(len(h), p[2]),
        )

    if model == "M3_EXP_EXP":
        return (
            p[0] * np.exp(-h / p[1]),
            p[2] * np.exp(-h / p[3]),
        )

    raise ValueError(model)


def gaussian_nll(obs, pred, se):

    z = (obs - pred) / se

    return float(
        0.5 * np.sum(
            np.log(2 * np.pi)
            + 2 * np.log(se)
            + z**2
        )
    )


PERSISTENCE_MODELS = {
    "M0_CONST_CONST": {
        "k": 2,
        "bounds": [(0, 1), (0, 1)],
        "starts": [
            [0.95, 0.95],
            [0.90, 0.90],
            [0.99, 0.95],
        ],
    },

    "M1_CONST_EXP": {
        "k": 3,
        "bounds": [(0, 1), (0, 1), (30, 20000)],
        "starts": [
            [0.95, 0.95, 500],
            [0.95, 0.95, 3000],
            [0.95, 0.95, 10000],
        ],
    },

    "M2_EXP_CONST": {
        "k": 3,
        "bounds": [(0, 1), (30, 20000), (0, 1)],
        "starts": [
            [0.95, 500, 0.95],
            [0.95, 3000, 0.95],
            [0.95, 10000, 0.95],
        ],
    },

    "M3_EXP_EXP": {
        "k": 4,
        "bounds": [
            (0, 1),
            (30, 20000),
            (0, 1),
            (30, 20000),
        ],
        "starts": [
            [0.95, 500, 0.95, 500],
            [0.95, 3000, 0.95, 3000],
            [0.95, 10000, 0.95, 10000],
        ],
    },
}


def persistence_objective(p, model):

    pred_i, pred_s = persistence_values(
        model,
        p,
        H,
    )

    return (
        gaussian_nll(OBS_I, pred_i, SE_I)
        + gaussian_nll(OBS_S, pred_s, SE_S)
    )


persistence_rows = []
PERSISTENCE_FITS = {}

N_INFO = 2 * len(FIT_BANDS)


for model, spec in PERSISTENCE_MODELS.items():

    runs = [
        minimize(
            persistence_objective,
            x0=np.asarray(x0, float),
            args=(model,),
            method="L-BFGS-B",
            bounds=spec["bounds"],
            options={
                "maxiter": 5000,
                "ftol": 1e-12,
                "gtol": 1e-9,
            },
        )
        for x0 in spec["starts"]
    ]


    best = min(
        runs,
        key=lambda r: float(r.fun),
    )

    p = np.asarray(best.x, float)
    k = spec["k"]
    nll = float(best.fun)

    aic = 2 * k + 2 * nll

    aicc = (
        aic
        +
        2 * k * (k + 1)
        / (N_INFO - k - 1)
    )

    bic = np.log(N_INFO) * k + 2 * nll


    row = {
        "model": model,
        "success": bool(best.success),
        "k": k,
        "NLL": nll,
        "AIC": aic,
        "AICc": aicc,
        "BIC": bic,

        "c_intercept": np.nan,
        "tau_intercept_days": np.nan,

        "c_slope": np.nan,
        "tau_slope_days": np.nan,
    }


    if model == "M0_CONST_CONST":
        row["c_intercept"], row["c_slope"] = p

    elif model == "M1_CONST_EXP":
        (
            row["c_intercept"],
            row["c_slope"],
            row["tau_slope_days"],
        ) = p

    elif model == "M2_EXP_CONST":
        (
            row["c_intercept"],
            row["tau_intercept_days"],
            row["c_slope"],
        ) = p

    else:
        (
            row["c_intercept"],
            row["tau_intercept_days"],
            row["c_slope"],
            row["tau_slope_days"],
        ) = p


    persistence_rows.append(row)

    PERSISTENCE_FITS[model] = {
        "result": best,
        "params": p,
    }


V6_PERSISTENCE_MODEL_COMPARISON = pd.DataFrame(
    persistence_rows
)

V6_PERSISTENCE_MODEL_COMPARISON["delta_AICc"] = (
    V6_PERSISTENCE_MODEL_COMPARISON["AICc"]
    - V6_PERSISTENCE_MODEL_COMPARISON["AICc"].min()
)

V6_PERSISTENCE_MODEL_COMPARISON = (
    V6_PERSISTENCE_MODEL_COMPARISON.sort_values(
        ["AICc", "BIC"]
    )
    .reset_index(drop=True)
)


V6_SELECTED_PERSISTENCE_MODEL = str(
    V6_PERSISTENCE_MODEL_COMPARISON.iloc[0]["model"]
)

V6_SELECTED_PERSISTENCE_PARAMS = np.asarray(
    PERSISTENCE_FITS[
        V6_SELECTED_PERSISTENCE_MODEL
    ]["params"],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# Apply selected persistence to calibration rows
# --------------------------------------------------------------------------------------------------

rho_i, rho_s = persistence_values(
    V6_SELECTED_PERSISTENCE_MODEL,
    V6_SELECTED_PERSISTENCE_PARAMS,
    CAL["state_gap_days"].to_numpy(float),
)

CAL["v6_rho_intercept"] = rho_i
CAL["v6_rho_slope"] = rho_s

CAL["persistent_structural_component"] = (
    CAL["v6_rho_intercept"]
    * CAL["athlete_intercept_component"]
    +
    CAL["v6_rho_slope"]
    * CAL["athlete_slope_component"]
)


# --------------------------------------------------------------------------------------------------
# Offset calibration
# --------------------------------------------------------------------------------------------------

CAL["measurement_variance_group"] = (
    CAL["model_group"]
    .map(V5_MEASUREMENT_VARIANCES)
)

if CAL["measurement_variance_group"].isna().any():
    raise RuntimeError("Missing V5 measurement variance.")


CAL["measurement_variance_date_mean"] = (
    CAL["measurement_variance_group"]
    /
    CAL["n_results_on_date"].astype(float)
)

CAL["calibration_variance"] = (
    CAL["state_prior_variance"]
    +
    CAL["measurement_variance_date_mean"]
)

CAL["calibration_weight"] = (
    1.0
    /
    CAL["calibration_variance"]
)

CAL["offset_target"] = (
    CAL["actual_structural_component"]
    -
    CAL["persistent_structural_component"]
)


def weighted_mean(v, w):

    v = np.asarray(v, float)
    w = np.asarray(w, float)

    return float(
        np.sum(v * w)
        /
        np.sum(w)
    )


GLOBAL_OFFSET = weighted_mean(
    CAL["offset_target"],
    CAL["calibration_weight"],
)


SEX_OFFSETS = {
    str(sex):
        weighted_mean(
            d["offset_target"],
            d["calibration_weight"],
        )
    for sex, d in CAL.groupby("sex_group")
}


GROUP_OFFSETS = {
    str(group):
        weighted_mean(
            d["offset_target"],
            d["calibration_weight"],
        )
    for group, d in CAL.groupby("model_group")
}


OFFSET_MODELS = {
    "Z0_NONE": {
        "type": "NONE",
        "k": 0,
        "values": {},
    },

    "Z1_COMMON": {
        "type": "COMMON",
        "k": 1,
        "values": {"ALL": GLOBAL_OFFSET},
    },

    "Z2_SEX": {
        "type": "SEX",
        "k": len(SEX_OFFSETS),
        "values": SEX_OFFSETS,
    },

    "Z3_GROUP": {
        "type": "GROUP",
        "k": len(GROUP_OFFSETS),
        "values": GROUP_OFFSETS,
    },
}


def offset_vector(df, spec):

    if spec["type"] == "NONE":
        return np.zeros(len(df))

    if spec["type"] == "COMMON":
        return np.full(
            len(df),
            float(spec["values"]["ALL"]),
        )

    key = (
        "sex_group"
        if spec["type"] == "SEX"
        else
        "model_group"
    )

    values = (
        df[key]
        .astype(str)
        .map(spec["values"])
    )

    if values.isna().any():
        raise RuntimeError(
            f"Missing {spec['type']} offset."
        )

    return values.to_numpy(float)


LOG_2PI = np.log(2 * np.pi)

offset_rows = []

for model, spec in OFFSET_MODELS.items():

    offset = offset_vector(CAL, spec)

    error = (
        CAL["actual_structural_component"].to_numpy(float)
        -
        (
            CAL["persistent_structural_component"].to_numpy(float)
            + offset
        )
    )

    variance = CAL["calibration_variance"].to_numpy(float)

    nll = float(
        0.5 * np.sum(
            LOG_2PI
            + np.log(variance)
            + error**2 / variance
        )
    )

    k = spec["k"]
    n = len(CAL)

    aic = 2 * k + 2 * nll

    aicc = (
        aic
        +
        2 * k * (k + 1)
        / (n - k - 1)
    )

    bic = np.log(n) * k + 2 * nll

    offset_rows.append({
        "model": model,
        "offset_type": spec["type"],
        "k": k,
        "NLL": nll,
        "AIC": aic,
        "AICc": aicc,
        "BIC": bic,
    })


V6_OFFSET_MODEL_COMPARISON = pd.DataFrame(
    offset_rows
)

V6_OFFSET_MODEL_COMPARISON["delta_AICc"] = (
    V6_OFFSET_MODEL_COMPARISON["AICc"]
    - V6_OFFSET_MODEL_COMPARISON["AICc"].min()
)

V6_OFFSET_MODEL_COMPARISON = (
    V6_OFFSET_MODEL_COMPARISON.sort_values(
        ["AICc", "BIC"]
    )
    .reset_index(drop=True)
)


V6_SELECTED_OFFSET_MODEL = str(
    V6_OFFSET_MODEL_COMPARISON.iloc[0]["model"]
)

selected_offset_spec = OFFSET_MODELS[
    V6_SELECTED_OFFSET_MODEL
]

V6_OFFSET_STRUCTURE = selected_offset_spec["type"]

V6_OFFSET_LOOKUP = {
    str(k): float(v)
    for k, v in selected_offset_spec["values"].items()
}


# --------------------------------------------------------------------------------------------------
# Serialize selected persistence law
# --------------------------------------------------------------------------------------------------

selected_persistence_row = (
    V6_PERSISTENCE_MODEL_COMPARISON.iloc[0]
)


def maybe_float(x):
    return None if pd.isna(x) else float(x)


V6_PERSISTENCE_SPEC = {
    "model":
        V6_SELECTED_PERSISTENCE_MODEL,

    "c_intercept":
        maybe_float(
            selected_persistence_row["c_intercept"]
        ),

    "tau_intercept_days":
        maybe_float(
            selected_persistence_row["tau_intercept_days"]
        ),

    "c_slope":
        maybe_float(
            selected_persistence_row["c_slope"]
        ),

    "tau_slope_days":
        maybe_float(
            selected_persistence_row["tau_slope_days"]
        ),
}


V6_MEAN_SPEC = {
    "model":
        "V6_STRUCTURAL_MEAN",

    "calibration_split":
        "VALIDATION",

    "calibration_population":
        "FIRST_VALIDATION_DATE_PER_TRAIN_SEEN_ATHLETE",

    "persistence":
        V6_PERSISTENCE_SPEC,

    "offset_model":
        V6_SELECTED_OFFSET_MODEL,

    "offset_structure":
        V6_OFFSET_STRUCTURE,

    "offsets":
        V6_OFFSET_LOOKUP,

    "mean_formula":
        (
            "population_component "
            "+ rho_intercept(h)*athlete_intercept_component "
            "+ rho_slope(h)*athlete_slope_component "
            "+ offset "
            "+ V5_state_prior_mean"
        ),
}


# --------------------------------------------------------------------------------------------------
# QA + save
# --------------------------------------------------------------------------------------------------

CELL_19_PASS = bool(
    RECONSTRUCTION_PASS
    and len(FIT_BANDS) >= 4
    and V6_PERSISTENCE_MODEL_COMPARISON["success"].all()
    and np.isfinite(
        V6_PERSISTENCE_MODEL_COMPARISON[
            ["AIC", "AICc", "BIC"]
        ].to_numpy(float)
    ).all()
    and np.isfinite(
        V6_OFFSET_MODEL_COMPARISON[
            ["AIC", "AICc", "BIC"]
        ].to_numpy(float)
    ).all()
)


V6_MEAN_SPEC_PATH = (
    TRAINING_RUN_DIR
    / "v6_mean_spec.json"
)

V6_MEAN_CALIBRATION_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_mean_calibration.parquet"
)

V6_PERSISTENCE_COMPARISON_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_persistence_model_comparison.parquet"
)

V6_OFFSET_COMPARISON_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_offset_model_comparison.parquet"
)


with V6_MEAN_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        V6_MEAN_SPEC,
        f,
        indent=2,
        default=str,
    )


CAL.to_parquet(
    V6_MEAN_CALIBRATION_PATH,
    index=False,
)

V6_PERSISTENCE_MODEL_COMPARISON.to_parquet(
    V6_PERSISTENCE_COMPARISON_PATH,
    index=False,
)

V6_OFFSET_MODEL_COMPARISON.to_parquet(
    V6_OFFSET_COMPARISON_PATH,
    index=False,
)


TRAINING_MANIFEST["v6_mean_calibration"] = {
    "status":
        "PASSED" if CELL_19_PASS else "FAILED",

    "persistence":
        V6_PERSISTENCE_SPEC,

    "offset_model":
        V6_SELECTED_OFFSET_MODEL,

    "offset_structure":
        V6_OFFSET_STRUCTURE,

    "offsets":
        V6_OFFSET_LOOKUP,

    "mean_spec_path":
        str(V6_MEAN_SPEC_PATH),
}

TRAINING_MANIFEST["status"] = (
    "V6_MEAN_SPEC_READY"
    if CELL_19_PASS
    else "V6_MEAN_CALIBRATION_FAILED"
)

with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


V6_CALIBRATION = CAL
NOTEBOOK_2_CELL_19_PASS = CELL_19_PASS


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 130)
print("V6 TRAINING — MEAN CALIBRATION")
print("=" * 130)

print()
print("EMPIRICAL PERSISTENCE")
display(V6_EMPIRICAL_PERSISTENCE)

print()
print("PERSISTENCE MODEL COMPARISON")
display(
    V6_PERSISTENCE_MODEL_COMPARISON[
        [
            "model",
            "AICc",
            "delta_AICc",
            "BIC",
            "c_intercept",
            "tau_intercept_days",
            "c_slope",
            "tau_slope_days",
        ]
    ]
)

print()
print("OFFSET MODEL COMPARISON")
display(
    V6_OFFSET_MODEL_COMPARISON[
        [
            "model",
            "offset_type",
            "AICc",
            "delta_AICc",
            "BIC",
        ]
    ]
)

print()
print("FINAL MEAN SPEC")
print("Persistence :", V6_PERSISTENCE_SPEC)
print("Offset      :", V6_SELECTED_OFFSET_MODEL)
print("Values      :", V6_OFFSET_LOOKUP)

print()
print("=" * 130)
print(
    "CELL 19 STATUS: "
    + (
        "V6 MEAN CALIBRATION PASSED"
        if CELL_19_PASS
        else
        "V6 MEAN CALIBRATION REVIEW REQUIRED"
    )
)
print("=" * 130)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 10
# Original source SHA256: fe501c6e3ff19506151aee765db4d4f93307a1d66656e995f026336b41f50314
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 20 — V6 UNCERTAINTY CALIBRATION + ROBUSTNESS — VALIDATION ONLY
# ==================================================================================================
#
# 1. Apply frozen Cell-19 V6 mean to all established VALIDATION rows
# 2. Compare:
#
#       U0: variance = B_sex
#       U1: variance = B_sex + G_sex * horizon_years
#
# 3. Fit variance-standardized Student-t
# 4. Select by AICc
# 5. Refit selected law at 365 / 730 / 1095 / 1825 / unlimited gaps
# 6. Freeze v6_uncertainty_spec.json
#
# TEST is not used.
# ==================================================================================================

import json
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# Prerequisites
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_19_PASS",
    "V5_ROLLING_PREDICTIONS",

    "V6_SELECTED_PERSISTENCE_MODEL",
    "V6_SELECTED_PERSISTENCE_PARAMS",
    "V6_OFFSET_STRUCTURE",
    "V6_OFFSET_LOOKUP",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(f"  - {x}" for x in missing)
    )

if not NOTEBOOK_2_CELL_19_PASS:
    raise RuntimeError("Cell 19 did not pass.")


# --------------------------------------------------------------------------------------------------
# Frozen V6 mean helpers
# --------------------------------------------------------------------------------------------------

def v6_persistence(h):

    h = np.asarray(h, float)
    p = np.asarray(
        V6_SELECTED_PERSISTENCE_PARAMS,
        float,
    )

    model = V6_SELECTED_PERSISTENCE_MODEL

    if model == "M0_CONST_CONST":
        return (
            np.full(len(h), p[0]),
            np.full(len(h), p[1]),
        )

    if model == "M1_CONST_EXP":
        return (
            np.full(len(h), p[0]),
            p[1] * np.exp(-h / p[2]),
        )

    if model == "M2_EXP_CONST":
        return (
            p[0] * np.exp(-h / p[1]),
            np.full(len(h), p[2]),
        )

    if model == "M3_EXP_EXP":
        return (
            p[0] * np.exp(-h / p[1]),
            p[2] * np.exp(-h / p[3]),
        )

    raise ValueError(model)


def v6_offset(df):

    if V6_OFFSET_STRUCTURE == "NONE":
        return np.zeros(len(df))

    if V6_OFFSET_STRUCTURE == "COMMON":
        return np.full(
            len(df),
            V6_OFFSET_LOOKUP["ALL"],
        )

    key = (
        "sex_group"
        if V6_OFFSET_STRUCTURE == "SEX"
        else
        "model_group"
    )

    values = (
        df[key]
        .astype(str)
        .map(V6_OFFSET_LOOKUP)
    )

    if values.isna().any():
        raise RuntimeError("Missing V6 offset.")

    return values.to_numpy(float)


# --------------------------------------------------------------------------------------------------
# Established VALIDATION rows
# --------------------------------------------------------------------------------------------------

VAL = (
    V5_ROLLING_PREDICTIONS.loc[
        V5_ROLLING_PREDICTIONS["prediction_split"].eq("VALIDATION")
        &
        V5_ROLLING_PREDICTIONS["athlete_seen_in_training"]
    ]
    .copy()
    .reset_index(drop=True)
)


if VAL["state_gap_days"].isna().any():
    raise RuntimeError(
        "Established VALIDATION row has missing state gap."
    )


VAL["population_component"] = (
    VAL["v4_structural_mean"]
    - VAL["v4_athlete_intercept_component"]
    - VAL["v4_athlete_slope_component"]
)


rho_i, rho_s = v6_persistence(
    VAL["state_gap_days"].to_numpy(float)
)

offset = v6_offset(VAL)


VAL["v6_prediction_mean"] = (
    VAL["population_component"].to_numpy(float)
    +
    rho_i
    * VAL["v4_athlete_intercept_component"].to_numpy(float)
    +
    rho_s
    * VAL["v4_athlete_slope_component"].to_numpy(float)
    +
    offset
    +
    VAL["state_prior_mean"].to_numpy(float)
)


VAL["v6_error"] = (
    VAL["mark_numeric_m"]
    - VAL["v6_prediction_mean"]
)

VAL["horizon_years"] = (
    VAL["state_gap_days"].astype(float)
    / 365.25
)


# --------------------------------------------------------------------------------------------------
# Student-t likelihood
# --------------------------------------------------------------------------------------------------

def student_t_nll(error, variance, nu):

    if (
        nu <= 2
        or
        np.any(variance <= 0)
        or
        not np.isfinite(variance).all()
    ):
        return 1e100

    scale = (
        np.sqrt(variance)
        *
        np.sqrt((nu - 2) / nu)
    )

    logp = student_t.logpdf(
        error,
        df=nu,
        loc=0,
        scale=scale,
    )

    return (
        1e100
        if not np.isfinite(logp).all()
        else float(-logp.sum())
    )


def fit_uncertainty_model(df, model):

    error = df["v6_error"].to_numpy(float)
    h = df["horizon_years"].to_numpy(float)

    is_m = (
        df["sex_group"]
        .astype(str)
        .to_numpy()
        ==
        "M"
    )

    is_w = ~is_m

    var_m = np.mean(error[is_m] ** 2)
    var_w = np.mean(error[is_w] ** 2)


    if model == "U0_SEX_FLAT":

        starts = [
            [var_m, var_w, 8],
            [var_m, var_w, 16],
            [0.75 * var_m, 0.75 * var_w, 30],
        ]

        bounds = [
            (0.05, 400),
            (0.05, 400),
            (2.05, 100),
        ]


        def unpack(p):

            B_M, B_W, nu = p

            variance = np.where(
                is_m,
                B_M,
                B_W,
            )

            return B_M, B_W, 0.0, 0.0, nu, variance


    elif model == "U1_SEX_LINEAR":

        starts = [
            [
                0.75 * var_m,
                0.75 * var_w,
                1,
                1,
                8,
            ],
            [
                0.60 * var_m,
                0.60 * var_w,
                5,
                2,
                16,
            ],
            [
                0.50 * var_m,
                0.50 * var_w,
                10,
                5,
                30,
            ],
        ]

        bounds = [
            (0.05, 400),
            (0.05, 400),
            (0, 200),
            (0, 200),
            (2.05, 100),
        ]


        def unpack(p):

            B_M, B_W, G_M, G_W, nu = p

            variance = np.where(
                is_m,
                B_M + G_M * h,
                B_W + G_W * h,
            )

            return B_M, B_W, G_M, G_W, nu, variance


    else:
        raise ValueError(model)


    def objective(p):

        *_, nu, variance = unpack(p)

        return student_t_nll(
            error,
            variance,
            nu,
        )


    runs = [
        minimize(
            objective,
            np.asarray(x0, float),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": 5000,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "maxls": 50,
            },
        )
        for x0 in starts
    ]


    best = min(
        runs,
        key=lambda r: float(r.fun),
    )


    B_M, B_W, G_M, G_W, nu, variance = unpack(
        np.asarray(best.x, float)
    )


    return {
        "success": bool(best.success),
        "NLL": float(best.fun),

        "B_M": float(B_M),
        "B_W": float(B_W),

        "G_M": float(G_M),
        "G_W": float(G_W),

        "nu": float(nu),

        "variance": variance,
    }


# --------------------------------------------------------------------------------------------------
# Compare U0 vs U1
# --------------------------------------------------------------------------------------------------

candidate_specs = {
    "U0_SEX_FLAT": 3,
    "U1_SEX_LINEAR": 5,
}

uncertainty_rows = []
UNCERTAINTY_FITS = {}

n = len(VAL)


for model, k in candidate_specs.items():

    fit = fit_uncertainty_model(
        VAL,
        model,
    )

    UNCERTAINTY_FITS[model] = fit

    aic = 2 * k + 2 * fit["NLL"]

    aicc = (
        aic
        +
        2 * k * (k + 1)
        / (n - k - 1)
    )

    bic = (
        np.log(n) * k
        +
        2 * fit["NLL"]
    )


    uncertainty_rows.append({
        "model": model,
        "success": fit["success"],
        "k": k,

        "AIC": aic,
        "AICc": aicc,
        "BIC": bic,

        "B_M": fit["B_M"],
        "B_W": fit["B_W"],

        "G_M": fit["G_M"],
        "G_W": fit["G_W"],

        "nu": fit["nu"],
    })


V6_UNCERTAINTY_MODEL_COMPARISON = pd.DataFrame(
    uncertainty_rows
)

V6_UNCERTAINTY_MODEL_COMPARISON["delta_AICc"] = (
    V6_UNCERTAINTY_MODEL_COMPARISON["AICc"]
    - V6_UNCERTAINTY_MODEL_COMPARISON["AICc"].min()
)

V6_UNCERTAINTY_MODEL_COMPARISON = (
    V6_UNCERTAINTY_MODEL_COMPARISON.sort_values(
        ["AICc", "BIC"]
    )
    .reset_index(drop=True)
)


V6_UNCERTAINTY_MODEL = str(
    V6_UNCERTAINTY_MODEL_COMPARISON.iloc[0]["model"]
)

BEST = UNCERTAINTY_FITS[
    V6_UNCERTAINTY_MODEL
]


V6_B_M = BEST["B_M"]
V6_B_W = BEST["B_W"]

V6_G_M = BEST["G_M"]
V6_G_W = BEST["G_W"]

V6_NU = BEST["nu"]


# --------------------------------------------------------------------------------------------------
# Validation intervals
# --------------------------------------------------------------------------------------------------

VAL["v6_prediction_variance"] = BEST["variance"]

VAL["v6_prediction_sigma"] = np.sqrt(
    VAL["v6_prediction_variance"]
)

VAL["v6_standardized_error"] = (
    VAL["v6_error"]
    /
    VAL["v6_prediction_sigma"]
)


LEVELS = [0.50, 0.80, 0.95]


def coverage_summary(df, nu):

    rows = []

    for level in LEVELS:

        q = (
            student_t.ppf(
                (1 + level) / 2,
                df=nu,
            )
            *
            np.sqrt((nu - 2) / nu)
        )

        half_width = (
            q
            *
            df["v6_prediction_sigma"]
        )

        lower = (
            df["v6_prediction_mean"]
            -
            half_width
        )

        upper = (
            df["v6_prediction_mean"]
            +
            half_width
        )

        rows.append({
            "level": level,
            "target_pct": 100 * level,

            "coverage_pct":
                100
                * df["mark_numeric_m"]
                .between(
                    lower,
                    upper,
                    inclusive="both",
                )
                .mean(),

            "mean_width_m":
                float(
                    (2 * half_width).mean()
                ),
        })

    return pd.DataFrame(rows)


V6_VALIDATION_COVERAGE = coverage_summary(
    VAL,
    V6_NU,
)


# --------------------------------------------------------------------------------------------------
# Horizon sensitivity — refit selected uncertainty law only
# --------------------------------------------------------------------------------------------------

cutoffs = [
    ("365d", 365),
    ("730d", 730),
    ("1095d", 1095),
    ("1825d", 1825),
    ("Unlimited", None),
]


sensitivity_rows = []


for label, cutoff in cutoffs:

    d = (
        VAL.copy()
        if cutoff is None
        else
        VAL.loc[
            VAL["state_gap_days"] <= cutoff
        ].copy()
    )


    fit = fit_uncertainty_model(
        d,
        V6_UNCERTAINTY_MODEL,
    )


    d["v6_prediction_variance"] = fit["variance"]

    d["v6_prediction_sigma"] = np.sqrt(
        d["v6_prediction_variance"]
    )


    coverage = coverage_summary(
        d,
        fit["nu"],
    ).set_index("level")


    sensitivity_rows.append({
        "max_gap": label,

        "rows": len(d),
        "athletes": d["aaAthleteId"].nunique(),

        "B_M": fit["B_M"],
        "B_W": fit["B_W"],

        "G_M": fit["G_M"],
        "G_W": fit["G_W"],

        "nu": fit["nu"],

        "coverage_50":
            coverage.loc[0.50, "coverage_pct"],

        "coverage_80":
            coverage.loc[0.80, "coverage_pct"],

        "coverage_95":
            coverage.loc[0.95, "coverage_pct"],

        "success":
            fit["success"],
    })


V6_UNCERTAINTY_HORIZON_SENSITIVITY = pd.DataFrame(
    sensitivity_rows
)


# --------------------------------------------------------------------------------------------------
# Freeze uncertainty specification
# --------------------------------------------------------------------------------------------------

V6_UNCERTAINTY_SPEC = {
    "model":
        V6_UNCERTAINTY_MODEL,

    "distribution":
        "variance_standardized_student_t",

    "nu":
        V6_NU,

    "B": {
        "M": V6_B_M,
        "W": V6_B_W,
    },

    "G": {
        "M": V6_G_M,
        "W": V6_G_W,
    },

    "variance_formula":
        "B_sex + G_sex * horizon_days / 365.25",

    "calibration_split":
        "VALIDATION",

    "calibration_population":
        "TRAIN_SEEN_ATHLETES",
}


# --------------------------------------------------------------------------------------------------
# QA + save
# --------------------------------------------------------------------------------------------------

CELL_20_PASS = bool(
    V6_UNCERTAINTY_MODEL_COMPARISON["success"].all()
    and
    V6_UNCERTAINTY_HORIZON_SENSITIVITY["success"].all()
    and
    np.isfinite(
        VAL[
            [
                "v6_prediction_mean",
                "v6_prediction_variance",
                "v6_prediction_sigma",
                "v6_standardized_error",
            ]
        ].to_numpy(float)
    ).all()
    and
    (VAL["v6_prediction_variance"] > 0).all()
)


V6_UNCERTAINTY_SPEC_PATH = (
    TRAINING_RUN_DIR
    / "v6_uncertainty_spec.json"
)

V6_UNCERTAINTY_COMPARISON_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_uncertainty_model_comparison.parquet"
)

V6_UNCERTAINTY_SENSITIVITY_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_uncertainty_horizon_sensitivity.parquet"
)

V6_VALIDATION_UNCERTAINTY_PATH = (
    TRAINING_CHECKPOINT_DIR
    / "v6_validation_uncertainty.parquet"
)


with V6_UNCERTAINTY_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        V6_UNCERTAINTY_SPEC,
        f,
        indent=2,
        default=str,
    )


V6_UNCERTAINTY_MODEL_COMPARISON.to_parquet(
    V6_UNCERTAINTY_COMPARISON_PATH,
    index=False,
)

V6_UNCERTAINTY_HORIZON_SENSITIVITY.to_parquet(
    V6_UNCERTAINTY_SENSITIVITY_PATH,
    index=False,
)

VAL.to_parquet(
    V6_VALIDATION_UNCERTAINTY_PATH,
    index=False,
)


TRAINING_MANIFEST["v6_uncertainty_calibration"] = {
    "status":
        "PASSED" if CELL_20_PASS else "FAILED",

    "spec":
        V6_UNCERTAINTY_SPEC,

    "spec_path":
        str(V6_UNCERTAINTY_SPEC_PATH),
}

TRAINING_MANIFEST["status"] = (
    "V6_FULL_SPEC_READY"
    if CELL_20_PASS
    else "V6_UNCERTAINTY_CALIBRATION_FAILED"
)

with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


V6_VALIDATION_UNCERTAINTY = VAL

NOTEBOOK_2_CELL_20_PASS = CELL_20_PASS


# --------------------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 130)
print("V6 TRAINING — UNCERTAINTY CALIBRATION + ROBUSTNESS")
print("=" * 130)

print()
print("UNCERTAINTY MODEL COMPARISON")
display(
    V6_UNCERTAINTY_MODEL_COMPARISON[
        [
            "model",
            "AICc",
            "delta_AICc",
            "BIC",
            "B_M",
            "B_W",
            "G_M",
            "G_W",
            "nu",
        ]
    ]
)

print()
print("VALIDATION COVERAGE")
display(V6_VALIDATION_COVERAGE)

print()
print("HORIZON SENSITIVITY")
display(V6_UNCERTAINTY_HORIZON_SENSITIVITY)

print()
print("FINAL UNCERTAINTY SPEC")
print(V6_UNCERTAINTY_SPEC)

print()
print("=" * 130)
print(
    "CELL 20 STATUS: "
    + (
        "V6 UNCERTAINTY CALIBRATION PASSED"
        if CELL_20_PASS
        else
        "V6 UNCERTAINTY CALIBRATION REVIEW REQUIRED"
    )
)
print("=" * 130)



V6 TRAINING — MEAN CALIBRATION

EMPIRICAL PERSISTENCE


,gap_band,n,median_gap_days,rho_intercept,intercept_se,rho_slope,slope_se,status
0,01-30,2,28.0,NaN,NaN,NaN,NaN,INSUFFICIENT_ROWS
1,31-60,45,43.0,0.954484,0.074123,0.538024,0.295965,FITTED
2,61-90,57,76.0,0.870979,0.045333,0.866234,0.232918,FITTED
3,91-120,118,106.0,1.005929,0.047505,1.245068,0.180226,FITTED
4,121-180,428,158.0,0.954296,0.029767,0.924713,0.064139,FITTED
5,181-365,1410,248.0,0.951976,0.012625,0.950578,0.039737,FITTED



PERSISTENCE MODEL COMPARISON


,model,AICc,delta_AICc,BIC,c_intercept,tau_intercept_days,c_slope,tau_slope_days
0,M0_CONST_CONST,-20.520568,0.000000,-21.629683,0.950516,NaN,0.946859,NaN
1,M1_CONST_EXP,-16.191745,4.328823,-19.283989,0.950516,NaN,0.957018,20000.0
2,M2_EXP_CONST,-15.913813,4.606755,-19.006057,0.960735,20000.0,0.946859,NaN
3,M3_EXP_EXP,-9.870704,10.649864,-16.660363,0.960735,20000.0,0.957018,20000.0



OFFSET MODEL COMPARISON


...


VALIDATION COVERAGE


,level,target_pct,coverage_pct,mean_width_m
0,0.50,50.0,48.797572,4.274967
1,0.80,80.0,79.733831,8.273155
2,0.95,95.0,95.260332,13.097985



HORIZON SENSITIVITY


,max_gap,rows,athletes,B_M,B_W,G_M,G_W,nu,coverage_50,coverage_80,coverage_95,success
0,365d,4044,2133,11.732141,7.122991,3.171165,4.406734,17.905684,48.936696,79.772502,95.227498,True
1,730d,4223,2256,11.547293,7.212698,4.024829,3.986718,17.295146,48.875207,79.753730,95.287710,True
2,1095d,4260,2281,11.387149,7.198142,4.733523,4.082166,16.753629,48.826291,79.788732,95.211268,True
3,1825d,4278,2296,11.512280,7.223888,4.291454,3.978103,16.642497,48.854605,79.756896,95.254792,True
4,Unlimited,4283,2299,11.485511,7.217274,4.382833,3.995875,16.738659,48.797572,79.733831,95.260332,True



FINAL UNCERTAINTY SPEC
{'model': 'U1_SEX_LINEAR', 'distribution': 'variance_standardized_student_t', 'nu': 16.73865891564449, 'B': {'M': 11.485510752250018, 'W': 7.217274066737105}, 'G': {'M': 4.382833269868732, 'W': 3.995875249730842}, 'variance_formula': 'B_sex + G_sex * horizon_days / 365.25', 'calibration_split': 'VALIDATION', 'calibration_population': 'TRAIN_SEEN_ATHLETES'}

CELL 20 STATUS: V6 UNCERTAINTY CALIBRATION PASSED


## Stage 08 — Frozen V6 TEST, POST_TEST and preliminary gate

Evaluate the fully frozen pre-V6.1 V6 candidate on TEST and POST_TEST and record its preliminary production gate.

**Legacy source cells represented:** 11, 12, 13  
**Execution rule:** run only after the preceding stage has completed successfully.


In [9]:
# ==================================================================================================
# REFACTORED STAGE 08 — Frozen V6 TEST, POST_TEST and preliminary gate
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 11
# Original source SHA256: eeb97dd3d20966856ca4c995e36a7513b8b0c853e98a587319f67bce44a72b3a
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 21 — COMPLETE FROZEN V6 TEST EVALUATION
# ==================================================================================================
#
# The complete V6 model is frozen before this cell.
#
# This cell performs NO fitting.
#
# Point prediction:
#       V4 → V5 → V6
#
# Uncertainty:
#       established / TRAIN-seen athletes only
#
# Cold-start athletes:
#       point prediction retained
#       uncertainty marked UNCALIBRATED
#
# ==================================================================================================

import json
import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_19_PASS",
    "NOTEBOOK_2_CELL_20_PASS",

    "V5_ROLLING_PREDICTIONS",
    "V6_VALIDATION_UNCERTAINTY",

    "V6_SELECTED_PERSISTENCE_MODEL",
    "V6_SELECTED_PERSISTENCE_PARAMS",
    "V6_OFFSET_STRUCTURE",
    "V6_OFFSET_LOOKUP",

    "V6_UNCERTAINTY_MODEL",
    "V6_B_M",
    "V6_B_W",
    "V6_G_M",
    "V6_G_W",
    "V6_NU",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


if not (
    NOTEBOOK_2_CELL_19_PASS
    and
    NOTEBOOK_2_CELL_20_PASS
):

    raise RuntimeError(
        "Cells 19 and 20 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FROZEN V6 MEAN HELPERS
# --------------------------------------------------------------------------------------------------

def v6_persistence(h):

    h = np.asarray(
        h,
        dtype=float,
    )


    p = np.asarray(
        V6_SELECTED_PERSISTENCE_PARAMS,
        dtype=float,
    )


    model = (
        V6_SELECTED_PERSISTENCE_MODEL
    )


    if model == "M0_CONST_CONST":

        return (
            np.full(
                len(h),
                p[0],
            ),

            np.full(
                len(h),
                p[1],
            ),
        )


    if model == "M1_CONST_EXP":

        return (
            np.full(
                len(h),
                p[0],
            ),

            p[1]
            *
            np.exp(
                -h
                /
                p[2]
            ),
        )


    if model == "M2_EXP_CONST":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            np.full(
                len(h),
                p[2],
            ),
        )


    if model == "M3_EXP_EXP":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            p[2]
            *
            np.exp(
                -h
                /
                p[3]
            ),
        )


    raise ValueError(
        f"Unknown persistence model: {model}"
    )


def v6_offset(df):

    if V6_OFFSET_STRUCTURE == "NONE":

        return np.zeros(
            len(df),
            dtype=float,
        )


    if V6_OFFSET_STRUCTURE == "COMMON":

        return np.full(
            len(df),
            float(
                V6_OFFSET_LOOKUP[
                    "ALL"
                ]
            ),
        )


    if V6_OFFSET_STRUCTURE == "SEX":

        key = "sex_group"


    elif V6_OFFSET_STRUCTURE == "GROUP":

        key = "model_group"


    else:

        raise ValueError(
            f"Unknown offset structure: "
            f"{V6_OFFSET_STRUCTURE}"
        )


    values = (
        df[
            key
        ]
        .astype(str)
        .map(
            V6_OFFSET_LOOKUP
        )
    )


    if values.isna().any():

        raise RuntimeError(
            "Missing frozen V6 offset."
        )


    return values.to_numpy(
        dtype=float
    )


# --------------------------------------------------------------------------------------------------
# 3. LOAD TEST DATA
# --------------------------------------------------------------------------------------------------

TEST = (
    V5_ROLLING_PREDICTIONS.loc[
        V5_ROLLING_PREDICTIONS[
            "prediction_split"
        ]
        .eq(
            "TEST"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


seen = (
    TEST[
        "athlete_seen_in_training"
    ]
    .to_numpy(
        dtype=bool
    )
)


# --------------------------------------------------------------------------------------------------
# 4. FROZEN V6 POINT PREDICTION
# --------------------------------------------------------------------------------------------------

TEST[
    "v6_population_component"
] = (
    TEST[
        "v4_structural_mean"
    ]

    -

    TEST[
        "v4_athlete_intercept_component"
    ]

    -

    TEST[
        "v4_athlete_slope_component"
    ]
)


if TEST.loc[
    seen,
    "state_gap_days",
].isna().any():

    raise RuntimeError(
        "Established TEST row has missing state gap."
    )


rho_i = np.ones(
    len(TEST),
    dtype=float,
)

rho_s = np.ones(
    len(TEST),
    dtype=float,
)

offset = np.zeros(
    len(TEST),
    dtype=float,
)


rho_i_seen, rho_s_seen = v6_persistence(
    TEST.loc[
        seen,
        "state_gap_days",
    ]
    .to_numpy(
        dtype=float
    )
)


rho_i[
    seen
] = rho_i_seen


rho_s[
    seen
] = rho_s_seen


offset[
    seen
] = v6_offset(
    TEST.loc[
        seen
    ]
)


TEST[
    "v6_rho_intercept"
] = rho_i


TEST[
    "v6_rho_slope"
] = rho_s


TEST[
    "v6_offset"
] = offset


v6_seen_prediction = (
    TEST[
        "v6_population_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    rho_i
    *
    TEST[
        "v4_athlete_intercept_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    rho_s
    *
    TEST[
        "v4_athlete_slope_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    offset

    +

    TEST[
        "state_prior_mean"
    ]
    .to_numpy(
        dtype=float
    )
)


# Cold-start point mean remains V5.
TEST[
    "v6_prediction_mean"
] = np.where(
    seen,

    v6_seen_prediction,

    TEST[
        "v5_prediction_mean"
    ]
    .to_numpy(
        dtype=float
    ),
)


TEST[
    "v4_error"
] = (
    TEST[
        "mark_numeric_m"
    ]

    -

    TEST[
        "v4_structural_mean"
    ]
)


TEST[
    "v5_error"
] = (
    TEST[
        "mark_numeric_m"
    ]

    -

    TEST[
        "v5_prediction_mean"
    ]
)


TEST[
    "v6_error"
] = (
    TEST[
        "mark_numeric_m"
    ]

    -

    TEST[
        "v6_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 5. POINT METRIC HELPER
# --------------------------------------------------------------------------------------------------

def point_metrics(df):

    def metrics(
        error,
    ):

        e = np.asarray(
            error,
            dtype=float,
        )


        return (
            float(
                np.mean(
                    np.abs(
                        e
                    )
                )
            ),

            float(
                np.sqrt(
                    np.mean(
                        e
                        **
                        2
                    )
                )
            ),

            float(
                np.mean(
                    e
                )
            ),
        )


    v4 = metrics(
        df[
            "v4_error"
        ]
    )


    v5 = metrics(
        df[
            "v5_error"
        ]
    )


    v6 = metrics(
        df[
            "v6_error"
        ]
    )


    return pd.Series(
        {
            "n":
                int(
                    len(
                        df
                    )
                ),

            "athletes":
                int(
                    df[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4[0],

            "V5_MAE":
                v5[0],

            "V6_MAE":
                v6[0],

            "V5_to_V6_MAE_gain_m":
                (
                    v5[0]
                    -
                    v6[0]
                ),

            "V4_to_V6_MAE_gain_m":
                (
                    v4[0]
                    -
                    v6[0]
                ),

            "V4_RMSE":
                v4[1],

            "V5_RMSE":
                v5[1],

            "V6_RMSE":
                v6[1],

            "V5_to_V6_RMSE_gain_m":
                (
                    v5[1]
                    -
                    v6[1]
                ),

            "V4_to_V6_RMSE_gain_m":
                (
                    v4[1]
                    -
                    v6[1]
                ),

            "V4_Bias":
                v4[2],

            "V5_Bias":
                v5[2],

            "V6_Bias":
                v6[2],
        }
    )


# --------------------------------------------------------------------------------------------------
# 6. POINT PREDICTION RESULTS
# --------------------------------------------------------------------------------------------------

TEST_POINT_OVERALL = (
    point_metrics(
        TEST
    )
    .to_frame()
    .T
)


TEST[
    "training_status"
] = np.where(
    seen,
    "SEEN",
    "UNSEEN",
)


TEST_POINT_SEEN = (
    TEST.groupby(
        "training_status",
        sort=False,
    )
    .apply(
        point_metrics,
        include_groups=False,
    )
    .reset_index()
)


EST = (
    TEST.loc[
        seen
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


ESTABLISHED_POINT_METRICS = (
    point_metrics(
        EST
    )
)


# --------------------------------------------------------------------------------------------------
# 7. FROZEN UNCERTAINTY — ESTABLISHED ATHLETES ONLY
# --------------------------------------------------------------------------------------------------

EST[
    "horizon_years"
] = (
    EST[
        "state_gap_days"
    ]
    .astype(float)

    /

    365.25
)


EST[
    "v6_prediction_variance"
] = np.where(

    EST[
        "sex_group"
    ]
    .eq(
        "M"
    ),

    V6_B_M
    +
    V6_G_M
    *
    EST[
        "horizon_years"
    ],

    V6_B_W
    +
    V6_G_W
    *
    EST[
        "horizon_years"
    ],
)


EST[
    "v6_prediction_sigma"
] = np.sqrt(
    EST[
        "v6_prediction_variance"
    ]
)


EST[
    "v6_standardized_error"
] = (
    EST[
        "v6_error"
    ]

    /

    EST[
        "v6_prediction_sigma"
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. PREDICTIVE INTERVAL LEVELS
# --------------------------------------------------------------------------------------------------

LEVELS = [
    0.50,
    0.80,
    0.95,
]


for level in LEVELS:

    pct = int(
        level
        *
        100
    )


    q = float(
        student_t.ppf(
            (
                1.0
                +
                level
            )
            /
            2.0,

            df=
                V6_NU,
        )

        *

        np.sqrt(
            (
                V6_NU
                -
                2.0
            )

            /

            V6_NU
        )
    )


    half_width = (
        q
        *
        EST[
            "v6_prediction_sigma"
        ]
    )


    EST[
        f"v6_lower_{pct}"
    ] = (
        EST[
            "v6_prediction_mean"
        ]

        -

        half_width
    )


    EST[
        f"v6_upper_{pct}"
    ] = (
        EST[
            "v6_prediction_mean"
        ]

        +

        half_width
    )


    EST[
        f"v6_width_{pct}"
    ] = (
        2.0
        *
        half_width
    )


    EST[
        f"v6_covered_{pct}"
    ] = (
        EST[
            "mark_numeric_m"
        ]
        .between(
            EST[
                f"v6_lower_{pct}"
            ],

            EST[
                f"v6_upper_{pct}"
            ],

            inclusive="both",
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. UNCERTAINTY METRIC HELPER — AMENDED
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# This function does NOT require pre-existing:
#
#       v6_covered_50
#       v6_covered_80
#       v6_covered_95
#
# It reconstructs intervals directly from:
#
#       v6_prediction_mean
#       v6_prediction_sigma
#       V6_NU
#
# This means it works identically for both:
#
#       EST
#       V6_VALIDATION_UNCERTAINTY
#
# --------------------------------------------------------------------------------------------------

def uncertainty_metrics(df):

    out = {
        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "standardized_error_mean":
            float(
                df[
                    "v6_standardized_error"
                ]
                .mean()
            ),

        "standardized_error_sd":
            float(
                df[
                    "v6_standardized_error"
                ]
                .std()
            ),
    }


    for level in LEVELS:

        pct = int(
            level
            *
            100
        )


        q = float(
            student_t.ppf(
                (
                    1.0
                    +
                    level
                )
                /
                2.0,

                df=
                    V6_NU,
            )

            *

            np.sqrt(
                (
                    V6_NU
                    -
                    2.0
                )

                /

                V6_NU
            )
        )


        half_width = (
            q
            *
            df[
                "v6_prediction_sigma"
            ]
        )


        lower = (
            df[
                "v6_prediction_mean"
            ]

            -

            half_width
        )


        upper = (
            df[
                "v6_prediction_mean"
            ]

            +

            half_width
        )


        covered = (
            df[
                "mark_numeric_m"
            ]
            .between(
                lower,
                upper,
                inclusive="both",
            )
        )


        out[
            f"coverage_{pct}"
        ] = float(
            100.0
            *
            covered.mean()
        )


        out[
            f"width_{pct}"
        ] = float(
            (
                2.0
                *
                half_width
            )
            .mean()
        )


    return pd.Series(
        out
    )


# --------------------------------------------------------------------------------------------------
# 10. TEST UNCERTAINTY RESULTS
# --------------------------------------------------------------------------------------------------

TEST_UNCERTAINTY_ESTABLISHED = (
    uncertainty_metrics(
        EST
    )
    .to_frame()
    .T
)


TEST_UNCERTAINTY_BY_SEX = (
    EST.groupby(
        "sex_group",
        sort=True,
    )
    .apply(
        uncertainty_metrics,
        include_groups=False,
    )
    .reset_index()
)


TEST_UNCERTAINTY_BY_GROUP = (
    EST.groupby(
        "model_group",
        sort=False,
    )
    .apply(
        uncertainty_metrics,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 11. VALIDATION → TEST UNCERTAINTY GENERALIZATION
# --------------------------------------------------------------------------------------------------

VALIDATION_UNCERTAINTY_METRICS = (
    uncertainty_metrics(
        V6_VALIDATION_UNCERTAINTY
    )
    .to_frame()
    .T
)


VALIDATION_UNCERTAINTY_METRICS.insert(
    0,
    "split",
    "VALIDATION",
)


TEST_REFERENCE = (
    TEST_UNCERTAINTY_ESTABLISHED.copy()
)


TEST_REFERENCE.insert(
    0,
    "split",
    "TEST_ESTABLISHED",
)


VALIDATION_VS_TEST = pd.concat(
    [
        VALIDATION_UNCERTAINTY_METRICS,
        TEST_REFERENCE,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 12. PAIRED ATHLETE-CLUSTER BOOTSTRAP — ESTABLISHED V5 → V6 POINT GAIN
# --------------------------------------------------------------------------------------------------

EST[
    "v5_abs"
] = (
    EST[
        "v5_error"
    ]
    .abs()
)


EST[
    "v6_abs"
] = (
    EST[
        "v6_error"
    ]
    .abs()
)


EST[
    "v5_sq"
] = (
    EST[
        "v5_error"
    ]
    **
    2
)


EST[
    "v6_sq"
] = (
    EST[
        "v6_error"
    ]
    **
    2
)


cluster = (
    EST.groupby(
        "aaAthleteId",
        as_index=False,
    )
    .agg(
        n=(
            "v5_error",
            "size",
        ),

        v5_abs=(
            "v5_abs",
            "sum",
        ),

        v6_abs=(
            "v6_abs",
            "sum",
        ),

        v5_sq=(
            "v5_sq",
            "sum",
        ),

        v6_sq=(
            "v6_sq",
            "sum",
        ),
    )
)


rng = np.random.default_rng(
    42
)


B = 2000

n_cluster = len(
    cluster
)


cluster_n = (
    cluster[
        "n"
    ]
    .to_numpy(
        dtype=float
    )
)


cluster_v5_abs = (
    cluster[
        "v5_abs"
    ]
    .to_numpy(
        dtype=float
    )
)


cluster_v6_abs = (
    cluster[
        "v6_abs"
    ]
    .to_numpy(
        dtype=float
    )
)


cluster_v5_sq = (
    cluster[
        "v5_sq"
    ]
    .to_numpy(
        dtype=float
    )
)


cluster_v6_sq = (
    cluster[
        "v6_sq"
    ]
    .to_numpy(
        dtype=float
    )
)


boot_mae = np.empty(
    B
)

boot_rmse = np.empty(
    B
)


for b in range(
    B
):

    idx = rng.integers(
        0,
        n_cluster,
        size=n_cluster,
    )


    n_rows = (
        cluster_n[
            idx
        ]
        .sum()
    )


    mae5 = (
        cluster_v5_abs[
            idx
        ]
        .sum()

        /

        n_rows
    )


    mae6 = (
        cluster_v6_abs[
            idx
        ]
        .sum()

        /

        n_rows
    )


    rmse5 = np.sqrt(
        cluster_v5_sq[
            idx
        ]
        .sum()

        /

        n_rows
    )


    rmse6 = np.sqrt(
        cluster_v6_sq[
            idx
        ]
        .sum()

        /

        n_rows
    )


    boot_mae[
        b
    ] = (
        mae5
        -
        mae6
    )


    boot_rmse[
        b
    ] = (
        rmse5
        -
        rmse6
    )


BOOTSTRAP_SUMMARY = pd.DataFrame(
    [
        {
            "metric":
                "V5_to_V6_MAE_gain_m",

            "estimate":
                ESTABLISHED_POINT_METRICS[
                    "V5_to_V6_MAE_gain_m"
                ],

            "ci95_lo":
                float(
                    np.quantile(
                        boot_mae,
                        0.025,
                    )
                ),

            "ci95_hi":
                float(
                    np.quantile(
                        boot_mae,
                        0.975,
                    )
                ),

            "positive_pct":
                float(
                    100.0
                    *
                    np.mean(
                        boot_mae
                        >
                        0
                    )
                ),
        },

        {
            "metric":
                "V5_to_V6_RMSE_gain_m",

            "estimate":
                ESTABLISHED_POINT_METRICS[
                    "V5_to_V6_RMSE_gain_m"
                ],

            "ci95_lo":
                float(
                    np.quantile(
                        boot_rmse,
                        0.025,
                    )
                ),

            "ci95_hi":
                float(
                    np.quantile(
                        boot_rmse,
                        0.975,
                    )
                ),

            "positive_pct":
                float(
                    100.0
                    *
                    np.mean(
                        boot_rmse
                        >
                        0
                    )
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. COLD-START STATUS
# --------------------------------------------------------------------------------------------------

COLD = (
    TEST.loc[
        ~seen
    ]
    .copy()
)


COLD_START_SUMMARY = pd.DataFrame(
    [
        {
            "status":
                "COLD_START_UNCALIBRATED",

            "rows":
                int(
                    len(
                        COLD
                    )
                ),

            "athletes":
                int(
                    COLD[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "rows_with_state_history":
                int(
                    COLD[
                        "state_source_date"
                    ]
                    .notna()
                    .sum()
                ),

            "rows_without_state_history":
                int(
                    COLD[
                        "state_source_date"
                    ]
                    .isna()
                    .sum()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. QA
# --------------------------------------------------------------------------------------------------

uncertainty_columns = [
    "v6_prediction_variance",
    "v6_prediction_sigma",
    "v6_standardized_error",

    "v6_lower_50",
    "v6_upper_50",

    "v6_lower_80",
    "v6_upper_80",

    "v6_lower_95",
    "v6_upper_95",
]


N_NONFINITE = int(
    (
        ~np.isfinite(
            EST[
                uncertainty_columns
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        EST[
            "v6_prediction_variance"
        ]
        >
        0
    )
    .all()
)


INTERVAL_ORDER_PASS = bool(

    (
        EST[
            "v6_lower_95"
        ]
        <=
        EST[
            "v6_lower_80"
        ]
    )
    .all()

    and

    (
        EST[
            "v6_lower_80"
        ]
        <=
        EST[
            "v6_lower_50"
        ]
    )
    .all()

    and

    (
        EST[
            "v6_upper_50"
        ]
        <=
        EST[
            "v6_upper_80"
        ]
    )
    .all()

    and

    (
        EST[
            "v6_upper_80"
        ]
        <=
        EST[
            "v6_upper_95"
        ]
    )
    .all()
)


UNSEEN_V6_EQUALS_V5 = bool(
    np.allclose(
        TEST.loc[
            ~seen,
            "v6_prediction_mean",
        ],

        TEST.loc[
            ~seen,
            "v5_prediction_mean",
        ],

        rtol=0.0,
        atol=1e-12,
    )
)


CELL_21_PASS = bool(

    N_NONFINITE == 0

    and

    POSITIVE_VARIANCE_PASS

    and

    INTERVAL_ORDER_PASS

    and

    UNSEEN_V6_EQUALS_V5
)


# --------------------------------------------------------------------------------------------------
# 15. SAVE ARTIFACTS
# --------------------------------------------------------------------------------------------------

V6_TEST_PREDICTIONS_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_test_predictions.parquet"
)


V6_TEST_POINT_METRICS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_point_metrics.parquet"
)


V6_TEST_UNCERTAINTY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_uncertainty_established.parquet"
)


V6_TEST_UNCERTAINTY_BY_SEX_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_uncertainty_established_by_sex.parquet"
)


V6_TEST_UNCERTAINTY_BY_GROUP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_uncertainty_established_by_group.parquet"
)


V6_VALIDATION_VS_TEST_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_validation_vs_test_uncertainty.parquet"
)


V6_TEST_BOOTSTRAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_cluster_bootstrap.parquet"
)


V6_COLD_START_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_cold_start_status.parquet"
)


TEST.to_parquet(
    V6_TEST_PREDICTIONS_PATH,
    index=False,
)


TEST_POINT_OVERALL.to_parquet(
    V6_TEST_POINT_METRICS_PATH,
    index=False,
)


TEST_UNCERTAINTY_ESTABLISHED.to_parquet(
    V6_TEST_UNCERTAINTY_PATH,
    index=False,
)


TEST_UNCERTAINTY_BY_SEX.to_parquet(
    V6_TEST_UNCERTAINTY_BY_SEX_PATH,
    index=False,
)


TEST_UNCERTAINTY_BY_GROUP.to_parquet(
    V6_TEST_UNCERTAINTY_BY_GROUP_PATH,
    index=False,
)


VALIDATION_VS_TEST.to_parquet(
    V6_VALIDATION_VS_TEST_PATH,
    index=False,
)


BOOTSTRAP_SUMMARY.to_parquet(
    V6_TEST_BOOTSTRAP_PATH,
    index=False,
)


COLD_START_SUMMARY.to_parquet(
    V6_COLD_START_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 16. UPDATE MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v6_test_evaluation"
] = {

    "status":
        (
            "PASSED"
            if CELL_21_PASS
            else "FAILED"
        ),

    "parameters_frozen":
        True,

    "test_rows":
        int(
            len(
                TEST
            )
        ),

    "test_athletes":
        int(
            TEST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "established_rows":
        int(
            len(
                EST
            )
        ),

    "established_athletes":
        int(
            EST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "established_point_metrics": {
        key:
            float(
                value
            )

        for key, value
        in ESTABLISHED_POINT_METRICS.items()
    },

    "established_uncertainty": {
        key:
            float(
                value
            )

        for key, value
        in TEST_UNCERTAINTY_ESTABLISHED.iloc[
            0
        ]
        .items()
    },

    "cold_start_uncertainty":
        "UNCALIBRATED",

    "prediction_path":
        str(
            V6_TEST_PREDICTIONS_PATH
        ),

    "bootstrap_path":
        str(
            V6_TEST_BOOTSTRAP_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V6_TEST_EVALUATED"
    if CELL_21_PASS
    else "V6_TEST_EVALUATION_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 17. GLOBALS
# --------------------------------------------------------------------------------------------------

V6_TEST_PREDICTIONS = (
    TEST
)


V6_TEST_ESTABLISHED = (
    EST
)


V6_TEST_UNCERTAINTY_ESTABLISHED = (
    TEST_UNCERTAINTY_ESTABLISHED
)


V6_TEST_UNCERTAINTY_BY_SEX = (
    TEST_UNCERTAINTY_BY_SEX
)


V6_TEST_UNCERTAINTY_BY_GROUP = (
    TEST_UNCERTAINTY_BY_GROUP
)


V6_TEST_BOOTSTRAP_SUMMARY = (
    BOOTSTRAP_SUMMARY
)


NOTEBOOK_2_CELL_21_PASS = (
    CELL_21_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 135)
print("V6 TRAINING — COMPLETE FROZEN V6 TEST EVALUATION")
print("=" * 135)


print()
print("POINT PREDICTION — OVERALL")
print("-" * 135)

display(
    TEST_POINT_OVERALL
)


print()
print("POINT PREDICTION — SEEN VS UNSEEN")
print("-" * 135)

display(
    TEST_POINT_SEEN
)


print()
print("PRIMARY — ESTABLISHED ATHLETES")
print("-" * 135)

display(
    ESTABLISHED_POINT_METRICS
    .to_frame(
        "value"
    )
)


print()
print("PAIRED ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 135)

display(
    BOOTSTRAP_SUMMARY
)


print()
print("UNCERTAINTY — ESTABLISHED")
print("-" * 135)

display(
    TEST_UNCERTAINTY_ESTABLISHED
)


print()
print("VALIDATION → TEST GENERALIZATION")
print("-" * 135)

display(
    VALIDATION_VS_TEST
)


print()
print("ESTABLISHED — BY SEX")
print("-" * 135)

display(
    TEST_UNCERTAINTY_BY_SEX
)


print()
print("ESTABLISHED — BY MODEL GROUP")
print("-" * 135)

display(
    TEST_UNCERTAINTY_BY_GROUP
)


print()
print("COLD-START STATUS")
print("-" * 135)

display(
    COLD_START_SUMMARY
)


print()
print("QA")
print("-" * 135)

print(
    f"Nonfinite established uncertainty : "
    f"{N_NONFINITE}"
)

print(
    f"Positive variance                  : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Interval ordering                  : "
    f"{INTERVAL_ORDER_PASS}"
)

print(
    f"Unseen V6 point mean = V5          : "
    f"{UNSEEN_V6_EQUALS_V5}"
)


print()
print("=" * 135)


if CELL_21_PASS:

    print(
        "CELL 21 STATUS: COMPLETE FROZEN V6 TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 21 STATUS: V6 TEST EVALUATION REVIEW REQUIRED"
    )


print("=" * 135)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 12
# Original source SHA256: 33334618289a3b728d0b979225e4c701d87c0c4dcfeb83bced74e48a414b249e
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 22 — FROZEN V6 POST_TEST EVALUATION
# ==================================================================================================
#
# Purpose
# -------
# Evaluate the completely frozen V6 model on POST_TEST:
#
#       POST_TEST >= 2026-08-31
#
# No fitting.
# No model selection.
# No recalibration.
#
# POST_TEST is a recent-production / prospective-style diagnostic.
#
# Point prediction:
#       V4 → V5 → V6
#
# Uncertainty:
#       established / TRAIN-seen athletes only
#
# Cold-start athletes:
#       V6 point mean = V5
#       uncertainty = UNCALIBRATED
#
# ==================================================================================================

import json
import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_21_PASS",

    "V5_ROLLING_PREDICTIONS",

    "V6_SELECTED_PERSISTENCE_MODEL",
    "V6_SELECTED_PERSISTENCE_PARAMS",

    "V6_OFFSET_STRUCTURE",
    "V6_OFFSET_LOOKUP",

    "V6_B_M",
    "V6_B_W",
    "V6_G_M",
    "V6_G_W",
    "V6_NU",

    "V6_TEST_ESTABLISHED",
    "V6_TEST_UNCERTAINTY_ESTABLISHED",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


if not NOTEBOOK_2_CELL_21_PASS:

    raise RuntimeError(
        "Cell 21 must pass before POST_TEST evaluation."
    )


# --------------------------------------------------------------------------------------------------
# 2. FROZEN V6 MEAN HELPERS
# --------------------------------------------------------------------------------------------------

def post_v6_persistence(h):

    h = np.asarray(
        h,
        dtype=float,
    )


    p = np.asarray(
        V6_SELECTED_PERSISTENCE_PARAMS,
        dtype=float,
    )


    model = (
        V6_SELECTED_PERSISTENCE_MODEL
    )


    if model == "M0_CONST_CONST":

        return (
            np.full(
                len(h),
                p[0],
            ),

            np.full(
                len(h),
                p[1],
            ),
        )


    if model == "M1_CONST_EXP":

        return (
            np.full(
                len(h),
                p[0],
            ),

            p[1]
            *
            np.exp(
                -h
                /
                p[2]
            ),
        )


    if model == "M2_EXP_CONST":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            np.full(
                len(h),
                p[2],
            ),
        )


    if model == "M3_EXP_EXP":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            p[2]
            *
            np.exp(
                -h
                /
                p[3]
            ),
        )


    raise ValueError(
        f"Unknown persistence model: {model}"
    )


def post_v6_offset(df):

    if V6_OFFSET_STRUCTURE == "NONE":

        return np.zeros(
            len(df),
            dtype=float,
        )


    if V6_OFFSET_STRUCTURE == "COMMON":

        return np.full(
            len(df),
            float(
                V6_OFFSET_LOOKUP[
                    "ALL"
                ]
            ),
        )


    if V6_OFFSET_STRUCTURE == "SEX":

        key = "sex_group"


    elif V6_OFFSET_STRUCTURE == "GROUP":

        key = "model_group"


    else:

        raise ValueError(
            f"Unknown offset structure: "
            f"{V6_OFFSET_STRUCTURE}"
        )


    values = (
        df[
            key
        ]
        .astype(str)
        .map(
            V6_OFFSET_LOOKUP
        )
    )


    if values.isna().any():

        raise RuntimeError(
            "Missing frozen V6 offset."
        )


    return values.to_numpy(
        dtype=float
    )


# --------------------------------------------------------------------------------------------------
# 3. LOAD POST_TEST
# --------------------------------------------------------------------------------------------------

POST = (
    V5_ROLLING_PREDICTIONS.loc[
        V5_ROLLING_PREDICTIONS[
            "prediction_split"
        ]
        .eq(
            "POST_TEST"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(POST) == 0:

    raise RuntimeError(
        "No POST_TEST rows are available."
    )


POST[
    "_post_row_id"
] = np.arange(
    len(POST)
)


seen = (
    POST[
        "athlete_seen_in_training"
    ]
    .to_numpy(
        dtype=bool
    )
)


# --------------------------------------------------------------------------------------------------
# 4. FROZEN V6 POINT PREDICTION
# --------------------------------------------------------------------------------------------------

POST[
    "v6_population_component"
] = (
    POST[
        "v4_structural_mean"
    ]

    -

    POST[
        "v4_athlete_intercept_component"
    ]

    -

    POST[
        "v4_athlete_slope_component"
    ]
)


if POST.loc[
    seen,
    "state_gap_days",
].isna().any():

    raise RuntimeError(
        "Established POST_TEST row has missing state gap."
    )


# Start everyone from V5.
POST[
    "v6_prediction_mean"
] = (
    POST[
        "v5_prediction_mean"
    ]
    .to_numpy(
        dtype=float
    )
)


POST[
    "v6_rho_intercept"
] = np.nan


POST[
    "v6_rho_slope"
] = np.nan


POST[
    "v6_offset"
] = np.nan


# Apply V6 only to established athletes.
POST_SEEN = (
    POST.loc[
        seen
    ]
    .copy()
)


rho_i, rho_s = post_v6_persistence(
    POST_SEEN[
        "state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


offset = post_v6_offset(
    POST_SEEN
)


seen_prediction = (
    POST_SEEN[
        "v6_population_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    rho_i
    *
    POST_SEEN[
        "v4_athlete_intercept_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    rho_s
    *
    POST_SEEN[
        "v4_athlete_slope_component"
    ]
    .to_numpy(
        dtype=float
    )

    +

    offset

    +

    POST_SEEN[
        "state_prior_mean"
    ]
    .to_numpy(
        dtype=float
    )
)


seen_ids = (
    POST_SEEN[
        "_post_row_id"
    ]
    .to_numpy()
)


POST.loc[
    seen_ids,
    "v6_prediction_mean",
] = seen_prediction


POST.loc[
    seen_ids,
    "v6_rho_intercept",
] = rho_i


POST.loc[
    seen_ids,
    "v6_rho_slope",
] = rho_s


POST.loc[
    seen_ids,
    "v6_offset",
] = offset


# --------------------------------------------------------------------------------------------------
# 5. ERRORS
# --------------------------------------------------------------------------------------------------

POST[
    "v4_error"
] = (
    POST[
        "mark_numeric_m"
    ]

    -

    POST[
        "v4_structural_mean"
    ]
)


POST[
    "v5_error"
] = (
    POST[
        "mark_numeric_m"
    ]

    -

    POST[
        "v5_prediction_mean"
    ]
)


POST[
    "v6_error"
] = (
    POST[
        "mark_numeric_m"
    ]

    -

    POST[
        "v6_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. POINT METRIC HELPER
# --------------------------------------------------------------------------------------------------

def post_point_metrics(df):

    def metrics(error):

        e = np.asarray(
            error,
            dtype=float,
        )

        return (
            float(
                np.mean(
                    np.abs(e)
                )
            ),

            float(
                np.sqrt(
                    np.mean(
                        e ** 2
                    )
                )
            ),

            float(
                np.mean(e)
            ),
        )


    v4 = metrics(
        df["v4_error"]
    )

    v5 = metrics(
        df["v5_error"]
    )

    v6 = metrics(
        df["v6_error"]
    )


    return pd.Series(
        {
            "n":
                int(len(df)),

            "athletes":
                int(
                    df[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4[0],

            "V5_MAE":
                v5[0],

            "V6_MAE":
                v6[0],

            "V5_to_V6_MAE_gain_m":
                v5[0] - v6[0],

            "V4_to_V6_MAE_gain_m":
                v4[0] - v6[0],

            "V4_RMSE":
                v4[1],

            "V5_RMSE":
                v5[1],

            "V6_RMSE":
                v6[1],

            "V5_to_V6_RMSE_gain_m":
                v5[1] - v6[1],

            "V4_to_V6_RMSE_gain_m":
                v4[1] - v6[1],

            "V4_Bias":
                v4[2],

            "V5_Bias":
                v5[2],

            "V6_Bias":
                v6[2],
        }
    )


# --------------------------------------------------------------------------------------------------
# 7. POINT RESULTS
# --------------------------------------------------------------------------------------------------

POST_POINT_OVERALL = (
    post_point_metrics(
        POST
    )
    .to_frame()
    .T
)


POST[
    "training_status"
] = np.where(
    seen,
    "SEEN",
    "UNSEEN",
)


POST_POINT_SEEN_UNSEEN = (
    POST.groupby(
        "training_status",
        sort=False,
    )
    .apply(
        post_point_metrics,
        include_groups=False,
    )
    .reset_index()
)


POST_EST = (
    POST.loc[
        seen
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


POST_ESTABLISHED_POINT_METRICS = (
    post_point_metrics(
        POST_EST
    )
)


# --------------------------------------------------------------------------------------------------
# 8. FROZEN UNCERTAINTY — ESTABLISHED ONLY
# --------------------------------------------------------------------------------------------------

POST_EST[
    "horizon_years"
] = (
    POST_EST[
        "state_gap_days"
    ]
    .astype(float)

    /

    365.25
)


POST_EST[
    "v6_prediction_variance"
] = np.where(

    POST_EST[
        "sex_group"
    ]
    .eq(
        "M"
    ),

    V6_B_M
    +
    V6_G_M
    *
    POST_EST[
        "horizon_years"
    ],

    V6_B_W
    +
    V6_G_W
    *
    POST_EST[
        "horizon_years"
    ],
)


POST_EST[
    "v6_prediction_sigma"
] = np.sqrt(
    POST_EST[
        "v6_prediction_variance"
    ]
)


POST_EST[
    "v6_standardized_error"
] = (
    POST_EST[
        "v6_error"
    ]

    /

    POST_EST[
        "v6_prediction_sigma"
    ]
)


LEVELS = [
    0.50,
    0.80,
    0.95,
]


for level in LEVELS:

    pct = int(
        level * 100
    )


    q = float(
        student_t.ppf(
            (
                1.0
                +
                level
            )
            /
            2.0,

            df=
                V6_NU,
        )

        *

        np.sqrt(
            (
                V6_NU
                -
                2.0
            )

            /

            V6_NU
        )
    )


    half_width = (
        q
        *
        POST_EST[
            "v6_prediction_sigma"
        ]
    )


    POST_EST[
        f"v6_lower_{pct}"
    ] = (
        POST_EST[
            "v6_prediction_mean"
        ]

        -

        half_width
    )


    POST_EST[
        f"v6_upper_{pct}"
    ] = (
        POST_EST[
            "v6_prediction_mean"
        ]

        +

        half_width
    )


    POST_EST[
        f"v6_width_{pct}"
    ] = (
        2.0
        *
        half_width
    )


    POST_EST[
        f"v6_covered_{pct}"
    ] = (
        POST_EST[
            "mark_numeric_m"
        ]
        .between(
            POST_EST[
                f"v6_lower_{pct}"
            ],

            POST_EST[
                f"v6_upper_{pct}"
            ],

            inclusive="both",
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. UNCERTAINTY METRIC HELPER
# --------------------------------------------------------------------------------------------------

def post_uncertainty_metrics(df):

    out = {
        "n":
            int(len(df)),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "standardized_error_mean":
            float(
                df[
                    "v6_standardized_error"
                ]
                .mean()
            ),

        "standardized_error_sd":
            float(
                df[
                    "v6_standardized_error"
                ]
                .std()
            ),
    }


    for level in LEVELS:

        pct = int(
            level * 100
        )


        out[
            f"coverage_{pct}"
        ] = float(
            100.0
            *
            df[
                f"v6_covered_{pct}"
            ]
            .mean()
        )


        out[
            f"width_{pct}"
        ] = float(
            df[
                f"v6_width_{pct}"
            ]
            .mean()
        )


    return pd.Series(out)


# --------------------------------------------------------------------------------------------------
# 10. POST_TEST UNCERTAINTY RESULTS
# --------------------------------------------------------------------------------------------------

POST_UNCERTAINTY_ESTABLISHED = (
    post_uncertainty_metrics(
        POST_EST
    )
    .to_frame()
    .T
)


POST_UNCERTAINTY_BY_SEX = (
    POST_EST.groupby(
        "sex_group",
        sort=True,
    )
    .apply(
        post_uncertainty_metrics,
        include_groups=False,
    )
    .reset_index()
)


POST_UNCERTAINTY_BY_GROUP = (
    POST_EST.groupby(
        "model_group",
        sort=False,
    )
    .apply(
        post_uncertainty_metrics,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 11. TEST → POST_TEST COMPARISON
# --------------------------------------------------------------------------------------------------

TEST_POINT_REFERENCE = (
    post_point_metrics(
        V6_TEST_ESTABLISHED
    )
    .to_frame()
    .T
)


TEST_POINT_REFERENCE.insert(
    0,
    "split",
    "TEST_ESTABLISHED",
)


POST_POINT_REFERENCE = (
    POST_ESTABLISHED_POINT_METRICS
    .to_frame()
    .T
)


POST_POINT_REFERENCE.insert(
    0,
    "split",
    "POST_TEST_ESTABLISHED",
)


TEST_VS_POST_POINT = pd.concat(
    [
        TEST_POINT_REFERENCE,
        POST_POINT_REFERENCE,
    ],
    ignore_index=True,
)


TEST_UNCERTAINTY_REFERENCE = (
    V6_TEST_UNCERTAINTY_ESTABLISHED.copy()
)


TEST_UNCERTAINTY_REFERENCE.insert(
    0,
    "split",
    "TEST_ESTABLISHED",
)


POST_UNCERTAINTY_REFERENCE = (
    POST_UNCERTAINTY_ESTABLISHED.copy()
)


POST_UNCERTAINTY_REFERENCE.insert(
    0,
    "split",
    "POST_TEST_ESTABLISHED",
)


TEST_VS_POST_UNCERTAINTY = pd.concat(
    [
        TEST_UNCERTAINTY_REFERENCE,
        POST_UNCERTAINTY_REFERENCE,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 12. COLD-START STATUS
# --------------------------------------------------------------------------------------------------

POST_COLD = (
    POST.loc[
        ~seen
    ]
    .copy()
)


POST_COLD_START_SUMMARY = pd.DataFrame(
    [
        {
            "status":
                "COLD_START_UNCALIBRATED",

            "rows":
                int(
                    len(
                        POST_COLD
                    )
                ),

            "athletes":
                int(
                    POST_COLD[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "rows_with_state_history":
                int(
                    POST_COLD[
                        "state_source_date"
                    ]
                    .notna()
                    .sum()
                ),

            "rows_without_state_history":
                int(
                    POST_COLD[
                        "state_source_date"
                    ]
                    .isna()
                    .sum()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. QA
# --------------------------------------------------------------------------------------------------

uncertainty_columns = [
    "v6_prediction_variance",
    "v6_prediction_sigma",
    "v6_standardized_error",

    "v6_lower_50",
    "v6_upper_50",

    "v6_lower_80",
    "v6_upper_80",

    "v6_lower_95",
    "v6_upper_95",
]


N_NONFINITE_ESTABLISHED = int(
    (
        ~np.isfinite(
            POST_EST[
                uncertainty_columns
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
    .sum()
)


POINT_FINITE_PASS = bool(
    np.isfinite(
        POST[
            [
                "v4_error",
                "v5_error",
                "v6_error",
                "v6_prediction_mean",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        POST_EST[
            "v6_prediction_variance"
        ]
        >
        0
    )
    .all()
)


INTERVAL_ORDER_PASS = bool(

    (
        POST_EST[
            "v6_lower_95"
        ]
        <=
        POST_EST[
            "v6_lower_80"
        ]
    )
    .all()

    and

    (
        POST_EST[
            "v6_lower_80"
        ]
        <=
        POST_EST[
            "v6_lower_50"
        ]
    )
    .all()

    and

    (
        POST_EST[
            "v6_upper_50"
        ]
        <=
        POST_EST[
            "v6_upper_80"
        ]
    )
    .all()

    and

    (
        POST_EST[
            "v6_upper_80"
        ]
        <=
        POST_EST[
            "v6_upper_95"
        ]
    )
    .all()
)


COLD_START_V6_EQUALS_V5 = bool(
    np.allclose(
        POST.loc[
            ~seen,
            "v6_prediction_mean",
        ],

        POST.loc[
            ~seen,
            "v5_prediction_mean",
        ],

        rtol=0.0,
        atol=1e-12,
    )
)


ROW_ACCOUNTING_PASS = bool(
    len(POST_EST)
    +
    len(POST_COLD)
    ==
    len(POST)
)


CELL_22_PASS = bool(

    len(POST) > 0

    and

    len(POST_EST) > 0

    and

    POINT_FINITE_PASS

    and

    N_NONFINITE_ESTABLISHED == 0

    and

    POSITIVE_VARIANCE_PASS

    and

    INTERVAL_ORDER_PASS

    and

    COLD_START_V6_EQUALS_V5

    and

    ROW_ACCOUNTING_PASS
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE
# --------------------------------------------------------------------------------------------------

V6_POST_TEST_PREDICTIONS_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_post_test_predictions.parquet"
)


V6_POST_TEST_POINT_METRICS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_post_test_point_metrics.parquet"
)


V6_POST_TEST_UNCERTAINTY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_post_test_uncertainty_established.parquet"
)


V6_POST_TEST_BY_SEX_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_post_test_uncertainty_established_by_sex.parquet"
)


V6_POST_TEST_BY_GROUP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_post_test_uncertainty_established_by_group.parquet"
)


V6_TEST_POST_POINT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_vs_post_test_point_metrics.parquet"
)


V6_TEST_POST_UNCERTAINTY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_test_vs_post_test_uncertainty.parquet"
)


POST.to_parquet(
    V6_POST_TEST_PREDICTIONS_PATH,
    index=False,
)


POST_POINT_OVERALL.to_parquet(
    V6_POST_TEST_POINT_METRICS_PATH,
    index=False,
)


POST_UNCERTAINTY_ESTABLISHED.to_parquet(
    V6_POST_TEST_UNCERTAINTY_PATH,
    index=False,
)


POST_UNCERTAINTY_BY_SEX.to_parquet(
    V6_POST_TEST_BY_SEX_PATH,
    index=False,
)


POST_UNCERTAINTY_BY_GROUP.to_parquet(
    V6_POST_TEST_BY_GROUP_PATH,
    index=False,
)


TEST_VS_POST_POINT.to_parquet(
    V6_TEST_POST_POINT_PATH,
    index=False,
)


TEST_VS_POST_UNCERTAINTY.to_parquet(
    V6_TEST_POST_UNCERTAINTY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 15. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v6_post_test_evaluation"
] = {

    "status":
        (
            "PASSED"
            if CELL_22_PASS
            else "FAILED"
        ),

    "purpose":
        "RECENT_PRODUCTION_DIAGNOSTIC_ONLY",

    "parameters_frozen":
        True,

    "post_test_rows":
        int(
            len(POST)
        ),

    "post_test_athletes":
        int(
            POST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "established_rows":
        int(
            len(
                POST_EST
            )
        ),

    "established_athletes":
        int(
            POST_EST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "established_point_metrics": {
        key:
            float(value)

        for key, value
        in POST_ESTABLISHED_POINT_METRICS.items()
    },

    "established_uncertainty": {
        key:
            float(value)

        for key, value
        in POST_UNCERTAINTY_ESTABLISHED.iloc[
            0
        ].items()
    },

    "cold_start_uncertainty":
        "UNCALIBRATED",
}


TRAINING_MANIFEST[
    "status"
] = (
    "V6_POST_TEST_EVALUATED"
    if CELL_22_PASS
    else
    "V6_POST_TEST_EVALUATION_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 16. GLOBALS
# --------------------------------------------------------------------------------------------------

V6_POST_TEST_PREDICTIONS = (
    POST
)


V6_POST_TEST_ESTABLISHED = (
    POST_EST
)


V6_POST_TEST_ESTABLISHED_POINT_METRICS = (
    POST_ESTABLISHED_POINT_METRICS
)


V6_POST_TEST_UNCERTAINTY_ESTABLISHED = (
    POST_UNCERTAINTY_ESTABLISHED
)


V6_POST_TEST_UNCERTAINTY_BY_SEX = (
    POST_UNCERTAINTY_BY_SEX
)


V6_POST_TEST_UNCERTAINTY_BY_GROUP = (
    POST_UNCERTAINTY_BY_GROUP
)


NOTEBOOK_2_CELL_22_PASS = (
    CELL_22_PASS
)


# --------------------------------------------------------------------------------------------------
# 17. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 140)
print("V6 TRAINING — FROZEN POST_TEST EVALUATION")
print("=" * 140)


print()
print("POINT PREDICTION — OVERALL")
print("-" * 140)

display(
    POST_POINT_OVERALL
)


print()
print("POINT PREDICTION — SEEN VS UNSEEN")
print("-" * 140)

display(
    POST_POINT_SEEN_UNSEEN
)


print()
print("PRIMARY — ESTABLISHED POST_TEST")
print("-" * 140)

display(
    POST_ESTABLISHED_POINT_METRICS
    .to_frame(
        "value"
    )
)


print()
print("TEST → POST_TEST POINT PERFORMANCE")
print("-" * 140)

display(
    TEST_VS_POST_POINT
)


print()
print("UNCERTAINTY — ESTABLISHED POST_TEST")
print("-" * 140)

display(
    POST_UNCERTAINTY_ESTABLISHED
)


print()
print("TEST → POST_TEST UNCERTAINTY")
print("-" * 140)

display(
    TEST_VS_POST_UNCERTAINTY
)


print()
print("ESTABLISHED POST_TEST — BY SEX")
print("-" * 140)

display(
    POST_UNCERTAINTY_BY_SEX
)


print()
print("ESTABLISHED POST_TEST — BY MODEL GROUP")
print("-" * 140)

display(
    POST_UNCERTAINTY_BY_GROUP
)


print()
print("COLD-START STATUS")
print("-" * 140)

display(
    POST_COLD_START_SUMMARY
)


print()
print("QA")
print("-" * 140)

print(
    f"Point predictions finite            : "
    f"{POINT_FINITE_PASS}"
)

print(
    f"Nonfinite established uncertainty   : "
    f"{N_NONFINITE_ESTABLISHED}"
)

print(
    f"Positive variance                   : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Interval ordering                   : "
    f"{INTERVAL_ORDER_PASS}"
)

print(
    f"Cold-start V6 point mean = V5       : "
    f"{COLD_START_V6_EQUALS_V5}"
)

print(
    f"Row accounting                      : "
    f"{ROW_ACCOUNTING_PASS}"
)


print()
print("=" * 140)

if CELL_22_PASS:

    print(
        "CELL 22 STATUS: FROZEN V6 POST_TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 22 STATUS: V6 POST_TEST EVALUATION REVIEW REQUIRED"
    )

print("=" * 140)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 13
# Original source SHA256: 45e1b65ec52f1fd66be22f2a95bd5f0727910ce3d5c7e2e5ae6c751276976188
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 23 — V6 PRODUCTION RELEASE GATE
# ==================================================================================================
#
# Purpose
# -------
# Determine whether the current V6 training run is ready to be packaged
# as a release candidate.
#
# NO fitting.
# NO recalibration.
# NO model selection.
#
#
# Hard release checks
# -------------------
#
# 1. Cells 19–22 passed.
# 2. Required model artifacts exist.
# 3. TEST established predictions are finite.
# 4. No statistically robust evidence that V6 degrades V5 on TEST:
#
#       cluster-bootstrap 95% CI upper bound must not be < 0.
#
# 5. TEST 50 / 80 / 95% uncertainty calibration is statistically
#    compatible with the nominal target using Wilson intervals.
# 6. Cold-start predictions remain explicitly outside calibrated V6
#    uncertainty.
# 7. POST_TEST technical / chronological diagnostic passed.
#
# POST_TEST is NOT used for parameter tuning.
#
# ==================================================================================================

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_19_PASS",
    "NOTEBOOK_2_CELL_20_PASS",
    "NOTEBOOK_2_CELL_21_PASS",
    "NOTEBOOK_2_CELL_22_PASS",

    "V6_TEST_PREDICTIONS",
    "V6_TEST_ESTABLISHED",
    "V6_TEST_UNCERTAINTY_ESTABLISHED",
    "V6_TEST_BOOTSTRAP_SUMMARY",

    "V6_POST_TEST_PREDICTIONS",
    "V6_POST_TEST_ESTABLISHED",
    "V6_POST_TEST_UNCERTAINTY_ESTABLISHED",

    "V6_MEAN_SPEC",
    "V6_UNCERTAINTY_SPEC",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def wilson_interval(
    successes,
    n,
    z=1.959963984540054,
):

    successes = int(successes)
    n = int(n)


    if n <= 0:

        return (
            np.nan,
            np.nan,
        )


    p = successes / n

    z2 = z ** 2


    denominator = (
        1.0
        +
        z2 / n
    )


    center = (
        p
        +
        z2 / (2.0 * n)
    ) / denominator


    half = (
        z
        *
        np.sqrt(
            (
                p * (1.0 - p) / n
            )
            +
            (
                z2
                /
                (4.0 * n ** 2)
            )
        )

        /

        denominator
    )


    return (
        center - half,
        center + half,
    )


def point_metrics(df):

    def values(error):

        e = np.asarray(
            error,
            dtype=float,
        )

        return {
            "MAE":
                float(
                    np.mean(
                        np.abs(e)
                    )
                ),

            "RMSE":
                float(
                    np.sqrt(
                        np.mean(
                            e ** 2
                        )
                    )
                ),

            "Bias":
                float(
                    np.mean(e)
                ),
        }


    v4 = values(
        df["v4_error"]
    )

    v5 = values(
        df["v5_error"]
    )

    v6 = values(
        df["v6_error"]
    )


    return {
        "V4_MAE":
            v4["MAE"],

        "V5_MAE":
            v5["MAE"],

        "V6_MAE":
            v6["MAE"],

        "V5_to_V6_MAE_gain_m":
            v5["MAE"]
            -
            v6["MAE"],

        "V4_RMSE":
            v4["RMSE"],

        "V5_RMSE":
            v5["RMSE"],

        "V6_RMSE":
            v6["RMSE"],

        "V5_to_V6_RMSE_gain_m":
            v5["RMSE"]
            -
            v6["RMSE"],

        "V6_Bias":
            v6["Bias"],
    }


# --------------------------------------------------------------------------------------------------
# 3. PREVIOUS-CELL STATUS
# --------------------------------------------------------------------------------------------------

PREVIOUS_CELLS_PASS = bool(
    NOTEBOOK_2_CELL_19_PASS
    and
    NOTEBOOK_2_CELL_20_PASS
    and
    NOTEBOOK_2_CELL_21_PASS
    and
    NOTEBOOK_2_CELL_22_PASS
)


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED RELEASE ARTIFACTS
# --------------------------------------------------------------------------------------------------

REQUIRED_ARTIFACTS = {
    "v5_ou_parameters":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v5_ou_parameters.json",

    "v5_terminal_states":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v5_train_terminal_states.parquet",

    "v6_mean_spec":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v6_mean_spec.json",

    "v6_uncertainty_spec":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v6_uncertainty_spec.json",

    "v6_test_predictions":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v6_test_predictions.parquet",

    "v6_post_test_predictions":
        Path(
            TRAINING_RUN_DIR
        )
        /
        "v6_post_test_predictions.parquet",

    "training_manifest":
        Path(
            TRAINING_MANIFEST_PATH
        ),
}


artifact_rows = []


for name, path in REQUIRED_ARTIFACTS.items():

    exists = (
        path.exists()
        and
        path.is_file()
    )


    artifact_rows.append({
        "artifact":
            name,

        "path":
            str(path),

        "exists":
            bool(exists),

        "size_bytes":
            (
                int(
                    path.stat().st_size
                )
                if exists
                else
                np.nan
            ),

        "sha256":
            (
                sha256_file(path)
                if exists
                else
                None
            ),
    })


RELEASE_ARTIFACT_AUDIT = pd.DataFrame(
    artifact_rows
)


REQUIRED_ARTIFACTS_PASS = bool(
    RELEASE_ARTIFACT_AUDIT[
        "exists"
    ]
    .all()
)


# --------------------------------------------------------------------------------------------------
# 5. TEST ESTABLISHED POINT METRICS
# --------------------------------------------------------------------------------------------------

TEST_EST_POINT = point_metrics(
    V6_TEST_ESTABLISHED
)


TEST_POINT_FINITE_PASS = bool(
    np.isfinite(
        np.asarray(
            list(
                TEST_EST_POINT.values()
            ),
            dtype=float,
        )
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 6. CLUSTER-BOOTSTRAP DEGRADATION CHECK
# --------------------------------------------------------------------------------------------------
#
# gain > 0:
#       V6 better than V5
#
# gain < 0:
#       V6 worse than V5
#
# We hard-fail only when the ENTIRE 95% CI is below zero.
# --------------------------------------------------------------------------------------------------

BOOT = (
    V6_TEST_BOOTSTRAP_SUMMARY.copy()
)


required_bootstrap_metrics = {
    "V5_to_V6_MAE_gain_m",
    "V5_to_V6_RMSE_gain_m",
}


BOOTSTRAP_ROWS_PASS = bool(
    required_bootstrap_metrics
    .issubset(
        set(
            BOOT[
                "metric"
            ]
        )
    )
)


if BOOTSTRAP_ROWS_PASS:

    mae_boot = (
        BOOT.loc[
            BOOT[
                "metric"
            ]
            .eq(
                "V5_to_V6_MAE_gain_m"
            )
        ]
        .iloc[0]
    )


    rmse_boot = (
        BOOT.loc[
            BOOT[
                "metric"
            ]
            .eq(
                "V5_to_V6_RMSE_gain_m"
            )
        ]
        .iloc[0]
    )


    MAE_ROBUST_DEGRADATION = bool(
        float(
            mae_boot[
                "ci95_hi"
            ]
        )
        <
        0
    )


    RMSE_ROBUST_DEGRADATION = bool(
        float(
            rmse_boot[
                "ci95_hi"
            ]
        )
        <
        0
    )


else:

    MAE_ROBUST_DEGRADATION = True
    RMSE_ROBUST_DEGRADATION = True


TEST_POINT_RELEASE_PASS = bool(
    BOOTSTRAP_ROWS_PASS
    and
    not MAE_ROBUST_DEGRADATION
    and
    not RMSE_ROBUST_DEGRADATION
)


# --------------------------------------------------------------------------------------------------
# 7. TEST UNCERTAINTY CALIBRATION GATE
# --------------------------------------------------------------------------------------------------
#
# Use Wilson intervals around observed coverage.
#
# The nominal target must lie inside the 95% Wilson interval.
# --------------------------------------------------------------------------------------------------

coverage_rows = []


for level in [
    0.50,
    0.80,
    0.95,
]:

    pct = int(
        level * 100
    )


    column = (
        f"v6_covered_{pct}"
    )


    if column not in V6_TEST_ESTABLISHED.columns:

        raise RuntimeError(
            f"Missing TEST interval column: {column}"
        )


    covered = (
        V6_TEST_ESTABLISHED[
            column
        ]
        .astype(bool)
    )


    n = int(
        len(
            covered
        )
    )


    successes = int(
        covered.sum()
    )


    observed = (
        successes
        /
        n
    )


    lower, upper = wilson_interval(
        successes,
        n,
    )


    target_inside = bool(
        lower
        <=
        level
        <=
        upper
    )


    coverage_rows.append({
        "interval":
            f"{pct}%",

        "target":
            level,

        "n":
            n,

        "covered":
            successes,

        "observed":
            observed,

        "wilson_95_lo":
            lower,

        "wilson_95_hi":
            upper,

        "target_inside_wilson_95":
            target_inside,
    })


TEST_COVERAGE_GATE = pd.DataFrame(
    coverage_rows
)


TEST_COVERAGE_RELEASE_PASS = bool(
    TEST_COVERAGE_GATE[
        "target_inside_wilson_95"
    ]
    .all()
)


# --------------------------------------------------------------------------------------------------
# 8. STANDARDIZED ERROR CHECK
# --------------------------------------------------------------------------------------------------

TEST_STANDARDIZED_ERROR_MEAN = float(
    V6_TEST_ESTABLISHED[
        "v6_standardized_error"
    ]
    .mean()
)


TEST_STANDARDIZED_ERROR_SD = float(
    V6_TEST_ESTABLISHED[
        "v6_standardized_error"
    ]
    .std()
)


TEST_STANDARDIZED_ERROR_FINITE_PASS = bool(
    np.isfinite(
        [
            TEST_STANDARDIZED_ERROR_MEAN,
            TEST_STANDARDIZED_ERROR_SD,
        ]
    )
    .all()

    and

    TEST_STANDARDIZED_ERROR_SD > 0
)


# --------------------------------------------------------------------------------------------------
# 9. COLD-START POLICY CHECK
# --------------------------------------------------------------------------------------------------

test_unseen = (
    ~V6_TEST_PREDICTIONS[
        "athlete_seen_in_training"
    ]
)


post_unseen = (
    ~V6_POST_TEST_PREDICTIONS[
        "athlete_seen_in_training"
    ]
)


TEST_COLD_START_POINT_PASS = bool(
    np.allclose(
        V6_TEST_PREDICTIONS.loc[
            test_unseen,
            "v6_prediction_mean",
        ],

        V6_TEST_PREDICTIONS.loc[
            test_unseen,
            "v5_prediction_mean",
        ],

        rtol=0.0,
        atol=1e-12,
    )
)


POST_COLD_START_POINT_PASS = bool(
    np.allclose(
        V6_POST_TEST_PREDICTIONS.loc[
            post_unseen,
            "v6_prediction_mean",
        ],

        V6_POST_TEST_PREDICTIONS.loc[
            post_unseen,
            "v5_prediction_mean",
        ],

        rtol=0.0,
        atol=1e-12,
    )
)


COLD_START_POLICY_PASS = bool(
    TEST_COLD_START_POINT_PASS
    and
    POST_COLD_START_POINT_PASS
)


# --------------------------------------------------------------------------------------------------
# 10. POST_TEST DIAGNOSTIC
# --------------------------------------------------------------------------------------------------
#
# POST_TEST is deliberately not used as another tuning set.
#
# We require:
#       technical validity
#       finite results
#
# We do NOT fit or alter the model based on its performance.
# --------------------------------------------------------------------------------------------------

POST_TEST_TECHNICAL_PASS = bool(
    NOTEBOOK_2_CELL_22_PASS
    and
    len(
        V6_POST_TEST_ESTABLISHED
    )
    >
    0
)


POST_TEST_POINT = point_metrics(
    V6_POST_TEST_ESTABLISHED
)


POST_TEST_POINT_FINITE_PASS = bool(
    np.isfinite(
        np.asarray(
            list(
                POST_TEST_POINT.values()
            ),
            dtype=float,
        )
    )
    .all()
)


POST_TEST_DIAGNOSTIC_PASS = bool(
    POST_TEST_TECHNICAL_PASS
    and
    POST_TEST_POINT_FINITE_PASS
)


# --------------------------------------------------------------------------------------------------
# 11. SPECIFICATION CONSISTENCY
# --------------------------------------------------------------------------------------------------

MEAN_SPEC_PASS = bool(
    isinstance(
        V6_MEAN_SPEC,
        dict,
    )

    and

    V6_MEAN_SPEC.get(
        "calibration_split"
    )
    ==
    "VALIDATION"
)


UNCERTAINTY_SPEC_PASS = bool(
    isinstance(
        V6_UNCERTAINTY_SPEC,
        dict,
    )

    and

    V6_UNCERTAINTY_SPEC.get(
        "calibration_split"
    )
    ==
    "VALIDATION"
)


SPECIFICATION_PASS = bool(
    MEAN_SPEC_PASS
    and
    UNCERTAINTY_SPEC_PASS
)


# --------------------------------------------------------------------------------------------------
# 12. RELEASE CHECK TABLE
# --------------------------------------------------------------------------------------------------

RELEASE_CHECKS = pd.DataFrame(
    [
        {
            "check":
                "Cells 19-22 passed",

            "category":
                "PIPELINE",

            "pass":
                PREVIOUS_CELLS_PASS,
        },

        {
            "check":
                "Required artifacts exist",

            "category":
                "ARTIFACT",

            "pass":
                REQUIRED_ARTIFACTS_PASS,
        },

        {
            "check":
                "Frozen model specs calibrated on VALIDATION",

            "category":
                "METHODOLOGY",

            "pass":
                SPECIFICATION_PASS,
        },

        {
            "check":
                "TEST established point metrics finite",

            "category":
                "TEST_POINT",

            "pass":
                TEST_POINT_FINITE_PASS,
        },

        {
            "check":
                "No robust TEST MAE degradation vs V5",

            "category":
                "TEST_POINT",

            "pass":
                not MAE_ROBUST_DEGRADATION,
        },

        {
            "check":
                "No robust TEST RMSE degradation vs V5",

            "category":
                "TEST_POINT",

            "pass":
                not RMSE_ROBUST_DEGRADATION,
        },

        {
            "check":
                "TEST 50/80/95 coverage compatible with nominal",

            "category":
                "TEST_UNCERTAINTY",

            "pass":
                TEST_COVERAGE_RELEASE_PASS,
        },

        {
            "check":
                "TEST standardized errors finite",

            "category":
                "TEST_UNCERTAINTY",

            "pass":
                TEST_STANDARDIZED_ERROR_FINITE_PASS,
        },

        {
            "check":
                "Cold-start regime remains explicitly separate",

            "category":
                "COLD_START",

            "pass":
                COLD_START_POLICY_PASS,
        },

        {
            "check":
                "POST_TEST technical diagnostic passed",

            "category":
                "POST_TEST",

            "pass":
                POST_TEST_DIAGNOSTIC_PASS,
        },
    ]
)


RELEASE_GATE_PASS = bool(
    RELEASE_CHECKS[
        "pass"
    ]
    .all()
)


# --------------------------------------------------------------------------------------------------
# 13. RELEASE-GATE SUMMARY
# --------------------------------------------------------------------------------------------------

RELEASE_GATE_SUMMARY = {

    "status":
        (
            "PASSED"
            if RELEASE_GATE_PASS
            else
            "FAILED"
        ),

    "release_candidate":
        bool(
            RELEASE_GATE_PASS
        ),

    "methodology": {
        "mean_calibration_split":
            "VALIDATION",

        "uncertainty_calibration_split":
            "VALIDATION",

        "frozen_test_evaluation":
            True,

        "post_test_role":
            "RECENT_PRODUCTION_DIAGNOSTIC_ONLY",

        "cold_start_uncertainty":
            "UNCALIBRATED_SEPARATE_REGIME",
    },

    "test_established": {
        **{
            key:
                float(value)

            for key, value
            in TEST_EST_POINT.items()
        },

        "standardized_error_mean":
            TEST_STANDARDIZED_ERROR_MEAN,

        "standardized_error_sd":
            TEST_STANDARDIZED_ERROR_SD,
    },

    "post_test_established": {
        key:
            float(value)

        for key, value
        in POST_TEST_POINT.items()
    },

    "bootstrap": {

        "mae_robust_degradation":
            MAE_ROBUST_DEGRADATION,

        "rmse_robust_degradation":
            RMSE_ROBUST_DEGRADATION,
    },
}


# --------------------------------------------------------------------------------------------------
# 14. SAVE RELEASE GATE
# --------------------------------------------------------------------------------------------------

V6_RELEASE_GATE_JSON_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_release_gate.json"
)


V6_RELEASE_GATE_CHECKS_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_release_gate_checks.parquet"
)


V6_RELEASE_ARTIFACT_AUDIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_release_artifact_audit.parquet"
)


V6_RELEASE_COVERAGE_GATE_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_release_test_coverage_gate.parquet"
)


with V6_RELEASE_GATE_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RELEASE_GATE_SUMMARY,
        f,
        indent=2,
        default=str,
    )


RELEASE_CHECKS.to_parquet(
    V6_RELEASE_GATE_CHECKS_PATH,
    index=False,
)


RELEASE_ARTIFACT_AUDIT.to_parquet(
    V6_RELEASE_ARTIFACT_AUDIT_PATH,
    index=False,
)


TEST_COVERAGE_GATE.to_parquet(
    V6_RELEASE_COVERAGE_GATE_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 15. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v6_release_gate"
] = {

    "status":
        (
            "PASSED"
            if RELEASE_GATE_PASS
            else
            "FAILED"
        ),

    "release_candidate":
        bool(
            RELEASE_GATE_PASS
        ),

    "gate_path":
        str(
            V6_RELEASE_GATE_JSON_PATH
        ),

    "checks_path":
        str(
            V6_RELEASE_GATE_CHECKS_PATH
        ),

    "artifact_audit_path":
        str(
            V6_RELEASE_ARTIFACT_AUDIT_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "V6_RELEASE_GATE_PASSED"
    if RELEASE_GATE_PASS
    else
    "V6_RELEASE_GATE_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 16. GLOBALS
# --------------------------------------------------------------------------------------------------

V6_RELEASE_GATE_SUMMARY = (
    RELEASE_GATE_SUMMARY
)


V6_RELEASE_GATE_CHECKS = (
    RELEASE_CHECKS
)


V6_RELEASE_ARTIFACT_AUDIT = (
    RELEASE_ARTIFACT_AUDIT
)


V6_RELEASE_TEST_COVERAGE_GATE = (
    TEST_COVERAGE_GATE
)


NOTEBOOK_2_CELL_23_PASS = (
    RELEASE_GATE_PASS
)


# --------------------------------------------------------------------------------------------------
# 17. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 145)
print("V6 TRAINING — PRODUCTION RELEASE GATE")
print("=" * 145)


print()
print("RELEASE CHECKS")
print("-" * 145)

display(
    RELEASE_CHECKS
)


print()
print("TEST POINT PERFORMANCE — ESTABLISHED")
print("-" * 145)

display(
    pd.DataFrame(
        [
            TEST_EST_POINT
        ]
    )
)


print()
print("PAIRED ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 145)

display(
    V6_TEST_BOOTSTRAP_SUMMARY
)


print()
print("TEST UNCERTAINTY CALIBRATION GATE")
print("-" * 145)

coverage_display = (
    TEST_COVERAGE_GATE.copy()
)

for column in [
    "target",
    "observed",
    "wilson_95_lo",
    "wilson_95_hi",
]:

    coverage_display[
        column
    ] = (
        100.0
        *
        coverage_display[
            column
        ]
    )


display(
    coverage_display
)


print()
print("TEST STANDARDIZED ERROR")
print("-" * 145)

print(
    f"Mean : "
    f"{TEST_STANDARDIZED_ERROR_MEAN:.6f}"
)

print(
    f"SD   : "
    f"{TEST_STANDARDIZED_ERROR_SD:.6f}"
)


print()
print("POST_TEST — ESTABLISHED DIAGNOSTIC")
print("-" * 145)

display(
    pd.DataFrame(
        [
            POST_TEST_POINT
        ]
    )
)


print()
print("REQUIRED ARTIFACT AUDIT")
print("-" * 145)

display(
    RELEASE_ARTIFACT_AUDIT[
        [
            "artifact",
            "exists",
            "size_bytes",
            "sha256",
        ]
    ]
)


print()
print("RELEASE DECISION")
print("-" * 145)

print(
    f"Release candidate : "
    f"{RELEASE_GATE_PASS}"
)


print()
print("=" * 145)

if RELEASE_GATE_PASS:

    print(
        "CELL 23 STATUS: V6 PRODUCTION RELEASE GATE PASSED"
    )

else:

    print(
        "CELL 23 STATUS: V6 PRODUCTION RELEASE GATE FAILED — REVIEW REQUIRED"
    )

print("=" * 145)



V6 TRAINING — COMPLETE FROZEN V6 TEST EVALUATION

POINT PREDICTION — OVERALL
---------------------------------------------------------------------------------------------------------------------------------------


,n,athletes,V4_MAE,V5_MAE,V6_MAE,V5_to_V6_MAE_gain_m,V4_to_V6_MAE_gain_m,V4_RMSE,V5_RMSE,V6_RMSE,V5_to_V6_RMSE_gain_m,V4_to_V6_RMSE_gain_m,V4_Bias,V5_Bias,V6_Bias
0,12865.0,5200.0,3.7357,3.274069,3.287566,-0.013497,0.448134,5.325524,4.781999,4.784199,-0.002201,0.541324,0.100082,-0.108767,0.054525



POINT PREDICTION — SEEN VS UNSEEN
---------------------------------------------------------------------------------------------------------------------------------------


,training_status,n,athletes,V4_MAE,V5_MAE,V6_MAE,V5_to_V6_MAE_gain_m,V4_to_V6_MAE_gain_m,V4_RMSE,V5_RMSE,V6_RMSE,V5_to_V6_RMSE_gain_m,V4_to_V6_RMSE_gain_m,V4_Bias,V5_Bias,V6_Bias
0,SEEN,10941.0,4023.0,2.914155,2.550114,2.565985,-0.015871,0.348171,3.802461,3.371135,3.374805,-0.003669,0.427656,0.285539,0.111497,0.303504
1,UNSEEN,1924.0,1177.0,8.407490,7.390906,7.390906,0.000000,1.016584,10.364313,9.395750,9.395750,0.000000,0.968563,-0.954533,-1.361317,-1.361317



PRIMARY — ESTABLISHED ATHLETES
---------------------------------------------------------------------------------------------------------------------------------------


...


TEST STANDARDIZED ERROR
-------------------------------------------------------------------------------------------------------------------------------------------------
Mean : 0.105709
SD   : 1.019726

POST_TEST — ESTABLISHED DIAGNOSTIC
-------------------------------------------------------------------------------------------------------------------------------------------------


,V4_MAE,V5_MAE,V6_MAE,V5_to_V6_MAE_gain_m,V4_RMSE,V5_RMSE,V6_RMSE,V5_to_V6_RMSE_gain_m,V6_Bias
0,3.227096,2.756045,2.753548,0.002497,4.126472,3.605815,3.610494,-0.004679,0.066628



REQUIRED ARTIFACT AUDIT
-------------------------------------------------------------------------------------------------------------------------------------------------


,artifact,exists,size_bytes,sha256
0,v5_ou_parameters,True,959,7b1ef7bb43bf171c4a66ac0e0cb41f3941bddbac2af50d...
1,v5_terminal_states,True,456725,6dfed0896c8fa445719ae83ba36f2fcf81f281aae1b873...
2,v6_mean_spec,True,631,2899887295e4d86bdffe55a57e9d0d0c2181953f67e5fa...
3,v6_uncertainty_spec,True,400,388ba4e0710ece252a4dccb1f482837d1dd272ac6675e9...
4,v6_test_predictions,True,4775623,375b360bbc13b215db818d85c37210b09083a10afc23a3...
5,v6_post_test_predictions,True,338935,1decac58842fd1e2613b34f96f9e72a001542a390672e6...
6,training_manifest,True,14020,08f56ae6b2fc693aaf43bac76693ae14b94367c5d863d2...



RELEASE DECISION
-------------------------------------------------------------------------------------------------------------------------------------------------
Release candidate : False

CELL 23 STATUS: V6 PRODUCTION RELEASE GATE FAILED — REVIEW REQUIRED


## Stage 09 — V6 degradation diagnostics and V6.1 candidate

Diagnose short-gap degradation, confirm the VALIDATION effect, and define the train-anchored V6.1 OU freshness blend without using TEST.

**Legacy source cells represented:** 14, 15, 16  
**Execution rule:** run only after the preceding stage has completed successfully.


In [10]:
# ==================================================================================================
# REFACTORED STAGE 09 — V6 degradation diagnostics and V6.1 candidate
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 14
# Original source SHA256: 877dd2779750e917ba5bc9ecd81fd9a6f3950f96f9dc8438f8b0b9977a29268d
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 24 — V5 → V6 MEAN DEGRADATION DIAGNOSTIC
# ==================================================================================================
#
# Purpose
# -------
# Explain the small but statistically robust TEST MAE degradation detected
# by the Cell-23 release gate.
#
# NO fitting.
# NO parameter selection.
# NO recalibration.
# NO release-rule changes.
#
#
# For an established athlete:
#
#   V5 =
#       V4 structural mean
#       + V5 transient state
#
#
#   V6 =
#       population component
#       + rho_I * athlete intercept
#       + rho_S * athlete slope component
#       + offset
#       + V5 transient state
#
#
# Therefore approximately:
#
#   V6 - V5
#
#       =
#
#   (rho_I - 1) * athlete_intercept
#   +
#   (rho_S - 1) * athlete_slope_component
#   +
#   offset
#
#
# The cell examines:
#
#   • total V6 adjustment relative to V5
#   • intercept-shrinkage contribution
#   • slope-shrinkage contribution
#   • offset contribution
#   • whether the adjustment moved toward or away from the actual result
#   • model-group effects
#   • sex effects
#   • horizon effects
#   • age effects
#   • adjustment-magnitude effects
#   • athlete-level concentration of gains/losses
#
# ==================================================================================================

import json
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_21_PASS",
    "NOTEBOOK_2_CELL_23_PASS",
    "V6_TEST_ESTABLISHED",
    "V6_TEST_BOOTSTRAP_SUMMARY",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


# Cell 23 is EXPECTED to have failed because this is a failed-release diagnostic.
if NOTEBOOK_2_CELL_23_PASS:

    print(
        "WARNING: Cell 23 passed. "
        "This diagnostic can still run, but it was designed primarily "
        "for failed-release investigation."
    )


# --------------------------------------------------------------------------------------------------
# 2. DIAGNOSTIC POPULATION
# --------------------------------------------------------------------------------------------------

D = (
    V6_TEST_ESTABLISHED.copy()
    .reset_index(
        drop=True
    )
)


required_columns = [
    "aaAthleteId",

    "mark_numeric_m",

    "v4_structural_mean",
    "v5_prediction_mean",
    "v6_prediction_mean",

    "state_prior_mean",

    "v4_athlete_intercept_component",
    "v4_athlete_slope_component",

    "v6_rho_intercept",
    "v6_rho_slope",
    "v6_offset",

    "state_gap_days",

    "sex_group",
    "model_group",
]


missing_columns = [
    x
    for x in required_columns
    if x not in D.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing TEST diagnostic columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 3. RECONSTRUCT V5 AND DECOMPOSE V6 − V5
# --------------------------------------------------------------------------------------------------

D[
    "v5_reference_reconstruction"
] = (
    D[
        "v4_structural_mean"
    ]
    +
    D[
        "state_prior_mean"
    ]
)


D[
    "v5_reconstruction_residual"
] = (
    D[
        "v5_prediction_mean"
    ]
    -
    D[
        "v5_reference_reconstruction"
    ]
)


# Contribution caused by shrinking the athlete intercept.
D[
    "v6_intercept_adjustment"
] = (
    (
        D[
            "v6_rho_intercept"
        ]
        -
        1.0
    )
    *
    D[
        "v4_athlete_intercept_component"
    ]
)


# Contribution caused by shrinking the athlete slope component.
D[
    "v6_slope_adjustment"
] = (
    (
        D[
            "v6_rho_slope"
        ]
        -
        1.0
    )
    *
    D[
        "v4_athlete_slope_component"
    ]
)


# Explicit calibrated offset.
D[
    "v6_offset_adjustment"
] = (
    D[
        "v6_offset"
    ]
)


# If V5 differs numerically from V4 + state, account for it explicitly.
D[
    "v5_reference_adjustment"
] = (
    -
    D[
        "v5_reconstruction_residual"
    ]
)


D[
    "v6_expected_adjustment_vs_v5"
] = (
    D[
        "v6_intercept_adjustment"
    ]
    +
    D[
        "v6_slope_adjustment"
    ]
    +
    D[
        "v6_offset_adjustment"
    ]
    +
    D[
        "v5_reference_adjustment"
    ]
)


D[
    "v6_actual_adjustment_vs_v5"
] = (
    D[
        "v6_prediction_mean"
    ]
    -
    D[
        "v5_prediction_mean"
    ]
)


D[
    "adjustment_reconstruction_error"
] = (
    D[
        "v6_actual_adjustment_vs_v5"
    ]
    -
    D[
        "v6_expected_adjustment_vs_v5"
    ]
)


MAX_DECOMPOSITION_ERROR = float(
    D[
        "adjustment_reconstruction_error"
    ]
    .abs()
    .max()
)


# --------------------------------------------------------------------------------------------------
# 4. ERROR IMPROVEMENT / DETERIORATION
# --------------------------------------------------------------------------------------------------

D[
    "v5_error"
] = (
    D[
        "mark_numeric_m"
    ]
    -
    D[
        "v5_prediction_mean"
    ]
)


D[
    "v6_error"
] = (
    D[
        "mark_numeric_m"
    ]
    -
    D[
        "v6_prediction_mean"
    ]
)


D[
    "v5_abs_error"
] = (
    D[
        "v5_error"
    ]
    .abs()
)


D[
    "v6_abs_error"
] = (
    D[
        "v6_error"
    ]
    .abs()
)


D[
    "mae_gain_m"
] = (
    D[
        "v5_abs_error"
    ]
    -
    D[
        "v6_abs_error"
    ]
)


D[
    "squared_error_gain_m2"
] = (
    D[
        "v5_error"
    ]
    ** 2

    -

    D[
        "v6_error"
    ]
    ** 2
)


D[
    "v6_helped"
] = (
    D[
        "mae_gain_m"
    ]
    >
    0
)


D[
    "v6_hurt"
] = (
    D[
        "mae_gain_m"
    ]
    <
    0
)


D[
    "same_abs_error"
] = (
    np.isclose(
        D[
            "mae_gain_m"
        ],
        0.0,
        atol=1e-12,
    )
)


# --------------------------------------------------------------------------------------------------
# 5. WAS V6 MOVING IN THE CORRECT DIRECTION?
# --------------------------------------------------------------------------------------------------
#
# V5 residual:
#
#       actual - V5 prediction
#
# If this residual is positive, the prediction needs to move upward.
#
# If:
#
#       (V6 - V5) * V5 residual > 0
#
# the V6 structural adjustment moved in the correct direction.
# --------------------------------------------------------------------------------------------------

D[
    "direction_product"
] = (
    D[
        "v6_actual_adjustment_vs_v5"
    ]
    *
    D[
        "v5_error"
    ]
)


D[
    "adjustment_correct_direction"
] = (
    D[
        "direction_product"
    ]
    >
    0
)


D[
    "adjustment_wrong_direction"
] = (
    D[
        "direction_product"
    ]
    <
    0
)


# Same direction as residual but so large it passes the actual result
# far enough to increase absolute error.
D[
    "correct_direction_but_overshot"
] = (
    D[
        "adjustment_correct_direction"
    ]
    &
    (
        D[
            "v6_actual_adjustment_vs_v5"
        ]
        .abs()

        >

        2.0
        *
        D[
            "v5_error"
        ]
        .abs()
    )
)


# --------------------------------------------------------------------------------------------------
# 6. OVERALL SUMMARY
# --------------------------------------------------------------------------------------------------

V5_MAE = float(
    D[
        "v5_abs_error"
    ]
    .mean()
)


V6_MAE = float(
    D[
        "v6_abs_error"
    ]
    .mean()
)


V5_RMSE = float(
    np.sqrt(
        np.mean(
            D[
                "v5_error"
            ]
            ** 2
        )
    )
)


V6_RMSE = float(
    np.sqrt(
        np.mean(
            D[
                "v6_error"
            ]
            ** 2
        )
    )
)


OVERALL_DIAGNOSTIC = pd.DataFrame(
    [
        {
            "rows":
                int(
                    len(D)
                ),

            "athletes":
                int(
                    D[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "MAE_gain_m":
                V5_MAE
                -
                V6_MAE,

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "RMSE_gain_m":
                V5_RMSE
                -
                V6_RMSE,

            "mean_V6_adjustment_m":
                float(
                    D[
                        "v6_actual_adjustment_vs_v5"
                    ]
                    .mean()
                ),

            "mean_abs_V6_adjustment_m":
                float(
                    D[
                        "v6_actual_adjustment_vs_v5"
                    ]
                    .abs()
                    .mean()
                ),

            "median_abs_V6_adjustment_m":
                float(
                    D[
                        "v6_actual_adjustment_vs_v5"
                    ]
                    .abs()
                    .median()
                ),

            "rows_helped_pct":
                float(
                    100
                    *
                    D[
                        "v6_helped"
                    ]
                    .mean()
                ),

            "rows_hurt_pct":
                float(
                    100
                    *
                    D[
                        "v6_hurt"
                    ]
                    .mean()
                ),

            "correct_direction_pct":
                float(
                    100
                    *
                    D[
                        "adjustment_correct_direction"
                    ]
                    .mean()
                ),

            "wrong_direction_pct":
                float(
                    100
                    *
                    D[
                        "adjustment_wrong_direction"
                    ]
                    .mean()
                ),

            "overshoot_pct":
                float(
                    100
                    *
                    D[
                        "correct_direction_but_overshot"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. COMPONENT DECOMPOSITION
# --------------------------------------------------------------------------------------------------

component_rows = []


for label, column in [
    (
        "Intercept shrinkage",
        "v6_intercept_adjustment",
    ),
    (
        "Slope shrinkage",
        "v6_slope_adjustment",
    ),
    (
        "Offset",
        "v6_offset_adjustment",
    ),
    (
        "V5 reconstruction difference",
        "v5_reference_adjustment",
    ),
    (
        "Total V6 adjustment",
        "v6_actual_adjustment_vs_v5",
    ),
]:

    x = (
        D[
            column
        ]
        .to_numpy(
            dtype=float
        )
    )


    component_rows.append(
        {
            "component":
                label,

            "mean_m":
                float(
                    np.mean(x)
                ),

            "mean_abs_m":
                float(
                    np.mean(
                        np.abs(x)
                    )
                ),

            "median_m":
                float(
                    np.median(x)
                ),

            "sd_m":
                float(
                    np.std(
                        x,
                        ddof=1,
                    )
                ),

            "p05_m":
                float(
                    np.quantile(
                        x,
                        0.05,
                    )
                ),

            "p95_m":
                float(
                    np.quantile(
                        x,
                        0.95,
                    )
                ),
        }
    )


COMPONENT_DECOMPOSITION = pd.DataFrame(
    component_rows
)


# --------------------------------------------------------------------------------------------------
# 8. SUBGROUP SUMMARY HELPER
# --------------------------------------------------------------------------------------------------

def diagnostic_summary(
    df,
    group_column,
):

    rows = []


    for group_value, g in df.groupby(
        group_column,
        observed=True,
        sort=False,
    ):

        v5_mae = float(
            g[
                "v5_abs_error"
            ]
            .mean()
        )


        v6_mae = float(
            g[
                "v6_abs_error"
            ]
            .mean()
        )


        v5_rmse = float(
            np.sqrt(
                np.mean(
                    g[
                        "v5_error"
                    ]
                    ** 2
                )
            )
        )


        v6_rmse = float(
            np.sqrt(
                np.mean(
                    g[
                        "v6_error"
                    ]
                    ** 2
                )
            )
        )


        rows.append(
            {
                group_column:
                    group_value,

                "n":
                    int(
                        len(g)
                    ),

                "athletes":
                    int(
                        g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "V5_MAE":
                    v5_mae,

                "V6_MAE":
                    v6_mae,

                "MAE_gain_m":
                    v5_mae
                    -
                    v6_mae,

                "V5_RMSE":
                    v5_rmse,

                "V6_RMSE":
                    v6_rmse,

                "RMSE_gain_m":
                    v5_rmse
                    -
                    v6_rmse,

                "mean_adjustment_m":
                    float(
                        g[
                            "v6_actual_adjustment_vs_v5"
                        ]
                        .mean()
                    ),

                "mean_abs_adjustment_m":
                    float(
                        g[
                            "v6_actual_adjustment_vs_v5"
                        ]
                        .abs()
                        .mean()
                    ),

                "mean_intercept_adjustment_m":
                    float(
                        g[
                            "v6_intercept_adjustment"
                        ]
                        .mean()
                    ),

                "mean_slope_adjustment_m":
                    float(
                        g[
                            "v6_slope_adjustment"
                        ]
                        .mean()
                    ),

                "mean_offset_m":
                    float(
                        g[
                            "v6_offset_adjustment"
                        ]
                        .mean()
                    ),

                "rows_helped_pct":
                    float(
                        100
                        *
                        g[
                            "v6_helped"
                        ]
                        .mean()
                    ),

                "rows_hurt_pct":
                    float(
                        100
                        *
                        g[
                            "v6_hurt"
                        ]
                        .mean()
                    ),

                "correct_direction_pct":
                    float(
                        100
                        *
                        g[
                            "adjustment_correct_direction"
                        ]
                        .mean()
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# --------------------------------------------------------------------------------------------------
# 9. BY SEX
# --------------------------------------------------------------------------------------------------

DIAGNOSTIC_BY_SEX = diagnostic_summary(
    D,
    "sex_group",
)


# --------------------------------------------------------------------------------------------------
# 10. BY MODEL GROUP
# --------------------------------------------------------------------------------------------------

DIAGNOSTIC_BY_GROUP = diagnostic_summary(
    D,
    "model_group",
)


# --------------------------------------------------------------------------------------------------
# 11. BY STATE-GAP / FORECAST-HISTORY HORIZON
# --------------------------------------------------------------------------------------------------

D[
    "state_gap_band"
] = pd.cut(
    D[
        "state_gap_days"
    ],
    bins=[
        0,
        30,
        60,
        90,
        120,
        180,
        365,
        730,
        1825,
        np.inf,
    ],
    labels=[
        "01-30",
        "31-60",
        "61-90",
        "91-120",
        "121-180",
        "181-365",
        "366-730",
        "731-1825",
        "1826+",
    ],
    include_lowest=True,
)


DIAGNOSTIC_BY_GAP = diagnostic_summary(
    D,
    "state_gap_band",
)


# --------------------------------------------------------------------------------------------------
# 12. BY AGE
# --------------------------------------------------------------------------------------------------

DIAGNOSTIC_BY_AGE = None


if "age_years" in D.columns:

    D[
        "age_band"
    ] = pd.cut(
        D[
            "age_years"
        ],
        bins=[
            13,
            16,
            18,
            20,
            23,
            26,
            30,
            35.000001,
        ],
        labels=[
            "13-15",
            "16-17",
            "18-19",
            "20-22",
            "23-25",
            "26-29",
            "30-35",
        ],
        include_lowest=True,
        right=False,
    )


    DIAGNOSTIC_BY_AGE = diagnostic_summary(
        D,
        "age_band",
    )


# --------------------------------------------------------------------------------------------------
# 13. BY MAGNITUDE OF THE V6 ADJUSTMENT
# --------------------------------------------------------------------------------------------------

D[
    "abs_v6_adjustment"
] = (
    D[
        "v6_actual_adjustment_vs_v5"
    ]
    .abs()
)


try:

    D[
        "adjustment_magnitude_quintile"
    ] = pd.qcut(
        D[
            "abs_v6_adjustment"
        ],
        q=5,
        labels=[
            "Q1 smallest",
            "Q2",
            "Q3",
            "Q4",
            "Q5 largest",
        ],
        duplicates="drop",
    )


    DIAGNOSTIC_BY_ADJUSTMENT_SIZE = (
        diagnostic_summary(
            D,
            "adjustment_magnitude_quintile",
        )
    )


except ValueError:

    DIAGNOSTIC_BY_ADJUSTMENT_SIZE = (
        pd.DataFrame()
    )


# --------------------------------------------------------------------------------------------------
# 14. DIRECTION OF STRUCTURAL ADJUSTMENT
# --------------------------------------------------------------------------------------------------

D[
    "adjustment_direction_class"
] = np.select(
    [
        D[
            "adjustment_wrong_direction"
        ],

        D[
            "correct_direction_but_overshot"
        ],

        D[
            "adjustment_correct_direction"
        ],
    ],
    [
        "WRONG_DIRECTION",
        "CORRECT_BUT_OVERSHOT",
        "CORRECT_DIRECTION",
    ],
    default=
        "ZERO_OR_UNDEFINED",
)


DIAGNOSTIC_BY_DIRECTION = (
    diagnostic_summary(
        D,
        "adjustment_direction_class",
    )
)


# --------------------------------------------------------------------------------------------------
# 15. ATHLETE-LEVEL CONCENTRATION
# --------------------------------------------------------------------------------------------------

athlete_rows = []


for athlete_id, g in D.groupby(
    "aaAthleteId",
    sort=False,
):

    athlete_rows.append(
        {
            "aaAthleteId":
                athlete_id,

            "n":
                int(
                    len(g)
                ),

            "sex_group":
                g[
                    "sex_group"
                ]
                .iloc[0],

            "mean_age_years":
                (
                    float(
                        g[
                            "age_years"
                        ]
                        .mean()
                    )
                    if "age_years" in g.columns
                    else
                    np.nan
                ),

            "V5_MAE":
                float(
                    g[
                        "v5_abs_error"
                    ]
                    .mean()
                ),

            "V6_MAE":
                float(
                    g[
                        "v6_abs_error"
                    ]
                    .mean()
                ),

            "MAE_gain_m":
                float(
                    g[
                        "mae_gain_m"
                    ]
                    .mean()
                ),

            "mean_adjustment_m":
                float(
                    g[
                        "v6_actual_adjustment_vs_v5"
                    ]
                    .mean()
                ),

            "mean_abs_adjustment_m":
                float(
                    g[
                        "v6_actual_adjustment_vs_v5"
                    ]
                    .abs()
                    .mean()
                ),

            "rows_helped_pct":
                float(
                    100
                    *
                    g[
                        "v6_helped"
                    ]
                    .mean()
                ),
        }
    )


ATHLETE_DIAGNOSTIC = pd.DataFrame(
    athlete_rows
)


ATHLETE_HELPED_PCT = float(
    100
    *
    (
        ATHLETE_DIAGNOSTIC[
            "MAE_gain_m"
        ]
        >
        0
    )
    .mean()
)


ATHLETE_HURT_PCT = float(
    100
    *
    (
        ATHLETE_DIAGNOSTIC[
            "MAE_gain_m"
        ]
        <
        0
    )
    .mean()
)


MOST_HURT_ATHLETES = (
    ATHLETE_DIAGNOSTIC.sort_values(
        "MAE_gain_m",
        ascending=True,
    )
    .head(15)
    .reset_index(drop=True)
)


MOST_HELPED_ATHLETES = (
    ATHLETE_DIAGNOSTIC.sort_values(
        "MAE_gain_m",
        ascending=False,
    )
    .head(15)
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 16. ASSOCIATION DIAGNOSTICS
# --------------------------------------------------------------------------------------------------

ASSOCIATION_DIAGNOSTICS = pd.DataFrame(
    [
        {
            "relationship":
                "V6 adjustment vs V5 residual",

            "correlation":
                float(
                    D[
                        [
                            "v6_actual_adjustment_vs_v5",
                            "v5_error",
                        ]
                    ]
                    .corr()
                    .iloc[
                        0,
                        1
                    ]
                ),
        },

        {
            "relationship":
                "|V6 adjustment| vs MAE gain",

            "correlation":
                float(
                    D[
                        [
                            "abs_v6_adjustment",
                            "mae_gain_m",
                        ]
                    ]
                    .corr()
                    .iloc[
                        0,
                        1
                    ]
                ),
        },

        {
            "relationship":
                "Intercept adjustment vs MAE gain",

            "correlation":
                float(
                    D[
                        [
                            "v6_intercept_adjustment",
                            "mae_gain_m",
                        ]
                    ]
                    .corr()
                    .iloc[
                        0,
                        1
                    ]
                ),
        },

        {
            "relationship":
                "Slope adjustment vs MAE gain",

            "correlation":
                float(
                    D[
                        [
                            "v6_slope_adjustment",
                            "mae_gain_m",
                        ]
                    ]
                    .corr()
                    .iloc[
                        0,
                        1
                    ]
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. QA
# --------------------------------------------------------------------------------------------------

DECOMPOSITION_PASS = bool(
    MAX_DECOMPOSITION_ERROR
    <
    1e-9
)


FINITE_DIAGNOSTIC_PASS = bool(
    np.isfinite(
        D[
            [
                "v5_error",
                "v6_error",
                "mae_gain_m",

                "v6_intercept_adjustment",
                "v6_slope_adjustment",
                "v6_offset_adjustment",

                "v6_actual_adjustment_vs_v5",
                "v6_expected_adjustment_vs_v5",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


ROW_ACCOUNTING_PASS = bool(
    len(D)
    ==
    len(
        V6_TEST_ESTABLISHED
    )
)


CELL_24_PASS = bool(
    DECOMPOSITION_PASS
    and
    FINITE_DIAGNOSTIC_PASS
    and
    ROW_ACCOUNTING_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. SAVE
# --------------------------------------------------------------------------------------------------

V6_MEAN_DIAGNOSTIC_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_mean_degradation_diagnostic.parquet"
)


V6_MEAN_DIAGNOSTIC_GROUP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_mean_degradation_by_group.parquet"
)


V6_MEAN_DIAGNOSTIC_GAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_mean_degradation_by_gap.parquet"
)


V6_MEAN_DIAGNOSTIC_ATHLETE_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_mean_degradation_by_athlete.parquet"
)


V6_MEAN_DIAGNOSTIC_SUMMARY_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_mean_degradation_diagnostic.json"
)


D.to_parquet(
    V6_MEAN_DIAGNOSTIC_PATH,
    index=False,
)


DIAGNOSTIC_BY_GROUP.to_parquet(
    V6_MEAN_DIAGNOSTIC_GROUP_PATH,
    index=False,
)


DIAGNOSTIC_BY_GAP.to_parquet(
    V6_MEAN_DIAGNOSTIC_GAP_PATH,
    index=False,
)


ATHLETE_DIAGNOSTIC.to_parquet(
    V6_MEAN_DIAGNOSTIC_ATHLETE_PATH,
    index=False,
)


diagnostic_json = {
    "status":
        (
            "PASSED"
            if CELL_24_PASS
            else
            "FAILED"
        ),

    "purpose":
        "FAILED_RELEASE_FORENSIC_DIAGNOSTIC",

    "no_fitting":
        True,

    "no_parameter_selection":
        True,

    "test_rows":
        int(
            len(D)
        ),

    "test_athletes":
        int(
            D[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "V5_MAE":
        V5_MAE,

    "V6_MAE":
        V6_MAE,

    "MAE_gain_m":
        V5_MAE
        -
        V6_MAE,

    "V5_RMSE":
        V5_RMSE,

    "V6_RMSE":
        V6_RMSE,

    "RMSE_gain_m":
        V5_RMSE
        -
        V6_RMSE,

    "athletes_helped_pct":
        ATHLETE_HELPED_PCT,

    "athletes_hurt_pct":
        ATHLETE_HURT_PCT,

    "max_decomposition_error":
        MAX_DECOMPOSITION_ERROR,
}


with V6_MEAN_DIAGNOSTIC_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        diagnostic_json,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 19. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v6_failed_release_mean_diagnostic"
] = diagnostic_json


# Important:
# Do NOT change the failed release-gate status to PASSED.
TRAINING_MANIFEST[
    "status"
] = (
    "V6_RELEASE_GATE_FAILED_DIAGNOSED"
    if CELL_24_PASS
    else
    "V6_RELEASE_DIAGNOSTIC_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 20. GLOBALS
# --------------------------------------------------------------------------------------------------

V6_MEAN_DEGRADATION_DIAGNOSTIC = D

V6_MEAN_DEGRADATION_BY_SEX = (
    DIAGNOSTIC_BY_SEX
)

V6_MEAN_DEGRADATION_BY_GROUP = (
    DIAGNOSTIC_BY_GROUP
)

V6_MEAN_DEGRADATION_BY_GAP = (
    DIAGNOSTIC_BY_GAP
)

V6_MEAN_DEGRADATION_BY_AGE = (
    DIAGNOSTIC_BY_AGE
)

V6_MEAN_DEGRADATION_BY_ADJUSTMENT_SIZE = (
    DIAGNOSTIC_BY_ADJUSTMENT_SIZE
)

V6_MEAN_DEGRADATION_BY_DIRECTION = (
    DIAGNOSTIC_BY_DIRECTION
)

V6_MEAN_DEGRADATION_BY_ATHLETE = (
    ATHLETE_DIAGNOSTIC
)

NOTEBOOK_2_CELL_24_PASS = (
    CELL_24_PASS
)


# --------------------------------------------------------------------------------------------------
# 21. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 145)
print("V6 TRAINING — V5 → V6 MEAN DEGRADATION DIAGNOSTIC")
print("=" * 145)


print()
print("OVERALL")
print("-" * 145)

display(
    OVERALL_DIAGNOSTIC
)


print()
print("V6 ADJUSTMENT DECOMPOSITION")
print("-" * 145)

display(
    COMPONENT_DECOMPOSITION
)


print()
print("BY SEX")
print("-" * 145)

display(
    DIAGNOSTIC_BY_SEX
)


print()
print("BY MODEL GROUP")
print("-" * 145)

display(
    DIAGNOSTIC_BY_GROUP
)


print()
print("BY STATE GAP")
print("-" * 145)

display(
    DIAGNOSTIC_BY_GAP
)


if DIAGNOSTIC_BY_AGE is not None:

    print()
    print("BY AGE")
    print("-" * 145)

    display(
        DIAGNOSTIC_BY_AGE
    )


print()
print("BY V6 ADJUSTMENT MAGNITUDE")
print("-" * 145)

display(
    DIAGNOSTIC_BY_ADJUSTMENT_SIZE
)


print()
print("BY ADJUSTMENT DIRECTION")
print("-" * 145)

display(
    DIAGNOSTIC_BY_DIRECTION
)


print()
print("ATHLETE-LEVEL DISTRIBUTION")
print("-" * 145)

print(
    f"Athletes helped : "
    f"{ATHLETE_HELPED_PCT:.2f}%"
)

print(
    f"Athletes hurt   : "
    f"{ATHLETE_HURT_PCT:.2f}%"
)


print()
print("MOST HURT ATHLETES")
print("-" * 145)

display(
    MOST_HURT_ATHLETES
)


print()
print("MOST HELPED ATHLETES")
print("-" * 145)

display(
    MOST_HELPED_ATHLETES
)


print()
print("ASSOCIATION DIAGNOSTICS")
print("-" * 145)

display(
    ASSOCIATION_DIAGNOSTICS
)


print()
print("QA")
print("-" * 145)

print(
    f"Maximum decomposition error : "
    f"{MAX_DECOMPOSITION_ERROR:.12e}"
)

print(
    f"Decomposition identity       : "
    f"{DECOMPOSITION_PASS}"
)

print(
    f"Finite diagnostics           : "
    f"{FINITE_DIAGNOSTIC_PASS}"
)

print(
    f"Row accounting               : "
    f"{ROW_ACCOUNTING_PASS}"
)


print()
print("=" * 145)

if CELL_24_PASS:

    print(
        "CELL 24 STATUS: V5 → V6 MEAN DEGRADATION DIAGNOSTIC PASSED"
    )

else:

    print(
        "CELL 24 STATUS: V5 → V6 MEAN DIAGNOSTIC REVIEW REQUIRED"
    )

print("=" * 145)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 15
# Original source SHA256: db247211ec42db10f949f37b623556ae3e6413fa91dafeeada763887194db473
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 25 — VALIDATION CONFIRMATION OF SHORT-GAP V6 EFFECT
# ==================================================================================================
#
# Purpose
# -------
# Test whether the TEST finding from Cell 24 is also present in VALIDATION:
#
#       fresh V5 state  -> V6 structural adjustment may hurt
#       stale V5 state  -> V6 structural adjustment may help
#
#
# IMPORTANT
# ---------
# This is DIAGNOSTIC ONLY.
#
# NO fitting.
# NO new persistence parameters.
# NO new cutoff selection.
# NO TEST-based tuning.
#
#
# Uses:
#       all rolling established-athlete VALIDATION rows
#
# not just the first validation date per athlete used by Cell 19.
#
#
# Primary quantities
# ------------------
#
#       MAE_gain = MAE_V5 - MAE_V6
#
#       positive -> V6 better
#       negative -> V6 worse
#
#
# Pre-existing horizon bands:
#
#       01-30
#       31-60
#       61-90
#       91-120
#       121-180
#       181-365
#       366-730
#       731-1825
#       1826+
#
#
# Also reports:
#
#       FRESH = 1-30 days
#       STALE = >30 days
#
# with paired athlete-cluster bootstrap confidence intervals.
#
# ==================================================================================================

import json
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_20_PASS",
    "NOTEBOOK_2_CELL_24_PASS",

    "V6_VALIDATION_UNCERTAINTY",

    "V6_SELECTED_PERSISTENCE_MODEL",
    "V6_SELECTED_PERSISTENCE_PARAMS",

    "V6_OFFSET_STRUCTURE",
    "V6_OFFSET_LOOKUP",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


if not NOTEBOOK_2_CELL_20_PASS:

    raise RuntimeError(
        "Cell 20 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. VALIDATION POPULATION
# --------------------------------------------------------------------------------------------------

V = (
    V6_VALIDATION_UNCERTAINTY.copy()
    .reset_index(
        drop=True
    )
)


if len(V) == 0:

    raise RuntimeError(
        "No VALIDATION rows available."
    )


if not V[
    "athlete_seen_in_training"
].all():

    raise RuntimeError(
        "V6_VALIDATION_UNCERTAINTY should contain only TRAIN-seen athletes."
    )


if V[
    "state_gap_days"
].isna().any():

    raise RuntimeError(
        "Established VALIDATION row has missing state gap."
    )


# --------------------------------------------------------------------------------------------------
# 3. FROZEN V6 PERSISTENCE HELPER
# --------------------------------------------------------------------------------------------------

def validation_v6_persistence(h):

    h = np.asarray(
        h,
        dtype=float,
    )


    p = np.asarray(
        V6_SELECTED_PERSISTENCE_PARAMS,
        dtype=float,
    )


    model = (
        V6_SELECTED_PERSISTENCE_MODEL
    )


    if model == "M0_CONST_CONST":

        return (
            np.full(
                len(h),
                p[0],
            ),

            np.full(
                len(h),
                p[1],
            ),
        )


    if model == "M1_CONST_EXP":

        return (
            np.full(
                len(h),
                p[0],
            ),

            p[1]
            *
            np.exp(
                -h
                /
                p[2]
            ),
        )


    if model == "M2_EXP_CONST":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            np.full(
                len(h),
                p[2],
            ),
        )


    if model == "M3_EXP_EXP":

        return (
            p[0]
            *
            np.exp(
                -h
                /
                p[1]
            ),

            p[2]
            *
            np.exp(
                -h
                /
                p[3]
            ),
        )


    raise ValueError(
        f"Unknown persistence model: {model}"
    )


# --------------------------------------------------------------------------------------------------
# 4. FROZEN OFFSET HELPER
# --------------------------------------------------------------------------------------------------

def validation_v6_offset(df):

    if V6_OFFSET_STRUCTURE == "NONE":

        return np.zeros(
            len(df),
            dtype=float,
        )


    if V6_OFFSET_STRUCTURE == "COMMON":

        return np.full(
            len(df),
            float(
                V6_OFFSET_LOOKUP[
                    "ALL"
                ]
            ),
        )


    if V6_OFFSET_STRUCTURE == "SEX":

        key = "sex_group"


    elif V6_OFFSET_STRUCTURE == "GROUP":

        key = "model_group"


    else:

        raise ValueError(
            f"Unknown offset structure: "
            f"{V6_OFFSET_STRUCTURE}"
        )


    values = (
        df[
            key
        ]
        .astype(str)
        .map(
            V6_OFFSET_LOOKUP
        )
    )


    if values.isna().any():

        raise RuntimeError(
            "Missing V6 offset."
        )


    return values.to_numpy(
        dtype=float
    )


# --------------------------------------------------------------------------------------------------
# 5. RECONSTRUCT V6 STRUCTURAL ADJUSTMENT
# --------------------------------------------------------------------------------------------------

rho_i, rho_s = (
    validation_v6_persistence(
        V[
            "state_gap_days"
        ]
        .to_numpy(
            dtype=float
        )
    )
)


offset = validation_v6_offset(
    V
)


V[
    "validation_rho_intercept"
] = rho_i


V[
    "validation_rho_slope"
] = rho_s


V[
    "validation_offset"
] = offset


V[
    "intercept_adjustment"
] = (
    (
        rho_i
        -
        1.0
    )

    *

    V[
        "v4_athlete_intercept_component"
    ]
    .to_numpy(
        dtype=float
    )
)


V[
    "slope_adjustment"
] = (
    (
        rho_s
        -
        1.0
    )

    *

    V[
        "v4_athlete_slope_component"
    ]
    .to_numpy(
        dtype=float
    )
)


V[
    "offset_adjustment"
] = offset


V[
    "actual_adjustment_vs_v5"
] = (
    V[
        "v6_prediction_mean"
    ]

    -

    V[
        "v5_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. ERRORS AND ROW-LEVEL GAIN
# --------------------------------------------------------------------------------------------------

V[
    "v5_error"
] = (
    V[
        "mark_numeric_m"
    ]

    -

    V[
        "v5_prediction_mean"
    ]
)


V[
    "v6_error"
] = (
    V[
        "mark_numeric_m"
    ]

    -

    V[
        "v6_prediction_mean"
    ]
)


V[
    "v5_abs_error"
] = (
    V[
        "v5_error"
    ]
    .abs()
)


V[
    "v6_abs_error"
] = (
    V[
        "v6_error"
    ]
    .abs()
)


V[
    "mae_gain_m"
] = (
    V[
        "v5_abs_error"
    ]

    -

    V[
        "v6_abs_error"
    ]
)


V[
    "v5_sq_error"
] = (
    V[
        "v5_error"
    ]
    **
    2
)


V[
    "v6_sq_error"
] = (
    V[
        "v6_error"
    ]
    **
    2
)


V[
    "direction_product"
] = (
    V[
        "actual_adjustment_vs_v5"
    ]

    *

    V[
        "v5_error"
    ]
)


V[
    "correct_direction"
] = (
    V[
        "direction_product"
    ]
    >
    0
)


V[
    "wrong_direction"
] = (
    V[
        "direction_product"
    ]
    <
    0
)


# --------------------------------------------------------------------------------------------------
# 7. PRE-EXISTING STATE-GAP BANDS
# --------------------------------------------------------------------------------------------------

V[
    "state_gap_band"
] = pd.cut(
    V[
        "state_gap_days"
    ],

    bins=[
        0,
        30,
        60,
        90,
        120,
        180,
        365,
        730,
        1825,
        np.inf,
    ],

    labels=[
        "01-30",
        "31-60",
        "61-90",
        "91-120",
        "121-180",
        "181-365",
        "366-730",
        "731-1825",
        "1826+",
    ],

    include_lowest=True,
)


V[
    "fresh_stale_regime"
] = np.where(
    V[
        "state_gap_days"
    ]
    <=
    30,

    "FRESH_01_30",

    "STALE_31_PLUS",
)


# --------------------------------------------------------------------------------------------------
# 8. SUMMARY HELPER
# --------------------------------------------------------------------------------------------------

def validation_summary(
    df,
    group_column,
):

    rows = []


    for group_value, g in df.groupby(
        group_column,
        observed=True,
        sort=False,
    ):

        v5_mae = float(
            g[
                "v5_abs_error"
            ]
            .mean()
        )


        v6_mae = float(
            g[
                "v6_abs_error"
            ]
            .mean()
        )


        v5_rmse = float(
            np.sqrt(
                g[
                    "v5_sq_error"
                ]
                .mean()
            )
        )


        v6_rmse = float(
            np.sqrt(
                g[
                    "v6_sq_error"
                ]
                .mean()
            )
        )


        rows.append(
            {
                group_column:
                    group_value,

                "n":
                    int(
                        len(g)
                    ),

                "athletes":
                    int(
                        g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "V5_MAE":
                    v5_mae,

                "V6_MAE":
                    v6_mae,

                "MAE_gain_m":
                    v5_mae
                    -
                    v6_mae,

                "V5_RMSE":
                    v5_rmse,

                "V6_RMSE":
                    v6_rmse,

                "RMSE_gain_m":
                    v5_rmse
                    -
                    v6_rmse,

                "mean_adjustment_m":
                    float(
                        g[
                            "actual_adjustment_vs_v5"
                        ]
                        .mean()
                    ),

                "mean_abs_adjustment_m":
                    float(
                        g[
                            "actual_adjustment_vs_v5"
                        ]
                        .abs()
                        .mean()
                    ),

                "mean_intercept_adjustment_m":
                    float(
                        g[
                            "intercept_adjustment"
                        ]
                        .mean()
                    ),

                "mean_slope_adjustment_m":
                    float(
                        g[
                            "slope_adjustment"
                        ]
                        .mean()
                    ),

                "mean_offset_m":
                    float(
                        g[
                            "offset_adjustment"
                        ]
                        .mean()
                    ),

                "rows_helped_pct":
                    float(
                        100.0
                        *
                        (
                            g[
                                "mae_gain_m"
                            ]
                            >
                            0
                        )
                        .mean()
                    ),

                "rows_hurt_pct":
                    float(
                        100.0
                        *
                        (
                            g[
                                "mae_gain_m"
                            ]
                            <
                            0
                        )
                        .mean()
                    ),

                "correct_direction_pct":
                    float(
                        100.0
                        *
                        g[
                            "correct_direction"
                        ]
                        .mean()
                    ),

                "wrong_direction_pct":
                    float(
                        100.0
                        *
                        g[
                            "wrong_direction"
                        ]
                        .mean()
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# --------------------------------------------------------------------------------------------------
# 9. VALIDATION RESULTS BY GAP
# --------------------------------------------------------------------------------------------------

VALIDATION_BY_GAP = (
    validation_summary(
        V,
        "state_gap_band",
    )
)


VALIDATION_FRESH_STALE = (
    validation_summary(
        V,
        "fresh_stale_regime",
    )
)


# --------------------------------------------------------------------------------------------------
# 10. PAIRED ATHLETE-CLUSTER BOOTSTRAP HELPER
# --------------------------------------------------------------------------------------------------

def athlete_cluster_bootstrap(
    df,
    B=2000,
    seed=42,
):

    cluster = (
        df.groupby(
            "aaAthleteId",
            as_index=False,
        )
        .agg(
            n=(
                "v5_error",
                "size",
            ),

            v5_abs=(
                "v5_abs_error",
                "sum",
            ),

            v6_abs=(
                "v6_abs_error",
                "sum",
            ),

            v5_sq=(
                "v5_sq_error",
                "sum",
            ),

            v6_sq=(
                "v6_sq_error",
                "sum",
            ),
        )
    )


    if len(cluster) == 0:

        return {
            "athletes": 0,

            "mae_gain":
                np.nan,

            "mae_ci95_lo":
                np.nan,

            "mae_ci95_hi":
                np.nan,

            "mae_positive_pct":
                np.nan,

            "rmse_gain":
                np.nan,

            "rmse_ci95_lo":
                np.nan,

            "rmse_ci95_hi":
                np.nan,

            "rmse_positive_pct":
                np.nan,
        }


    n_cluster = len(cluster)


    c_n = (
        cluster[
            "n"
        ]
        .to_numpy(
            dtype=float
        )
    )


    c_v5_abs = (
        cluster[
            "v5_abs"
        ]
        .to_numpy(
            dtype=float
        )
    )


    c_v6_abs = (
        cluster[
            "v6_abs"
        ]
        .to_numpy(
            dtype=float
        )
    )


    c_v5_sq = (
        cluster[
            "v5_sq"
        ]
        .to_numpy(
            dtype=float
        )
    )


    c_v6_sq = (
        cluster[
            "v6_sq"
        ]
        .to_numpy(
            dtype=float
        )
    )


    total_n = c_n.sum()


    observed_mae_gain = float(
        (
            c_v5_abs.sum()
            -
            c_v6_abs.sum()
        )
        /
        total_n
    )


    observed_rmse_gain = float(
        np.sqrt(
            c_v5_sq.sum()
            /
            total_n
        )
        -
        np.sqrt(
            c_v6_sq.sum()
            /
            total_n
        )
    )


    rng = np.random.default_rng(
        seed
    )


    mae_draws = np.empty(
        B
    )


    rmse_draws = np.empty(
        B
    )


    for b in range(B):

        idx = rng.integers(
            0,
            n_cluster,
            size=n_cluster,
        )


        n_b = (
            c_n[
                idx
            ]
            .sum()
        )


        mae5 = (
            c_v5_abs[
                idx
            ]
            .sum()

            /

            n_b
        )


        mae6 = (
            c_v6_abs[
                idx
            ]
            .sum()

            /

            n_b
        )


        rmse5 = np.sqrt(
            c_v5_sq[
                idx
            ]
            .sum()

            /

            n_b
        )


        rmse6 = np.sqrt(
            c_v6_sq[
                idx
            ]
            .sum()

            /

            n_b
        )


        mae_draws[
            b
        ] = (
            mae5
            -
            mae6
        )


        rmse_draws[
            b
        ] = (
            rmse5
            -
            rmse6
        )


    return {
        "athletes":
            int(
                n_cluster
            ),

        "mae_gain":
            observed_mae_gain,

        "mae_ci95_lo":
            float(
                np.quantile(
                    mae_draws,
                    0.025,
                )
            ),

        "mae_ci95_hi":
            float(
                np.quantile(
                    mae_draws,
                    0.975,
                )
            ),

        "mae_positive_pct":
            float(
                100.0
                *
                np.mean(
                    mae_draws
                    >
                    0
                )
            ),

        "rmse_gain":
            observed_rmse_gain,

        "rmse_ci95_lo":
            float(
                np.quantile(
                    rmse_draws,
                    0.025,
                )
            ),

        "rmse_ci95_hi":
            float(
                np.quantile(
                    rmse_draws,
                    0.975,
                )
            ),

        "rmse_positive_pct":
            float(
                100.0
                *
                np.mean(
                    rmse_draws
                    >
                    0
                )
            ),
    }


# --------------------------------------------------------------------------------------------------
# 11. BOOTSTRAP BY GAP BAND
# --------------------------------------------------------------------------------------------------

bootstrap_gap_rows = []


for band, g in V.groupby(
    "state_gap_band",
    observed=True,
    sort=False,
):

    boot = athlete_cluster_bootstrap(
        g,
        B=2000,
        seed=42,
    )


    bootstrap_gap_rows.append(
        {
            "state_gap_band":
                str(band),

            "n":
                int(
                    len(g)
                ),

            **boot,
        }
    )


VALIDATION_GAP_BOOTSTRAP = pd.DataFrame(
    bootstrap_gap_rows
)


# --------------------------------------------------------------------------------------------------
# 12. BOOTSTRAP FRESH VS STALE
# --------------------------------------------------------------------------------------------------

bootstrap_regime_rows = []


for regime, g in V.groupby(
    "fresh_stale_regime",
    sort=False,
):

    boot = athlete_cluster_bootstrap(
        g,
        B=5000,
        seed=42,
    )


    bootstrap_regime_rows.append(
        {
            "regime":
                regime,

            "n":
                int(
                    len(g)
                ),

            **boot,
        }
    )


VALIDATION_FRESH_STALE_BOOTSTRAP = pd.DataFrame(
    bootstrap_regime_rows
)


# --------------------------------------------------------------------------------------------------
# 13. DIRECT VALIDATION HYPOTHESIS CHECK
# --------------------------------------------------------------------------------------------------

fresh_row = (
    VALIDATION_FRESH_STALE.loc[
        VALIDATION_FRESH_STALE[
            "fresh_stale_regime"
        ]
        .eq(
            "FRESH_01_30"
        )
    ]
)


stale_row = (
    VALIDATION_FRESH_STALE.loc[
        VALIDATION_FRESH_STALE[
            "fresh_stale_regime"
        ]
        .eq(
            "STALE_31_PLUS"
        )
    ]
)


if len(fresh_row) != 1 or len(stale_row) != 1:

    raise RuntimeError(
        "Expected exactly one FRESH and one STALE validation row."
    )


FRESH_MAE_GAIN = float(
    fresh_row.iloc[
        0
    ][
        "MAE_gain_m"
    ]
)


STALE_MAE_GAIN = float(
    stale_row.iloc[
        0
    ][
        "MAE_gain_m"
    ]
)


fresh_boot = (
    VALIDATION_FRESH_STALE_BOOTSTRAP.loc[
        VALIDATION_FRESH_STALE_BOOTSTRAP[
            "regime"
        ]
        .eq(
            "FRESH_01_30"
        )
    ]
    .iloc[0]
)


stale_boot = (
    VALIDATION_FRESH_STALE_BOOTSTRAP.loc[
        VALIDATION_FRESH_STALE_BOOTSTRAP[
            "regime"
        ]
        .eq(
            "STALE_31_PLUS"
        )
    ]
    .iloc[0]
)


FRESH_ROBUSTLY_WORSE = bool(
    float(
        fresh_boot[
            "mae_ci95_hi"
        ]
    )
    <
    0
)


STALE_ROBUSTLY_BETTER = bool(
    float(
        stale_boot[
            "mae_ci95_lo"
        ]
    )
    >
    0
)


SAME_DIRECTION_AS_TEST = bool(
    FRESH_MAE_GAIN
    <
    0

    and

    STALE_MAE_GAIN
    >
    0
)


STRONG_VALIDATION_CONFIRMATION = bool(
    SAME_DIRECTION_AS_TEST
    and
    FRESH_ROBUSTLY_WORSE
    and
    STALE_ROBUSTLY_BETTER
)


# --------------------------------------------------------------------------------------------------
# 14. OPTIONAL MODEL-GROUP CHECK WITHIN FRESH REGIME
# --------------------------------------------------------------------------------------------------

VALIDATION_FRESH_BY_GROUP = (
    validation_summary(
        V.loc[
            V[
                "fresh_stale_regime"
            ]
            .eq(
                "FRESH_01_30"
            )
        ],
        "model_group",
    )
)


VALIDATION_STALE_BY_GROUP = (
    validation_summary(
        V.loc[
            V[
                "fresh_stale_regime"
            ]
            .eq(
                "STALE_31_PLUS"
            )
        ],
        "model_group",
    )
)


# --------------------------------------------------------------------------------------------------
# 15. QA
# --------------------------------------------------------------------------------------------------

FINITE_PASS = bool(
    np.isfinite(
        V[
            [
                "v5_error",
                "v6_error",
                "mae_gain_m",
                "actual_adjustment_vs_v5",

                "intercept_adjustment",
                "slope_adjustment",
                "offset_adjustment",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


ROW_ACCOUNTING_PASS = bool(
    len(V)
    ==
    VALIDATION_FRESH_STALE[
        "n"
    ]
    .sum()
)


BOOTSTRAP_FINITE_PASS = bool(
    np.isfinite(
        VALIDATION_FRESH_STALE_BOOTSTRAP[
            [
                "mae_gain",
                "mae_ci95_lo",
                "mae_ci95_hi",

                "rmse_gain",
                "rmse_ci95_lo",
                "rmse_ci95_hi",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


CELL_25_PASS = bool(
    FINITE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    BOOTSTRAP_FINITE_PASS
)


# --------------------------------------------------------------------------------------------------
# 16. SAVE
# --------------------------------------------------------------------------------------------------

V6_VALIDATION_GAP_DIAGNOSTIC_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_validation_gap_interaction.parquet"
)


V6_VALIDATION_GAP_BOOTSTRAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_validation_gap_interaction_bootstrap.parquet"
)


V6_VALIDATION_FRESH_STALE_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_validation_fresh_vs_stale.parquet"
)


V6_VALIDATION_FRESH_STALE_BOOTSTRAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v6_validation_fresh_vs_stale_bootstrap.parquet"
)


V6_VALIDATION_GAP_CONFIRMATION_JSON_PATH = (
    TRAINING_RUN_DIR
    /
    "v6_validation_short_gap_confirmation.json"
)


VALIDATION_BY_GAP.to_parquet(
    V6_VALIDATION_GAP_DIAGNOSTIC_PATH,
    index=False,
)


VALIDATION_GAP_BOOTSTRAP.to_parquet(
    V6_VALIDATION_GAP_BOOTSTRAP_PATH,
    index=False,
)


VALIDATION_FRESH_STALE.to_parquet(
    V6_VALIDATION_FRESH_STALE_PATH,
    index=False,
)


VALIDATION_FRESH_STALE_BOOTSTRAP.to_parquet(
    V6_VALIDATION_FRESH_STALE_BOOTSTRAP_PATH,
    index=False,
)


confirmation_json = {

    "status":
        (
            "PASSED"
            if CELL_25_PASS
            else
            "FAILED"
        ),

    "purpose":
        "VALIDATION_CONFIRMATION_OF_SHORT_GAP_INTERACTION",

    "no_fitting":
        True,

    "no_parameter_selection":
        True,

    "fresh_definition":
        "state_gap_days <= 30",

    "fresh_mae_gain_m":
        FRESH_MAE_GAIN,

    "stale_mae_gain_m":
        STALE_MAE_GAIN,

    "fresh_bootstrap_ci95": [
        float(
            fresh_boot[
                "mae_ci95_lo"
            ]
        ),

        float(
            fresh_boot[
                "mae_ci95_hi"
            ]
        ),
    ],

    "stale_bootstrap_ci95": [
        float(
            stale_boot[
                "mae_ci95_lo"
            ]
        ),

        float(
            stale_boot[
                "mae_ci95_hi"
            ]
        ),
    ],

    "same_direction_as_test":
        SAME_DIRECTION_AS_TEST,

    "fresh_robustly_worse":
        FRESH_ROBUSTLY_WORSE,

    "stale_robustly_better":
        STALE_ROBUSTLY_BETTER,

    "strong_validation_confirmation":
        STRONG_VALIDATION_CONFIRMATION,
}


with V6_VALIDATION_GAP_CONFIRMATION_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        confirmation_json,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 17. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v6_validation_short_gap_diagnostic"
] = confirmation_json


# Preserve failed release status.
TRAINING_MANIFEST[
    "status"
] = (
    "V6_RELEASE_GATE_FAILED_VALIDATION_DIAGNOSED"
    if CELL_25_PASS
    else
    "V6_VALIDATION_DIAGNOSTIC_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 18. GLOBALS
# --------------------------------------------------------------------------------------------------

V6_VALIDATION_SHORT_GAP_DATA = V

V6_VALIDATION_BY_GAP = (
    VALIDATION_BY_GAP
)

V6_VALIDATION_FRESH_STALE = (
    VALIDATION_FRESH_STALE
)

V6_VALIDATION_GAP_BOOTSTRAP = (
    VALIDATION_GAP_BOOTSTRAP
)

V6_VALIDATION_FRESH_STALE_BOOTSTRAP = (
    VALIDATION_FRESH_STALE_BOOTSTRAP
)

V6_VALIDATION_FRESH_BY_GROUP = (
    VALIDATION_FRESH_BY_GROUP
)

V6_VALIDATION_STALE_BY_GROUP = (
    VALIDATION_STALE_BY_GROUP
)

NOTEBOOK_2_CELL_25_PASS = (
    CELL_25_PASS
)


# --------------------------------------------------------------------------------------------------
# 19. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 145)
print("V6 TRAINING — VALIDATION CONFIRMATION OF SHORT-GAP INTERACTION")
print("=" * 145)


print()
print("VALIDATION — BY STATE GAP")
print("-" * 145)

display(
    VALIDATION_BY_GAP
)


print()
print("VALIDATION — STATE-GAP BOOTSTRAP")
print("-" * 145)

display(
    VALIDATION_GAP_BOOTSTRAP
)


print()
print("PRIMARY — FRESH VS STALE")
print("-" * 145)

display(
    VALIDATION_FRESH_STALE
)


print()
print("FRESH VS STALE — ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 145)

display(
    VALIDATION_FRESH_STALE_BOOTSTRAP
)


print()
print("FRESH REGIME — BY MODEL GROUP")
print("-" * 145)

display(
    VALIDATION_FRESH_BY_GROUP
)


print()
print("STALE REGIME — BY MODEL GROUP")
print("-" * 145)

display(
    VALIDATION_STALE_BY_GROUP
)


print()
print("HYPOTHESIS CHECK")
print("-" * 145)

print(
    f"Fresh 1-30d MAE gain             : "
    f"{FRESH_MAE_GAIN:+.6f} m"
)

print(
    f"Stale 31+d MAE gain              : "
    f"{STALE_MAE_GAIN:+.6f} m"
)

print(
    f"Same sign pattern as TEST        : "
    f"{SAME_DIRECTION_AS_TEST}"
)

print(
    f"Fresh robustly worse             : "
    f"{FRESH_ROBUSTLY_WORSE}"
)

print(
    f"Stale robustly better            : "
    f"{STALE_ROBUSTLY_BETTER}"
)

print(
    f"Strong validation confirmation   : "
    f"{STRONG_VALIDATION_CONFIRMATION}"
)


print()
print("QA")
print("-" * 145)

print(
    f"Finite diagnostic values         : "
    f"{FINITE_PASS}"
)

print(
    f"Row accounting                   : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Bootstrap finite                 : "
    f"{BOOTSTRAP_FINITE_PASS}"
)


print()
print("=" * 145)

if CELL_25_PASS:

    print(
        "CELL 25 STATUS: VALIDATION SHORT-GAP DIAGNOSTIC PASSED"
    )

else:

    print(
        "CELL 25 STATUS: VALIDATION SHORT-GAP DIAGNOSTIC REVIEW REQUIRED"
    )

print("=" * 145)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 16
# Original source SHA256: 7798be3483cf5e303c2dc2818f263bce8e6630421a2dc5055882b19e511272e1
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 26 — V6.1 TRAIN-ANCHORED OU FRESHNESS BLEND — VALIDATION
# ==================================================================================================
#
# Purpose
# -------
# Evaluate ONE principled state-freshness modification:
#
#       weight(h) = 1 - exp(-h / tau_V5)
#
#       V6.1 = V5 + weight(h) * (V6 - V5)
#
#
# tau_V5 was estimated from TRAIN in Cell 16.
#
# NO TEST data.
# NO cutoff fitting.
# NO new parameter optimization.
# NO uncertainty recalibration yet.
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_25_PASS",
    "V6_VALIDATION_UNCERTAINTY",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


if not NOTEBOOK_2_CELL_25_PASS:

    raise RuntimeError(
        "Cell 25 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. LOAD TRAIN-ESTIMATED V5 TAU
# --------------------------------------------------------------------------------------------------

V5_PARAMETER_PATH = (
    Path(TRAINING_RUN_DIR)
    /
    "v5_ou_parameters.json"
)


if not V5_PARAMETER_PATH.exists():

    raise RuntimeError(
        f"Missing V5 parameter file:\n"
        f"{V5_PARAMETER_PATH}"
    )


with V5_PARAMETER_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    v5_parameter_json = json.load(f)


def find_tau_days(obj):

    candidate_keys = {
        "tau",
        "tau_days",
        "state_tau",
        "state_tau_days",
        "ou_tau",
        "ou_tau_days",
    }


    if isinstance(obj, dict):

        for key, value in obj.items():

            if (
                str(key).lower()
                in candidate_keys

                and

                isinstance(
                    value,
                    (int, float),
                )
            ):

                return float(value)


        for value in obj.values():

            result = find_tau_days(
                value
            )

            if result is not None:

                return result


    return None


V5_TAU_DAYS = find_tau_days(
    v5_parameter_json
)


if (
    V5_TAU_DAYS is None
    or
    not np.isfinite(V5_TAU_DAYS)
    or
    V5_TAU_DAYS <= 0
):

    raise RuntimeError(
        "Could not recover positive finite V5 tau "
        "from v5_ou_parameters.json."
    )


# --------------------------------------------------------------------------------------------------
# 3. VALIDATION DATA ONLY
# --------------------------------------------------------------------------------------------------

V = (
    V6_VALIDATION_UNCERTAINTY.copy()
    .reset_index(
        drop=True
    )
)


required_columns = [
    "aaAthleteId",
    "state_gap_days",
    "mark_numeric_m",
    "v5_prediction_mean",
    "v6_prediction_mean",
    "sex_group",
    "model_group",
]


missing_columns = [
    x
    for x in required_columns
    if x not in V.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing validation columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_columns
        )
    )


if V["state_gap_days"].isna().any():

    raise RuntimeError(
        "Established validation row has missing state gap."
    )


# --------------------------------------------------------------------------------------------------
# 4. OU FRESHNESS WEIGHT
# --------------------------------------------------------------------------------------------------
#
# phi(h):
#       fraction of transient state persistence remaining
#
# weight(h):
#       amount of full V6 structural adjustment allowed
#
#
#       phi(h)   = exp(-h / tau)
#
#       weight   = 1 - phi(h)
#
# --------------------------------------------------------------------------------------------------

V[
    "v5_state_persistence_phi"
] = np.exp(
    -
    V[
        "state_gap_days"
    ]
    .astype(float)

    /

    V5_TAU_DAYS
)


V[
    "v61_structural_weight"
] = (
    1.0
    -
    V[
        "v5_state_persistence_phi"
    ]
)


if not V[
    "v61_structural_weight"
].between(
    0,
    1,
    inclusive="both",
).all():

    raise RuntimeError(
        "Invalid V6.1 structural weights."
    )


# --------------------------------------------------------------------------------------------------
# 5. V6.1 POINT PREDICTION
# --------------------------------------------------------------------------------------------------

V[
    "full_v6_adjustment"
] = (
    V[
        "v6_prediction_mean"
    ]

    -

    V[
        "v5_prediction_mean"
    ]
)


V[
    "v61_applied_adjustment"
] = (
    V[
        "v61_structural_weight"
    ]

    *

    V[
        "full_v6_adjustment"
    ]
)


V[
    "v61_prediction_mean"
] = (
    V[
        "v5_prediction_mean"
    ]

    +

    V[
        "v61_applied_adjustment"
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. ERRORS
# --------------------------------------------------------------------------------------------------

for model in [
    "v5",
    "v6",
    "v61",
]:

    V[
        f"{model}_error"
    ] = (
        V[
            "mark_numeric_m"
        ]

        -

        V[
            f"{model}_prediction_mean"
        ]
    )


    V[
        f"{model}_abs_error"
    ] = (
        V[
            f"{model}_error"
        ]
        .abs()
    )


    V[
        f"{model}_sq_error"
    ] = (
        V[
            f"{model}_error"
        ]
        **
        2
    )


# --------------------------------------------------------------------------------------------------
# 7. STATE-GAP BANDS
# --------------------------------------------------------------------------------------------------

V[
    "state_gap_band"
] = pd.cut(
    V[
        "state_gap_days"
    ],

    bins=[
        0,
        30,
        60,
        90,
        120,
        180,
        365,
        730,
        1825,
        np.inf,
    ],

    labels=[
        "01-30",
        "31-60",
        "61-90",
        "91-120",
        "121-180",
        "181-365",
        "366-730",
        "731-1825",
        "1826+",
    ],

    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 8. METRIC HELPER
# --------------------------------------------------------------------------------------------------

def compare_models(df):

    out = {
        "n":
            int(len(df)),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "mean_gap_days":
            float(
                df[
                    "state_gap_days"
                ]
                .mean()
            ),

        "mean_v61_weight":
            float(
                df[
                    "v61_structural_weight"
                ]
                .mean()
            ),
    }


    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        out[
            f"{model.upper()}_MAE"
        ] = float(
            df[
                f"{model}_abs_error"
            ]
            .mean()
        )


        out[
            f"{model.upper()}_RMSE"
        ] = float(
            np.sqrt(
                df[
                    f"{model}_sq_error"
                ]
                .mean()
            )
        )


    out[
        "V5_to_V61_MAE_gain_m"
    ] = (
        out["V5_MAE"]
        -
        out["V61_MAE"]
    )


    out[
        "V6_to_V61_MAE_gain_m"
    ] = (
        out["V6_MAE"]
        -
        out["V61_MAE"]
    )


    out[
        "V5_to_V61_RMSE_gain_m"
    ] = (
        out["V5_RMSE"]
        -
        out["V61_RMSE"]
    )


    out[
        "V6_to_V61_RMSE_gain_m"
    ] = (
        out["V6_RMSE"]
        -
        out["V61_RMSE"]
    )


    return pd.Series(out)


# --------------------------------------------------------------------------------------------------
# 9. OVERALL + BY GAP
# --------------------------------------------------------------------------------------------------

V61_VALIDATION_OVERALL = (
    compare_models(
        V
    )
    .to_frame()
    .T
)


V61_VALIDATION_BY_GAP = (
    V.groupby(
        "state_gap_band",
        observed=True,
        sort=False,
    )
    .apply(
        compare_models,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 10. ATHLETE-CLUSTER BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def cluster_bootstrap(
    df,
    B=5000,
    seed=42,
):

    cluster = (
        df.groupby(
            "aaAthleteId",
            as_index=False,
        )
        .agg(
            n=(
                "v5_error",
                "size",
            ),

            v5_abs=(
                "v5_abs_error",
                "sum",
            ),

            v6_abs=(
                "v6_abs_error",
                "sum",
            ),

            v61_abs=(
                "v61_abs_error",
                "sum",
            ),

            v5_sq=(
                "v5_sq_error",
                "sum",
            ),

            v6_sq=(
                "v6_sq_error",
                "sum",
            ),

            v61_sq=(
                "v61_sq_error",
                "sum",
            ),
        )
    )


    c_n = cluster[
        "n"
    ].to_numpy(float)


    arrays = {
        x:
            cluster[x]
            .to_numpy(float)

        for x in [
            "v5_abs",
            "v6_abs",
            "v61_abs",
            "v5_sq",
            "v6_sq",
            "v61_sq",
        ]
    }


    rng = np.random.default_rng(
        seed
    )


    n_cluster = len(cluster)


    mae_v5_v61 = np.empty(B)
    mae_v6_v61 = np.empty(B)

    rmse_v5_v61 = np.empty(B)
    rmse_v6_v61 = np.empty(B)


    for b in range(B):

        idx = rng.integers(
            0,
            n_cluster,
            size=n_cluster,
        )


        n_b = c_n[
            idx
        ].sum()


        mae5 = (
            arrays[
                "v5_abs"
            ][
                idx
            ].sum()
            /
            n_b
        )


        mae6 = (
            arrays[
                "v6_abs"
            ][
                idx
            ].sum()
            /
            n_b
        )


        mae61 = (
            arrays[
                "v61_abs"
            ][
                idx
            ].sum()
            /
            n_b
        )


        rmse5 = np.sqrt(
            arrays[
                "v5_sq"
            ][
                idx
            ].sum()
            /
            n_b
        )


        rmse6 = np.sqrt(
            arrays[
                "v6_sq"
            ][
                idx
            ].sum()
            /
            n_b
        )


        rmse61 = np.sqrt(
            arrays[
                "v61_sq"
            ][
                idx
            ].sum()
            /
            n_b
        )


        mae_v5_v61[b] = (
            mae5 - mae61
        )

        mae_v6_v61[b] = (
            mae6 - mae61
        )

        rmse_v5_v61[b] = (
            rmse5 - rmse61
        )

        rmse_v6_v61[b] = (
            rmse6 - rmse61
        )


    def summarize(
        name,
        draws,
    ):

        return {
            "comparison":
                name,

            "estimate":
                float(
                    np.mean(draws)
                ),

            "ci95_lo":
                float(
                    np.quantile(
                        draws,
                        0.025,
                    )
                ),

            "ci95_hi":
                float(
                    np.quantile(
                        draws,
                        0.975,
                    )
                ),

            "positive_pct":
                float(
                    100
                    *
                    np.mean(
                        draws
                        >
                        0
                    )
                ),
        }


    return pd.DataFrame(
        [
            summarize(
                "V5_to_V61_MAE_gain",
                mae_v5_v61,
            ),

            summarize(
                "V6_to_V61_MAE_gain",
                mae_v6_v61,
            ),

            summarize(
                "V5_to_V61_RMSE_gain",
                rmse_v5_v61,
            ),

            summarize(
                "V6_to_V61_RMSE_gain",
                rmse_v6_v61,
            ),
        ]
    )


V61_VALIDATION_BOOTSTRAP = (
    cluster_bootstrap(
        V,
        B=5000,
        seed=42,
    )
)


# --------------------------------------------------------------------------------------------------
# 11. WEIGHT REFERENCE TABLE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


V61_WEIGHT_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS,

        "V5_state_phi":
            np.exp(
                -REFERENCE_DAYS
                /
                V5_TAU_DAYS
            ),
    }
)


V61_WEIGHT_REFERENCE[
    "V61_structural_weight"
] = (
    1.0
    -
    V61_WEIGHT_REFERENCE[
        "V5_state_phi"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. QA
# --------------------------------------------------------------------------------------------------

FINITE_PASS = bool(
    np.isfinite(
        V[
            [
                "v5_prediction_mean",
                "v6_prediction_mean",
                "v61_prediction_mean",

                "v5_error",
                "v6_error",
                "v61_error",

                "v5_state_persistence_phi",
                "v61_structural_weight",
            ]
        ]
        .to_numpy(float)
    )
    .all()
)


WEIGHT_BOUNDS_PASS = bool(
    V[
        "v61_structural_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


NO_PARAMETER_FIT_PASS = True


CELL_26_PASS = bool(
    FINITE_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    NO_PARAMETER_FIT_PASS
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE
# --------------------------------------------------------------------------------------------------

V61_VALIDATION_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v61_ou_freshness_blend_validation.parquet"
)


V61_VALIDATION_SUMMARY_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v61_ou_freshness_blend_summary.parquet"
)


V61_VALIDATION_BY_GAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v61_ou_freshness_blend_by_gap.parquet"
)


V61_BOOTSTRAP_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "v61_ou_freshness_blend_bootstrap.parquet"
)


V61_SPEC_PATH = (
    TRAINING_RUN_DIR
    /
    "v61_ou_freshness_blend_candidate.json"
)


V.to_parquet(
    V61_VALIDATION_PATH,
    index=False,
)


V61_VALIDATION_OVERALL.to_parquet(
    V61_VALIDATION_SUMMARY_PATH,
    index=False,
)


V61_VALIDATION_BY_GAP.to_parquet(
    V61_VALIDATION_BY_GAP_PATH,
    index=False,
)


V61_VALIDATION_BOOTSTRAP.to_parquet(
    V61_BOOTSTRAP_PATH,
    index=False,
)


V61_CANDIDATE_SPEC = {
    "status":
        "VALIDATION_CANDIDATE_ONLY",

    "model":
        "V6.1_OU_FRESHNESS_BLEND",

    "tau_source":
        "V5 TRAIN OU",

    "tau_days":
        float(
            V5_TAU_DAYS
        ),

    "weight_formula":
        "1 - exp(-state_gap_days / tau_days)",

    "prediction_formula":
        (
            "V5_prediction + "
            "weight * (V6_prediction - V5_prediction)"
        ),

    "parameters_fitted_in_cell":
        False,

    "test_used":
        False,

    "uncertainty_recalibrated":
        False,
}


with V61_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        V61_CANDIDATE_SPEC,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 14. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "v61_ou_freshness_candidate"
] = {
    "status":
        (
            "VALIDATION_EVALUATED"
            if CELL_26_PASS
            else
            "FAILED"
        ),

    "candidate_spec_path":
        str(
            V61_SPEC_PATH
        ),
}


# Do not alter the failed V6 release decision.
TRAINING_MANIFEST[
    "status"
] = (
    "V6_RELEASE_FAILED_V61_VALIDATION_CANDIDATE"
    if CELL_26_PASS
    else
    "V61_VALIDATION_CANDIDATE_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 15. GLOBALS
# --------------------------------------------------------------------------------------------------

V61_VALIDATION_DATA = V

V61_VALIDATION_SUMMARY = (
    V61_VALIDATION_OVERALL
)

V61_VALIDATION_GAP_SUMMARY = (
    V61_VALIDATION_BY_GAP
)

V61_VALIDATION_BOOTSTRAP_SUMMARY = (
    V61_VALIDATION_BOOTSTRAP
)

NOTEBOOK_2_CELL_26_PASS = (
    CELL_26_PASS
)


# --------------------------------------------------------------------------------------------------
# 16. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 145)
print("V6.1 — TRAIN-ANCHORED OU FRESHNESS BLEND — VALIDATION")
print("=" * 145)


print()
print("TRAIN-ESTIMATED OU PARAMETER")
print("-" * 145)

print(
    f"V5 tau days : "
    f"{V5_TAU_DAYS:.6f}"
)


print()
print("FRESHNESS WEIGHT REFERENCE")
print("-" * 145)

display(
    V61_WEIGHT_REFERENCE
)


print()
print("VALIDATION — OVERALL")
print("-" * 145)

display(
    V61_VALIDATION_OVERALL
)


print()
print("VALIDATION — BY STATE GAP")
print("-" * 145)

display(
    V61_VALIDATION_BY_GAP
)


print()
print("ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 145)

display(
    V61_VALIDATION_BOOTSTRAP
)


print()
print("QA")
print("-" * 145)

print(
    f"Finite values       : "
    f"{FINITE_PASS}"
)

print(
    f"Weight bounds       : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"New parameters fit  : "
    f"{not NO_PARAMETER_FIT_PASS}"
)

print(
    f"TEST used           : False"
)


print()
print("=" * 145)

if CELL_26_PASS:

    print(
        "CELL 26 STATUS: V6.1 OU FRESHNESS CANDIDATE VALIDATION PASSED"
    )

else:

    print(
        "CELL 26 STATUS: V6.1 VALIDATION REVIEW REQUIRED"
    )

print("=" * 145)



V6 TRAINING — V5 → V6 MEAN DEGRADATION DIAGNOSTIC

OVERALL
-------------------------------------------------------------------------------------------------------------------------------------------------


,rows,athletes,V5_MAE,V6_MAE,MAE_gain_m,V5_RMSE,V6_RMSE,RMSE_gain_m,mean_V6_adjustment_m,mean_abs_V6_adjustment_m,median_abs_V6_adjustment_m,rows_helped_pct,rows_hurt_pct,correct_direction_pct,wrong_direction_pct,overshoot_pct
0,10941,4023,2.550114,2.565985,-0.015871,3.371135,3.374805,-0.003669,-0.192007,0.401166,0.353384,48.267983,51.732017,51.028242,48.971758,2.76026



V6 ADJUSTMENT DECOMPOSITION
-------------------------------------------------------------------------------------------------------------------------------------------------


,component,mean_m,mean_abs_m,median_m,sd_m,p05_m,p95_m
0,Intercept shrinkage,-0.250425,0.427965,-0.265257,0.457704,-0.991379,0.561902
1,Slope shrinkage,-0.036137,0.076709,-0.001733,0.144555,-0.333117,0.127102
2,Offset,0.094556,0.239657,-0.143748,0.240568,-0.143748,0.337389
3,V5 reconstruction difference,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Total V6 adjustment,-0.192007,0.401166,-0.239676,0.448976,-0.873193,0.650258



BY SEX
-------------------------------------------------------------------------------------------------------------------------------------------------


...


VALIDATION — BY STATE GAP
-------------------------------------------------------------------------------------------------------------------------------------------------


,state_gap_band,n,athletes,mean_gap_days,mean_v61_weight,V5_MAE,V5_RMSE,V6_MAE,V6_RMSE,V61_MAE,V61_RMSE,V5_to_V61_MAE_gain_m,V6_to_V61_MAE_gain_m,V5_to_V61_RMSE_gain_m,V6_to_V61_RMSE_gain_m
0,181-365,1410.0,1410.0,255.078723,0.914560,2.781927,3.556174,2.769174,3.532289,2.768466,3.532101,0.013462,7.079197e-04,0.024073,1.879297e-04
1,366-730,179.0,179.0,565.513966,0.993498,3.175985,4.157947,3.131432,4.080035,3.131952,4.080430,0.044033,-5.200136e-04,0.077517,-3.950156e-04
2,31-60,349.0,308.0,40.656160,0.335554,2.324205,3.055138,2.384204,3.090692,2.338567,3.060410,-0.014362,4.563675e-02,-0.005271,3.028264e-02
3,01-30,1559.0,828.0,13.521488,0.125421,2.352127,2.991153,2.409215,3.061101,2.357463,2.997637,-0.005336,5.175201e-02,-0.006484,6.346454e-02
4,121-180,432.0,428.0,155.268519,0.789501,2.685530,3.464056,2.633857,3.414087,2.641090,3.420643,0.044440,-7.232663e-03,0.043412,-6.556447e-03
5,731-1825,55.0,55.0,1083.927273,0.999892,3.794199,5.158236,3.774696,5.111859,3.774677,5.111839,0.019523,1.937684e-05,0.046397,2.067307e-05
6,1826+,5.0,5.0,3109.000000,1.000000,5.866815,7.189991,5.838986,7.017823,5.838986,7.017823,0.027829,3.399503e-11,0.172168,-2.137703e-10
7,91-120,126.0,121.0,105.214286,0.653549,2.397657,2.876223,2.458965,2.919751,2.432313,2.897547,-0.034656,2.665170e-02,-0.021324,2.220453e-02
8,61-90,168.0,162.0,70.422619,0.507316,2.468684,3.085143,2.451297,3.036544,2.452749,3.050159,0.015935,-1.451871e-03,0.034984,-1.361498e-02



ATHLETE-CLUSTER BOOTSTRAP
-------------------------------------------------------------------------------------------------------------------------------------------------


,comparison,estimate,ci95_lo,ci95_hi,positive_pct
0,V5_to_V61_MAE_gain,0.007409,-0.001380,0.016060,95.18
1,V6_to_V61_MAE_gain,0.022786,0.014146,0.031503,100.00
2,V5_to_V61_RMSE_gain,0.016416,0.005302,0.027979,99.84
3,V6_to_V61_RMSE_gain,0.022746,0.013455,0.031931,100.00



QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Finite values       : True
Weight bounds       : True
New parameters fit  : False
TEST used           : False

CELL 26 STATUS: V6.1 OU FRESHNESS CANDIDATE VALIDATION PASSED


## Stage 10 — Historical fold registry

Define and audit the independent historical temporal folds used to decide the final mean architecture.

**Legacy source cells represented:** 17  
**Execution rule:** run only after the preceding stage has completed successfully.


In [11]:
# ==================================================================================================
# REFACTORED STAGE 10 — Historical fold registry
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 17
# Original source SHA256: db75b2ef41569ea96ec00d4b1ac517ecf6acaec71c0c6c7ea09aa9466cafaac5
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 27 — HISTORICAL TEMPORAL BACKTEST FOLD DEFINITION + DATA AUDIT
# ==================================================================================================
#
# Purpose
# -------
# Define historical pseudo-production folds that mirror the current 2026 design:
#
#   TRAIN       : all history through previous Dec 31
#   VALIDATION  : Jan 1 -> Apr 30
#   TEST        : May 1 -> Aug 30
#
#
# Historical folds:
#
#   HB_2023:
#       TRAIN <= 2022-12-31
#       VALIDATION 2023-01-01 -> 2023-04-30
#       TEST       2023-05-01 -> 2023-08-30
#
#   HB_2024:
#       TRAIN <= 2023-12-31
#       VALIDATION 2024-01-01 -> 2024-04-30
#       TEST       2024-05-01 -> 2024-08-30
#
#   HB_2025:
#       TRAIN <= 2024-12-31
#       VALIDATION 2025-01-01 -> 2025-04-30
#       TEST       2025-05-01 -> 2025-08-30
#
#
# IMPORTANT
# ---------
# This cell:
#
#   • does NOT fit V4
#   • does NOT fit V5
#   • does NOT fit V6/V6.1
#   • does NOT calculate forecast accuracy
#   • does NOT select folds based on performance
#
# It only verifies that the historical data are adequate for fair backtesting.
#
#
# Established athlete definition:
#
#       athlete appears in that fold's TRAIN period
#
# This matches the current 2026 evaluation definition.
#
#
# Chronology audit:
#
# State gaps are calculated using previous DISTINCT athlete dates.
# Validation dates are allowed to update chronology before later validation/test dates,
# exactly as production rolling-origin forecasting would operate.
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PATHS
# --------------------------------------------------------------------------------------------------

ACTIVE_CANONICAL_POINTER = Path(
    "/Users/veesheenyuen/Documents/code/veeyuen/SAA/"
    "world_athletics_global/final/ACTIVE_CANONICAL.json"
)


FALLBACK_CANONICAL_PATH = Path(
    "/Users/veesheenyuen/Documents/code/veeyuen/SAA/"
    "world_athletics_global/final/"
    "javelin_reconciled_base_1986_2026.parquet"
)


if not ACTIVE_CANONICAL_POINTER.exists():

    raise RuntimeError(
        f"Active canonical pointer not found:\n"
        f"{ACTIVE_CANONICAL_POINTER}"
    )


with ACTIVE_CANONICAL_POINTER.open(
    "r",
    encoding="utf-8",
) as f:

    active_pointer = json.load(f)


# --------------------------------------------------------------------------------------------------
# 2. RECOVER ACTIVE CANONICAL PARQUET PATH
# --------------------------------------------------------------------------------------------------

def find_existing_parquet_path(obj):

    if isinstance(obj, dict):

        for value in obj.values():

            result = find_existing_parquet_path(
                value
            )

            if result is not None:

                return result


    elif isinstance(obj, list):

        for value in obj:

            result = find_existing_parquet_path(
                value
            )

            if result is not None:

                return result


    elif isinstance(obj, str):

        if obj.lower().endswith(".parquet"):

            p = Path(obj)

            if p.exists():

                return p


    return None


CANONICAL_PATH = find_existing_parquet_path(
    active_pointer
)


if CANONICAL_PATH is None:

    if FALLBACK_CANONICAL_PATH.exists():

        CANONICAL_PATH = (
            FALLBACK_CANONICAL_PATH
        )

    else:

        raise RuntimeError(
            "Could not resolve active canonical parquet."
        )


# --------------------------------------------------------------------------------------------------
# 3. LOAD CANONICAL DATA
# --------------------------------------------------------------------------------------------------

canonical = pd.read_parquet(
    CANONICAL_PATH
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",
    "age_years",
    "model_group",
    "v4_base_eligible",
]


missing = [
    x
    for x in required_columns
    if x not in canonical.columns
]


if missing:

    raise RuntimeError(
        "Canonical data missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing
        )
    )


canonical[
    "result_date_dt"
] = pd.to_datetime(
    canonical[
        "result_date_dt"
    ]
)


# --------------------------------------------------------------------------------------------------
# 4. RECREATE MODEL-ELIGIBLE BASE
# --------------------------------------------------------------------------------------------------
#
# Matches the production modelling domain:
#
#   • V4-base eligible
#   • age 13 through 35 inclusive
#   • four core sex/implement groups
#
# --------------------------------------------------------------------------------------------------

CORE_GROUPS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


BACKTEST_BASE = (
    canonical.loc[
        canonical[
            "v4_base_eligible"
        ]
        .fillna(False)
        .astype(bool)

        &

        canonical[
            "age_years"
        ]
        .between(
            13,
            35,
            inclusive="both",
        )

        &

        canonical[
            "model_group"
        ]
        .isin(
            CORE_GROUPS
        )
    ]
    .copy()
)


BACKTEST_BASE = (
    BACKTEST_BASE.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 5. FIXED HISTORICAL BACKTEST FOLDS
# --------------------------------------------------------------------------------------------------

BACKTEST_FOLDS = pd.DataFrame(
    [
        {
            "fold":
                "HB_2023",

            "train_end":
                pd.Timestamp(
                    "2022-12-31"
                ),

            "validation_start":
                pd.Timestamp(
                    "2023-01-01"
                ),

            "validation_end":
                pd.Timestamp(
                    "2023-04-30"
                ),

            "test_start":
                pd.Timestamp(
                    "2023-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2023-08-30"
                ),
        },

        {
            "fold":
                "HB_2024",

            "train_end":
                pd.Timestamp(
                    "2023-12-31"
                ),

            "validation_start":
                pd.Timestamp(
                    "2024-01-01"
                ),

            "validation_end":
                pd.Timestamp(
                    "2024-04-30"
                ),

            "test_start":
                pd.Timestamp(
                    "2024-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2024-08-30"
                ),
        },

        {
            "fold":
                "HB_2025",

            "train_end":
                pd.Timestamp(
                    "2024-12-31"
                ),

            "validation_start":
                pd.Timestamp(
                    "2025-01-01"
                ),

            "validation_end":
                pd.Timestamp(
                    "2025-04-30"
                ),

            "test_start":
                pd.Timestamp(
                    "2025-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2025-08-30"
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. DISTINCT-DATE STATE GAP HELPER
# --------------------------------------------------------------------------------------------------

def add_rolling_previous_date(
    df,
):

    date_table = (
        df[
            [
                "aaAthleteId",
                "result_date_dt",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "aaAthleteId",
                "result_date_dt",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    date_table[
        "previous_distinct_date"
    ] = (
        date_table.groupby(
            "aaAthleteId",
            sort=False,
        )[
            "result_date_dt"
        ]
        .shift(1)
    )


    date_table[
        "state_gap_days"
    ] = (
        date_table[
            "result_date_dt"
        ]

        -

        date_table[
            "previous_distinct_date"
        ]
    ).dt.days


    return df.merge(
        date_table,
        on=[
            "aaAthleteId",
            "result_date_dt",
        ],
        how="left",
        validate="many_to_one",
    )


# --------------------------------------------------------------------------------------------------
# 7. FOLD AUDIT
# --------------------------------------------------------------------------------------------------

fold_rows = []
group_rows = []
gap_rows = []


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


for _, spec in BACKTEST_FOLDS.iterrows():

    fold_name = spec[
        "fold"
    ]


    train_end = spec[
        "train_end"
    ]


    validation_start = spec[
        "validation_start"
    ]


    validation_end = spec[
        "validation_end"
    ]


    test_start = spec[
        "test_start"
    ]


    test_end = spec[
        "test_end"
    ]


    # ----------------------------------------------------------------------------------------------
    # TRAIN
    # ----------------------------------------------------------------------------------------------

    train = BACKTEST_BASE.loc[
        BACKTEST_BASE[
            "result_date_dt"
        ]
        <=
        train_end
    ].copy()


    train_athletes = set(
        train[
            "aaAthleteId"
        ]
        .unique()
    )


    # ----------------------------------------------------------------------------------------------
    # VALIDATION
    # ----------------------------------------------------------------------------------------------

    validation = BACKTEST_BASE.loc[
        BACKTEST_BASE[
            "result_date_dt"
        ]
        .between(
            validation_start,
            validation_end,
            inclusive="both",
        )
    ].copy()


    validation[
        "seen_in_fold_train"
    ] = (
        validation[
            "aaAthleteId"
        ]
        .isin(
            train_athletes
        )
    )


    validation_established = (
        validation.loc[
            validation[
                "seen_in_fold_train"
            ]
        ]
        .copy()
    )


    # ----------------------------------------------------------------------------------------------
    # TEST
    # ----------------------------------------------------------------------------------------------

    test = BACKTEST_BASE.loc[
        BACKTEST_BASE[
            "result_date_dt"
        ]
        .between(
            test_start,
            test_end,
            inclusive="both",
        )
    ].copy()


    test[
        "seen_in_fold_train"
    ] = (
        test[
            "aaAthleteId"
        ]
        .isin(
            train_athletes
        )
    )


    test_established = (
        test.loc[
            test[
                "seen_in_fold_train"
            ]
        ]
        .copy()
    )


    # ----------------------------------------------------------------------------------------------
    # ROLLING CHRONOLOGY THROUGH TEST END
    #
    # This does not use marks to fit anything.
    # It simply determines the previous DISTINCT competition date available at each point.
    # ----------------------------------------------------------------------------------------------

    chronology = BACKTEST_BASE.loc[
        (
            BACKTEST_BASE[
                "aaAthleteId"
            ]
            .isin(
                train_athletes
            )
        )

        &

        (
            BACKTEST_BASE[
                "result_date_dt"
            ]
            <=
            test_end
        )
    ].copy()


    chronology = add_rolling_previous_date(
        chronology
    )


    validation_chronology = (
        chronology.loc[
            chronology[
                "result_date_dt"
            ]
            .between(
                validation_start,
                validation_end,
                inclusive="both",
            )
        ]
        .copy()
    )


    test_chronology = (
        chronology.loc[
            chronology[
                "result_date_dt"
            ]
            .between(
                test_start,
                test_end,
                inclusive="both",
            )
        ]
        .copy()
    )


    validation_chronology[
        "state_gap_band"
    ] = pd.cut(
        validation_chronology[
            "state_gap_days"
        ],
        bins=GAP_BINS,
        labels=GAP_LABELS,
        include_lowest=True,
    )


    test_chronology[
        "state_gap_band"
    ] = pd.cut(
        test_chronology[
            "state_gap_days"
        ],
        bins=GAP_BINS,
        labels=GAP_LABELS,
        include_lowest=True,
    )


    # ----------------------------------------------------------------------------------------------
    # DATA-SUFFICIENCY CHECKS
    #
    # These thresholds concern sample size only.
    # They do NOT depend on forecast performance.
    # ----------------------------------------------------------------------------------------------

    train_rows_pass = bool(
        len(train)
        >=
        50000
    )


    train_athletes_pass = bool(
        train[
            "aaAthleteId"
        ]
        .nunique()
        >=
        5000
    )


    validation_established_pass = bool(
        len(
            validation_established
        )
        >=
        500
    )


    test_established_pass = bool(
        len(
            test_established
        )
        >=
        1000
    )


    test_established_athletes_pass = bool(
        test_established[
            "aaAthleteId"
        ]
        .nunique()
        >=
        500
    )


    all_groups_train_pass = bool(
        set(
            CORE_GROUPS
        )
        .issubset(
            set(
                train[
                    "model_group"
                ]
                .dropna()
                .unique()
            )
        )
    )


    all_groups_validation_pass = bool(
        set(
            CORE_GROUPS
        )
        .issubset(
            set(
                validation_established[
                    "model_group"
                ]
                .dropna()
                .unique()
            )
        )
    )


    all_groups_test_pass = bool(
        set(
            CORE_GROUPS
        )
        .issubset(
            set(
                test_established[
                    "model_group"
                ]
                .dropna()
                .unique()
            )
        )
    )


    chronology_pass = bool(
        validation_chronology[
            "state_gap_days"
        ]
        .notna()
        .all()

        and

        test_chronology[
            "state_gap_days"
        ]
        .notna()
        .all()

        and

        (
            validation_chronology[
                "state_gap_days"
            ]
            >
            0
        )
        .all()

        and

        (
            test_chronology[
                "state_gap_days"
            ]
            >
            0
        )
        .all()
    )


    fold_pass = bool(
        train_rows_pass
        and
        train_athletes_pass
        and
        validation_established_pass
        and
        test_established_pass
        and
        test_established_athletes_pass
        and
        all_groups_train_pass
        and
        all_groups_validation_pass
        and
        all_groups_test_pass
        and
        chronology_pass
    )


    # ----------------------------------------------------------------------------------------------
    # FOLD SUMMARY
    # ----------------------------------------------------------------------------------------------

    fold_rows.append(
        {
            "fold":
                fold_name,

            "train_end":
                train_end,

            "validation_start":
                validation_start,

            "validation_end":
                validation_end,

            "test_start":
                test_start,

            "test_end":
                test_end,

            "train_rows":
                int(
                    len(train)
                ),

            "train_athletes":
                int(
                    train[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "train_distinct_athlete_dates":
                int(
                    train[
                        [
                            "aaAthleteId",
                            "result_date_dt",
                        ]
                    ]
                    .drop_duplicates()
                    .shape[0]
                ),

            "validation_rows":
                int(
                    len(validation)
                ),

            "validation_athletes":
                int(
                    validation[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "validation_established_rows":
                int(
                    len(
                        validation_established
                    )
                ),

            "validation_established_athletes":
                int(
                    validation_established[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "validation_seen_pct":
                float(
                    100.0
                    *
                    validation[
                        "seen_in_fold_train"
                    ]
                    .mean()
                )
                if len(validation)
                else
                np.nan,

            "test_rows":
                int(
                    len(test)
                ),

            "test_athletes":
                int(
                    test[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "test_established_rows":
                int(
                    len(
                        test_established
                    )
                ),

            "test_established_athletes":
                int(
                    test_established[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "test_seen_pct":
                float(
                    100.0
                    *
                    test[
                        "seen_in_fold_train"
                    ]
                    .mean()
                )
                if len(test)
                else
                np.nan,

            "validation_median_state_gap_days":
                float(
                    validation_chronology[
                        "state_gap_days"
                    ]
                    .median()
                ),

            "test_median_state_gap_days":
                float(
                    test_chronology[
                        "state_gap_days"
                    ]
                    .median()
                ),

            "train_rows_pass":
                train_rows_pass,

            "train_athletes_pass":
                train_athletes_pass,

            "validation_established_pass":
                validation_established_pass,

            "test_established_pass":
                test_established_pass,

            "test_established_athletes_pass":
                test_established_athletes_pass,

            "all_groups_train_pass":
                all_groups_train_pass,

            "all_groups_validation_pass":
                all_groups_validation_pass,

            "all_groups_test_pass":
                all_groups_test_pass,

            "chronology_pass":
                chronology_pass,

            "fold_pass":
                fold_pass,
        }
    )


    # ----------------------------------------------------------------------------------------------
    # GROUP AUDIT
    # ----------------------------------------------------------------------------------------------

    for group in CORE_GROUPS:

        train_g = train.loc[
            train[
                "model_group"
            ]
            .eq(
                group
            )
        ]


        val_g = validation_established.loc[
            validation_established[
                "model_group"
            ]
            .eq(
                group
            )
        ]


        test_g = test_established.loc[
            test_established[
                "model_group"
            ]
            .eq(
                group
            )
        ]


        group_rows.append(
            {
                "fold":
                    fold_name,

                "model_group":
                    group,

                "train_rows":
                    int(
                        len(
                            train_g
                        )
                    ),

                "train_athletes":
                    int(
                        train_g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "validation_established_rows":
                    int(
                        len(
                            val_g
                        )
                    ),

                "validation_established_athletes":
                    int(
                        val_g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "test_established_rows":
                    int(
                        len(
                            test_g
                        )
                    ),

                "test_established_athletes":
                    int(
                        test_g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),
            }
        )


    # ----------------------------------------------------------------------------------------------
    # TEST STATE-GAP AUDIT
    # ----------------------------------------------------------------------------------------------

    gap_counts = (
        test_chronology.groupby(
            "state_gap_band",
            observed=True,
        )
        .agg(
            rows=(
                "aaAthleteId",
                "size",
            ),

            athletes=(
                "aaAthleteId",
                "nunique",
            ),

            mean_gap_days=(
                "state_gap_days",
                "mean",
            ),

            median_gap_days=(
                "state_gap_days",
                "median",
            ),
        )
        .reset_index()
    )


    for _, gap_row in gap_counts.iterrows():

        gap_rows.append(
            {
                "fold":
                    fold_name,

                "state_gap_band":
                    str(
                        gap_row[
                            "state_gap_band"
                        ]
                    ),

                "rows":
                    int(
                        gap_row[
                            "rows"
                        ]
                    ),

                "athletes":
                    int(
                        gap_row[
                            "athletes"
                        ]
                    ),

                "mean_gap_days":
                    float(
                        gap_row[
                            "mean_gap_days"
                        ]
                    ),

                "median_gap_days":
                    float(
                        gap_row[
                            "median_gap_days"
                        ]
                    ),
            }
        )


# --------------------------------------------------------------------------------------------------
# 8. AUDIT TABLES
# --------------------------------------------------------------------------------------------------

HISTORICAL_BACKTEST_FOLD_AUDIT = pd.DataFrame(
    fold_rows
)


HISTORICAL_BACKTEST_GROUP_AUDIT = pd.DataFrame(
    group_rows
)


HISTORICAL_BACKTEST_GAP_AUDIT = pd.DataFrame(
    gap_rows
)


ELIGIBLE_BACKTEST_FOLDS = (
    HISTORICAL_BACKTEST_FOLD_AUDIT.loc[
        HISTORICAL_BACKTEST_FOLD_AUDIT[
            "fold_pass"
        ]
    ][
        "fold"
    ]
    .tolist()
)


CELL_27_PASS = bool(
    len(
        ELIGIBLE_BACKTEST_FOLDS
    )
    >=
    2
)


# --------------------------------------------------------------------------------------------------
# 9. SAVE
# --------------------------------------------------------------------------------------------------

HISTORICAL_BACKTEST_FOLD_AUDIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "historical_backtest_fold_audit.parquet"
)


HISTORICAL_BACKTEST_GROUP_AUDIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "historical_backtest_group_audit.parquet"
)


HISTORICAL_BACKTEST_GAP_AUDIT_PATH = (
    TRAINING_CHECKPOINT_DIR
    /
    "historical_backtest_gap_audit.parquet"
)


HISTORICAL_BACKTEST_SPEC_PATH = (
    TRAINING_RUN_DIR
    /
    "historical_backtest_spec.json"
)


HISTORICAL_BACKTEST_FOLD_AUDIT.to_parquet(
    HISTORICAL_BACKTEST_FOLD_AUDIT_PATH,
    index=False,
)


HISTORICAL_BACKTEST_GROUP_AUDIT.to_parquet(
    HISTORICAL_BACKTEST_GROUP_AUDIT_PATH,
    index=False,
)


HISTORICAL_BACKTEST_GAP_AUDIT.to_parquet(
    HISTORICAL_BACKTEST_GAP_AUDIT_PATH,
    index=False,
)


HISTORICAL_BACKTEST_SPEC = {
    "status":
        (
            "AUDIT_PASSED"
            if CELL_27_PASS
            else
            "AUDIT_FAILED"
        ),

    "canonical_path":
        str(
            CANONICAL_PATH
        ),

    "evaluation_design":
        {
            "train":
                "all eligible history through prior Dec 31",

            "validation":
                "Jan 1 through Apr 30",

            "test":
                "May 1 through Aug 30",

            "established_definition":
                "athlete present in fold TRAIN",

            "same_day_rule":
                "previous DISTINCT athlete date",

            "performance_used_for_fold_selection":
                False,
        },

    "folds":
        [
            {
                key:
                    (
                        value.isoformat()
                        if isinstance(
                            value,
                            pd.Timestamp,
                        )
                        else
                        value
                    )

                for key, value in row.items()
            }

            for row in BACKTEST_FOLDS.to_dict(
                orient="records"
            )
        ],

    "eligible_folds":
        ELIGIBLE_BACKTEST_FOLDS,
}


with HISTORICAL_BACKTEST_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        HISTORICAL_BACKTEST_SPEC,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 10. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
] = {
    "status":
        (
            "FOLD_AUDIT_PASSED"
            if CELL_27_PASS
            else
            "FOLD_AUDIT_FAILED"
        ),

    "spec_path":
        str(
            HISTORICAL_BACKTEST_SPEC_PATH
        ),

    "eligible_folds":
        ELIGIBLE_BACKTEST_FOLDS,
}


# Preserve current failed-release state.
TRAINING_MANIFEST[
    "status"
] = (
    "V6_RELEASE_FAILED_HISTORICAL_BACKTEST_READY"
    if CELL_27_PASS
    else
    "HISTORICAL_BACKTEST_FOLD_AUDIT_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 11. GLOBALS
# --------------------------------------------------------------------------------------------------

HISTORICAL_BACKTEST_BASE = (
    BACKTEST_BASE
)


HISTORICAL_BACKTEST_FOLDS = (
    BACKTEST_FOLDS
)


NOTEBOOK_2_CELL_27_PASS = (
    CELL_27_PASS
)


# --------------------------------------------------------------------------------------------------
# 12. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 150)
print("V6.1 — HISTORICAL TEMPORAL BACKTEST FOLD AUDIT")
print("=" * 150)


print()
print("ACTIVE CANONICAL")
print("-" * 150)

print(
    CANONICAL_PATH
)

print(
    f"Model-eligible rows : "
    f"{len(BACKTEST_BASE):,}"
)

print(
    f"Athletes            : "
    f"{BACKTEST_BASE['aaAthleteId'].nunique():,}"
)

print(
    f"Date range          : "
    f"{BACKTEST_BASE['result_date_dt'].min().date()} "
    f"-> "
    f"{BACKTEST_BASE['result_date_dt'].max().date()}"
)


print()
print("FOLD AUDIT")
print("-" * 150)

display(
    HISTORICAL_BACKTEST_FOLD_AUDIT
)


print()
print("MODEL-GROUP AUDIT")
print("-" * 150)

display(
    HISTORICAL_BACKTEST_GROUP_AUDIT
)


print()
print("TEST STATE-GAP AUDIT")
print("-" * 150)

display(
    HISTORICAL_BACKTEST_GAP_AUDIT
)


print()
print("ELIGIBLE HISTORICAL FOLDS")
print("-" * 150)

print(
    ELIGIBLE_BACKTEST_FOLDS
)


print()
print("=" * 150)

if CELL_27_PASS:

    print(
        "CELL 27 STATUS: HISTORICAL BACKTEST FOLD AUDIT PASSED"
    )

else:

    print(
        "CELL 27 STATUS: HISTORICAL BACKTEST FOLD AUDIT REVIEW REQUIRED"
    )

print("=" * 150)



V6.1 — HISTORICAL TEMPORAL BACKTEST FOLD AUDIT

ACTIVE CANONICAL
------------------------------------------------------------------------------------------------------------------------------------------------------
/Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026.parquet
Model-eligible rows : 189,128
Athletes            : 14,478
Date range          : 1991-08-10 -> 2026-09-18

FOLD AUDIT
------------------------------------------------------------------------------------------------------------------------------------------------------


,fold,train_end,validation_start,validation_end,test_start,test_end,train_rows,train_athletes,train_distinct_athlete_dates,validation_rows,...,train_rows_pass,train_athletes_pass,validation_established_pass,test_established_pass,test_established_athletes_pass,all_groups_train_pass,all_groups_validation_pass,all_groups_test_pass,chronology_pass,fold_pass
0,HB_2023,2022-12-31,2023-01-01,2023-04-30,2023-05-01,2023-08-30,110977,7442,110397,3719,...,True,True,True,True,True,True,True,True,True,True
1,HB_2024,2023-12-31,2024-01-01,2024-04-30,2024-05-01,2024-08-30,129573,9336,128862,4900,...,True,True,True,True,True,True,True,True,True,True
2,HB_2025,2024-12-31,2025-01-01,2025-04-30,2025-05-01,2025-08-30,150052,11207,149184,4755,...,True,True,True,True,True,True,True,True,True,True



MODEL-GROUP AUDIT
------------------------------------------------------------------------------------------------------------------------------------------------------


,fold,model_group,train_rows,train_athletes,validation_established_rows,validation_established_athletes,test_established_rows,test_established_athletes
0,HB_2023,M_700,6831,1711,96,56,266,104
1,HB_2023,M_800,49427,3111,1381,764,4499,1678
2,HB_2023,W_500,11987,2221,273,144,622,241
3,HB_2023,W_600,42732,2630,1342,735,4723,1599
4,HB_2024,M_700,7831,2038,108,67,374,136
5,HB_2024,M_800,57459,4132,1965,1064,5038,1903
6,HB_2024,W_500,13606,2572,252,140,560,231
7,HB_2024,W_600,50677,3633,1963,991,5082,1808
8,HB_2025,M_700,9148,2447,133,84,324,128
9,HB_2025,M_800,66350,4996,1985,1073,5268,1979



TEST STATE-GAP AUDIT
------------------------------------------------------------------------------------------------------------------------------------------------------


,fold,state_gap_band,rows,athletes,mean_gap_days,median_gap_days
0,HB_2023,01-30,5779,1914,13.923170,13.0
1,HB_2023,31-60,1331,1129,42.297521,41.0
2,HB_2023,61-90,508,496,71.870079,70.0
3,HB_2023,91-120,146,146,100.575342,98.0
4,HB_2023,121-180,69,69,144.608696,144.0
5,HB_2023,181-365,1581,1575,290.415560,293.0
6,HB_2023,366-730,475,470,535.448421,588.0
7,HB_2023,731-1825,186,186,1073.376344,1002.0
8,HB_2023,1826+,35,35,2440.457143,2170.0
9,HB_2024,01-30,6070,2057,13.683855,13.0



ELIGIBLE HISTORICAL FOLDS
------------------------------------------------------------------------------------------------------------------------------------------------------
['HB_2023', 'HB_2024', 'HB_2025']

CELL 27 STATUS: HISTORICAL BACKTEST FOLD AUDIT PASSED


## Stage 11 — HB_2023 model reconstruction

Rebuild the 2023 historical V4/V5 training state and persist immutable fold-specific audit artifacts.

**Legacy source cells represented:** 18, 19, 20, 21  
**Execution rule:** run only after the preceding stage has completed successfully.


In [12]:
# ==================================================================================================
# REFACTORED STAGE 11 — HB_2023 model reconstruction
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 18
# Original source SHA256: 2d8f7b81118b841eab930e4e6c609378b3d80fb0294baeeea708ca7f3725d8e2
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 28 — HB_2023 HISTORICAL V4 REFIT PREPARATION
# ==================================================================================================
#
# Purpose
# -------
# Reconstruct the first historical backtest fold using ONLY information
# that would have been available at the time.
#
#
# HB_2023:
#
#   TRAIN       <= 2022-12-31
#   VALIDATION   2023-01-01 -> 2023-04-30
#   TEST         2023-05-01 -> 2023-08-30
#
#
# This cell:
#
#   • materializes the historical fold
#   • defines V4 features exactly as in the production model
#   • creates a TRAIN-only athlete index
#   • builds the 7-column cubic age spline from historical TRAIN only
#   • reuses the TRAIN spline design for VALIDATION and TEST
#   • reconstructs leakage-safe previous-distinct-date chronology
#   • runs leakage / feature / spline QA
#
#
# This cell DOES NOT fit PyMC yet.
#
# ==================================================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from patsy import (
    dmatrix,
    build_design_matrices,
)


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_27_PASS",

    "HISTORICAL_BACKTEST_BASE",
    "HISTORICAL_BACKTEST_FOLDS",

    "TRAINING_RUN_DIR",
    "TRAINING_CHECKPOINT_DIR",

    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_27_PASS:

    raise RuntimeError(
        "Cell 27 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. SELECT FIRST HISTORICAL FOLD
# --------------------------------------------------------------------------------------------------

HB_FOLD_NAME = "HB_2023"


fold_match = (
    HISTORICAL_BACKTEST_FOLDS.loc[
        HISTORICAL_BACKTEST_FOLDS[
            "fold"
        ]
        .eq(
            HB_FOLD_NAME
        )
    ]
)


if len(fold_match) != 1:

    raise RuntimeError(
        f"Could not uniquely identify {HB_FOLD_NAME}."
    )


FOLD_SPEC = (
    fold_match
    .iloc[0]
)


TRAIN_END = pd.Timestamp(
    FOLD_SPEC[
        "train_end"
    ]
)


VALIDATION_START = pd.Timestamp(
    FOLD_SPEC[
        "validation_start"
    ]
)


VALIDATION_END = pd.Timestamp(
    FOLD_SPEC[
        "validation_end"
    ]
)


TEST_START = pd.Timestamp(
    FOLD_SPEC[
        "test_start"
    ]
)


TEST_END = pd.Timestamp(
    FOLD_SPEC[
        "test_end"
    ]
)


# --------------------------------------------------------------------------------------------------
# 3. MATERIALIZE FOLD
# --------------------------------------------------------------------------------------------------

BASE = (
    HISTORICAL_BACKTEST_BASE.copy()
)


TRAIN = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    ]
    .copy()
)


VALIDATION_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            VALIDATION_START,
            VALIDATION_END,
            inclusive="both",
        )
    ]
    .copy()
)


TEST_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            TEST_START,
            TEST_END,
            inclusive="both",
        )
    ]
    .copy()
)


TRAIN_ATHLETES = set(
    TRAIN[
        "aaAthleteId"
    ]
    .unique()
)


VALIDATION_ALL[
    "athlete_seen_in_training"
] = (
    VALIDATION_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


TEST_ALL[
    "athlete_seen_in_training"
] = (
    TEST_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


# Historical V6.1 backtest is focused on established athletes.
VALIDATION = (
    VALIDATION_ALL.loc[
        VALIDATION_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


TEST = (
    TEST_ALL.loc[
        TEST_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 4. EXACT V4 FEATURE DEFINITIONS
# --------------------------------------------------------------------------------------------------

SEX_INDEX = {
    "M": 0,
    "W": 1,
}


GROUP_INDEX = {
    "M_700": 0,
    "M_800": 1,
    "W_500": 2,
    "W_600": 3,
}


YOUTH_GROUPS = {
    "M_700",
    "W_500",
}


AGE_CENTER = 18.0
AGE_SCALE = 5.0


def add_v4_features(
    df,
):

    out = df.copy()


    out[
        "sex_group"
    ] = (
        out[
            "model_group"
        ]
        .astype(str)
        .str[
            0
        ]
    )


    out[
        "sex_idx"
    ] = (
        out[
            "sex_group"
        ]
        .map(
            SEX_INDEX
        )
    )


    out[
        "group_idx"
    ] = (
        out[
            "model_group"
        ]
        .map(
            GROUP_INDEX
        )
    )


    out[
        "is_youth_implement"
    ] = (
        out[
            "model_group"
        ]
        .isin(
            YOUTH_GROUPS
        )
        .astype(float)
    )


    out[
        "age_scaled"
    ] = (
        (
            out[
                "age_years"
            ]
            .astype(float)

            -

            AGE_CENTER
        )

        /

        AGE_SCALE
    )


    return out


TRAIN = add_v4_features(
    TRAIN
)


VALIDATION = add_v4_features(
    VALIDATION
)


TEST = add_v4_features(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 5. TRAIN-ONLY ATHLETE INDEX
# --------------------------------------------------------------------------------------------------

ATHLETE_LEVELS = (
    pd.Series(
        TRAIN[
            "aaAthleteId"
        ]
        .unique()
    )
    .sort_values()
    .tolist()
)


ATHLETE_INDEX = {
    athlete_id:
        idx

    for idx, athlete_id
    in enumerate(
        ATHLETE_LEVELS
    )
}


TRAIN[
    "athlete_idx"
] = (
    TRAIN[
        "aaAthleteId"
    ]
    .map(
        ATHLETE_INDEX
    )
    .astype(int)
)


VALIDATION[
    "athlete_idx"
] = (
    VALIDATION[
        "aaAthleteId"
    ]
    .map(
        ATHLETE_INDEX
    )
)


TEST[
    "athlete_idx"
] = (
    TEST[
        "aaAthleteId"
    ]
    .map(
        ATHLETE_INDEX
    )
)


# These are established-only datasets, so every row must map.
if VALIDATION[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "Established VALIDATION athlete missing from TRAIN athlete map."
    )


if TEST[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "Established TEST athlete missing from TRAIN athlete map."
    )


VALIDATION[
    "athlete_idx"
] = (
    VALIDATION[
        "athlete_idx"
    ]
    .astype(int)
)


TEST[
    "athlete_idx"
] = (
    TEST[
        "athlete_idx"
    ]
    .astype(int)
)


# --------------------------------------------------------------------------------------------------
# 6. TRAIN-ONLY 7-COLUMN CUBIC AGE SPLINE
# --------------------------------------------------------------------------------------------------
#
# Exact V4 form:
#
#   bs(
#       age,
#       df=7,
#       degree=3,
#       include_intercept=False,
#       lower_bound=13,
#       upper_bound=35
#   ) - 1
#
#
# Patsy determines the four internal knots from TRAIN.
#
# The TRAIN design_info is then reused unchanged for VALIDATION and TEST.
#
# --------------------------------------------------------------------------------------------------

SPLINE_FORMULA = (
    "bs("
    "age, "
    "df=7, "
    "degree=3, "
    "include_intercept=False, "
    "lower_bound=13, "
    "upper_bound=35"
    ") - 1"
)


B_TRAIN_RAW_DF = dmatrix(
    SPLINE_FORMULA,

    {
        "age":
            TRAIN[
                "age_years"
            ]
            .to_numpy(
                dtype=float
            )
    },

    return_type="dataframe",
)


SPLINE_DESIGN_INFO = (
    B_TRAIN_RAW_DF.design_info
)


B_TRAIN_RAW = np.asarray(
    B_TRAIN_RAW_DF,
    dtype=float,
)


B_VALIDATION_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                VALIDATION[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


B_TEST_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                TEST[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 7. TRAIN-ONLY SPLINE CENTERING
# --------------------------------------------------------------------------------------------------

SPLINE_COLUMN_MEANS = (
    B_TRAIN_RAW.mean(
        axis=0
    )
)


B_TRAIN = (
    B_TRAIN_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_VALIDATION = (
    B_VALIDATION_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_TEST = (
    B_TEST_RAW
    -
    SPLINE_COLUMN_MEANS
)


# Patsy with df=7 / degree=3 uses four interior knots.
# These quantiles are printed for transparent fold auditing.
SPLINE_INTERNAL_KNOT_QUANTILES = np.array(
    [
        0.20,
        0.40,
        0.60,
        0.80,
    ],
    dtype=float,
)


SPLINE_INTERNAL_KNOTS = np.quantile(
    TRAIN[
        "age_years"
    ]
    .to_numpy(
        dtype=float
    ),

    SPLINE_INTERNAL_KNOT_QUANTILES,
)


# --------------------------------------------------------------------------------------------------
# 8. ATTACH SPLINE COLUMNS FOR AUDIT / PERSISTENCE
# --------------------------------------------------------------------------------------------------

SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(
        B_TRAIN.shape[
            1
        ]
    )
]


for i, column in enumerate(
    SPLINE_COLUMNS
):

    TRAIN[
        column
    ] = (
        B_TRAIN[
            :,
            i
        ]
    )


    VALIDATION[
        column
    ] = (
        B_VALIDATION[
            :,
            i
        ]
    )


    TEST[
        column
    ] = (
        B_TEST[
            :,
            i
        ]
    )


# --------------------------------------------------------------------------------------------------
# 9. LEAKAGE-SAFE DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------
#
# Build chronology using only dates available through the historical TEST end.
#
# Multiple same-day performances remain multiple observations,
# but all share the same previous DISTINCT competition date.
#
# --------------------------------------------------------------------------------------------------

CHRONOLOGY_SOURCE = (
    BASE.loc[
        (
            BASE[
                "aaAthleteId"
            ]
            .isin(
                TRAIN_ATHLETES
            )
        )

        &

        (
            BASE[
                "result_date_dt"
            ]
            <=
            TEST_END
        )
    ][
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


CHRONOLOGY_SOURCE[
    "previous_distinct_date"
] = (
    CHRONOLOGY_SOURCE.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(1)
)


CHRONOLOGY_SOURCE[
    "state_gap_days"
] = (
    CHRONOLOGY_SOURCE[
        "result_date_dt"
    ]

    -

    CHRONOLOGY_SOURCE[
        "previous_distinct_date"
    ]
).dt.days


def attach_chronology(
    df,
):

    return df.merge(
        CHRONOLOGY_SOURCE,

        on=[
            "aaAthleteId",
            "result_date_dt",
        ],

        how="left",

        validate="many_to_one",
    )


TRAIN = attach_chronology(
    TRAIN
)


VALIDATION = attach_chronology(
    VALIDATION
)


TEST = attach_chronology(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 10. QA
# --------------------------------------------------------------------------------------------------

DATE_SPLIT_PASS = bool(
    (
        TRAIN[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    )
    .all()

    and

    VALIDATION[
        "result_date_dt"
    ]
    .between(
        VALIDATION_START,
        VALIDATION_END,
        inclusive="both",
    )
    .all()

    and

    TEST[
        "result_date_dt"
    ]
    .between(
        TEST_START,
        TEST_END,
        inclusive="both",
    )
    .all()
)


TRAIN_ONLY_ATHLETE_MAP_PASS = bool(
    VALIDATION[
        "athlete_idx"
    ]
    .notna()
    .all()

    and

    TEST[
        "athlete_idx"
    ]
    .notna()
    .all()
)


FEATURE_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "sex_idx",
                "group_idx",
                "is_youth_implement",
                "age_scaled",
                "athlete_idx",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()

    and

    np.isfinite(
        VALIDATION[
            [
                "sex_idx",
                "group_idx",
                "is_youth_implement",
                "age_scaled",
                "athlete_idx",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()

    and

    np.isfinite(
        TEST[
            [
                "sex_idx",
                "group_idx",
                "is_youth_implement",
                "age_scaled",
                "athlete_idx",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


SPLINE_DIMENSION_PASS = bool(
    B_TRAIN.shape[
        1
    ]
    ==
    7

    and

    B_VALIDATION.shape[
        1
    ]
    ==
    7

    and

    B_TEST.shape[
        1
    ]
    ==
    7
)


SPLINE_FINITE_PASS = bool(
    np.isfinite(
        B_TRAIN
    )
    .all()

    and

    np.isfinite(
        B_VALIDATION
    )
    .all()

    and

    np.isfinite(
        B_TEST
    )
    .all()
)


SPLINE_CENTERING_PASS = bool(
    np.max(
        np.abs(
            B_TRAIN.mean(
                axis=0
            )
        )
    )
    <
    1e-10
)


VALIDATION_CHRONOLOGY_PASS = bool(
    VALIDATION[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        VALIDATION[
            "state_gap_days"
        ]
        >
        0
    )
    .all()

    and

    (
        VALIDATION[
            "previous_distinct_date"
        ]
        <
        VALIDATION[
            "result_date_dt"
        ]
    )
    .all()
)


TEST_CHRONOLOGY_PASS = bool(
    TEST[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        TEST[
            "state_gap_days"
        ]
        >
        0
    )
    .all()

    and

    (
        TEST[
            "previous_distinct_date"
        ]
        <
        TEST[
            "result_date_dt"
        ]
    )
    .all()
)


NO_FUTURE_TRAIN_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    TRAIN_END
)


CELL_28_PASS = bool(
    DATE_SPLIT_PASS
    and
    TRAIN_ONLY_ATHLETE_MAP_PASS
    and
    FEATURE_FINITE_PASS
    and
    SPLINE_DIMENSION_PASS
    and
    SPLINE_FINITE_PASS
    and
    SPLINE_CENTERING_PASS
    and
    VALIDATION_CHRONOLOGY_PASS
    and
    TEST_CHRONOLOGY_PASS
    and
    NO_FUTURE_TRAIN_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 11. SAVE HISTORICAL FOLD PREPARATION
# --------------------------------------------------------------------------------------------------

HB2023_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    HB_FOLD_NAME
)


HB2023_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


HB2023_TRAIN_PATH = (
    HB2023_DIR
    /
    "train_prepared.parquet"
)


HB2023_VALIDATION_PATH = (
    HB2023_DIR
    /
    "validation_established_prepared.parquet"
)


HB2023_TEST_PATH = (
    HB2023_DIR
    /
    "test_established_prepared.parquet"
)


HB2023_PREP_SPEC_PATH = (
    HB2023_DIR
    /
    "fold_preparation_spec.json"
)


TRAIN.to_parquet(
    HB2023_TRAIN_PATH,
    index=False,
)


VALIDATION.to_parquet(
    HB2023_VALIDATION_PATH,
    index=False,
)


TEST.to_parquet(
    HB2023_TEST_PATH,
    index=False,
)


HB2023_PREP_SPEC = {
    "fold":
        HB_FOLD_NAME,

    "status":
        (
            "PASSED"
            if CELL_28_PASS
            else
            "FAILED"
        ),

    "train_end":
        TRAIN_END.isoformat(),

    "validation_start":
        VALIDATION_START.isoformat(),

    "validation_end":
        VALIDATION_END.isoformat(),

    "test_start":
        TEST_START.isoformat(),

    "test_end":
        TEST_END.isoformat(),

    "train_rows":
        int(
            len(TRAIN)
        ),

    "train_athletes":
        int(
            TRAIN[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "validation_established_rows":
        int(
            len(VALIDATION)
        ),

    "validation_established_athletes":
        int(
            VALIDATION[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "test_established_rows":
        int(
            len(TEST)
        ),

    "test_established_athletes":
        int(
            TEST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "sex_index":
        SEX_INDEX,

    "group_index":
        GROUP_INDEX,

    "youth_groups":
        sorted(
            YOUTH_GROUPS
        ),

    "age_center":
        AGE_CENTER,

    "age_scale":
        AGE_SCALE,

    "spline_formula":
        SPLINE_FORMULA,

    "spline_internal_knots":
        [
            float(x)
            for x in SPLINE_INTERNAL_KNOTS
        ],

    "spline_column_means":
        [
            float(x)
            for x in SPLINE_COLUMN_MEANS
        ],

    "spline_columns":
        SPLINE_COLUMNS,

    "athlete_index_count":
        int(
            len(
                ATHLETE_INDEX
            )
        ),
}


with HB2023_PREP_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        HB2023_PREP_SPEC,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 12. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    HB_FOLD_NAME
] = {
    "preparation_status":
        (
            "PASSED"
            if CELL_28_PASS
            else
            "FAILED"
        ),

    "directory":
        str(
            HB2023_DIR
        ),

    "preparation_spec":
        str(
            HB2023_PREP_SPEC_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_PREPARED_FOR_V4_REFIT"
    if CELL_28_PASS
    else
    "HB_2023_PREPARATION_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 13. GLOBALS FOR NEXT CELL
# --------------------------------------------------------------------------------------------------

HB2023_TRAIN = TRAIN

HB2023_VALIDATION = VALIDATION

HB2023_TEST = TEST


HB2023_ATHLETE_LEVELS = (
    ATHLETE_LEVELS
)


HB2023_ATHLETE_INDEX = (
    ATHLETE_INDEX
)


HB2023_SPLINE_DESIGN_INFO = (
    SPLINE_DESIGN_INFO
)


HB2023_SPLINE_COLUMN_MEANS = (
    SPLINE_COLUMN_MEANS
)


HB2023_SPLINE_INTERNAL_KNOTS = (
    SPLINE_INTERNAL_KNOTS
)


HB2023_B_TRAIN = (
    B_TRAIN
)


HB2023_B_VALIDATION = (
    B_VALIDATION
)


HB2023_B_TEST = (
    B_TEST
)


NOTEBOOK_2_CELL_28_PASS = (
    CELL_28_PASS
)


# --------------------------------------------------------------------------------------------------
# 14. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 150)
print("HB_2023 — HISTORICAL V4 REFIT PREPARATION")
print("=" * 150)


print()
print("TEMPORAL SPLIT")
print("-" * 150)

print(
    f"TRAIN       : <= {TRAIN_END.date()}"
)

print(
    f"VALIDATION  : "
    f"{VALIDATION_START.date()} -> {VALIDATION_END.date()}"
)

print(
    f"TEST        : "
    f"{TEST_START.date()} -> {TEST_END.date()}"
)


print()
print("POPULATION")
print("-" * 150)

print(
    f"TRAIN rows / athletes          : "
    f"{len(TRAIN):,} / "
    f"{TRAIN['aaAthleteId'].nunique():,}"
)

print(
    f"VALIDATION established         : "
    f"{len(VALIDATION):,} / "
    f"{VALIDATION['aaAthleteId'].nunique():,}"
)

print(
    f"TEST established               : "
    f"{len(TEST):,} / "
    f"{TEST['aaAthleteId'].nunique():,}"
)


print()
print("TRAIN-ONLY SPLINE")
print("-" * 150)

print(
    f"Basis shape                    : "
    f"{B_TRAIN.shape}"
)

print(
    "Internal knots                : "
    +
    np.array2string(
        SPLINE_INTERNAL_KNOTS,
        precision=10,
    )
)

print(
    "Column means                  : "
    +
    np.array2string(
        SPLINE_COLUMN_MEANS,
        precision=10,
    )
)

print(
    "Centered TRAIN mean           : "
    +
    np.array2string(
        B_TRAIN.mean(
            axis=0
        ),
        precision=3,
        suppress_small=True,
    )
)


print()
print("STATE-GAP SUMMARY")
print("-" * 150)

gap_summary = pd.DataFrame(
    [
        {
            "split":
                "VALIDATION",

            "n":
                len(VALIDATION),

            "median_gap_days":
                float(
                    VALIDATION[
                        "state_gap_days"
                    ]
                    .median()
                ),

            "mean_gap_days":
                float(
                    VALIDATION[
                        "state_gap_days"
                    ]
                    .mean()
                ),

            "min_gap_days":
                int(
                    VALIDATION[
                        "state_gap_days"
                    ]
                    .min()
                ),

            "max_gap_days":
                int(
                    VALIDATION[
                        "state_gap_days"
                    ]
                    .max()
                ),
        },

        {
            "split":
                "TEST",

            "n":
                len(TEST),

            "median_gap_days":
                float(
                    TEST[
                        "state_gap_days"
                    ]
                    .median()
                ),

            "mean_gap_days":
                float(
                    TEST[
                        "state_gap_days"
                    ]
                    .mean()
                ),

            "min_gap_days":
                int(
                    TEST[
                        "state_gap_days"
                    ]
                    .min()
                ),

            "max_gap_days":
                int(
                    TEST[
                        "state_gap_days"
                    ]
                    .max()
                ),
        },
    ]
)


display(
    gap_summary
)


print()
print("QA")
print("-" * 150)

print(
    f"Temporal split                 : "
    f"{DATE_SPLIT_PASS}"
)

print(
    f"TRAIN-only athlete map         : "
    f"{TRAIN_ONLY_ATHLETE_MAP_PASS}"
)

print(
    f"Finite V4 features             : "
    f"{FEATURE_FINITE_PASS}"
)

print(
    f"7-column spline                : "
    f"{SPLINE_DIMENSION_PASS}"
)

print(
    f"Finite spline values           : "
    f"{SPLINE_FINITE_PASS}"
)

print(
    f"TRAIN-centered spline          : "
    f"{SPLINE_CENTERING_PASS}"
)

print(
    f"VALIDATION chronology          : "
    f"{VALIDATION_CHRONOLOGY_PASS}"
)

print(
    f"TEST chronology                : "
    f"{TEST_CHRONOLOGY_PASS}"
)

print(
    f"No future TRAIN data           : "
    f"{NO_FUTURE_TRAIN_DATA_PASS}"
)


print()
print("=" * 150)

if CELL_28_PASS:

    print(
        "CELL 28 STATUS: HB_2023 HISTORICAL V4 PREPARATION PASSED"
    )

else:

    print(
        "CELL 28 STATUS: HB_2023 PREPARATION REVIEW REQUIRED"
    )

print("=" * 150)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 19
# Original source SHA256: 02046bd9c284f8d68f4a57d48740e4c923b0f6a9f21bb44abd8dee746cfc8e26
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 29 — HB_2023 HISTORICAL V4 BAYESIAN REFIT
# ==================================================================================================
#
# Purpose
# -------
# Fit the production V4 structural Bayesian model from scratch using ONLY:
#
#       HB_2023 TRAIN <= 2022-12-31
#
#
# Model
# -----
#
#   y_it ~ StudentT(
#       nu,
#       mu_it,
#       sigma_group[g]
#   )
#
#
#   mu_it =
#
#       alpha_sex
#
#       + spline_sex(age)
#
#       + youth_implement_effect_sex * I(youth implement)
#
#       + athlete_intercept_i
#
#       + athlete_slope_i * age_scaled
#
#
#   age_scaled = (age - 18) / 5
#
#
# Exact production priors
# -----------------------
#
#   alpha_sex ~ Normal(45, 15)
#
#   sigma_spline ~ HalfNormal(2)
#
#   spline_start ~ Normal(0, 3)
#
#   spline_delta ~ Normal(0, sigma_spline)
#
#   beta_spline =
#       [spline_start,
#        spline_start + cumulative spline_delta]
#
#   youth_implement_effect ~ Normal(0, 8)
#
#   sigma_athlete_intercept ~ HalfNormal(10)
#   athlete_intercept_z ~ Normal(0, 1)
#   athlete_intercept = sigma_athlete_intercept * z
#
#   sigma_athlete_slope ~ HalfNormal(4)
#   athlete_slope_z ~ Normal(0, 1)
#   athlete_slope = sigma_athlete_slope * z
#
#   sigma_group ~ HalfNormal(5)
#
#   nu_minus_two ~ Exponential(0.1)
#   nu = nu_minus_two + 2
#
#
# Inference
# ---------
#
#   ADVI iterations : 40,000
#   Posterior draws : 2,000
#   Random seed     : 42
#
#
# IMPORTANT
# ---------
#
# This historical refit does NOT use:
#
#   • 2023 VALIDATION marks for fitting
#   • 2023 TEST marks for fitting
#   • 2024-2026 data
#
# ==================================================================================================

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

import pymc as pm
import pytensor.tensor as pt


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_28_PASS",

    "HB2023_TRAIN",
    "HB2023_B_TRAIN",

    "HB2023_ATHLETE_LEVELS",
    "HB2023_ATHLETE_INDEX",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_28_PASS:

    raise RuntimeError(
        "Cell 28 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED INFERENCE SETTINGS
# --------------------------------------------------------------------------------------------------

ADVI_ITERATIONS = 40_000
POSTERIOR_DRAWS = 2_000
RANDOM_SEED = 42


SEX_LABELS = [
    "M",
    "W",
]


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_SEX = 2
N_GROUP = 4
N_SPLINE = 7
N_SPLINE_DELTA = 6


# --------------------------------------------------------------------------------------------------
# 3. TRAIN ARRAYS
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2023_TRAIN.copy()
    .reset_index(
        drop=True
    )
)


B_TRAIN = np.asarray(
    HB2023_B_TRAIN,
    dtype=np.float64,
)


y_train = (
    TRAIN[
        "mark_numeric_m"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


sex_idx_train = (
    TRAIN[
        "sex_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


group_idx_train = (
    TRAIN[
        "group_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


athlete_idx_train = (
    TRAIN[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


youth_train = (
    TRAIN[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


age_scaled_train = (
    TRAIN[
        "age_scaled"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


N_ATHLETES = len(
    HB2023_ATHLETE_LEVELS
)


# --------------------------------------------------------------------------------------------------
# 4. PRE-FIT QA
# --------------------------------------------------------------------------------------------------

if B_TRAIN.shape != (
    len(TRAIN),
    N_SPLINE,
):

    raise RuntimeError(
        "Unexpected HB_2023 spline shape: "
        f"{B_TRAIN.shape}"
    )


if not np.isfinite(
    y_train
).all():

    raise RuntimeError(
        "Non-finite TRAIN marks."
    )


if not np.isfinite(
    B_TRAIN
).all():

    raise RuntimeError(
        "Non-finite TRAIN spline values."
    )


if (
    sex_idx_train.min() < 0
    or
    sex_idx_train.max() >= N_SEX
):

    raise RuntimeError(
        "Invalid sex index."
    )


if (
    group_idx_train.min() < 0
    or
    group_idx_train.max() >= N_GROUP
):

    raise RuntimeError(
        "Invalid group index."
    )


if (
    athlete_idx_train.min() < 0
    or
    athlete_idx_train.max() >= N_ATHLETES
):

    raise RuntimeError(
        "Invalid athlete index."
    )


# --------------------------------------------------------------------------------------------------
# 5. MODEL COORDINATES
# --------------------------------------------------------------------------------------------------

coords = {
    "sex":
        SEX_LABELS,

    "group":
        GROUP_LABELS,

    "spline_basis":
        np.arange(
            N_SPLINE
        ),

    "spline_step":
        np.arange(
            N_SPLINE_DELTA
        ),

    "athlete":
        np.asarray(
            HB2023_ATHLETE_LEVELS
        ),

    "observation":
        np.arange(
            len(TRAIN)
        ),
}


# --------------------------------------------------------------------------------------------------
# 6. BUILD + FIT V4
# --------------------------------------------------------------------------------------------------

with pm.Model(
    coords=coords
) as HB2023_V4_MODEL:

    # ----------------------------------------------------------------------------------------------
    # Static model inputs
    # ----------------------------------------------------------------------------------------------

    sex_idx_data = pm.Data(
        "sex_idx_data",
        sex_idx_train,
        dims="observation",
    )


    group_idx_data = pm.Data(
        "group_idx_data",
        group_idx_train,
        dims="observation",
    )


    athlete_idx_data = pm.Data(
        "athlete_idx_data",
        athlete_idx_train,
        dims="observation",
    )


    youth_data = pm.Data(
        "youth_data",
        youth_train,
        dims="observation",
    )


    age_scaled_data = pm.Data(
        "age_scaled_data",
        age_scaled_train,
        dims="observation",
    )


    spline_data = pm.Data(
        "spline_data",
        B_TRAIN,
        dims=(
            "observation",
            "spline_basis",
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Population intercept by sex
    # ----------------------------------------------------------------------------------------------

    alpha_sex = pm.Normal(
        "alpha_sex",
        mu=45.0,
        sigma=15.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Sex-specific spline random walk
    # ----------------------------------------------------------------------------------------------

    sigma_spline = pm.HalfNormal(
        "sigma_spline",
        sigma=2.0,
        dims="sex",
    )


    spline_start = pm.Normal(
        "spline_start",
        mu=0.0,
        sigma=3.0,
        dims="sex",
    )


    spline_delta = pm.Normal(
        "spline_delta",
        mu=0.0,
        sigma=sigma_spline[
            :,
            None
        ],
        dims=(
            "sex",
            "spline_step",
        ),
    )


    beta_spline = pm.Deterministic(
        "beta_spline",

        pt.concatenate(
            [
                spline_start[
                    :,
                    None
                ],

                spline_start[
                    :,
                    None
                ]
                +
                pt.cumsum(
                    spline_delta,
                    axis=1,
                ),
            ],
            axis=1,
        ),

        dims=(
            "sex",
            "spline_basis",
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Youth implement effect by sex
    # ----------------------------------------------------------------------------------------------

    youth_implement_effect = pm.Normal(
        "youth_implement_effect",
        mu=0.0,
        sigma=8.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random intercept
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_intercept = pm.HalfNormal(
        "sigma_athlete_intercept",
        sigma=10.0,
    )


    athlete_intercept_z = pm.Normal(
        "athlete_intercept_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_intercept = pm.Deterministic(
        "athlete_intercept",

        sigma_athlete_intercept
        *
        athlete_intercept_z,

        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random age slope
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_slope = pm.HalfNormal(
        "sigma_athlete_slope",
        sigma=4.0,
    )


    athlete_slope_z = pm.Normal(
        "athlete_slope_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_slope = pm.Deterministic(
        "athlete_slope",

        sigma_athlete_slope
        *
        athlete_slope_z,

        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Group-specific observation scale
    # ----------------------------------------------------------------------------------------------

    sigma_group = pm.HalfNormal(
        "sigma_group",
        sigma=5.0,
        dims="group",
    )


    # ----------------------------------------------------------------------------------------------
    # Student-t degrees of freedom
    # ----------------------------------------------------------------------------------------------

    nu_minus_two = pm.Exponential(
        "nu_minus_two",
        lam=0.1,
    )


    nu = pm.Deterministic(
        "nu",
        nu_minus_two
        +
        2.0,
    )


    # ----------------------------------------------------------------------------------------------
    # Mean structure
    # ----------------------------------------------------------------------------------------------

    spline_effect = (
        spline_data
        *
        beta_spline[
            sex_idx_data
        ]
    ).sum(
        axis=1
    )


    mu = (
        alpha_sex[
            sex_idx_data
        ]

        +

        spline_effect

        +

        youth_implement_effect[
            sex_idx_data
        ]
        *
        youth_data

        +

        athlete_intercept[
            athlete_idx_data
        ]

        +

        athlete_slope[
            athlete_idx_data
        ]
        *
        age_scaled_data
    )


    # ----------------------------------------------------------------------------------------------
    # Likelihood
    # ----------------------------------------------------------------------------------------------

    observed = pm.StudentT(
        "observed",

        nu=nu,

        mu=mu,

        sigma=sigma_group[
            group_idx_data
        ],

        observed=y_train,

        dims="observation",
    )


    # ----------------------------------------------------------------------------------------------
    # ADVI
    # ----------------------------------------------------------------------------------------------

    HB2023_V4_APPROX = pm.fit(
        n=ADVI_ITERATIONS,
        method="advi",
        random_seed=RANDOM_SEED,
        progressbar=True,
    )


    # ----------------------------------------------------------------------------------------------
    # Posterior draws from variational approximation
    # ----------------------------------------------------------------------------------------------

    HB2023_V4_IDATA = (
        HB2023_V4_APPROX.sample(
            draws=POSTERIOR_DRAWS,
            random_seed=RANDOM_SEED,
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. ADVI HISTORY
# --------------------------------------------------------------------------------------------------

ADVI_HISTORY = np.asarray(
    HB2023_V4_APPROX.hist,
    dtype=float,
)


ADVI_FINITE_PASS = bool(
    len(
        ADVI_HISTORY
    )
    ==
    ADVI_ITERATIONS

    and

    np.isfinite(
        ADVI_HISTORY
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 8. POSTERIOR MEAN HELPER
# --------------------------------------------------------------------------------------------------

def posterior_mean(
    name,
):

    da = (
        HB2023_V4_IDATA
        .posterior[
            name
        ]
    )


    dims_to_average = [
        dim
        for dim in [
            "chain",
            "draw",
        ]
        if dim in da.dims
    ]


    return (
        da.mean(
            dim=dims_to_average
        )
        .values
    )


# --------------------------------------------------------------------------------------------------
# 9. EXTRACT REQUIRED POSTERIOR MEANS
# --------------------------------------------------------------------------------------------------

HB2023_V4_POSTERIOR_MEANS = {

    "alpha_sex":
        np.asarray(
            posterior_mean(
                "alpha_sex"
            ),
            dtype=float,
        ),

    "beta_spline":
        np.asarray(
            posterior_mean(
                "beta_spline"
            ),
            dtype=float,
        ),

    "youth_implement_effect":
        np.asarray(
            posterior_mean(
                "youth_implement_effect"
            ),
            dtype=float,
        ),

    "athlete_intercept":
        np.asarray(
            posterior_mean(
                "athlete_intercept"
            ),
            dtype=float,
        ),

    "athlete_slope":
        np.asarray(
            posterior_mean(
                "athlete_slope"
            ),
            dtype=float,
        ),

    "sigma_group":
        np.asarray(
            posterior_mean(
                "sigma_group"
            ),
            dtype=float,
        ),

    "nu":
        float(
            np.asarray(
                posterior_mean(
                    "nu"
                )
            )
        ),

    "sigma_athlete_intercept":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_intercept"
                )
            )
        ),

    "sigma_athlete_slope":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_slope"
                )
            )
        ),

    "sigma_spline":
        np.asarray(
            posterior_mean(
                "sigma_spline"
            ),
            dtype=float,
        ),
}


# --------------------------------------------------------------------------------------------------
# 10. POSTERIOR SHAPE QA
# --------------------------------------------------------------------------------------------------

POSTERIOR_SHAPE_PASS = bool(

    HB2023_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ].shape
    ==
    (
        N_SEX,
    )

    and

    HB2023_V4_POSTERIOR_MEANS[
        "beta_spline"
    ].shape
    ==
    (
        N_SEX,
        N_SPLINE,
    )

    and

    HB2023_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ].shape
    ==
    (
        N_SEX,
    )

    and

    HB2023_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ].shape
    ==
    (
        N_ATHLETES,
    )

    and

    HB2023_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ].shape
    ==
    (
        N_ATHLETES,
    )

    and

    HB2023_V4_POSTERIOR_MEANS[
        "sigma_group"
    ].shape
    ==
    (
        N_GROUP,
    )
)


# --------------------------------------------------------------------------------------------------
# 11. RECONSTRUCT TRAIN STRUCTURAL MEAN FROM POSTERIOR MEANS
# --------------------------------------------------------------------------------------------------

alpha_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ]
)


beta_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "beta_spline"
    ]
)


youth_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ]
)


athlete_intercept_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ]
)


athlete_slope_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ]
)


sigma_group_hat = (
    HB2023_V4_POSTERIOR_MEANS[
        "sigma_group"
    ]
)


TRAIN[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx_train
    ]

    +

    np.sum(
        B_TRAIN
        *
        beta_hat[
            sex_idx_train
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx_train
    ]
    *
    youth_train
)


TRAIN[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx_train
    ]
)


TRAIN[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx_train
    ]

    *

    age_scaled_train
)


TRAIN[
    "v4_structural_mean"
] = (
    TRAIN[
        "v4_population_component"
    ]

    +

    TRAIN[
        "v4_athlete_intercept_component"
    ]

    +

    TRAIN[
        "v4_athlete_slope_component"
    ]
)


TRAIN[
    "v4_residual"
] = (
    TRAIN[
        "mark_numeric_m"
    ]

    -

    TRAIN[
        "v4_structural_mean"
    ]
)


TRAIN[
    "v4_error"
] = (
    TRAIN[
        "v4_residual"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. TRAIN FIT METRICS
# --------------------------------------------------------------------------------------------------

TRAIN_MAE = float(
    TRAIN[
        "v4_error"
    ]
    .abs()
    .mean()
)


TRAIN_RMSE = float(
    np.sqrt(
        np.mean(
            TRAIN[
                "v4_error"
            ]
            **
            2
        )
    )
)


TRAIN_BIAS = float(
    TRAIN[
        "v4_error"
    ]
    .mean()
)


TRAIN_METRICS = pd.DataFrame(
    [
        {
            "fold":
                "HB_2023",

            "n":
                int(
                    len(TRAIN)
                ),

            "athletes":
                int(
                    TRAIN[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "MAE":
                TRAIN_MAE,

            "RMSE":
                TRAIN_RMSE,

            "Bias":
                TRAIN_BIAS,
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. POSTERIOR HYPERPARAMETER SUMMARY
# --------------------------------------------------------------------------------------------------

POSTERIOR_SUMMARY = pd.DataFrame(
    [
        {
            "parameter":
                "nu",

            "value":
                HB2023_V4_POSTERIOR_MEANS[
                    "nu"
                ],
        },

        {
            "parameter":
                "sigma_athlete_intercept",

            "value":
                HB2023_V4_POSTERIOR_MEANS[
                    "sigma_athlete_intercept"
                ],
        },

        {
            "parameter":
                "sigma_athlete_slope",

            "value":
                HB2023_V4_POSTERIOR_MEANS[
                    "sigma_athlete_slope"
                ],
        },

        *[
            {
                "parameter":
                    f"sigma_group_{group}",

                "value":
                    float(
                        sigma_group_hat[
                            i
                        ]
                    ),
            }

            for i, group
            in enumerate(
                GROUP_LABELS
            )
        ],
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE COMPACT HISTORICAL POSTERIOR ARTIFACTS
# --------------------------------------------------------------------------------------------------
#
# We intentionally save posterior MEANS required for the historical backtest,
# rather than storing hundreds of MB of athlete-level posterior draws per fold.
#
# The inference method itself remains identical to the production fit:
# 2,000 draws are generated and averaged.
#
# --------------------------------------------------------------------------------------------------

HB2023_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    "HB_2023"
)


HB2023_POSTERIOR_MEANS_PATH = (
    HB2023_DIR
    /
    "v4_posterior_means.npz"
)


HB2023_ADVI_HISTORY_PATH = (
    HB2023_DIR
    /
    "v4_advi_history.npy"
)


HB2023_TRAIN_STRUCTURAL_PATH = (
    HB2023_DIR
    /
    "v4_train_structural_predictions.parquet"
)


HB2023_V4_FIT_SPEC_PATH = (
    HB2023_DIR
    /
    "v4_fit_spec.json"
)


np.savez_compressed(
    HB2023_POSTERIOR_MEANS_PATH,

    alpha_sex=
        HB2023_V4_POSTERIOR_MEANS[
            "alpha_sex"
        ],

    beta_spline=
        HB2023_V4_POSTERIOR_MEANS[
            "beta_spline"
        ],

    youth_implement_effect=
        HB2023_V4_POSTERIOR_MEANS[
            "youth_implement_effect"
        ],

    athlete_intercept=
        HB2023_V4_POSTERIOR_MEANS[
            "athlete_intercept"
        ],

    athlete_slope=
        HB2023_V4_POSTERIOR_MEANS[
            "athlete_slope"
        ],

    sigma_group=
        HB2023_V4_POSTERIOR_MEANS[
            "sigma_group"
        ],

    nu=np.array(
        [
            HB2023_V4_POSTERIOR_MEANS[
                "nu"
            ]
        ],
        dtype=float,
    ),

    sigma_athlete_intercept=np.array(
        [
            HB2023_V4_POSTERIOR_MEANS[
                "sigma_athlete_intercept"
            ]
        ],
        dtype=float,
    ),

    sigma_athlete_slope=np.array(
        [
            HB2023_V4_POSTERIOR_MEANS[
                "sigma_athlete_slope"
            ]
        ],
        dtype=float,
    ),

    sigma_spline=
        HB2023_V4_POSTERIOR_MEANS[
            "sigma_spline"
        ],
)


np.save(
    HB2023_ADVI_HISTORY_PATH,
    ADVI_HISTORY,
)


TRAIN.to_parquet(
    HB2023_TRAIN_STRUCTURAL_PATH,
    index=False,
)


HB2023_V4_FIT_SPEC = {

    "fold":
        "HB_2023",

    "status":
        "PASSED",

    "train_end":
        "2022-12-31",

    "train_rows":
        int(
            len(TRAIN)
        ),

    "train_athletes":
        int(
            N_ATHLETES
        ),

    "likelihood":
        "StudentT",

    "advi_iterations":
        ADVI_ITERATIONS,

    "posterior_draws":
        POSTERIOR_DRAWS,

    "random_seed":
        RANDOM_SEED,

    "priors": {
        "alpha_sex":
            "Normal(45,15)",

        "sigma_spline":
            "HalfNormal(2)",

        "spline_start":
            "Normal(0,3)",

        "spline_delta":
            "Normal(0,sigma_spline)",

        "youth_implement_effect":
            "Normal(0,8)",

        "sigma_athlete_intercept":
            "HalfNormal(10)",

        "athlete_intercept_z":
            "Normal(0,1)",

        "sigma_athlete_slope":
            "HalfNormal(4)",

        "athlete_slope_z":
            "Normal(0,1)",

        "sigma_group":
            "HalfNormal(5)",

        "nu_minus_two":
            "Exponential(0.1)",
    },

    "posterior_means_path":
        str(
            HB2023_POSTERIOR_MEANS_PATH
        ),

    "advi_history_path":
        str(
            HB2023_ADVI_HISTORY_PATH
        ),

    "train_structural_path":
        str(
            HB2023_TRAIN_STRUCTURAL_PATH
        ),

    "train_metrics": {
        "MAE":
            TRAIN_MAE,

        "RMSE":
            TRAIN_RMSE,

        "Bias":
            TRAIN_BIAS,
    },
}


with HB2023_V4_FIT_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        HB2023_V4_FIT_SPEC,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 15. QA
# --------------------------------------------------------------------------------------------------

POSTERIOR_FINITE_PASS = bool(

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "alpha_sex"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "beta_spline"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "youth_implement_effect"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "athlete_intercept"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "athlete_slope"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "sigma_group"
        ]
    )
    .all()

    and

    np.isfinite(
        HB2023_V4_POSTERIOR_MEANS[
            "nu"
        ]
    )
)


POSITIVE_SCALE_PASS = bool(

    HB2023_V4_POSTERIOR_MEANS[
        "sigma_athlete_intercept"
    ]
    >
    0

    and

    HB2023_V4_POSTERIOR_MEANS[
        "sigma_athlete_slope"
    ]
    >
    0

    and

    (
        HB2023_V4_POSTERIOR_MEANS[
            "sigma_group"
        ]
        >
        0
    )
    .all()

    and

    HB2023_V4_POSTERIOR_MEANS[
        "nu"
    ]
    >
    2
)


TRAIN_PREDICTION_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "v4_population_component",
                "v4_athlete_intercept_component",
                "v4_athlete_slope_component",
                "v4_structural_mean",
                "v4_residual",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


ARTIFACT_PASS = bool(
    HB2023_POSTERIOR_MEANS_PATH.exists()
    and
    HB2023_ADVI_HISTORY_PATH.exists()
    and
    HB2023_TRAIN_STRUCTURAL_PATH.exists()
    and
    HB2023_V4_FIT_SPEC_PATH.exists()
)


CELL_29_PASS = bool(
    ADVI_FINITE_PASS
    and
    POSTERIOR_SHAPE_PASS
    and
    POSTERIOR_FINITE_PASS
    and
    POSITIVE_SCALE_PASS
    and
    TRAIN_PREDICTION_FINITE_PASS
    and
    ARTIFACT_PASS
)


# --------------------------------------------------------------------------------------------------
# 16. MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    "HB_2023"
][
    "v4_refit"
] = {
    "status":
        (
            "PASSED"
            if CELL_29_PASS
            else
            "FAILED"
        ),

    "fit_spec":
        str(
            HB2023_V4_FIT_SPEC_PATH
        ),

    "posterior_means":
        str(
            HB2023_POSTERIOR_MEANS_PATH
        ),

    "train_structural_predictions":
        str(
            HB2023_TRAIN_STRUCTURAL_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_V4_REFIT_COMPLETE"
    if CELL_29_PASS
    else
    "HB_2023_V4_REFIT_FAILED"
)


with TRAINING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 17. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_TRAIN_V4 = TRAIN


NOTEBOOK_2_CELL_29_PASS = (
    CELL_29_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. RELEASE LARGE DRAW OBJECTS FROM MEMORY
# --------------------------------------------------------------------------------------------------
#
# All information required by the historical point-forecast backtest
# has been persisted above.
#
# --------------------------------------------------------------------------------------------------

del HB2023_V4_IDATA

gc.collect()


# --------------------------------------------------------------------------------------------------
# 19. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 150)
print("HB_2023 — HISTORICAL V4 BAYESIAN REFIT")
print("=" * 150)


print()
print("FIT CONFIGURATION")
print("-" * 150)

print(
    f"TRAIN rows              : "
    f"{len(TRAIN):,}"
)

print(
    f"TRAIN athletes          : "
    f"{N_ATHLETES:,}"
)

print(
    f"ADVI iterations         : "
    f"{ADVI_ITERATIONS:,}"
)

print(
    f"Posterior draws         : "
    f"{POSTERIOR_DRAWS:,}"
)

print(
    f"Random seed             : "
    f"{RANDOM_SEED}"
)


print()
print("ADVI")
print("-" * 150)

print(
    f"History length          : "
    f"{len(ADVI_HISTORY):,}"
)

print(
    f"Initial loss            : "
    f"{ADVI_HISTORY[0]:,.6f}"
)

print(
    f"Final loss              : "
    f"{ADVI_HISTORY[-1]:,.6f}"
)

print(
    f"Mean final 1,000 loss   : "
    f"{ADVI_HISTORY[-1000:].mean():,.6f}"
)


print()
print("POSTERIOR HYPERPARAMETER MEANS")
print("-" * 150)

display(
    POSTERIOR_SUMMARY
)


print()
print("TRAIN STRUCTURAL FIT")
print("-" * 150)

display(
    TRAIN_METRICS
)


print()
print("QA")
print("-" * 150)

print(
    f"ADVI finite             : "
    f"{ADVI_FINITE_PASS}"
)

print(
    f"Posterior shapes        : "
    f"{POSTERIOR_SHAPE_PASS}"
)

print(
    f"Posterior finite        : "
    f"{POSTERIOR_FINITE_PASS}"
)

print(
    f"Positive scales / nu>2 : "
    f"{POSITIVE_SCALE_PASS}"
)

print(
    f"TRAIN predictions finite: "
    f"{TRAIN_PREDICTION_FINITE_PASS}"
)

print(
    f"Artifacts written       : "
    f"{ARTIFACT_PASS}"
)


print()
print("=" * 150)

if CELL_29_PASS:

    print(
        "CELL 29 STATUS: HB_2023 HISTORICAL V4 REFIT PASSED"
    )

else:

    print(
        "CELL 29 STATUS: HB_2023 V4 REFIT REVIEW REQUIRED"
    )

print("=" * 150)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 20
# Original source SHA256: 2daee72bd0c4889443c883c1f848e8988fca78d2fbc88d0967e728a37e752340
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 30 — HB_2023 HISTORICAL V5 OU REFIT + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Refit the V5 continuous-time OU transient-state model using ONLY
# HB_2023 TRAIN residuals from the historical V4 refit.
#
#
# State observation
# -----------------
# One observation per athlete × DISTINCT competition date:
#
#       r_d = mean(V4 residuals on that athlete-date)
#
#
# Observation variance
# --------------------
#
#       R_d = sum_g n_dg * R_g / n_d^2
#
#
# State evolution
# ---------------
#
#       z_{t+Δ} | z_t
#           ~ Normal(
#                 exp(-Δ/tau) * z_t,
#                 Vz * [1 - exp(-2Δ/tau)]
#             )
#
#
# Estimated TRAIN-only parameters
# --------------------------------
#
#       tau
#       Vz
#       R_M700
#       R_M800
#       R_W500
#       R_W600
#
#
# IMPORTANT
# ---------
# • NO 2023 validation data used.
# • NO 2023 test data used.
# • NO current V5 parameter values used as optimizer targets.
# • Multiple same-day performances are collapsed to one athlete-level state observation.
#
#
# AUDIT / VERSIONING
# ------------------
# Every execution creates a NEW immutable attempt directory.
#
# Saved:
#   • state input
#   • optimizer starts/results
#   • fitted parameters
#   • complete fit/audit manifest
#   • SHA256 hashes
#   • parent-artifact hashes
#   • software versions
#   • git commit when available
#
# A lightweight LATEST_V5_OU_REFIT.json pointer is atomically updated.
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_29_PASS",

    "HB2023_TRAIN_V4",
    "HB2023_V4_POSTERIOR_MEANS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_29_PASS:

    raise RuntimeError(
        "Cell 29 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED MODEL DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

CELL_ID = "CELL_30_HB2023_V5_OU_REFIT"
STAGE_SCHEMA_VERSION = "1.0.0"

GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_GROUPS = len(
    GROUP_LABELS
)


# Broad numerical bounds.
TAU_MIN = 2.0
TAU_MAX = 5000.0

VAR_MIN = 1e-4
VAR_MAX = 400.0

N_STARTS = 10


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE VERSIONED ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2023_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V5_STAGE_ROOT = (
    HB2023_DIR
    /
    "v5_ou_refit"
)


V5_ATTEMPT_DIR = (
    V5_STAGE_ROOT
    /
    ATTEMPT_ID
)


V5_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HASH / AUDIT HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 5. HISTORICAL TRAIN RESIDUALS
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2023_TRAIN_V4.copy()
    .reset_index(
        drop=True
    )
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "model_group",
    "v4_residual",
]


missing_columns = [
    column
    for column in required_columns
    if column not in TRAIN.columns
]


if missing_columns:

    raise RuntimeError(
        "HB2023_TRAIN_V4 missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


TRAIN[
    "result_date_dt"
] = pd.to_datetime(
    TRAIN[
        "result_date_dt"
    ]
)


if TRAIN[
    "v4_residual"
].isna().any():

    raise RuntimeError(
        "HB_2023 TRAIN contains missing V4 residuals."
    )


# --------------------------------------------------------------------------------------------------
# 6. ONE STATE OBSERVATION PER ATHLETE × DISTINCT DATE
# --------------------------------------------------------------------------------------------------

daily = (
    TRAIN.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        state_observation=(
            "v4_residual",
            "mean",
        ),

        n_performances=(
            "v4_residual",
            "size",
        ),
    )
)


group_counts = pd.crosstab(
    index=[
        TRAIN[
            "aaAthleteId"
        ],

        TRAIN[
            "result_date_dt"
        ],
    ],

    columns=
        TRAIN[
            "model_group"
        ],
)


group_counts = (
    group_counts.reindex(
        columns=GROUP_LABELS,
        fill_value=0,
    )
    .reset_index()
)


group_counts.columns.name = None


for group in GROUP_LABELS:

    group_counts = group_counts.rename(
        columns={
            group:
                f"n_{group}"
        }
    )


daily = daily.merge(
    group_counts,

    on=[
        "aaAthleteId",
        "result_date_dt",
    ],

    how="left",

    validate="one_to_one",
)


daily = (
    daily.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------

daily[
    "previous_state_date"
] = (
    daily.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(1)
)


daily[
    "state_gap_days"
] = (
    daily[
        "result_date_dt"
    ]

    -

    daily[
        "previous_state_date"
    ]
).dt.days


daily[
    "athlete_state_start"
] = (
    daily[
        "previous_state_date"
    ]
    .isna()
)


# --------------------------------------------------------------------------------------------------
# 8. STATE-INPUT QA
# --------------------------------------------------------------------------------------------------

DUPLICATE_STATE_COUNT = int(
    daily.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
)


NONPOSITIVE_GAP_COUNT = int(
    (
        daily.loc[
            ~daily[
                "athlete_state_start"
            ],
            "state_gap_days",
        ]
        <=
        0
    )
    .sum()
)


FIRST_DATE_COUNT = int(
    daily[
        "athlete_state_start"
    ]
    .sum()
)


N_STATE_ATHLETES = int(
    daily[
        "aaAthleteId"
    ]
    .nunique()
)


FIRST_DATE_ACCOUNTING_PASS = bool(
    FIRST_DATE_COUNT
    ==
    N_STATE_ATHLETES
)


COUNT_ACCOUNTING_PASS = bool(

    (
        daily[
            [
                f"n_{group}"
                for group in GROUP_LABELS
            ]
        ]
        .sum(
            axis=1
        )
        .to_numpy()

        ==

        daily[
            "n_performances"
        ]
        .to_numpy()
    )
    .all()
)


STATE_INPUT_PASS = bool(
    DUPLICATE_STATE_COUNT
    ==
    0

    and

    NONPOSITIVE_GAP_COUNT
    ==
    0

    and

    FIRST_DATE_ACCOUNTING_PASS

    and

    COUNT_ACCOUNTING_PASS

    and

    np.isfinite(
        daily[
            "state_observation"
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


if not STATE_INPUT_PASS:

    raise RuntimeError(
        "Historical V5 state-input QA failed."
    )


# --------------------------------------------------------------------------------------------------
# 9. NUMERICAL ARRAYS
# --------------------------------------------------------------------------------------------------

state_y = (
    daily[
        "state_observation"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_n = (
    daily[
        "n_performances"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_counts = (
    daily[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_start = (
    daily[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=np.int8
    )
)


state_gap = (
    daily[
        "state_gap_days"
    ]
    .fillna(
        0
    )
    .to_numpy(
        dtype=np.float64
    )
)


# --------------------------------------------------------------------------------------------------
# 10. KALMAN NEGATIVE LOG-LIKELIHOOD
# --------------------------------------------------------------------------------------------------

LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


def kalman_nll_python(
    theta_log,
):

    tau = float(
        np.exp(
            theta_log[
                0
            ]
        )
    )


    Vz = float(
        np.exp(
            theta_log[
                1
            ]
        )
    )


    Rg = np.exp(
        theta_log[
            2:
            2 + N_GROUPS
        ]
    )


    m_post = 0.0
    P_post = Vz

    nll = 0.0


    for i in range(
        len(
            state_y
        )
    ):

        if state_start[
            i
        ]:

            m_prior = 0.0
            P_prior = Vz


        else:

            phi = math.exp(
                -
                state_gap[
                    i
                ]
                /
                tau
            )


            m_prior = (
                phi
                *
                m_post
            )


            P_prior = (
                phi
                *
                phi
                *
                P_post

                +

                Vz
                *
                (
                    1.0
                    -
                    phi
                    *
                    phi
                )
            )


        R_d = float(
            np.dot(
                state_counts[
                    i
                ],
                Rg,
            )

            /

            (
                state_n[
                    i
                ]
                **
                2
            )
        )


        S = (
            P_prior
            +
            R_d
        )


        if (
            not np.isfinite(S)
            or
            S <= 0
        ):

            return np.inf


        innovation = (
            state_y[
                i
            ]
            -
            m_prior
        )


        nll += 0.5 * (
            LOG_2PI
            +
            np.log(
                S
            )
            +
            innovation
            *
            innovation
            /
            S
        )


        K = (
            P_prior
            /
            S
        )


        m_post = (
            m_prior
            +
            K
            *
            innovation
        )


        P_post = (
            (
                1.0
                -
                K
            )
            *
            P_prior
        )


        P_post = max(
            P_post,
            1e-12,
        )


    return float(
        nll
    )


# --------------------------------------------------------------------------------------------------
# 11. OPTIONAL NUMBA ACCELERATION
# --------------------------------------------------------------------------------------------------

NUMBA_AVAILABLE = False


try:

    import numba
    from numba import njit


    @njit(
        cache=False
    )
    def kalman_nll_numba(
        theta_log,
        state_y_,
        state_n_,
        state_counts_,
        state_start_,
        state_gap_,
    ):

        tau = np.exp(
            theta_log[
                0
            ]
        )


        Vz = np.exp(
            theta_log[
                1
            ]
        )


        Rg = np.exp(
            theta_log[
                2:
                2 + N_GROUPS
            ]
        )


        m_post = 0.0
        P_post = Vz

        nll = 0.0


        for i in range(
            state_y_.shape[
                0
            ]
        ):

            if state_start_[
                i
            ] == 1:

                m_prior = 0.0
                P_prior = Vz


            else:

                phi = np.exp(
                    -
                    state_gap_[
                        i
                    ]
                    /
                    tau
                )


                m_prior = (
                    phi
                    *
                    m_post
                )


                P_prior = (
                    phi
                    *
                    phi
                    *
                    P_post

                    +

                    Vz
                    *
                    (
                        1.0
                        -
                        phi
                        *
                        phi
                    )
                )


            weighted_R = 0.0


            for g in range(
                N_GROUPS
            ):

                weighted_R += (
                    state_counts_[
                        i,
                        g
                    ]
                    *
                    Rg[
                        g
                    ]
                )


            R_d = (
                weighted_R
                /
                (
                    state_n_[
                        i
                    ]
                    *
                    state_n_[
                        i
                    ]
                )
            )


            S = (
                P_prior
                +
                R_d
            )


            if (
                not np.isfinite(
                    S
                )
                or
                S <= 0
            ):

                return np.inf


            innovation = (
                state_y_[
                    i
                ]
                -
                m_prior
            )


            nll += 0.5 * (
                LOG_2PI
                +
                np.log(
                    S
                )
                +
                innovation
                *
                innovation
                /
                S
            )


            K = (
                P_prior
                /
                S
            )


            m_post = (
                m_prior
                +
                K
                *
                innovation
            )


            P_post = (
                (
                    1.0
                    -
                    K
                )
                *
                P_prior
            )


            if P_post < 1e-12:

                P_post = 1e-12


        return nll


    # Compile once.
    _ = kalman_nll_numba(
        np.log(
            np.array(
                [
                    100.0,
                    3.0,
                    10.0,
                    8.0,
                    6.0,
                    6.0,
                ],
                dtype=np.float64,
            )
        ),

        state_y[
            : min(
                10,
                len(
                    state_y
                ),
            )
        ],

        state_n[
            : min(
                10,
                len(
                    state_n
                ),
            )
        ],

        state_counts[
            : min(
                10,
                len(
                    state_counts
                ),
            )
        ],

        state_start[
            : min(
                10,
                len(
                    state_start
                ),
            )
        ],

        state_gap[
            : min(
                10,
                len(
                    state_gap
                ),
            )
        ],
    )


    NUMBA_AVAILABLE = True


except Exception:

    NUMBA_AVAILABLE = False


def objective(
    theta_log,
):

    if NUMBA_AVAILABLE:

        return float(
            kalman_nll_numba(
                theta_log,
                state_y,
                state_n,
                state_counts,
                state_start,
                state_gap,
            )
        )


    return kalman_nll_python(
        theta_log
    )


# --------------------------------------------------------------------------------------------------
# 12. DETERMINISTIC OPTIMIZER STARTS
# --------------------------------------------------------------------------------------------------

v4_sigma_group = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "sigma_group"
    ],
    dtype=float,
)


R_START_BASE = (
    v4_sigma_group
    **
    2
)


STATE_VARIANCE_DIAGNOSTIC = float(
    np.var(
        state_y,
        ddof=1,
    )
)


start_specs = [
    (30.0,  0.10, 0.75),
    (45.0,  0.25, 1.00),
    (60.0,  0.20, 1.25),
    (90.0,  0.30, 1.00),
    (120.0, 0.50, 0.75),
    (180.0, 0.75, 1.00),
    (240.0, 1.00, 1.25),
    (365.0, 1.50, 1.00),
    (730.0, 2.00, 0.75),
    (150.0, 0.25, 1.25),
]


if len(
    start_specs
) != N_STARTS:

    raise RuntimeError(
        "Unexpected optimizer start count."
    )


log_bounds = [
    (
        np.log(
            TAU_MIN
        ),
        np.log(
            TAU_MAX
        ),
    ),

    (
        np.log(
            VAR_MIN
        ),
        np.log(
            VAR_MAX
        ),
    ),

    *[
        (
            np.log(
                VAR_MIN
            ),
            np.log(
                VAR_MAX
            ),
        )

        for _ in range(
            N_GROUPS
        )
    ],
]


optimizer_rows = []
optimizer_results = []


for start_id, (
    tau_start,
    vz_fraction,
    r_multiplier,
) in enumerate(
    start_specs,
    start=1,
):

    Vz_start = float(
        np.clip(
            STATE_VARIANCE_DIAGNOSTIC
            *
            vz_fraction,

            0.05,
            50.0,
        )
    )


    R_start = np.clip(
        R_START_BASE
        *
        r_multiplier,

        0.05,
        100.0,
    )


    x0 = np.log(
        np.concatenate(
            [
                [
                    tau_start,
                    Vz_start,
                ],

                R_start,
            ]
        )
    )


    result = minimize(
        objective,

        x0=x0,

        method="L-BFGS-B",

        bounds=log_bounds,

        options={
            "maxiter":
                3000,

            "ftol":
                1e-11,

            "gtol":
                1e-7,

            "maxls":
                50,
        },
    )


    optimizer_results.append(
        result
    )


    fitted = np.exp(
        result.x
    )


    optimizer_rows.append(
        {
            "start_id":
                start_id,

            "success":
                bool(
                    result.success
                ),

            "status":
                int(
                    result.status
                ),

            "message":
                str(
                    result.message
                ),

            "iterations":
                int(
                    result.nit
                ),

            "n_function_evals":
                int(
                    result.nfev
                ),

            "nll":
                float(
                    result.fun
                ),

            "tau_days":
                float(
                    fitted[
                        0
                    ]
                ),

            "state_variance":
                float(
                    fitted[
                        1
                    ]
                ),

            "state_sigma":
                float(
                    np.sqrt(
                        fitted[
                            1
                        ]
                    )
                ),

            "R_M700":
                float(
                    fitted[
                        2
                    ]
                ),

            "R_M800":
                float(
                    fitted[
                        3
                    ]
                ),

            "R_W500":
                float(
                    fitted[
                        4
                    ]
                ),

            "R_W600":
                float(
                    fitted[
                        5
                    ]
                ),
        }
    )


OPTIMIZER_AUDIT = pd.DataFrame(
    optimizer_rows
)


# --------------------------------------------------------------------------------------------------
# 13. SELECT BEST SUCCESSFUL OPTIMIZER RESULT
# --------------------------------------------------------------------------------------------------

successful_indices = [
    i
    for i, result in enumerate(
        optimizer_results
    )
    if (
        result.success
        and
        np.isfinite(
            result.fun
        )
    )
]


if not successful_indices:

    raise RuntimeError(
        "No successful HB_2023 V5 OU optimizer starts."
    )


best_index = min(
    successful_indices,
    key=lambda i:
        optimizer_results[
            i
        ].fun,
)


BEST_RESULT = (
    optimizer_results[
        best_index
    ]
)


BEST_PARAMS = np.exp(
    BEST_RESULT.x
)


V5_TAU_DAYS = float(
    BEST_PARAMS[
        0
    ]
)


V5_STATE_VARIANCE = float(
    BEST_PARAMS[
        1
    ]
)


V5_STATE_SIGMA = float(
    np.sqrt(
        V5_STATE_VARIANCE
    )
)


V5_R_GROUP = {
    group:
        float(
            BEST_PARAMS[
                2 + i
            ]
        )

    for i, group
    in enumerate(
        GROUP_LABELS
    )
}


V5_HALF_LIFE_DAYS = float(
    V5_TAU_DAYS
    *
    np.log(
        2.0
    )
)


BEST_NLL = float(
    BEST_RESULT.fun
)


N_PARAMETERS = (
    2
    +
    N_GROUPS
)


V5_AIC = float(
    2
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


V5_BIC = float(
    np.log(
        len(
            daily
        )
    )
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


# --------------------------------------------------------------------------------------------------
# 14. PERSISTENCE REFERENCE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        1,
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS,

        "phi":
            np.exp(
                -
                REFERENCE_DAYS
                /
                V5_TAU_DAYS
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 15. OPTIMIZER / BOUNDARY QA
# --------------------------------------------------------------------------------------------------

SUCCESS_COUNT = int(
    OPTIMIZER_AUDIT[
        "success"
    ]
    .sum()
)


BEST_LOG_PARAMS = (
    BEST_RESULT.x
)


BOUNDARY_TOL = 1e-4


BOUNDARY_FLAGS = []


for value, (
    lower,
    upper,
) in zip(
    BEST_LOG_PARAMS,
    log_bounds,
):

    boundary_hit = bool(
        abs(
            value
            -
            lower
        )
        <
        BOUNDARY_TOL

        or

        abs(
            value
            -
            upper
        )
        <
        BOUNDARY_TOL
    )


    BOUNDARY_FLAGS.append(
        boundary_hit
    )


NO_BOUNDARY_HIT_PASS = bool(
    not any(
        BOUNDARY_FLAGS
    )
)


ALL_PARAMS_FINITE_PASS = bool(
    np.isfinite(
        BEST_PARAMS
    )
    .all()
)


ALL_PARAMS_POSITIVE_PASS = bool(
    (
        BEST_PARAMS
        >
        0
    )
    .all()
)


MULTISTART_PASS = bool(
    SUCCESS_COUNT
    >=
    2
)


# --------------------------------------------------------------------------------------------------
# 16. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

STATE_INPUT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_state_input.parquet"
)


OPTIMIZER_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_optimizer_starts.parquet"
)


PARAMETER_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_ou_parameters.json"
)


FIT_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_fit_audit.json"
)


daily.to_parquet(
    STATE_INPUT_PATH,
    index=False,
)


OPTIMIZER_AUDIT.to_parquet(
    OPTIMIZER_AUDIT_PATH,
    index=False,
)


V5_PARAMETER_SPEC = {
    "model":
        "V5_CONTINUOUS_TIME_OU",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "fit_population":
        "HB_2023 TRAIN only",

    "train_end":
        "2022-12-31",

    "state_rows":
        int(
            len(
                daily
            )
        ),

    "state_athletes":
        N_STATE_ATHLETES,

    "parameters": {
        "tau_days":
            V5_TAU_DAYS,

        "half_life_days":
            V5_HALF_LIFE_DAYS,

        "state_variance":
            V5_STATE_VARIANCE,

        "state_sigma":
            V5_STATE_SIGMA,

        "measurement_variance_by_group":
            V5_R_GROUP,
    },

    "fit_statistics": {
        "negative_log_likelihood":
            BEST_NLL,

        "AIC":
            V5_AIC,

        "BIC":
            V5_BIC,

        "n_parameters":
            N_PARAMETERS,
    },

    "optimizer": {
        "method":
            "L-BFGS-B",

        "deterministic_starts":
            N_STARTS,

        "successful_starts":
            SUCCESS_COUNT,

        "best_start_id":
            int(
                best_index
                +
                1
            ),

        "bounds": {
            "tau_days": [
                TAU_MIN,
                TAU_MAX,
            ],

            "variance_parameters": [
                VAR_MIN,
                VAR_MAX,
            ],
        },
    },

    "measurement_variance_formula":
        "R_d = sum_g(n_dg * R_g) / n_d^2",

    "state_transition":
        (
            "phi(delta)=exp(-delta/tau); "
            "Var innovation=Vz*(1-phi(delta)^2)"
        ),

    "group_order":
        GROUP_LABELS,

    "randomness":
        "No stochastic fitting; deterministic optimizer starts",

    "numba_used":
        NUMBA_AVAILABLE,
}


atomic_json_write(
    V5_PARAMETER_SPEC,
    PARAMETER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 17. PARENT / ARTIFACT PROVENANCE
# --------------------------------------------------------------------------------------------------

PARENT_ARTIFACT_CANDIDATES = {
    "v4_posterior_means":
        (
            HB2023_DIR
            /
            "v4_posterior_means.npz"
        ),

    "v4_train_structural_predictions":
        (
            HB2023_DIR
            /
            "v4_train_structural_predictions.parquet"
        ),

    "v4_fit_spec":
        (
            HB2023_DIR
            /
            "v4_fit_spec.json"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in (
    PARENT_ARTIFACT_CANDIDATES.items()
):

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


OUTPUT_ARTIFACTS = {}


for name, path in {
    "state_input":
        STATE_INPUT_PATH,

    "optimizer_audit":
        OPTIMIZER_AUDIT_PATH,

    "parameter_spec":
        PARAMETER_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 18. COMPLETE AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

FIT_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "model_version":
        str(
            TRAINING_MANIFEST.get(
                "model_version",
                "UNKNOWN",
            )
        ),

    "training_run_directory":
        str(
            TRAINING_RUN_DIR
        ),

    "historical_fold":
        FOLD_NAME,

    "stage":
        "V5_OU_REFIT",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        "PENDING_QA",

    "change_reason":
        (
            "Historical temporal backtest of the "
            "V6.1 OU-freshness architecture."
        ),

    "parent_stage":
        "HB_2023 V4 historical refit",

    "data_vintage_note":
        (
            "Historical fold is reconstructed from the current "
            "2026 canonical snapshot and restricted by event/result date. "
            "Later corrections to historical records may therefore be present; "
            "no future-dated performances are used."
        ),

    "data_scope": {
        "train_end":
            "2022-12-31",

        "performance_rows":
            int(
                len(
                    TRAIN
                )
            ),

        "state_rows":
            int(
                len(
                    daily
                )
            ),

        "athletes":
            N_STATE_ATHLETES,

        "first_state_dates":
            FIRST_DATE_COUNT,

        "same_day_multi_performance_dates":
            int(
                (
                    daily[
                        "n_performances"
                    ]
                    >
                    1
                )
                .sum()
            ),

        "same_day_multi_group_dates":
            int(
                (
                    (
                        daily[
                            [
                                f"n_{group}"
                                for group in GROUP_LABELS
                            ]
                        ]
                        >
                        0
                    )
                    .sum(
                        axis=1
                    )
                    >
                    1
                )
                .sum()
            ),
    },

    "model_specification":
        V5_PARAMETER_SPEC,

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "scipy":
            scipy.__version__,

        "numba":
            (
                numba.__version__
                if NUMBA_AVAILABLE
                else
                None
            ),
    },

    "git_commit":
        GIT_COMMIT,
}


# --------------------------------------------------------------------------------------------------
# 19. FINAL QA
# --------------------------------------------------------------------------------------------------

CELL_30_PASS = bool(
    STATE_INPUT_PASS
    and
    MULTISTART_PASS
    and
    ALL_PARAMS_FINITE_PASS
    and
    ALL_PARAMS_POSITIVE_PASS
    and
    NO_BOUNDARY_HIT_PASS
)


FIT_AUDIT[
    "qa"
] = {
    "duplicate_athlete_date_states":
        DUPLICATE_STATE_COUNT,

    "nonpositive_state_gaps":
        NONPOSITIVE_GAP_COUNT,

    "first_date_accounting":
        FIRST_DATE_ACCOUNTING_PASS,

    "performance_count_accounting":
        COUNT_ACCOUNTING_PASS,

    "state_input_pass":
        STATE_INPUT_PASS,

    "successful_optimizer_starts":
        SUCCESS_COUNT,

    "multistart_pass":
        MULTISTART_PASS,

    "parameters_finite":
        ALL_PARAMS_FINITE_PASS,

    "parameters_positive":
        ALL_PARAMS_POSITIVE_PASS,

    "no_parameter_at_boundary":
        NO_BOUNDARY_HIT_PASS,

    "cell_pass":
        CELL_30_PASS,
}


FIT_AUDIT[
    "status"
] = (
    "PASSED"
    if CELL_30_PASS
    else
    "FAILED"
)


atomic_json_write(
    FIT_AUDIT,
    FIT_AUDIT_PATH,
)


# Add audit file itself after writing it.
FIT_AUDIT_SHA256 = sha256_file(
    FIT_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 20. ATOMIC POINTER TO LATEST ATTEMPT
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    V5_STAGE_ROOT
    /
    "LATEST_V5_OU_REFIT.json"
)


LATEST_POINTER = {
    "fold":
        FOLD_NAME,

    "stage":
        "V5_OU_REFIT",

    "attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_30_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            V5_ATTEMPT_DIR
        ),

    "parameter_path":
        str(
            PARAMETER_PATH
        ),

    "fit_audit_path":
        str(
            FIT_AUDIT_PATH
        ),

    "fit_audit_sha256":
        FIT_AUDIT_SHA256,

    "created_utc":
        RUN_UTC.isoformat(),
}


atomic_json_write(
    LATEST_POINTER,
    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST.setdefault(
    "historical_backtest",
    {}
)


TRAINING_MANIFEST[
    "historical_backtest"
].setdefault(
    FOLD_NAME,
    {}
)


TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_ou_refit"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_30_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "attempt_directory":
        str(
            V5_ATTEMPT_DIR
        ),

    "fit_audit":
        str(
            FIT_AUDIT_PATH
        ),

    "fit_audit_sha256":
        FIT_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_V5_OU_REFIT_COMPLETE"
    if CELL_30_PASS
    else
    "HB_2023_V5_OU_REFIT_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. GLOBALS FOR NEXT CELL
# --------------------------------------------------------------------------------------------------

HB2023_V5_STATE_INPUT = daily

HB2023_V5_TAU_DAYS = V5_TAU_DAYS

HB2023_V5_HALF_LIFE_DAYS = (
    V5_HALF_LIFE_DAYS
)

HB2023_V5_STATE_VARIANCE = (
    V5_STATE_VARIANCE
)

HB2023_V5_STATE_SIGMA = (
    V5_STATE_SIGMA
)

HB2023_V5_R_GROUP = (
    V5_R_GROUP
)

HB2023_V5_ATTEMPT_DIR = (
    V5_ATTEMPT_DIR
)

HB2023_V5_PARAMETER_PATH = (
    PARAMETER_PATH
)

HB2023_V5_FIT_AUDIT_PATH = (
    FIT_AUDIT_PATH
)

NOTEBOOK_2_CELL_30_PASS = (
    CELL_30_PASS
)


# --------------------------------------------------------------------------------------------------
# 23. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 155)
print("HB_2023 — HISTORICAL V5 CONTINUOUS-TIME OU REFIT")
print("=" * 155)


print()
print("STATE INPUT")
print("-" * 155)

print(
    f"Performance residual rows       : "
    f"{len(TRAIN):,}"
)

print(
    f"Distinct athlete-date states    : "
    f"{len(daily):,}"
)

print(
    f"Athletes                        : "
    f"{N_STATE_ATHLETES:,}"
)

print(
    f"Multi-performance state dates   : "
    f"{(daily['n_performances'] > 1).sum():,}"
)

print(
    "Multi-group state dates         : "
    f"{((daily[[f'n_{g}' for g in GROUP_LABELS]] > 0).sum(axis=1) > 1).sum():,}"
)


print()
print("OPTIMIZER ROBUSTNESS")
print("-" * 155)

display(
    OPTIMIZER_AUDIT
)


print()
print("BEST V5 OU PARAMETERS")
print("-" * 155)

parameter_table = pd.DataFrame(
    [
        {
            "parameter":
                "tau_days",

            "value":
                V5_TAU_DAYS,
        },

        {
            "parameter":
                "half_life_days",

            "value":
                V5_HALF_LIFE_DAYS,
        },

        {
            "parameter":
                "state_variance",

            "value":
                V5_STATE_VARIANCE,
        },

        {
            "parameter":
                "state_sigma",

            "value":
                V5_STATE_SIGMA,
        },

        *[
            {
                "parameter":
                    f"R_{group}",

                "value":
                    V5_R_GROUP[
                        group
                    ],
            }

            for group in GROUP_LABELS
        ],
    ]
)


display(
    parameter_table
)


print()
print("PERSISTENCE REFERENCE")
print("-" * 155)

display(
    PERSISTENCE_REFERENCE
)


print()
print("FIT STATISTICS")
print("-" * 155)

print(
    f"NLL                             : "
    f"{BEST_NLL:,.6f}"
)

print(
    f"AIC                             : "
    f"{V5_AIC:,.6f}"
)

print(
    f"BIC                             : "
    f"{V5_BIC:,.6f}"
)

print(
    f"Successful starts               : "
    f"{SUCCESS_COUNT}/{N_STARTS}"
)

print(
    f"Numba used                      : "
    f"{NUMBA_AVAILABLE}"
)


print()
print("AUDIT / VERSIONING")
print("-" * 155)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{V5_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                  : "
    f"{FIT_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{FIT_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)

print(
    f"Latest-attempt pointer          : "
    f"{LATEST_POINTER_PATH}"
)


print()
print("QA")
print("-" * 155)

print(
    f"Duplicate state dates           : "
    f"{DUPLICATE_STATE_COUNT}"
)

print(
    f"Nonpositive gaps                : "
    f"{NONPOSITIVE_GAP_COUNT}"
)

print(
    f"First-date accounting           : "
    f"{FIRST_DATE_ACCOUNTING_PASS}"
)

print(
    f"Performance-count accounting    : "
    f"{COUNT_ACCOUNTING_PASS}"
)

print(
    f"State input valid               : "
    f"{STATE_INPUT_PASS}"
)

print(
    f"Multi-start optimizer           : "
    f"{MULTISTART_PASS}"
)

print(
    f"Parameters finite               : "
    f"{ALL_PARAMS_FINITE_PASS}"
)

print(
    f"Parameters positive             : "
    f"{ALL_PARAMS_POSITIVE_PASS}"
)

print(
    f"No parameter at boundary        : "
    f"{NO_BOUNDARY_HIT_PASS}"
)


print()
print("=" * 155)

if CELL_30_PASS:

    print(
        "CELL 30 STATUS: HB_2023 HISTORICAL V5 OU REFIT PASSED"
    )

else:

    print(
        "CELL 30 STATUS: HB_2023 V5 OU REFIT REVIEW REQUIRED"
    )

print("=" * 155)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 21
# Original source SHA256: 105e2497e4a3adb1009714cc5beb62dda386125a15c7ca0c3091f74f113532f7
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 31 — HB_2023 HISTORICAL V5 FILTERED STATE RECOVERY + TERMINAL STATE PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Replay the fitted HB_2023 V5 OU model through TRAIN in exact chronological order.
#
# For every athlete × distinct competition date:
#
#   1. propagate previous filtered state forward by the OU transition
#   2. construct the athlete-date measurement variance
#   3. calculate innovation
#   4. apply Kalman update
#   5. save prior and posterior state
#
#
# Output
# ------
#
#   • one filtered-state row per athlete × date
#   • one terminal state per TRAIN athlete
#   • innovation diagnostics
#
#
# These terminal states are the leakage-safe starting states for:
#
#       2023 VALIDATION
#       then rolling through to
#       2023 TEST
#
#
# IMPORTANT
# ---------
# NO fitting.
# NO parameter changes.
# NO validation/test marks used.
#
#
# AUDIT / VERSIONING
# ------------------
# Creates a NEW immutable state-recovery attempt directory.
#
# Includes:
#   • parent V5 parameter hash
#   • parent V5 state-input hash
#   • filtered state artifact hash
#   • terminal state artifact hash
#   • innovation summary
#   • replay NLL reconciliation
#   • complete audit manifest
#   • atomic latest-state pointer
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_30_PASS",

    "HB2023_V5_STATE_INPUT",

    "HB2023_V5_TAU_DAYS",
    "HB2023_V5_STATE_VARIANCE",
    "HB2023_V5_R_GROUP",

    "HB2023_V5_ATTEMPT_DIR",
    "HB2023_V5_PARAMETER_PATH",
    "HB2023_V5_FIT_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_30_PASS:

    raise RuntimeError(
        "Cell 30 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

CELL_ID = "CELL_31_HB2023_V5_STATE_RECOVERY"

STATE_RECOVERY_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE VERSIONED ATTEMPT
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2023_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


STATE_STAGE_ROOT = (
    HB2023_DIR
    /
    "v5_state_recovery"
)


STATE_ATTEMPT_DIR = (
    STATE_STAGE_ROOT
    /
    ATTEMPT_ID
)


STATE_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. LOAD / VERIFY FROZEN V5 PARAMETER SPEC
# --------------------------------------------------------------------------------------------------

with Path(
    HB2023_V5_PARAMETER_PATH
).open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_SPEC = json.load(
        f
    )


FIT_NLL = float(
    V5_PARAMETER_SPEC[
        "fit_statistics"
    ][
        "negative_log_likelihood"
    ]
)


TAU = float(
    HB2023_V5_TAU_DAYS
)


VZ = float(
    HB2023_V5_STATE_VARIANCE
)


R_GROUP = {
    group:
        float(
            HB2023_V5_R_GROUP[
                group
            ]
        )

    for group in GROUP_LABELS
}


# --------------------------------------------------------------------------------------------------
# 6. STATE INPUT
# --------------------------------------------------------------------------------------------------

S = (
    HB2023_V5_STATE_INPUT.copy()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


S[
    "result_date_dt"
] = pd.to_datetime(
    S[
        "result_date_dt"
    ]
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "state_observation",
    "n_performances",
    "athlete_state_start",
    "state_gap_days",

    *[
        f"n_{group}"
        for group in GROUP_LABELS
    ],
]


missing_columns = [
    column
    for column in required_columns
    if column not in S.columns
]


if missing_columns:

    raise RuntimeError(
        "State input missing columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. MEASUREMENT VARIANCE PER ATHLETE-DATE
# --------------------------------------------------------------------------------------------------

counts = (
    S[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


R_VECTOR = np.array(
    [
        R_GROUP[
            group
        ]
        for group in GROUP_LABELS
    ],
    dtype=float,
)


n_perf = (
    S[
        "n_performances"
    ]
    .to_numpy(
        dtype=float
    )
)


measurement_variance = (
    counts
    @
    R_VECTOR
) / (
    n_perf
    **
    2
)


S[
    "measurement_variance"
] = (
    measurement_variance
)


# --------------------------------------------------------------------------------------------------
# 8. REPLAY KALMAN FILTER
# --------------------------------------------------------------------------------------------------

N = len(
    S
)


prior_mean = np.empty(
    N,
    dtype=float,
)


prior_variance = np.empty(
    N,
    dtype=float,
)


phi_used = np.empty(
    N,
    dtype=float,
)


innovation = np.empty(
    N,
    dtype=float,
)


innovation_variance = np.empty(
    N,
    dtype=float,
)


standardized_innovation = np.empty(
    N,
    dtype=float,
)


kalman_gain = np.empty(
    N,
    dtype=float,
)


filtered_mean = np.empty(
    N,
    dtype=float,
)


filtered_variance = np.empty(
    N,
    dtype=float,
)


log_likelihood_contribution = np.empty(
    N,
    dtype=float,
)


y = (
    S[
        "state_observation"
    ]
    .to_numpy(
        dtype=float
    )
)


gap = (
    S[
        "state_gap_days"
    ]
    .fillna(
        0
    )
    .to_numpy(
        dtype=float
    )
)


state_start = (
    S[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=bool
    )
)


m_post = 0.0
P_post = VZ


for i in range(
    N
):

    # ----------------------------------------------------------------------------------------------
    # Prior
    # ----------------------------------------------------------------------------------------------

    if state_start[
        i
    ]:

        phi = 0.0

        m_prior = 0.0

        P_prior = VZ


    else:

        phi = math.exp(
            -
            gap[
                i
            ]
            /
            TAU
        )


        m_prior = (
            phi
            *
            m_post
        )


        P_prior = (
            phi
            *
            phi
            *
            P_post

            +

            VZ
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


    # ----------------------------------------------------------------------------------------------
    # Measurement update
    # ----------------------------------------------------------------------------------------------

    R_d = measurement_variance[
        i
    ]


    S_var = (
        P_prior
        +
        R_d
    )


    innov = (
        y[
            i
        ]
        -
        m_prior
    )


    z = (
        innov
        /
        np.sqrt(
            S_var
        )
    )


    K = (
        P_prior
        /
        S_var
    )


    m_post = (
        m_prior
        +
        K
        *
        innov
    )


    P_post = (
        (
            1.0
            -
            K
        )
        *
        P_prior
    )


    P_post = max(
        P_post,
        1e-12,
    )


    ll = -0.5 * (
        LOG_2PI
        +
        np.log(
            S_var
        )
        +
        innov
        *
        innov
        /
        S_var
    )


    # ----------------------------------------------------------------------------------------------
    # Store
    # ----------------------------------------------------------------------------------------------

    phi_used[
        i
    ] = phi


    prior_mean[
        i
    ] = m_prior


    prior_variance[
        i
    ] = P_prior


    innovation[
        i
    ] = innov


    innovation_variance[
        i
    ] = S_var


    standardized_innovation[
        i
    ] = z


    kalman_gain[
        i
    ] = K


    filtered_mean[
        i
    ] = m_post


    filtered_variance[
        i
    ] = P_post


    log_likelihood_contribution[
        i
    ] = ll


# --------------------------------------------------------------------------------------------------
# 9. ATTACH FILTER OUTPUT
# --------------------------------------------------------------------------------------------------

S[
    "state_phi"
] = phi_used


S[
    "state_prior_mean"
] = prior_mean


S[
    "state_prior_variance"
] = prior_variance


S[
    "innovation"
] = innovation


S[
    "innovation_variance"
] = (
    innovation_variance
)


S[
    "standardized_innovation"
] = (
    standardized_innovation
)


S[
    "kalman_gain"
] = kalman_gain


S[
    "state_filtered_mean"
] = filtered_mean


S[
    "state_filtered_variance"
] = (
    filtered_variance
)


S[
    "log_likelihood_contribution"
] = (
    log_likelihood_contribution
)


# --------------------------------------------------------------------------------------------------
# 10. REPLAY NLL
# --------------------------------------------------------------------------------------------------

REPLAY_NLL = float(
    -
    S[
        "log_likelihood_contribution"
    ]
    .sum()
)


NLL_ABS_DIFF = float(
    abs(
        REPLAY_NLL
        -
        FIT_NLL
    )
)


NLL_RECONCILIATION_PASS = bool(
    NLL_ABS_DIFF
    <
    1e-5
)


# --------------------------------------------------------------------------------------------------
# 11. ATHLETE HISTORY METADATA
# --------------------------------------------------------------------------------------------------
#
# Keep this inside the terminal-state artifact so future prediction code
# does not have to reconstruct it and so "history_span_days" is always available.
#
# --------------------------------------------------------------------------------------------------

ATHLETE_HISTORY = (
    S.groupby(
        "aaAthleteId",
        as_index=False,
    )
    .agg(
        first_state_date=(
            "result_date_dt",
            "min",
        ),

        last_state_date=(
            "result_date_dt",
            "max",
        ),

        n_state_dates=(
            "result_date_dt",
            "size",
        ),

        n_state_performances=(
            "n_performances",
            "sum",
        ),
    )
)


ATHLETE_HISTORY[
    "history_span_days"
] = (
    ATHLETE_HISTORY[
        "last_state_date"
    ]

    -

    ATHLETE_HISTORY[
        "first_state_date"
    ]
).dt.days


# --------------------------------------------------------------------------------------------------
# 12. TERMINAL FILTERED STATE PER ATHLETE
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    S.groupby(
        "aaAthleteId",
        sort=False,
        as_index=False,
    )
    .tail(1)
    .copy()
)


TERMINAL = TERMINAL[
    [
        "aaAthleteId",
        "result_date_dt",
        "state_filtered_mean",
        "state_filtered_variance",
        "state_observation",
        "measurement_variance",
    ]
].rename(
    columns={
        "result_date_dt":
            "terminal_state_date",

        "state_filtered_mean":
            "terminal_state_mean",

        "state_filtered_variance":
            "terminal_state_variance",

        "state_observation":
            "terminal_state_observation",

        "measurement_variance":
            "terminal_measurement_variance",
    }
)


TERMINAL = TERMINAL.merge(
    ATHLETE_HISTORY,
    on="aaAthleteId",
    how="left",
    validate="one_to_one",
)


TERMINAL[
    "terminal_state_sigma"
] = np.sqrt(
    TERMINAL[
        "terminal_state_variance"
    ]
)


TERMINAL = (
    TERMINAL.sort_values(
        "aaAthleteId"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 13. INNOVATION DIAGNOSTICS
# --------------------------------------------------------------------------------------------------

Z = (
    S[
        "standardized_innovation"
    ]
    .to_numpy(
        dtype=float
    )
)


INNOVATION_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "n_states":
                int(
                    len(
                        S
                    )
                ),

            "athletes":
                int(
                    S[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "mean_standardized_innovation":
                float(
                    np.mean(
                        Z
                    )
                ),

            "sd_standardized_innovation":
                float(
                    np.std(
                        Z,
                        ddof=1,
                    )
                ),

            "median_standardized_innovation":
                float(
                    np.median(
                        Z
                    )
                ),

            "pct_abs_z_gt_2":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(
                            Z
                        )
                        >
                        2.0
                    )
                ),

            "pct_abs_z_gt_3":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(
                            Z
                        )
                        >
                        3.0
                    )
                ),

            "mean_kalman_gain":
                float(
                    S[
                        "kalman_gain"
                    ]
                    .mean()
                ),

            "mean_filtered_variance":
                float(
                    S[
                        "state_filtered_variance"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. QA
# --------------------------------------------------------------------------------------------------

ROW_ACCOUNTING_PASS = bool(
    len(
        S
    )
    ==
    len(
        HB2023_V5_STATE_INPUT
    )
)


DUPLICATE_STATE_PASS = bool(
    not S.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .any()
)


FINITE_FILTER_PASS = bool(
    np.isfinite(
        S[
            [
                "measurement_variance",
                "state_prior_mean",
                "state_prior_variance",
                "innovation",
                "innovation_variance",
                "standardized_innovation",
                "kalman_gain",
                "state_filtered_mean",
                "state_filtered_variance",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        S[
            "measurement_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        S[
            "state_prior_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        S[
            "innovation_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        S[
            "state_filtered_variance"
        ]
        >
        0
    )
    .all()
)


GAIN_BOUNDS_PASS = bool(
    S[
        "kalman_gain"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


FIRST_STATE_PRIOR_PASS = bool(

    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_mean",
        ]
        .to_numpy(
            dtype=float
        ),

        0.0,
        atol=1e-12,
    )

    and

    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_variance",
        ]
        .to_numpy(
            dtype=float
        ),

        VZ,
        atol=1e-10,
        rtol=1e-10,
    )
)


TERMINAL_ACCOUNTING_PASS = bool(
    len(
        TERMINAL
    )
    ==
    S[
        "aaAthleteId"
    ]
    .nunique()

    and

    TERMINAL[
        "aaAthleteId"
    ]
    .is_unique
)


TERMINAL_DATE_PASS = bool(
    (
        TERMINAL[
            "terminal_state_date"
        ]

        ==

        TERMINAL[
            "last_state_date"
        ]
    )
    .all()
)


HISTORY_METADATA_PASS = bool(
    (
        TERMINAL[
            "n_state_dates"
        ]
        >=
        1
    )
    .all()

    and

    (
        TERMINAL[
            "history_span_days"
        ]
        >=
        0
    )
    .all()

    and

    (
        TERMINAL[
            "n_state_performances"
        ]
        >=
        TERMINAL[
            "n_state_dates"
        ]
    )
    .all()
)


CELL_31_PASS = bool(
    ROW_ACCOUNTING_PASS
    and
    DUPLICATE_STATE_PASS
    and
    FINITE_FILTER_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    GAIN_BOUNDS_PASS
    and
    FIRST_STATE_PRIOR_PASS
    and
    TERMINAL_ACCOUNTING_PASS
    and
    TERMINAL_DATE_PASS
    and
    HISTORY_METADATA_PASS
    and
    NLL_RECONCILIATION_PASS
)


# --------------------------------------------------------------------------------------------------
# 15. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

FILTERED_STATE_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_train_filtered_states.parquet"
)


TERMINAL_STATE_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_train_terminal_states.parquet"
)


INNOVATION_SUMMARY_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_innovation_summary.parquet"
)


STATE_RECOVERY_AUDIT_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_state_recovery_audit.json"
)


S.to_parquet(
    FILTERED_STATE_PATH,
    index=False,
)


TERMINAL.to_parquet(
    TERMINAL_STATE_PATH,
    index=False,
)


INNOVATION_SUMMARY.to_parquet(
    INNOVATION_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 16. PARENT ARTIFACT PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2023_V5_PARAMETER_PATH
        ),

    "v5_fit_audit":
        Path(
            HB2023_V5_FIT_AUDIT_PATH
        ),

    "v5_state_input":
        (
            Path(
                HB2023_V5_ATTEMPT_DIR
            )
            /
            "v5_state_input.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


# --------------------------------------------------------------------------------------------------
# 17. OUTPUT ARTIFACT PROVENANCE
# --------------------------------------------------------------------------------------------------

output_paths = {
    "filtered_states":
        FILTERED_STATE_PATH,

    "terminal_states":
        TERMINAL_STATE_PATH,

    "innovation_summary":
        INNOVATION_SUMMARY_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 18. COMPLETE AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

STATE_RECOVERY_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "model_version":
        str(
            TRAINING_MANIFEST.get(
                "model_version",
                "UNKNOWN",
            )
        ),

    "historical_fold":
        FOLD_NAME,

    "stage":
        "V5_STATE_RECOVERY",

    "stage_version":
        STATE_RECOVERY_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_31_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Recover leakage-safe TRAIN filtered and terminal OU states "
            "for the HB_2023 historical temporal backtest."
        ),

    "parent_v5_refit_attempt":
        str(
            HB2023_V5_ATTEMPT_DIR
        ),

    "parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "replay": {
        "fit_nll":
            FIT_NLL,

        "replay_nll":
            REPLAY_NLL,

        "absolute_nll_difference":
            NLL_ABS_DIFF,

        "nll_reconciliation_pass":
            NLL_RECONCILIATION_PASS,
    },

    "state_population": {
        "rows":
            int(
                len(
                    S
                )
            ),

        "athletes":
            int(
                S[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "terminal_states":
            int(
                len(
                    TERMINAL
                )
            ),
    },

    "terminal_state_schema": [
        "aaAthleteId",
        "terminal_state_date",
        "terminal_state_mean",
        "terminal_state_variance",
        "terminal_state_sigma",
        "first_state_date",
        "last_state_date",
        "n_state_dates",
        "n_state_performances",
        "history_span_days",
    ],

    "qa": {
        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "duplicate_state_dates":
            DUPLICATE_STATE_PASS,

        "finite_filter":
            FINITE_FILTER_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "kalman_gain_bounds":
            GAIN_BOUNDS_PASS,

        "first_state_prior":
            FIRST_STATE_PRIOR_PASS,

        "terminal_accounting":
            TERMINAL_ACCOUNTING_PASS,

        "terminal_dates":
            TERMINAL_DATE_PASS,

        "history_metadata":
            HISTORY_METADATA_PASS,

        "nll_reconciliation":
            NLL_RECONCILIATION_PASS,

        "cell_pass":
            CELL_31_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    STATE_RECOVERY_AUDIT,
    STATE_RECOVERY_AUDIT_PATH,
)


STATE_RECOVERY_AUDIT_SHA256 = (
    sha256_file(
        STATE_RECOVERY_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 19. ATOMIC LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_STATE_POINTER_PATH = (
    STATE_STAGE_ROOT
    /
    "LATEST_V5_STATE_RECOVERY.json"
)


LATEST_STATE_POINTER = {
    "fold":
        FOLD_NAME,

    "stage":
        "V5_STATE_RECOVERY",

    "attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_31_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            STATE_ATTEMPT_DIR
        ),

    "filtered_states":
        str(
            FILTERED_STATE_PATH
        ),

    "terminal_states":
        str(
            TERMINAL_STATE_PATH
        ),

    "innovation_summary":
        str(
            INNOVATION_SUMMARY_PATH
        ),

    "audit_manifest":
        str(
            STATE_RECOVERY_AUDIT_PATH
        ),

    "audit_sha256":
        STATE_RECOVERY_AUDIT_SHA256,

    "created_utc":
        RUN_UTC.isoformat(),
}


atomic_json_write(
    LATEST_STATE_POINTER,
    LATEST_STATE_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_state_recovery"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_31_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_STATE_POINTER_PATH
        ),

    "attempt_directory":
        str(
            STATE_ATTEMPT_DIR
        ),

    "terminal_states":
        str(
            TERMINAL_STATE_PATH
        ),

    "audit_manifest":
        str(
            STATE_RECOVERY_AUDIT_PATH
        ),

    "audit_sha256":
        STATE_RECOVERY_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_V5_STATE_RECOVERY_COMPLETE"
    if CELL_31_PASS
    else
    "HB_2023_V5_STATE_RECOVERY_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_V5_FILTERED_STATES = S

HB2023_V5_TERMINAL_STATES = TERMINAL

HB2023_V5_INNOVATION_SUMMARY = (
    INNOVATION_SUMMARY
)

HB2023_V5_STATE_RECOVERY_ATTEMPT_DIR = (
    STATE_ATTEMPT_DIR
)

HB2023_V5_TERMINAL_STATE_PATH = (
    TERMINAL_STATE_PATH
)

HB2023_V5_STATE_RECOVERY_AUDIT_PATH = (
    STATE_RECOVERY_AUDIT_PATH
)

NOTEBOOK_2_CELL_31_PASS = (
    CELL_31_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 155)
print("HB_2023 — HISTORICAL V5 FILTERED STATE RECOVERY")
print("=" * 155)


print()
print("STATE REPLAY")
print("-" * 155)

print(
    f"Filtered athlete-date states    : "
    f"{len(S):,}"
)

print(
    f"Athletes                        : "
    f"{S['aaAthleteId'].nunique():,}"
)

print(
    f"Terminal athlete states         : "
    f"{len(TERMINAL):,}"
)


print()
print("NLL RECONCILIATION")
print("-" * 155)

print(
    f"Fit NLL                         : "
    f"{FIT_NLL:,.9f}"
)

print(
    f"Replay NLL                      : "
    f"{REPLAY_NLL:,.9f}"
)

print(
    f"Absolute difference             : "
    f"{NLL_ABS_DIFF:.12e}"
)

print(
    f"Reconciliation pass             : "
    f"{NLL_RECONCILIATION_PASS}"
)


print()
print("INNOVATION DIAGNOSTICS")
print("-" * 155)

display(
    INNOVATION_SUMMARY
)


print()
print("TERMINAL STATE HISTORY SUMMARY")
print("-" * 155)

terminal_history_summary = pd.DataFrame(
    [
        {
            "athletes":
                int(
                    len(
                        TERMINAL
                    )
                ),

            "median_state_dates":
                float(
                    TERMINAL[
                        "n_state_dates"
                    ]
                    .median()
                ),

            "median_history_span_days":
                float(
                    TERMINAL[
                        "history_span_days"
                    ]
                    .median()
                ),

            "max_history_span_days":
                int(
                    TERMINAL[
                        "history_span_days"
                    ]
                    .max()
                ),

            "median_terminal_state_sigma":
                float(
                    TERMINAL[
                        "terminal_state_sigma"
                    ]
                    .median()
                ),
        }
    ]
)


display(
    terminal_history_summary
)


print()
print("AUDIT / VERSIONING")
print("-" * 155)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{STATE_ATTEMPT_DIR}"
)

print(
    f"Terminal-state artifact         : "
    f"{TERMINAL_STATE_PATH}"
)

print(
    f"Audit manifest                  : "
    f"{STATE_RECOVERY_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{STATE_RECOVERY_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 155)

print(
    f"Row accounting                  : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Unique athlete-date states      : "
    f"{DUPLICATE_STATE_PASS}"
)

print(
    f"Finite filter quantities        : "
    f"{FINITE_FILTER_PASS}"
)

print(
    f"Positive variances              : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Kalman gains in [0,1]           : "
    f"{GAIN_BOUNDS_PASS}"
)

print(
    f"First-state priors correct      : "
    f"{FIRST_STATE_PRIOR_PASS}"
)

print(
    f"One terminal state / athlete    : "
    f"{TERMINAL_ACCOUNTING_PASS}"
)

print(
    f"Terminal dates correct          : "
    f"{TERMINAL_DATE_PASS}"
)

print(
    f"History metadata valid          : "
    f"{HISTORY_METADATA_PASS}"
)


print()
print("=" * 155)

if CELL_31_PASS:

    print(
        "CELL 31 STATUS: HB_2023 HISTORICAL V5 STATE RECOVERY PASSED"
    )

else:

    print(
        "CELL 31 STATUS: HB_2023 V5 STATE RECOVERY REVIEW REQUIRED"
    )

print("=" * 155)



HB_2023 — HISTORICAL V4 REFIT PREPARATION

TEMPORAL SPLIT
------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN       : <= 2022-12-31
VALIDATION  : 2023-01-01 -> 2023-04-30
TEST        : 2023-05-01 -> 2023-08-30

POPULATION
------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN rows / athletes          : 110,977 / 7,442
VALIDATION established         : 3,092 / 1,676
TEST established               : 10,110 / 3,539

TRAIN-ONLY SPLINE
------------------------------------------------------------------------------------------------------------------------------------------------------
Basis shape                    : (110977, 7)
Internal knots                : [18.3850455519 20.3043183638 22.2619218738 25.3256398146]
Column means                  : [0.0684499177 0.1510330762 0.2898914833 

,split,n,median_gap_days,mean_gap_days,min_gap_days,max_gap_days
0,VALIDATION,3092,120.5,165.055627,1,2656
1,TEST,10110,25.0,118.346291,1,4248



QA
------------------------------------------------------------------------------------------------------------------------------------------------------
Temporal split                 : True
TRAIN-only athlete map         : True
Finite V4 features             : True
7-column spline                : True
Finite spline values           : True
TRAIN-centered spline          : True
VALIDATION chronology          : True
TEST chronology                : True
No future TRAIN data           : True

CELL 28 STATUS: HB_2023 HISTORICAL V4 PREPARATION PASSED


Output()

Finished [100%]: Average Loss = 3.0202e+05


...


FIT STATISTICS
-----------------------------------------------------------------------------------------------------------------------------------------------------------
NLL                             : 275,601.114782
AIC                             : 551,214.229565
BIC                             : 551,271.900594
Successful starts               : 10/10
Numba used                      : True

AUDIT / VERSIONING
-----------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T132200_053751Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v5_ou_refit/20260922T132200_053751Z
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/

,fold,n_states,athletes,mean_standardized_innovation,sd_standardized_innovation,median_standardized_innovation,pct_abs_z_gt_2,pct_abs_z_gt_3,mean_kalman_gain,mean_filtered_variance
0,HB_2023,110397,7442,-0.032213,0.999485,0.001798,4.321675,0.734621,0.271012,1.680863



TERMINAL STATE HISTORY SUMMARY
-----------------------------------------------------------------------------------------------------------------------------------------------------------


,athletes,median_state_dates,median_history_span_days,max_history_span_days,median_terminal_state_sigma
0,7442,7.0,854.0,7028,1.349987



AUDIT / VERSIONING
-----------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T132203_663300Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v5_state_recovery/20260922T132203_663300Z
Terminal-state artifact         : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v5_state_recovery/20260922T132203_663300Z/v5_train_terminal_states.parquet
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v5_state_recovery/20260922T132203_663300Z/v5_state_recovery_audit.json
Audit SHA256                    : 96ece62515128067c926d70fbfbb020c7c

## Stage 12 — HB_2023 replay, calibration and evidence freeze

Replay rolling forecasts, calibrate V6 on historical VALIDATION, compare V5/V6/V6.1 on untouched TEST, and freeze evidence.

**Legacy source cells represented:** 22, 23, 24, 25  
**Execution rule:** run only after the preceding stage has completed successfully.


In [13]:
# ==================================================================================================
# REFACTORED STAGE 12 — HB_2023 replay, calibration and evidence freeze
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 22
# Original source SHA256: 358e2a30649620c40f2ba7799e3dcaab5044972ef89f24d9b2e678da1002469b
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 32 — HB_2023 ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY
# ==================================================================================================
#
# Purpose
# -------
# Generate genuine rolling-origin historical forecasts for HB_2023.
#
#
# Starting point:
#
#       TRAIN terminal OU state as of <= 2022-12-31
#
#
# Then chronologically:
#
#       predict athlete-date
#       score prediction
#       observe same-day performance residual(s)
#       update OU state ONCE for that athlete-date
#       carry updated state forward
#
#
# VALIDATION:
#
#       2023-01-01 -> 2023-04-30
#
# TEST:
#
#       2023-05-01 -> 2023-08-30
#
#
# IMPORTANT
# ---------
# • V4 parameters are frozen from historical TRAIN.
# • V5 parameters are frozen from historical TRAIN.
# • VALIDATION results may update state before later dates.
# • TEST results may update state before later TEST dates.
# • Same-day rows cannot see each other before prediction.
# • One state update per athlete × distinct date.
# • No V6/V6.1 fitting occurs in this cell.
#
#
# AUDIT / VERSIONING
# ------------------
# Every execution creates a new immutable rolling-forecast attempt.
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_31_PASS",

    "HB2023_VALIDATION",
    "HB2023_TEST",

    "HB2023_V4_POSTERIOR_MEANS",
    "HB2023_V5_TERMINAL_STATES",

    "HB2023_V5_TAU_DAYS",
    "HB2023_V5_STATE_VARIANCE",
    "HB2023_V5_R_GROUP",

    "HB2023_V5_PARAMETER_PATH",
    "HB2023_V5_STATE_RECOVERY_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_31_PASS:

    raise RuntimeError(
        "Cell 31 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

CELL_ID = "CELL_32_HB2023_ROLLING_V4_V5"

STAGE_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(7)
]


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2023_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


ROLLING_STAGE_ROOT = (
    HB2023_DIR
    /
    "rolling_v4_v5"
)


ROLLING_ATTEMPT_DIR = (
    ROLLING_STAGE_ROOT
    /
    ATTEMPT_ID
)


ROLLING_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. FREEZE HISTORICAL V4 PARAMETERS
# --------------------------------------------------------------------------------------------------

alpha_hat = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ],
    dtype=float,
)


beta_hat = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "beta_spline"
    ],
    dtype=float,
)


youth_hat = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ],
    dtype=float,
)


athlete_intercept_hat = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ],
    dtype=float,
)


athlete_slope_hat = np.asarray(
    HB2023_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 6. FREEZE HISTORICAL V5 PARAMETERS
# --------------------------------------------------------------------------------------------------

TAU = float(
    HB2023_V5_TAU_DAYS
)


VZ = float(
    HB2023_V5_STATE_VARIANCE
)


R_GROUP = {
    group:
        float(
            HB2023_V5_R_GROUP[
                group
            ]
        )

    for group in GROUP_LABELS
}


# --------------------------------------------------------------------------------------------------
# 7. PREPARE VALIDATION + TEST
# --------------------------------------------------------------------------------------------------

VALIDATION = (
    HB2023_VALIDATION.copy()
    .reset_index(
        drop=True
    )
)


TEST = (
    HB2023_TEST.copy()
    .reset_index(
        drop=True
    )
)


VALIDATION[
    "evaluation_split"
] = "VALIDATION"


TEST[
    "evaluation_split"
] = "TEST"


VALIDATION[
    "split_row_id"
] = np.arange(
    len(
        VALIDATION
    ),
    dtype=int,
)


TEST[
    "split_row_id"
] = np.arange(
    len(
        TEST
    ),
    dtype=int,
)


PRED = pd.concat(
    [
        VALIDATION,
        TEST,
    ],
    ignore_index=True,
)


PRED[
    "result_date_dt"
] = pd.to_datetime(
    PRED[
        "result_date_dt"
    ]
)


PRED[
    "_forecast_row_id"
] = np.arange(
    len(
        PRED
    ),
    dtype=int,
)


# --------------------------------------------------------------------------------------------------
# 8. REQUIRED COLUMN QA
# --------------------------------------------------------------------------------------------------

required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",

    "sex_idx",
    "group_idx",
    "athlete_idx",

    "is_youth_implement",
    "age_scaled",

    "model_group",

    *SPLINE_COLUMNS,
]


missing_columns = [
    column
    for column in required_columns
    if column not in PRED.columns
]


if missing_columns:

    raise RuntimeError(
        "Prediction data missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. HISTORICAL V4 STRUCTURAL PREDICTION
# --------------------------------------------------------------------------------------------------

sex_idx = (
    PRED[
        "sex_idx"
    ]
    .to_numpy(
        dtype=int
    )
)


athlete_idx = (
    PRED[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=int
    )
)


youth = (
    PRED[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=float
    )
)


age_scaled = (
    PRED[
        "age_scaled"
    ]
    .to_numpy(
        dtype=float
    )
)


B = (
    PRED[
        SPLINE_COLUMNS
    ]
    .to_numpy(
        dtype=float
    )
)


PRED[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx
    ]

    +

    np.sum(
        B
        *
        beta_hat[
            sex_idx
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx
    ]
    *
    youth
)


PRED[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx
    ]
)


PRED[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx
    ]

    *

    age_scaled
)


PRED[
    "v4_structural_mean"
] = (
    PRED[
        "v4_population_component"
    ]

    +

    PRED[
        "v4_athlete_intercept_component"
    ]

    +

    PRED[
        "v4_athlete_slope_component"
    ]
)


PRED[
    "v4_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v4_structural_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. TERMINAL TRAIN STATE LOOKUP
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    HB2023_V5_TERMINAL_STATES.copy()
)


TERMINAL[
    "terminal_state_date"
] = pd.to_datetime(
    TERMINAL[
        "terminal_state_date"
    ]
)


terminal_lookup = (
    TERMINAL.set_index(
        "aaAthleteId"
    )
)


prediction_athletes = set(
    PRED[
        "aaAthleteId"
    ]
    .unique()
)


terminal_athletes = set(
    TERMINAL[
        "aaAthleteId"
    ]
    .unique()
)


MISSING_TERMINAL_ATHLETES = (
    prediction_athletes
    -
    terminal_athletes
)


if MISSING_TERMINAL_ATHLETES:

    raise RuntimeError(
        "Established prediction athletes missing TRAIN terminal states:\n"
        +
        "\n".join(
            str(x)
            for x in sorted(
                MISSING_TERMINAL_ATHLETES
            )[:20]
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. ALLOCATE ROLLING STATE OUTPUT
# --------------------------------------------------------------------------------------------------

N = len(
    PRED
)


state_prior_mean = np.full(
    N,
    np.nan,
    dtype=float,
)


state_prior_variance = np.full(
    N,
    np.nan,
    dtype=float,
)


state_gap_days_dynamic = np.full(
    N,
    np.nan,
    dtype=float,
)


state_phi = np.full(
    N,
    np.nan,
    dtype=float,
)


v5_prediction_mean = np.full(
    N,
    np.nan,
    dtype=float,
)


# Date-level update audit.
date_state_rows = []


# --------------------------------------------------------------------------------------------------
# 12. ROLLING-ORIGIN FILTER
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# Predictions are written BEFORE the current date's result is used to update state.
#
# All performances on the same athlete-date share one prior state.
#
# --------------------------------------------------------------------------------------------------

for athlete_id, athlete_rows in PRED.groupby(
    "aaAthleteId",
    sort=False,
):

    terminal_row = (
        terminal_lookup.loc[
            athlete_id
        ]
    )


    current_mean = float(
        terminal_row[
            "terminal_state_mean"
        ]
    )


    current_variance = float(
        terminal_row[
            "terminal_state_variance"
        ]
    )


    current_date = pd.Timestamp(
        terminal_row[
            "terminal_state_date"
        ]
    )


    athlete_rows = (
        athlete_rows.sort_values(
            [
                "result_date_dt",
                "_forecast_row_id",
            ]
        )
    )


    for forecast_date, date_rows in athlete_rows.groupby(
        "result_date_dt",
        sort=True,
    ):

        forecast_date = pd.Timestamp(
            forecast_date
        )


        delta_days = int(
            (
                forecast_date
                -
                current_date
            ).days
        )


        if delta_days <= 0:

            raise RuntimeError(
                f"Non-positive rolling state gap "
                f"for athlete {athlete_id}: "
                f"{current_date} -> {forecast_date}"
            )


        phi = float(
            np.exp(
                -
                delta_days
                /
                TAU
            )
        )


        prior_mean = float(
            phi
            *
            current_mean
        )


        prior_variance = float(
            phi
            *
            phi
            *
            current_variance

            +

            VZ
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


        row_ids = (
            date_rows[
                "_forecast_row_id"
            ]
            .to_numpy(
                dtype=int
            )
        )


        state_prior_mean[
            row_ids
        ] = (
            prior_mean
        )


        state_prior_variance[
            row_ids
        ] = (
            prior_variance
        )


        state_gap_days_dynamic[
            row_ids
        ] = (
            delta_days
        )


        state_phi[
            row_ids
        ] = (
            phi
        )


        v5_prediction_mean[
            row_ids
        ] = (
            PRED.loc[
                row_ids,
                "v4_structural_mean",
            ]
            .to_numpy(
                dtype=float
            )

            +

            prior_mean
        )


        # ------------------------------------------------------------------------------------------
        # AFTER prediction: observe same-day residual(s) and update state.
        # ------------------------------------------------------------------------------------------

        residuals = (
            PRED.loc[
                row_ids,
                "v4_error",
            ]
            .to_numpy(
                dtype=float
            )
        )


        state_observation = float(
            np.mean(
                residuals
            )
        )


        n_date = len(
            row_ids
        )


        group_counts = np.array(
            [
                int(
                    (
                        PRED.loc[
                            row_ids,
                            "model_group",
                        ]
                        ==
                        group
                    )
                    .sum()
                )

                for group in GROUP_LABELS
            ],
            dtype=float,
        )


        R_d = float(
            np.dot(
                group_counts,
                np.array(
                    [
                        R_GROUP[
                            group
                        ]
                        for group in GROUP_LABELS
                    ],
                    dtype=float,
                ),
            )

            /

            (
                n_date
                **
                2
            )
        )


        innovation = float(
            state_observation
            -
            prior_mean
        )


        innovation_variance = float(
            prior_variance
            +
            R_d
        )


        kalman_gain = float(
            prior_variance
            /
            innovation_variance
        )


        posterior_mean = float(
            prior_mean
            +
            kalman_gain
            *
            innovation
        )


        posterior_variance = float(
            (
                1.0
                -
                kalman_gain
            )
            *
            prior_variance
        )


        posterior_variance = max(
            posterior_variance,
            1e-12,
        )


        date_state_rows.append(
            {
                "aaAthleteId":
                    athlete_id,

                "result_date_dt":
                    forecast_date,

                "evaluation_split":
                    date_rows[
                        "evaluation_split"
                    ]
                    .iloc[0],

                "state_gap_days":
                    delta_days,

                "state_phi":
                    phi,

                "state_prior_mean":
                    prior_mean,

                "state_prior_variance":
                    prior_variance,

                "state_observation":
                    state_observation,

                "n_performances":
                    int(
                        n_date
                    ),

                "measurement_variance":
                    R_d,

                "innovation":
                    innovation,

                "innovation_variance":
                    innovation_variance,

                "standardized_innovation":
                    innovation
                    /
                    np.sqrt(
                        innovation_variance
                    ),

                "kalman_gain":
                    kalman_gain,

                "state_filtered_mean":
                    posterior_mean,

                "state_filtered_variance":
                    posterior_variance,

                **{
                    f"n_{group}":
                        int(
                            group_counts[
                                i
                            ]
                        )

                    for i, group
                    in enumerate(
                        GROUP_LABELS
                    )
                },
            }
        )


        # State update becomes available only AFTER this date.
        current_mean = posterior_mean
        current_variance = posterior_variance
        current_date = forecast_date


# --------------------------------------------------------------------------------------------------
# 13. ATTACH ROLLING PREDICTIONS
# --------------------------------------------------------------------------------------------------

PRED[
    "state_prior_mean"
] = (
    state_prior_mean
)


PRED[
    "state_prior_variance"
] = (
    state_prior_variance
)


PRED[
    "state_prior_sigma"
] = np.sqrt(
    state_prior_variance
)


PRED[
    "rolling_state_gap_days"
] = (
    state_gap_days_dynamic
)


PRED[
    "state_phi"
] = (
    state_phi
)


PRED[
    "v5_prediction_mean"
] = (
    v5_prediction_mean
)


PRED[
    "v5_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v5_prediction_mean"
    ]
)


DATE_STATES = pd.DataFrame(
    date_state_rows
)


# --------------------------------------------------------------------------------------------------
# 14. CHRONOLOGY RECONCILIATION
# --------------------------------------------------------------------------------------------------
#
# Cell 28 independently constructed previous-distinct-date gaps.
#
# The dynamic rolling replay should exactly reproduce those gaps.
#
# --------------------------------------------------------------------------------------------------

if "state_gap_days" not in PRED.columns:

    raise RuntimeError(
        "Prepared historical fold lacks reference state_gap_days."
    )


GAP_RECONCILIATION_MAX_DIFF = float(
    np.max(
        np.abs(
            PRED[
                "rolling_state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )

            -

            PRED[
                "state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
)


GAP_RECONCILIATION_PASS = bool(
    GAP_RECONCILIATION_MAX_DIFF
    <
    1e-12
)


# --------------------------------------------------------------------------------------------------
# 15. SPLIT BACK TO VALIDATION / TEST
# --------------------------------------------------------------------------------------------------

ROLLING_VALIDATION = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "VALIDATION"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


ROLLING_TEST = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "TEST"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 16. POINT-METRIC HELPER
# --------------------------------------------------------------------------------------------------

def point_metrics(
    df,
):

    rows = []


    for model in [
        "V4",
        "V5",
    ]:

        error = (
            df[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        rows.append(
            {
                "model":
                    model,

                "n":
                    int(
                        len(
                            df
                        )
                    ),

                "athletes":
                    int(
                        df[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error
                                **
                                2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


VALIDATION_METRICS = (
    point_metrics(
        ROLLING_VALIDATION
    )
)


VALIDATION_METRICS[
    "split"
] = "VALIDATION"


TEST_METRICS = (
    point_metrics(
        ROLLING_TEST
    )
)


TEST_METRICS[
    "split"
] = "TEST"


POINT_METRICS = pd.concat(
    [
        VALIDATION_METRICS,
        TEST_METRICS,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 17. STATE-GAP METRICS
# --------------------------------------------------------------------------------------------------

GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


PRED[
    "state_gap_band"
] = pd.cut(
    PRED[
        "rolling_state_gap_days"
    ],

    bins=GAP_BINS,

    labels=GAP_LABELS,

    include_lowest=True,
)


gap_metric_rows = []


for (
    split,
    gap_band
), g in PRED.groupby(
    [
        "evaluation_split",
        "state_gap_band",
    ],
    observed=True,
):

    for model in [
        "V4",
        "V5",
    ]:

        error = (
            g[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        gap_metric_rows.append(
            {
                "split":
                    split,

                "state_gap_band":
                    str(
                        gap_band
                    ),

                "model":
                    model,

                "n":
                    int(
                        len(
                            g
                        )
                    ),

                "athletes":
                    int(
                        g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "mean_gap_days":
                    float(
                        g[
                            "rolling_state_gap_days"
                        ]
                        .mean()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error
                                **
                                2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )


GAP_METRICS = pd.DataFrame(
    gap_metric_rows
)


# --------------------------------------------------------------------------------------------------
# 18. V5 GAIN TABLE
# --------------------------------------------------------------------------------------------------

GAIN_ROWS = []


for split, g in PRED.groupby(
    "evaluation_split",
    sort=False,
):

    v4_error = (
        g[
            "v4_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v5_error = (
        g[
            "v5_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v4_mae = float(
        np.mean(
            np.abs(
                v4_error
            )
        )
    )


    v5_mae = float(
        np.mean(
            np.abs(
                v5_error
            )
        )
    )


    v4_rmse = float(
        np.sqrt(
            np.mean(
                v4_error
                **
                2
            )
        )
    )


    v5_rmse = float(
        np.sqrt(
            np.mean(
                v5_error
                **
                2
            )
        )
    )


    GAIN_ROWS.append(
        {
            "split":
                split,

            "n":
                int(
                    len(
                        g
                    )
                ),

            "athletes":
                int(
                    g[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4_mae,

            "V5_MAE":
                v5_mae,

            "V4_to_V5_MAE_gain_m":
                v4_mae
                -
                v5_mae,

            "V4_RMSE":
                v4_rmse,

            "V5_RMSE":
                v5_rmse,

            "V4_to_V5_RMSE_gain_m":
                v4_rmse
                -
                v5_rmse,

            "V5_Bias":
                float(
                    np.mean(
                        v5_error
                    )
                ),
        }
    )


V5_GAIN_SUMMARY = pd.DataFrame(
    GAIN_ROWS
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

FINITE_PREDICTION_PASS = bool(
    np.isfinite(
        PRED[
            [
                "v4_structural_mean",
                "v5_prediction_mean",

                "v4_error",
                "v5_error",

                "state_prior_mean",
                "state_prior_variance",
                "state_prior_sigma",

                "rolling_state_gap_days",
                "state_phi",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        PRED[
            "state_prior_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        DATE_STATES[
            "state_filtered_variance"
        ]
        >
        0
    )
    .all()
)


NO_SAME_DAY_LEAKAGE_PASS = bool(
    DATE_STATES.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
    ==
    0
)


ROW_ACCOUNTING_PASS = bool(
    len(
        ROLLING_VALIDATION
    )
    ==
    len(
        HB2023_VALIDATION
    )

    and

    len(
        ROLLING_TEST
    )
    ==
    len(
        HB2023_TEST
    )
)


SPLIT_CHRONOLOGY_PASS = bool(
    ROLLING_VALIDATION[
        "result_date_dt"
    ]
    .max()

    <

    ROLLING_TEST[
        "result_date_dt"
    ]
    .min()
)


CELL_32_PASS = bool(
    FINITE_PREDICTION_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    NO_SAME_DAY_LEAKAGE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    SPLIT_CHRONOLOGY_PASS
    and
    GAP_RECONCILIATION_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

VALIDATION_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "validation_rolling_predictions.parquet"
)


TEST_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "test_rolling_predictions.parquet"
)


DATE_STATE_UPDATES_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "rolling_date_state_updates.parquet"
)


POINT_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "point_metrics.parquet"
)


GAP_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "gap_metrics.parquet"
)


GAIN_SUMMARY_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "v4_to_v5_gain_summary.parquet"
)


ROLLING_AUDIT_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "rolling_forecast_audit.json"
)


ROLLING_VALIDATION.to_parquet(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)


ROLLING_TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)


DATE_STATES.to_parquet(
    DATE_STATE_UPDATES_PATH,
    index=False,
)


POINT_METRICS.to_parquet(
    POINT_METRICS_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


V5_GAIN_SUMMARY.to_parquet(
    GAIN_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. AUDIT PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2023_V5_PARAMETER_PATH
        ),

    "v5_state_recovery_audit":
        Path(
            HB2023_V5_STATE_RECOVERY_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


output_paths = {
    "validation_predictions":
        VALIDATION_PREDICTIONS_PATH,

    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "date_state_updates":
        DATE_STATE_UPDATES_PATH,

    "point_metrics":
        POINT_METRICS_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "v5_gain_summary":
        GAIN_SUMMARY_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


ROLLING_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "ROLLING_V4_V5_FORECAST",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_32_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Leakage-safe rolling historical V4/V5 forecasts "
            "for the HB_2023 V6.1 temporal backtest."
        ),

    "temporal_windows": {
        "validation":
            "2023-01-01 through 2023-04-30",

        "test":
            "2023-05-01 through 2023-08-30",
    },

    "state_update_rule":
        (
            "Predict all rows on an athlete-date using the pre-date state; "
            "then update state once from the mean same-day V4 residual."
        ),

    "v5_parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "population": {
        "validation_rows":
            int(
                len(
                    ROLLING_VALIDATION
                )
            ),

        "validation_athletes":
            int(
                ROLLING_VALIDATION[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "test_rows":
            int(
                len(
                    ROLLING_TEST
                )
            ),

        "test_athletes":
            int(
                ROLLING_TEST[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "rolling_athlete_dates":
            int(
                len(
                    DATE_STATES
                )
            ),
    },

    "qa": {
        "finite_predictions":
            FINITE_PREDICTION_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "same_day_leakage":
            NO_SAME_DAY_LEAKAGE_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "split_chronology":
            SPLIT_CHRONOLOGY_PASS,

        "gap_reconciliation":
            GAP_RECONCILIATION_PASS,

        "gap_reconciliation_max_diff":
            GAP_RECONCILIATION_MAX_DIFF,

        "cell_pass":
            CELL_32_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    ROLLING_AUDIT,
    ROLLING_AUDIT_PATH,
)


ROLLING_AUDIT_SHA256 = (
    sha256_file(
        ROLLING_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 22. ATOMIC LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_ROLLING_POINTER_PATH = (
    ROLLING_STAGE_ROOT
    /
    "LATEST_ROLLING_V4_V5.json"
)


LATEST_ROLLING_POINTER = {
    "fold":
        FOLD_NAME,

    "stage":
        "ROLLING_V4_V5_FORECAST",

    "attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_32_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            ROLLING_ATTEMPT_DIR
        ),

    "validation_predictions":
        str(
            VALIDATION_PREDICTIONS_PATH
        ),

    "test_predictions":
        str(
            TEST_PREDICTIONS_PATH
        ),

    "audit_manifest":
        str(
            ROLLING_AUDIT_PATH
        ),

    "audit_sha256":
        ROLLING_AUDIT_SHA256,

    "created_utc":
        RUN_UTC.isoformat(),
}


atomic_json_write(
    LATEST_ROLLING_POINTER,
    LATEST_ROLLING_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "rolling_v4_v5"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_32_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_ROLLING_POINTER_PATH
        ),

    "attempt_directory":
        str(
            ROLLING_ATTEMPT_DIR
        ),

    "audit_manifest":
        str(
            ROLLING_AUDIT_PATH
        ),

    "audit_sha256":
        ROLLING_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_ROLLING_V4_V5_COMPLETE"
    if CELL_32_PASS
    else
    "HB_2023_ROLLING_V4_V5_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_ROLLING_VALIDATION = (
    ROLLING_VALIDATION
)


HB2023_ROLLING_TEST = (
    ROLLING_TEST
)


HB2023_ROLLING_DATE_STATES = (
    DATE_STATES
)


HB2023_V4_V5_POINT_METRICS = (
    POINT_METRICS
)


HB2023_V4_V5_GAP_METRICS = (
    GAP_METRICS
)


HB2023_V4_TO_V5_GAIN_SUMMARY = (
    V5_GAIN_SUMMARY
)


HB2023_ROLLING_ATTEMPT_DIR = (
    ROLLING_ATTEMPT_DIR
)


NOTEBOOK_2_CELL_32_PASS = (
    CELL_32_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2023 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY")
print("=" * 160)


print()
print("OVERALL POINT PERFORMANCE")
print("-" * 160)

display(
    V5_GAIN_SUMMARY
)


print()
print("POINT METRICS")
print("-" * 160)

display(
    POINT_METRICS
)


print()
print("BY STATE GAP")
print("-" * 160)

display(
    GAP_METRICS
)


print()
print("ROLLING STATE UPDATE SUMMARY")
print("-" * 160)

state_summary = pd.DataFrame(
    [
        {
            "athlete_dates":
                int(
                    len(
                        DATE_STATES
                    )
                ),

            "athletes":
                int(
                    DATE_STATES[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "mean_standardized_innovation":
                float(
                    DATE_STATES[
                        "standardized_innovation"
                    ]
                    .mean()
                ),

            "sd_standardized_innovation":
                float(
                    DATE_STATES[
                        "standardized_innovation"
                    ]
                    .std(
                        ddof=1
                    )
                ),

            "mean_kalman_gain":
                float(
                    DATE_STATES[
                        "kalman_gain"
                    ]
                    .mean()
                ),
        }
    ]
)


display(
    state_summary
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{ROLLING_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{ROLLING_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{ROLLING_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Finite predictions             : "
    f"{FINITE_PREDICTION_PASS}"
)

print(
    f"Positive variances             : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"No same-day leakage            : "
    f"{NO_SAME_DAY_LEAKAGE_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Split chronology               : "
    f"{SPLIT_CHRONOLOGY_PASS}"
)

print(
    f"Gap reconciliation             : "
    f"{GAP_RECONCILIATION_PASS}"
)

print(
    f"Maximum gap discrepancy        : "
    f"{GAP_RECONCILIATION_MAX_DIFF:.12e}"
)


print()
print("=" * 160)

if CELL_32_PASS:

    print(
        "CELL 32 STATUS: HB_2023 ROLLING V4/V5 FORECAST PASSED"
    )

else:

    print(
        "CELL 32 STATUS: HB_2023 ROLLING V4/V5 FORECAST REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 23
# Original source SHA256: 9720f7dc3fabc764f995db6879290d109fc3997fb7f4828e2ef05354bb68b252
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 33 — HB_2023 HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY
# ==================================================================================================
#
# Purpose
# -------
# Reproduce the V6 structural-persistence calibration at the HB_2023 historical cutoff.
#
#
# Inputs
# ------
# Frozen historical models:
#
#   V4 fitted only through 2022-12-31
#   V5 OU fitted only through 2022-12-31
#
#
# Calibration data:
#
#   HB_2023 VALIDATION only:
#       2023-01-01 -> 2023-04-30
#
#
# TEST:
#
#   NOT USED
#
#
# Calibration sample
# ------------------
# First VALIDATION date per:
#
#       athlete × model_group
#
# restricted to:
#
#       1 <= rolling_state_gap_days <= 365
#
# Multiple performances on that same athlete/group/date are averaged so one competition date
# cannot receive extra structural-calibration weight merely because multiple marks were recorded.
#
#
# Structural model
# ----------------
#
# V5 already contains the full historical V4 athlete effects:
#
#   V5 =
#       population
#       + athlete_intercept
#       + athlete_slope_component
#       + OU_state
#
#
# Define:
#
#   base_without_athlete_structure =
#
#       V5
#       - athlete_intercept_component
#       - athlete_slope_component
#
#
# V6 full structural mean:
#
#   V6_FULL =
#
#       base_without_athlete_structure
#       + rho_I(h) * athlete_intercept_component
#       + rho_S(h) * athlete_slope_component
#       + frozen_offset
#
#
# Persistence candidates
# ----------------------
#
#   M0_CONST_CONST
#       rho_I(h) = c_I
#       rho_S(h) = c_S
#
#   M1_CONST_EXP
#       rho_I(h) = c_I
#       rho_S(h) = exp(-h / tau_S)
#
#   M2_EXP_CONST
#       rho_I(h) = exp(-h / tau_I)
#       rho_S(h) = c_S
#
#   M3_EXP_EXP
#       rho_I(h) = exp(-h / tau_I)
#       rho_S(h) = exp(-h / tau_S)
#
#
# Candidate selection
# -------------------
# Persistence is selected by Gaussian likelihood / AICc on sex × gap-band centered variables.
# This prevents level offsets from determining the persistence coefficients.
#
# After persistence is frozen, compare offset structures:
#
#       NONE
#       COMMON
#       SEX
#       GROUP
#
# again by Gaussian likelihood / AICc.
#
#
# Audit / versioning
# ------------------
# Every execution creates an immutable attempt directory.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import norm


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_32_PASS",

    "HB2023_ROLLING_VALIDATION",
    "HB2023_ROLLING_TEST",

    "HB2023_ROLLING_ATTEMPT_DIR",

    "HB2023_V5_TAU_DAYS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_32_PASS:

    raise RuntimeError(
        "Cell 32 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

CELL_ID = "CELL_33_HB2023_V6_MEAN_CALIBRATION"

STAGE_SCHEMA_VERSION = "1.0.0"


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
]


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2023_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V6_STAGE_ROOT = (
    HB2023_DIR
    /
    "v6_mean_calibration"
)


V6_ATTEMPT_DIR = (
    V6_STAGE_ROOT
    /
    ATTEMPT_ID
)


V6_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(
        path
    )

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


def gaussian_fit_statistics(
    residual,
    n_mean_parameters,
):

    residual = np.asarray(
        residual,
        dtype=float,
    )


    n = len(
        residual
    )


    rss = float(
        np.sum(
            residual
            **
            2
        )
    )


    sigma2 = max(
        rss
        /
        n,
        1e-12,
    )


    nll = float(
        0.5
        *
        n
        *
        (
            np.log(
                2.0
                *
                np.pi
                *
                sigma2
            )
            +
            1.0
        )
    )


    # Mean parameters + Gaussian sigma.
    k = int(
        n_mean_parameters
        +
        1
    )


    aic = float(
        2.0
        *
        k
        +
        2.0
        *
        nll
    )


    if n <= (
        k
        +
        1
    ):

        aicc = np.inf


    else:

        aicc = float(
            aic

            +

            (
                2.0
                *
                k
                *
                (
                    k
                    +
                    1
                )
            )
            /
            (
                n
                -
                k
                -
                1
            )
        )


    bic = float(
        np.log(
            n
        )
        *
        k

        +

        2.0
        *
        nll
    )


    return {
        "n":
            int(
                n
            ),

        "rss":
            rss,

        "sigma":
            float(
                np.sqrt(
                    sigma2
                )
            ),

        "nll":
            nll,

        "k":
            k,

        "AIC":
            aic,

        "AICc":
            aicc,

        "BIC":
            bic,
    }


# --------------------------------------------------------------------------------------------------
# 5. VALIDATION ONLY
# --------------------------------------------------------------------------------------------------

VAL = (
    HB2023_ROLLING_VALIDATION.copy()
)


# Explicit QA: absolutely no TEST data enter calibration.
TEST_TOUCHED = False


VAL[
    "sex_group"
] = (
    VAL[
        "model_group"
    ]
    .astype(str)
    .str[
        0
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. FIRST VALIDATION DATE PER ATHLETE × MODEL GROUP
# --------------------------------------------------------------------------------------------------

first_date = (
    VAL.groupby(
        [
            "aaAthleteId",
            "model_group",
        ]
    )[
        "result_date_dt"
    ]
    .transform(
        "min"
    )
)


CAL_ROWS = (
    VAL.loc[
        VAL[
            "result_date_dt"
        ]
        .eq(
            first_date
        )
    ]
    .copy()
)


# One row per athlete × group × first date.
CAL = (
    CAL_ROWS.groupby(
        [
            "aaAthleteId",
            "model_group",
            "sex_group",
            "result_date_dt",
        ],
        as_index=False,
    )
    .agg(
        mark_numeric_m=(
            "mark_numeric_m",
            "mean",
        ),

        v5_prediction_mean=(
            "v5_prediction_mean",
            "mean",
        ),

        state_prior_mean=(
            "state_prior_mean",
            "mean",
        ),

        rolling_state_gap_days=(
            "rolling_state_gap_days",
            "first",
        ),

        v4_population_component=(
            "v4_population_component",
            "mean",
        ),

        v4_athlete_intercept_component=(
            "v4_athlete_intercept_component",
            "mean",
        ),

        v4_athlete_slope_component=(
            "v4_athlete_slope_component",
            "mean",
        ),

        n_same_day_rows=(
            "mark_numeric_m",
            "size",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 7. CALIBRATION DOMAIN — 1 TO 365 DAYS
# --------------------------------------------------------------------------------------------------

CAL = (
    CAL.loc[
        CAL[
            "rolling_state_gap_days"
        ]
        .between(
            1,
            365,
            inclusive="both",
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


CAL[
    "gap_band"
] = pd.cut(
    CAL[
        "rolling_state_gap_days"
    ],

    bins=GAP_BINS,

    labels=GAP_LABELS,

    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 8. STRUCTURAL CALIBRATION TARGET
# --------------------------------------------------------------------------------------------------

CAL[
    "base_without_athlete_structure"
] = (
    CAL[
        "v5_prediction_mean"
    ]

    -

    CAL[
        "v4_athlete_intercept_component"
    ]

    -

    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "structural_target"
] = (
    CAL[
        "mark_numeric_m"
    ]

    -

    CAL[
        "base_without_athlete_structure"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. CENTER BY SEX × GAP BAND
# --------------------------------------------------------------------------------------------------
#
# Persistence is estimated from within-cell covariation, while level offsets are handled afterward.
#
# --------------------------------------------------------------------------------------------------

CENTER_GROUP = [
    "sex_group",
    "gap_band",
]


for column in [
    "structural_target",
    "v4_athlete_intercept_component",
    "v4_athlete_slope_component",
]:

    group_mean = (
        CAL.groupby(
            CENTER_GROUP,
            observed=True,
        )[
            column
        ]
        .transform(
            "mean"
        )
    )


    CAL[
        f"{column}_centered"
    ] = (
        CAL[
            column
        ]

        -

        group_mean
    )


# --------------------------------------------------------------------------------------------------
# 10. EMPIRICAL HC3 OLS DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

y_centered = (
    CAL[
        "structural_target_centered"
    ]
    .to_numpy(
        dtype=float
    )
)


I_centered = (
    CAL[
        "v4_athlete_intercept_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)


S_centered = (
    CAL[
        "v4_athlete_slope_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)


X = np.column_stack(
    [
        np.ones(
            len(
                CAL
            )
        ),

        I_centered,

        S_centered,
    ]
)


beta_ols = np.linalg.lstsq(
    X,
    y_centered,
    rcond=None,
)[0]


ols_residual = (
    y_centered
    -
    X
    @
    beta_ols
)


XtX_inv = np.linalg.inv(
    X.T
    @
    X
)


hat_diag = np.sum(
    (
        X
        @
        XtX_inv
    )
    *
    X,
    axis=1,
)


hc3_scale = (
    ols_residual
    /
    np.maximum(
        1.0
        -
        hat_diag,
        1e-8,
    )
) ** 2


meat = (
    X.T
    @
    (
        X
        *
        hc3_scale[
            :,
            None
        ]
    )
)


HC3_COV = (
    XtX_inv
    @
    meat
    @
    XtX_inv
)


HC3_SE = np.sqrt(
    np.diag(
        HC3_COV
    )
)


HC3_Z = (
    beta_ols
    /
    HC3_SE
)


HC3_P = (
    2.0
    *
    norm.sf(
        np.abs(
            HC3_Z
        )
    )
)


HC3_SUMMARY = pd.DataFrame(
    {
        "term": [
            "intercept",
            "rho_intercept_empirical",
            "rho_slope_empirical",
        ],

        "estimate":
            beta_ols,

        "HC3_SE":
            HC3_SE,

        "z":
            HC3_Z,

        "p_value":
            HC3_P,
    }
)


# --------------------------------------------------------------------------------------------------
# 11. PERSISTENCE FUNCTIONS
# --------------------------------------------------------------------------------------------------

h = (
    CAL[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


I = I_centered

S = S_centered

Y = y_centered


def persistence_values(
    model_name,
    params,
    horizon,
):

    horizon = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        c_I, c_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M1_CONST_EXP":

        c_I, tau_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    elif model_name == "M2_EXP_CONST":

        tau_I, c_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M3_EXP_EXP":

        tau_I, tau_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    else:

        raise ValueError(
            model_name
        )


    return (
        rho_I,
        rho_S,
    )


def persistence_sse(
    model_name,
    params,
):

    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )


    pred = (
        rho_I
        *
        I

        +

        rho_S
        *
        S
    )


    residual = (
        Y
        -
        pred
    )


    return float(
        np.sum(
            residual
            **
            2
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. PERSISTENCE MODEL CONFIGURATION
# --------------------------------------------------------------------------------------------------

PERSISTENCE_CONFIG = {

    "M0_CONST_CONST": {
        "bounds": [
            (0.0, 1.5),
            (0.0, 1.5),
        ],

        "starts": [
            (0.50, 0.50),
            (0.75, 0.75),
            (1.00, 1.00),
            (1.25, 1.25),
            (0.75, 1.00),
            (1.00, 0.75),
        ],

        "parameter_names": [
            "c_intercept",
            "c_slope",
        ],
    },

    "M1_CONST_EXP": {
        "bounds": [
            (0.0, 1.5),
            (1.0, 5000.0),
        ],

        "starts": [
            (0.75, 60.0),
            (0.90, 120.0),
            (1.00, 365.0),
            (1.10, 730.0),
            (1.00, 1500.0),
        ],

        "parameter_names": [
            "c_intercept",
            "tau_slope_days",
        ],
    },

    "M2_EXP_CONST": {
        "bounds": [
            (1.0, 5000.0),
            (0.0, 1.5),
        ],

        "starts": [
            (60.0, 0.75),
            (120.0, 0.90),
            (365.0, 1.00),
            (730.0, 1.10),
            (1500.0, 1.00),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "c_slope",
        ],
    },

    "M3_EXP_EXP": {
        "bounds": [
            (1.0, 5000.0),
            (1.0, 5000.0),
        ],

        "starts": [
            (60.0, 60.0),
            (120.0, 120.0),
            (365.0, 365.0),
            (730.0, 365.0),
            (365.0, 730.0),
            (1500.0, 1500.0),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "tau_slope_days",
        ],
    },
}


# --------------------------------------------------------------------------------------------------
# 13. FIT PERSISTENCE CANDIDATES
# --------------------------------------------------------------------------------------------------

candidate_rows = []

candidate_best = {}


for model_name, config in (
    PERSISTENCE_CONFIG.items()
):

    results = []


    for start_id, x0 in enumerate(
        config[
            "starts"
        ],
        start=1,
    ):

        result = minimize(
            lambda p:
                persistence_sse(
                    model_name,
                    p,
                ),

            x0=np.asarray(
                x0,
                dtype=float,
            ),

            method="L-BFGS-B",

            bounds=config[
                "bounds"
            ],

            options={
                "maxiter":
                    3000,

                "ftol":
                    1e-12,

                "gtol":
                    1e-8,
            },
        )


        results.append(
            result
        )


    successful = [
        result
        for result in results
        if (
            result.success
            and
            np.isfinite(
                result.fun
            )
        )
    ]


    if not successful:

        raise RuntimeError(
            f"No successful optimization for {model_name}."
        )


    best = min(
        successful,
        key=lambda r:
            r.fun,
    )


    params = (
        np.asarray(
            best.x,
            dtype=float,
        )
    )


    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )


    residual = (
        Y
        -
        (
            rho_I
            *
            I

            +

            rho_S
            *
            S
        )
    )


    stats = gaussian_fit_statistics(
        residual,
        n_mean_parameters=len(
            params
        ),
    )


    parameter_dict = {
        name:
            float(
                value
            )

        for name, value in zip(
            config[
                "parameter_names"
            ],
            params,
        )
    }


    candidate_rows.append(
        {
            "model":
                model_name,

            "successful_starts":
                int(
                    sum(
                        result.success
                        for result in results
                    )
                ),

            **parameter_dict,

            **stats,
        }
    )


    candidate_best[
        model_name
    ] = {
        "params":
            params,

        "parameter_dict":
            parameter_dict,

        "fit":
            stats,
    }


PERSISTENCE_CANDIDATES = pd.DataFrame(
    candidate_rows
)


PERSISTENCE_CANDIDATES[
    "delta_AICc"
] = (
    PERSISTENCE_CANDIDATES[
        "AICc"
    ]

    -

    PERSISTENCE_CANDIDATES[
        "AICc"
    ]
    .min()
)


PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_PERSISTENCE_MODEL = str(
    PERSISTENCE_CANDIDATES.loc[
        0,
        "model",
    ]
)


SELECTED_PERSISTENCE_PARAMS = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "params"
    ]
)


SELECTED_PERSISTENCE_PARAMETER_DICT = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "parameter_dict"
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. FROZEN PERSISTENCE ON RAW CALIBRATION SAMPLE
# --------------------------------------------------------------------------------------------------

rho_I_raw, rho_S_raw = (
    persistence_values(
        SELECTED_PERSISTENCE_MODEL,
        SELECTED_PERSISTENCE_PARAMS,
        CAL[
            "rolling_state_gap_days"
        ]
        .to_numpy(
            dtype=float
        ),
    )
)


CAL[
    "rho_intercept"
] = (
    rho_I_raw
)


CAL[
    "rho_slope"
] = (
    rho_S_raw
)


CAL[
    "v6_without_offset"
] = (
    CAL[
        "base_without_athlete_structure"
    ]

    +

    CAL[
        "rho_intercept"
    ]
    *
    CAL[
        "v4_athlete_intercept_component"
    ]

    +

    CAL[
        "rho_slope"
    ]
    *
    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "offset_target"
] = (
    CAL[
        "mark_numeric_m"
    ]

    -

    CAL[
        "v6_without_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. OFFSET CANDIDATES
# --------------------------------------------------------------------------------------------------

offset_candidate_rows = []

offset_specs = {}


# NONE
offset_pred = np.zeros(
    len(
        CAL
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    offset_pred,

    n_mean_parameters=0,
)


offset_candidate_rows.append(
    {
        "offset_model":
            "NONE",

        **stats,
    }
)


offset_specs[
    "NONE"
] = {}


# COMMON
common_offset = float(
    CAL[
        "offset_target"
    ]
    .mean()
)


offset_pred = np.full(
    len(
        CAL
    ),
    common_offset,
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    offset_pred,

    n_mean_parameters=1,
)


offset_candidate_rows.append(
    {
        "offset_model":
            "COMMON",

        **stats,
    }
)


offset_specs[
    "COMMON"
] = {
    "COMMON":
        common_offset
}


# SEX
sex_offsets = (
    CAL.groupby(
        "sex_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


offset_pred = (
    CAL[
        "sex_group"
    ]
    .map(
        sex_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    offset_pred,

    n_mean_parameters=len(
        sex_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "SEX",

        **stats,
    }
)


offset_specs[
    "SEX"
] = {
    str(k):
        float(v)

    for k, v in sex_offsets.items()
}


# GROUP
group_offsets = (
    CAL.groupby(
        "model_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


offset_pred = (
    CAL[
        "model_group"
    ]
    .map(
        group_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    offset_pred,

    n_mean_parameters=len(
        group_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "GROUP",

        **stats,
    }
)


offset_specs[
    "GROUP"
] = {
    str(k):
        float(v)

    for k, v in group_offsets.items()
}


OFFSET_CANDIDATES = pd.DataFrame(
    offset_candidate_rows
)


OFFSET_CANDIDATES[
    "delta_AICc"
] = (
    OFFSET_CANDIDATES[
        "AICc"
    ]

    -

    OFFSET_CANDIDATES[
        "AICc"
    ]
    .min()
)


OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_OFFSET_MODEL = str(
    OFFSET_CANDIDATES.loc[
        0,
        "offset_model",
    ]
)


SELECTED_OFFSETS = (
    offset_specs[
        SELECTED_OFFSET_MODEL
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. APPLY SELECTED OFFSET TO CALIBRATION SAMPLE
# --------------------------------------------------------------------------------------------------

if SELECTED_OFFSET_MODEL == "NONE":

    CAL[
        "v6_offset"
    ] = 0.0


elif SELECTED_OFFSET_MODEL == "COMMON":

    CAL[
        "v6_offset"
    ] = float(
        SELECTED_OFFSETS[
            "COMMON"
        ]
    )


elif SELECTED_OFFSET_MODEL == "SEX":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "sex_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


elif SELECTED_OFFSET_MODEL == "GROUP":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "model_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        SELECTED_OFFSET_MODEL
    )


CAL[
    "v6_prediction_mean"
] = (
    CAL[
        "v6_without_offset"
    ]

    +

    CAL[
        "v6_offset"
    ]
)


CAL[
    "v5_error"
] = (
    CAL[
        "mark_numeric_m"
    ]

    -

    CAL[
        "v5_prediction_mean"
    ]
)


CAL[
    "v6_error"
] = (
    CAL[
        "mark_numeric_m"
    ]

    -

    CAL[
        "v6_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. VALIDATION CALIBRATION METRICS
# --------------------------------------------------------------------------------------------------

V5_MAE = float(
    CAL[
        "v5_error"
    ]
    .abs()
    .mean()
)


V6_MAE = float(
    CAL[
        "v6_error"
    ]
    .abs()
    .mean()
)


V5_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v5_error"
            ]
            **
            2
        )
    )
)


V6_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v6_error"
            ]
            **
            2
        )
    )
)


CALIBRATION_METRICS = pd.DataFrame(
    [
        {
            "n":
                int(
                    len(
                        CAL
                    )
                ),

            "athletes":
                int(
                    CAL[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V5_to_V6_MAE_gain_m":
                V5_MAE
                -
                V6_MAE,

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V5_to_V6_RMSE_gain_m":
                V5_RMSE
                -
                V6_RMSE,

            "V6_Bias":
                float(
                    CAL[
                        "v6_error"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. PERSISTENCE REFERENCE TABLE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


REF_RHO_I, REF_RHO_S = (
    persistence_values(
        SELECTED_PERSISTENCE_MODEL,
        SELECTED_PERSISTENCE_PARAMS,
        REFERENCE_DAYS,
    )
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS.astype(
                int
            ),

        "rho_intercept":
            REF_RHO_I,

        "rho_slope":
            REF_RHO_S,
    }
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

CALIBRATION_DATE_PASS = bool(
    CAL[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2023-01-01"
        ),
        pd.Timestamp(
            "2023-04-30"
        ),
        inclusive="both",
    )
    .all()
)


GAP_DOMAIN_PASS = bool(
    CAL[
        "rolling_state_gap_days"
    ]
    .between(
        1,
        365,
        inclusive="both",
    )
    .all()
)


ONE_FIRST_DATE_PASS = bool(
    not CAL.duplicated(
        subset=[
            "aaAthleteId",
            "model_group",
        ]
    )
    .any()
)


FINITE_PASS = bool(
    np.isfinite(
        CAL[
            [
                "structural_target",
                "rho_intercept",
                "rho_slope",
                "v6_offset",
                "v6_prediction_mean",
                "v6_error",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


PERSISTENCE_SELECTION_PASS = bool(
    len(
        PERSISTENCE_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        PERSISTENCE_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


OFFSET_SELECTION_PASS = bool(
    len(
        OFFSET_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        OFFSET_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


TEST_NOT_USED_PASS = bool(
    not TEST_TOUCHED
)


CELL_33_PASS = bool(
    CALIBRATION_DATE_PASS
    and
    GAP_DOMAIN_PASS
    and
    ONE_FIRST_DATE_PASS
    and
    FINITE_PASS
    and
    PERSISTENCE_SELECTION_PASS
    and
    OFFSET_SELECTION_PASS
    and
    TEST_NOT_USED_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

CALIBRATION_SAMPLE_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_validation_calibration_sample.parquet"
)


PERSISTENCE_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_persistence_candidates.parquet"
)


OFFSET_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_offset_candidates.parquet"
)


HC3_SUMMARY_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_empirical_persistence_hc3.parquet"
)


V6_MEAN_SPEC_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_spec.json"
)


V6_AUDIT_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_calibration_audit.json"
)


CAL.to_parquet(
    CALIBRATION_SAMPLE_PATH,
    index=False,
)


PERSISTENCE_CANDIDATES.to_parquet(
    PERSISTENCE_CANDIDATES_PATH,
    index=False,
)


OFFSET_CANDIDATES.to_parquet(
    OFFSET_CANDIDATES_PATH,
    index=False,
)


HC3_SUMMARY.to_parquet(
    HC3_SUMMARY_PATH,
    index=False,
)


V6_MEAN_SPEC = {
    "model":
        "HISTORICAL_V6_FULL_STRUCTURAL_CORRECTION",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "calibration_window":
        {
            "start":
                "2023-01-01",

            "end":
                "2023-04-30",
        },

    "calibration_rule":
        (
            "First validation date per athlete × model_group; "
            "gap restricted to 1–365 days."
        ),

    "selected_persistence_model":
        SELECTED_PERSISTENCE_MODEL,

    "selected_persistence_parameters":
        SELECTED_PERSISTENCE_PARAMETER_DICT,

    "selected_offset_model":
        SELECTED_OFFSET_MODEL,

    "selected_offsets":
        SELECTED_OFFSETS,

    "formula":
        (
            "V6_FULL = "
            "V5 - athlete_intercept - athlete_slope_component "
            "+ rho_I(h)*athlete_intercept "
            "+ rho_S(h)*athlete_slope_component "
            "+ offset"
        ),

    "test_used":
        False,
}


atomic_json_write(
    V6_MEAN_SPEC,
    V6_MEAN_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "rolling_forecast_audit":
        (
            Path(
                HB2023_ROLLING_ATTEMPT_DIR
            )
            /
            "rolling_forecast_audit.json"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


output_paths = {
    "calibration_sample":
        CALIBRATION_SAMPLE_PATH,

    "persistence_candidates":
        PERSISTENCE_CANDIDATES_PATH,

    "offset_candidates":
        OFFSET_CANDIDATES_PATH,

    "hc3_summary":
        HC3_SUMMARY_PATH,

    "v6_mean_spec":
        V6_MEAN_SPEC_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


V6_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V6_MEAN_CALIBRATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_33_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Historical validation-only reconstruction of the "
            "V6 structural-persistence mean model for HB_2023."
        ),

    "test_used":
        False,

    "sample": {
        "rows":
            int(
                len(
                    CAL
                )
            ),

        "athletes":
            int(
                CAL[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "groups":
            sorted(
                CAL[
                    "model_group"
                ]
                .unique()
                .tolist()
            ),

        "minimum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ]
                .min()
            ),

        "maximum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ]
                .max()
            ),
    },

    "selected_model":
        V6_MEAN_SPEC,

    "calibration_metrics":
        CALIBRATION_METRICS.iloc[
            0
        ]
        .to_dict(),

    "qa": {
        "calibration_date_pass":
            CALIBRATION_DATE_PASS,

        "gap_domain_pass":
            GAP_DOMAIN_PASS,

        "one_first_date_per_athlete_group":
            ONE_FIRST_DATE_PASS,

        "finite_outputs":
            FINITE_PASS,

        "persistence_selection":
            PERSISTENCE_SELECTION_PASS,

        "offset_selection":
            OFFSET_SELECTION_PASS,

        "test_not_used":
            TEST_NOT_USED_PASS,

        "cell_pass":
            CELL_33_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    V6_AUDIT,
    V6_AUDIT_PATH,
)


V6_AUDIT_SHA256 = (
    sha256_file(
        V6_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 22. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_V6_POINTER_PATH = (
    V6_STAGE_ROOT
    /
    "LATEST_V6_MEAN_CALIBRATION.json"
)


LATEST_V6_POINTER = {
    "fold":
        FOLD_NAME,

    "stage":
        "V6_MEAN_CALIBRATION",

    "attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_33_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            V6_ATTEMPT_DIR
        ),

    "v6_mean_spec":
        str(
            V6_MEAN_SPEC_PATH
        ),

    "audit_manifest":
        str(
            V6_AUDIT_PATH
        ),

    "audit_sha256":
        V6_AUDIT_SHA256,

    "created_utc":
        RUN_UTC.isoformat(),
}


atomic_json_write(
    LATEST_V6_POINTER,
    LATEST_V6_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v6_mean_calibration"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_33_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            V6_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_V6_POINTER_PATH
        ),

    "mean_spec":
        str(
            V6_MEAN_SPEC_PATH
        ),

    "audit_manifest":
        str(
            V6_AUDIT_PATH
        ),

    "audit_sha256":
        V6_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_V6_MEAN_CALIBRATION_COMPLETE"
    if CELL_33_PASS
    else
    "HB_2023_V6_MEAN_CALIBRATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_V6_CALIBRATION_SAMPLE = (
    CAL
)


HB2023_V6_PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES
)


HB2023_V6_OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES
)


HB2023_V6_MEAN_SPEC = (
    V6_MEAN_SPEC
)


HB2023_V6_ATTEMPT_DIR = (
    V6_ATTEMPT_DIR
)


HB2023_V6_MEAN_SPEC_PATH = (
    V6_MEAN_SPEC_PATH
)


NOTEBOOK_2_CELL_33_PASS = (
    CELL_33_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2023 — HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY")
print("=" * 160)


print()
print("CALIBRATION SAMPLE")
print("-" * 160)

print(
    f"Rows                           : "
    f"{len(CAL):,}"
)

print(
    f"Athletes                       : "
    f"{CAL['aaAthleteId'].nunique():,}"
)

print(
    f"Gap range                      : "
    f"{CAL['rolling_state_gap_days'].min():.0f}"
    f" -> "
    f"{CAL['rolling_state_gap_days'].max():.0f} days"
)

print(
    f"TEST used                      : "
    f"{TEST_TOUCHED}"
)


print()
print("EMPIRICAL CENTERED HC3 PERSISTENCE")
print("-" * 160)

display(
    HC3_SUMMARY
)


print()
print("PERSISTENCE MODEL COMPARISON")
print("-" * 160)

display(
    PERSISTENCE_CANDIDATES
)


print()
print(
    f"Selected persistence model     : "
    f"{SELECTED_PERSISTENCE_MODEL}"
)

print(
    f"Selected parameters            : "
    f"{SELECTED_PERSISTENCE_PARAMETER_DICT}"
)


print()
print("SELECTED PERSISTENCE BY HORIZON")
print("-" * 160)

display(
    PERSISTENCE_REFERENCE
)


print()
print("OFFSET MODEL COMPARISON")
print("-" * 160)

display(
    OFFSET_CANDIDATES
)


print()
print(
    f"Selected offset model          : "
    f"{SELECTED_OFFSET_MODEL}"
)

print(
    f"Selected offsets               : "
    f"{SELECTED_OFFSETS}"
)


print()
print("VALIDATION CALIBRATION METRICS")
print("-" * 160)

display(
    CALIBRATION_METRICS
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{V6_ATTEMPT_DIR}"
)

print(
    f"Mean specification             : "
    f"{V6_MEAN_SPEC_PATH}"
)

print(
    f"Audit manifest                 : "
    f"{V6_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{V6_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"VALIDATION dates only          : "
    f"{CALIBRATION_DATE_PASS}"
)

print(
    f"Gap domain 1–365 days          : "
    f"{GAP_DOMAIN_PASS}"
)

print(
    f"First athlete/group date only  : "
    f"{ONE_FIRST_DATE_PASS}"
)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"Persistence candidates valid   : "
    f"{PERSISTENCE_SELECTION_PASS}"
)

print(
    f"Offset candidates valid        : "
    f"{OFFSET_SELECTION_PASS}"
)

print(
    f"TEST untouched                 : "
    f"{TEST_NOT_USED_PASS}"
)


print()
print("=" * 160)

if CELL_33_PASS:

    print(
        "CELL 33 STATUS: HB_2023 HISTORICAL V6 MEAN CALIBRATION PASSED"
    )

else:

    print(
        "CELL 33 STATUS: HB_2023 V6 MEAN CALIBRATION REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 24
# Original source SHA256: 27866d3e5251caafd2c1a4b4633fadcc5fff746232bababab56c08b16320db12
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 34 — HB_2023 UNTOUCHED TEST: V5 vs V6 vs V6.1 OU-FRESHNESS BLEND
# ==================================================================================================
#
# Purpose
# -------
# First genuinely independent historical evaluation of the V6.1 architecture.
#
#
# Frozen before TEST:
#
#   V4                fitted <= 2022-12-31
#   V5 OU             fitted <= 2022-12-31
#   V6 mean spec      calibrated on 2023-01-01 -> 2023-04-30 VALIDATION only
#
#
# TEST:
#
#   2023-05-01 -> 2023-08-30
#
#
# V6.1
# ----
#
#   w(h) = 1 - exp(-h / tau_V5)
#
#   V6.1 =
#       V5
#       + w(h) * (V6_FULL - V5)
#
#
# No V6.1 parameters are fitted.
# tau comes entirely from historical TRAIN.
#
#
# Evaluation
# ----------
#   • overall MAE / RMSE / Bias
#   • by state-gap band
#   • by model group
#   • paired athlete-cluster bootstrap
#
#
# Positive "gain" means the second model is better.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_33_PASS",

    "HB2023_ROLLING_TEST",

    "HB2023_V6_MEAN_SPEC",
    "HB2023_V6_MEAN_SPEC_PATH",
    "HB2023_V6_ATTEMPT_DIR",

    "HB2023_ROLLING_ATTEMPT_DIR",

    "HB2023_V5_TAU_DAYS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_33_PASS:

    raise RuntimeError(
        "Cell 33 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

CELL_ID = "CELL_34_HB2023_TEST_V6_V61"

STAGE_SCHEMA_VERSION = "1.0.0"

BOOTSTRAP_SEED = 42
BOOTSTRAP_REPS = 5000


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


# Broader operational bands for bootstrap inference.
BOOTSTRAP_HORIZONS = {
    "ALL":
        (0, np.inf),

    "01-30":
        (0, 30),

    "31-60":
        (30, 60),

    "61-120":
        (60, 120),

    "121-365":
        (120, 365),

    "366+":
        (365, np.inf),
}


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2023_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


TEST_STAGE_ROOT = (
    HB2023_DIR
    /
    "v61_test_evaluation"
)


TEST_ATTEMPT_DIR = (
    TEST_STAGE_ROOT
    /
    ATTEMPT_ID
)


TEST_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. LOAD FROZEN TEST + V6 SPECIFICATION
# --------------------------------------------------------------------------------------------------

TEST = (
    HB2023_ROLLING_TEST.copy()
    .reset_index(
        drop=True
    )
)


V6_SPEC = dict(
    HB2023_V6_MEAN_SPEC
)


TAU_V5 = float(
    HB2023_V5_TAU_DAYS
)


SELECTED_MODEL = V6_SPEC[
    "selected_persistence_model"
]


PERSISTENCE_PARAMS = V6_SPEC[
    "selected_persistence_parameters"
]


OFFSET_MODEL = V6_SPEC[
    "selected_offset_model"
]


OFFSETS = V6_SPEC[
    "selected_offsets"
]


TEST[
    "sex_group"
] = (
    TEST[
        "model_group"
    ]
    .astype(str)
    .str[
        0
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. FROZEN PERSISTENCE FUNCTION
# --------------------------------------------------------------------------------------------------

def frozen_persistence(
    model_name,
    params,
    horizon,
):

    h = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M1_CONST_EXP":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    elif model_name == "M2_EXP_CONST":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M3_EXP_EXP":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    else:

        raise RuntimeError(
            f"Unknown persistence model: {model_name}"
        )


    return rho_I, rho_S


# --------------------------------------------------------------------------------------------------
# 7. APPLY FROZEN V6 STRUCTURAL SPECIFICATION
# --------------------------------------------------------------------------------------------------

h = (
    TEST[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


rho_I, rho_S = frozen_persistence(
    SELECTED_MODEL,
    PERSISTENCE_PARAMS,
    h,
)


TEST[
    "v6_rho_intercept"
] = rho_I


TEST[
    "v6_rho_slope"
] = rho_S


TEST[
    "v6_base_without_athlete_structure"
] = (
    TEST[
        "v5_prediction_mean"
    ]

    -

    TEST[
        "v4_athlete_intercept_component"
    ]

    -

    TEST[
        "v4_athlete_slope_component"
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. FROZEN OFFSET
# --------------------------------------------------------------------------------------------------

if OFFSET_MODEL == "NONE":

    TEST[
        "v6_offset"
    ] = 0.0


elif OFFSET_MODEL == "COMMON":

    TEST[
        "v6_offset"
    ] = float(
        OFFSETS[
            "COMMON"
        ]
    )


elif OFFSET_MODEL == "SEX":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "sex_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


elif OFFSET_MODEL == "GROUP":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "model_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        f"Unknown offset model: {OFFSET_MODEL}"
    )


if TEST[
    "v6_offset"
].isna().any():

    raise RuntimeError(
        "Frozen V6 offset produced missing values."
    )


# --------------------------------------------------------------------------------------------------
# 9. V6 FULL PREDICTION
# --------------------------------------------------------------------------------------------------

TEST[
    "v6_prediction_mean"
] = (
    TEST[
        "v6_base_without_athlete_structure"
    ]

    +

    TEST[
        "v6_rho_intercept"
    ]
    *
    TEST[
        "v4_athlete_intercept_component"
    ]

    +

    TEST[
        "v6_rho_slope"
    ]
    *
    TEST[
        "v4_athlete_slope_component"
    ]

    +

    TEST[
        "v6_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. V6.1 TRAIN-ANCHORED OU FRESHNESS BLEND
# --------------------------------------------------------------------------------------------------
#
# NO parameter fitting.
#
# tau comes entirely from HB_2023 TRAIN.
#
# --------------------------------------------------------------------------------------------------

TEST[
    "v61_ou_phi"
] = np.exp(
    -
    TEST[
        "rolling_state_gap_days"
    ]
    /
    TAU_V5
)


TEST[
    "v61_weight"
] = (
    1.0
    -
    TEST[
        "v61_ou_phi"
    ]
)


TEST[
    "v61_prediction_mean"
] = (
    TEST[
        "v5_prediction_mean"
    ]

    +

    TEST[
        "v61_weight"
    ]
    *
    (
        TEST[
            "v6_prediction_mean"
        ]

        -

        TEST[
            "v5_prediction_mean"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ERRORS
# --------------------------------------------------------------------------------------------------

for model, column in {
    "v5":
        "v5_prediction_mean",

    "v6":
        "v6_prediction_mean",

    "v61":
        "v61_prediction_mean",
}.items():

    TEST[
        f"{model}_error"
    ] = (
        TEST[
            "mark_numeric_m"
        ]

        -

        TEST[
            column
        ]
    )


# --------------------------------------------------------------------------------------------------
# 12. GAP BANDS
# --------------------------------------------------------------------------------------------------

TEST[
    "state_gap_band"
] = pd.cut(
    TEST[
        "rolling_state_gap_days"
    ],

    bins=GAP_BINS,

    labels=GAP_LABELS,

    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 13. METRIC HELPERS
# --------------------------------------------------------------------------------------------------

def metric_row(
    df,
    model,
):

    error = (
        df[
            f"{model}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    return {
        "model":
            model.upper()
            if model != "v61"
            else
            "V6.1",

        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error
                        **
                        2
                    )
                )
            ),

        "Bias":
            float(
                np.mean(
                    error
                )
            ),
    }


# --------------------------------------------------------------------------------------------------
# 14. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

OVERALL_METRICS = pd.DataFrame(
    [
        metric_row(
            TEST,
            model,
        )

        for model in [
            "v5",
            "v6",
            "v61",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. PAIRWISE GAIN HELPER
# --------------------------------------------------------------------------------------------------

def pairwise_gain(
    df,
    baseline,
    candidate,
):

    e0 = (
        df[
            f"{baseline}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    e1 = (
        df[
            f"{candidate}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    mae0 = float(
        np.mean(
            np.abs(
                e0
            )
        )
    )


    mae1 = float(
        np.mean(
            np.abs(
                e1
            )
        )
    )


    rmse0 = float(
        np.sqrt(
            np.mean(
                e0
                **
                2
            )
        )
    )


    rmse1 = float(
        np.sqrt(
            np.mean(
                e1
                **
                2
            )
        )
    )


    return {
        "baseline":
            (
                baseline.upper()
                if baseline != "v61"
                else
                "V6.1"
            ),

        "candidate":
            (
                candidate.upper()
                if candidate != "v61"
                else
                "V6.1"
            ),

        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "baseline_MAE":
            mae0,

        "candidate_MAE":
            mae1,

        "MAE_gain_m":
            mae0
            -
            mae1,

        "baseline_RMSE":
            rmse0,

        "candidate_RMSE":
            rmse1,

        "RMSE_gain_m":
            rmse0
            -
            rmse1,
    }


PAIRWISE_OVERALL = pd.DataFrame(
    [
        pairwise_gain(
            TEST,
            "v5",
            "v6",
        ),

        pairwise_gain(
            TEST,
            "v5",
            "v61",
        ),

        pairwise_gain(
            TEST,
            "v6",
            "v61",
        ),
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. GAP METRICS
# --------------------------------------------------------------------------------------------------

gap_rows = []


for gap_band, g in TEST.groupby(
    "state_gap_band",
    observed=True,
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )


        row[
            "state_gap_band"
        ] = str(
            gap_band
        )


        row[
            "mean_gap_days"
        ] = float(
            g[
                "rolling_state_gap_days"
            ]
            .mean()
        )


        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ]
            .mean()
        )


        gap_rows.append(
            row
        )


GAP_METRICS = pd.DataFrame(
    gap_rows
)


# --------------------------------------------------------------------------------------------------
# 17. GAP PAIRWISE GAINS
# --------------------------------------------------------------------------------------------------

gap_gain_rows = []


for gap_band, g in TEST.groupby(
    "state_gap_band",
    observed=True,
):

    for baseline, candidate in [
        (
            "v5",
            "v6",
        ),
        (
            "v5",
            "v61",
        ),
        (
            "v6",
            "v61",
        ),
    ]:

        row = pairwise_gain(
            g,
            baseline,
            candidate,
        )


        row[
            "state_gap_band"
        ] = str(
            gap_band
        )


        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ]
            .mean()
        )


        gap_gain_rows.append(
            row
        )


GAP_GAINS = pd.DataFrame(
    gap_gain_rows
)


# --------------------------------------------------------------------------------------------------
# 18. MODEL-GROUP METRICS
# --------------------------------------------------------------------------------------------------

group_rows = []


for model_group, g in TEST.groupby(
    "model_group"
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )


        row[
            "model_group"
        ] = model_group


        group_rows.append(
            row
        )


GROUP_METRICS = pd.DataFrame(
    group_rows
)


# --------------------------------------------------------------------------------------------------
# 19. ATHLETE-CLUSTER BOOTSTRAP
# --------------------------------------------------------------------------------------------------
#
# Resamples ATHLETES, not individual rows.
#
# This preserves within-athlete dependence.
#
# --------------------------------------------------------------------------------------------------

def cluster_bootstrap_pair(
    df,
    baseline,
    candidate,
    reps=5000,
    seed=42,
):

    d = (
        df[
            [
                "aaAthleteId",
                f"{baseline}_error",
                f"{candidate}_error",
            ]
        ]
        .copy()
    )


    d[
        "abs_gain"
    ] = (
        d[
            f"{baseline}_error"
        ]
        .abs()

        -

        d[
            f"{candidate}_error"
        ]
        .abs()
    )


    d[
        "baseline_sq"
    ] = (
        d[
            f"{baseline}_error"
        ]
        **
        2
    )


    d[
        "candidate_sq"
    ] = (
        d[
            f"{candidate}_error"
        ]
        **
        2
    )


    cluster = (
        d.groupby(
            "aaAthleteId",
            as_index=False,
        )
        .agg(
            n=(
                "abs_gain",
                "size",
            ),

            abs_gain_sum=(
                "abs_gain",
                "sum",
            ),

            baseline_sq_sum=(
                "baseline_sq",
                "sum",
            ),

            candidate_sq_sum=(
                "candidate_sq",
                "sum",
            ),
        )
    )


    n_cluster = len(
        cluster
    )


    n = (
        cluster[
            "n"
        ]
        .to_numpy(
            dtype=float
        )
    )


    abs_gain_sum = (
        cluster[
            "abs_gain_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    baseline_sq_sum = (
        cluster[
            "baseline_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    candidate_sq_sum = (
        cluster[
            "candidate_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rng = np.random.default_rng(
        seed
    )


    mae_boot = np.empty(
        reps,
        dtype=float,
    )


    rmse_boot = np.empty(
        reps,
        dtype=float,
    )


    for b in range(
        reps
    ):

        idx = rng.integers(
            0,
            n_cluster,
            size=n_cluster,
        )


        total_n = float(
            n[
                idx
            ]
            .sum()
        )


        mae_boot[
            b
        ] = (
            abs_gain_sum[
                idx
            ]
            .sum()

            /

            total_n
        )


        baseline_rmse = np.sqrt(
            baseline_sq_sum[
                idx
            ]
            .sum()

            /

            total_n
        )


        candidate_rmse = np.sqrt(
            candidate_sq_sum[
                idx
            ]
            .sum()

            /

            total_n
        )


        rmse_boot[
            b
        ] = (
            baseline_rmse
            -
            candidate_rmse
        )


    observed = pairwise_gain(
        df,
        baseline,
        candidate,
    )


    return {
        "baseline":
            observed[
                "baseline"
            ],

        "candidate":
            observed[
                "candidate"
            ],

        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                n_cluster
            ),

        "MAE_gain_m":
            observed[
                "MAE_gain_m"
            ],

        "MAE_CI_low":
            float(
                np.quantile(
                    mae_boot,
                    0.025,
                )
            ),

        "MAE_CI_high":
            float(
                np.quantile(
                    mae_boot,
                    0.975,
                )
            ),

        "MAE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    mae_boot
                    >
                    0
                )
            ),

        "RMSE_gain_m":
            observed[
                "RMSE_gain_m"
            ],

        "RMSE_CI_low":
            float(
                np.quantile(
                    rmse_boot,
                    0.025,
                )
            ),

        "RMSE_CI_high":
            float(
                np.quantile(
                    rmse_boot,
                    0.975,
                )
            ),

        "RMSE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    rmse_boot
                    >
                    0
                )
            ),
    }


# --------------------------------------------------------------------------------------------------
# 20. BOOTSTRAP — OVERALL + OPERATIONAL HORIZONS
# --------------------------------------------------------------------------------------------------

bootstrap_rows = []


for horizon_name, (
    lower,
    upper,
) in BOOTSTRAP_HORIZONS.items():

    if horizon_name == "ALL":

        subset = TEST.copy()


    else:

        subset = TEST.loc[
            (
                TEST[
                    "rolling_state_gap_days"
                ]
                >
                lower
            )

            &

            (
                TEST[
                    "rolling_state_gap_days"
                ]
                <=
                upper
            )
        ].copy()


    if len(
        subset
    ) == 0:

        continue


    for baseline, candidate in [
        (
            "v5",
            "v6",
        ),
        (
            "v5",
            "v61",
        ),
        (
            "v6",
            "v61",
        ),
    ]:

        row = cluster_bootstrap_pair(
            subset,
            baseline,
            candidate,
            reps=BOOTSTRAP_REPS,
            seed=BOOTSTRAP_SEED,
        )


        row[
            "horizon"
        ] = horizon_name


        bootstrap_rows.append(
            row
        )


BOOTSTRAP_SUMMARY = pd.DataFrame(
    bootstrap_rows
)


# --------------------------------------------------------------------------------------------------
# 21. QA
# --------------------------------------------------------------------------------------------------

FINITE_PASS = bool(
    np.isfinite(
        TEST[
            [
                "v5_prediction_mean",
                "v6_prediction_mean",
                "v61_prediction_mean",

                "v5_error",
                "v6_error",
                "v61_error",

                "v6_rho_intercept",
                "v6_rho_slope",

                "v61_ou_phi",
                "v61_weight",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


WEIGHT_BOUNDS_PASS = bool(
    TEST[
        "v61_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


TEST_DATE_PASS = bool(
    TEST[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2023-05-01"
        ),
        pd.Timestamp(
            "2023-08-30"
        ),
        inclusive="both",
    )
    .all()
)


NO_NEW_PARAMETERS_PASS = True


ROW_ACCOUNTING_PASS = bool(
    len(
        TEST
    )
    ==
    len(
        HB2023_ROLLING_TEST
    )
)


V61_ENDPOINT_LOGIC_PASS = bool(

    (
        TEST.loc[
            TEST[
                "rolling_state_gap_days"
            ]
            <=
            30,
            "v61_weight",
        ]
        .mean()
        <
        0.35
    )

    and

    (
        TEST.loc[
            TEST[
                "rolling_state_gap_days"
            ]
            >
            365,
            "v61_weight",
        ]
        .mean()
        >
        0.95
    )
)


CELL_34_PASS = bool(
    FINITE_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    TEST_DATE_PASS
    and
    NO_NEW_PARAMETERS_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    V61_ENDPOINT_LOGIC_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

TEST_PREDICTIONS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "hb2023_test_v5_v6_v61_predictions.parquet"
)


OVERALL_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "overall_metrics.parquet"
)


PAIRWISE_OVERALL_PATH = (
    TEST_ATTEMPT_DIR
    /
    "pairwise_overall_gains.parquet"
)


GAP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_metrics.parquet"
)


GAP_GAINS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_gains.parquet"
)


GROUP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "group_metrics.parquet"
)


BOOTSTRAP_PATH = (
    TEST_ATTEMPT_DIR
    /
    "athlete_cluster_bootstrap.parquet"
)


TEST_AUDIT_PATH = (
    TEST_ATTEMPT_DIR
    /
    "v61_test_evaluation_audit.json"
)


TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)


OVERALL_METRICS.to_parquet(
    OVERALL_METRICS_PATH,
    index=False,
)


PAIRWISE_OVERALL.to_parquet(
    PAIRWISE_OVERALL_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


GAP_GAINS.to_parquet(
    GAP_GAINS_PATH,
    index=False,
)


GROUP_METRICS.to_parquet(
    GROUP_METRICS_PATH,
    index=False,
)


BOOTSTRAP_SUMMARY.to_parquet(
    BOOTSTRAP_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 23. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v6_mean_spec":
        Path(
            HB2023_V6_MEAN_SPEC_PATH
        ),

    "rolling_test_predictions":
        (
            Path(
                HB2023_ROLLING_ATTEMPT_DIR
            )
            /
            "test_rolling_predictions.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


output_paths = {
    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "overall_metrics":
        OVERALL_METRICS_PATH,

    "pairwise_overall":
        PAIRWISE_OVERALL_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "gap_gains":
        GAP_GAINS_PATH,

    "group_metrics":
        GROUP_METRICS_PATH,

    "athlete_cluster_bootstrap":
        BOOTSTRAP_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 24. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

TEST_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V61_UNTOUCHED_TEST_EVALUATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_34_PASS
            else
            "FAILED"
        ),

    "evaluation_window": {
        "start":
            "2023-05-01",

        "end":
            "2023-08-30",
    },

    "test_role":
        (
            "Untouched historical TEST. "
            "No V6 or V6.1 parameters fitted on this period."
        ),

    "frozen_v6_spec": {
        "persistence_model":
            SELECTED_MODEL,

        "persistence_parameters":
            PERSISTENCE_PARAMS,

        "offset_model":
            OFFSET_MODEL,

        "offsets":
            OFFSETS,
    },

    "v61_spec": {
        "formula":
            "V5 + (1-exp(-gap/tau_V5))*(V6_FULL-V5)",

        "tau_days":
            TAU_V5,

        "tau_source":
            "HB_2023 TRAIN-only V5 OU fit",

        "new_fitted_parameters":
            0,
    },

    "bootstrap": {
        "method":
            "paired athlete-cluster bootstrap",

        "replications":
            BOOTSTRAP_REPS,

        "seed":
            BOOTSTRAP_SEED,
    },

    "qa": {
        "finite":
            FINITE_PASS,

        "weight_bounds":
            WEIGHT_BOUNDS_PASS,

        "test_dates":
            TEST_DATE_PASS,

        "no_new_parameters":
            NO_NEW_PARAMETERS_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "endpoint_logic":
            V61_ENDPOINT_LOGIC_PASS,

        "cell_pass":
            CELL_34_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    TEST_AUDIT,
    TEST_AUDIT_PATH,
)


TEST_AUDIT_SHA256 = (
    sha256_file(
        TEST_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 25. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    TEST_STAGE_ROOT
    /
    "LATEST_V61_TEST_EVALUATION.json"
)


LATEST_POINTER = {
    "fold":
        FOLD_NAME,

    "stage":
        "V61_UNTOUCHED_TEST_EVALUATION",

    "attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_34_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            TEST_ATTEMPT_DIR
        ),

    "test_predictions":
        str(
            TEST_PREDICTIONS_PATH
        ),

    "bootstrap":
        str(
            BOOTSTRAP_PATH
        ),

    "audit_manifest":
        str(
            TEST_AUDIT_PATH
        ),

    "audit_sha256":
        TEST_AUDIT_SHA256,

    "created_utc":
        RUN_UTC.isoformat(),
}


atomic_json_write(
    LATEST_POINTER,
    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 26. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v61_test_evaluation"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_34_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            TEST_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "audit_manifest":
        str(
            TEST_AUDIT_PATH
        ),

    "audit_sha256":
        TEST_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_V61_TEST_EVALUATION_COMPLETE"
    if CELL_34_PASS
    else
    "HB_2023_V61_TEST_EVALUATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 27. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_V61_TEST = TEST

HB2023_V61_OVERALL_METRICS = (
    OVERALL_METRICS
)

HB2023_V61_PAIRWISE_OVERALL = (
    PAIRWISE_OVERALL
)

HB2023_V61_GAP_METRICS = (
    GAP_METRICS
)

HB2023_V61_GAP_GAINS = (
    GAP_GAINS
)

HB2023_V61_GROUP_METRICS = (
    GROUP_METRICS
)

HB2023_V61_BOOTSTRAP = (
    BOOTSTRAP_SUMMARY
)

HB2023_V61_TEST_ATTEMPT_DIR = (
    TEST_ATTEMPT_DIR
)

NOTEBOOK_2_CELL_34_PASS = (
    CELL_34_PASS
)


# --------------------------------------------------------------------------------------------------
# 28. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2023 — UNTOUCHED TEST: V5 vs V6 vs V6.1")
print("=" * 165)


print()
print("FROZEN MODEL SPECIFICATION")
print("-" * 165)

print(
    f"V6 persistence model           : "
    f"{SELECTED_MODEL}"
)

print(
    f"V6 persistence parameters      : "
    f"{PERSISTENCE_PARAMS}"
)

print(
    f"V6 offset model                : "
    f"{OFFSET_MODEL}"
)

print(
    f"V6 offsets                     : "
    f"{OFFSETS}"
)

print(
    f"V6.1 TRAIN-only tau            : "
    f"{TAU_V5:.6f} days"
)


print()
print("OVERALL TEST METRICS")
print("-" * 165)

display(
    OVERALL_METRICS
)


print()
print("OVERALL PAIRWISE GAINS")
print("-" * 165)

display(
    PAIRWISE_OVERALL
)


print()
print("BY STATE GAP — METRICS")
print("-" * 165)

display(
    GAP_METRICS
)


print()
print("BY STATE GAP — PAIRWISE GAINS")
print("-" * 165)

display(
    GAP_GAINS
)


print()
print("BY MODEL GROUP")
print("-" * 165)

display(
    GROUP_METRICS
)


print()
print("PAIRED ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 165)

display(
    BOOTSTRAP_SUMMARY
)


print()
print("V6.1 WEIGHT REFERENCE")
print("-" * 165)

weight_reference_days = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


weight_reference = pd.DataFrame(
    {
        "gap_days":
            weight_reference_days.astype(
                int
            ),

        "OU_phi":
            np.exp(
                -
                weight_reference_days
                /
                TAU_V5
            ),
    }
)


weight_reference[
    "V61_weight"
] = (
    1.0
    -
    weight_reference[
        "OU_phi"
    ]
)


display(
    weight_reference
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{TEST_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{TEST_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{TEST_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 165)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"V6.1 weights within [0,1]      : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"TEST dates only                : "
    f"{TEST_DATE_PASS}"
)

print(
    f"No new fitted parameters       : "
    f"{NO_NEW_PARAMETERS_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Fresh/stale endpoint logic     : "
    f"{V61_ENDPOINT_LOGIC_PASS}"
)


print()
print("=" * 165)

if CELL_34_PASS:

    print(
        "CELL 34 STATUS: HB_2023 V6.1 UNTOUCHED TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 34 STATUS: HB_2023 V6.1 TEST EVALUATION REVIEW REQUIRED"
    )

print("=" * 165)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 25
# Original source SHA256: 07e9167cccf7b5ac497e8571a4e67f2c23686f5cffcb290028738b146091280a
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 35 — HB_2023 HISTORICAL EVIDENCE FREEZE + CROSS-FOLD REGISTRY ENTRY
# ==================================================================================================
#
# Purpose
# -------
# Persist the completed HB_2023 historical backtest as an immutable evidence package.
#
# NO model fitting.
# NO parameter changes.
# NO threshold tuning.
#
# This records exactly what HB_2023 showed so that the final V6.1 decision can
# compare HB_2023 / HB_2024 / HB_2025 without reconstructing notebook outputs.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_34_PASS",

    "HB2023_V61_OVERALL_METRICS",
    "HB2023_V61_PAIRWISE_OVERALL",
    "HB2023_V61_GAP_GAINS",
    "HB2023_V61_GROUP_METRICS",
    "HB2023_V61_BOOTSTRAP",

    "HB2023_V61_TEST_ATTEMPT_DIR",

    "HB2023_V5_TAU_DAYS",
    "HB2023_V6_MEAN_SPEC",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_34_PASS:

    raise RuntimeError(
        "Cell 34 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE EVIDENCE DIRECTORY
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2023"

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2023_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


EVIDENCE_ROOT = (
    HB2023_DIR
    /
    "fold_evidence"
)


EVIDENCE_DIR = (
    EVIDENCE_ROOT
    /
    ATTEMPT_ID
)


EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. COPY METRICS INTO STABLE TABLES
# --------------------------------------------------------------------------------------------------

OVERALL = (
    HB2023_V61_OVERALL_METRICS.copy()
)


PAIRWISE = (
    HB2023_V61_PAIRWISE_OVERALL.copy()
)


GAP_GAINS = (
    HB2023_V61_GAP_GAINS.copy()
)


GROUP_METRICS = (
    HB2023_V61_GROUP_METRICS.copy()
)


BOOTSTRAP = (
    HB2023_V61_BOOTSTRAP.copy()
)


# --------------------------------------------------------------------------------------------------
# 5. EXTRACT KEY EVIDENCE
# --------------------------------------------------------------------------------------------------

def get_pairwise(
    baseline,
    candidate,
):

    row = (
        PAIRWISE.loc[
            PAIRWISE[
                "baseline"
            ].eq(
                baseline
            )

            &

            PAIRWISE[
                "candidate"
            ].eq(
                candidate
            )
        ]
    )


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover {baseline} -> {candidate}."
        )


    return row.iloc[0]


def get_bootstrap(
    horizon,
    baseline,
    candidate,
):

    row = (
        BOOTSTRAP.loc[
            BOOTSTRAP[
                "horizon"
            ].eq(
                horizon
            )

            &

            BOOTSTRAP[
                "baseline"
            ].eq(
                baseline
            )

            &

            BOOTSTRAP[
                "candidate"
            ].eq(
                candidate
            )
        ]
    )


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover bootstrap "
            f"{horizon}: {baseline} -> {candidate}."
        )


    return row.iloc[0]


V5_TO_V61 = get_pairwise(
    "V5",
    "V6.1",
)


V6_TO_V61 = get_pairwise(
    "V6",
    "V6.1",
)


BOOT_ALL_V5_V61 = get_bootstrap(
    "ALL",
    "V5",
    "V6.1",
)


BOOT_FRESH_V5_V6 = get_bootstrap(
    "01-30",
    "V5",
    "V6",
)


BOOT_FRESH_V5_V61 = get_bootstrap(
    "01-30",
    "V5",
    "V6.1",
)


BOOT_31_60_V5_V61 = get_bootstrap(
    "31-60",
    "V5",
    "V6.1",
)


BOOT_61_120_V5_V61 = get_bootstrap(
    "61-120",
    "V5",
    "V6.1",
)


BOOT_121_365_V5_V61 = get_bootstrap(
    "121-365",
    "V5",
    "V6.1",
)


BOOT_366_PLUS_V5_V61 = get_bootstrap(
    "366+",
    "V5",
    "V6.1",
)


# --------------------------------------------------------------------------------------------------
# 6. MODEL METRIC LOOKUP
# --------------------------------------------------------------------------------------------------

overall_lookup = (
    OVERALL.set_index(
        "model"
    )
)


V5_MAE = float(
    overall_lookup.loc[
        "V5",
        "MAE",
    ]
)


V6_MAE = float(
    overall_lookup.loc[
        "V6",
        "MAE",
    ]
)


V61_MAE = float(
    overall_lookup.loc[
        "V6.1",
        "MAE",
    ]
)


V5_RMSE = float(
    overall_lookup.loc[
        "V5",
        "RMSE",
    ]
)


V6_RMSE = float(
    overall_lookup.loc[
        "V6",
        "RMSE",
    ]
)


V61_RMSE = float(
    overall_lookup.loc[
        "V6.1",
        "RMSE",
    ]
)


V5_BIAS = float(
    overall_lookup.loc[
        "V5",
        "Bias",
    ]
)


V6_BIAS = float(
    overall_lookup.loc[
        "V6",
        "Bias",
    ]
)


V61_BIAS = float(
    overall_lookup.loc[
        "V6.1",
        "Bias",
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. DESCRIPTIVE EVIDENCE FLAGS
# --------------------------------------------------------------------------------------------------
#
# These are NOT release thresholds.
#
# They simply record whether the observed bootstrap interval is entirely
# above/below zero for this historical fold.
#
# --------------------------------------------------------------------------------------------------

EVIDENCE_FLAGS = {

    "v61_overall_mae_better_than_v5":
        bool(
            V61_MAE
            <
            V5_MAE
        ),

    "v61_overall_mae_gain_ci_above_zero":
        bool(
            BOOT_ALL_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "full_v6_fresh_mae_worse_than_v5":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_gain_m"
            ]
            <
            0
        ),

    "full_v6_fresh_mae_ci_below_zero":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_CI_high"
            ]
            <
            0
        ),

    "v61_fresh_mae_better_than_v5":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_gain_m"
            ]
            >
            0
        ),

    "v61_fresh_mae_gain_ci_above_zero":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_31_60_mae_gain_ci_above_zero":
        bool(
            BOOT_31_60_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_61_120_mae_gain_ci_above_zero":
        bool(
            BOOT_61_120_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_121_365_mae_gain_ci_above_zero":
        bool(
            BOOT_121_365_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_366_plus_mae_gain_ci_above_zero":
        bool(
            BOOT_366_PLUS_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),
}


# --------------------------------------------------------------------------------------------------
# 8. COMPACT CROSS-FOLD SUMMARY ROW
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "test_start":
                pd.Timestamp(
                    "2023-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2023-08-30"
                ),

            "test_rows":
                int(
                    OVERALL.loc[
                        OVERALL[
                            "model"
                        ].eq(
                            "V5"
                        ),
                        "n",
                    ]
                    .iloc[0]
                ),

            "test_athletes":
                int(
                    OVERALL.loc[
                        OVERALL[
                            "model"
                        ].eq(
                            "V5"
                        ),
                        "athletes",
                    ]
                    .iloc[0]
                ),

            "v5_tau_days":
                float(
                    HB2023_V5_TAU_DAYS
                ),

            "v6_persistence_model":
                HB2023_V6_MEAN_SPEC[
                    "selected_persistence_model"
                ],

            "v6_offset_model":
                HB2023_V6_MEAN_SPEC[
                    "selected_offset_model"
                ],

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V61_MAE":
                V61_MAE,

            "V5_to_V61_MAE_gain_m":
                float(
                    V5_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V61_MAE_CI_low":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V61_MAE_CI_high":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V61_RMSE":
                V61_RMSE,

            "V5_to_V61_RMSE_gain_m":
                float(
                    V5_TO_V61[
                        "RMSE_gain_m"
                    ]
                ),

            "V5_Bias":
                V5_BIAS,

            "V6_Bias":
                V6_BIAS,

            "V61_Bias":
                V61_BIAS,

            "fresh_V5_to_V6_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V61_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V61_CI_low":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V61_CI_high":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. SAVE ARTIFACTS
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY_PATH = (
    EVIDENCE_DIR
    /
    "fold_summary.parquet"
)


EVIDENCE_JSON_PATH = (
    EVIDENCE_DIR
    /
    "fold_evidence.json"
)


FOLD_SUMMARY.to_parquet(
    FOLD_SUMMARY_PATH,
    index=False,
)


# Parent Cell 34 audit.
CELL34_AUDIT_PATH = (
    Path(
        HB2023_V61_TEST_ATTEMPT_DIR
    )
    /
    "v61_test_evaluation_audit.json"
)


EVIDENCE_SPEC = {
    "evidence_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        "EVIDENCE_FROZEN",

    "purpose":
        (
            "Immutable fold-level evidence record for eventual "
            "cross-fold V6.1 architecture assessment."
        ),

    "decision_note":
        (
            "This artifact does not promote V6.1 and does not define "
            "release thresholds. It records HB_2023 evidence only."
        ),

    "overall": {
        "V5": {
            "MAE":
                V5_MAE,

            "RMSE":
                V5_RMSE,

            "Bias":
                V5_BIAS,
        },

        "V6": {
            "MAE":
                V6_MAE,

            "RMSE":
                V6_RMSE,

            "Bias":
                V6_BIAS,
        },

        "V6.1": {
            "MAE":
                V61_MAE,

            "RMSE":
                V61_RMSE,

            "Bias":
                V61_BIAS,
        },
    },

    "v5_to_v61_overall": {
        "MAE_gain_m":
            float(
                BOOT_ALL_V5_V61[
                    "MAE_gain_m"
                ]
            ),

        "MAE_CI_95": [
            float(
                BOOT_ALL_V5_V61[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_ALL_V5_V61[
                    "MAE_CI_high"
                ]
            ),
        ],

        "RMSE_gain_m":
            float(
                BOOT_ALL_V5_V61[
                    "RMSE_gain_m"
                ]
            ),

        "RMSE_CI_95": [
            float(
                BOOT_ALL_V5_V61[
                    "RMSE_CI_low"
                ]
            ),

            float(
                BOOT_ALL_V5_V61[
                    "RMSE_CI_high"
                ]
            ),
        ],
    },

    "fresh_01_30_days": {
        "V5_to_V6_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V6[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V61_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V61_MAE_CI_95": [
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_high"
                ]
            ),
        ],
    },

    "evidence_flags":
        EVIDENCE_FLAGS,

    "v5_tau_days":
        float(
            HB2023_V5_TAU_DAYS
        ),

    "v6_mean_spec":
        HB2023_V6_MEAN_SPEC,

    "source_test_evaluation": {
        "directory":
            str(
                HB2023_V61_TEST_ATTEMPT_DIR
            ),

        "audit_path":
            str(
                CELL34_AUDIT_PATH
            ),

        "audit_sha256":
            (
                sha256_file(
                    CELL34_AUDIT_PATH
                )
                if CELL34_AUDIT_PATH.exists()
                else
                None
            ),
    },

    "artifacts": {
        "fold_summary": {
            "path":
                str(
                    FOLD_SUMMARY_PATH
                ),

            "sha256":
                sha256_file(
                    FOLD_SUMMARY_PATH
                ),
        },
    },
}


atomic_json_write(
    EVIDENCE_SPEC,
    EVIDENCE_JSON_PATH,
)


EVIDENCE_SHA256 = sha256_file(
    EVIDENCE_JSON_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. LATEST EVIDENCE POINTER
# --------------------------------------------------------------------------------------------------

LATEST_EVIDENCE_POINTER = (
    EVIDENCE_ROOT
    /
    "LATEST_FOLD_EVIDENCE.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            "EVIDENCE_FROZEN",

        "evidence_directory":
            str(
                EVIDENCE_DIR
            ),

        "evidence_json":
            str(
                EVIDENCE_JSON_PATH
            ),

        "evidence_sha256":
            EVIDENCE_SHA256,

        "fold_summary":
            str(
                FOLD_SUMMARY_PATH
            ),

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_EVIDENCE_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 11. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "fold_evidence"
] = {
    "status":
        "EVIDENCE_FROZEN",

    "attempt_id":
        ATTEMPT_ID,

    "evidence_directory":
        str(
            EVIDENCE_DIR
        ),

    "evidence_json":
        str(
            EVIDENCE_JSON_PATH
        ),

    "evidence_sha256":
        EVIDENCE_SHA256,

    "fold_summary":
        str(
            FOLD_SUMMARY_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = "HB_2023_HISTORICAL_EVIDENCE_FROZEN"


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 12. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_FOLD_SUMMARY = (
    FOLD_SUMMARY
)


HB2023_FOLD_EVIDENCE = (
    EVIDENCE_SPEC
)


HB2023_FOLD_EVIDENCE_PATH = (
    EVIDENCE_JSON_PATH
)


NOTEBOOK_2_CELL_35_PASS = True


# --------------------------------------------------------------------------------------------------
# 13. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2023 — HISTORICAL EVIDENCE FREEZE")
print("=" * 165)


print()
print("CROSS-FOLD SUMMARY ROW")
print("-" * 165)

display(
    FOLD_SUMMARY
)


print()
print("EVIDENCE FLAGS")
print("-" * 165)

display(
    pd.DataFrame(
        [
            {
                "evidence":
                    key,

                "value":
                    value,
            }

            for key, value
            in EVIDENCE_FLAGS.items()
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable evidence directory   : "
    f"{EVIDENCE_DIR}"
)

print(
    f"Evidence JSON                  : "
    f"{EVIDENCE_JSON_PATH}"
)

print(
    f"Evidence SHA256                : "
    f"{EVIDENCE_SHA256}"
)

print(
    f"Latest evidence pointer        : "
    f"{LATEST_EVIDENCE_POINTER}"
)


print()
print("=" * 165)
print(
    "CELL 35 STATUS: HB_2023 HISTORICAL EVIDENCE FROZEN"
)
print("=" * 165)



HB_2023 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY

OVERALL POINT PERFORMANCE
----------------------------------------------------------------------------------------------------------------------------------------------------------------


,split,n,athletes,V4_MAE,V5_MAE,V4_to_V5_MAE_gain_m,V4_RMSE,V5_RMSE,V4_to_V5_RMSE_gain_m,V5_Bias
0,VALIDATION,3092,1676,2.754150,2.600320,0.153830,3.646641,3.450355,0.196287,0.012977
1,TEST,10110,3539,2.924484,2.552473,0.372011,3.846364,3.398251,0.448113,0.058338



POINT METRICS
----------------------------------------------------------------------------------------------------------------------------------------------------------------


,model,n,athletes,MAE,RMSE,Bias,split
0,V4,3092,1676,2.754150,3.646641,-0.009108,VALIDATION
1,V5,3092,1676,2.600320,3.450355,0.012977,VALIDATION
2,V4,10110,3539,2.924484,3.846364,0.154905,TEST
3,V5,10110,3539,2.552473,3.398251,0.058338,TEST



BY STATE GAP
----------------------------------------------------------------------------------------------------------------------------------------------------------------


...


AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T132535_765044Z
Immutable attempt directory    : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v61_test_evaluation/20260922T132535_765044Z
Audit manifest                 : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/v61_test_evaluation/20260922T132535_765044Z/v61_test_evaluation_audit.json
Audit SHA256                   : 5cf6815c010702f085fc654323a1d87ef622cdad13ee1e1ffc1ed164987c82e7
Git commit                     : a7cb73d264fa6eb35402298c3bf7f921b1c8641e

QA
----------------------------------------------------------------------------------------------------------------------------

,fold,test_start,test_end,test_rows,test_athletes,v5_tau_days,v6_persistence_model,v6_offset_model,V5_MAE,V6_MAE,...,V6_RMSE,V61_RMSE,V5_to_V61_RMSE_gain_m,V5_Bias,V6_Bias,V61_Bias,fresh_V5_to_V6_MAE_gain_m,fresh_V5_to_V61_MAE_gain_m,fresh_V5_to_V61_CI_low,fresh_V5_to_V61_CI_high
0,HB_2023,2023-05-01,2023-08-30,10110,3539,87.924374,M0_CONST_CONST,GROUP,2.552473,2.55015,...,3.371475,3.369035,0.029216,0.058338,0.259753,0.056991,-0.03484,0.004596,0.001344,0.00794



EVIDENCE FLAGS
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,evidence,value
0,v61_overall_mae_better_than_v5,True
1,v61_overall_mae_gain_ci_above_zero,True
2,full_v6_fresh_mae_worse_than_v5,True
3,full_v6_fresh_mae_ci_below_zero,True
4,v61_fresh_mae_better_than_v5,True
5,v61_fresh_mae_gain_ci_above_zero,True
6,v61_31_60_mae_gain_ci_above_zero,True
7,v61_61_120_mae_gain_ci_above_zero,True
8,v61_121_365_mae_gain_ci_above_zero,True
9,v61_366_plus_mae_gain_ci_above_zero,True



AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T132538_121984Z
Immutable evidence directory   : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/fold_evidence/20260922T132538_121984Z
Evidence JSON                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/fold_evidence/20260922T132538_121984Z/fold_evidence.json
Evidence SHA256                : f4cea6885da7c01c45eccdded177d2ddb7786babdab30da390121e8839f54c68
Latest evidence pointer        : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2023/fold_evidence/LATEST_FOLD_EVIDENCE.json

CELL 35 STATUS: 

## Stage 13 — HB_2024 model reconstruction

Rebuild the 2024 historical V4/V5 training state and persist immutable fold-specific audit artifacts.

**Legacy source cells represented:** 26, 27, 28, 29  
**Execution rule:** run only after the preceding stage has completed successfully.


In [14]:
# ==================================================================================================
# REFACTORED STAGE 13 — HB_2024 model reconstruction
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 26
# Original source SHA256: 2ca53d0f681f096a9e759cdb2c09d6a651ce4ff82bfea472ce4543492cb1d2f7
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 36 — HB_2024 HISTORICAL V4 REFIT PREPARATION + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# Historical fold:
#
#   TRAIN       <= 2023-12-31
#   VALIDATION   2024-01-01 -> 2024-04-30
#   TEST         2024-05-01 -> 2024-08-30
#
#
# Purpose
# -------
# Recreate the exact leakage-safe V4 modelling inputs for HB_2024:
#
#   • established-athlete VALIDATION and TEST
#   • production V4 feature definitions
#   • TRAIN-only athlete index
#   • TRAIN-only 7-column cubic age spline
#   • distinct-date chronology
#
#
# NO fitting occurs here.
#
#
# Audit/versioning
# ----------------
# Every execution creates a new immutable preparation attempt.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd

from patsy import (
    dmatrix,
    build_design_matrices,
)


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_35_PASS",

    "HISTORICAL_BACKTEST_BASE",
    "HISTORICAL_BACKTEST_FOLDS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_35_PASS:

    raise RuntimeError(
        "Cell 35 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 3. FOLD SPECIFICATION
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_36_HB2024_PREPARATION"

PREPARATION_SCHEMA_VERSION = "1.0.0"


fold_match = (
    HISTORICAL_BACKTEST_FOLDS.loc[
        HISTORICAL_BACKTEST_FOLDS[
            "fold"
        ]
        .eq(
            FOLD_NAME
        )
    ]
)


if len(
    fold_match
) != 1:

    raise RuntimeError(
        f"Could not uniquely identify {FOLD_NAME}."
    )


FOLD_SPEC = (
    fold_match.iloc[
        0
    ]
)


TRAIN_END = pd.Timestamp(
    FOLD_SPEC[
        "train_end"
    ]
)


VALIDATION_START = pd.Timestamp(
    FOLD_SPEC[
        "validation_start"
    ]
)


VALIDATION_END = pd.Timestamp(
    FOLD_SPEC[
        "validation_end"
    ]
)


TEST_START = pd.Timestamp(
    FOLD_SPEC[
        "test_start"
    ]
)


TEST_END = pd.Timestamp(
    FOLD_SPEC[
        "test_end"
    ]
)


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2024_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


PREP_ROOT = (
    HB2024_DIR
    /
    "preparation"
)


PREP_ATTEMPT_DIR = (
    PREP_ROOT
    /
    ATTEMPT_ID
)


PREP_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. MATERIALIZE TEMPORAL SPLITS
# --------------------------------------------------------------------------------------------------

BASE = (
    HISTORICAL_BACKTEST_BASE.copy()
)


TRAIN = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    ]
    .copy()
)


VALIDATION_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            VALIDATION_START,
            VALIDATION_END,
            inclusive="both",
        )
    ]
    .copy()
)


TEST_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            TEST_START,
            TEST_END,
            inclusive="both",
        )
    ]
    .copy()
)


TRAIN_ATHLETES = set(
    TRAIN[
        "aaAthleteId"
    ]
    .unique()
)


VALIDATION_ALL[
    "athlete_seen_in_training"
] = (
    VALIDATION_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


TEST_ALL[
    "athlete_seen_in_training"
] = (
    TEST_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


VALIDATION = (
    VALIDATION_ALL.loc[
        VALIDATION_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


TEST = (
    TEST_ALL.loc[
        TEST_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 6. V4 FEATURE DEFINITIONS — UNCHANGED FROM HB_2023 / PRODUCTION
# --------------------------------------------------------------------------------------------------

SEX_INDEX = {
    "M": 0,
    "W": 1,
}


GROUP_INDEX = {
    "M_700": 0,
    "M_800": 1,
    "W_500": 2,
    "W_600": 3,
}


YOUTH_GROUPS = {
    "M_700",
    "W_500",
}


AGE_CENTER = 18.0
AGE_SCALE = 5.0


def add_v4_features(
    df,
):

    out = df.copy()


    out[
        "sex_group"
    ] = (
        out[
            "model_group"
        ]
        .astype(str)
        .str[
            0
        ]
    )


    out[
        "sex_idx"
    ] = (
        out[
            "sex_group"
        ]
        .map(
            SEX_INDEX
        )
    )


    out[
        "group_idx"
    ] = (
        out[
            "model_group"
        ]
        .map(
            GROUP_INDEX
        )
    )


    out[
        "is_youth_implement"
    ] = (
        out[
            "model_group"
        ]
        .isin(
            YOUTH_GROUPS
        )
        .astype(float)
    )


    out[
        "age_scaled"
    ] = (
        (
            out[
                "age_years"
            ]
            .astype(float)

            -

            AGE_CENTER
        )

        /

        AGE_SCALE
    )


    return out


TRAIN = add_v4_features(
    TRAIN
)


VALIDATION = add_v4_features(
    VALIDATION
)


TEST = add_v4_features(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 7. TRAIN-ONLY ATHLETE INDEX
# --------------------------------------------------------------------------------------------------

ATHLETE_LEVELS = (
    pd.Series(
        TRAIN[
            "aaAthleteId"
        ]
        .unique()
    )
    .sort_values()
    .tolist()
)


ATHLETE_INDEX = {
    athlete_id:
        idx

    for idx, athlete_id
    in enumerate(
        ATHLETE_LEVELS
    )
}


for df in [
    TRAIN,
    VALIDATION,
    TEST,
]:

    df[
        "athlete_idx"
    ] = (
        df[
            "aaAthleteId"
        ]
        .map(
            ATHLETE_INDEX
        )
    )


if VALIDATION[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "HB_2024 established VALIDATION athlete missing TRAIN index."
    )


if TEST[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "HB_2024 established TEST athlete missing TRAIN index."
    )


TRAIN[
    "athlete_idx"
] = (
    TRAIN[
        "athlete_idx"
    ]
    .astype(int)
)


VALIDATION[
    "athlete_idx"
] = (
    VALIDATION[
        "athlete_idx"
    ]
    .astype(int)
)


TEST[
    "athlete_idx"
] = (
    TEST[
        "athlete_idx"
    ]
    .astype(int)
)


# --------------------------------------------------------------------------------------------------
# 8. TRAIN-ONLY CUBIC SPLINE
# --------------------------------------------------------------------------------------------------

SPLINE_FORMULA = (
    "bs("
    "age, "
    "df=7, "
    "degree=3, "
    "include_intercept=False, "
    "lower_bound=13, "
    "upper_bound=35"
    ") - 1"
)


B_TRAIN_RAW_DF = dmatrix(
    SPLINE_FORMULA,

    {
        "age":
            TRAIN[
                "age_years"
            ]
            .to_numpy(
                dtype=float
            )
    },

    return_type="dataframe",
)


SPLINE_DESIGN_INFO = (
    B_TRAIN_RAW_DF.design_info
)


B_TRAIN_RAW = np.asarray(
    B_TRAIN_RAW_DF,
    dtype=float,
)


B_VALIDATION_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                VALIDATION[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


B_TEST_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                TEST[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


# TRAIN-only centering.
SPLINE_COLUMN_MEANS = (
    B_TRAIN_RAW.mean(
        axis=0
    )
)


B_TRAIN = (
    B_TRAIN_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_VALIDATION = (
    B_VALIDATION_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_TEST = (
    B_TEST_RAW
    -
    SPLINE_COLUMN_MEANS
)


# Audit-friendly knot representation.
SPLINE_INTERNAL_KNOTS = np.quantile(
    TRAIN[
        "age_years"
    ]
    .to_numpy(
        dtype=float
    ),

    [
        0.20,
        0.40,
        0.60,
        0.80,
    ],
)


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(
        7
    )
]


for i, column in enumerate(
    SPLINE_COLUMNS
):

    TRAIN[
        column
    ] = (
        B_TRAIN[
            :,
            i
        ]
    )


    VALIDATION[
        column
    ] = (
        B_VALIDATION[
            :,
            i
        ]
    )


    TEST[
        column
    ] = (
        B_TEST[
            :,
            i
        ]
    )


# --------------------------------------------------------------------------------------------------
# 9. LEAKAGE-SAFE DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------

CHRONOLOGY = (
    BASE.loc[
        (
            BASE[
                "aaAthleteId"
            ]
            .isin(
                TRAIN_ATHLETES
            )
        )

        &

        (
            BASE[
                "result_date_dt"
            ]
            <=
            TEST_END
        )
    ][
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


CHRONOLOGY[
    "previous_distinct_date"
] = (
    CHRONOLOGY.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(
        1
    )
)


CHRONOLOGY[
    "state_gap_days"
] = (
    CHRONOLOGY[
        "result_date_dt"
    ]

    -

    CHRONOLOGY[
        "previous_distinct_date"
    ]
).dt.days


def attach_chronology(
    df,
):

    return df.merge(
        CHRONOLOGY,

        on=[
            "aaAthleteId",
            "result_date_dt",
        ],

        how="left",

        validate="many_to_one",
    )


TRAIN = attach_chronology(
    TRAIN
)


VALIDATION = attach_chronology(
    VALIDATION
)


TEST = attach_chronology(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 10. QA
# --------------------------------------------------------------------------------------------------

TEMPORAL_SPLIT_PASS = bool(

    (
        TRAIN[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    )
    .all()

    and

    VALIDATION[
        "result_date_dt"
    ]
    .between(
        VALIDATION_START,
        VALIDATION_END,
        inclusive="both",
    )
    .all()

    and

    TEST[
        "result_date_dt"
    ]
    .between(
        TEST_START,
        TEST_END,
        inclusive="both",
    )
    .all()
)


ATHLETE_INDEX_PASS = bool(

    TRAIN[
        "athlete_idx"
    ]
    .notna()
    .all()

    and

    VALIDATION[
        "athlete_idx"
    ]
    .notna()
    .all()

    and

    TEST[
        "athlete_idx"
    ]
    .notna()
    .all()
)


FEATURE_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "sex_idx",
                "group_idx",
                "is_youth_implement",
                "age_scaled",
                "athlete_idx",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


SPLINE_PASS = bool(

    B_TRAIN.shape[
        1
    ]
    ==
    7

    and

    B_VALIDATION.shape[
        1
    ]
    ==
    7

    and

    B_TEST.shape[
        1
    ]
    ==
    7

    and

    np.isfinite(
        B_TRAIN
    )
    .all()

    and

    np.isfinite(
        B_VALIDATION
    )
    .all()

    and

    np.isfinite(
        B_TEST
    )
    .all()

    and

    np.max(
        np.abs(
            B_TRAIN.mean(
                axis=0
            )
        )
    )
    <
    1e-10
)


VALIDATION_CHRONOLOGY_PASS = bool(

    VALIDATION[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        VALIDATION[
            "state_gap_days"
        ]
        >
        0
    )
    .all()
)


TEST_CHRONOLOGY_PASS = bool(

    TEST[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        TEST[
            "state_gap_days"
        ]
        >
        0
    )
    .all()
)


NO_FUTURE_TRAIN_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    TRAIN_END
)


CELL_36_PASS = bool(
    TEMPORAL_SPLIT_PASS
    and
    ATHLETE_INDEX_PASS
    and
    FEATURE_FINITE_PASS
    and
    SPLINE_PASS
    and
    VALIDATION_CHRONOLOGY_PASS
    and
    TEST_CHRONOLOGY_PASS
    and
    NO_FUTURE_TRAIN_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 11. SAVE IMMUTABLE PREPARATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

TRAIN_PATH = (
    PREP_ATTEMPT_DIR
    /
    "train_prepared.parquet"
)


VALIDATION_PATH = (
    PREP_ATTEMPT_DIR
    /
    "validation_established_prepared.parquet"
)


TEST_PATH = (
    PREP_ATTEMPT_DIR
    /
    "test_established_prepared.parquet"
)


PREP_SPEC_PATH = (
    PREP_ATTEMPT_DIR
    /
    "preparation_spec.json"
)


PREP_AUDIT_PATH = (
    PREP_ATTEMPT_DIR
    /
    "preparation_audit.json"
)


TRAIN.to_parquet(
    TRAIN_PATH,
    index=False,
)


VALIDATION.to_parquet(
    VALIDATION_PATH,
    index=False,
)


TEST.to_parquet(
    TEST_PATH,
    index=False,
)


PREP_SPEC = {
    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "schema_version":
        PREPARATION_SCHEMA_VERSION,

    "status":
        (
            "PASSED"
            if CELL_36_PASS
            else
            "FAILED"
        ),

    "train_end":
        TRAIN_END.isoformat(),

    "validation_start":
        VALIDATION_START.isoformat(),

    "validation_end":
        VALIDATION_END.isoformat(),

    "test_start":
        TEST_START.isoformat(),

    "test_end":
        TEST_END.isoformat(),

    "train_rows":
        int(
            len(
                TRAIN
            )
        ),

    "train_athletes":
        int(
            TRAIN[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "validation_rows":
        int(
            len(
                VALIDATION
            )
        ),

    "validation_athletes":
        int(
            VALIDATION[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "test_rows":
        int(
            len(
                TEST
            )
        ),

    "test_athletes":
        int(
            TEST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "age_center":
        AGE_CENTER,

    "age_scale":
        AGE_SCALE,

    "sex_index":
        SEX_INDEX,

    "group_index":
        GROUP_INDEX,

    "youth_groups":
        sorted(
            YOUTH_GROUPS
        ),

    "spline_formula":
        SPLINE_FORMULA,

    "spline_internal_knots":
        [
            float(
                x
            )
            for x in SPLINE_INTERNAL_KNOTS
        ],

    "spline_column_means":
        [
            float(
                x
            )
            for x in SPLINE_COLUMN_MEANS
        ],

    "spline_columns":
        SPLINE_COLUMNS,

    "athlete_index_count":
        int(
            len(
                ATHLETE_INDEX
            )
        ),

    "data_vintage_note":
        (
            "Historical fold reconstructed from the current canonical snapshot "
            "and restricted by event/result date. Later corrections to historical "
            "records may therefore be present; no future-dated performances are used."
        ),
}


atomic_json_write(
    PREP_SPEC,
    PREP_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 12. ARTIFACT HASHES
# --------------------------------------------------------------------------------------------------

OUTPUT_ARTIFACTS = {
    "train_prepared": {
        "path":
            str(
                TRAIN_PATH
            ),

        "sha256":
            sha256_file(
                TRAIN_PATH
            ),
    },

    "validation_prepared": {
        "path":
            str(
                VALIDATION_PATH
            ),

        "sha256":
            sha256_file(
                VALIDATION_PATH
            ),
    },

    "test_prepared": {
        "path":
            str(
                TEST_PATH
            ),

        "sha256":
            sha256_file(
                TEST_PATH
            ),
    },

    "preparation_spec": {
        "path":
            str(
                PREP_SPEC_PATH
            ),

        "sha256":
            sha256_file(
                PREP_SPEC_PATH
            ),
    },
}


PARENT_ARTIFACTS = {}


if "CANONICAL_PATH" in globals():

    canonical_path = Path(
        CANONICAL_PATH
    )


    PARENT_ARTIFACTS[
        "active_canonical"
    ] = {
        "path":
            str(
                canonical_path
            ),

        "exists":
            canonical_path.exists(),

        "sha256":
            (
                sha256_file(
                    canonical_path
                )
                if canonical_path.exists()
                else
                None
            ),
    }


if "HB2023_FOLD_EVIDENCE_PATH" in globals():

    previous_evidence_path = Path(
        HB2023_FOLD_EVIDENCE_PATH
    )


    PARENT_ARTIFACTS[
        "previous_fold_evidence"
    ] = {
        "path":
            str(
                previous_evidence_path
            ),

        "exists":
            previous_evidence_path.exists(),

        "sha256":
            (
                sha256_file(
                    previous_evidence_path
                )
                if previous_evidence_path.exists()
                else
                None
            ),
    }


# --------------------------------------------------------------------------------------------------
# 13. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

PREP_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "HISTORICAL_FOLD_PREPARATION",

    "stage_version":
        PREPARATION_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_36_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Prepare HB_2024 as the second independent historical "
            "backtest of the frozen V6.1 architecture."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "v61_formula_changed_from_HB_2023":
        False,

    "fold_spec":
        PREP_SPEC,

    "qa": {
        "temporal_split":
            TEMPORAL_SPLIT_PASS,

        "train_only_athlete_index":
            ATHLETE_INDEX_PASS,

        "finite_features":
            FEATURE_FINITE_PASS,

        "spline":
            SPLINE_PASS,

        "validation_chronology":
            VALIDATION_CHRONOLOGY_PASS,

        "test_chronology":
            TEST_CHRONOLOGY_PASS,

        "no_future_train_data":
            NO_FUTURE_TRAIN_DATA_PASS,

        "cell_pass":
            CELL_36_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,
}


atomic_json_write(
    PREP_AUDIT,
    PREP_AUDIT_PATH,
)


PREP_AUDIT_SHA256 = (
    sha256_file(
        PREP_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 14. LATEST PREPARATION POINTER
# --------------------------------------------------------------------------------------------------

LATEST_PREP_POINTER_PATH = (
    PREP_ROOT
    /
    "LATEST_PREPARATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_36_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                PREP_ATTEMPT_DIR
            ),

        "train_prepared":
            str(
                TRAIN_PATH
            ),

        "validation_prepared":
            str(
                VALIDATION_PATH
            ),

        "test_prepared":
            str(
                TEST_PATH
            ),

        "audit_manifest":
            str(
                PREP_AUDIT_PATH
            ),

        "audit_sha256":
            PREP_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_PREP_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 15. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
].setdefault(
    FOLD_NAME,
    {}
)


TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "preparation"
] = {
    "status":
        (
            "PASSED"
            if CELL_36_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            PREP_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_PREP_POINTER_PATH
        ),

    "audit_manifest":
        str(
            PREP_AUDIT_PATH
        ),

    "audit_sha256":
        PREP_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_PREPARED_FOR_V4_REFIT"
    if CELL_36_PASS
    else
    "HB_2024_PREPARATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 16. GLOBALS FOR NEXT CELL
# --------------------------------------------------------------------------------------------------

HB2024_TRAIN = TRAIN

HB2024_VALIDATION = VALIDATION

HB2024_TEST = TEST


HB2024_ATHLETE_LEVELS = (
    ATHLETE_LEVELS
)


HB2024_ATHLETE_INDEX = (
    ATHLETE_INDEX
)


HB2024_SPLINE_DESIGN_INFO = (
    SPLINE_DESIGN_INFO
)


HB2024_SPLINE_COLUMN_MEANS = (
    SPLINE_COLUMN_MEANS
)


HB2024_SPLINE_INTERNAL_KNOTS = (
    SPLINE_INTERNAL_KNOTS
)


HB2024_B_TRAIN = (
    B_TRAIN
)


HB2024_B_VALIDATION = (
    B_VALIDATION
)


HB2024_B_TEST = (
    B_TEST
)


HB2024_PREP_ATTEMPT_DIR = (
    PREP_ATTEMPT_DIR
)


HB2024_PREP_AUDIT_PATH = (
    PREP_AUDIT_PATH
)


NOTEBOOK_2_CELL_36_PASS = (
    CELL_36_PASS
)


# --------------------------------------------------------------------------------------------------
# 17. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2024 — HISTORICAL V4 REFIT PREPARATION")
print("=" * 160)


print()
print("TEMPORAL SPLIT")
print("-" * 160)

print(
    f"TRAIN       : <= {TRAIN_END.date()}"
)

print(
    f"VALIDATION  : "
    f"{VALIDATION_START.date()} -> {VALIDATION_END.date()}"
)

print(
    f"TEST        : "
    f"{TEST_START.date()} -> {TEST_END.date()}"
)


print()
print("POPULATION")
print("-" * 160)

print(
    f"TRAIN rows / athletes          : "
    f"{len(TRAIN):,} / "
    f"{TRAIN['aaAthleteId'].nunique():,}"
)

print(
    f"VALIDATION established         : "
    f"{len(VALIDATION):,} / "
    f"{VALIDATION['aaAthleteId'].nunique():,}"
)

print(
    f"TEST established               : "
    f"{len(TEST):,} / "
    f"{TEST['aaAthleteId'].nunique():,}"
)


print()
print("TRAIN-ONLY SPLINE")
print("-" * 160)

print(
    f"Basis shape                    : "
    f"{B_TRAIN.shape}"
)

print(
    "Internal knots                : "
    +
    np.array2string(
        SPLINE_INTERNAL_KNOTS,
        precision=10,
    )
)

print(
    "Column means                  : "
    +
    np.array2string(
        SPLINE_COLUMN_MEANS,
        precision=10,
    )
)

print(
    "Centered TRAIN mean           : "
    +
    np.array2string(
        B_TRAIN.mean(
            axis=0
        ),
        precision=3,
        suppress_small=True,
    )
)


print()
print("STATE-GAP SUMMARY")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "split":
                    "VALIDATION",

                "n":
                    len(
                        VALIDATION
                    ),

                "median_gap_days":
                    float(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .median()
                    ),

                "mean_gap_days":
                    float(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .mean()
                    ),

                "min_gap_days":
                    int(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .min()
                    ),

                "max_gap_days":
                    int(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .max()
                    ),
            },

            {
                "split":
                    "TEST",

                "n":
                    len(
                        TEST
                    ),

                "median_gap_days":
                    float(
                        TEST[
                            "state_gap_days"
                        ]
                        .median()
                    ),

                "mean_gap_days":
                    float(
                        TEST[
                            "state_gap_days"
                        ]
                        .mean()
                    ),

                "min_gap_days":
                    int(
                        TEST[
                            "state_gap_days"
                        ]
                        .min()
                    ),

                "max_gap_days":
                    int(
                        TEST[
                            "state_gap_days"
                        ]
                        .max()
                    ),
            },
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable preparation directory: "
    f"{PREP_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{PREP_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{PREP_AUDIT_SHA256}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Temporal split                 : "
    f"{TEMPORAL_SPLIT_PASS}"
)

print(
    f"TRAIN-only athlete index       : "
    f"{ATHLETE_INDEX_PASS}"
)

print(
    f"Finite V4 features             : "
    f"{FEATURE_FINITE_PASS}"
)

print(
    f"7-column TRAIN spline          : "
    f"{SPLINE_PASS}"
)

print(
    f"VALIDATION chronology          : "
    f"{VALIDATION_CHRONOLOGY_PASS}"
)

print(
    f"TEST chronology                : "
    f"{TEST_CHRONOLOGY_PASS}"
)

print(
    f"No future TRAIN data           : "
    f"{NO_FUTURE_TRAIN_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_36_PASS:

    print(
        "CELL 36 STATUS: HB_2024 HISTORICAL V4 PREPARATION PASSED"
    )

else:

    print(
        "CELL 36 STATUS: HB_2024 PREPARATION REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 27
# Original source SHA256: fd7cd9269a047342f69a2453d5ed40fede8894a3ae8b34246e8f9ff82f820e38
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 37 — HB_2024 HISTORICAL V4 BAYESIAN REFIT + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# TRAIN:
#       <= 2023-12-31
#
# Inference:
#       ADVI          = 40,000 iterations
#       posterior     = 2,000 draws
#       random seed   = 42
#
# Model specification is intentionally UNCHANGED from HB_2023.
#
# NO 2024 VALIDATION or TEST marks are used for fitting.
#
# ==================================================================================================

import gc
import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd

import pymc as pm
import pytensor
import pytensor.tensor as pt


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_36_PASS",

    "HB2024_TRAIN",
    "HB2024_B_TRAIN",

    "HB2024_ATHLETE_LEVELS",
    "HB2024_ATHLETE_INDEX",

    "HB2024_PREP_ATTEMPT_DIR",
    "HB2024_PREP_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_36_PASS:

    raise RuntimeError(
        "Cell 36 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED MODEL / INFERENCE SETTINGS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_37_HB2024_V4_REFIT"

STAGE_SCHEMA_VERSION = "1.0.0"


ADVI_ITERATIONS = 40_000
POSTERIOR_DRAWS = 2_000
RANDOM_SEED = 42


SEX_LABELS = [
    "M",
    "W",
]


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_SEX = 2
N_GROUP = 4
N_SPLINE = 7
N_SPLINE_DELTA = 6


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )


        return result.stdout.strip()


    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE FIT ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2024_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V4_ROOT = (
    HB2024_DIR
    /
    "v4_refit"
)


V4_ATTEMPT_DIR = (
    V4_ROOT
    /
    ATTEMPT_ID
)


V4_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. TRAIN ARRAYS
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2024_TRAIN.copy()
    .reset_index(
        drop=True
    )
)


B_TRAIN = np.asarray(
    HB2024_B_TRAIN,
    dtype=np.float64,
)


y_train = (
    TRAIN[
        "mark_numeric_m"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


sex_idx_train = (
    TRAIN[
        "sex_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


group_idx_train = (
    TRAIN[
        "group_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


athlete_idx_train = (
    TRAIN[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


youth_train = (
    TRAIN[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


age_scaled_train = (
    TRAIN[
        "age_scaled"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


N_ATHLETES = len(
    HB2024_ATHLETE_LEVELS
)


# --------------------------------------------------------------------------------------------------
# 6. PRE-FIT QA
# --------------------------------------------------------------------------------------------------

if B_TRAIN.shape != (
    len(
        TRAIN
    ),
    N_SPLINE,
):

    raise RuntimeError(
        f"Unexpected spline shape: {B_TRAIN.shape}"
    )


if not np.isfinite(
    y_train
).all():

    raise RuntimeError(
        "Non-finite TRAIN marks."
    )


if not np.isfinite(
    B_TRAIN
).all():

    raise RuntimeError(
        "Non-finite TRAIN spline."
    )


if (
    sex_idx_train.min() < 0
    or
    sex_idx_train.max() >= N_SEX
):

    raise RuntimeError(
        "Invalid sex index."
    )


if (
    group_idx_train.min() < 0
    or
    group_idx_train.max() >= N_GROUP
):

    raise RuntimeError(
        "Invalid group index."
    )


if (
    athlete_idx_train.min() < 0
    or
    athlete_idx_train.max() >= N_ATHLETES
):

    raise RuntimeError(
        "Invalid athlete index."
    )


if TRAIN[
    "result_date_dt"
].max() > pd.Timestamp(
    "2023-12-31"
):

    raise RuntimeError(
        "Future data detected in HB_2024 TRAIN."
    )


# --------------------------------------------------------------------------------------------------
# 7. MODEL COORDINATES
# --------------------------------------------------------------------------------------------------

coords = {
    "sex":
        SEX_LABELS,

    "group":
        GROUP_LABELS,

    "spline_basis":
        np.arange(
            N_SPLINE
        ),

    "spline_step":
        np.arange(
            N_SPLINE_DELTA
        ),

    "athlete":
        np.asarray(
            HB2024_ATHLETE_LEVELS
        ),

    "observation":
        np.arange(
            len(
                TRAIN
            )
        ),
}


# --------------------------------------------------------------------------------------------------
# 8. BUILD + FIT HISTORICAL V4
# --------------------------------------------------------------------------------------------------

with pm.Model(
    coords=coords
) as HB2024_V4_MODEL:

    # ----------------------------------------------------------------------------------------------
    # Model inputs
    # ----------------------------------------------------------------------------------------------

    sex_idx_data = pm.Data(
        "sex_idx_data",
        sex_idx_train,
        dims="observation",
    )


    group_idx_data = pm.Data(
        "group_idx_data",
        group_idx_train,
        dims="observation",
    )


    athlete_idx_data = pm.Data(
        "athlete_idx_data",
        athlete_idx_train,
        dims="observation",
    )


    youth_data = pm.Data(
        "youth_data",
        youth_train,
        dims="observation",
    )


    age_scaled_data = pm.Data(
        "age_scaled_data",
        age_scaled_train,
        dims="observation",
    )


    spline_data = pm.Data(
        "spline_data",
        B_TRAIN,
        dims=(
            "observation",
            "spline_basis",
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Sex population intercept
    # ----------------------------------------------------------------------------------------------

    alpha_sex = pm.Normal(
        "alpha_sex",
        mu=45.0,
        sigma=15.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Sex-specific spline random walk
    # ----------------------------------------------------------------------------------------------

    sigma_spline = pm.HalfNormal(
        "sigma_spline",
        sigma=2.0,
        dims="sex",
    )


    spline_start = pm.Normal(
        "spline_start",
        mu=0.0,
        sigma=3.0,
        dims="sex",
    )


    spline_delta = pm.Normal(
        "spline_delta",
        mu=0.0,
        sigma=sigma_spline[
            :,
            None
        ],
        dims=(
            "sex",
            "spline_step",
        ),
    )


    beta_spline = pm.Deterministic(
        "beta_spline",

        pt.concatenate(
            [
                spline_start[
                    :,
                    None
                ],

                spline_start[
                    :,
                    None
                ]
                +
                pt.cumsum(
                    spline_delta,
                    axis=1,
                ),
            ],
            axis=1,
        ),

        dims=(
            "sex",
            "spline_basis",
        ),
    )


    # ----------------------------------------------------------------------------------------------
    # Youth implement effect
    # ----------------------------------------------------------------------------------------------

    youth_implement_effect = pm.Normal(
        "youth_implement_effect",
        mu=0.0,
        sigma=8.0,
        dims="sex",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random intercept
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_intercept = pm.HalfNormal(
        "sigma_athlete_intercept",
        sigma=10.0,
    )


    athlete_intercept_z = pm.Normal(
        "athlete_intercept_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_intercept = pm.Deterministic(
        "athlete_intercept",

        sigma_athlete_intercept
        *
        athlete_intercept_z,

        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Athlete random age slope
    # ----------------------------------------------------------------------------------------------

    sigma_athlete_slope = pm.HalfNormal(
        "sigma_athlete_slope",
        sigma=4.0,
    )


    athlete_slope_z = pm.Normal(
        "athlete_slope_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_slope = pm.Deterministic(
        "athlete_slope",

        sigma_athlete_slope
        *
        athlete_slope_z,

        dims="athlete",
    )


    # ----------------------------------------------------------------------------------------------
    # Observation scale
    # ----------------------------------------------------------------------------------------------

    sigma_group = pm.HalfNormal(
        "sigma_group",
        sigma=5.0,
        dims="group",
    )


    # ----------------------------------------------------------------------------------------------
    # Student-t degrees of freedom
    # ----------------------------------------------------------------------------------------------

    nu_minus_two = pm.Exponential(
        "nu_minus_two",
        lam=0.1,
    )


    nu = pm.Deterministic(
        "nu",
        nu_minus_two
        +
        2.0,
    )


    # ----------------------------------------------------------------------------------------------
    # Structural mean
    # ----------------------------------------------------------------------------------------------

    spline_effect = (
        spline_data
        *
        beta_spline[
            sex_idx_data
        ]
    ).sum(
        axis=1
    )


    mu = (
        alpha_sex[
            sex_idx_data
        ]

        +

        spline_effect

        +

        youth_implement_effect[
            sex_idx_data
        ]
        *
        youth_data

        +

        athlete_intercept[
            athlete_idx_data
        ]

        +

        athlete_slope[
            athlete_idx_data
        ]
        *
        age_scaled_data
    )


    # ----------------------------------------------------------------------------------------------
    # Student-t likelihood
    # ----------------------------------------------------------------------------------------------

    observed = pm.StudentT(
        "observed",

        nu=nu,

        mu=mu,

        sigma=sigma_group[
            group_idx_data
        ],

        observed=y_train,

        dims="observation",
    )


    # ----------------------------------------------------------------------------------------------
    # ADVI
    # ----------------------------------------------------------------------------------------------

    HB2024_V4_APPROX = pm.fit(
        n=ADVI_ITERATIONS,
        method="advi",
        random_seed=RANDOM_SEED,
        progressbar=True,
    )


    # ----------------------------------------------------------------------------------------------
    # Posterior draws
    # ----------------------------------------------------------------------------------------------

    HB2024_V4_IDATA = (
        HB2024_V4_APPROX.sample(
            draws=POSTERIOR_DRAWS,
            random_seed=RANDOM_SEED,
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. ADVI HISTORY
# --------------------------------------------------------------------------------------------------

ADVI_HISTORY = np.asarray(
    HB2024_V4_APPROX.hist,
    dtype=float,
)


ADVI_FINITE_PASS = bool(
    len(
        ADVI_HISTORY
    )
    ==
    ADVI_ITERATIONS

    and

    np.isfinite(
        ADVI_HISTORY
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 10. POSTERIOR MEAN HELPER
# --------------------------------------------------------------------------------------------------

def posterior_mean(
    name,
):

    da = (
        HB2024_V4_IDATA
        .posterior[
            name
        ]
    )


    average_dims = [
        dim
        for dim in [
            "chain",
            "draw",
        ]
        if dim in da.dims
    ]


    return (
        da.mean(
            dim=average_dims
        )
        .values
    )


# --------------------------------------------------------------------------------------------------
# 11. REQUIRED POSTERIOR MEANS
# --------------------------------------------------------------------------------------------------

HB2024_V4_POSTERIOR_MEANS = {

    "alpha_sex":
        np.asarray(
            posterior_mean(
                "alpha_sex"
            ),
            dtype=float,
        ),

    "beta_spline":
        np.asarray(
            posterior_mean(
                "beta_spline"
            ),
            dtype=float,
        ),

    "youth_implement_effect":
        np.asarray(
            posterior_mean(
                "youth_implement_effect"
            ),
            dtype=float,
        ),

    "athlete_intercept":
        np.asarray(
            posterior_mean(
                "athlete_intercept"
            ),
            dtype=float,
        ),

    "athlete_slope":
        np.asarray(
            posterior_mean(
                "athlete_slope"
            ),
            dtype=float,
        ),

    "sigma_group":
        np.asarray(
            posterior_mean(
                "sigma_group"
            ),
            dtype=float,
        ),

    "nu":
        float(
            np.asarray(
                posterior_mean(
                    "nu"
                )
            )
        ),

    "sigma_athlete_intercept":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_intercept"
                )
            )
        ),

    "sigma_athlete_slope":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_slope"
                )
            )
        ),

    "sigma_spline":
        np.asarray(
            posterior_mean(
                "sigma_spline"
            ),
            dtype=float,
        ),
}


# --------------------------------------------------------------------------------------------------
# 12. POSTERIOR QA
# --------------------------------------------------------------------------------------------------

POSTERIOR_SHAPE_PASS = bool(

    HB2024_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ].shape
    ==
    (
        N_SEX,
    )

    and

    HB2024_V4_POSTERIOR_MEANS[
        "beta_spline"
    ].shape
    ==
    (
        N_SEX,
        N_SPLINE,
    )

    and

    HB2024_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ].shape
    ==
    (
        N_SEX,
    )

    and

    HB2024_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ].shape
    ==
    (
        N_ATHLETES,
    )

    and

    HB2024_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ].shape
    ==
    (
        N_ATHLETES,
    )

    and

    HB2024_V4_POSTERIOR_MEANS[
        "sigma_group"
    ].shape
    ==
    (
        N_GROUP,
    )
)


POSTERIOR_FINITE_PASS = bool(
    all(
        np.isfinite(
            np.asarray(
                value
            )
        )
        .all()

        for value in (
            HB2024_V4_POSTERIOR_MEANS.values()
        )
    )
)


POSITIVE_SCALE_PASS = bool(

    HB2024_V4_POSTERIOR_MEANS[
        "sigma_athlete_intercept"
    ]
    >
    0

    and

    HB2024_V4_POSTERIOR_MEANS[
        "sigma_athlete_slope"
    ]
    >
    0

    and

    (
        HB2024_V4_POSTERIOR_MEANS[
            "sigma_group"
        ]
        >
        0
    )
    .all()

    and

    HB2024_V4_POSTERIOR_MEANS[
        "nu"
    ]
    >
    2
)


# --------------------------------------------------------------------------------------------------
# 13. RECONSTRUCT TRAIN STRUCTURAL MEAN
# --------------------------------------------------------------------------------------------------

alpha_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ]
)


beta_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "beta_spline"
    ]
)


youth_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ]
)


athlete_intercept_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ]
)


athlete_slope_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ]
)


sigma_group_hat = (
    HB2024_V4_POSTERIOR_MEANS[
        "sigma_group"
    ]
)


TRAIN[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx_train
    ]

    +

    np.sum(
        B_TRAIN
        *
        beta_hat[
            sex_idx_train
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx_train
    ]
    *
    youth_train
)


TRAIN[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx_train
    ]
)


TRAIN[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx_train
    ]

    *

    age_scaled_train
)


TRAIN[
    "v4_structural_mean"
] = (
    TRAIN[
        "v4_population_component"
    ]

    +

    TRAIN[
        "v4_athlete_intercept_component"
    ]

    +

    TRAIN[
        "v4_athlete_slope_component"
    ]
)


TRAIN[
    "v4_residual"
] = (
    TRAIN[
        "mark_numeric_m"
    ]

    -

    TRAIN[
        "v4_structural_mean"
    ]
)


TRAIN[
    "v4_error"
] = (
    TRAIN[
        "v4_residual"
    ]
)


TRAIN_PREDICTION_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "v4_population_component",
                "v4_athlete_intercept_component",
                "v4_athlete_slope_component",
                "v4_structural_mean",
                "v4_residual",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 14. TRAIN METRICS
# --------------------------------------------------------------------------------------------------

TRAIN_MAE = float(
    TRAIN[
        "v4_error"
    ]
    .abs()
    .mean()
)


TRAIN_RMSE = float(
    np.sqrt(
        np.mean(
            TRAIN[
                "v4_error"
            ]
            **
            2
        )
    )
)


TRAIN_BIAS = float(
    TRAIN[
        "v4_error"
    ]
    .mean()
)


TRAIN_METRICS = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "n":
                int(
                    len(
                        TRAIN
                    )
                ),

            "athletes":
                int(
                    TRAIN[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "MAE":
                TRAIN_MAE,

            "RMSE":
                TRAIN_RMSE,

            "Bias":
                TRAIN_BIAS,
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. POSTERIOR HYPERPARAMETER SUMMARY
# --------------------------------------------------------------------------------------------------

POSTERIOR_SUMMARY = pd.DataFrame(
    [
        {
            "parameter":
                "nu",

            "value":
                HB2024_V4_POSTERIOR_MEANS[
                    "nu"
                ],
        },

        {
            "parameter":
                "sigma_athlete_intercept",

            "value":
                HB2024_V4_POSTERIOR_MEANS[
                    "sigma_athlete_intercept"
                ],
        },

        {
            "parameter":
                "sigma_athlete_slope",

            "value":
                HB2024_V4_POSTERIOR_MEANS[
                    "sigma_athlete_slope"
                ],
        },

        *[
            {
                "parameter":
                    f"sigma_group_{group}",

                "value":
                    float(
                        sigma_group_hat[
                            i
                        ]
                    ),
            }

            for i, group
            in enumerate(
                GROUP_LABELS
            )
        ],
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL QA
# --------------------------------------------------------------------------------------------------

NO_FUTURE_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2023-12-31"
    )
)


CELL_37_PASS = bool(
    ADVI_FINITE_PASS
    and
    POSTERIOR_SHAPE_PASS
    and
    POSTERIOR_FINITE_PASS
    and
    POSITIVE_SCALE_PASS
    and
    TRAIN_PREDICTION_FINITE_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 17. SAVE IMMUTABLE FIT ARTIFACTS
# --------------------------------------------------------------------------------------------------

POSTERIOR_MEANS_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_posterior_means.npz"
)


ADVI_HISTORY_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_advi_history.npy"
)


TRAIN_STRUCTURAL_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_train_structural_predictions.parquet"
)


FIT_SPEC_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_fit_spec.json"
)


FIT_AUDIT_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_fit_audit.json"
)


np.savez_compressed(
    POSTERIOR_MEANS_PATH,

    alpha_sex=
        HB2024_V4_POSTERIOR_MEANS[
            "alpha_sex"
        ],

    beta_spline=
        HB2024_V4_POSTERIOR_MEANS[
            "beta_spline"
        ],

    youth_implement_effect=
        HB2024_V4_POSTERIOR_MEANS[
            "youth_implement_effect"
        ],

    athlete_intercept=
        HB2024_V4_POSTERIOR_MEANS[
            "athlete_intercept"
        ],

    athlete_slope=
        HB2024_V4_POSTERIOR_MEANS[
            "athlete_slope"
        ],

    sigma_group=
        HB2024_V4_POSTERIOR_MEANS[
            "sigma_group"
        ],

    nu=np.array(
        [
            HB2024_V4_POSTERIOR_MEANS[
                "nu"
            ]
        ]
    ),

    sigma_athlete_intercept=np.array(
        [
            HB2024_V4_POSTERIOR_MEANS[
                "sigma_athlete_intercept"
            ]
        ]
    ),

    sigma_athlete_slope=np.array(
        [
            HB2024_V4_POSTERIOR_MEANS[
                "sigma_athlete_slope"
            ]
        ]
    ),

    sigma_spline=
        HB2024_V4_POSTERIOR_MEANS[
            "sigma_spline"
        ],
)


np.save(
    ADVI_HISTORY_PATH,
    ADVI_HISTORY,
)


TRAIN.to_parquet(
    TRAIN_STRUCTURAL_PATH,
    index=False,
)


FIT_SPEC = {
    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "status":
        (
            "PASSED"
            if CELL_37_PASS
            else
            "FAILED"
        ),

    "train_end":
        "2023-12-31",

    "train_rows":
        int(
            len(
                TRAIN
            )
        ),

    "train_athletes":
        int(
            N_ATHLETES
        ),

    "model_specification_changed_from_HB_2023":
        False,

    "likelihood":
        "StudentT",

    "inference": {
        "method":
            "ADVI",

        "iterations":
            ADVI_ITERATIONS,

        "posterior_draws":
            POSTERIOR_DRAWS,

        "random_seed":
            RANDOM_SEED,
    },

    "priors": {
        "alpha_sex":
            "Normal(45,15)",

        "sigma_spline":
            "HalfNormal(2)",

        "spline_start":
            "Normal(0,3)",

        "spline_delta":
            "Normal(0,sigma_spline)",

        "youth_implement_effect":
            "Normal(0,8)",

        "sigma_athlete_intercept":
            "HalfNormal(10)",

        "athlete_intercept_z":
            "Normal(0,1)",

        "sigma_athlete_slope":
            "HalfNormal(4)",

        "athlete_slope_z":
            "Normal(0,1)",

        "sigma_group":
            "HalfNormal(5)",

        "nu_minus_two":
            "Exponential(0.1)",
    },

    "train_metrics": {
        "MAE":
            TRAIN_MAE,

        "RMSE":
            TRAIN_RMSE,

        "Bias":
            TRAIN_BIAS,
    },
}


atomic_json_write(
    FIT_SPEC,
    FIT_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 18. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "preparation_audit":
        Path(
            HB2024_PREP_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


output_paths = {
    "posterior_means":
        POSTERIOR_MEANS_PATH,

    "advi_history":
        ADVI_HISTORY_PATH,

    "train_structural_predictions":
        TRAIN_STRUCTURAL_PATH,

    "fit_spec":
        FIT_SPEC_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


FIT_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V4_BAYESIAN_REFIT",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_37_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Second historical temporal replication of the frozen "
            "V4 -> V5 -> V6 -> V6.1 modelling architecture."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "fit_specification":
        FIT_SPEC,

    "posterior_hyperparameters":
        {
            row[
                "parameter"
            ]:
                float(
                    row[
                        "value"
                    ]
                )

            for _, row in POSTERIOR_SUMMARY.iterrows()
        },

    "advi": {
        "history_length":
            int(
                len(
                    ADVI_HISTORY
                )
            ),

        "initial_loss":
            float(
                ADVI_HISTORY[
                    0
                ]
            ),

        "final_loss":
            float(
                ADVI_HISTORY[
                    -1
                ]
            ),

        "mean_final_1000_loss":
            float(
                ADVI_HISTORY[
                    -1000:
                ]
                .mean()
            ),
    },

    "qa": {
        "advi_finite":
            ADVI_FINITE_PASS,

        "posterior_shapes":
            POSTERIOR_SHAPE_PASS,

        "posterior_finite":
            POSTERIOR_FINITE_PASS,

        "positive_scales_nu":
            POSITIVE_SCALE_PASS,

        "train_predictions_finite":
            TRAIN_PREDICTION_FINITE_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_37_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "pymc":
            pm.__version__,

        "pytensor":
            pytensor.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    FIT_AUDIT,
    FIT_AUDIT_PATH,
)


FIT_AUDIT_SHA256 = sha256_file(
    FIT_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. LATEST V4 REFIT POINTER
# --------------------------------------------------------------------------------------------------

LATEST_V4_POINTER_PATH = (
    V4_ROOT
    /
    "LATEST_V4_REFIT.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_37_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V4_ATTEMPT_DIR
            ),

        "posterior_means":
            str(
                POSTERIOR_MEANS_PATH
            ),

        "train_structural_predictions":
            str(
                TRAIN_STRUCTURAL_PATH
            ),

        "fit_spec":
            str(
                FIT_SPEC_PATH
            ),

        "audit_manifest":
            str(
                FIT_AUDIT_PATH
            ),

        "audit_sha256":
            FIT_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_V4_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v4_refit"
] = {
    "status":
        (
            "PASSED"
            if CELL_37_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            V4_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_V4_POINTER_PATH
        ),

    "posterior_means":
        str(
            POSTERIOR_MEANS_PATH
        ),

    "train_structural_predictions":
        str(
            TRAIN_STRUCTURAL_PATH
        ),

    "audit_manifest":
        str(
            FIT_AUDIT_PATH
        ),

    "audit_sha256":
        FIT_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_V4_REFIT_COMPLETE"
    if CELL_37_PASS
    else
    "HB_2024_V4_REFIT_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_TRAIN_V4 = TRAIN


HB2024_V4_ATTEMPT_DIR = (
    V4_ATTEMPT_DIR
)


HB2024_V4_POSTERIOR_MEANS_PATH = (
    POSTERIOR_MEANS_PATH
)


HB2024_V4_FIT_AUDIT_PATH = (
    FIT_AUDIT_PATH
)


NOTEBOOK_2_CELL_37_PASS = (
    CELL_37_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. RELEASE LARGE POSTERIOR DRAW OBJECT
# --------------------------------------------------------------------------------------------------

del HB2024_V4_IDATA

gc.collect()


# --------------------------------------------------------------------------------------------------
# 23. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2024 — HISTORICAL V4 BAYESIAN REFIT")
print("=" * 160)


print()
print("FIT CONFIGURATION")
print("-" * 160)

print(
    f"TRAIN rows              : "
    f"{len(TRAIN):,}"
)

print(
    f"TRAIN athletes          : "
    f"{N_ATHLETES:,}"
)

print(
    f"ADVI iterations         : "
    f"{ADVI_ITERATIONS:,}"
)

print(
    f"Posterior draws         : "
    f"{POSTERIOR_DRAWS:,}"
)

print(
    f"Random seed             : "
    f"{RANDOM_SEED}"
)


print()
print("ADVI")
print("-" * 160)

print(
    f"History length          : "
    f"{len(ADVI_HISTORY):,}"
)

print(
    f"Initial loss            : "
    f"{ADVI_HISTORY[0]:,.6f}"
)

print(
    f"Final loss              : "
    f"{ADVI_HISTORY[-1]:,.6f}"
)

print(
    f"Mean final 1,000 loss   : "
    f"{ADVI_HISTORY[-1000:].mean():,.6f}"
)


print()
print("POSTERIOR HYPERPARAMETER MEANS")
print("-" * 160)

display(
    POSTERIOR_SUMMARY
)


print()
print("TRAIN STRUCTURAL FIT")
print("-" * 160)

display(
    TRAIN_METRICS
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID              : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable fit directory : "
    f"{V4_ATTEMPT_DIR}"
)

print(
    f"Audit manifest          : "
    f"{FIT_AUDIT_PATH}"
)

print(
    f"Audit SHA256            : "
    f"{FIT_AUDIT_SHA256}"
)

print(
    f"Git commit              : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"ADVI finite             : "
    f"{ADVI_FINITE_PASS}"
)

print(
    f"Posterior shapes        : "
    f"{POSTERIOR_SHAPE_PASS}"
)

print(
    f"Posterior finite        : "
    f"{POSTERIOR_FINITE_PASS}"
)

print(
    f"Positive scales / nu>2 : "
    f"{POSITIVE_SCALE_PASS}"
)

print(
    f"TRAIN predictions finite: "
    f"{TRAIN_PREDICTION_FINITE_PASS}"
)

print(
    f"No future TRAIN data    : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_37_PASS:

    print(
        "CELL 37 STATUS: HB_2024 HISTORICAL V4 REFIT PASSED"
    )

else:

    print(
        "CELL 37 STATUS: HB_2024 V4 REFIT REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 28
# Original source SHA256: ba97607fccb3032b67043adf05079eb2a1e262c30bc2c4006caf98f82618d8aa
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 38 — HB_2024 HISTORICAL V5 OU REFIT + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Refit the continuous-time V5 OU transient-state model using ONLY
# HB_2024 TRAIN residuals from the historical V4 refit.
#
# TRAIN:
#       <= 2023-12-31
#
# NO 2024 VALIDATION or TEST data used.
#
# Model specification and optimizer procedure are unchanged from HB_2023.
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd
import scipy

from scipy.optimize import minimize


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_37_PASS",

    "HB2024_TRAIN_V4",
    "HB2024_V4_POSTERIOR_MEANS",

    "HB2024_V4_ATTEMPT_DIR",
    "HB2024_V4_FIT_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_37_PASS:

    raise RuntimeError(
        "Cell 37 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_38_HB2024_V5_OU_REFIT"

STAGE_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_GROUPS = len(
    GROUP_LABELS
)


TAU_MIN = 2.0
TAU_MAX = 5000.0

VAR_MIN = 1e-4
VAR_MAX = 400.0

N_STARTS = 10


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2024_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V5_STAGE_ROOT = (
    HB2024_DIR
    /
    "v5_ou_refit"
)


V5_ATTEMPT_DIR = (
    V5_STAGE_ROOT
    /
    ATTEMPT_ID
)


V5_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. TRAIN RESIDUAL INPUT
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2024_TRAIN_V4.copy()
    .reset_index(
        drop=True
    )
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "model_group",
    "v4_residual",
]


missing_columns = [
    column
    for column in required_columns
    if column not in TRAIN.columns
]


if missing_columns:

    raise RuntimeError(
        "HB2024_TRAIN_V4 missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


TRAIN[
    "result_date_dt"
] = pd.to_datetime(
    TRAIN[
        "result_date_dt"
    ]
)


if TRAIN[
    "v4_residual"
].isna().any():

    raise RuntimeError(
        "HB_2024 TRAIN contains missing V4 residuals."
    )


if TRAIN[
    "result_date_dt"
].max() > pd.Timestamp(
    "2023-12-31"
):

    raise RuntimeError(
        "Future data detected in HB_2024 V5 fit input."
    )


# --------------------------------------------------------------------------------------------------
# 6. ONE STATE OBSERVATION PER ATHLETE × DISTINCT DATE
# --------------------------------------------------------------------------------------------------

daily = (
    TRAIN.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        state_observation=(
            "v4_residual",
            "mean",
        ),

        n_performances=(
            "v4_residual",
            "size",
        ),
    )
)


group_counts = pd.crosstab(
    index=[
        TRAIN[
            "aaAthleteId"
        ],

        TRAIN[
            "result_date_dt"
        ],
    ],

    columns=
        TRAIN[
            "model_group"
        ],
)


group_counts = (
    group_counts.reindex(
        columns=GROUP_LABELS,
        fill_value=0,
    )
    .reset_index()
)


group_counts.columns.name = None


for group in GROUP_LABELS:

    group_counts = group_counts.rename(
        columns={
            group:
                f"n_{group}"
        }
    )


daily = daily.merge(
    group_counts,

    on=[
        "aaAthleteId",
        "result_date_dt",
    ],

    how="left",

    validate="one_to_one",
)


daily = (
    daily.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------

daily[
    "previous_state_date"
] = (
    daily.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(1)
)


daily[
    "state_gap_days"
] = (
    daily[
        "result_date_dt"
    ]

    -

    daily[
        "previous_state_date"
    ]
).dt.days


daily[
    "athlete_state_start"
] = (
    daily[
        "previous_state_date"
    ]
    .isna()
)


# --------------------------------------------------------------------------------------------------
# 8. STATE INPUT QA
# --------------------------------------------------------------------------------------------------

DUPLICATE_STATE_COUNT = int(
    daily.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
)


NONPOSITIVE_GAP_COUNT = int(
    (
        daily.loc[
            ~daily[
                "athlete_state_start"
            ],
            "state_gap_days",
        ]
        <=
        0
    )
    .sum()
)


FIRST_DATE_COUNT = int(
    daily[
        "athlete_state_start"
    ]
    .sum()
)


N_STATE_ATHLETES = int(
    daily[
        "aaAthleteId"
    ]
    .nunique()
)


FIRST_DATE_ACCOUNTING_PASS = bool(
    FIRST_DATE_COUNT
    ==
    N_STATE_ATHLETES
)


COUNT_ACCOUNTING_PASS = bool(
    (
        daily[
            [
                f"n_{group}"
                for group in GROUP_LABELS
            ]
        ]
        .sum(
            axis=1
        )
        .to_numpy()

        ==

        daily[
            "n_performances"
        ]
        .to_numpy()
    )
    .all()
)


STATE_INPUT_PASS = bool(
    DUPLICATE_STATE_COUNT
    ==
    0

    and

    NONPOSITIVE_GAP_COUNT
    ==
    0

    and

    FIRST_DATE_ACCOUNTING_PASS

    and

    COUNT_ACCOUNTING_PASS

    and

    np.isfinite(
        daily[
            "state_observation"
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


if not STATE_INPUT_PASS:

    raise RuntimeError(
        "HB_2024 V5 state-input QA failed."
    )


# --------------------------------------------------------------------------------------------------
# 9. NUMERICAL ARRAYS
# --------------------------------------------------------------------------------------------------

state_y = (
    daily[
        "state_observation"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_n = (
    daily[
        "n_performances"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_counts = (
    daily[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_start = (
    daily[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=np.int8
    )
)


state_gap = (
    daily[
        "state_gap_days"
    ]
    .fillna(
        0
    )
    .to_numpy(
        dtype=np.float64
    )
)


# --------------------------------------------------------------------------------------------------
# 10. KALMAN NEGATIVE LOG-LIKELIHOOD
# --------------------------------------------------------------------------------------------------

LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


def kalman_nll_python(
    theta_log,
):

    tau = float(
        np.exp(
            theta_log[
                0
            ]
        )
    )


    Vz = float(
        np.exp(
            theta_log[
                1
            ]
        )
    )


    Rg = np.exp(
        theta_log[
            2:
            2 + N_GROUPS
        ]
    )


    m_post = 0.0
    P_post = Vz

    nll = 0.0


    for i in range(
        len(
            state_y
        )
    ):

        if state_start[
            i
        ]:

            m_prior = 0.0
            P_prior = Vz


        else:

            phi = math.exp(
                -
                state_gap[
                    i
                ]
                /
                tau
            )


            m_prior = (
                phi
                *
                m_post
            )


            P_prior = (
                phi
                *
                phi
                *
                P_post

                +

                Vz
                *
                (
                    1.0
                    -
                    phi
                    *
                    phi
                )
            )


        R_d = float(
            np.dot(
                state_counts[
                    i
                ],
                Rg,
            )

            /

            (
                state_n[
                    i
                ]
                **
                2
            )
        )


        S = (
            P_prior
            +
            R_d
        )


        if (
            not np.isfinite(
                S
            )
            or
            S <= 0
        ):

            return np.inf


        innovation = (
            state_y[
                i
            ]
            -
            m_prior
        )


        nll += 0.5 * (
            LOG_2PI

            +

            np.log(
                S
            )

            +

            innovation
            *
            innovation
            /
            S
        )


        K = (
            P_prior
            /
            S
        )


        m_post = (
            m_prior
            +
            K
            *
            innovation
        )


        P_post = (
            (
                1.0
                -
                K
            )
            *
            P_prior
        )


        P_post = max(
            P_post,
            1e-12,
        )


    return float(
        nll
    )


# --------------------------------------------------------------------------------------------------
# 11. OPTIONAL NUMBA ACCELERATION
# --------------------------------------------------------------------------------------------------

NUMBA_AVAILABLE = False


try:

    import numba

    from numba import njit


    @njit(
        cache=False
    )
    def kalman_nll_numba(
        theta_log,
        state_y_,
        state_n_,
        state_counts_,
        state_start_,
        state_gap_,
    ):

        tau = np.exp(
            theta_log[
                0
            ]
        )


        Vz = np.exp(
            theta_log[
                1
            ]
        )


        Rg = np.exp(
            theta_log[
                2:
                2 + N_GROUPS
            ]
        )


        m_post = 0.0
        P_post = Vz

        nll = 0.0


        for i in range(
            state_y_.shape[
                0
            ]
        ):

            if state_start_[
                i
            ] == 1:

                m_prior = 0.0
                P_prior = Vz


            else:

                phi = np.exp(
                    -
                    state_gap_[
                        i
                    ]
                    /
                    tau
                )


                m_prior = (
                    phi
                    *
                    m_post
                )


                P_prior = (
                    phi
                    *
                    phi
                    *
                    P_post

                    +

                    Vz
                    *
                    (
                        1.0
                        -
                        phi
                        *
                        phi
                    )
                )


            weighted_R = 0.0


            for g in range(
                N_GROUPS
            ):

                weighted_R += (
                    state_counts_[
                        i,
                        g
                    ]
                    *
                    Rg[
                        g
                    ]
                )


            R_d = (
                weighted_R
                /
                (
                    state_n_[
                        i
                    ]
                    *
                    state_n_[
                        i
                    ]
                )
            )


            S = (
                P_prior
                +
                R_d
            )


            if (
                not np.isfinite(
                    S
                )
                or
                S <= 0
            ):

                return np.inf


            innovation = (
                state_y_[
                    i
                ]
                -
                m_prior
            )


            nll += 0.5 * (
                LOG_2PI

                +

                np.log(
                    S
                )

                +

                innovation
                *
                innovation
                /
                S
            )


            K = (
                P_prior
                /
                S
            )


            m_post = (
                m_prior
                +
                K
                *
                innovation
            )


            P_post = (
                (
                    1.0
                    -
                    K
                )
                *
                P_prior
            )


            if P_post < 1e-12:

                P_post = 1e-12


        return nll


    # Compile once.
    _ = kalman_nll_numba(
        np.log(
            np.array(
                [
                    100.0,
                    3.0,
                    10.0,
                    8.0,
                    6.0,
                    6.0,
                ],
                dtype=np.float64,
            )
        ),

        state_y[
            :10
        ],

        state_n[
            :10
        ],

        state_counts[
            :10
        ],

        state_start[
            :10
        ],

        state_gap[
            :10
        ],
    )


    NUMBA_AVAILABLE = True


except Exception:

    NUMBA_AVAILABLE = False


def objective(
    theta_log,
):

    if NUMBA_AVAILABLE:

        return float(
            kalman_nll_numba(
                theta_log,
                state_y,
                state_n,
                state_counts,
                state_start,
                state_gap,
            )
        )


    return kalman_nll_python(
        theta_log
    )


# --------------------------------------------------------------------------------------------------
# 12. DETERMINISTIC MULTI-START OPTIMIZATION
# --------------------------------------------------------------------------------------------------

v4_sigma_group = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "sigma_group"
    ],
    dtype=float,
)


R_START_BASE = (
    v4_sigma_group
    **
    2
)


STATE_VARIANCE_DIAGNOSTIC = float(
    np.var(
        state_y,
        ddof=1,
    )
)


start_specs = [
    (30.0,  0.10, 0.75),
    (45.0,  0.25, 1.00),
    (60.0,  0.20, 1.25),
    (90.0,  0.30, 1.00),
    (120.0, 0.50, 0.75),
    (180.0, 0.75, 1.00),
    (240.0, 1.00, 1.25),
    (365.0, 1.50, 1.00),
    (730.0, 2.00, 0.75),
    (150.0, 0.25, 1.25),
]


log_bounds = [
    (
        np.log(
            TAU_MIN
        ),
        np.log(
            TAU_MAX
        ),
    ),

    (
        np.log(
            VAR_MIN
        ),
        np.log(
            VAR_MAX
        ),
    ),

    *[
        (
            np.log(
                VAR_MIN
            ),
            np.log(
                VAR_MAX
            ),
        )

        for _ in range(
            N_GROUPS
        )
    ],
]


optimizer_rows = []
optimizer_results = []


for start_id, (
    tau_start,
    vz_fraction,
    r_multiplier,
) in enumerate(
    start_specs,
    start=1,
):

    Vz_start = float(
        np.clip(
            STATE_VARIANCE_DIAGNOSTIC
            *
            vz_fraction,

            0.05,
            50.0,
        )
    )


    R_start = np.clip(
        R_START_BASE
        *
        r_multiplier,

        0.05,
        100.0,
    )


    x0 = np.log(
        np.concatenate(
            [
                [
                    tau_start,
                    Vz_start,
                ],

                R_start,
            ]
        )
    )


    result = minimize(
        objective,

        x0=x0,

        method="L-BFGS-B",

        bounds=log_bounds,

        options={
            "maxiter":
                3000,

            "ftol":
                1e-11,

            "gtol":
                1e-7,

            "maxls":
                50,
        },
    )


    optimizer_results.append(
        result
    )


    fitted = np.exp(
        result.x
    )


    optimizer_rows.append(
        {
            "start_id":
                start_id,

            "success":
                bool(
                    result.success
                ),

            "status":
                int(
                    result.status
                ),

            "message":
                str(
                    result.message
                ),

            "iterations":
                int(
                    result.nit
                ),

            "n_function_evals":
                int(
                    result.nfev
                ),

            "nll":
                float(
                    result.fun
                ),

            "tau_days":
                float(
                    fitted[
                        0
                    ]
                ),

            "state_variance":
                float(
                    fitted[
                        1
                    ]
                ),

            "state_sigma":
                float(
                    np.sqrt(
                        fitted[
                            1
                        ]
                    )
                ),

            "R_M700":
                float(
                    fitted[
                        2
                    ]
                ),

            "R_M800":
                float(
                    fitted[
                        3
                    ]
                ),

            "R_W500":
                float(
                    fitted[
                        4
                    ]
                ),

            "R_W600":
                float(
                    fitted[
                        5
                    ]
                ),
        }
    )


OPTIMIZER_AUDIT = pd.DataFrame(
    optimizer_rows
)


# --------------------------------------------------------------------------------------------------
# 13. SELECT BEST SUCCESSFUL RESULT
# --------------------------------------------------------------------------------------------------

successful_indices = [
    i
    for i, result in enumerate(
        optimizer_results
    )
    if (
        result.success
        and
        np.isfinite(
            result.fun
        )
    )
]


if not successful_indices:

    raise RuntimeError(
        "No successful HB_2024 V5 OU optimizer starts."
    )


best_index = min(
    successful_indices,
    key=lambda i:
        optimizer_results[
            i
        ].fun,
)


BEST_RESULT = (
    optimizer_results[
        best_index
    ]
)


BEST_PARAMS = np.exp(
    BEST_RESULT.x
)


V5_TAU_DAYS = float(
    BEST_PARAMS[
        0
    ]
)


V5_STATE_VARIANCE = float(
    BEST_PARAMS[
        1
    ]
)


V5_STATE_SIGMA = float(
    np.sqrt(
        V5_STATE_VARIANCE
    )
)


V5_R_GROUP = {
    group:
        float(
            BEST_PARAMS[
                2 + i
            ]
        )

    for i, group
    in enumerate(
        GROUP_LABELS
    )
}


V5_HALF_LIFE_DAYS = float(
    V5_TAU_DAYS
    *
    np.log(
        2.0
    )
)


BEST_NLL = float(
    BEST_RESULT.fun
)


N_PARAMETERS = (
    2
    +
    N_GROUPS
)


V5_AIC = float(
    2
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


V5_BIC = float(
    np.log(
        len(
            daily
        )
    )
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


# --------------------------------------------------------------------------------------------------
# 14. PERSISTENCE REFERENCE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        1,
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS,

        "phi":
            np.exp(
                -
                REFERENCE_DAYS
                /
                V5_TAU_DAYS
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 15. QA
# --------------------------------------------------------------------------------------------------

SUCCESS_COUNT = int(
    OPTIMIZER_AUDIT[
        "success"
    ]
    .sum()
)


BOUNDARY_TOL = 1e-4


BOUNDARY_FLAGS = []


for value, (
    lower,
    upper,
) in zip(
    BEST_RESULT.x,
    log_bounds,
):

    BOUNDARY_FLAGS.append(
        bool(
            abs(
                value
                -
                lower
            )
            <
            BOUNDARY_TOL

            or

            abs(
                value
                -
                upper
            )
            <
            BOUNDARY_TOL
        )
    )


NO_BOUNDARY_HIT_PASS = bool(
    not any(
        BOUNDARY_FLAGS
    )
)


ALL_PARAMS_FINITE_PASS = bool(
    np.isfinite(
        BEST_PARAMS
    )
    .all()
)


ALL_PARAMS_POSITIVE_PASS = bool(
    (
        BEST_PARAMS
        >
        0
    )
    .all()
)


MULTISTART_PASS = bool(
    SUCCESS_COUNT
    >=
    2
)


NO_FUTURE_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2023-12-31"
    )
)


CELL_38_PASS = bool(
    STATE_INPUT_PASS
    and
    MULTISTART_PASS
    and
    ALL_PARAMS_FINITE_PASS
    and
    ALL_PARAMS_POSITIVE_PASS
    and
    NO_BOUNDARY_HIT_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 16. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

STATE_INPUT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_state_input.parquet"
)


OPTIMIZER_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_optimizer_starts.parquet"
)


PARAMETER_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_ou_parameters.json"
)


FIT_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_fit_audit.json"
)


daily.to_parquet(
    STATE_INPUT_PATH,
    index=False,
)


OPTIMIZER_AUDIT.to_parquet(
    OPTIMIZER_AUDIT_PATH,
    index=False,
)


V5_PARAMETER_SPEC = {
    "model":
        "V5_CONTINUOUS_TIME_OU",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "model_logic_changed_from_HB_2023":
        False,

    "fit_population":
        "HB_2024 TRAIN only",

    "train_end":
        "2023-12-31",

    "state_rows":
        int(
            len(
                daily
            )
        ),

    "state_athletes":
        N_STATE_ATHLETES,

    "parameters": {
        "tau_days":
            V5_TAU_DAYS,

        "half_life_days":
            V5_HALF_LIFE_DAYS,

        "state_variance":
            V5_STATE_VARIANCE,

        "state_sigma":
            V5_STATE_SIGMA,

        "measurement_variance_by_group":
            V5_R_GROUP,
    },

    "fit_statistics": {
        "negative_log_likelihood":
            BEST_NLL,

        "AIC":
            V5_AIC,

        "BIC":
            V5_BIC,

        "n_parameters":
            N_PARAMETERS,
    },

    "optimizer": {
        "method":
            "L-BFGS-B",

        "deterministic_starts":
            N_STARTS,

        "successful_starts":
            SUCCESS_COUNT,

        "best_start_id":
            int(
                best_index
                +
                1
            ),
    },

    "measurement_variance_formula":
        "R_d = sum_g(n_dg * R_g) / n_d^2",

    "state_transition":
        (
            "phi(delta)=exp(-delta/tau); "
            "Var innovation=Vz*(1-phi(delta)^2)"
        ),

    "group_order":
        GROUP_LABELS,

    "randomness":
        "No stochastic fitting; deterministic optimizer starts",

    "numba_used":
        NUMBA_AVAILABLE,
}


atomic_json_write(
    V5_PARAMETER_SPEC,
    PARAMETER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 17. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v4_posterior_means":
        (
            Path(
                HB2024_V4_ATTEMPT_DIR
            )
            /
            "v4_posterior_means.npz"
        ),

    "v4_train_structural_predictions":
        (
            Path(
                HB2024_V4_ATTEMPT_DIR
            )
            /
            "v4_train_structural_predictions.parquet"
        ),

    "v4_fit_spec":
        (
            Path(
                HB2024_V4_ATTEMPT_DIR
            )
            /
            "v4_fit_spec.json"
        ),

    "v4_fit_audit":
        Path(
            HB2024_V4_FIT_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


OUTPUT_ARTIFACTS = {}


for name, path in {
    "state_input":
        STATE_INPUT_PATH,

    "optimizer_audit":
        OPTIMIZER_AUDIT_PATH,

    "parameter_spec":
        PARAMETER_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 18. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

FIT_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V5_OU_REFIT",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_38_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Second historical temporal replication of the "
            "frozen V5 OU specification."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "data_vintage_note":
        (
            "Historical fold reconstructed from the current 2026 canonical "
            "snapshot and restricted by event/result date. Later historical "
            "corrections may therefore be present; no future-dated performances "
            "are used."
        ),

    "state_population": {
        "performance_rows":
            int(
                len(
                    TRAIN
                )
            ),

        "state_rows":
            int(
                len(
                    daily
                )
            ),

        "athletes":
            N_STATE_ATHLETES,

        "multi_performance_dates":
            int(
                (
                    daily[
                        "n_performances"
                    ]
                    >
                    1
                )
                .sum()
            ),

        "multi_group_dates":
            int(
                (
                    (
                        daily[
                            [
                                f"n_{group}"
                                for group in GROUP_LABELS
                            ]
                        ]
                        >
                        0
                    )
                    .sum(
                        axis=1
                    )
                    >
                    1
                )
                .sum()
            ),
    },

    "model_specification":
        V5_PARAMETER_SPEC,

    "qa": {
        "duplicate_state_dates":
            DUPLICATE_STATE_COUNT,

        "nonpositive_state_gaps":
            NONPOSITIVE_GAP_COUNT,

        "first_date_accounting":
            FIRST_DATE_ACCOUNTING_PASS,

        "performance_count_accounting":
            COUNT_ACCOUNTING_PASS,

        "state_input_pass":
            STATE_INPUT_PASS,

        "successful_optimizer_starts":
            SUCCESS_COUNT,

        "multistart_pass":
            MULTISTART_PASS,

        "parameters_finite":
            ALL_PARAMS_FINITE_PASS,

        "parameters_positive":
            ALL_PARAMS_POSITIVE_PASS,

        "no_parameter_at_boundary":
            NO_BOUNDARY_HIT_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_38_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "scipy":
            scipy.__version__,

        "numba":
            (
                numba.__version__
                if NUMBA_AVAILABLE
                else
                None
            ),
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    FIT_AUDIT,
    FIT_AUDIT_PATH,
)


FIT_AUDIT_SHA256 = sha256_file(
    FIT_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    V5_STAGE_ROOT
    /
    "LATEST_V5_OU_REFIT.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V5_OU_REFIT",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_38_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V5_ATTEMPT_DIR
            ),

        "parameter_path":
            str(
                PARAMETER_PATH
            ),

        "fit_audit":
            str(
                FIT_AUDIT_PATH
            ),

        "fit_audit_sha256":
            FIT_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_ou_refit"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_38_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "attempt_directory":
        str(
            V5_ATTEMPT_DIR
        ),

    "fit_audit":
        str(
            FIT_AUDIT_PATH
        ),

    "fit_audit_sha256":
        FIT_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_V5_OU_REFIT_COMPLETE"
    if CELL_38_PASS
    else
    "HB_2024_V5_OU_REFIT_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. GLOBALS FOR CELL 39
# --------------------------------------------------------------------------------------------------

HB2024_V5_STATE_INPUT = daily

HB2024_V5_TAU_DAYS = (
    V5_TAU_DAYS
)

HB2024_V5_HALF_LIFE_DAYS = (
    V5_HALF_LIFE_DAYS
)

HB2024_V5_STATE_VARIANCE = (
    V5_STATE_VARIANCE
)

HB2024_V5_STATE_SIGMA = (
    V5_STATE_SIGMA
)

HB2024_V5_R_GROUP = (
    V5_R_GROUP
)

HB2024_V5_ATTEMPT_DIR = (
    V5_ATTEMPT_DIR
)

HB2024_V5_PARAMETER_PATH = (
    PARAMETER_PATH
)

HB2024_V5_FIT_AUDIT_PATH = (
    FIT_AUDIT_PATH
)

NOTEBOOK_2_CELL_38_PASS = (
    CELL_38_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2024 — HISTORICAL V5 CONTINUOUS-TIME OU REFIT")
print("=" * 160)


print()
print("STATE INPUT")
print("-" * 160)

print(
    f"Performance residual rows       : "
    f"{len(TRAIN):,}"
)

print(
    f"Distinct athlete-date states    : "
    f"{len(daily):,}"
)

print(
    f"Athletes                        : "
    f"{N_STATE_ATHLETES:,}"
)

print(
    f"Multi-performance state dates   : "
    f"{(daily['n_performances'] > 1).sum():,}"
)

print(
    "Multi-group state dates         : "
    f"{((daily[[f'n_{g}' for g in GROUP_LABELS]] > 0).sum(axis=1) > 1).sum():,}"
)


print()
print("OPTIMIZER ROBUSTNESS")
print("-" * 160)

display(
    OPTIMIZER_AUDIT
)


print()
print("BEST V5 OU PARAMETERS")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "parameter":
                    "tau_days",

                "value":
                    V5_TAU_DAYS,
            },

            {
                "parameter":
                    "half_life_days",

                "value":
                    V5_HALF_LIFE_DAYS,
            },

            {
                "parameter":
                    "state_variance",

                "value":
                    V5_STATE_VARIANCE,
            },

            {
                "parameter":
                    "state_sigma",

                "value":
                    V5_STATE_SIGMA,
            },

            *[
                {
                    "parameter":
                        f"R_{group}",

                    "value":
                        V5_R_GROUP[
                            group
                        ],
                }

                for group in GROUP_LABELS
            ],
        ]
    )
)


print()
print("PERSISTENCE REFERENCE")
print("-" * 160)

display(
    PERSISTENCE_REFERENCE
)


print()
print("FIT STATISTICS")
print("-" * 160)

print(
    f"NLL                             : "
    f"{BEST_NLL:,.6f}"
)

print(
    f"AIC                             : "
    f"{V5_AIC:,.6f}"
)

print(
    f"BIC                             : "
    f"{V5_BIC:,.6f}"
)

print(
    f"Successful starts               : "
    f"{SUCCESS_COUNT}/{N_STARTS}"
)

print(
    f"Numba used                      : "
    f"{NUMBA_AVAILABLE}"
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{V5_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                  : "
    f"{FIT_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{FIT_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Duplicate state dates           : "
    f"{DUPLICATE_STATE_COUNT}"
)

print(
    f"Nonpositive gaps                : "
    f"{NONPOSITIVE_GAP_COUNT}"
)

print(
    f"First-date accounting           : "
    f"{FIRST_DATE_ACCOUNTING_PASS}"
)

print(
    f"Performance-count accounting    : "
    f"{COUNT_ACCOUNTING_PASS}"
)

print(
    f"State input valid               : "
    f"{STATE_INPUT_PASS}"
)

print(
    f"Multi-start optimizer           : "
    f"{MULTISTART_PASS}"
)

print(
    f"Parameters finite               : "
    f"{ALL_PARAMS_FINITE_PASS}"
)

print(
    f"Parameters positive             : "
    f"{ALL_PARAMS_POSITIVE_PASS}"
)

print(
    f"No parameter at boundary        : "
    f"{NO_BOUNDARY_HIT_PASS}"
)

print(
    f"No future TRAIN data            : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_38_PASS:

    print(
        "CELL 38 STATUS: HB_2024 HISTORICAL V5 OU REFIT PASSED"
    )

else:

    print(
        "CELL 38 STATUS: HB_2024 V5 OU REFIT REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 29
# Original source SHA256: 21b1a881aec6ce574edc4ca00ec03210f6689ec3332e9eee116aaa1a97d4a0dc
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 39 — HB_2024 HISTORICAL V5 FILTERED STATE RECOVERY + TERMINAL STATE PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Replay the frozen HB_2024 V5 OU model through TRAIN:
#
#       <= 2023-12-31
#
# Recover:
#   • filtered state after every athlete-date
#   • terminal state for every TRAIN athlete
#   • history metadata
#   • innovation diagnostics
#
# NO fitting.
# NO parameter changes.
# NO VALIDATION / TEST information used.
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_38_PASS",

    "HB2024_V5_STATE_INPUT",
    "HB2024_V5_TAU_DAYS",
    "HB2024_V5_STATE_VARIANCE",
    "HB2024_V5_R_GROUP",

    "HB2024_V5_ATTEMPT_DIR",
    "HB2024_V5_PARAMETER_PATH",
    "HB2024_V5_FIT_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_38_PASS:
    raise RuntimeError(
        "Cell 38 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_39_HB2024_V5_STATE_RECOVERY"

STAGE_SCHEMA_VERSION = "1.0.0"

GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]

LOG_2PI = float(
    np.log(
        2.0 * np.pi
    )
)


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2024_DIR = (
    Path(TRAINING_RUN_DIR)
    / "historical_backtest"
    / FOLD_NAME
)

STATE_STAGE_ROOT = (
    HB2024_DIR
    / "v5_state_recovery"
)

STATE_ATTEMPT_DIR = (
    STATE_STAGE_ROOT
    / ATTEMPT_ID
)

STATE_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. FROZEN V5 PARAMETERS
# --------------------------------------------------------------------------------------------------

with Path(
    HB2024_V5_PARAMETER_PATH
).open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_SPEC = json.load(
        f
    )


FIT_NLL = float(
    V5_PARAMETER_SPEC[
        "fit_statistics"
    ][
        "negative_log_likelihood"
    ]
)

TAU = float(
    HB2024_V5_TAU_DAYS
)

VZ = float(
    HB2024_V5_STATE_VARIANCE
)

R_GROUP = {
    group:
        float(
            HB2024_V5_R_GROUP[
                group
            ]
        )
    for group in GROUP_LABELS
}


# --------------------------------------------------------------------------------------------------
# 6. STATE INPUT
# --------------------------------------------------------------------------------------------------

S = (
    HB2024_V5_STATE_INPUT.copy()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)

S[
    "result_date_dt"
] = pd.to_datetime(
    S[
        "result_date_dt"
    ]
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "state_observation",
    "n_performances",
    "athlete_state_start",
    "state_gap_days",

    *[
        f"n_{group}"
        for group in GROUP_LABELS
    ],
]

missing_columns = [
    column
    for column in required_columns
    if column not in S.columns
]

if missing_columns:
    raise RuntimeError(
        "State input missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. MEASUREMENT VARIANCE
# --------------------------------------------------------------------------------------------------

counts = (
    S[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=float
    )
)

R_VECTOR = np.array(
    [
        R_GROUP[
            group
        ]
        for group in GROUP_LABELS
    ],
    dtype=float,
)

n_perf = (
    S[
        "n_performances"
    ]
    .to_numpy(
        dtype=float
    )
)

measurement_variance = (
    counts @ R_VECTOR
) / (
    n_perf ** 2
)

S[
    "measurement_variance"
] = measurement_variance


# --------------------------------------------------------------------------------------------------
# 8. REPLAY KALMAN FILTER
# --------------------------------------------------------------------------------------------------

N = len(S)

prior_mean = np.empty(
    N,
    dtype=float,
)

prior_variance = np.empty(
    N,
    dtype=float,
)

phi_used = np.empty(
    N,
    dtype=float,
)

innovation = np.empty(
    N,
    dtype=float,
)

innovation_variance = np.empty(
    N,
    dtype=float,
)

standardized_innovation = np.empty(
    N,
    dtype=float,
)

kalman_gain = np.empty(
    N,
    dtype=float,
)

filtered_mean = np.empty(
    N,
    dtype=float,
)

filtered_variance = np.empty(
    N,
    dtype=float,
)

log_likelihood_contribution = np.empty(
    N,
    dtype=float,
)


y = (
    S[
        "state_observation"
    ]
    .to_numpy(
        dtype=float
    )
)

gap = (
    S[
        "state_gap_days"
    ]
    .fillna(0)
    .to_numpy(
        dtype=float
    )
)

state_start = (
    S[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=bool
    )
)


m_post = 0.0
P_post = VZ


for i in range(N):

    # ----------------------------------------------------------------------------------------------
    # OU prior
    # ----------------------------------------------------------------------------------------------

    if state_start[i]:

        phi = 0.0

        m_prior = 0.0
        P_prior = VZ

    else:

        phi = math.exp(
            -gap[i] / TAU
        )

        m_prior = (
            phi
            * m_post
        )

        P_prior = (
            phi
            * phi
            * P_post

            +

            VZ
            * (
                1.0
                -
                phi
                * phi
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Measurement update
    # ----------------------------------------------------------------------------------------------

    R_d = measurement_variance[i]

    S_var = (
        P_prior
        +
        R_d
    )

    innov = (
        y[i]
        -
        m_prior
    )

    z = (
        innov
        /
        np.sqrt(
            S_var
        )
    )

    K = (
        P_prior
        /
        S_var
    )

    m_post = (
        m_prior
        +
        K
        * innov
    )

    P_post = (
        (
            1.0
            -
            K
        )
        * P_prior
    )

    P_post = max(
        P_post,
        1e-12,
    )

    ll = -0.5 * (
        LOG_2PI
        +
        np.log(
            S_var
        )
        +
        innov
        * innov
        / S_var
    )

    # ----------------------------------------------------------------------------------------------
    # Store
    # ----------------------------------------------------------------------------------------------

    phi_used[i] = phi

    prior_mean[i] = m_prior
    prior_variance[i] = P_prior

    innovation[i] = innov
    innovation_variance[i] = S_var
    standardized_innovation[i] = z

    kalman_gain[i] = K

    filtered_mean[i] = m_post
    filtered_variance[i] = P_post

    log_likelihood_contribution[i] = ll


# --------------------------------------------------------------------------------------------------
# 9. ATTACH FILTER OUTPUT
# --------------------------------------------------------------------------------------------------

S[
    "state_phi"
] = phi_used

S[
    "state_prior_mean"
] = prior_mean

S[
    "state_prior_variance"
] = prior_variance

S[
    "innovation"
] = innovation

S[
    "innovation_variance"
] = innovation_variance

S[
    "standardized_innovation"
] = standardized_innovation

S[
    "kalman_gain"
] = kalman_gain

S[
    "state_filtered_mean"
] = filtered_mean

S[
    "state_filtered_variance"
] = filtered_variance

S[
    "log_likelihood_contribution"
] = log_likelihood_contribution


# --------------------------------------------------------------------------------------------------
# 10. NLL RECONCILIATION
# --------------------------------------------------------------------------------------------------

REPLAY_NLL = float(
    -S[
        "log_likelihood_contribution"
    ].sum()
)

NLL_ABS_DIFF = float(
    abs(
        REPLAY_NLL
        -
        FIT_NLL
    )
)

NLL_RECONCILIATION_PASS = bool(
    NLL_ABS_DIFF
    <
    1e-5
)


# --------------------------------------------------------------------------------------------------
# 11. ATHLETE HISTORY METADATA
# --------------------------------------------------------------------------------------------------

ATHLETE_HISTORY = (
    S.groupby(
        "aaAthleteId",
        as_index=False,
    )
    .agg(
        first_state_date=(
            "result_date_dt",
            "min",
        ),

        last_state_date=(
            "result_date_dt",
            "max",
        ),

        n_state_dates=(
            "result_date_dt",
            "size",
        ),

        n_state_performances=(
            "n_performances",
            "sum",
        ),
    )
)


ATHLETE_HISTORY[
    "history_span_days"
] = (
    ATHLETE_HISTORY[
        "last_state_date"
    ]

    -

    ATHLETE_HISTORY[
        "first_state_date"
    ]
).dt.days


# --------------------------------------------------------------------------------------------------
# 12. TERMINAL STATE PER ATHLETE
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    S.groupby(
        "aaAthleteId",
        sort=False,
        as_index=False,
    )
    .tail(1)
    .copy()
)


TERMINAL = TERMINAL[
    [
        "aaAthleteId",
        "result_date_dt",
        "state_filtered_mean",
        "state_filtered_variance",
        "state_observation",
        "measurement_variance",
    ]
].rename(
    columns={
        "result_date_dt":
            "terminal_state_date",

        "state_filtered_mean":
            "terminal_state_mean",

        "state_filtered_variance":
            "terminal_state_variance",

        "state_observation":
            "terminal_state_observation",

        "measurement_variance":
            "terminal_measurement_variance",
    }
)


TERMINAL = TERMINAL.merge(
    ATHLETE_HISTORY,
    on="aaAthleteId",
    how="left",
    validate="one_to_one",
)


TERMINAL[
    "terminal_state_sigma"
] = np.sqrt(
    TERMINAL[
        "terminal_state_variance"
    ]
)


TERMINAL = (
    TERMINAL.sort_values(
        "aaAthleteId"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 13. INNOVATION SUMMARY
# --------------------------------------------------------------------------------------------------

Z = (
    S[
        "standardized_innovation"
    ]
    .to_numpy(
        dtype=float
    )
)


INNOVATION_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "n_states":
                int(
                    len(S)
                ),

            "athletes":
                int(
                    S[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "mean_standardized_innovation":
                float(
                    np.mean(Z)
                ),

            "sd_standardized_innovation":
                float(
                    np.std(
                        Z,
                        ddof=1,
                    )
                ),

            "median_standardized_innovation":
                float(
                    np.median(Z)
                ),

            "pct_abs_z_gt_2":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(Z) > 2.0
                    )
                ),

            "pct_abs_z_gt_3":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(Z) > 3.0
                    )
                ),

            "mean_kalman_gain":
                float(
                    S[
                        "kalman_gain"
                    ]
                    .mean()
                ),

            "mean_filtered_variance":
                float(
                    S[
                        "state_filtered_variance"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. QA
# --------------------------------------------------------------------------------------------------

ROW_ACCOUNTING_PASS = bool(
    len(S)
    ==
    len(
        HB2024_V5_STATE_INPUT
    )
)


UNIQUE_STATE_PASS = bool(
    not S.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .any()
)


FINITE_FILTER_PASS = bool(
    np.isfinite(
        S[
            [
                "measurement_variance",
                "state_prior_mean",
                "state_prior_variance",
                "innovation",
                "innovation_variance",
                "standardized_innovation",
                "kalman_gain",
                "state_filtered_mean",
                "state_filtered_variance",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        S[
            "measurement_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "state_prior_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "innovation_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "state_filtered_variance"
        ]
        > 0
    )
    .all()
)


GAIN_BOUNDS_PASS = bool(
    S[
        "kalman_gain"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


FIRST_STATE_PRIOR_PASS = bool(
    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_mean",
        ],
        0.0,
        atol=1e-12,
    )

    and

    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_variance",
        ],
        VZ,
        atol=1e-10,
        rtol=1e-10,
    )
)


TERMINAL_ACCOUNTING_PASS = bool(
    len(TERMINAL)
    ==
    S[
        "aaAthleteId"
    ]
    .nunique()

    and

    TERMINAL[
        "aaAthleteId"
    ]
    .is_unique
)


TERMINAL_DATE_PASS = bool(
    (
        TERMINAL[
            "terminal_state_date"
        ]
        ==
        TERMINAL[
            "last_state_date"
        ]
    )
    .all()
)


HISTORY_METADATA_PASS = bool(
    (
        TERMINAL[
            "n_state_dates"
        ]
        >=
        1
    )
    .all()

    and

    (
        TERMINAL[
            "history_span_days"
        ]
        >=
        0
    )
    .all()

    and

    (
        TERMINAL[
            "n_state_performances"
        ]
        >=
        TERMINAL[
            "n_state_dates"
        ]
    )
    .all()
)


NO_FUTURE_DATA_PASS = bool(
    S[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2023-12-31"
    )
)


CELL_39_PASS = bool(
    ROW_ACCOUNTING_PASS
    and
    UNIQUE_STATE_PASS
    and
    FINITE_FILTER_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    GAIN_BOUNDS_PASS
    and
    FIRST_STATE_PRIOR_PASS
    and
    TERMINAL_ACCOUNTING_PASS
    and
    TERMINAL_DATE_PASS
    and
    HISTORY_METADATA_PASS
    and
    NLL_RECONCILIATION_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 15. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

FILTERED_STATE_PATH = (
    STATE_ATTEMPT_DIR
    / "v5_train_filtered_states.parquet"
)

TERMINAL_STATE_PATH = (
    STATE_ATTEMPT_DIR
    / "v5_train_terminal_states.parquet"
)

INNOVATION_SUMMARY_PATH = (
    STATE_ATTEMPT_DIR
    / "v5_innovation_summary.parquet"
)

STATE_RECOVERY_AUDIT_PATH = (
    STATE_ATTEMPT_DIR
    / "v5_state_recovery_audit.json"
)


S.to_parquet(
    FILTERED_STATE_PATH,
    index=False,
)

TERMINAL.to_parquet(
    TERMINAL_STATE_PATH,
    index=False,
)

INNOVATION_SUMMARY.to_parquet(
    INNOVATION_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 16. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2024_V5_PARAMETER_PATH
        ),

    "v5_fit_audit":
        Path(
            HB2024_V5_FIT_AUDIT_PATH
        ),

    "v5_state_input":
        (
            Path(
                HB2024_V5_ATTEMPT_DIR
            )
            / "v5_state_input.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(path)
                if path.exists()
                else None
            ),
    }


OUTPUT_ARTIFACTS = {}


for name, path in {
    "filtered_states":
        FILTERED_STATE_PATH,

    "terminal_states":
        TERMINAL_STATE_PATH,

    "innovation_summary":
        INNOVATION_SUMMARY_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(path),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 17. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

STATE_RECOVERY_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V5_STATE_RECOVERY",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_39_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Recover leakage-safe HB_2024 TRAIN filtered and "
            "terminal OU states."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "replay": {
        "fit_nll":
            FIT_NLL,

        "replay_nll":
            REPLAY_NLL,

        "absolute_nll_difference":
            NLL_ABS_DIFF,

        "nll_reconciliation_pass":
            NLL_RECONCILIATION_PASS,
    },

    "state_population": {
        "rows":
            int(
                len(S)
            ),

        "athletes":
            int(
                S[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "terminal_states":
            int(
                len(TERMINAL)
            ),
    },

    "terminal_state_schema": [
        "aaAthleteId",
        "terminal_state_date",
        "terminal_state_mean",
        "terminal_state_variance",
        "terminal_state_sigma",
        "first_state_date",
        "last_state_date",
        "n_state_dates",
        "n_state_performances",
        "history_span_days",
    ],

    "qa": {
        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "unique_state_dates":
            UNIQUE_STATE_PASS,

        "finite_filter":
            FINITE_FILTER_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "kalman_gain_bounds":
            GAIN_BOUNDS_PASS,

        "first_state_prior":
            FIRST_STATE_PRIOR_PASS,

        "terminal_accounting":
            TERMINAL_ACCOUNTING_PASS,

        "terminal_dates":
            TERMINAL_DATE_PASS,

        "history_metadata":
            HISTORY_METADATA_PASS,

        "nll_reconciliation":
            NLL_RECONCILIATION_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_39_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    STATE_RECOVERY_AUDIT,
    STATE_RECOVERY_AUDIT_PATH,
)


STATE_RECOVERY_AUDIT_SHA256 = sha256_file(
    STATE_RECOVERY_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 18. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_STATE_POINTER_PATH = (
    STATE_STAGE_ROOT
    / "LATEST_V5_STATE_RECOVERY.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V5_STATE_RECOVERY",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_39_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                STATE_ATTEMPT_DIR
            ),

        "filtered_states":
            str(
                FILTERED_STATE_PATH
            ),

        "terminal_states":
            str(
                TERMINAL_STATE_PATH
            ),

        "innovation_summary":
            str(
                INNOVATION_SUMMARY_PATH
            ),

        "audit_manifest":
            str(
                STATE_RECOVERY_AUDIT_PATH
            ),

        "audit_sha256":
            STATE_RECOVERY_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_STATE_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 19. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_state_recovery"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_39_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_STATE_POINTER_PATH
        ),

    "attempt_directory":
        str(
            STATE_ATTEMPT_DIR
        ),

    "terminal_states":
        str(
            TERMINAL_STATE_PATH
        ),

    "audit_manifest":
        str(
            STATE_RECOVERY_AUDIT_PATH
        ),

    "audit_sha256":
        STATE_RECOVERY_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_V5_STATE_RECOVERY_COMPLETE"
    if CELL_39_PASS
    else
    "HB_2024_V5_STATE_RECOVERY_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_V5_FILTERED_STATES = S

HB2024_V5_TERMINAL_STATES = TERMINAL

HB2024_V5_INNOVATION_SUMMARY = (
    INNOVATION_SUMMARY
)

HB2024_V5_STATE_RECOVERY_ATTEMPT_DIR = (
    STATE_ATTEMPT_DIR
)

HB2024_V5_TERMINAL_STATE_PATH = (
    TERMINAL_STATE_PATH
)

HB2024_V5_STATE_RECOVERY_AUDIT_PATH = (
    STATE_RECOVERY_AUDIT_PATH
)

NOTEBOOK_2_CELL_39_PASS = (
    CELL_39_PASS
)


# --------------------------------------------------------------------------------------------------
# 21. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2024 — HISTORICAL V5 FILTERED STATE RECOVERY")
print("=" * 160)


print()
print("STATE REPLAY")
print("-" * 160)

print(
    f"Filtered athlete-date states    : "
    f"{len(S):,}"
)

print(
    f"Athletes                        : "
    f"{S['aaAthleteId'].nunique():,}"
)

print(
    f"Terminal athlete states         : "
    f"{len(TERMINAL):,}"
)


print()
print("NLL RECONCILIATION")
print("-" * 160)

print(
    f"Fit NLL                         : "
    f"{FIT_NLL:,.9f}"
)

print(
    f"Replay NLL                      : "
    f"{REPLAY_NLL:,.9f}"
)

print(
    f"Absolute difference             : "
    f"{NLL_ABS_DIFF:.12e}"
)

print(
    f"Reconciliation pass             : "
    f"{NLL_RECONCILIATION_PASS}"
)


print()
print("INNOVATION DIAGNOSTICS")
print("-" * 160)

display(
    INNOVATION_SUMMARY
)


print()
print("TERMINAL STATE HISTORY SUMMARY")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "athletes":
                    int(
                        len(TERMINAL)
                    ),

                "median_state_dates":
                    float(
                        TERMINAL[
                            "n_state_dates"
                        ]
                        .median()
                    ),

                "median_history_span_days":
                    float(
                        TERMINAL[
                            "history_span_days"
                        ]
                        .median()
                    ),

                "max_history_span_days":
                    int(
                        TERMINAL[
                            "history_span_days"
                        ]
                        .max()
                    ),

                "median_terminal_state_sigma":
                    float(
                        TERMINAL[
                            "terminal_state_sigma"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{STATE_ATTEMPT_DIR}"
)

print(
    f"Terminal-state artifact         : "
    f"{TERMINAL_STATE_PATH}"
)

print(
    f"Audit manifest                  : "
    f"{STATE_RECOVERY_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{STATE_RECOVERY_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Row accounting                  : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Unique athlete-date states      : "
    f"{UNIQUE_STATE_PASS}"
)

print(
    f"Finite filter quantities        : "
    f"{FINITE_FILTER_PASS}"
)

print(
    f"Positive variances              : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Kalman gains in [0,1]           : "
    f"{GAIN_BOUNDS_PASS}"
)

print(
    f"First-state priors correct      : "
    f"{FIRST_STATE_PRIOR_PASS}"
)

print(
    f"One terminal state / athlete    : "
    f"{TERMINAL_ACCOUNTING_PASS}"
)

print(
    f"Terminal dates correct          : "
    f"{TERMINAL_DATE_PASS}"
)

print(
    f"History metadata valid          : "
    f"{HISTORY_METADATA_PASS}"
)

print(
    f"No future TRAIN data            : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_39_PASS:

    print(
        "CELL 39 STATUS: HB_2024 HISTORICAL V5 STATE RECOVERY PASSED"
    )

else:

    print(
        "CELL 39 STATUS: HB_2024 V5 STATE RECOVERY REVIEW REQUIRED"
    )

print("=" * 160)



HB_2024 — HISTORICAL V4 REFIT PREPARATION

TEMPORAL SPLIT
----------------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN       : <= 2023-12-31
VALIDATION  : 2024-01-01 -> 2024-04-30
TEST        : 2024-05-01 -> 2024-08-30

POPULATION
----------------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN rows / athletes          : 129,573 / 9,336
VALIDATION established         : 4,288 / 2,219
TEST established               : 11,054 / 3,997

TRAIN-ONLY SPLINE
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Basis shape                    : (129573, 7)
Internal knots                : [18.1769646194 20.2167053396 22.2153774547 25.2955228376]
Column means                  : [0.0675803

,split,n,median_gap_days,mean_gap_days,min_gap_days,max_gap_days
0,VALIDATION,4288,93.0,160.984142,1,4979
1,TEST,11054,27.0,109.385109,1,4053



AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T132655_786390Z
Immutable preparation directory: /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/preparation/20260922T132655_786390Z
Audit manifest                 : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/preparation/20260922T132655_786390Z/preparation_audit.json
Audit SHA256                   : 94191f69800e9ca47469dac65430073141485c78ad728345471882f14bf1ee20

QA
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Temporal split                 : True
TRAIN-only athlete index    

Output()

Finished [100%]: Average Loss = 3.5357e+05


...


FIT STATISTICS
----------------------------------------------------------------------------------------------------------------------------------------------------------------
NLL                             : 320,996.065033
AIC                             : 642,004.130066
BIC                             : 642,062.729050
Successful starts               : 10/10
Numba used                      : True

AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T134153_499008Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v5_ou_refit/20260922T134153_499008Z
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical

,fold,n_states,athletes,mean_standardized_innovation,sd_standardized_innovation,median_standardized_innovation,pct_abs_z_gt_2,pct_abs_z_gt_3,mean_kalman_gain,mean_filtered_variance
0,HB_2024,128862,9336,-0.033746,0.999436,-0.000459,4.352718,0.755071,0.271999,1.666034



TERMINAL STATE HISTORY SUMMARY
----------------------------------------------------------------------------------------------------------------------------------------------------------------


,athletes,median_state_dates,median_history_span_days,max_history_span_days,median_terminal_state_sigma
0,9336,6.0,773.0,7028,1.340667



AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T134158_957815Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v5_state_recovery/20260922T134158_957815Z
Terminal-state artifact         : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v5_state_recovery/20260922T134158_957815Z/v5_train_terminal_states.parquet
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v5_state_recovery/20260922T134158_957815Z/v5_state_recovery_audit.json
Audit SHA256                    : 2abf18596af6b1b351147c361ca16

## Stage 14 — HB_2024 replay, calibration and evidence freeze

Replay rolling forecasts, calibrate V6 on historical VALIDATION, compare V5/V6/V6.1 on untouched TEST, and freeze evidence.

**Legacy source cells represented:** 30, 31, 32, 33  
**Execution rule:** run only after the preceding stage has completed successfully.


In [15]:
# ==================================================================================================
# REFACTORED STAGE 14 — HB_2024 replay, calibration and evidence freeze
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 30
# Original source SHA256: 824df3da26eb53bb9213a3dac8c1360fb60456ad8ffd652f8e4421ab769151d9
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 40 — HB_2024 ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY
# ==================================================================================================
#
# Purpose
# -------
# Generate leakage-safe rolling-origin historical forecasts for HB_2024.
#
# Sequence:
#
#   TRAIN terminal state
#       ↓
#   predict VALIDATION athlete-date
#       ↓
#   observe that date
#       ↓
#   update OU state once
#       ↓
#   continue chronologically through VALIDATION and TEST
#
# Same-day performances share exactly the same pre-date state.
#
# NO V6 fitting occurs here.
# NO TEST information is used before its chronological observation date.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_39_PASS",

    "HB2024_VALIDATION",
    "HB2024_TEST",

    "HB2024_V4_POSTERIOR_MEANS",
    "HB2024_V5_TERMINAL_STATES",

    "HB2024_V5_TAU_DAYS",
    "HB2024_V5_STATE_VARIANCE",
    "HB2024_V5_R_GROUP",

    "HB2024_V5_PARAMETER_PATH",
    "HB2024_V5_STATE_RECOVERY_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_39_PASS:
    raise RuntimeError(
        "Cell 39 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_40_HB2024_ROLLING_V4_V5"

STAGE_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(7)
]


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2024_DIR = (
    Path(TRAINING_RUN_DIR)
    / "historical_backtest"
    / FOLD_NAME
)

ROLLING_STAGE_ROOT = (
    HB2024_DIR
    / "rolling_v4_v5"
)

ROLLING_ATTEMPT_DIR = (
    ROLLING_STAGE_ROOT
    / ATTEMPT_ID
)

ROLLING_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(obj, path):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(start_path):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. FREEZE HISTORICAL V4 PARAMETERS
# --------------------------------------------------------------------------------------------------

alpha_hat = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ],
    dtype=float,
)

beta_hat = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "beta_spline"
    ],
    dtype=float,
)

youth_hat = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ],
    dtype=float,
)

athlete_intercept_hat = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ],
    dtype=float,
)

athlete_slope_hat = np.asarray(
    HB2024_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 6. FREEZE V5 PARAMETERS
# --------------------------------------------------------------------------------------------------

TAU = float(
    HB2024_V5_TAU_DAYS
)

VZ = float(
    HB2024_V5_STATE_VARIANCE
)

R_GROUP = {
    group:
        float(
            HB2024_V5_R_GROUP[
                group
            ]
        )
    for group in GROUP_LABELS
}


R_VECTOR = np.array(
    [
        R_GROUP[
            group
        ]
        for group in GROUP_LABELS
    ],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 7. PREPARE VALIDATION + TEST
# --------------------------------------------------------------------------------------------------

VALIDATION = (
    HB2024_VALIDATION.copy()
    .reset_index(drop=True)
)

TEST = (
    HB2024_TEST.copy()
    .reset_index(drop=True)
)


VALIDATION[
    "evaluation_split"
] = "VALIDATION"

TEST[
    "evaluation_split"
] = "TEST"


VALIDATION[
    "split_row_id"
] = np.arange(
    len(VALIDATION)
)

TEST[
    "split_row_id"
] = np.arange(
    len(TEST)
)


PRED = pd.concat(
    [
        VALIDATION,
        TEST,
    ],
    ignore_index=True,
)


PRED[
    "result_date_dt"
] = pd.to_datetime(
    PRED[
        "result_date_dt"
    ]
)


PRED[
    "_forecast_row_id"
] = np.arange(
    len(PRED),
    dtype=int,
)


# --------------------------------------------------------------------------------------------------
# 8. REQUIRED COLUMN QA
# --------------------------------------------------------------------------------------------------

required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",

    "sex_idx",
    "group_idx",
    "athlete_idx",

    "is_youth_implement",
    "age_scaled",

    "model_group",

    *SPLINE_COLUMNS,
]


missing_columns = [
    column
    for column in required_columns
    if column not in PRED.columns
]


if missing_columns:
    raise RuntimeError(
        "Prediction data missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. RECONSTRUCT HISTORICAL V4 STRUCTURAL PREDICTIONS
# --------------------------------------------------------------------------------------------------

sex_idx = (
    PRED[
        "sex_idx"
    ]
    .to_numpy(
        dtype=int
    )
)

athlete_idx = (
    PRED[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=int
    )
)

youth = (
    PRED[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=float
    )
)

age_scaled = (
    PRED[
        "age_scaled"
    ]
    .to_numpy(
        dtype=float
    )
)

B = (
    PRED[
        SPLINE_COLUMNS
    ]
    .to_numpy(
        dtype=float
    )
)


PRED[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx
    ]

    +

    np.sum(
        B
        *
        beta_hat[
            sex_idx
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx
    ]
    *
    youth
)


PRED[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx
    ]
)


PRED[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx
    ]
    *
    age_scaled
)


PRED[
    "v4_structural_mean"
] = (
    PRED[
        "v4_population_component"
    ]

    +

    PRED[
        "v4_athlete_intercept_component"
    ]

    +

    PRED[
        "v4_athlete_slope_component"
    ]
)


PRED[
    "v4_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v4_structural_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. TRAIN TERMINAL STATE LOOKUP
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    HB2024_V5_TERMINAL_STATES.copy()
)

TERMINAL[
    "terminal_state_date"
] = pd.to_datetime(
    TERMINAL[
        "terminal_state_date"
    ]
)


terminal_lookup = (
    TERMINAL.set_index(
        "aaAthleteId"
    )
)


prediction_athletes = set(
    PRED[
        "aaAthleteId"
    ]
    .unique()
)

terminal_athletes = set(
    TERMINAL[
        "aaAthleteId"
    ]
    .unique()
)


MISSING_TERMINAL_ATHLETES = (
    prediction_athletes
    -
    terminal_athletes
)


if MISSING_TERMINAL_ATHLETES:

    raise RuntimeError(
        "Established prediction athletes missing TRAIN terminal states:\n"
        +
        "\n".join(
            str(x)
            for x in sorted(
                MISSING_TERMINAL_ATHLETES
            )[:20]
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. ALLOCATE OUTPUT ARRAYS
# --------------------------------------------------------------------------------------------------

N = len(PRED)

state_prior_mean = np.full(
    N,
    np.nan,
)

state_prior_variance = np.full(
    N,
    np.nan,
)

state_gap_dynamic = np.full(
    N,
    np.nan,
)

state_phi = np.full(
    N,
    np.nan,
)

v5_prediction_mean = np.full(
    N,
    np.nan,
)


date_state_rows = []


# --------------------------------------------------------------------------------------------------
# 12. ROLLING-ORIGIN FORECAST / UPDATE
# --------------------------------------------------------------------------------------------------

for athlete_id, athlete_rows in PRED.groupby(
    "aaAthleteId",
    sort=False,
):

    terminal_row = (
        terminal_lookup.loc[
            athlete_id
        ]
    )


    current_mean = float(
        terminal_row[
            "terminal_state_mean"
        ]
    )

    current_variance = float(
        terminal_row[
            "terminal_state_variance"
        ]
    )

    current_date = pd.Timestamp(
        terminal_row[
            "terminal_state_date"
        ]
    )


    athlete_rows = (
        athlete_rows.sort_values(
            [
                "result_date_dt",
                "_forecast_row_id",
            ]
        )
    )


    for forecast_date, date_rows in athlete_rows.groupby(
        "result_date_dt",
        sort=True,
    ):

        forecast_date = pd.Timestamp(
            forecast_date
        )


        delta_days = int(
            (
                forecast_date
                -
                current_date
            ).days
        )


        if delta_days <= 0:

            raise RuntimeError(
                f"Non-positive state gap for athlete "
                f"{athlete_id}: "
                f"{current_date} -> {forecast_date}"
            )


        # ------------------------------------------------------------------------------------------
        # PRE-DATE OU PROPAGATION
        # ------------------------------------------------------------------------------------------

        phi = float(
            np.exp(
                -delta_days
                /
                TAU
            )
        )


        prior_mean = float(
            phi
            *
            current_mean
        )


        prior_variance = float(
            phi
            *
            phi
            *
            current_variance

            +

            VZ
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


        row_ids = (
            date_rows[
                "_forecast_row_id"
            ]
            .to_numpy(
                dtype=int
            )
        )


        # ------------------------------------------------------------------------------------------
        # PREDICTION BEFORE OBSERVING THIS DATE
        # ------------------------------------------------------------------------------------------

        state_prior_mean[
            row_ids
        ] = prior_mean

        state_prior_variance[
            row_ids
        ] = prior_variance

        state_gap_dynamic[
            row_ids
        ] = delta_days

        state_phi[
            row_ids
        ] = phi


        v5_prediction_mean[
            row_ids
        ] = (
            PRED.loc[
                row_ids,
                "v4_structural_mean",
            ]
            .to_numpy(
                dtype=float
            )

            +

            prior_mean
        )


        # ------------------------------------------------------------------------------------------
        # AFTER PREDICTION: OBSERVE ALL SAME-DAY RESULTS
        # ------------------------------------------------------------------------------------------

        residuals = (
            PRED.loc[
                row_ids,
                "v4_error",
            ]
            .to_numpy(
                dtype=float
            )
        )


        state_observation = float(
            np.mean(
                residuals
            )
        )


        n_date = int(
            len(row_ids)
        )


        group_counts = np.array(
            [
                int(
                    (
                        PRED.loc[
                            row_ids,
                            "model_group",
                        ]
                        ==
                        group
                    )
                    .sum()
                )
                for group in GROUP_LABELS
            ],
            dtype=float,
        )


        R_d = float(
            np.dot(
                group_counts,
                R_VECTOR,
            )

            /

            (
                n_date
                **
                2
            )
        )


        innovation = float(
            state_observation
            -
            prior_mean
        )


        innovation_variance = float(
            prior_variance
            +
            R_d
        )


        K = float(
            prior_variance
            /
            innovation_variance
        )


        posterior_mean = float(
            prior_mean
            +
            K
            *
            innovation
        )


        posterior_variance = float(
            (
                1.0
                -
                K
            )
            *
            prior_variance
        )


        posterior_variance = max(
            posterior_variance,
            1e-12,
        )


        date_state_rows.append(
            {
                "aaAthleteId":
                    athlete_id,

                "result_date_dt":
                    forecast_date,

                "evaluation_split":
                    date_rows[
                        "evaluation_split"
                    ]
                    .iloc[0],

                "state_gap_days":
                    delta_days,

                "state_phi":
                    phi,

                "state_prior_mean":
                    prior_mean,

                "state_prior_variance":
                    prior_variance,

                "state_observation":
                    state_observation,

                "n_performances":
                    n_date,

                "measurement_variance":
                    R_d,

                "innovation":
                    innovation,

                "innovation_variance":
                    innovation_variance,

                "standardized_innovation":
                    innovation
                    /
                    np.sqrt(
                        innovation_variance
                    ),

                "kalman_gain":
                    K,

                "state_filtered_mean":
                    posterior_mean,

                "state_filtered_variance":
                    posterior_variance,

                **{
                    f"n_{group}":
                        int(
                            group_counts[i]
                        )

                    for i, group
                    in enumerate(
                        GROUP_LABELS
                    )
                },
            }
        )


        # State becomes available for FUTURE dates only.
        current_mean = posterior_mean
        current_variance = posterior_variance
        current_date = forecast_date


# --------------------------------------------------------------------------------------------------
# 13. ATTACH ROLLING OUTPUT
# --------------------------------------------------------------------------------------------------

PRED[
    "state_prior_mean"
] = state_prior_mean

PRED[
    "state_prior_variance"
] = state_prior_variance

PRED[
    "state_prior_sigma"
] = np.sqrt(
    state_prior_variance
)

PRED[
    "rolling_state_gap_days"
] = state_gap_dynamic

PRED[
    "state_phi"
] = state_phi

PRED[
    "v5_prediction_mean"
] = v5_prediction_mean


PRED[
    "v5_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v5_prediction_mean"
    ]
)


DATE_STATES = pd.DataFrame(
    date_state_rows
)


# --------------------------------------------------------------------------------------------------
# 14. CHRONOLOGY RECONCILIATION
# --------------------------------------------------------------------------------------------------

if "state_gap_days" not in PRED.columns:

    raise RuntimeError(
        "Prepared historical fold lacks reference state_gap_days."
    )


GAP_RECONCILIATION_MAX_DIFF = float(
    np.max(
        np.abs(
            PRED[
                "rolling_state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )

            -

            PRED[
                "state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
)


GAP_RECONCILIATION_PASS = bool(
    GAP_RECONCILIATION_MAX_DIFF
    <
    1e-12
)


# --------------------------------------------------------------------------------------------------
# 15. SPLIT BACK INTO VALIDATION / TEST
# --------------------------------------------------------------------------------------------------

ROLLING_VALIDATION = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "VALIDATION"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


ROLLING_TEST = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "TEST"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 16. METRIC HELPERS
# --------------------------------------------------------------------------------------------------

def point_metrics(df):

    rows = []

    for model in [
        "V4",
        "V5",
    ]:

        error = (
            df[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )

        rows.append(
            {
                "model":
                    model,

                "n":
                    int(
                        len(df)
                    ),

                "athletes":
                    int(
                        df[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error ** 2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


VALIDATION_METRICS = point_metrics(
    ROLLING_VALIDATION
)

VALIDATION_METRICS[
    "split"
] = "VALIDATION"


TEST_METRICS = point_metrics(
    ROLLING_TEST
)

TEST_METRICS[
    "split"
] = "TEST"


POINT_METRICS = pd.concat(
    [
        VALIDATION_METRICS,
        TEST_METRICS,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 17. OVERALL V4 -> V5 GAIN
# --------------------------------------------------------------------------------------------------

gain_rows = []


for split, g in PRED.groupby(
    "evaluation_split",
    sort=False,
):

    e4 = (
        g[
            "v4_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    e5 = (
        g[
            "v5_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v4_mae = float(
        np.mean(
            np.abs(e4)
        )
    )

    v5_mae = float(
        np.mean(
            np.abs(e5)
        )
    )

    v4_rmse = float(
        np.sqrt(
            np.mean(
                e4 ** 2
            )
        )
    )

    v5_rmse = float(
        np.sqrt(
            np.mean(
                e5 ** 2
            )
        )
    )


    gain_rows.append(
        {
            "split":
                split,

            "n":
                int(
                    len(g)
                ),

            "athletes":
                int(
                    g[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4_mae,

            "V5_MAE":
                v5_mae,

            "V4_to_V5_MAE_gain_m":
                v4_mae
                -
                v5_mae,

            "V4_RMSE":
                v4_rmse,

            "V5_RMSE":
                v5_rmse,

            "V4_to_V5_RMSE_gain_m":
                v4_rmse
                -
                v5_rmse,

            "V5_Bias":
                float(
                    np.mean(e5)
                ),
        }
    )


V5_GAIN_SUMMARY = pd.DataFrame(
    gain_rows
)


# --------------------------------------------------------------------------------------------------
# 18. STATE-GAP METRICS
# --------------------------------------------------------------------------------------------------

GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


PRED[
    "state_gap_band"
] = pd.cut(
    PRED[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


gap_rows = []


for (
    split,
    gap_band
), g in PRED.groupby(
    [
        "evaluation_split",
        "state_gap_band",
    ],
    observed=True,
):

    for model in [
        "V4",
        "V5",
    ]:

        error = (
            g[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        gap_rows.append(
            {
                "split":
                    split,

                "state_gap_band":
                    str(
                        gap_band
                    ),

                "model":
                    model,

                "n":
                    int(
                        len(g)
                    ),

                "athletes":
                    int(
                        g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "mean_gap_days":
                    float(
                        g[
                            "rolling_state_gap_days"
                        ]
                        .mean()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error ** 2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )


GAP_METRICS = pd.DataFrame(
    gap_rows
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

FINITE_PREDICTION_PASS = bool(
    np.isfinite(
        PRED[
            [
                "v4_structural_mean",
                "v5_prediction_mean",
                "v4_error",
                "v5_error",
                "state_prior_mean",
                "state_prior_variance",
                "rolling_state_gap_days",
                "state_phi",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        PRED[
            "state_prior_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        DATE_STATES[
            "state_filtered_variance"
        ]
        >
        0
    )
    .all()
)


NO_SAME_DAY_LEAKAGE_PASS = bool(
    DATE_STATES.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
    ==
    0
)


ROW_ACCOUNTING_PASS = bool(
    len(
        ROLLING_VALIDATION
    )
    ==
    len(
        HB2024_VALIDATION
    )

    and

    len(
        ROLLING_TEST
    )
    ==
    len(
        HB2024_TEST
    )
)


SPLIT_CHRONOLOGY_PASS = bool(
    ROLLING_VALIDATION[
        "result_date_dt"
    ]
    .max()

    <

    ROLLING_TEST[
        "result_date_dt"
    ]
    .min()
)


CELL_40_PASS = bool(
    FINITE_PREDICTION_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    NO_SAME_DAY_LEAKAGE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    SPLIT_CHRONOLOGY_PASS
    and
    GAP_RECONCILIATION_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

VALIDATION_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    / "validation_rolling_predictions.parquet"
)

TEST_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    / "test_rolling_predictions.parquet"
)

DATE_STATE_UPDATES_PATH = (
    ROLLING_ATTEMPT_DIR
    / "rolling_date_state_updates.parquet"
)

POINT_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    / "point_metrics.parquet"
)

GAP_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    / "gap_metrics.parquet"
)

GAIN_SUMMARY_PATH = (
    ROLLING_ATTEMPT_DIR
    / "v4_to_v5_gain_summary.parquet"
)

ROLLING_AUDIT_PATH = (
    ROLLING_ATTEMPT_DIR
    / "rolling_forecast_audit.json"
)


ROLLING_VALIDATION.to_parquet(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)

ROLLING_TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)

DATE_STATES.to_parquet(
    DATE_STATE_UPDATES_PATH,
    index=False,
)

POINT_METRICS.to_parquet(
    POINT_METRICS_PATH,
    index=False,
)

GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)

V5_GAIN_SUMMARY.to_parquet(
    GAIN_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. AUDIT PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2024_V5_PARAMETER_PATH
        ),

    "v5_state_recovery_audit":
        Path(
            HB2024_V5_STATE_RECOVERY_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(path)
                if path.exists()
                else None
            ),
    }


output_paths = {
    "validation_predictions":
        VALIDATION_PREDICTIONS_PATH,

    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "date_state_updates":
        DATE_STATE_UPDATES_PATH,

    "point_metrics":
        POINT_METRICS_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "v5_gain_summary":
        GAIN_SUMMARY_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(path),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


ROLLING_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "ROLLING_V4_V5_FORECAST",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_40_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Leakage-safe HB_2024 rolling V4/V5 forecast replication."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "temporal_windows": {
        "validation":
            "2024-01-01 through 2024-04-30",

        "test":
            "2024-05-01 through 2024-08-30",
    },

    "state_update_rule":
        (
            "Predict all performances on an athlete-date using the pre-date state; "
            "then update once using the mean same-day V4 residual."
        ),

    "v5_parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "population": {
        "validation_rows":
            int(
                len(
                    ROLLING_VALIDATION
                )
            ),

        "validation_athletes":
            int(
                ROLLING_VALIDATION[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "test_rows":
            int(
                len(
                    ROLLING_TEST
                )
            ),

        "test_athletes":
            int(
                ROLLING_TEST[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "rolling_athlete_dates":
            int(
                len(
                    DATE_STATES
                )
            ),
    },

    "qa": {
        "finite_predictions":
            FINITE_PREDICTION_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "no_same_day_leakage":
            NO_SAME_DAY_LEAKAGE_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "split_chronology":
            SPLIT_CHRONOLOGY_PASS,

        "gap_reconciliation":
            GAP_RECONCILIATION_PASS,

        "gap_reconciliation_max_diff":
            GAP_RECONCILIATION_MAX_DIFF,

        "cell_pass":
            CELL_40_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    ROLLING_AUDIT,
    ROLLING_AUDIT_PATH,
)


ROLLING_AUDIT_SHA256 = sha256_file(
    ROLLING_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_ROLLING_POINTER_PATH = (
    ROLLING_STAGE_ROOT
    / "LATEST_ROLLING_V4_V5.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "ROLLING_V4_V5_FORECAST",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_40_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                ROLLING_ATTEMPT_DIR
            ),

        "validation_predictions":
            str(
                VALIDATION_PREDICTIONS_PATH
            ),

        "test_predictions":
            str(
                TEST_PREDICTIONS_PATH
            ),

        "audit_manifest":
            str(
                ROLLING_AUDIT_PATH
            ),

        "audit_sha256":
            ROLLING_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_ROLLING_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "rolling_v4_v5"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_40_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_ROLLING_POINTER_PATH
        ),

    "attempt_directory":
        str(
            ROLLING_ATTEMPT_DIR
        ),

    "audit_manifest":
        str(
            ROLLING_AUDIT_PATH
        ),

    "audit_sha256":
        ROLLING_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_ROLLING_V4_V5_COMPLETE"
    if CELL_40_PASS
    else
    "HB_2024_ROLLING_V4_V5_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_ROLLING_VALIDATION = (
    ROLLING_VALIDATION
)

HB2024_ROLLING_TEST = (
    ROLLING_TEST
)

HB2024_ROLLING_DATE_STATES = (
    DATE_STATES
)

HB2024_V4_V5_POINT_METRICS = (
    POINT_METRICS
)

HB2024_V4_V5_GAP_METRICS = (
    GAP_METRICS
)

HB2024_V4_TO_V5_GAIN_SUMMARY = (
    V5_GAIN_SUMMARY
)

HB2024_ROLLING_ATTEMPT_DIR = (
    ROLLING_ATTEMPT_DIR
)

NOTEBOOK_2_CELL_40_PASS = (
    CELL_40_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2024 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY")
print("=" * 165)


print()
print("OVERALL POINT PERFORMANCE")
print("-" * 165)

display(
    V5_GAIN_SUMMARY
)


print()
print("POINT METRICS")
print("-" * 165)

display(
    POINT_METRICS
)


print()
print("BY STATE GAP")
print("-" * 165)

display(
    GAP_METRICS
)


print()
print("ROLLING STATE UPDATE SUMMARY")
print("-" * 165)

display(
    pd.DataFrame(
        [
            {
                "athlete_dates":
                    int(
                        len(
                            DATE_STATES
                        )
                    ),

                "athletes":
                    int(
                        DATE_STATES[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "mean_standardized_innovation":
                    float(
                        DATE_STATES[
                            "standardized_innovation"
                        ]
                        .mean()
                    ),

                "sd_standardized_innovation":
                    float(
                        DATE_STATES[
                            "standardized_innovation"
                        ]
                        .std(
                            ddof=1
                        )
                    ),

                "mean_kalman_gain":
                    float(
                        DATE_STATES[
                            "kalman_gain"
                        ]
                        .mean()
                    ),
            }
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{ROLLING_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{ROLLING_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{ROLLING_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 165)

print(
    f"Finite predictions             : "
    f"{FINITE_PREDICTION_PASS}"
)

print(
    f"Positive variances             : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"No same-day leakage            : "
    f"{NO_SAME_DAY_LEAKAGE_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Split chronology               : "
    f"{SPLIT_CHRONOLOGY_PASS}"
)

print(
    f"Gap reconciliation             : "
    f"{GAP_RECONCILIATION_PASS}"
)

print(
    f"Maximum gap discrepancy        : "
    f"{GAP_RECONCILIATION_MAX_DIFF:.12e}"
)


print()
print("=" * 165)

if CELL_40_PASS:

    print(
        "CELL 40 STATUS: HB_2024 ROLLING V4/V5 FORECAST PASSED"
    )

else:

    print(
        "CELL 40 STATUS: HB_2024 ROLLING V4/V5 FORECAST REVIEW REQUIRED"
    )

print("=" * 165)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 31
# Original source SHA256: c3b9b43779020120d981cb329b8c2b9b086c25d1338704d85e08773ff969e01c
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 41 — HB_2024 HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY
# ==================================================================================================
#
# TRAIN models are already frozen:
#
#   V4 fitted <= 2023-12-31
#   V5 fitted <= 2023-12-31
#
# Calibration:
#
#   2024-01-01 -> 2024-04-30 VALIDATION only
#
# TEST:
#
#   NOT USED
#
# Model-selection logic is intentionally identical to HB_2023 Cell 33.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import norm


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_40_PASS",

    "HB2024_ROLLING_VALIDATION",
    "HB2024_ROLLING_ATTEMPT_DIR",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_40_PASS:

    raise RuntimeError(
        "Cell 40 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_41_HB2024_V6_MEAN_CALIBRATION"

STAGE_SCHEMA_VERSION = "1.0.0"


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
]


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def gaussian_fit_statistics(
    residual,
    n_mean_parameters,
):

    residual = np.asarray(
        residual,
        dtype=float,
    )

    n = len(
        residual
    )

    rss = float(
        np.sum(
            residual ** 2
        )
    )

    sigma2 = max(
        rss / n,
        1e-12,
    )

    nll = float(
        0.5
        *
        n
        *
        (
            np.log(
                2.0
                *
                np.pi
                *
                sigma2
            )
            +
            1.0
        )
    )

    # Mean parameters + Gaussian residual sigma.
    k = int(
        n_mean_parameters
        +
        1
    )

    aic = float(
        2.0 * k
        +
        2.0 * nll
    )

    if n <= k + 1:

        aicc = np.inf

    else:

        aicc = float(
            aic
            +
            (
                2.0
                *
                k
                *
                (
                    k + 1
                )
            )
            /
            (
                n
                -
                k
                -
                1
            )
        )

    bic = float(
        np.log(n)
        *
        k
        +
        2.0
        *
        nll
    )

    return {
        "n":
            int(n),

        "rss":
            rss,

        "sigma":
            float(
                np.sqrt(
                    sigma2
                )
            ),

        "nll":
            nll,

        "k":
            k,

        "AIC":
            aic,

        "AICc":
            aicc,

        "BIC":
            bic,
    }


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2024_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)

V6_STAGE_ROOT = (
    HB2024_DIR
    /
    "v6_mean_calibration"
)

V6_ATTEMPT_DIR = (
    V6_STAGE_ROOT
    /
    ATTEMPT_ID
)

V6_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. VALIDATION ONLY
# --------------------------------------------------------------------------------------------------

VAL = (
    HB2024_ROLLING_VALIDATION.copy()
)

VAL[
    "sex_group"
] = (
    VAL[
        "model_group"
    ]
    .astype(str)
    .str[0]
)

TEST_TOUCHED = False


# --------------------------------------------------------------------------------------------------
# 6. FIRST VALIDATION DATE PER ATHLETE × MODEL GROUP
# --------------------------------------------------------------------------------------------------

first_date = (
    VAL.groupby(
        [
            "aaAthleteId",
            "model_group",
        ]
    )[
        "result_date_dt"
    ]
    .transform(
        "min"
    )
)


CAL_ROWS = (
    VAL.loc[
        VAL[
            "result_date_dt"
        ].eq(
            first_date
        )
    ]
    .copy()
)


# Average multiple performances on same first athlete/group/date.
CAL = (
    CAL_ROWS.groupby(
        [
            "aaAthleteId",
            "model_group",
            "sex_group",
            "result_date_dt",
        ],
        as_index=False,
    )
    .agg(
        mark_numeric_m=(
            "mark_numeric_m",
            "mean",
        ),

        v5_prediction_mean=(
            "v5_prediction_mean",
            "mean",
        ),

        state_prior_mean=(
            "state_prior_mean",
            "mean",
        ),

        rolling_state_gap_days=(
            "rolling_state_gap_days",
            "first",
        ),

        v4_population_component=(
            "v4_population_component",
            "mean",
        ),

        v4_athlete_intercept_component=(
            "v4_athlete_intercept_component",
            "mean",
        ),

        v4_athlete_slope_component=(
            "v4_athlete_slope_component",
            "mean",
        ),

        n_same_day_rows=(
            "mark_numeric_m",
            "size",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 7. CALIBRATION DOMAIN
# --------------------------------------------------------------------------------------------------

CAL = (
    CAL.loc[
        CAL[
            "rolling_state_gap_days"
        ].between(
            1,
            365,
            inclusive="both",
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


CAL[
    "gap_band"
] = pd.cut(
    CAL[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 8. STRUCTURAL TARGET
# --------------------------------------------------------------------------------------------------

CAL[
    "base_without_athlete_structure"
] = (
    CAL[
        "v5_prediction_mean"
    ]
    -
    CAL[
        "v4_athlete_intercept_component"
    ]
    -
    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "structural_target"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "base_without_athlete_structure"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. CENTER BY SEX × GAP BAND
# --------------------------------------------------------------------------------------------------

CENTER_GROUP = [
    "sex_group",
    "gap_band",
]


for column in [
    "structural_target",
    "v4_athlete_intercept_component",
    "v4_athlete_slope_component",
]:

    group_mean = (
        CAL.groupby(
            CENTER_GROUP,
            observed=True,
        )[
            column
        ]
        .transform(
            "mean"
        )
    )

    CAL[
        f"{column}_centered"
    ] = (
        CAL[
            column
        ]
        -
        group_mean
    )


# --------------------------------------------------------------------------------------------------
# 10. EMPIRICAL HC3 PERSISTENCE DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

Y = (
    CAL[
        "structural_target_centered"
    ]
    .to_numpy(
        dtype=float
    )
)

I = (
    CAL[
        "v4_athlete_intercept_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)

S = (
    CAL[
        "v4_athlete_slope_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)


X = np.column_stack(
    [
        np.ones(
            len(CAL)
        ),
        I,
        S,
    ]
)


beta_ols = np.linalg.lstsq(
    X,
    Y,
    rcond=None,
)[0]


ols_residual = (
    Y
    -
    X @ beta_ols
)


XtX_inv = np.linalg.inv(
    X.T @ X
)


hat_diag = np.sum(
    (
        X @ XtX_inv
    )
    *
    X,
    axis=1,
)


hc3_scale = (
    ols_residual
    /
    np.maximum(
        1.0
        -
        hat_diag,
        1e-8,
    )
) ** 2


HC3_COV = (
    XtX_inv
    @
    (
        X.T
        @
        (
            X
            *
            hc3_scale[
                :,
                None
            ]
        )
    )
    @
    XtX_inv
)


HC3_SE = np.sqrt(
    np.diag(
        HC3_COV
    )
)


HC3_Z = (
    beta_ols
    /
    HC3_SE
)


HC3_P = (
    2.0
    *
    norm.sf(
        np.abs(
            HC3_Z
        )
    )
)


HC3_SUMMARY = pd.DataFrame(
    {
        "term": [
            "intercept",
            "rho_intercept_empirical",
            "rho_slope_empirical",
        ],

        "estimate":
            beta_ols,

        "HC3_SE":
            HC3_SE,

        "z":
            HC3_Z,

        "p_value":
            HC3_P,
    }
)


# --------------------------------------------------------------------------------------------------
# 11. PERSISTENCE FUNCTIONS
# --------------------------------------------------------------------------------------------------

h = (
    CAL[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


def persistence_values(
    model_name,
    params,
    horizon,
):

    horizon = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        c_I, c_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M1_CONST_EXP":

        c_I, tau_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    elif model_name == "M2_EXP_CONST":

        tau_I, c_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M3_EXP_EXP":

        tau_I, tau_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    else:

        raise ValueError(
            model_name
        )


    return rho_I, rho_S


def persistence_sse(
    model_name,
    params,
):

    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )

    prediction = (
        rho_I * I
        +
        rho_S * S
    )

    residual = (
        Y
        -
        prediction
    )

    return float(
        np.sum(
            residual ** 2
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. SAME FOUR CANDIDATE MODELS AS HB_2023
# --------------------------------------------------------------------------------------------------

PERSISTENCE_CONFIG = {

    "M0_CONST_CONST": {
        "bounds": [
            (0.0, 1.5),
            (0.0, 1.5),
        ],

        "starts": [
            (0.50, 0.50),
            (0.75, 0.75),
            (1.00, 1.00),
            (1.25, 1.25),
            (0.75, 1.00),
            (1.00, 0.75),
        ],

        "parameter_names": [
            "c_intercept",
            "c_slope",
        ],
    },

    "M1_CONST_EXP": {
        "bounds": [
            (0.0, 1.5),
            (1.0, 5000.0),
        ],

        "starts": [
            (0.75, 60.0),
            (0.90, 120.0),
            (1.00, 365.0),
            (1.10, 730.0),
            (1.00, 1500.0),
        ],

        "parameter_names": [
            "c_intercept",
            "tau_slope_days",
        ],
    },

    "M2_EXP_CONST": {
        "bounds": [
            (1.0, 5000.0),
            (0.0, 1.5),
        ],

        "starts": [
            (60.0, 0.75),
            (120.0, 0.90),
            (365.0, 1.00),
            (730.0, 1.10),
            (1500.0, 1.00),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "c_slope",
        ],
    },

    "M3_EXP_EXP": {
        "bounds": [
            (1.0, 5000.0),
            (1.0, 5000.0),
        ],

        "starts": [
            (60.0, 60.0),
            (120.0, 120.0),
            (365.0, 365.0),
            (730.0, 365.0),
            (365.0, 730.0),
            (1500.0, 1500.0),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "tau_slope_days",
        ],
    },
}


# --------------------------------------------------------------------------------------------------
# 13. FIT PERSISTENCE CANDIDATES
# --------------------------------------------------------------------------------------------------

candidate_rows = []

candidate_best = {}


for model_name, config in PERSISTENCE_CONFIG.items():

    results = []


    for x0 in config[
        "starts"
    ]:

        result = minimize(
            lambda p:
                persistence_sse(
                    model_name,
                    p,
                ),

            x0=np.asarray(
                x0,
                dtype=float,
            ),

            method="L-BFGS-B",

            bounds=config[
                "bounds"
            ],

            options={
                "maxiter":
                    3000,

                "ftol":
                    1e-12,

                "gtol":
                    1e-8,
            },
        )

        results.append(
            result
        )


    successful = [
        result
        for result in results
        if (
            result.success
            and
            np.isfinite(
                result.fun
            )
        )
    ]


    if not successful:

        raise RuntimeError(
            f"No successful optimization for {model_name}."
        )


    best = min(
        successful,
        key=lambda r:
            r.fun,
    )


    params = np.asarray(
        best.x,
        dtype=float,
    )


    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )


    residual = (
        Y
        -
        (
            rho_I * I
            +
            rho_S * S
        )
    )


    stats = gaussian_fit_statistics(
        residual,
        n_mean_parameters=len(
            params
        ),
    )


    parameter_dict = {
        name:
            float(value)

        for name, value
        in zip(
            config[
                "parameter_names"
            ],
            params,
        )
    }


    candidate_rows.append(
        {
            "model":
                model_name,

            "successful_starts":
                int(
                    sum(
                        r.success
                        for r in results
                    )
                ),

            **parameter_dict,

            **stats,
        }
    )


    candidate_best[
        model_name
    ] = {
        "params":
            params,

        "parameter_dict":
            parameter_dict,
    }


PERSISTENCE_CANDIDATES = pd.DataFrame(
    candidate_rows
)


PERSISTENCE_CANDIDATES[
    "delta_AICc"
] = (
    PERSISTENCE_CANDIDATES[
        "AICc"
    ]
    -
    PERSISTENCE_CANDIDATES[
        "AICc"
    ].min()
)


PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_PERSISTENCE_MODEL = str(
    PERSISTENCE_CANDIDATES.loc[
        0,
        "model",
    ]
)


SELECTED_PERSISTENCE_PARAMS = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "params"
    ]
)


SELECTED_PERSISTENCE_PARAMETER_DICT = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "parameter_dict"
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. APPLY SELECTED PERSISTENCE TO RAW CALIBRATION SAMPLE
# --------------------------------------------------------------------------------------------------

rho_I_raw, rho_S_raw = persistence_values(
    SELECTED_PERSISTENCE_MODEL,
    SELECTED_PERSISTENCE_PARAMS,
    CAL[
        "rolling_state_gap_days"
    ].to_numpy(
        dtype=float
    ),
)


CAL[
    "rho_intercept"
] = rho_I_raw

CAL[
    "rho_slope"
] = rho_S_raw


CAL[
    "v6_without_offset"
] = (
    CAL[
        "base_without_athlete_structure"
    ]
    +
    CAL[
        "rho_intercept"
    ]
    *
    CAL[
        "v4_athlete_intercept_component"
    ]
    +
    CAL[
        "rho_slope"
    ]
    *
    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "offset_target"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v6_without_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. OFFSET MODEL COMPARISON
# --------------------------------------------------------------------------------------------------

offset_candidate_rows = []

offset_specs = {}


# NONE
residual = (
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
)

stats = gaussian_fit_statistics(
    residual,
    n_mean_parameters=0,
)

offset_candidate_rows.append(
    {
        "offset_model":
            "NONE",

        **stats,
    }
)

offset_specs[
    "NONE"
] = {}


# COMMON
common_offset = float(
    CAL[
        "offset_target"
    ]
    .mean()
)


residual = (
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    common_offset
)


stats = gaussian_fit_statistics(
    residual,
    n_mean_parameters=1,
)


offset_candidate_rows.append(
    {
        "offset_model":
            "COMMON",

        **stats,
    }
)


offset_specs[
    "COMMON"
] = {
    "COMMON":
        common_offset
}


# SEX
sex_offsets = (
    CAL.groupby(
        "sex_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


pred_offset = (
    CAL[
        "sex_group"
    ]
    .map(
        sex_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    pred_offset,

    n_mean_parameters=len(
        sex_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "SEX",

        **stats,
    }
)


offset_specs[
    "SEX"
] = {
    str(k):
        float(v)

    for k, v
    in sex_offsets.items()
}


# GROUP
group_offsets = (
    CAL.groupby(
        "model_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


pred_offset = (
    CAL[
        "model_group"
    ]
    .map(
        group_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    pred_offset,

    n_mean_parameters=len(
        group_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "GROUP",

        **stats,
    }
)


offset_specs[
    "GROUP"
] = {
    str(k):
        float(v)

    for k, v
    in group_offsets.items()
}


OFFSET_CANDIDATES = pd.DataFrame(
    offset_candidate_rows
)


OFFSET_CANDIDATES[
    "delta_AICc"
] = (
    OFFSET_CANDIDATES[
        "AICc"
    ]
    -
    OFFSET_CANDIDATES[
        "AICc"
    ].min()
)


OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_OFFSET_MODEL = str(
    OFFSET_CANDIDATES.loc[
        0,
        "offset_model",
    ]
)


SELECTED_OFFSETS = (
    offset_specs[
        SELECTED_OFFSET_MODEL
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. APPLY SELECTED OFFSET
# --------------------------------------------------------------------------------------------------

if SELECTED_OFFSET_MODEL == "NONE":

    CAL[
        "v6_offset"
    ] = 0.0


elif SELECTED_OFFSET_MODEL == "COMMON":

    CAL[
        "v6_offset"
    ] = float(
        SELECTED_OFFSETS[
            "COMMON"
        ]
    )


elif SELECTED_OFFSET_MODEL == "SEX":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "sex_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


elif SELECTED_OFFSET_MODEL == "GROUP":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "model_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        SELECTED_OFFSET_MODEL
    )


CAL[
    "v6_prediction_mean"
] = (
    CAL[
        "v6_without_offset"
    ]
    +
    CAL[
        "v6_offset"
    ]
)


CAL[
    "v5_error"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v5_prediction_mean"
    ]
)


CAL[
    "v6_error"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v6_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. CALIBRATION METRICS
# --------------------------------------------------------------------------------------------------

V5_MAE = float(
    CAL[
        "v5_error"
    ]
    .abs()
    .mean()
)


V6_MAE = float(
    CAL[
        "v6_error"
    ]
    .abs()
    .mean()
)


V5_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v5_error"
            ] ** 2
        )
    )
)


V6_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v6_error"
            ] ** 2
        )
    )
)


CALIBRATION_METRICS = pd.DataFrame(
    [
        {
            "n":
                int(
                    len(CAL)
                ),

            "athletes":
                int(
                    CAL[
                        "aaAthleteId"
                    ].nunique()
                ),

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V5_to_V6_MAE_gain_m":
                V5_MAE
                -
                V6_MAE,

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V5_to_V6_RMSE_gain_m":
                V5_RMSE
                -
                V6_RMSE,

            "V6_Bias":
                float(
                    CAL[
                        "v6_error"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. PERSISTENCE REFERENCE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


REF_RHO_I, REF_RHO_S = persistence_values(
    SELECTED_PERSISTENCE_MODEL,
    SELECTED_PERSISTENCE_PARAMS,
    REFERENCE_DAYS,
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS.astype(
                int
            ),

        "rho_intercept":
            REF_RHO_I,

        "rho_slope":
            REF_RHO_S,
    }
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

CALIBRATION_DATE_PASS = bool(
    CAL[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2024-01-01"
        ),
        pd.Timestamp(
            "2024-04-30"
        ),
        inclusive="both",
    )
    .all()
)


GAP_DOMAIN_PASS = bool(
    CAL[
        "rolling_state_gap_days"
    ]
    .between(
        1,
        365,
        inclusive="both",
    )
    .all()
)


ONE_FIRST_DATE_PASS = bool(
    not CAL.duplicated(
        subset=[
            "aaAthleteId",
            "model_group",
        ]
    )
    .any()
)


FINITE_PASS = bool(
    np.isfinite(
        CAL[
            [
                "structural_target",
                "rho_intercept",
                "rho_slope",
                "v6_offset",
                "v6_prediction_mean",
                "v6_error",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


PERSISTENCE_SELECTION_PASS = bool(
    len(
        PERSISTENCE_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        PERSISTENCE_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


OFFSET_SELECTION_PASS = bool(
    len(
        OFFSET_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        OFFSET_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


TEST_NOT_USED_PASS = bool(
    not TEST_TOUCHED
)


CELL_41_PASS = bool(
    CALIBRATION_DATE_PASS
    and
    GAP_DOMAIN_PASS
    and
    ONE_FIRST_DATE_PASS
    and
    FINITE_PASS
    and
    PERSISTENCE_SELECTION_PASS
    and
    OFFSET_SELECTION_PASS
    and
    TEST_NOT_USED_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

CALIBRATION_SAMPLE_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_validation_calibration_sample.parquet"
)

PERSISTENCE_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_persistence_candidates.parquet"
)

OFFSET_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_offset_candidates.parquet"
)

HC3_SUMMARY_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_empirical_persistence_hc3.parquet"
)

V6_MEAN_SPEC_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_spec.json"
)

V6_AUDIT_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_calibration_audit.json"
)


CAL.to_parquet(
    CALIBRATION_SAMPLE_PATH,
    index=False,
)

PERSISTENCE_CANDIDATES.to_parquet(
    PERSISTENCE_CANDIDATES_PATH,
    index=False,
)

OFFSET_CANDIDATES.to_parquet(
    OFFSET_CANDIDATES_PATH,
    index=False,
)

HC3_SUMMARY.to_parquet(
    HC3_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. FREEZE MODEL SPEC
# --------------------------------------------------------------------------------------------------

V6_MEAN_SPEC = {
    "model":
        "HISTORICAL_V6_FULL_STRUCTURAL_CORRECTION",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "calibration_window": {
        "start":
            "2024-01-01",

        "end":
            "2024-04-30",
    },

    "calibration_rule":
        (
            "First validation date per athlete × model_group; "
            "gap restricted to 1–365 days."
        ),

    "selected_persistence_model":
        SELECTED_PERSISTENCE_MODEL,

    "selected_persistence_parameters":
        SELECTED_PERSISTENCE_PARAMETER_DICT,

    "selected_offset_model":
        SELECTED_OFFSET_MODEL,

    "selected_offsets":
        SELECTED_OFFSETS,

    "formula":
        (
            "V6_FULL = "
            "V5 - athlete_intercept - athlete_slope_component "
            "+ rho_I(h)*athlete_intercept "
            "+ rho_S(h)*athlete_slope_component "
            "+ offset"
        ),

    "test_used":
        False,

    "architecture_changed_from_HB_2023":
        False,
}


atomic_json_write(
    V6_MEAN_SPEC,
    V6_MEAN_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. PROVENANCE
# --------------------------------------------------------------------------------------------------

rolling_audit_path = (
    Path(
        HB2024_ROLLING_ATTEMPT_DIR
    )
    /
    "rolling_forecast_audit.json"
)


PARENT_ARTIFACTS = {
    "rolling_forecast_audit": {
        "path":
            str(
                rolling_audit_path
            ),

        "exists":
            bool(
                rolling_audit_path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    rolling_audit_path
                )
                if rolling_audit_path.exists()
                else None
            ),
    }
}


output_paths = {
    "calibration_sample":
        CALIBRATION_SAMPLE_PATH,

    "persistence_candidates":
        PERSISTENCE_CANDIDATES_PATH,

    "offset_candidates":
        OFFSET_CANDIDATES_PATH,

    "hc3_summary":
        HC3_SUMMARY_PATH,

    "v6_mean_spec":
        V6_MEAN_SPEC_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 23. AUDIT
# --------------------------------------------------------------------------------------------------

V6_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V6_MEAN_CALIBRATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_41_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Second independent historical validation-only "
            "V6 structural-persistence calibration."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "test_used":
        False,

    "sample": {
        "rows":
            int(
                len(CAL)
            ),

        "athletes":
            int(
                CAL[
                    "aaAthleteId"
                ].nunique()
            ),

        "minimum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ].min()
            ),

        "maximum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ].max()
            ),
    },

    "selected_model":
        V6_MEAN_SPEC,

    "calibration_metrics":
        CALIBRATION_METRICS.iloc[
            0
        ].to_dict(),

    "qa": {
        "calibration_date_pass":
            CALIBRATION_DATE_PASS,

        "gap_domain_pass":
            GAP_DOMAIN_PASS,

        "one_first_date_per_athlete_group":
            ONE_FIRST_DATE_PASS,

        "finite_outputs":
            FINITE_PASS,

        "persistence_selection":
            PERSISTENCE_SELECTION_PASS,

        "offset_selection":
            OFFSET_SELECTION_PASS,

        "test_not_used":
            TEST_NOT_USED_PASS,

        "cell_pass":
            CELL_41_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    V6_AUDIT,
    V6_AUDIT_PATH,
)


V6_AUDIT_SHA256 = sha256_file(
    V6_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_V6_POINTER_PATH = (
    V6_STAGE_ROOT
    /
    "LATEST_V6_MEAN_CALIBRATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V6_MEAN_CALIBRATION",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_41_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V6_ATTEMPT_DIR
            ),

        "v6_mean_spec":
            str(
                V6_MEAN_SPEC_PATH
            ),

        "audit_manifest":
            str(
                V6_AUDIT_PATH
            ),

        "audit_sha256":
            V6_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_V6_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 25. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v6_mean_calibration"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_41_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            V6_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_V6_POINTER_PATH
        ),

    "mean_spec":
        str(
            V6_MEAN_SPEC_PATH
        ),

    "audit_manifest":
        str(
            V6_AUDIT_PATH
        ),

    "audit_sha256":
        V6_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_V6_MEAN_CALIBRATION_COMPLETE"
    if CELL_41_PASS
    else
    "HB_2024_V6_MEAN_CALIBRATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 26. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_V6_CALIBRATION_SAMPLE = CAL

HB2024_V6_PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES
)

HB2024_V6_OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES
)

HB2024_V6_MEAN_SPEC = (
    V6_MEAN_SPEC
)

HB2024_V6_ATTEMPT_DIR = (
    V6_ATTEMPT_DIR
)

HB2024_V6_MEAN_SPEC_PATH = (
    V6_MEAN_SPEC_PATH
)

NOTEBOOK_2_CELL_41_PASS = (
    CELL_41_PASS
)


# --------------------------------------------------------------------------------------------------
# 27. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2024 — HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY")
print("=" * 160)


print()
print("CALIBRATION SAMPLE")
print("-" * 160)

print(
    f"Rows                           : "
    f"{len(CAL):,}"
)

print(
    f"Athletes                       : "
    f"{CAL['aaAthleteId'].nunique():,}"
)

print(
    f"Gap range                      : "
    f"{CAL['rolling_state_gap_days'].min():.0f}"
    f" -> "
    f"{CAL['rolling_state_gap_days'].max():.0f} days"
)

print(
    f"TEST used                      : "
    f"{TEST_TOUCHED}"
)


print()
print("EMPIRICAL CENTERED HC3 PERSISTENCE")
print("-" * 160)

display(
    HC3_SUMMARY
)


print()
print("PERSISTENCE MODEL COMPARISON")
print("-" * 160)

display(
    PERSISTENCE_CANDIDATES
)


print()
print(
    f"Selected persistence model     : "
    f"{SELECTED_PERSISTENCE_MODEL}"
)

print(
    f"Selected parameters            : "
    f"{SELECTED_PERSISTENCE_PARAMETER_DICT}"
)


print()
print("SELECTED PERSISTENCE BY HORIZON")
print("-" * 160)

display(
    PERSISTENCE_REFERENCE
)


print()
print("OFFSET MODEL COMPARISON")
print("-" * 160)

display(
    OFFSET_CANDIDATES
)


print()
print(
    f"Selected offset model          : "
    f"{SELECTED_OFFSET_MODEL}"
)

print(
    f"Selected offsets               : "
    f"{SELECTED_OFFSETS}"
)


print()
print("VALIDATION CALIBRATION METRICS")
print("-" * 160)

display(
    CALIBRATION_METRICS
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{V6_ATTEMPT_DIR}"
)

print(
    f"Mean specification             : "
    f"{V6_MEAN_SPEC_PATH}"
)

print(
    f"Audit manifest                 : "
    f"{V6_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{V6_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"VALIDATION dates only          : "
    f"{CALIBRATION_DATE_PASS}"
)

print(
    f"Gap domain 1–365 days          : "
    f"{GAP_DOMAIN_PASS}"
)

print(
    f"First athlete/group date only  : "
    f"{ONE_FIRST_DATE_PASS}"
)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"Persistence candidates valid   : "
    f"{PERSISTENCE_SELECTION_PASS}"
)

print(
    f"Offset candidates valid        : "
    f"{OFFSET_SELECTION_PASS}"
)

print(
    f"TEST untouched                 : "
    f"{TEST_NOT_USED_PASS}"
)


print()
print("=" * 160)

if CELL_41_PASS:

    print(
        "CELL 41 STATUS: HB_2024 HISTORICAL V6 MEAN CALIBRATION PASSED"
    )

else:

    print(
        "CELL 41 STATUS: HB_2024 V6 MEAN CALIBRATION REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 32
# Original source SHA256: dfdc4743d0e5001568df42b040b43d9fca25f59800e11f993b22ec3c651b07cf
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 42 — HB_2024 UNTOUCHED TEST: V5 vs V6 vs V6.1 OU-FRESHNESS BLEND
# ==================================================================================================
#
# Frozen before TEST:
#
#   V4          fitted <= 2023-12-31
#   V5 OU       fitted <= 2023-12-31
#   V6 mean     calibrated on 2024-01-01 -> 2024-04-30 VALIDATION only
#
# TEST:
#
#   2024-05-01 -> 2024-08-30
#
# V6.1:
#
#   w(h) = 1 - exp(-h / tau_V5)
#
#   V6.1 = V5 + w(h) * (V6_FULL - V5)
#
# No V6.1 parameters are fitted.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_41_PASS",

    "HB2024_ROLLING_TEST",

    "HB2024_V6_MEAN_SPEC",
    "HB2024_V6_MEAN_SPEC_PATH",
    "HB2024_V6_ATTEMPT_DIR",

    "HB2024_ROLLING_ATTEMPT_DIR",

    "HB2024_V5_TAU_DAYS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_41_PASS:

    raise RuntimeError(
        "Cell 41 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

CELL_ID = "CELL_42_HB2024_TEST_V6_V61"

STAGE_SCHEMA_VERSION = "1.0.0"

BOOTSTRAP_SEED = 42
BOOTSTRAP_REPS = 5000


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


BOOTSTRAP_HORIZONS = {
    "ALL":
        (0, np.inf),

    "01-30":
        (0, 30),

    "31-60":
        (30, 60),

    "61-120":
        (60, 120),

    "121-365":
        (120, 365),

    "366+":
        (365, np.inf),
}


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2024_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)

TEST_STAGE_ROOT = (
    HB2024_DIR
    /
    "v61_test_evaluation"
)

TEST_ATTEMPT_DIR = (
    TEST_STAGE_ROOT
    /
    ATTEMPT_ID
)

TEST_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. FROZEN TEST + SPECIFICATION
# --------------------------------------------------------------------------------------------------

TEST = (
    HB2024_ROLLING_TEST.copy()
    .reset_index(
        drop=True
    )
)


V6_SPEC = dict(
    HB2024_V6_MEAN_SPEC
)


TAU_V5 = float(
    HB2024_V5_TAU_DAYS
)


SELECTED_MODEL = (
    V6_SPEC[
        "selected_persistence_model"
    ]
)


PERSISTENCE_PARAMS = (
    V6_SPEC[
        "selected_persistence_parameters"
    ]
)


OFFSET_MODEL = (
    V6_SPEC[
        "selected_offset_model"
    ]
)


OFFSETS = (
    V6_SPEC[
        "selected_offsets"
    ]
)


TEST[
    "sex_group"
] = (
    TEST[
        "model_group"
    ]
    .astype(str)
    .str[0]
)


# --------------------------------------------------------------------------------------------------
# 6. FROZEN V6 PERSISTENCE
# --------------------------------------------------------------------------------------------------

def frozen_persistence(
    model_name,
    params,
    horizon,
):

    h = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M1_CONST_EXP":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    elif model_name == "M2_EXP_CONST":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M3_EXP_EXP":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    else:

        raise RuntimeError(
            f"Unknown persistence model: {model_name}"
        )


    return rho_I, rho_S


# --------------------------------------------------------------------------------------------------
# 7. APPLY FROZEN V6 STRUCTURAL SPECIFICATION
# --------------------------------------------------------------------------------------------------

h = (
    TEST[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


rho_I, rho_S = frozen_persistence(
    SELECTED_MODEL,
    PERSISTENCE_PARAMS,
    h,
)


TEST[
    "v6_rho_intercept"
] = rho_I

TEST[
    "v6_rho_slope"
] = rho_S


TEST[
    "v6_base_without_athlete_structure"
] = (
    TEST[
        "v5_prediction_mean"
    ]
    -
    TEST[
        "v4_athlete_intercept_component"
    ]
    -
    TEST[
        "v4_athlete_slope_component"
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. APPLY FROZEN OFFSET
# --------------------------------------------------------------------------------------------------

if OFFSET_MODEL == "NONE":

    TEST[
        "v6_offset"
    ] = 0.0


elif OFFSET_MODEL == "COMMON":

    TEST[
        "v6_offset"
    ] = float(
        OFFSETS[
            "COMMON"
        ]
    )


elif OFFSET_MODEL == "SEX":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "sex_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


elif OFFSET_MODEL == "GROUP":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "model_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        f"Unknown offset model: {OFFSET_MODEL}"
    )


if TEST[
    "v6_offset"
].isna().any():

    raise RuntimeError(
        "Frozen V6 offset produced missing values."
    )


# --------------------------------------------------------------------------------------------------
# 9. FULL V6 MEAN
# --------------------------------------------------------------------------------------------------

TEST[
    "v6_prediction_mean"
] = (
    TEST[
        "v6_base_without_athlete_structure"
    ]

    +

    TEST[
        "v6_rho_intercept"
    ]
    *
    TEST[
        "v4_athlete_intercept_component"
    ]

    +

    TEST[
        "v6_rho_slope"
    ]
    *
    TEST[
        "v4_athlete_slope_component"
    ]

    +

    TEST[
        "v6_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. V6.1 — TRAIN-ONLY OU FRESHNESS BLEND
# --------------------------------------------------------------------------------------------------

TEST[
    "v61_ou_phi"
] = np.exp(
    -
    TEST[
        "rolling_state_gap_days"
    ]
    /
    TAU_V5
)


TEST[
    "v61_weight"
] = (
    1.0
    -
    TEST[
        "v61_ou_phi"
    ]
)


TEST[
    "v61_prediction_mean"
] = (
    TEST[
        "v5_prediction_mean"
    ]

    +

    TEST[
        "v61_weight"
    ]
    *
    (
        TEST[
            "v6_prediction_mean"
        ]
        -
        TEST[
            "v5_prediction_mean"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ERRORS
# --------------------------------------------------------------------------------------------------

for model, column in {
    "v5":
        "v5_prediction_mean",

    "v6":
        "v6_prediction_mean",

    "v61":
        "v61_prediction_mean",
}.items():

    TEST[
        f"{model}_error"
    ] = (
        TEST[
            "mark_numeric_m"
        ]
        -
        TEST[
            column
        ]
    )


# --------------------------------------------------------------------------------------------------
# 12. GAP BAND
# --------------------------------------------------------------------------------------------------

TEST[
    "state_gap_band"
] = pd.cut(
    TEST[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 13. METRIC HELPERS
# --------------------------------------------------------------------------------------------------

def metric_row(
    df,
    model,
):

    error = (
        df[
            f"{model}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    return {
        "model":
            (
                model.upper()
                if model != "v61"
                else "V6.1"
            ),

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ].nunique()
            ),

        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "Bias":
            float(
                np.mean(
                    error
                )
            ),
    }


def pairwise_gain(
    df,
    baseline,
    candidate,
):

    e0 = (
        df[
            f"{baseline}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    e1 = (
        df[
            f"{candidate}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    mae0 = float(
        np.mean(
            np.abs(
                e0
            )
        )
    )

    mae1 = float(
        np.mean(
            np.abs(
                e1
            )
        )
    )

    rmse0 = float(
        np.sqrt(
            np.mean(
                e0 ** 2
            )
        )
    )

    rmse1 = float(
        np.sqrt(
            np.mean(
                e1 ** 2
            )
        )
    )

    return {
        "baseline":
            (
                baseline.upper()
                if baseline != "v61"
                else "V6.1"
            ),

        "candidate":
            (
                candidate.upper()
                if candidate != "v61"
                else "V6.1"
            ),

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ].nunique()
            ),

        "baseline_MAE":
            mae0,

        "candidate_MAE":
            mae1,

        "MAE_gain_m":
            mae0
            -
            mae1,

        "baseline_RMSE":
            rmse0,

        "candidate_RMSE":
            rmse1,

        "RMSE_gain_m":
            rmse0
            -
            rmse1,
    }


# --------------------------------------------------------------------------------------------------
# 14. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

OVERALL_METRICS = pd.DataFrame(
    [
        metric_row(
            TEST,
            model,
        )

        for model in [
            "v5",
            "v6",
            "v61",
        ]
    ]
)


PAIRWISE_OVERALL = pd.DataFrame(
    [
        pairwise_gain(
            TEST,
            "v5",
            "v6",
        ),

        pairwise_gain(
            TEST,
            "v5",
            "v61",
        ),

        pairwise_gain(
            TEST,
            "v6",
            "v61",
        ),
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. GAP METRICS + GAINS
# --------------------------------------------------------------------------------------------------

gap_metric_rows = []
gap_gain_rows = []


for gap_band, g in TEST.groupby(
    "state_gap_band",
    observed=True,
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )

        row[
            "state_gap_band"
        ] = str(
            gap_band
        )

        row[
            "mean_gap_days"
        ] = float(
            g[
                "rolling_state_gap_days"
            ].mean()
        )

        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ].mean()
        )

        gap_metric_rows.append(
            row
        )


    for baseline, candidate in [
        ("v5", "v6"),
        ("v5", "v61"),
        ("v6", "v61"),
    ]:

        row = pairwise_gain(
            g,
            baseline,
            candidate,
        )

        row[
            "state_gap_band"
        ] = str(
            gap_band
        )

        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ].mean()
        )

        gap_gain_rows.append(
            row
        )


GAP_METRICS = pd.DataFrame(
    gap_metric_rows
)

GAP_GAINS = pd.DataFrame(
    gap_gain_rows
)


# --------------------------------------------------------------------------------------------------
# 16. MODEL GROUP METRICS
# --------------------------------------------------------------------------------------------------

group_rows = []


for model_group, g in TEST.groupby(
    "model_group"
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )

        row[
            "model_group"
        ] = model_group

        group_rows.append(
            row
        )


GROUP_METRICS = pd.DataFrame(
    group_rows
)


# --------------------------------------------------------------------------------------------------
# 17. ATHLETE-CLUSTER BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def cluster_bootstrap_pair(
    df,
    baseline,
    candidate,
    reps=5000,
    seed=42,
):

    d = (
        df[
            [
                "aaAthleteId",
                f"{baseline}_error",
                f"{candidate}_error",
            ]
        ]
        .copy()
    )


    d[
        "abs_gain"
    ] = (
        d[
            f"{baseline}_error"
        ].abs()
        -
        d[
            f"{candidate}_error"
        ].abs()
    )


    d[
        "baseline_sq"
    ] = (
        d[
            f"{baseline}_error"
        ] ** 2
    )


    d[
        "candidate_sq"
    ] = (
        d[
            f"{candidate}_error"
        ] ** 2
    )


    cluster = (
        d.groupby(
            "aaAthleteId",
            as_index=False,
        )
        .agg(
            n=(
                "abs_gain",
                "size",
            ),

            abs_gain_sum=(
                "abs_gain",
                "sum",
            ),

            baseline_sq_sum=(
                "baseline_sq",
                "sum",
            ),

            candidate_sq_sum=(
                "candidate_sq",
                "sum",
            ),
        )
    )


    n_cluster = len(
        cluster
    )


    n = (
        cluster[
            "n"
        ]
        .to_numpy(
            dtype=float
        )
    )


    abs_gain_sum = (
        cluster[
            "abs_gain_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    baseline_sq_sum = (
        cluster[
            "baseline_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    candidate_sq_sum = (
        cluster[
            "candidate_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rng = np.random.default_rng(
        seed
    )


    mae_boot = np.empty(
        reps,
        dtype=float,
    )


    rmse_boot = np.empty(
        reps,
        dtype=float,
    )


    for b in range(reps):

        idx = rng.integers(
            0,
            n_cluster,
            size=n_cluster,
        )

        total_n = float(
            n[idx].sum()
        )

        mae_boot[b] = float(
            abs_gain_sum[
                idx
            ].sum()
            /
            total_n
        )

        baseline_rmse = np.sqrt(
            baseline_sq_sum[
                idx
            ].sum()
            /
            total_n
        )

        candidate_rmse = np.sqrt(
            candidate_sq_sum[
                idx
            ].sum()
            /
            total_n
        )

        rmse_boot[b] = (
            baseline_rmse
            -
            candidate_rmse
        )


    observed = pairwise_gain(
        df,
        baseline,
        candidate,
    )


    return {
        "baseline":
            observed[
                "baseline"
            ],

        "candidate":
            observed[
                "candidate"
            ],

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                n_cluster
            ),

        "MAE_gain_m":
            observed[
                "MAE_gain_m"
            ],

        "MAE_CI_low":
            float(
                np.quantile(
                    mae_boot,
                    0.025,
                )
            ),

        "MAE_CI_high":
            float(
                np.quantile(
                    mae_boot,
                    0.975,
                )
            ),

        "MAE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    mae_boot > 0
                )
            ),

        "RMSE_gain_m":
            observed[
                "RMSE_gain_m"
            ],

        "RMSE_CI_low":
            float(
                np.quantile(
                    rmse_boot,
                    0.025,
                )
            ),

        "RMSE_CI_high":
            float(
                np.quantile(
                    rmse_boot,
                    0.975,
                )
            ),

        "RMSE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    rmse_boot > 0
                )
            ),
    }


# --------------------------------------------------------------------------------------------------
# 18. BOOTSTRAP OVERALL + HORIZONS
# --------------------------------------------------------------------------------------------------

bootstrap_rows = []


for horizon_name, (
    lower,
    upper,
) in BOOTSTRAP_HORIZONS.items():

    if horizon_name == "ALL":

        subset = TEST.copy()

    else:

        subset = TEST.loc[
            (
                TEST[
                    "rolling_state_gap_days"
                ]
                >
                lower
            )
            &
            (
                TEST[
                    "rolling_state_gap_days"
                ]
                <=
                upper
            )
        ].copy()


    if len(subset) == 0:
        continue


    for baseline, candidate in [
        ("v5", "v6"),
        ("v5", "v61"),
        ("v6", "v61"),
    ]:

        row = cluster_bootstrap_pair(
            subset,
            baseline,
            candidate,
            reps=BOOTSTRAP_REPS,
            seed=BOOTSTRAP_SEED,
        )

        row[
            "horizon"
        ] = horizon_name

        bootstrap_rows.append(
            row
        )


BOOTSTRAP_SUMMARY = pd.DataFrame(
    bootstrap_rows
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

FINITE_PASS = bool(
    np.isfinite(
        TEST[
            [
                "v5_prediction_mean",
                "v6_prediction_mean",
                "v61_prediction_mean",

                "v5_error",
                "v6_error",
                "v61_error",

                "v6_rho_intercept",
                "v6_rho_slope",

                "v61_ou_phi",
                "v61_weight",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


WEIGHT_BOUNDS_PASS = bool(
    TEST[
        "v61_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


TEST_DATE_PASS = bool(
    TEST[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2024-05-01"
        ),
        pd.Timestamp(
            "2024-08-30"
        ),
        inclusive="both",
    )
    .all()
)


ROW_ACCOUNTING_PASS = bool(
    len(TEST)
    ==
    len(
        HB2024_ROLLING_TEST
    )
)


NO_NEW_PARAMETERS_PASS = True


V61_ENDPOINT_LOGIC_PASS = bool(
    TEST.loc[
        TEST[
            "rolling_state_gap_days"
        ]
        <=
        30,
        "v61_weight",
    ].mean()
    <
    0.35

    and

    TEST.loc[
        TEST[
            "rolling_state_gap_days"
        ]
        >
        365,
        "v61_weight",
    ].mean()
    >
    0.95
)


CELL_42_PASS = bool(
    FINITE_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    TEST_DATE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    NO_NEW_PARAMETERS_PASS
    and
    V61_ENDPOINT_LOGIC_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

TEST_PREDICTIONS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "hb2024_test_v5_v6_v61_predictions.parquet"
)

OVERALL_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "overall_metrics.parquet"
)

PAIRWISE_OVERALL_PATH = (
    TEST_ATTEMPT_DIR
    /
    "pairwise_overall_gains.parquet"
)

GAP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_metrics.parquet"
)

GAP_GAINS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_gains.parquet"
)

GROUP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "group_metrics.parquet"
)

BOOTSTRAP_PATH = (
    TEST_ATTEMPT_DIR
    /
    "athlete_cluster_bootstrap.parquet"
)

TEST_AUDIT_PATH = (
    TEST_ATTEMPT_DIR
    /
    "v61_test_evaluation_audit.json"
)


TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)

OVERALL_METRICS.to_parquet(
    OVERALL_METRICS_PATH,
    index=False,
)

PAIRWISE_OVERALL.to_parquet(
    PAIRWISE_OVERALL_PATH,
    index=False,
)

GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)

GAP_GAINS.to_parquet(
    GAP_GAINS_PATH,
    index=False,
)

GROUP_METRICS.to_parquet(
    GROUP_METRICS_PATH,
    index=False,
)

BOOTSTRAP_SUMMARY.to_parquet(
    BOOTSTRAP_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. PROVENANCE + AUDIT
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v6_mean_spec":
        Path(
            HB2024_V6_MEAN_SPEC_PATH
        ),

    "rolling_test_predictions":
        (
            Path(
                HB2024_ROLLING_ATTEMPT_DIR
            )
            /
            "test_rolling_predictions.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(path)
                if path.exists()
                else None
            ),
    }


output_paths = {
    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "overall_metrics":
        OVERALL_METRICS_PATH,

    "pairwise_overall":
        PAIRWISE_OVERALL_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "gap_gains":
        GAP_GAINS_PATH,

    "group_metrics":
        GROUP_METRICS_PATH,

    "athlete_cluster_bootstrap":
        BOOTSTRAP_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


TEST_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V61_UNTOUCHED_TEST_EVALUATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_42_PASS
            else
            "FAILED"
        ),

    "evaluation_window": {
        "start":
            "2024-05-01",

        "end":
            "2024-08-30",
    },

    "test_role":
        (
            "Untouched historical TEST. "
            "No V6 or V6.1 parameters fitted on this period."
        ),

    "frozen_v6_spec": {
        "persistence_model":
            SELECTED_MODEL,

        "persistence_parameters":
            PERSISTENCE_PARAMS,

        "offset_model":
            OFFSET_MODEL,

        "offsets":
            OFFSETS,
    },

    "v61_spec": {
        "formula":
            "V5 + (1-exp(-gap/tau_V5))*(V6_FULL-V5)",

        "tau_days":
            TAU_V5,

        "tau_source":
            "HB_2024 TRAIN-only V5 OU fit",

        "new_fitted_parameters":
            0,
    },

    "bootstrap": {
        "method":
            "paired athlete-cluster bootstrap",

        "replications":
            BOOTSTRAP_REPS,

        "seed":
            BOOTSTRAP_SEED,
    },

    "qa": {
        "finite":
            FINITE_PASS,

        "weight_bounds":
            WEIGHT_BOUNDS_PASS,

        "test_dates":
            TEST_DATE_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "no_new_parameters":
            NO_NEW_PARAMETERS_PASS,

        "endpoint_logic":
            V61_ENDPOINT_LOGIC_PASS,

        "cell_pass":
            CELL_42_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    TEST_AUDIT,
    TEST_AUDIT_PATH,
)


TEST_AUDIT_SHA256 = sha256_file(
    TEST_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    TEST_STAGE_ROOT
    /
    "LATEST_V61_TEST_EVALUATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V61_UNTOUCHED_TEST_EVALUATION",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_42_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                TEST_ATTEMPT_DIR
            ),

        "test_predictions":
            str(
                TEST_PREDICTIONS_PATH
            ),

        "bootstrap":
            str(
                BOOTSTRAP_PATH
            ),

        "audit_manifest":
            str(
                TEST_AUDIT_PATH
            ),

        "audit_sha256":
            TEST_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v61_test_evaluation"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_42_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            TEST_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "audit_manifest":
        str(
            TEST_AUDIT_PATH
        ),

    "audit_sha256":
        TEST_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_V61_TEST_EVALUATION_COMPLETE"
    if CELL_42_PASS
    else
    "HB_2024_V61_TEST_EVALUATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_V61_TEST = TEST

HB2024_V61_OVERALL_METRICS = (
    OVERALL_METRICS
)

HB2024_V61_PAIRWISE_OVERALL = (
    PAIRWISE_OVERALL
)

HB2024_V61_GAP_METRICS = (
    GAP_METRICS
)

HB2024_V61_GAP_GAINS = (
    GAP_GAINS
)

HB2024_V61_GROUP_METRICS = (
    GROUP_METRICS
)

HB2024_V61_BOOTSTRAP = (
    BOOTSTRAP_SUMMARY
)

HB2024_V61_TEST_ATTEMPT_DIR = (
    TEST_ATTEMPT_DIR
)

NOTEBOOK_2_CELL_42_PASS = (
    CELL_42_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2024 — UNTOUCHED TEST: V5 vs V6 vs V6.1")
print("=" * 165)


print()
print("FROZEN MODEL SPECIFICATION")
print("-" * 165)

print(
    f"V6 persistence model           : "
    f"{SELECTED_MODEL}"
)

print(
    f"V6 persistence parameters      : "
    f"{PERSISTENCE_PARAMS}"
)

print(
    f"V6 offset model                : "
    f"{OFFSET_MODEL}"
)

print(
    f"V6 offsets                     : "
    f"{OFFSETS}"
)

print(
    f"V6.1 TRAIN-only tau            : "
    f"{TAU_V5:.6f} days"
)


print()
print("OVERALL TEST METRICS")
print("-" * 165)

display(
    OVERALL_METRICS
)


print()
print("OVERALL PAIRWISE GAINS")
print("-" * 165)

display(
    PAIRWISE_OVERALL
)


print()
print("BY STATE GAP — METRICS")
print("-" * 165)

display(
    GAP_METRICS
)


print()
print("BY STATE GAP — PAIRWISE GAINS")
print("-" * 165)

display(
    GAP_GAINS
)


print()
print("BY MODEL GROUP")
print("-" * 165)

display(
    GROUP_METRICS
)


print()
print("PAIRED ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 165)

display(
    BOOTSTRAP_SUMMARY
)


print()
print("V6.1 WEIGHT REFERENCE")
print("-" * 165)

weight_days = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


WEIGHT_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            weight_days.astype(int),

        "OU_phi":
            np.exp(
                -
                weight_days
                /
                TAU_V5
            ),
    }
)


WEIGHT_REFERENCE[
    "V61_weight"
] = (
    1.0
    -
    WEIGHT_REFERENCE[
        "OU_phi"
    ]
)


display(
    WEIGHT_REFERENCE
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{TEST_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{TEST_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{TEST_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 165)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"V6.1 weights within [0,1]      : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"TEST dates only                : "
    f"{TEST_DATE_PASS}"
)

print(
    f"No new fitted parameters       : "
    f"{NO_NEW_PARAMETERS_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Fresh/stale endpoint logic     : "
    f"{V61_ENDPOINT_LOGIC_PASS}"
)


print()
print("=" * 165)

if CELL_42_PASS:

    print(
        "CELL 42 STATUS: HB_2024 V6.1 UNTOUCHED TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 42 STATUS: HB_2024 V6.1 TEST EVALUATION REVIEW REQUIRED"
    )

print("=" * 165)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 33
# Original source SHA256: e13422abc367f9ae83b118643b427869a2426f65da86d39eded8e3c23afb060c
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 43 — HB_2024 HISTORICAL EVIDENCE FREEZE
# ==================================================================================================
#
# Purpose
# -------
# Freeze HB_2024 untouched-test evidence exactly as observed.
#
# NO fitting.
# NO architecture changes.
# NO threshold changes.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_42_PASS",

    "HB2024_V61_OVERALL_METRICS",
    "HB2024_V61_PAIRWISE_OVERALL",
    "HB2024_V61_GAP_GAINS",
    "HB2024_V61_GROUP_METRICS",
    "HB2024_V61_BOOTSTRAP",

    "HB2024_V61_TEST_ATTEMPT_DIR",

    "HB2024_V5_TAU_DAYS",
    "HB2024_V6_MEAN_SPEC",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_42_PASS:

    raise RuntimeError(
        "Cell 42 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE EVIDENCE DIRECTORY
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2024"

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2024_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


EVIDENCE_ROOT = (
    HB2024_DIR
    /
    "fold_evidence"
)


EVIDENCE_DIR = (
    EVIDENCE_ROOT
    /
    ATTEMPT_ID
)


EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. INPUT TABLES
# --------------------------------------------------------------------------------------------------

OVERALL = (
    HB2024_V61_OVERALL_METRICS.copy()
)


PAIRWISE = (
    HB2024_V61_PAIRWISE_OVERALL.copy()
)


GAP_GAINS = (
    HB2024_V61_GAP_GAINS.copy()
)


GROUP_METRICS = (
    HB2024_V61_GROUP_METRICS.copy()
)


BOOTSTRAP = (
    HB2024_V61_BOOTSTRAP.copy()
)


# --------------------------------------------------------------------------------------------------
# 5. LOOKUP HELPERS
# --------------------------------------------------------------------------------------------------

def get_pairwise(
    baseline,
    candidate,
):

    row = PAIRWISE.loc[
        PAIRWISE[
            "baseline"
        ].eq(
            baseline
        )
        &
        PAIRWISE[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover "
            f"{baseline} -> {candidate}."
        )


    return row.iloc[0]


def get_bootstrap(
    horizon,
    baseline,
    candidate,
):

    row = BOOTSTRAP.loc[
        BOOTSTRAP[
            "horizon"
        ].eq(
            horizon
        )
        &
        BOOTSTRAP[
            "baseline"
        ].eq(
            baseline
        )
        &
        BOOTSTRAP[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover bootstrap "
            f"{horizon}: {baseline} -> {candidate}."
        )


    return row.iloc[0]


# --------------------------------------------------------------------------------------------------
# 6. KEY COMPARISONS
# --------------------------------------------------------------------------------------------------

V5_TO_V6 = get_pairwise(
    "V5",
    "V6",
)


V5_TO_V61 = get_pairwise(
    "V5",
    "V6.1",
)


V6_TO_V61 = get_pairwise(
    "V6",
    "V6.1",
)


BOOT_ALL_V5_V6 = get_bootstrap(
    "ALL",
    "V5",
    "V6",
)


BOOT_ALL_V5_V61 = get_bootstrap(
    "ALL",
    "V5",
    "V6.1",
)


BOOT_ALL_V6_V61 = get_bootstrap(
    "ALL",
    "V6",
    "V6.1",
)


BOOT_FRESH_V5_V6 = get_bootstrap(
    "01-30",
    "V5",
    "V6",
)


BOOT_FRESH_V5_V61 = get_bootstrap(
    "01-30",
    "V5",
    "V6.1",
)


BOOT_31_60_V5_V61 = get_bootstrap(
    "31-60",
    "V5",
    "V6.1",
)


BOOT_61_120_V5_V61 = get_bootstrap(
    "61-120",
    "V5",
    "V6.1",
)


BOOT_121_365_V5_V61 = get_bootstrap(
    "121-365",
    "V5",
    "V6.1",
)


BOOT_366_PLUS_V5_V61 = get_bootstrap(
    "366+",
    "V5",
    "V6.1",
)


# --------------------------------------------------------------------------------------------------
# 7. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

O = (
    OVERALL.set_index(
        "model"
    )
)


V5_MAE = float(
    O.loc[
        "V5",
        "MAE",
    ]
)


V6_MAE = float(
    O.loc[
        "V6",
        "MAE",
    ]
)


V61_MAE = float(
    O.loc[
        "V6.1",
        "MAE",
    ]
)


V5_RMSE = float(
    O.loc[
        "V5",
        "RMSE",
    ]
)


V6_RMSE = float(
    O.loc[
        "V6",
        "RMSE",
    ]
)


V61_RMSE = float(
    O.loc[
        "V6.1",
        "RMSE",
    ]
)


V5_BIAS = float(
    O.loc[
        "V5",
        "Bias",
    ]
)


V6_BIAS = float(
    O.loc[
        "V6",
        "Bias",
    ]
)


V61_BIAS = float(
    O.loc[
        "V6.1",
        "Bias",
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. DESCRIPTIVE EVIDENCE FLAGS
# --------------------------------------------------------------------------------------------------
#
# These are observations, NOT promotion rules.
#
# --------------------------------------------------------------------------------------------------

EVIDENCE_FLAGS = {

    "v6_overall_mae_better_than_v5":
        bool(
            V6_MAE
            <
            V5_MAE
        ),

    "v6_overall_mae_gain_ci_above_zero":
        bool(
            BOOT_ALL_V5_V6[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_overall_mae_better_than_v5":
        bool(
            V61_MAE
            <
            V5_MAE
        ),

    "v61_overall_mae_gain_ci_above_zero":
        bool(
            BOOT_ALL_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_overall_mae_better_than_v6":
        bool(
            V61_MAE
            <
            V6_MAE
        ),

    "v61_vs_v6_mae_ci_above_zero":
        bool(
            BOOT_ALL_V6_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v6_fresh_mae_better_than_v5":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_gain_m"
            ]
            >
            0
        ),

    "v6_fresh_mae_gain_ci_above_zero":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_fresh_mae_better_than_v5":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_gain_m"
            ]
            >
            0
        ),

    "v61_fresh_mae_gain_ci_above_zero":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_31_60_mae_gain_ci_above_zero":
        bool(
            BOOT_31_60_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_61_120_mae_gain_ci_above_zero":
        bool(
            BOOT_61_120_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_121_365_mae_gain_ci_above_zero":
        bool(
            BOOT_121_365_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_366_plus_mae_gain_ci_above_zero":
        bool(
            BOOT_366_PLUS_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),
}


# --------------------------------------------------------------------------------------------------
# 9. CROSS-FOLD SUMMARY ROW
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "test_start":
                pd.Timestamp(
                    "2024-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2024-08-30"
                ),

            "test_rows":
                int(
                    O.loc[
                        "V5",
                        "n",
                    ]
                ),

            "test_athletes":
                int(
                    O.loc[
                        "V5",
                        "athletes",
                    ]
                ),

            "v5_tau_days":
                float(
                    HB2024_V5_TAU_DAYS
                ),

            "v6_persistence_model":
                HB2024_V6_MEAN_SPEC[
                    "selected_persistence_model"
                ],

            "v6_offset_model":
                HB2024_V6_MEAN_SPEC[
                    "selected_offset_model"
                ],

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V61_MAE":
                V61_MAE,

            "V5_to_V6_MAE_gain_m":
                float(
                    V5_TO_V6[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V6_CI_low":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V6_CI_high":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "V5_to_V61_MAE_gain_m":
                float(
                    V5_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V61_CI_low":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V61_CI_high":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),

            "V6_to_V61_MAE_gain_m":
                float(
                    V6_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V61_RMSE":
                V61_RMSE,

            "V5_Bias":
                V5_BIAS,

            "V6_Bias":
                V6_BIAS,

            "V61_Bias":
                V61_BIAS,

            "fresh_V5_to_V6_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V6_CI_low":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V6_CI_high":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "fresh_V5_to_V61_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V61_CI_low":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V61_CI_high":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. SAVE IMMUTABLE EVIDENCE
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY_PATH = (
    EVIDENCE_DIR
    /
    "fold_summary.parquet"
)


EVIDENCE_JSON_PATH = (
    EVIDENCE_DIR
    /
    "fold_evidence.json"
)


FOLD_SUMMARY.to_parquet(
    FOLD_SUMMARY_PATH,
    index=False,
)


CELL42_AUDIT_PATH = (
    Path(
        HB2024_V61_TEST_ATTEMPT_DIR
    )
    /
    "v61_test_evaluation_audit.json"
)


EVIDENCE_SPEC = {
    "evidence_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        "EVIDENCE_FROZEN",

    "purpose":
        (
            "Immutable HB_2024 fold evidence for eventual "
            "cross-fold V6/V6.1 architecture assessment."
        ),

    "decision_note":
        (
            "No architecture or release decision is made here. "
            "HB_2024 evidence is frozen before HB_2025."
        ),

    "overall": {
        "V5": {
            "MAE":
                V5_MAE,

            "RMSE":
                V5_RMSE,

            "Bias":
                V5_BIAS,
        },

        "V6": {
            "MAE":
                V6_MAE,

            "RMSE":
                V6_RMSE,

            "Bias":
                V6_BIAS,
        },

        "V6.1": {
            "MAE":
                V61_MAE,

            "RMSE":
                V61_RMSE,

            "Bias":
                V61_BIAS,
        },
    },

    "overall_bootstrap": {
        "V5_to_V6": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },

        "V5_to_V61": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },

        "V6_to_V61": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V6_V61[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V6_V61[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V6_V61[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },
    },

    "fresh_01_30_days": {
        "V5_to_V6_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V6[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V6_CI95": [
            float(
                BOOT_FRESH_V5_V6[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_FRESH_V5_V6[
                    "MAE_CI_high"
                ]
            ),
        ],

        "V5_to_V61_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V61_CI95": [
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_high"
                ]
            ),
        ],
    },

    "evidence_flags":
        EVIDENCE_FLAGS,

    "v5_tau_days":
        float(
            HB2024_V5_TAU_DAYS
        ),

    "v6_mean_spec":
        HB2024_V6_MEAN_SPEC,

    "source_test_evaluation": {
        "directory":
            str(
                HB2024_V61_TEST_ATTEMPT_DIR
            ),

        "audit_path":
            str(
                CELL42_AUDIT_PATH
            ),

        "audit_sha256":
            (
                sha256_file(
                    CELL42_AUDIT_PATH
                )
                if CELL42_AUDIT_PATH.exists()
                else None
            ),
    },

    "artifacts": {
        "fold_summary": {
            "path":
                str(
                    FOLD_SUMMARY_PATH
                ),

            "sha256":
                sha256_file(
                    FOLD_SUMMARY_PATH
                ),
        },
    },
}


atomic_json_write(
    EVIDENCE_SPEC,
    EVIDENCE_JSON_PATH,
)


EVIDENCE_SHA256 = sha256_file(
    EVIDENCE_JSON_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_EVIDENCE_POINTER = (
    EVIDENCE_ROOT
    /
    "LATEST_FOLD_EVIDENCE.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            "EVIDENCE_FROZEN",

        "evidence_directory":
            str(
                EVIDENCE_DIR
            ),

        "evidence_json":
            str(
                EVIDENCE_JSON_PATH
            ),

        "evidence_sha256":
            EVIDENCE_SHA256,

        "fold_summary":
            str(
                FOLD_SUMMARY_PATH
            ),

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_EVIDENCE_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 12. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "fold_evidence"
] = {
    "status":
        "EVIDENCE_FROZEN",

    "attempt_id":
        ATTEMPT_ID,

    "evidence_directory":
        str(
            EVIDENCE_DIR
        ),

    "evidence_json":
        str(
            EVIDENCE_JSON_PATH
        ),

    "evidence_sha256":
        EVIDENCE_SHA256,

    "fold_summary":
        str(
            FOLD_SUMMARY_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2024_HISTORICAL_EVIDENCE_FROZEN"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 13. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2024_FOLD_SUMMARY = (
    FOLD_SUMMARY
)


HB2024_FOLD_EVIDENCE = (
    EVIDENCE_SPEC
)


HB2024_FOLD_EVIDENCE_PATH = (
    EVIDENCE_JSON_PATH
)


NOTEBOOK_2_CELL_43_PASS = True


# --------------------------------------------------------------------------------------------------
# 14. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2024 — HISTORICAL EVIDENCE FREEZE")
print("=" * 165)


print()
print("CROSS-FOLD SUMMARY ROW")
print("-" * 165)

display(
    FOLD_SUMMARY
)


print()
print("EVIDENCE FLAGS")
print("-" * 165)

display(
    pd.DataFrame(
        [
            {
                "evidence":
                    key,

                "value":
                    value,
            }

            for key, value
            in EVIDENCE_FLAGS.items()
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable evidence directory   : "
    f"{EVIDENCE_DIR}"
)

print(
    f"Evidence JSON                  : "
    f"{EVIDENCE_JSON_PATH}"
)

print(
    f"Evidence SHA256                : "
    f"{EVIDENCE_SHA256}"
)

print(
    f"Latest evidence pointer        : "
    f"{LATEST_EVIDENCE_POINTER}"
)


print()
print("=" * 165)
print(
    "CELL 43 STATUS: HB_2024 HISTORICAL EVIDENCE FROZEN"
)
print("=" * 165)



HB_2024 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY

OVERALL POINT PERFORMANCE
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,split,n,athletes,V4_MAE,V5_MAE,V4_to_V5_MAE_gain_m,V4_RMSE,V5_RMSE,V4_to_V5_RMSE_gain_m,V5_Bias
0,VALIDATION,4288,2219,2.801272,2.576002,0.225270,3.742295,3.477255,0.265040,-0.050276
1,TEST,11054,3997,2.908380,2.567806,0.340574,3.920283,3.509818,0.410465,0.110356



POINT METRICS
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,model,n,athletes,MAE,RMSE,Bias,split
0,V4,4288,2219,2.801272,3.742295,-0.071706,VALIDATION
1,V5,4288,2219,2.576002,3.477255,-0.050276,VALIDATION
2,V4,11054,3997,2.908380,3.920283,0.267816,TEST
3,V5,11054,3997,2.567806,3.509818,0.110356,TEST



BY STATE GAP
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


...


AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T134222_456257Z
Immutable attempt directory    : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v61_test_evaluation/20260922T134222_456257Z
Audit manifest                 : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/v61_test_evaluation/20260922T134222_456257Z/v61_test_evaluation_audit.json
Audit SHA256                   : ca5bd0cefcbaa83158cb1a5a7263bb0f35ca59b96123d89f67d154da5ed7863d
Git commit                     : a7cb73d264fa6eb35402298c3bf7f921b1c8641e

QA
----------------------------------------------------------------------------------------------------------------------------

,fold,test_start,test_end,test_rows,test_athletes,v5_tau_days,v6_persistence_model,v6_offset_model,V5_MAE,V6_MAE,...,V61_RMSE,V5_Bias,V6_Bias,V61_Bias,fresh_V5_to_V6_MAE_gain_m,fresh_V5_to_V6_CI_low,fresh_V5_to_V6_CI_high,fresh_V5_to_V61_MAE_gain_m,fresh_V5_to_V61_CI_low,fresh_V5_to_V61_CI_high
0,HB_2024,2024-05-01,2024-08-30,11054,3997,92.394693,M2_EXP_CONST,SEX,2.567806,2.557316,...,3.501533,0.110356,-0.021834,0.036609,0.012298,0.003561,0.020873,0.002729,0.001446,0.004016



EVIDENCE FLAGS
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,evidence,value
0,v6_overall_mae_better_than_v5,True
1,v6_overall_mae_gain_ci_above_zero,True
2,v61_overall_mae_better_than_v5,True
3,v61_overall_mae_gain_ci_above_zero,False
4,v61_overall_mae_better_than_v6,False
5,v61_vs_v6_mae_ci_above_zero,False
6,v6_fresh_mae_better_than_v5,True
7,v6_fresh_mae_gain_ci_above_zero,True
8,v61_fresh_mae_better_than_v5,True
9,v61_fresh_mae_gain_ci_above_zero,True



AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T134224_958272Z
Immutable evidence directory   : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/fold_evidence/20260922T134224_958272Z
Evidence JSON                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/fold_evidence/20260922T134224_958272Z/fold_evidence.json
Evidence SHA256                : 134501e945ef525d2bf30b721a265825154f5240bc0ecfb930cc8b570303f033
Latest evidence pointer        : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2024/fold_evidence/LATEST_FOLD_EVIDENCE.json

CELL 43 STATUS: 

## Stage 15 — HB_2025 model reconstruction

Rebuild the 2025 historical V4/V5 training state and persist immutable fold-specific audit artifacts.

**Legacy source cells represented:** 34, 35, 36, 37  
**Execution rule:** run only after the preceding stage has completed successfully.


In [16]:
# ==================================================================================================
# REFACTORED STAGE 15 — HB_2025 model reconstruction
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 34
# Original source SHA256: 6144179bf353687e24e8b10072855c021a08df93767e0f9c8a295c1a7b5e9058
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 44 — HB_2025 HISTORICAL V4 REFIT PREPARATION + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# Historical fold:
#
#   TRAIN       <= 2024-12-31
#   VALIDATION   2025-01-01 -> 2025-04-30
#   TEST         2025-05-01 -> 2025-08-30
#
#
# Purpose
# -------
# Prepare the third and final independent historical fold using the exact same
# feature engineering and leakage controls as HB_2023 and HB_2024.
#
# NO fitting occurs here.
# NO architecture changes.
# NO V6.1 changes.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd

from patsy import (
    dmatrix,
    build_design_matrices,
)


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_43_PASS",

    "HISTORICAL_BACKTEST_BASE",
    "HISTORICAL_BACKTEST_FOLDS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_43_PASS:

    raise RuntimeError(
        "Cell 43 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FOLD SPECIFICATION
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_44_HB2025_PREPARATION"

PREPARATION_SCHEMA_VERSION = "1.0.0"


fold_match = (
    HISTORICAL_BACKTEST_FOLDS.loc[
        HISTORICAL_BACKTEST_FOLDS[
            "fold"
        ].eq(
            FOLD_NAME
        )
    ]
)


if len(
    fold_match
) != 1:

    raise RuntimeError(
        f"Could not uniquely identify {FOLD_NAME}."
    )


FOLD_SPEC = (
    fold_match.iloc[0]
)


TRAIN_END = pd.Timestamp(
    FOLD_SPEC[
        "train_end"
    ]
)


VALIDATION_START = pd.Timestamp(
    FOLD_SPEC[
        "validation_start"
    ]
)


VALIDATION_END = pd.Timestamp(
    FOLD_SPEC[
        "validation_end"
    ]
)


TEST_START = pd.Timestamp(
    FOLD_SPEC[
        "test_start"
    ]
)


TEST_END = pd.Timestamp(
    FOLD_SPEC[
        "test_end"
    ]
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = (
    RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
)


HB2025_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


PREP_ROOT = (
    HB2025_DIR
    /
    "preparation"
)


PREP_ATTEMPT_DIR = (
    PREP_ROOT
    /
    ATTEMPT_ID
)


PREP_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. MATERIALIZE TEMPORAL SPLITS
# --------------------------------------------------------------------------------------------------

BASE = (
    HISTORICAL_BACKTEST_BASE.copy()
)


TRAIN = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    ]
    .copy()
)


VALIDATION_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            VALIDATION_START,
            VALIDATION_END,
            inclusive="both",
        )
    ]
    .copy()
)


TEST_ALL = (
    BASE.loc[
        BASE[
            "result_date_dt"
        ]
        .between(
            TEST_START,
            TEST_END,
            inclusive="both",
        )
    ]
    .copy()
)


TRAIN_ATHLETES = set(
    TRAIN[
        "aaAthleteId"
    ]
    .unique()
)


VALIDATION_ALL[
    "athlete_seen_in_training"
] = (
    VALIDATION_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


TEST_ALL[
    "athlete_seen_in_training"
] = (
    TEST_ALL[
        "aaAthleteId"
    ]
    .isin(
        TRAIN_ATHLETES
    )
)


VALIDATION = (
    VALIDATION_ALL.loc[
        VALIDATION_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


TEST = (
    TEST_ALL.loc[
        TEST_ALL[
            "athlete_seen_in_training"
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 6. V4 FEATURE DEFINITIONS — UNCHANGED
# --------------------------------------------------------------------------------------------------

SEX_INDEX = {
    "M": 0,
    "W": 1,
}


GROUP_INDEX = {
    "M_700": 0,
    "M_800": 1,
    "W_500": 2,
    "W_600": 3,
}


YOUTH_GROUPS = {
    "M_700",
    "W_500",
}


AGE_CENTER = 18.0
AGE_SCALE = 5.0


def add_v4_features(
    df,
):

    out = df.copy()


    out[
        "sex_group"
    ] = (
        out[
            "model_group"
        ]
        .astype(str)
        .str[0]
    )


    out[
        "sex_idx"
    ] = (
        out[
            "sex_group"
        ]
        .map(
            SEX_INDEX
        )
    )


    out[
        "group_idx"
    ] = (
        out[
            "model_group"
        ]
        .map(
            GROUP_INDEX
        )
    )


    out[
        "is_youth_implement"
    ] = (
        out[
            "model_group"
        ]
        .isin(
            YOUTH_GROUPS
        )
        .astype(float)
    )


    out[
        "age_scaled"
    ] = (
        (
            out[
                "age_years"
            ]
            .astype(float)

            -

            AGE_CENTER
        )

        /

        AGE_SCALE
    )


    return out


TRAIN = add_v4_features(
    TRAIN
)


VALIDATION = add_v4_features(
    VALIDATION
)


TEST = add_v4_features(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 7. TRAIN-ONLY ATHLETE INDEX
# --------------------------------------------------------------------------------------------------

ATHLETE_LEVELS = (
    pd.Series(
        TRAIN[
            "aaAthleteId"
        ]
        .unique()
    )
    .sort_values()
    .tolist()
)


ATHLETE_INDEX = {
    athlete_id:
        idx

    for idx, athlete_id
    in enumerate(
        ATHLETE_LEVELS
    )
}


for df in [
    TRAIN,
    VALIDATION,
    TEST,
]:

    df[
        "athlete_idx"
    ] = (
        df[
            "aaAthleteId"
        ]
        .map(
            ATHLETE_INDEX
        )
    )


if VALIDATION[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "HB_2025 established VALIDATION athlete missing TRAIN index."
    )


if TEST[
    "athlete_idx"
].isna().any():

    raise RuntimeError(
        "HB_2025 established TEST athlete missing TRAIN index."
    )


TRAIN[
    "athlete_idx"
] = (
    TRAIN[
        "athlete_idx"
    ]
    .astype(int)
)


VALIDATION[
    "athlete_idx"
] = (
    VALIDATION[
        "athlete_idx"
    ]
    .astype(int)
)


TEST[
    "athlete_idx"
] = (
    TEST[
        "athlete_idx"
    ]
    .astype(int)
)


# --------------------------------------------------------------------------------------------------
# 8. TRAIN-ONLY CUBIC SPLINE
# --------------------------------------------------------------------------------------------------

SPLINE_FORMULA = (
    "bs("
    "age, "
    "df=7, "
    "degree=3, "
    "include_intercept=False, "
    "lower_bound=13, "
    "upper_bound=35"
    ") - 1"
)


B_TRAIN_RAW_DF = dmatrix(
    SPLINE_FORMULA,

    {
        "age":
            TRAIN[
                "age_years"
            ]
            .to_numpy(
                dtype=float
            )
    },

    return_type="dataframe",
)


SPLINE_DESIGN_INFO = (
    B_TRAIN_RAW_DF.design_info
)


B_TRAIN_RAW = np.asarray(
    B_TRAIN_RAW_DF,
    dtype=float,
)


B_VALIDATION_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                VALIDATION[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


B_TEST_RAW = np.asarray(
    build_design_matrices(
        [
            SPLINE_DESIGN_INFO
        ],

        {
            "age":
                TEST[
                    "age_years"
                ]
                .to_numpy(
                    dtype=float
                )
        },
    )[0],

    dtype=float,
)


SPLINE_COLUMN_MEANS = (
    B_TRAIN_RAW.mean(
        axis=0
    )
)


B_TRAIN = (
    B_TRAIN_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_VALIDATION = (
    B_VALIDATION_RAW
    -
    SPLINE_COLUMN_MEANS
)


B_TEST = (
    B_TEST_RAW
    -
    SPLINE_COLUMN_MEANS
)


SPLINE_INTERNAL_KNOTS = np.quantile(
    TRAIN[
        "age_years"
    ]
    .to_numpy(
        dtype=float
    ),

    [
        0.20,
        0.40,
        0.60,
        0.80,
    ],
)


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(
        7
    )
]


for i, column in enumerate(
    SPLINE_COLUMNS
):

    TRAIN[
        column
    ] = (
        B_TRAIN[
            :,
            i
        ]
    )


    VALIDATION[
        column
    ] = (
        B_VALIDATION[
            :,
            i
        ]
    )


    TEST[
        column
    ] = (
        B_TEST[
            :,
            i
        ]
    )


# --------------------------------------------------------------------------------------------------
# 9. LEAKAGE-SAFE DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------

CHRONOLOGY = (
    BASE.loc[
        (
            BASE[
                "aaAthleteId"
            ]
            .isin(
                TRAIN_ATHLETES
            )
        )

        &

        (
            BASE[
                "result_date_dt"
            ]
            <=
            TEST_END
        )
    ][
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


CHRONOLOGY[
    "previous_distinct_date"
] = (
    CHRONOLOGY.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(1)
)


CHRONOLOGY[
    "state_gap_days"
] = (
    CHRONOLOGY[
        "result_date_dt"
    ]

    -

    CHRONOLOGY[
        "previous_distinct_date"
    ]
).dt.days


def attach_chronology(
    df,
):

    return df.merge(
        CHRONOLOGY,

        on=[
            "aaAthleteId",
            "result_date_dt",
        ],

        how="left",

        validate="many_to_one",
    )


TRAIN = attach_chronology(
    TRAIN
)


VALIDATION = attach_chronology(
    VALIDATION
)


TEST = attach_chronology(
    TEST
)


# --------------------------------------------------------------------------------------------------
# 10. QA
# --------------------------------------------------------------------------------------------------

TEMPORAL_SPLIT_PASS = bool(

    (
        TRAIN[
            "result_date_dt"
        ]
        <=
        TRAIN_END
    )
    .all()

    and

    VALIDATION[
        "result_date_dt"
    ]
    .between(
        VALIDATION_START,
        VALIDATION_END,
        inclusive="both",
    )
    .all()

    and

    TEST[
        "result_date_dt"
    ]
    .between(
        TEST_START,
        TEST_END,
        inclusive="both",
    )
    .all()
)


ATHLETE_INDEX_PASS = bool(

    TRAIN[
        "athlete_idx"
    ]
    .notna()
    .all()

    and

    VALIDATION[
        "athlete_idx"
    ]
    .notna()
    .all()

    and

    TEST[
        "athlete_idx"
    ]
    .notna()
    .all()
)


FEATURE_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "sex_idx",
                "group_idx",
                "is_youth_implement",
                "age_scaled",
                "athlete_idx",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


SPLINE_PASS = bool(

    B_TRAIN.shape[
        1
    ]
    ==
    7

    and

    B_VALIDATION.shape[
        1
    ]
    ==
    7

    and

    B_TEST.shape[
        1
    ]
    ==
    7

    and

    np.isfinite(
        B_TRAIN
    )
    .all()

    and

    np.isfinite(
        B_VALIDATION
    )
    .all()

    and

    np.isfinite(
        B_TEST
    )
    .all()

    and

    np.max(
        np.abs(
            B_TRAIN.mean(
                axis=0
            )
        )
    )
    <
    1e-10
)


VALIDATION_CHRONOLOGY_PASS = bool(

    VALIDATION[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        VALIDATION[
            "state_gap_days"
        ]
        >
        0
    )
    .all()
)


TEST_CHRONOLOGY_PASS = bool(

    TEST[
        "state_gap_days"
    ]
    .notna()
    .all()

    and

    (
        TEST[
            "state_gap_days"
        ]
        >
        0
    )
    .all()
)


NO_FUTURE_TRAIN_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    TRAIN_END
)


CELL_44_PASS = bool(
    TEMPORAL_SPLIT_PASS
    and
    ATHLETE_INDEX_PASS
    and
    FEATURE_FINITE_PASS
    and
    SPLINE_PASS
    and
    VALIDATION_CHRONOLOGY_PASS
    and
    TEST_CHRONOLOGY_PASS
    and
    NO_FUTURE_TRAIN_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 11. SAVE IMMUTABLE PREPARATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

TRAIN_PATH = (
    PREP_ATTEMPT_DIR
    /
    "train_prepared.parquet"
)


VALIDATION_PATH = (
    PREP_ATTEMPT_DIR
    /
    "validation_established_prepared.parquet"
)


TEST_PATH = (
    PREP_ATTEMPT_DIR
    /
    "test_established_prepared.parquet"
)


PREP_SPEC_PATH = (
    PREP_ATTEMPT_DIR
    /
    "preparation_spec.json"
)


PREP_AUDIT_PATH = (
    PREP_ATTEMPT_DIR
    /
    "preparation_audit.json"
)


TRAIN.to_parquet(
    TRAIN_PATH,
    index=False,
)


VALIDATION.to_parquet(
    VALIDATION_PATH,
    index=False,
)


TEST.to_parquet(
    TEST_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 12. PREPARATION SPECIFICATION
# --------------------------------------------------------------------------------------------------

PREP_SPEC = {
    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "schema_version":
        PREPARATION_SCHEMA_VERSION,

    "status":
        (
            "PASSED"
            if CELL_44_PASS
            else
            "FAILED"
        ),

    "train_end":
        TRAIN_END.isoformat(),

    "validation_start":
        VALIDATION_START.isoformat(),

    "validation_end":
        VALIDATION_END.isoformat(),

    "test_start":
        TEST_START.isoformat(),

    "test_end":
        TEST_END.isoformat(),

    "train_rows":
        int(
            len(
                TRAIN
            )
        ),

    "train_athletes":
        int(
            TRAIN[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "validation_rows":
        int(
            len(
                VALIDATION
            )
        ),

    "validation_athletes":
        int(
            VALIDATION[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "test_rows":
        int(
            len(
                TEST
            )
        ),

    "test_athletes":
        int(
            TEST[
                "aaAthleteId"
            ]
            .nunique()
        ),

    "age_center":
        AGE_CENTER,

    "age_scale":
        AGE_SCALE,

    "sex_index":
        SEX_INDEX,

    "group_index":
        GROUP_INDEX,

    "youth_groups":
        sorted(
            YOUTH_GROUPS
        ),

    "spline_formula":
        SPLINE_FORMULA,

    "spline_internal_knots":
        [
            float(x)
            for x in SPLINE_INTERNAL_KNOTS
        ],

    "spline_column_means":
        [
            float(x)
            for x in SPLINE_COLUMN_MEANS
        ],

    "spline_columns":
        SPLINE_COLUMNS,

    "athlete_index_count":
        int(
            len(
                ATHLETE_INDEX
            )
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "v61_formula_changed_from_previous_folds":
        False,

    "data_vintage_note":
        (
            "Historical fold reconstructed from the current 2026 canonical "
            "snapshot and restricted by event/result date. Later corrections "
            "to historical records may therefore be present; no future-dated "
            "performances are used."
        ),
}


atomic_json_write(
    PREP_SPEC,
    PREP_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 13. ARTIFACT HASHES
# --------------------------------------------------------------------------------------------------

OUTPUT_ARTIFACTS = {}


for name, path in {
    "train_prepared":
        TRAIN_PATH,

    "validation_prepared":
        VALIDATION_PATH,

    "test_prepared":
        TEST_PATH,

    "preparation_spec":
        PREP_SPEC_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


PARENT_ARTIFACTS = {}


if "CANONICAL_PATH" in globals():

    canonical_path = Path(
        CANONICAL_PATH
    )


    PARENT_ARTIFACTS[
        "active_canonical"
    ] = {
        "path":
            str(
                canonical_path
            ),

        "exists":
            bool(
                canonical_path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    canonical_path
                )
                if canonical_path.exists()
                else
                None
            ),
    }


if "HB2024_FOLD_EVIDENCE_PATH" in globals():

    previous_fold_path = Path(
        HB2024_FOLD_EVIDENCE_PATH
    )


    PARENT_ARTIFACTS[
        "previous_fold_evidence"
    ] = {
        "path":
            str(
                previous_fold_path
            ),

        "exists":
            bool(
                previous_fold_path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    previous_fold_path
                )
                if previous_fold_path.exists()
                else
                None
            ),
    }


# --------------------------------------------------------------------------------------------------
# 14. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

PREP_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "HISTORICAL_FOLD_PREPARATION",

    "stage_version":
        PREPARATION_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_44_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Prepare HB_2025 as the third and final independent "
            "historical backtest of the frozen V6.1 architecture."
        ),

    "model_logic_changed_from_HB_2023":
        False,

    "model_logic_changed_from_HB_2024":
        False,

    "v61_formula_changed":
        False,

    "fold_spec":
        PREP_SPEC,

    "qa": {
        "temporal_split":
            TEMPORAL_SPLIT_PASS,

        "train_only_athlete_index":
            ATHLETE_INDEX_PASS,

        "finite_features":
            FEATURE_FINITE_PASS,

        "spline":
            SPLINE_PASS,

        "validation_chronology":
            VALIDATION_CHRONOLOGY_PASS,

        "test_chronology":
            TEST_CHRONOLOGY_PASS,

        "no_future_train_data":
            NO_FUTURE_TRAIN_DATA_PASS,

        "cell_pass":
            CELL_44_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,
}


atomic_json_write(
    PREP_AUDIT,
    PREP_AUDIT_PATH,
)


PREP_AUDIT_SHA256 = (
    sha256_file(
        PREP_AUDIT_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 15. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_PREP_POINTER_PATH = (
    PREP_ROOT
    /
    "LATEST_PREPARATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_44_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                PREP_ATTEMPT_DIR
            ),

        "train_prepared":
            str(
                TRAIN_PATH
            ),

        "validation_prepared":
            str(
                VALIDATION_PATH
            ),

        "test_prepared":
            str(
                TEST_PATH
            ),

        "audit_manifest":
            str(
                PREP_AUDIT_PATH
            ),

        "audit_sha256":
            PREP_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_PREP_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 16. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
].setdefault(
    FOLD_NAME,
    {}
)


TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "preparation"
] = {
    "status":
        (
            "PASSED"
            if CELL_44_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            PREP_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_PREP_POINTER_PATH
        ),

    "audit_manifest":
        str(
            PREP_AUDIT_PATH
        ),

    "audit_sha256":
        PREP_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_PREPARED_FOR_V4_REFIT"
    if CELL_44_PASS
    else
    "HB_2025_PREPARATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 17. GLOBALS FOR NEXT CELL
# --------------------------------------------------------------------------------------------------

HB2025_TRAIN = TRAIN

HB2025_VALIDATION = VALIDATION

HB2025_TEST = TEST


HB2025_ATHLETE_LEVELS = (
    ATHLETE_LEVELS
)


HB2025_ATHLETE_INDEX = (
    ATHLETE_INDEX
)


HB2025_SPLINE_DESIGN_INFO = (
    SPLINE_DESIGN_INFO
)


HB2025_SPLINE_COLUMN_MEANS = (
    SPLINE_COLUMN_MEANS
)


HB2025_SPLINE_INTERNAL_KNOTS = (
    SPLINE_INTERNAL_KNOTS
)


HB2025_B_TRAIN = (
    B_TRAIN
)


HB2025_B_VALIDATION = (
    B_VALIDATION
)


HB2025_B_TEST = (
    B_TEST
)


HB2025_PREP_ATTEMPT_DIR = (
    PREP_ATTEMPT_DIR
)


HB2025_PREP_AUDIT_PATH = (
    PREP_AUDIT_PATH
)


NOTEBOOK_2_CELL_44_PASS = (
    CELL_44_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2025 — HISTORICAL V4 REFIT PREPARATION")
print("=" * 160)


print()
print("TEMPORAL SPLIT")
print("-" * 160)

print(
    f"TRAIN       : <= {TRAIN_END.date()}"
)

print(
    f"VALIDATION  : "
    f"{VALIDATION_START.date()} -> {VALIDATION_END.date()}"
)

print(
    f"TEST        : "
    f"{TEST_START.date()} -> {TEST_END.date()}"
)


print()
print("POPULATION")
print("-" * 160)

print(
    f"TRAIN rows / athletes          : "
    f"{len(TRAIN):,} / "
    f"{TRAIN['aaAthleteId'].nunique():,}"
)

print(
    f"VALIDATION established         : "
    f"{len(VALIDATION):,} / "
    f"{VALIDATION['aaAthleteId'].nunique():,}"
)

print(
    f"TEST established               : "
    f"{len(TEST):,} / "
    f"{TEST['aaAthleteId'].nunique():,}"
)


print()
print("TRAIN-ONLY SPLINE")
print("-" * 160)

print(
    f"Basis shape                    : "
    f"{B_TRAIN.shape}"
)

print(
    "Internal knots                : "
    +
    np.array2string(
        SPLINE_INTERNAL_KNOTS,
        precision=10,
    )
)

print(
    "Column means                  : "
    +
    np.array2string(
        SPLINE_COLUMN_MEANS,
        precision=10,
    )
)

print(
    "Centered TRAIN mean           : "
    +
    np.array2string(
        B_TRAIN.mean(
            axis=0
        ),
        precision=3,
        suppress_small=True,
    )
)


print()
print("STATE-GAP SUMMARY")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "split":
                    "VALIDATION",

                "n":
                    len(
                        VALIDATION
                    ),

                "median_gap_days":
                    float(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .median()
                    ),

                "mean_gap_days":
                    float(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .mean()
                    ),

                "min_gap_days":
                    int(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .min()
                    ),

                "max_gap_days":
                    int(
                        VALIDATION[
                            "state_gap_days"
                        ]
                        .max()
                    ),
            },

            {
                "split":
                    "TEST",

                "n":
                    len(
                        TEST
                    ),

                "median_gap_days":
                    float(
                        TEST[
                            "state_gap_days"
                        ]
                        .median()
                    ),

                "mean_gap_days":
                    float(
                        TEST[
                            "state_gap_days"
                        ]
                        .mean()
                    ),

                "min_gap_days":
                    int(
                        TEST[
                            "state_gap_days"
                        ]
                        .min()
                    ),

                "max_gap_days":
                    int(
                        TEST[
                            "state_gap_days"
                        ]
                        .max()
                    ),
            },
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable preparation directory: "
    f"{PREP_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{PREP_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{PREP_AUDIT_SHA256}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Temporal split                 : "
    f"{TEMPORAL_SPLIT_PASS}"
)

print(
    f"TRAIN-only athlete index       : "
    f"{ATHLETE_INDEX_PASS}"
)

print(
    f"Finite V4 features             : "
    f"{FEATURE_FINITE_PASS}"
)

print(
    f"7-column TRAIN spline          : "
    f"{SPLINE_PASS}"
)

print(
    f"VALIDATION chronology          : "
    f"{VALIDATION_CHRONOLOGY_PASS}"
)

print(
    f"TEST chronology                : "
    f"{TEST_CHRONOLOGY_PASS}"
)

print(
    f"No future TRAIN data           : "
    f"{NO_FUTURE_TRAIN_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_44_PASS:

    print(
        "CELL 44 STATUS: HB_2025 HISTORICAL V4 PREPARATION PASSED"
    )

else:

    print(
        "CELL 44 STATUS: HB_2025 PREPARATION REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 35
# Original source SHA256: b0d884c7e76196aeceac56510660cfbc562933ddf870ee24979c914c803b4999
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 45 — HB_2025 HISTORICAL V4 BAYESIAN REFIT + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# TRAIN:
#       <= 2024-12-31
#
# Inference:
#       ADVI          = 40,000 iterations
#       posterior     = 2,000 draws
#       random seed   = 42
#
# Model specification is intentionally UNCHANGED from HB_2023 / HB_2024.
#
# NO 2025 VALIDATION or TEST marks are used for fitting.
#
# ==================================================================================================

import gc
import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import (
    datetime,
    timezone,
)
from pathlib import Path

import numpy as np
import pandas as pd

import pymc as pm
import pytensor
import pytensor.tensor as pt


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_44_PASS",

    "HB2025_TRAIN",
    "HB2025_B_TRAIN",

    "HB2025_ATHLETE_LEVELS",
    "HB2025_ATHLETE_INDEX",

    "HB2025_PREP_ATTEMPT_DIR",
    "HB2025_PREP_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_44_PASS:

    raise RuntimeError(
        "Cell 44 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED MODEL / INFERENCE SETTINGS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_45_HB2025_V4_REFIT"

STAGE_SCHEMA_VERSION = "1.0.0"


ADVI_ITERATIONS = 40_000
POSTERIOR_DRAWS = 2_000
RANDOM_SEED = 42


SEX_LABELS = [
    "M",
    "W",
]


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_SEX = 2
N_GROUP = 4
N_SPLINE = 7
N_SPLINE_DELTA = 6


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE FIT ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2025_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V4_ROOT = (
    HB2025_DIR
    /
    "v4_refit"
)


V4_ATTEMPT_DIR = (
    V4_ROOT
    /
    ATTEMPT_ID
)


V4_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. TRAIN ARRAYS
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2025_TRAIN.copy()
    .reset_index(
        drop=True
    )
)


B_TRAIN = np.asarray(
    HB2025_B_TRAIN,
    dtype=np.float64,
)


y_train = (
    TRAIN[
        "mark_numeric_m"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


sex_idx_train = (
    TRAIN[
        "sex_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


group_idx_train = (
    TRAIN[
        "group_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


athlete_idx_train = (
    TRAIN[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


youth_train = (
    TRAIN[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


age_scaled_train = (
    TRAIN[
        "age_scaled"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


N_ATHLETES = len(
    HB2025_ATHLETE_LEVELS
)


# --------------------------------------------------------------------------------------------------
# 6. PRE-FIT QA
# --------------------------------------------------------------------------------------------------

if B_TRAIN.shape != (
    len(TRAIN),
    N_SPLINE,
):

    raise RuntimeError(
        f"Unexpected spline shape: {B_TRAIN.shape}"
    )


if not np.isfinite(
    y_train
).all():

    raise RuntimeError(
        "Non-finite TRAIN marks."
    )


if not np.isfinite(
    B_TRAIN
).all():

    raise RuntimeError(
        "Non-finite TRAIN spline."
    )


if (
    sex_idx_train.min() < 0
    or
    sex_idx_train.max() >= N_SEX
):

    raise RuntimeError(
        "Invalid sex index."
    )


if (
    group_idx_train.min() < 0
    or
    group_idx_train.max() >= N_GROUP
):

    raise RuntimeError(
        "Invalid group index."
    )


if (
    athlete_idx_train.min() < 0
    or
    athlete_idx_train.max() >= N_ATHLETES
):

    raise RuntimeError(
        "Invalid athlete index."
    )


if TRAIN[
    "result_date_dt"
].max() > pd.Timestamp(
    "2024-12-31"
):

    raise RuntimeError(
        "Future data detected in HB_2025 TRAIN."
    )


# --------------------------------------------------------------------------------------------------
# 7. MODEL COORDINATES
# --------------------------------------------------------------------------------------------------

coords = {
    "sex":
        SEX_LABELS,

    "group":
        GROUP_LABELS,

    "spline_basis":
        np.arange(
            N_SPLINE
        ),

    "spline_step":
        np.arange(
            N_SPLINE_DELTA
        ),

    "athlete":
        np.asarray(
            HB2025_ATHLETE_LEVELS
        ),

    "observation":
        np.arange(
            len(TRAIN)
        ),
}


# --------------------------------------------------------------------------------------------------
# 8. BUILD + FIT HISTORICAL V4
# --------------------------------------------------------------------------------------------------

with pm.Model(
    coords=coords
) as HB2025_V4_MODEL:

    sex_idx_data = pm.Data(
        "sex_idx_data",
        sex_idx_train,
        dims="observation",
    )


    group_idx_data = pm.Data(
        "group_idx_data",
        group_idx_train,
        dims="observation",
    )


    athlete_idx_data = pm.Data(
        "athlete_idx_data",
        athlete_idx_train,
        dims="observation",
    )


    youth_data = pm.Data(
        "youth_data",
        youth_train,
        dims="observation",
    )


    age_scaled_data = pm.Data(
        "age_scaled_data",
        age_scaled_train,
        dims="observation",
    )


    spline_data = pm.Data(
        "spline_data",
        B_TRAIN,
        dims=(
            "observation",
            "spline_basis",
        ),
    )


    # Population intercept
    alpha_sex = pm.Normal(
        "alpha_sex",
        mu=45.0,
        sigma=15.0,
        dims="sex",
    )


    # Sex-specific smooth spline
    sigma_spline = pm.HalfNormal(
        "sigma_spline",
        sigma=2.0,
        dims="sex",
    )


    spline_start = pm.Normal(
        "spline_start",
        mu=0.0,
        sigma=3.0,
        dims="sex",
    )


    spline_delta = pm.Normal(
        "spline_delta",
        mu=0.0,
        sigma=sigma_spline[
            :,
            None
        ],
        dims=(
            "sex",
            "spline_step",
        ),
    )


    beta_spline = pm.Deterministic(
        "beta_spline",

        pt.concatenate(
            [
                spline_start[
                    :,
                    None
                ],

                spline_start[
                    :,
                    None
                ]
                +
                pt.cumsum(
                    spline_delta,
                    axis=1,
                ),
            ],
            axis=1,
        ),

        dims=(
            "sex",
            "spline_basis",
        ),
    )


    # Youth implement effect
    youth_implement_effect = pm.Normal(
        "youth_implement_effect",
        mu=0.0,
        sigma=8.0,
        dims="sex",
    )


    # Athlete random intercept
    sigma_athlete_intercept = pm.HalfNormal(
        "sigma_athlete_intercept",
        sigma=10.0,
    )


    athlete_intercept_z = pm.Normal(
        "athlete_intercept_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_intercept = pm.Deterministic(
        "athlete_intercept",

        sigma_athlete_intercept
        *
        athlete_intercept_z,

        dims="athlete",
    )


    # Athlete random age slope
    sigma_athlete_slope = pm.HalfNormal(
        "sigma_athlete_slope",
        sigma=4.0,
    )


    athlete_slope_z = pm.Normal(
        "athlete_slope_z",
        mu=0.0,
        sigma=1.0,
        dims="athlete",
    )


    athlete_slope = pm.Deterministic(
        "athlete_slope",

        sigma_athlete_slope
        *
        athlete_slope_z,

        dims="athlete",
    )


    # Group-specific observation scale
    sigma_group = pm.HalfNormal(
        "sigma_group",
        sigma=5.0,
        dims="group",
    )


    # Student-t degrees of freedom
    nu_minus_two = pm.Exponential(
        "nu_minus_two",
        lam=0.1,
    )


    nu = pm.Deterministic(
        "nu",
        nu_minus_two
        +
        2.0,
    )


    # Structural mean
    spline_effect = (
        spline_data
        *
        beta_spline[
            sex_idx_data
        ]
    ).sum(
        axis=1
    )


    mu = (
        alpha_sex[
            sex_idx_data
        ]

        +

        spline_effect

        +

        youth_implement_effect[
            sex_idx_data
        ]
        *
        youth_data

        +

        athlete_intercept[
            athlete_idx_data
        ]

        +

        athlete_slope[
            athlete_idx_data
        ]
        *
        age_scaled_data
    )


    # Student-t likelihood
    observed = pm.StudentT(
        "observed",

        nu=nu,

        mu=mu,

        sigma=sigma_group[
            group_idx_data
        ],

        observed=y_train,

        dims="observation",
    )


    # ADVI
    HB2025_V4_APPROX = pm.fit(
        n=ADVI_ITERATIONS,
        method="advi",
        random_seed=RANDOM_SEED,
        progressbar=True,
    )


    # Posterior sample
    HB2025_V4_IDATA = (
        HB2025_V4_APPROX.sample(
            draws=POSTERIOR_DRAWS,
            random_seed=RANDOM_SEED,
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. ADVI HISTORY
# --------------------------------------------------------------------------------------------------

ADVI_HISTORY = np.asarray(
    HB2025_V4_APPROX.hist,
    dtype=float,
)


ADVI_FINITE_PASS = bool(
    len(
        ADVI_HISTORY
    )
    ==
    ADVI_ITERATIONS

    and

    np.isfinite(
        ADVI_HISTORY
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 10. POSTERIOR MEAN HELPER
# --------------------------------------------------------------------------------------------------

def posterior_mean(
    name,
):

    da = (
        HB2025_V4_IDATA
        .posterior[
            name
        ]
    )


    average_dims = [
        dim
        for dim in [
            "chain",
            "draw",
        ]
        if dim in da.dims
    ]


    return (
        da.mean(
            dim=average_dims
        )
        .values
    )


# --------------------------------------------------------------------------------------------------
# 11. POSTERIOR MEANS
# --------------------------------------------------------------------------------------------------

HB2025_V4_POSTERIOR_MEANS = {

    "alpha_sex":
        np.asarray(
            posterior_mean(
                "alpha_sex"
            ),
            dtype=float,
        ),

    "beta_spline":
        np.asarray(
            posterior_mean(
                "beta_spline"
            ),
            dtype=float,
        ),

    "youth_implement_effect":
        np.asarray(
            posterior_mean(
                "youth_implement_effect"
            ),
            dtype=float,
        ),

    "athlete_intercept":
        np.asarray(
            posterior_mean(
                "athlete_intercept"
            ),
            dtype=float,
        ),

    "athlete_slope":
        np.asarray(
            posterior_mean(
                "athlete_slope"
            ),
            dtype=float,
        ),

    "sigma_group":
        np.asarray(
            posterior_mean(
                "sigma_group"
            ),
            dtype=float,
        ),

    "nu":
        float(
            np.asarray(
                posterior_mean(
                    "nu"
                )
            )
        ),

    "sigma_athlete_intercept":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_intercept"
                )
            )
        ),

    "sigma_athlete_slope":
        float(
            np.asarray(
                posterior_mean(
                    "sigma_athlete_slope"
                )
            )
        ),

    "sigma_spline":
        np.asarray(
            posterior_mean(
                "sigma_spline"
            ),
            dtype=float,
        ),
}


# --------------------------------------------------------------------------------------------------
# 12. POSTERIOR QA
# --------------------------------------------------------------------------------------------------

POSTERIOR_SHAPE_PASS = bool(

    HB2025_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ].shape
    ==
    (N_SEX,)

    and

    HB2025_V4_POSTERIOR_MEANS[
        "beta_spline"
    ].shape
    ==
    (
        N_SEX,
        N_SPLINE,
    )

    and

    HB2025_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ].shape
    ==
    (N_SEX,)

    and

    HB2025_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ].shape
    ==
    (N_ATHLETES,)

    and

    HB2025_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ].shape
    ==
    (N_ATHLETES,)

    and

    HB2025_V4_POSTERIOR_MEANS[
        "sigma_group"
    ].shape
    ==
    (N_GROUP,)
)


POSTERIOR_FINITE_PASS = bool(
    all(
        np.isfinite(
            np.asarray(value)
        )
        .all()

        for value
        in HB2025_V4_POSTERIOR_MEANS.values()
    )
)


POSITIVE_SCALE_PASS = bool(

    HB2025_V4_POSTERIOR_MEANS[
        "sigma_athlete_intercept"
    ] > 0

    and

    HB2025_V4_POSTERIOR_MEANS[
        "sigma_athlete_slope"
    ] > 0

    and

    (
        HB2025_V4_POSTERIOR_MEANS[
            "sigma_group"
        ] > 0
    ).all()

    and

    HB2025_V4_POSTERIOR_MEANS[
        "nu"
    ] > 2
)


# --------------------------------------------------------------------------------------------------
# 13. RECONSTRUCT TRAIN STRUCTURAL MEAN
# --------------------------------------------------------------------------------------------------

alpha_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ]
)


beta_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "beta_spline"
    ]
)


youth_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ]
)


athlete_intercept_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ]
)


athlete_slope_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ]
)


sigma_group_hat = (
    HB2025_V4_POSTERIOR_MEANS[
        "sigma_group"
    ]
)


TRAIN[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx_train
    ]

    +

    np.sum(
        B_TRAIN
        *
        beta_hat[
            sex_idx_train
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx_train
    ]
    *
    youth_train
)


TRAIN[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx_train
    ]
)


TRAIN[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx_train
    ]
    *
    age_scaled_train
)


TRAIN[
    "v4_structural_mean"
] = (
    TRAIN[
        "v4_population_component"
    ]

    +

    TRAIN[
        "v4_athlete_intercept_component"
    ]

    +

    TRAIN[
        "v4_athlete_slope_component"
    ]
)


TRAIN[
    "v4_residual"
] = (
    TRAIN[
        "mark_numeric_m"
    ]

    -

    TRAIN[
        "v4_structural_mean"
    ]
)


TRAIN[
    "v4_error"
] = (
    TRAIN[
        "v4_residual"
    ]
)


TRAIN_PREDICTION_FINITE_PASS = bool(
    np.isfinite(
        TRAIN[
            [
                "v4_population_component",
                "v4_athlete_intercept_component",
                "v4_athlete_slope_component",
                "v4_structural_mean",
                "v4_residual",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 14. TRAIN METRICS
# --------------------------------------------------------------------------------------------------

TRAIN_MAE = float(
    TRAIN[
        "v4_error"
    ]
    .abs()
    .mean()
)


TRAIN_RMSE = float(
    np.sqrt(
        np.mean(
            TRAIN[
                "v4_error"
            ]
            ** 2
        )
    )
)


TRAIN_BIAS = float(
    TRAIN[
        "v4_error"
    ]
    .mean()
)


TRAIN_METRICS = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "n":
                int(
                    len(TRAIN)
                ),

            "athletes":
                int(
                    TRAIN[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "MAE":
                TRAIN_MAE,

            "RMSE":
                TRAIN_RMSE,

            "Bias":
                TRAIN_BIAS,
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. POSTERIOR HYPERPARAMETER SUMMARY
# --------------------------------------------------------------------------------------------------

POSTERIOR_SUMMARY = pd.DataFrame(
    [
        {
            "parameter":
                "nu",

            "value":
                HB2025_V4_POSTERIOR_MEANS[
                    "nu"
                ],
        },

        {
            "parameter":
                "sigma_athlete_intercept",

            "value":
                HB2025_V4_POSTERIOR_MEANS[
                    "sigma_athlete_intercept"
                ],
        },

        {
            "parameter":
                "sigma_athlete_slope",

            "value":
                HB2025_V4_POSTERIOR_MEANS[
                    "sigma_athlete_slope"
                ],
        },

        *[
            {
                "parameter":
                    f"sigma_group_{group}",

                "value":
                    float(
                        sigma_group_hat[i]
                    ),
            }

            for i, group
            in enumerate(
                GROUP_LABELS
            )
        ],
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL QA
# --------------------------------------------------------------------------------------------------

NO_FUTURE_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2024-12-31"
    )
)


CELL_45_PASS = bool(
    ADVI_FINITE_PASS
    and
    POSTERIOR_SHAPE_PASS
    and
    POSTERIOR_FINITE_PASS
    and
    POSITIVE_SCALE_PASS
    and
    TRAIN_PREDICTION_FINITE_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 17. SAVE IMMUTABLE FIT ARTIFACTS
# --------------------------------------------------------------------------------------------------

POSTERIOR_MEANS_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_posterior_means.npz"
)


ADVI_HISTORY_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_advi_history.npy"
)


TRAIN_STRUCTURAL_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_train_structural_predictions.parquet"
)


FIT_SPEC_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_fit_spec.json"
)


FIT_AUDIT_PATH = (
    V4_ATTEMPT_DIR
    /
    "v4_fit_audit.json"
)


np.savez_compressed(
    POSTERIOR_MEANS_PATH,

    alpha_sex=
        HB2025_V4_POSTERIOR_MEANS[
            "alpha_sex"
        ],

    beta_spline=
        HB2025_V4_POSTERIOR_MEANS[
            "beta_spline"
        ],

    youth_implement_effect=
        HB2025_V4_POSTERIOR_MEANS[
            "youth_implement_effect"
        ],

    athlete_intercept=
        HB2025_V4_POSTERIOR_MEANS[
            "athlete_intercept"
        ],

    athlete_slope=
        HB2025_V4_POSTERIOR_MEANS[
            "athlete_slope"
        ],

    sigma_group=
        HB2025_V4_POSTERIOR_MEANS[
            "sigma_group"
        ],

    nu=np.array(
        [
            HB2025_V4_POSTERIOR_MEANS[
                "nu"
            ]
        ]
    ),

    sigma_athlete_intercept=np.array(
        [
            HB2025_V4_POSTERIOR_MEANS[
                "sigma_athlete_intercept"
            ]
        ]
    ),

    sigma_athlete_slope=np.array(
        [
            HB2025_V4_POSTERIOR_MEANS[
                "sigma_athlete_slope"
            ]
        ]
    ),

    sigma_spline=
        HB2025_V4_POSTERIOR_MEANS[
            "sigma_spline"
        ],
)


np.save(
    ADVI_HISTORY_PATH,
    ADVI_HISTORY,
)


TRAIN.to_parquet(
    TRAIN_STRUCTURAL_PATH,
    index=False,
)


FIT_SPEC = {
    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "status":
        (
            "PASSED"
            if CELL_45_PASS
            else
            "FAILED"
        ),

    "train_end":
        "2024-12-31",

    "train_rows":
        int(
            len(TRAIN)
        ),

    "train_athletes":
        int(
            N_ATHLETES
        ),

    "model_specification_changed_from_previous_folds":
        False,

    "likelihood":
        "StudentT",

    "inference": {
        "method":
            "ADVI",

        "iterations":
            ADVI_ITERATIONS,

        "posterior_draws":
            POSTERIOR_DRAWS,

        "random_seed":
            RANDOM_SEED,
    },

    "priors": {
        "alpha_sex":
            "Normal(45,15)",

        "sigma_spline":
            "HalfNormal(2)",

        "spline_start":
            "Normal(0,3)",

        "spline_delta":
            "Normal(0,sigma_spline)",

        "youth_implement_effect":
            "Normal(0,8)",

        "sigma_athlete_intercept":
            "HalfNormal(10)",

        "athlete_intercept_z":
            "Normal(0,1)",

        "sigma_athlete_slope":
            "HalfNormal(4)",

        "athlete_slope_z":
            "Normal(0,1)",

        "sigma_group":
            "HalfNormal(5)",

        "nu_minus_two":
            "Exponential(0.1)",
    },

    "train_metrics": {
        "MAE":
            TRAIN_MAE,

        "RMSE":
            TRAIN_RMSE,

        "Bias":
            TRAIN_BIAS,
    },
}


atomic_json_write(
    FIT_SPEC,
    FIT_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 18. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "preparation_audit":
        Path(
            HB2025_PREP_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(path)
                if path.exists()
                else None
            ),
    }


output_paths = {
    "posterior_means":
        POSTERIOR_MEANS_PATH,

    "advi_history":
        ADVI_HISTORY_PATH,

    "train_structural_predictions":
        TRAIN_STRUCTURAL_PATH,

    "fit_spec":
        FIT_SPEC_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(path),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


FIT_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V4_BAYESIAN_REFIT",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_45_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Third and final historical temporal replication of "
            "the frozen V4 -> V5 -> V6 -> V6.1 architecture."
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "fit_specification":
        FIT_SPEC,

    "posterior_hyperparameters": {
        row[
            "parameter"
        ]:
            float(
                row[
                    "value"
                ]
            )

        for _, row
        in POSTERIOR_SUMMARY.iterrows()
    },

    "advi": {
        "history_length":
            int(
                len(
                    ADVI_HISTORY
                )
            ),

        "initial_loss":
            float(
                ADVI_HISTORY[0]
            ),

        "final_loss":
            float(
                ADVI_HISTORY[-1]
            ),

        "mean_final_1000_loss":
            float(
                ADVI_HISTORY[
                    -1000:
                ]
                .mean()
            ),
    },

    "qa": {
        "advi_finite":
            ADVI_FINITE_PASS,

        "posterior_shapes":
            POSTERIOR_SHAPE_PASS,

        "posterior_finite":
            POSTERIOR_FINITE_PASS,

        "positive_scales_nu":
            POSITIVE_SCALE_PASS,

        "train_predictions_finite":
            TRAIN_PREDICTION_FINITE_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_45_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "pymc":
            pm.__version__,

        "pytensor":
            pytensor.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    FIT_AUDIT,
    FIT_AUDIT_PATH,
)


FIT_AUDIT_SHA256 = sha256_file(
    FIT_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_V4_POINTER_PATH = (
    V4_ROOT
    /
    "LATEST_V4_REFIT.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_45_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V4_ATTEMPT_DIR
            ),

        "posterior_means":
            str(
                POSTERIOR_MEANS_PATH
            ),

        "train_structural_predictions":
            str(
                TRAIN_STRUCTURAL_PATH
            ),

        "fit_spec":
            str(
                FIT_SPEC_PATH
            ),

        "audit_manifest":
            str(
                FIT_AUDIT_PATH
            ),

        "audit_sha256":
            FIT_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_V4_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v4_refit"
] = {
    "status":
        (
            "PASSED"
            if CELL_45_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            V4_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_V4_POINTER_PATH
        ),

    "posterior_means":
        str(
            POSTERIOR_MEANS_PATH
        ),

    "train_structural_predictions":
        str(
            TRAIN_STRUCTURAL_PATH
        ),

    "audit_manifest":
        str(
            FIT_AUDIT_PATH
        ),

    "audit_sha256":
        FIT_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_V4_REFIT_COMPLETE"
    if CELL_45_PASS
    else
    "HB_2025_V4_REFIT_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_TRAIN_V4 = TRAIN


HB2025_V4_ATTEMPT_DIR = (
    V4_ATTEMPT_DIR
)


HB2025_V4_POSTERIOR_MEANS_PATH = (
    POSTERIOR_MEANS_PATH
)


HB2025_V4_FIT_AUDIT_PATH = (
    FIT_AUDIT_PATH
)


NOTEBOOK_2_CELL_45_PASS = (
    CELL_45_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. RELEASE LARGE POSTERIOR DRAW OBJECT
# --------------------------------------------------------------------------------------------------

del HB2025_V4_IDATA

gc.collect()


# --------------------------------------------------------------------------------------------------
# 23. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2025 — HISTORICAL V4 BAYESIAN REFIT")
print("=" * 160)


print()
print("FIT CONFIGURATION")
print("-" * 160)

print(
    f"TRAIN rows              : "
    f"{len(TRAIN):,}"
)

print(
    f"TRAIN athletes          : "
    f"{N_ATHLETES:,}"
)

print(
    f"ADVI iterations         : "
    f"{ADVI_ITERATIONS:,}"
)

print(
    f"Posterior draws         : "
    f"{POSTERIOR_DRAWS:,}"
)

print(
    f"Random seed             : "
    f"{RANDOM_SEED}"
)


print()
print("ADVI")
print("-" * 160)

print(
    f"History length          : "
    f"{len(ADVI_HISTORY):,}"
)

print(
    f"Initial loss            : "
    f"{ADVI_HISTORY[0]:,.6f}"
)

print(
    f"Final loss              : "
    f"{ADVI_HISTORY[-1]:,.6f}"
)

print(
    f"Mean final 1,000 loss   : "
    f"{ADVI_HISTORY[-1000:].mean():,.6f}"
)


print()
print("POSTERIOR HYPERPARAMETER MEANS")
print("-" * 160)

display(
    POSTERIOR_SUMMARY
)


print()
print("TRAIN STRUCTURAL FIT")
print("-" * 160)

display(
    TRAIN_METRICS
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID              : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable fit directory : "
    f"{V4_ATTEMPT_DIR}"
)

print(
    f"Audit manifest          : "
    f"{FIT_AUDIT_PATH}"
)

print(
    f"Audit SHA256            : "
    f"{FIT_AUDIT_SHA256}"
)

print(
    f"Git commit              : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"ADVI finite             : "
    f"{ADVI_FINITE_PASS}"
)

print(
    f"Posterior shapes        : "
    f"{POSTERIOR_SHAPE_PASS}"
)

print(
    f"Posterior finite        : "
    f"{POSTERIOR_FINITE_PASS}"
)

print(
    f"Positive scales / nu>2 : "
    f"{POSITIVE_SCALE_PASS}"
)

print(
    f"TRAIN predictions finite: "
    f"{TRAIN_PREDICTION_FINITE_PASS}"
)

print(
    f"No future TRAIN data    : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_45_PASS:

    print(
        "CELL 45 STATUS: HB_2025 HISTORICAL V4 REFIT PASSED"
    )

else:

    print(
        "CELL 45 STATUS: HB_2025 V4 REFIT REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 36
# Original source SHA256: 882acb6d4be958f8ace436396e7957d34bea570efc27f4ab9bb7283b14b680b8
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 46 — HB_2025 HISTORICAL V5 OU REFIT + IMMUTABLE AUDIT PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Refit the continuous-time V5 OU transient-state model using ONLY
# HB_2025 TRAIN residuals from the historical V4 refit.
#
# TRAIN:
#       <= 2024-12-31
#
# NO 2025 VALIDATION or TEST data used.
#
# Model specification and optimizer procedure are unchanged from HB_2023 / HB_2024.
#
# ==================================================================================================

import hashlib
import json
import math
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy

from scipy.optimize import minimize


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_45_PASS",

    "HB2025_TRAIN_V4",
    "HB2025_V4_POSTERIOR_MEANS",

    "HB2025_V4_ATTEMPT_DIR",
    "HB2025_V4_FIT_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_45_PASS:
    raise RuntimeError(
        "Cell 45 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_46_HB2025_V5_OU_REFIT"

STAGE_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


N_GROUPS = len(
    GROUP_LABELS
)


TAU_MIN = 2.0
TAU_MAX = 5000.0

VAR_MIN = 1e-4
VAR_MAX = 400.0

N_STARTS = 10


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2025_DIR = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


V5_STAGE_ROOT = (
    HB2025_DIR
    /
    "v5_ou_refit"
)


V5_ATTEMPT_DIR = (
    V5_STAGE_ROOT
    /
    ATTEMPT_ID
)


V5_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. TRAIN RESIDUAL INPUT
# --------------------------------------------------------------------------------------------------

TRAIN = (
    HB2025_TRAIN_V4.copy()
    .reset_index(
        drop=True
    )
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "model_group",
    "v4_residual",
]


missing_columns = [
    column
    for column in required_columns
    if column not in TRAIN.columns
]


if missing_columns:
    raise RuntimeError(
        "HB2025_TRAIN_V4 missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


TRAIN[
    "result_date_dt"
] = pd.to_datetime(
    TRAIN[
        "result_date_dt"
    ]
)


if TRAIN[
    "v4_residual"
].isna().any():

    raise RuntimeError(
        "HB_2025 TRAIN contains missing V4 residuals."
    )


if TRAIN[
    "result_date_dt"
].max() > pd.Timestamp(
    "2024-12-31"
):

    raise RuntimeError(
        "Future data detected in HB_2025 V5 fit input."
    )


# --------------------------------------------------------------------------------------------------
# 6. ONE STATE OBSERVATION PER ATHLETE × DISTINCT DATE
# --------------------------------------------------------------------------------------------------

daily = (
    TRAIN.groupby(
        [
            "aaAthleteId",
            "result_date_dt",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        state_observation=(
            "v4_residual",
            "mean",
        ),

        n_performances=(
            "v4_residual",
            "size",
        ),
    )
)


group_counts = pd.crosstab(
    index=[
        TRAIN[
            "aaAthleteId"
        ],

        TRAIN[
            "result_date_dt"
        ],
    ],

    columns=
        TRAIN[
            "model_group"
        ],
)


group_counts = (
    group_counts.reindex(
        columns=GROUP_LABELS,
        fill_value=0,
    )
    .reset_index()
)


group_counts.columns.name = None


for group in GROUP_LABELS:

    group_counts = group_counts.rename(
        columns={
            group:
                f"n_{group}"
        }
    )


daily = daily.merge(
    group_counts,

    on=[
        "aaAthleteId",
        "result_date_dt",
    ],

    how="left",

    validate="one_to_one",
)


daily = (
    daily.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DISTINCT-DATE CHRONOLOGY
# --------------------------------------------------------------------------------------------------

daily[
    "previous_state_date"
] = (
    daily.groupby(
        "aaAthleteId",
        sort=False,
    )[
        "result_date_dt"
    ]
    .shift(1)
)


daily[
    "state_gap_days"
] = (
    daily[
        "result_date_dt"
    ]
    -
    daily[
        "previous_state_date"
    ]
).dt.days


daily[
    "athlete_state_start"
] = (
    daily[
        "previous_state_date"
    ]
    .isna()
)


# --------------------------------------------------------------------------------------------------
# 8. STATE INPUT QA
# --------------------------------------------------------------------------------------------------

DUPLICATE_STATE_COUNT = int(
    daily.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
)


NONPOSITIVE_GAP_COUNT = int(
    (
        daily.loc[
            ~daily[
                "athlete_state_start"
            ],
            "state_gap_days",
        ]
        <=
        0
    )
    .sum()
)


FIRST_DATE_COUNT = int(
    daily[
        "athlete_state_start"
    ]
    .sum()
)


N_STATE_ATHLETES = int(
    daily[
        "aaAthleteId"
    ]
    .nunique()
)


FIRST_DATE_ACCOUNTING_PASS = bool(
    FIRST_DATE_COUNT
    ==
    N_STATE_ATHLETES
)


COUNT_ACCOUNTING_PASS = bool(
    (
        daily[
            [
                f"n_{group}"
                for group in GROUP_LABELS
            ]
        ]
        .sum(
            axis=1
        )
        .to_numpy()

        ==

        daily[
            "n_performances"
        ]
        .to_numpy()
    )
    .all()
)


STATE_INPUT_PASS = bool(
    DUPLICATE_STATE_COUNT == 0
    and
    NONPOSITIVE_GAP_COUNT == 0
    and
    FIRST_DATE_ACCOUNTING_PASS
    and
    COUNT_ACCOUNTING_PASS
    and
    np.isfinite(
        daily[
            "state_observation"
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


if not STATE_INPUT_PASS:
    raise RuntimeError(
        "HB_2025 V5 state-input QA failed."
    )


# --------------------------------------------------------------------------------------------------
# 9. NUMERICAL ARRAYS
# --------------------------------------------------------------------------------------------------

state_y = (
    daily[
        "state_observation"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_n = (
    daily[
        "n_performances"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_counts = (
    daily[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=np.float64
    )
)


state_start = (
    daily[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=np.int8
    )
)


state_gap = (
    daily[
        "state_gap_days"
    ]
    .fillna(
        0
    )
    .to_numpy(
        dtype=np.float64
    )
)


# --------------------------------------------------------------------------------------------------
# 10. KALMAN NEGATIVE LOG-LIKELIHOOD
# --------------------------------------------------------------------------------------------------

LOG_2PI = float(
    np.log(
        2.0
        *
        np.pi
    )
)


def kalman_nll_python(
    theta_log,
):

    tau = float(
        np.exp(
            theta_log[0]
        )
    )

    Vz = float(
        np.exp(
            theta_log[1]
        )
    )

    Rg = np.exp(
        theta_log[
            2:
            2 + N_GROUPS
        ]
    )


    m_post = 0.0
    P_post = Vz

    nll = 0.0


    for i in range(
        len(
            state_y
        )
    ):

        if state_start[i]:

            m_prior = 0.0
            P_prior = Vz

        else:

            phi = math.exp(
                -
                state_gap[i]
                /
                tau
            )


            m_prior = (
                phi
                *
                m_post
            )


            P_prior = (
                phi
                *
                phi
                *
                P_post

                +

                Vz
                *
                (
                    1.0
                    -
                    phi
                    *
                    phi
                )
            )


        R_d = float(
            np.dot(
                state_counts[i],
                Rg,
            )
            /
            (
                state_n[i]
                **
                2
            )
        )


        S = (
            P_prior
            +
            R_d
        )


        if (
            not np.isfinite(S)
            or
            S <= 0
        ):
            return np.inf


        innovation = (
            state_y[i]
            -
            m_prior
        )


        nll += 0.5 * (
            LOG_2PI
            +
            np.log(S)
            +
            innovation
            *
            innovation
            /
            S
        )


        K = (
            P_prior
            /
            S
        )


        m_post = (
            m_prior
            +
            K
            *
            innovation
        )


        P_post = (
            (
                1.0
                -
                K
            )
            *
            P_prior
        )


        P_post = max(
            P_post,
            1e-12,
        )


    return float(
        nll
    )


# --------------------------------------------------------------------------------------------------
# 11. OPTIONAL NUMBA ACCELERATION
# --------------------------------------------------------------------------------------------------

NUMBA_AVAILABLE = False


try:

    import numba

    from numba import njit


    @njit(
        cache=False
    )
    def kalman_nll_numba(
        theta_log,
        state_y_,
        state_n_,
        state_counts_,
        state_start_,
        state_gap_,
    ):

        tau = np.exp(
            theta_log[0]
        )


        Vz = np.exp(
            theta_log[1]
        )


        Rg = np.exp(
            theta_log[
                2:
                2 + N_GROUPS
            ]
        )


        m_post = 0.0
        P_post = Vz

        nll = 0.0


        for i in range(
            state_y_.shape[0]
        ):

            if state_start_[i] == 1:

                m_prior = 0.0
                P_prior = Vz

            else:

                phi = np.exp(
                    -
                    state_gap_[i]
                    /
                    tau
                )


                m_prior = (
                    phi
                    *
                    m_post
                )


                P_prior = (
                    phi
                    *
                    phi
                    *
                    P_post

                    +

                    Vz
                    *
                    (
                        1.0
                        -
                        phi
                        *
                        phi
                    )
                )


            weighted_R = 0.0


            for g in range(
                N_GROUPS
            ):

                weighted_R += (
                    state_counts_[
                        i,
                        g
                    ]
                    *
                    Rg[g]
                )


            R_d = (
                weighted_R
                /
                (
                    state_n_[i]
                    *
                    state_n_[i]
                )
            )


            S = (
                P_prior
                +
                R_d
            )


            if (
                not np.isfinite(S)
                or
                S <= 0
            ):
                return np.inf


            innovation = (
                state_y_[i]
                -
                m_prior
            )


            nll += 0.5 * (
                LOG_2PI
                +
                np.log(S)
                +
                innovation
                *
                innovation
                /
                S
            )


            K = (
                P_prior
                /
                S
            )


            m_post = (
                m_prior
                +
                K
                *
                innovation
            )


            P_post = (
                (
                    1.0
                    -
                    K
                )
                *
                P_prior
            )


            if P_post < 1e-12:
                P_post = 1e-12


        return nll


    _ = kalman_nll_numba(
        np.log(
            np.array(
                [
                    100.0,
                    3.0,
                    10.0,
                    8.0,
                    6.0,
                    6.0,
                ],
                dtype=np.float64,
            )
        ),

        state_y[:10],
        state_n[:10],
        state_counts[:10],
        state_start[:10],
        state_gap[:10],
    )


    NUMBA_AVAILABLE = True


except Exception:

    NUMBA_AVAILABLE = False


def objective(
    theta_log,
):

    if NUMBA_AVAILABLE:

        return float(
            kalman_nll_numba(
                theta_log,
                state_y,
                state_n,
                state_counts,
                state_start,
                state_gap,
            )
        )

    return kalman_nll_python(
        theta_log
    )


# --------------------------------------------------------------------------------------------------
# 12. DETERMINISTIC MULTI-START OPTIMIZATION
# --------------------------------------------------------------------------------------------------

v4_sigma_group = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "sigma_group"
    ],
    dtype=float,
)


R_START_BASE = (
    v4_sigma_group
    **
    2
)


STATE_VARIANCE_DIAGNOSTIC = float(
    np.var(
        state_y,
        ddof=1,
    )
)


start_specs = [
    (30.0,  0.10, 0.75),
    (45.0,  0.25, 1.00),
    (60.0,  0.20, 1.25),
    (90.0,  0.30, 1.00),
    (120.0, 0.50, 0.75),
    (180.0, 0.75, 1.00),
    (240.0, 1.00, 1.25),
    (365.0, 1.50, 1.00),
    (730.0, 2.00, 0.75),
    (150.0, 0.25, 1.25),
]


log_bounds = [
    (
        np.log(
            TAU_MIN
        ),
        np.log(
            TAU_MAX
        ),
    ),

    (
        np.log(
            VAR_MIN
        ),
        np.log(
            VAR_MAX
        ),
    ),

    *[
        (
            np.log(
                VAR_MIN
            ),
            np.log(
                VAR_MAX
            ),
        )

        for _ in range(
            N_GROUPS
        )
    ],
]


optimizer_rows = []
optimizer_results = []


for start_id, (
    tau_start,
    vz_fraction,
    r_multiplier,
) in enumerate(
    start_specs,
    start=1,
):

    Vz_start = float(
        np.clip(
            STATE_VARIANCE_DIAGNOSTIC
            *
            vz_fraction,

            0.05,
            50.0,
        )
    )


    R_start = np.clip(
        R_START_BASE
        *
        r_multiplier,

        0.05,
        100.0,
    )


    x0 = np.log(
        np.concatenate(
            [
                [
                    tau_start,
                    Vz_start,
                ],

                R_start,
            ]
        )
    )


    result = minimize(
        objective,

        x0=x0,

        method="L-BFGS-B",

        bounds=log_bounds,

        options={
            "maxiter":
                3000,

            "ftol":
                1e-11,

            "gtol":
                1e-7,

            "maxls":
                50,
        },
    )


    optimizer_results.append(
        result
    )


    fitted = np.exp(
        result.x
    )


    optimizer_rows.append(
        {
            "start_id":
                start_id,

            "success":
                bool(
                    result.success
                ),

            "status":
                int(
                    result.status
                ),

            "message":
                str(
                    result.message
                ),

            "iterations":
                int(
                    result.nit
                ),

            "n_function_evals":
                int(
                    result.nfev
                ),

            "nll":
                float(
                    result.fun
                ),

            "tau_days":
                float(
                    fitted[0]
                ),

            "state_variance":
                float(
                    fitted[1]
                ),

            "state_sigma":
                float(
                    np.sqrt(
                        fitted[1]
                    )
                ),

            "R_M700":
                float(
                    fitted[2]
                ),

            "R_M800":
                float(
                    fitted[3]
                ),

            "R_W500":
                float(
                    fitted[4]
                ),

            "R_W600":
                float(
                    fitted[5]
                ),
        }
    )


OPTIMIZER_AUDIT = pd.DataFrame(
    optimizer_rows
)


# --------------------------------------------------------------------------------------------------
# 13. SELECT BEST SUCCESSFUL RESULT
# --------------------------------------------------------------------------------------------------

successful_indices = [
    i
    for i, result in enumerate(
        optimizer_results
    )
    if (
        result.success
        and
        np.isfinite(
            result.fun
        )
    )
]


if not successful_indices:
    raise RuntimeError(
        "No successful HB_2025 V5 OU optimizer starts."
    )


best_index = min(
    successful_indices,
    key=lambda i:
        optimizer_results[i].fun,
)


BEST_RESULT = (
    optimizer_results[
        best_index
    ]
)


BEST_PARAMS = np.exp(
    BEST_RESULT.x
)


V5_TAU_DAYS = float(
    BEST_PARAMS[0]
)


V5_STATE_VARIANCE = float(
    BEST_PARAMS[1]
)


V5_STATE_SIGMA = float(
    np.sqrt(
        V5_STATE_VARIANCE
    )
)


V5_R_GROUP = {
    group:
        float(
            BEST_PARAMS[
                2 + i
            ]
        )

    for i, group
    in enumerate(
        GROUP_LABELS
    )
}


V5_HALF_LIFE_DAYS = float(
    V5_TAU_DAYS
    *
    np.log(
        2.0
    )
)


BEST_NLL = float(
    BEST_RESULT.fun
)


N_PARAMETERS = (
    2
    +
    N_GROUPS
)


V5_AIC = float(
    2
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


V5_BIC = float(
    np.log(
        len(
            daily
        )
    )
    *
    N_PARAMETERS

    +

    2
    *
    BEST_NLL
)


# --------------------------------------------------------------------------------------------------
# 14. PERSISTENCE REFERENCE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        1,
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS,

        "phi":
            np.exp(
                -
                REFERENCE_DAYS
                /
                V5_TAU_DAYS
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 15. QA
# --------------------------------------------------------------------------------------------------

SUCCESS_COUNT = int(
    OPTIMIZER_AUDIT[
        "success"
    ]
    .sum()
)


BOUNDARY_TOL = 1e-4


BOUNDARY_FLAGS = []


for value, (
    lower,
    upper,
) in zip(
    BEST_RESULT.x,
    log_bounds,
):

    BOUNDARY_FLAGS.append(
        bool(
            abs(
                value
                -
                lower
            )
            <
            BOUNDARY_TOL

            or

            abs(
                value
                -
                upper
            )
            <
            BOUNDARY_TOL
        )
    )


NO_BOUNDARY_HIT_PASS = bool(
    not any(
        BOUNDARY_FLAGS
    )
)


ALL_PARAMS_FINITE_PASS = bool(
    np.isfinite(
        BEST_PARAMS
    )
    .all()
)


ALL_PARAMS_POSITIVE_PASS = bool(
    (
        BEST_PARAMS
        >
        0
    )
    .all()
)


MULTISTART_PASS = bool(
    SUCCESS_COUNT
    >=
    2
)


NO_FUTURE_DATA_PASS = bool(
    TRAIN[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2024-12-31"
    )
)


CELL_46_PASS = bool(
    STATE_INPUT_PASS
    and
    MULTISTART_PASS
    and
    ALL_PARAMS_FINITE_PASS
    and
    ALL_PARAMS_POSITIVE_PASS
    and
    NO_BOUNDARY_HIT_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 16. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

STATE_INPUT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_state_input.parquet"
)


OPTIMIZER_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_optimizer_starts.parquet"
)


PARAMETER_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_ou_parameters.json"
)


FIT_AUDIT_PATH = (
    V5_ATTEMPT_DIR
    /
    "v5_fit_audit.json"
)


daily.to_parquet(
    STATE_INPUT_PATH,
    index=False,
)


OPTIMIZER_AUDIT.to_parquet(
    OPTIMIZER_AUDIT_PATH,
    index=False,
)


V5_PARAMETER_SPEC = {
    "model":
        "V5_CONTINUOUS_TIME_OU",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "model_logic_changed_from_previous_folds":
        False,

    "fit_population":
        "HB_2025 TRAIN only",

    "train_end":
        "2024-12-31",

    "state_rows":
        int(
            len(
                daily
            )
        ),

    "state_athletes":
        N_STATE_ATHLETES,

    "parameters": {
        "tau_days":
            V5_TAU_DAYS,

        "half_life_days":
            V5_HALF_LIFE_DAYS,

        "state_variance":
            V5_STATE_VARIANCE,

        "state_sigma":
            V5_STATE_SIGMA,

        "measurement_variance_by_group":
            V5_R_GROUP,
    },

    "fit_statistics": {
        "negative_log_likelihood":
            BEST_NLL,

        "AIC":
            V5_AIC,

        "BIC":
            V5_BIC,

        "n_parameters":
            N_PARAMETERS,
    },

    "optimizer": {
        "method":
            "L-BFGS-B",

        "deterministic_starts":
            N_STARTS,

        "successful_starts":
            SUCCESS_COUNT,

        "best_start_id":
            int(
                best_index
                +
                1
            ),
    },

    "measurement_variance_formula":
        "R_d = sum_g(n_dg * R_g) / n_d^2",

    "state_transition":
        (
            "phi(delta)=exp(-delta/tau); "
            "Var innovation=Vz*(1-phi(delta)^2)"
        ),

    "group_order":
        GROUP_LABELS,

    "randomness":
        "No stochastic fitting; deterministic optimizer starts",

    "numba_used":
        NUMBA_AVAILABLE,
}


atomic_json_write(
    V5_PARAMETER_SPEC,
    PARAMETER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 17. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v4_posterior_means":
        (
            Path(
                HB2025_V4_ATTEMPT_DIR
            )
            /
            "v4_posterior_means.npz"
        ),

    "v4_train_structural_predictions":
        (
            Path(
                HB2025_V4_ATTEMPT_DIR
            )
            /
            "v4_train_structural_predictions.parquet"
        ),

    "v4_fit_spec":
        (
            Path(
                HB2025_V4_ATTEMPT_DIR
            )
            /
            "v4_fit_spec.json"
        ),

    "v4_fit_audit":
        Path(
            HB2025_V4_FIT_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else
                None
            ),
    }


OUTPUT_ARTIFACTS = {}


for name, path in {
    "state_input":
        STATE_INPUT_PATH,

    "optimizer_audit":
        OPTIMIZER_AUDIT_PATH,

    "parameter_spec":
        PARAMETER_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 18. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

FIT_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V5_OU_REFIT",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_46_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Third and final historical temporal replication "
            "of the frozen V5 OU specification."
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "data_vintage_note":
        (
            "Historical fold reconstructed from the current 2026 canonical "
            "snapshot and restricted by event/result date. Later historical "
            "corrections may therefore be present; no future-dated performances "
            "are used."
        ),

    "state_population": {
        "performance_rows":
            int(
                len(
                    TRAIN
                )
            ),

        "state_rows":
            int(
                len(
                    daily
                )
            ),

        "athletes":
            N_STATE_ATHLETES,

        "multi_performance_dates":
            int(
                (
                    daily[
                        "n_performances"
                    ]
                    >
                    1
                )
                .sum()
            ),

        "multi_group_dates":
            int(
                (
                    (
                        daily[
                            [
                                f"n_{group}"
                                for group in GROUP_LABELS
                            ]
                        ]
                        >
                        0
                    )
                    .sum(
                        axis=1
                    )
                    >
                    1
                )
                .sum()
            ),
    },

    "model_specification":
        V5_PARAMETER_SPEC,

    "qa": {
        "duplicate_state_dates":
            DUPLICATE_STATE_COUNT,

        "nonpositive_state_gaps":
            NONPOSITIVE_GAP_COUNT,

        "first_date_accounting":
            FIRST_DATE_ACCOUNTING_PASS,

        "performance_count_accounting":
            COUNT_ACCOUNTING_PASS,

        "state_input_pass":
            STATE_INPUT_PASS,

        "successful_optimizer_starts":
            SUCCESS_COUNT,

        "multistart_pass":
            MULTISTART_PASS,

        "parameters_finite":
            ALL_PARAMS_FINITE_PASS,

        "parameters_positive":
            ALL_PARAMS_POSITIVE_PASS,

        "no_parameter_at_boundary":
            NO_BOUNDARY_HIT_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_46_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "scipy":
            scipy.__version__,

        "numba":
            (
                numba.__version__
                if NUMBA_AVAILABLE
                else
                None
            ),
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    FIT_AUDIT,
    FIT_AUDIT_PATH,
)


FIT_AUDIT_SHA256 = sha256_file(
    FIT_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    V5_STAGE_ROOT
    /
    "LATEST_V5_OU_REFIT.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V5_OU_REFIT",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_46_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V5_ATTEMPT_DIR
            ),

        "parameter_path":
            str(
                PARAMETER_PATH
            ),

        "fit_audit":
            str(
                FIT_AUDIT_PATH
            ),

        "fit_audit_sha256":
            FIT_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_ou_refit"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_46_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "attempt_directory":
        str(
            V5_ATTEMPT_DIR
        ),

    "fit_audit":
        str(
            FIT_AUDIT_PATH
        ),

    "fit_audit_sha256":
        FIT_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_V5_OU_REFIT_COMPLETE"
    if CELL_46_PASS
    else
    "HB_2025_V5_OU_REFIT_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. GLOBALS FOR CELL 47
# --------------------------------------------------------------------------------------------------

HB2025_V5_STATE_INPUT = (
    daily
)


HB2025_V5_TAU_DAYS = (
    V5_TAU_DAYS
)


HB2025_V5_HALF_LIFE_DAYS = (
    V5_HALF_LIFE_DAYS
)


HB2025_V5_STATE_VARIANCE = (
    V5_STATE_VARIANCE
)


HB2025_V5_STATE_SIGMA = (
    V5_STATE_SIGMA
)


HB2025_V5_R_GROUP = (
    V5_R_GROUP
)


HB2025_V5_ATTEMPT_DIR = (
    V5_ATTEMPT_DIR
)


HB2025_V5_PARAMETER_PATH = (
    PARAMETER_PATH
)


HB2025_V5_FIT_AUDIT_PATH = (
    FIT_AUDIT_PATH
)


NOTEBOOK_2_CELL_46_PASS = (
    CELL_46_PASS
)


# --------------------------------------------------------------------------------------------------
# 22. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2025 — HISTORICAL V5 CONTINUOUS-TIME OU REFIT")
print("=" * 160)


print()
print("STATE INPUT")
print("-" * 160)

print(
    f"Performance residual rows       : "
    f"{len(TRAIN):,}"
)

print(
    f"Distinct athlete-date states    : "
    f"{len(daily):,}"
)

print(
    f"Athletes                        : "
    f"{N_STATE_ATHLETES:,}"
)

print(
    f"Multi-performance state dates   : "
    f"{(daily['n_performances'] > 1).sum():,}"
)

print(
    "Multi-group state dates         : "
    f"{((daily[[f'n_{g}' for g in GROUP_LABELS]] > 0).sum(axis=1) > 1).sum():,}"
)


print()
print("OPTIMIZER ROBUSTNESS")
print("-" * 160)

display(
    OPTIMIZER_AUDIT
)


print()
print("BEST V5 OU PARAMETERS")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "parameter":
                    "tau_days",

                "value":
                    V5_TAU_DAYS,
            },

            {
                "parameter":
                    "half_life_days",

                "value":
                    V5_HALF_LIFE_DAYS,
            },

            {
                "parameter":
                    "state_variance",

                "value":
                    V5_STATE_VARIANCE,
            },

            {
                "parameter":
                    "state_sigma",

                "value":
                    V5_STATE_SIGMA,
            },

            *[
                {
                    "parameter":
                        f"R_{group}",

                    "value":
                        V5_R_GROUP[
                            group
                        ],
                }

                for group in GROUP_LABELS
            ],
        ]
    )
)


print()
print("PERSISTENCE REFERENCE")
print("-" * 160)

display(
    PERSISTENCE_REFERENCE
)


print()
print("FIT STATISTICS")
print("-" * 160)

print(
    f"NLL                             : "
    f"{BEST_NLL:,.6f}"
)

print(
    f"AIC                             : "
    f"{V5_AIC:,.6f}"
)

print(
    f"BIC                             : "
    f"{V5_BIC:,.6f}"
)

print(
    f"Successful starts               : "
    f"{SUCCESS_COUNT}/{N_STARTS}"
)

print(
    f"Numba used                      : "
    f"{NUMBA_AVAILABLE}"
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{V5_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                  : "
    f"{FIT_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{FIT_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Duplicate state dates           : "
    f"{DUPLICATE_STATE_COUNT}"
)

print(
    f"Nonpositive gaps                : "
    f"{NONPOSITIVE_GAP_COUNT}"
)

print(
    f"First-date accounting           : "
    f"{FIRST_DATE_ACCOUNTING_PASS}"
)

print(
    f"Performance-count accounting    : "
    f"{COUNT_ACCOUNTING_PASS}"
)

print(
    f"State input valid               : "
    f"{STATE_INPUT_PASS}"
)

print(
    f"Multi-start optimizer           : "
    f"{MULTISTART_PASS}"
)

print(
    f"Parameters finite               : "
    f"{ALL_PARAMS_FINITE_PASS}"
)

print(
    f"Parameters positive             : "
    f"{ALL_PARAMS_POSITIVE_PASS}"
)

print(
    f"No parameter at boundary        : "
    f"{NO_BOUNDARY_HIT_PASS}"
)

print(
    f"No future TRAIN data            : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_46_PASS:

    print(
        "CELL 46 STATUS: HB_2025 HISTORICAL V5 OU REFIT PASSED"
    )

else:

    print(
        "CELL 46 STATUS: HB_2025 V5 OU REFIT REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 37
# Original source SHA256: e2950ad1827364d0e1f28f6e30c4fcfd1b66175ea2ae8d66e313d3b80a599933
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 47 — HB_2025 HISTORICAL V5 FILTERED STATE RECOVERY + TERMINAL STATE PACKAGE
# ==================================================================================================
#
# Purpose
# -------
# Replay the frozen HB_2025 V5 OU model through TRAIN:
#
#       <= 2024-12-31
#
# Recover:
#   • filtered state after every athlete-date
#   • terminal state for every TRAIN athlete
#   • history metadata
#   • innovation diagnostics
#
# NO fitting.
# NO parameter changes.
# NO VALIDATION / TEST information used.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_46_PASS",

    "HB2025_V5_STATE_INPUT",
    "HB2025_V5_TAU_DAYS",
    "HB2025_V5_STATE_VARIANCE",
    "HB2025_V5_R_GROUP",

    "HB2025_V5_ATTEMPT_DIR",
    "HB2025_V5_PARAMETER_PATH",
    "HB2025_V5_FIT_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_46_PASS:
    raise RuntimeError(
        "Cell 46 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_47_HB2025_V5_STATE_RECOVERY"

STAGE_SCHEMA_VERSION = "1.0.0"

GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]

LOG_2PI = float(
    np.log(
        2.0 * np.pi
    )
)


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2025_DIR = (
    Path(TRAINING_RUN_DIR)
    / "historical_backtest"
    / FOLD_NAME
)

STATE_STAGE_ROOT = (
    HB2025_DIR
    / "v5_state_recovery"
)

STATE_ATTEMPT_DIR = (
    STATE_STAGE_ROOT
    / ATTEMPT_ID
)

STATE_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. FROZEN V5 PARAMETERS
# --------------------------------------------------------------------------------------------------

with Path(
    HB2025_V5_PARAMETER_PATH
).open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_SPEC = json.load(
        f
    )


FIT_NLL = float(
    V5_PARAMETER_SPEC[
        "fit_statistics"
    ][
        "negative_log_likelihood"
    ]
)

TAU = float(
    HB2025_V5_TAU_DAYS
)

VZ = float(
    HB2025_V5_STATE_VARIANCE
)

R_GROUP = {
    group:
        float(
            HB2025_V5_R_GROUP[
                group
            ]
        )
    for group in GROUP_LABELS
}


# --------------------------------------------------------------------------------------------------
# 6. STATE INPUT
# --------------------------------------------------------------------------------------------------

S = (
    HB2025_V5_STATE_INPUT.copy()
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .reset_index(
        drop=True
    )
)

S[
    "result_date_dt"
] = pd.to_datetime(
    S[
        "result_date_dt"
    ]
)


required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "state_observation",
    "n_performances",
    "athlete_state_start",
    "state_gap_days",

    *[
        f"n_{group}"
        for group in GROUP_LABELS
    ],
]

missing_columns = [
    column
    for column in required_columns
    if column not in S.columns
]

if missing_columns:
    raise RuntimeError(
        "State input missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. MEASUREMENT VARIANCE
# --------------------------------------------------------------------------------------------------

counts = (
    S[
        [
            f"n_{group}"
            for group in GROUP_LABELS
        ]
    ]
    .to_numpy(
        dtype=float
    )
)

R_VECTOR = np.array(
    [
        R_GROUP[
            group
        ]
        for group in GROUP_LABELS
    ],
    dtype=float,
)

n_perf = (
    S[
        "n_performances"
    ]
    .to_numpy(
        dtype=float
    )
)

measurement_variance = (
    counts @ R_VECTOR
) / (
    n_perf ** 2
)

S[
    "measurement_variance"
] = measurement_variance


# --------------------------------------------------------------------------------------------------
# 8. REPLAY KALMAN FILTER
# --------------------------------------------------------------------------------------------------

N = len(S)

prior_mean = np.empty(
    N,
    dtype=float,
)

prior_variance = np.empty(
    N,
    dtype=float,
)

phi_used = np.empty(
    N,
    dtype=float,
)

innovation = np.empty(
    N,
    dtype=float,
)

innovation_variance = np.empty(
    N,
    dtype=float,
)

standardized_innovation = np.empty(
    N,
    dtype=float,
)

kalman_gain = np.empty(
    N,
    dtype=float,
)

filtered_mean = np.empty(
    N,
    dtype=float,
)

filtered_variance = np.empty(
    N,
    dtype=float,
)

log_likelihood_contribution = np.empty(
    N,
    dtype=float,
)


y = (
    S[
        "state_observation"
    ]
    .to_numpy(
        dtype=float
    )
)

gap = (
    S[
        "state_gap_days"
    ]
    .fillna(0)
    .to_numpy(
        dtype=float
    )
)

state_start = (
    S[
        "athlete_state_start"
    ]
    .to_numpy(
        dtype=bool
    )
)


m_post = 0.0
P_post = VZ


for i in range(N):

    # ----------------------------------------------------------------------------------------------
    # OU PRIOR
    # ----------------------------------------------------------------------------------------------

    if state_start[i]:

        phi = 0.0

        m_prior = 0.0
        P_prior = VZ

    else:

        phi = float(
            np.exp(
                -gap[i] / TAU
            )
        )

        m_prior = (
            phi
            *
            m_post
        )

        P_prior = (
            phi
            *
            phi
            *
            P_post

            +

            VZ
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


    # ----------------------------------------------------------------------------------------------
    # MEASUREMENT UPDATE
    # ----------------------------------------------------------------------------------------------

    R_d = measurement_variance[i]

    S_var = (
        P_prior
        +
        R_d
    )

    innov = (
        y[i]
        -
        m_prior
    )

    z = (
        innov
        /
        np.sqrt(
            S_var
        )
    )

    K = (
        P_prior
        /
        S_var
    )

    m_post = (
        m_prior
        +
        K
        *
        innov
    )

    P_post = (
        (
            1.0
            -
            K
        )
        *
        P_prior
    )

    P_post = max(
        P_post,
        1e-12,
    )

    ll = -0.5 * (
        LOG_2PI
        +
        np.log(
            S_var
        )
        +
        innov
        *
        innov
        /
        S_var
    )


    # ----------------------------------------------------------------------------------------------
    # STORE
    # ----------------------------------------------------------------------------------------------

    phi_used[i] = phi

    prior_mean[i] = m_prior
    prior_variance[i] = P_prior

    innovation[i] = innov
    innovation_variance[i] = S_var
    standardized_innovation[i] = z

    kalman_gain[i] = K

    filtered_mean[i] = m_post
    filtered_variance[i] = P_post

    log_likelihood_contribution[i] = ll


# --------------------------------------------------------------------------------------------------
# 9. ATTACH FILTER OUTPUT
# --------------------------------------------------------------------------------------------------

S[
    "state_phi"
] = phi_used

S[
    "state_prior_mean"
] = prior_mean

S[
    "state_prior_variance"
] = prior_variance

S[
    "innovation"
] = innovation

S[
    "innovation_variance"
] = innovation_variance

S[
    "standardized_innovation"
] = standardized_innovation

S[
    "kalman_gain"
] = kalman_gain

S[
    "state_filtered_mean"
] = filtered_mean

S[
    "state_filtered_variance"
] = filtered_variance

S[
    "log_likelihood_contribution"
] = log_likelihood_contribution


# --------------------------------------------------------------------------------------------------
# 10. NLL RECONCILIATION
# --------------------------------------------------------------------------------------------------

REPLAY_NLL = float(
    -S[
        "log_likelihood_contribution"
    ].sum()
)

NLL_ABS_DIFF = float(
    abs(
        REPLAY_NLL
        -
        FIT_NLL
    )
)

NLL_RECONCILIATION_PASS = bool(
    NLL_ABS_DIFF
    <
    1e-5
)


# --------------------------------------------------------------------------------------------------
# 11. ATHLETE HISTORY METADATA
# --------------------------------------------------------------------------------------------------

ATHLETE_HISTORY = (
    S.groupby(
        "aaAthleteId",
        as_index=False,
    )
    .agg(
        first_state_date=(
            "result_date_dt",
            "min",
        ),

        last_state_date=(
            "result_date_dt",
            "max",
        ),

        n_state_dates=(
            "result_date_dt",
            "size",
        ),

        n_state_performances=(
            "n_performances",
            "sum",
        ),
    )
)


ATHLETE_HISTORY[
    "history_span_days"
] = (
    ATHLETE_HISTORY[
        "last_state_date"
    ]
    -
    ATHLETE_HISTORY[
        "first_state_date"
    ]
).dt.days


# --------------------------------------------------------------------------------------------------
# 12. TERMINAL STATE PER ATHLETE
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    S.groupby(
        "aaAthleteId",
        sort=False,
        as_index=False,
    )
    .tail(1)
    .copy()
)


TERMINAL = TERMINAL[
    [
        "aaAthleteId",
        "result_date_dt",
        "state_filtered_mean",
        "state_filtered_variance",
        "state_observation",
        "measurement_variance",
    ]
].rename(
    columns={
        "result_date_dt":
            "terminal_state_date",

        "state_filtered_mean":
            "terminal_state_mean",

        "state_filtered_variance":
            "terminal_state_variance",

        "state_observation":
            "terminal_state_observation",

        "measurement_variance":
            "terminal_measurement_variance",
    }
)


TERMINAL = TERMINAL.merge(
    ATHLETE_HISTORY,
    on="aaAthleteId",
    how="left",
    validate="one_to_one",
)


TERMINAL[
    "terminal_state_sigma"
] = np.sqrt(
    TERMINAL[
        "terminal_state_variance"
    ]
)


TERMINAL = (
    TERMINAL.sort_values(
        "aaAthleteId"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 13. INNOVATION SUMMARY
# --------------------------------------------------------------------------------------------------

Z = (
    S[
        "standardized_innovation"
    ]
    .to_numpy(
        dtype=float
    )
)


INNOVATION_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "n_states":
                int(
                    len(S)
                ),

            "athletes":
                int(
                    S[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "mean_standardized_innovation":
                float(
                    np.mean(Z)
                ),

            "sd_standardized_innovation":
                float(
                    np.std(
                        Z,
                        ddof=1,
                    )
                ),

            "median_standardized_innovation":
                float(
                    np.median(Z)
                ),

            "pct_abs_z_gt_2":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(Z) > 2.0
                    )
                ),

            "pct_abs_z_gt_3":
                float(
                    100.0
                    *
                    np.mean(
                        np.abs(Z) > 3.0
                    )
                ),

            "mean_kalman_gain":
                float(
                    S[
                        "kalman_gain"
                    ]
                    .mean()
                ),

            "mean_filtered_variance":
                float(
                    S[
                        "state_filtered_variance"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. QA
# --------------------------------------------------------------------------------------------------

ROW_ACCOUNTING_PASS = bool(
    len(S)
    ==
    len(
        HB2025_V5_STATE_INPUT
    )
)


UNIQUE_STATE_PASS = bool(
    not S.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .any()
)


FINITE_FILTER_PASS = bool(
    np.isfinite(
        S[
            [
                "measurement_variance",
                "state_prior_mean",
                "state_prior_variance",
                "innovation",
                "innovation_variance",
                "standardized_innovation",
                "kalman_gain",
                "state_filtered_mean",
                "state_filtered_variance",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        S[
            "measurement_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "state_prior_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "innovation_variance"
        ]
        > 0
    )
    .all()

    and

    (
        S[
            "state_filtered_variance"
        ]
        > 0
    )
    .all()
)


GAIN_BOUNDS_PASS = bool(
    S[
        "kalman_gain"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


FIRST_STATE_PRIOR_PASS = bool(
    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_mean",
        ],
        0.0,
        atol=1e-12,
    )

    and

    np.allclose(
        S.loc[
            S[
                "athlete_state_start"
            ],
            "state_prior_variance",
        ],
        VZ,
        atol=1e-10,
        rtol=1e-10,
    )
)


TERMINAL_ACCOUNTING_PASS = bool(
    len(TERMINAL)
    ==
    S[
        "aaAthleteId"
    ]
    .nunique()

    and

    TERMINAL[
        "aaAthleteId"
    ]
    .is_unique
)


TERMINAL_DATE_PASS = bool(
    (
        TERMINAL[
            "terminal_state_date"
        ]
        ==
        TERMINAL[
            "last_state_date"
        ]
    )
    .all()
)


HISTORY_METADATA_PASS = bool(
    (
        TERMINAL[
            "n_state_dates"
        ]
        >=
        1
    )
    .all()

    and

    (
        TERMINAL[
            "history_span_days"
        ]
        >=
        0
    )
    .all()

    and

    (
        TERMINAL[
            "n_state_performances"
        ]
        >=
        TERMINAL[
            "n_state_dates"
        ]
    )
    .all()
)


NO_FUTURE_DATA_PASS = bool(
    S[
        "result_date_dt"
    ]
    .max()
    <=
    pd.Timestamp(
        "2024-12-31"
    )
)


CELL_47_PASS = bool(
    ROW_ACCOUNTING_PASS
    and
    UNIQUE_STATE_PASS
    and
    FINITE_FILTER_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    GAIN_BOUNDS_PASS
    and
    FIRST_STATE_PRIOR_PASS
    and
    TERMINAL_ACCOUNTING_PASS
    and
    TERMINAL_DATE_PASS
    and
    HISTORY_METADATA_PASS
    and
    NLL_RECONCILIATION_PASS
    and
    NO_FUTURE_DATA_PASS
)


# --------------------------------------------------------------------------------------------------
# 15. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

FILTERED_STATE_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_train_filtered_states.parquet"
)


TERMINAL_STATE_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_train_terminal_states.parquet"
)


INNOVATION_SUMMARY_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_innovation_summary.parquet"
)


STATE_RECOVERY_AUDIT_PATH = (
    STATE_ATTEMPT_DIR
    /
    "v5_state_recovery_audit.json"
)


S.to_parquet(
    FILTERED_STATE_PATH,
    index=False,
)


TERMINAL.to_parquet(
    TERMINAL_STATE_PATH,
    index=False,
)


INNOVATION_SUMMARY.to_parquet(
    INNOVATION_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 16. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2025_V5_PARAMETER_PATH
        ),

    "v5_fit_audit":
        Path(
            HB2025_V5_FIT_AUDIT_PATH
        ),

    "v5_state_input":
        (
            Path(
                HB2025_V5_ATTEMPT_DIR
            )
            /
            "v5_state_input.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(path)
                if path.exists()
                else None
            ),
    }


OUTPUT_ARTIFACTS = {}


for name, path in {
    "filtered_states":
        FILTERED_STATE_PATH,

    "terminal_states":
        TERMINAL_STATE_PATH,

    "innovation_summary":
        INNOVATION_SUMMARY_PATH,
}.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(path),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 17. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

STATE_RECOVERY_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V5_STATE_RECOVERY",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_47_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Recover leakage-safe HB_2025 TRAIN filtered and "
            "terminal OU states."
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "replay": {
        "fit_nll":
            FIT_NLL,

        "replay_nll":
            REPLAY_NLL,

        "absolute_nll_difference":
            NLL_ABS_DIFF,

        "nll_reconciliation_pass":
            NLL_RECONCILIATION_PASS,
    },

    "state_population": {
        "rows":
            int(
                len(S)
            ),

        "athletes":
            int(
                S[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "terminal_states":
            int(
                len(TERMINAL)
            ),
    },

    "terminal_state_schema": [
        "aaAthleteId",
        "terminal_state_date",
        "terminal_state_mean",
        "terminal_state_variance",
        "terminal_state_sigma",
        "first_state_date",
        "last_state_date",
        "n_state_dates",
        "n_state_performances",
        "history_span_days",
    ],

    "qa": {
        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "unique_state_dates":
            UNIQUE_STATE_PASS,

        "finite_filter":
            FINITE_FILTER_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "kalman_gain_bounds":
            GAIN_BOUNDS_PASS,

        "first_state_prior":
            FIRST_STATE_PRIOR_PASS,

        "terminal_accounting":
            TERMINAL_ACCOUNTING_PASS,

        "terminal_dates":
            TERMINAL_DATE_PASS,

        "history_metadata":
            HISTORY_METADATA_PASS,

        "nll_reconciliation":
            NLL_RECONCILIATION_PASS,

        "no_future_data":
            NO_FUTURE_DATA_PASS,

        "cell_pass":
            CELL_47_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    STATE_RECOVERY_AUDIT,
    STATE_RECOVERY_AUDIT_PATH,
)


STATE_RECOVERY_AUDIT_SHA256 = sha256_file(
    STATE_RECOVERY_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 18. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_STATE_POINTER_PATH = (
    STATE_STAGE_ROOT
    /
    "LATEST_V5_STATE_RECOVERY.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V5_STATE_RECOVERY",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_47_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                STATE_ATTEMPT_DIR
            ),

        "filtered_states":
            str(
                FILTERED_STATE_PATH
            ),

        "terminal_states":
            str(
                TERMINAL_STATE_PATH
            ),

        "innovation_summary":
            str(
                INNOVATION_SUMMARY_PATH
            ),

        "audit_manifest":
            str(
                STATE_RECOVERY_AUDIT_PATH
            ),

        "audit_sha256":
            STATE_RECOVERY_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_STATE_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 19. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v5_state_recovery"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_47_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_STATE_POINTER_PATH
        ),

    "attempt_directory":
        str(
            STATE_ATTEMPT_DIR
        ),

    "terminal_states":
        str(
            TERMINAL_STATE_PATH
        ),

    "audit_manifest":
        str(
            STATE_RECOVERY_AUDIT_PATH
        ),

    "audit_sha256":
        STATE_RECOVERY_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_V5_STATE_RECOVERY_COMPLETE"
    if CELL_47_PASS
    else
    "HB_2025_V5_STATE_RECOVERY_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_V5_FILTERED_STATES = S

HB2025_V5_TERMINAL_STATES = TERMINAL

HB2025_V5_INNOVATION_SUMMARY = (
    INNOVATION_SUMMARY
)

HB2025_V5_STATE_RECOVERY_ATTEMPT_DIR = (
    STATE_ATTEMPT_DIR
)

HB2025_V5_TERMINAL_STATE_PATH = (
    TERMINAL_STATE_PATH
)

HB2025_V5_STATE_RECOVERY_AUDIT_PATH = (
    STATE_RECOVERY_AUDIT_PATH
)

NOTEBOOK_2_CELL_47_PASS = (
    CELL_47_PASS
)


# --------------------------------------------------------------------------------------------------
# 21. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2025 — HISTORICAL V5 FILTERED STATE RECOVERY")
print("=" * 160)


print()
print("STATE REPLAY")
print("-" * 160)

print(
    f"Filtered athlete-date states    : "
    f"{len(S):,}"
)

print(
    f"Athletes                        : "
    f"{S['aaAthleteId'].nunique():,}"
)

print(
    f"Terminal athlete states         : "
    f"{len(TERMINAL):,}"
)


print()
print("NLL RECONCILIATION")
print("-" * 160)

print(
    f"Fit NLL                         : "
    f"{FIT_NLL:,.9f}"
)

print(
    f"Replay NLL                      : "
    f"{REPLAY_NLL:,.9f}"
)

print(
    f"Absolute difference             : "
    f"{NLL_ABS_DIFF:.12e}"
)

print(
    f"Reconciliation pass             : "
    f"{NLL_RECONCILIATION_PASS}"
)


print()
print("INNOVATION DIAGNOSTICS")
print("-" * 160)

display(
    INNOVATION_SUMMARY
)


print()
print("TERMINAL STATE HISTORY SUMMARY")
print("-" * 160)

display(
    pd.DataFrame(
        [
            {
                "athletes":
                    int(
                        len(TERMINAL)
                    ),

                "median_state_dates":
                    float(
                        TERMINAL[
                            "n_state_dates"
                        ]
                        .median()
                    ),

                "median_history_span_days":
                    float(
                        TERMINAL[
                            "history_span_days"
                        ]
                        .median()
                    ),

                "max_history_span_days":
                    int(
                        TERMINAL[
                            "history_span_days"
                        ]
                        .max()
                    ),

                "median_terminal_state_sigma":
                    float(
                        TERMINAL[
                            "terminal_state_sigma"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory     : "
    f"{STATE_ATTEMPT_DIR}"
)

print(
    f"Terminal-state artifact         : "
    f"{TERMINAL_STATE_PATH}"
)

print(
    f"Audit manifest                  : "
    f"{STATE_RECOVERY_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{STATE_RECOVERY_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"Row accounting                  : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Unique athlete-date states      : "
    f"{UNIQUE_STATE_PASS}"
)

print(
    f"Finite filter quantities        : "
    f"{FINITE_FILTER_PASS}"
)

print(
    f"Positive variances              : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Kalman gains in [0,1]           : "
    f"{GAIN_BOUNDS_PASS}"
)

print(
    f"First-state priors correct      : "
    f"{FIRST_STATE_PRIOR_PASS}"
)

print(
    f"One terminal state / athlete    : "
    f"{TERMINAL_ACCOUNTING_PASS}"
)

print(
    f"Terminal dates correct          : "
    f"{TERMINAL_DATE_PASS}"
)

print(
    f"History metadata valid          : "
    f"{HISTORY_METADATA_PASS}"
)

print(
    f"No future TRAIN data            : "
    f"{NO_FUTURE_DATA_PASS}"
)


print()
print("=" * 160)

if CELL_47_PASS:

    print(
        "CELL 47 STATUS: HB_2025 HISTORICAL V5 STATE RECOVERY PASSED"
    )

else:

    print(
        "CELL 47 STATUS: HB_2025 V5 STATE RECOVERY REVIEW REQUIRED"
    )

print("=" * 160)



HB_2025 — HISTORICAL V4 REFIT PREPARATION

TEMPORAL SPLIT
----------------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN       : <= 2024-12-31
VALIDATION  : 2025-01-01 -> 2025-04-30
TEST        : 2025-05-01 -> 2025-08-30

POPULATION
----------------------------------------------------------------------------------------------------------------------------------------------------------------
TRAIN rows / athletes          : 150,052 / 11,207
VALIDATION established         : 4,226 / 2,258
TEST established               : 11,342 / 4,076

TRAIN-ONLY SPLINE
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Basis shape                    : (150052, 7)
Internal knots                : [18.023641827  20.1126648733 22.1496676865 25.2708816745]
Column means                  : [0.066615

,split,n,median_gap_days,mean_gap_days,min_gap_days,max_gap_days
0,VALIDATION,4226,105.0,160.318504,1,3427
1,TEST,11342,27.0,113.741844,1,4373



AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T134225_178790Z
Immutable preparation directory: /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/preparation/20260922T134225_178790Z
Audit manifest                 : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/preparation/20260922T134225_178790Z/preparation_audit.json
Audit SHA256                   : 333fb36218e06ffb59d6e69058216fbca240391fe9e22e7b25cb220477283e5f

QA
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Temporal split                 : True
TRAIN-only athlete index    

Output()

Finished [100%]: Average Loss = 4.0993e+05


...


FIT STATISTICS
----------------------------------------------------------------------------------------------------------------------------------------------------------------
NLL                             : 371,505.766490
AIC                             : 743,023.532979
BIC                             : 743,083.010594
Successful starts               : 10/10
Numba used                      : True

AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T140556_178919Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v5_ou_refit/20260922T140556_178919Z
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical

,fold,n_states,athletes,mean_standardized_innovation,sd_standardized_innovation,median_standardized_innovation,pct_abs_z_gt_2,pct_abs_z_gt_3,mean_kalman_gain,mean_filtered_variance
0,HB_2025,149184,11207,-0.03295,0.99946,-0.000318,4.319498,0.776223,0.271516,1.661822



TERMINAL STATE HISTORY SUMMARY
----------------------------------------------------------------------------------------------------------------------------------------------------------------


,athletes,median_state_dates,median_history_span_days,max_history_span_days,median_terminal_state_sigma
0,11207,5.0,754.0,7028,1.338583



AUDIT / VERSIONING
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T140602_131082Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v5_state_recovery/20260922T140602_131082Z
Terminal-state artifact         : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v5_state_recovery/20260922T140602_131082Z/v5_train_terminal_states.parquet
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v5_state_recovery/20260922T140602_131082Z/v5_state_recovery_audit.json
Audit SHA256                    : c9d18a3c69fd40d1a51f58c5cc309

## Stage 16 — HB_2025 replay, calibration and evidence freeze

Replay rolling forecasts, calibrate V6 on historical VALIDATION, compare V5/V6/V6.1 on untouched TEST, and freeze evidence.

**Legacy source cells represented:** 38, 39, 40, 41  
**Execution rule:** run only after the preceding stage has completed successfully.


In [17]:
# ==================================================================================================
# REFACTORED STAGE 16 — HB_2025 replay, calibration and evidence freeze
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 38
# Original source SHA256: 77bc6f71cc4d2596920884d16e48c78d2786592dc133553df3400239d47e8566
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 48 — HB_2025 ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY
# ==================================================================================================
#
# Purpose
# -------
# Generate leakage-safe rolling-origin historical forecasts for HB_2025.
#
# Sequence:
#
#   TRAIN terminal state
#       ↓
#   predict VALIDATION athlete-date
#       ↓
#   observe that date
#       ↓
#   update OU state once
#       ↓
#   continue chronologically through VALIDATION and TEST
#
# Same-day performances share exactly the same pre-date state.
#
# NO V6 fitting occurs here.
# NO TEST information is used before its chronological observation date.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_47_PASS",

    "HB2025_VALIDATION",
    "HB2025_TEST",

    "HB2025_V4_POSTERIOR_MEANS",
    "HB2025_V5_TERMINAL_STATES",

    "HB2025_V5_TAU_DAYS",
    "HB2025_V5_STATE_VARIANCE",
    "HB2025_V5_R_GROUP",

    "HB2025_V5_PARAMETER_PATH",
    "HB2025_V5_STATE_RECOVERY_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing prerequisites:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_47_PASS:
    raise RuntimeError(
        "Cell 47 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_48_HB2025_ROLLING_V4_V5"

STAGE_SCHEMA_VERSION = "1.0.0"


GROUP_LABELS = [
    "M_700",
    "M_800",
    "W_500",
    "W_600",
]


SPLINE_COLUMNS = [
    f"spline_{i}"
    for i in range(7)
]


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2025_DIR = (
    Path(TRAINING_RUN_DIR)
    / "historical_backtest"
    / FOLD_NAME
)

ROLLING_STAGE_ROOT = (
    HB2025_DIR
    / "rolling_v4_v5"
)

ROLLING_ATTEMPT_DIR = (
    ROLLING_STAGE_ROOT
    / ATTEMPT_ID
)

ROLLING_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 5. FREEZE HISTORICAL V4 PARAMETERS
# --------------------------------------------------------------------------------------------------

alpha_hat = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "alpha_sex"
    ],
    dtype=float,
)

beta_hat = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "beta_spline"
    ],
    dtype=float,
)

youth_hat = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "youth_implement_effect"
    ],
    dtype=float,
)

athlete_intercept_hat = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "athlete_intercept"
    ],
    dtype=float,
)

athlete_slope_hat = np.asarray(
    HB2025_V4_POSTERIOR_MEANS[
        "athlete_slope"
    ],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 6. FREEZE V5 PARAMETERS
# --------------------------------------------------------------------------------------------------

TAU = float(
    HB2025_V5_TAU_DAYS
)

VZ = float(
    HB2025_V5_STATE_VARIANCE
)

R_GROUP = {
    group:
        float(
            HB2025_V5_R_GROUP[
                group
            ]
        )
    for group in GROUP_LABELS
}


R_VECTOR = np.array(
    [
        R_GROUP[
            group
        ]
        for group in GROUP_LABELS
    ],
    dtype=float,
)


# --------------------------------------------------------------------------------------------------
# 7. PREPARE VALIDATION + TEST
# --------------------------------------------------------------------------------------------------

VALIDATION = (
    HB2025_VALIDATION.copy()
    .reset_index(
        drop=True
    )
)

TEST = (
    HB2025_TEST.copy()
    .reset_index(
        drop=True
    )
)


VALIDATION[
    "evaluation_split"
] = "VALIDATION"

TEST[
    "evaluation_split"
] = "TEST"


VALIDATION[
    "split_row_id"
] = np.arange(
    len(VALIDATION)
)

TEST[
    "split_row_id"
] = np.arange(
    len(TEST)
)


PRED = pd.concat(
    [
        VALIDATION,
        TEST,
    ],
    ignore_index=True,
)


PRED[
    "result_date_dt"
] = pd.to_datetime(
    PRED[
        "result_date_dt"
    ]
)


PRED[
    "_forecast_row_id"
] = np.arange(
    len(PRED),
    dtype=int,
)


# --------------------------------------------------------------------------------------------------
# 8. REQUIRED COLUMN QA
# --------------------------------------------------------------------------------------------------

required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",

    "sex_idx",
    "group_idx",
    "athlete_idx",

    "is_youth_implement",
    "age_scaled",

    "model_group",

    *SPLINE_COLUMNS,
]


missing_columns = [
    column
    for column in required_columns
    if column not in PRED.columns
]


if missing_columns:
    raise RuntimeError(
        "Prediction data missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. RECONSTRUCT HISTORICAL V4 STRUCTURAL PREDICTIONS
# --------------------------------------------------------------------------------------------------

sex_idx = (
    PRED[
        "sex_idx"
    ]
    .to_numpy(
        dtype=int
    )
)

athlete_idx = (
    PRED[
        "athlete_idx"
    ]
    .to_numpy(
        dtype=int
    )
)

youth = (
    PRED[
        "is_youth_implement"
    ]
    .to_numpy(
        dtype=float
    )
)

age_scaled = (
    PRED[
        "age_scaled"
    ]
    .to_numpy(
        dtype=float
    )
)

B = (
    PRED[
        SPLINE_COLUMNS
    ]
    .to_numpy(
        dtype=float
    )
)


PRED[
    "v4_population_component"
] = (
    alpha_hat[
        sex_idx
    ]

    +

    np.sum(
        B
        *
        beta_hat[
            sex_idx
        ],
        axis=1,
    )

    +

    youth_hat[
        sex_idx
    ]
    *
    youth
)


PRED[
    "v4_athlete_intercept_component"
] = (
    athlete_intercept_hat[
        athlete_idx
    ]
)


PRED[
    "v4_athlete_slope_component"
] = (
    athlete_slope_hat[
        athlete_idx
    ]
    *
    age_scaled
)


PRED[
    "v4_structural_mean"
] = (
    PRED[
        "v4_population_component"
    ]

    +

    PRED[
        "v4_athlete_intercept_component"
    ]

    +

    PRED[
        "v4_athlete_slope_component"
    ]
)


PRED[
    "v4_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v4_structural_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. TRAIN TERMINAL STATE LOOKUP
# --------------------------------------------------------------------------------------------------

TERMINAL = (
    HB2025_V5_TERMINAL_STATES.copy()
)


TERMINAL[
    "terminal_state_date"
] = pd.to_datetime(
    TERMINAL[
        "terminal_state_date"
    ]
)


terminal_lookup = (
    TERMINAL.set_index(
        "aaAthleteId"
    )
)


prediction_athletes = set(
    PRED[
        "aaAthleteId"
    ]
    .unique()
)


terminal_athletes = set(
    TERMINAL[
        "aaAthleteId"
    ]
    .unique()
)


MISSING_TERMINAL_ATHLETES = (
    prediction_athletes
    -
    terminal_athletes
)


if MISSING_TERMINAL_ATHLETES:

    raise RuntimeError(
        "Established prediction athletes missing TRAIN terminal states:\n"
        +
        "\n".join(
            str(x)
            for x in sorted(
                MISSING_TERMINAL_ATHLETES
            )[:20]
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. ALLOCATE OUTPUT ARRAYS
# --------------------------------------------------------------------------------------------------

N = len(PRED)


state_prior_mean = np.full(
    N,
    np.nan,
)


state_prior_variance = np.full(
    N,
    np.nan,
)


state_gap_dynamic = np.full(
    N,
    np.nan,
)


state_phi = np.full(
    N,
    np.nan,
)


v5_prediction_mean = np.full(
    N,
    np.nan,
)


date_state_rows = []


# --------------------------------------------------------------------------------------------------
# 12. ROLLING-ORIGIN FORECAST / UPDATE
# --------------------------------------------------------------------------------------------------

for athlete_id, athlete_rows in PRED.groupby(
    "aaAthleteId",
    sort=False,
):

    terminal_row = (
        terminal_lookup.loc[
            athlete_id
        ]
    )


    current_mean = float(
        terminal_row[
            "terminal_state_mean"
        ]
    )


    current_variance = float(
        terminal_row[
            "terminal_state_variance"
        ]
    )


    current_date = pd.Timestamp(
        terminal_row[
            "terminal_state_date"
        ]
    )


    athlete_rows = (
        athlete_rows.sort_values(
            [
                "result_date_dt",
                "_forecast_row_id",
            ]
        )
    )


    for forecast_date, date_rows in athlete_rows.groupby(
        "result_date_dt",
        sort=True,
    ):

        forecast_date = pd.Timestamp(
            forecast_date
        )


        delta_days = int(
            (
                forecast_date
                -
                current_date
            ).days
        )


        if delta_days <= 0:

            raise RuntimeError(
                f"Non-positive state gap for athlete "
                f"{athlete_id}: "
                f"{current_date} -> {forecast_date}"
            )


        # ------------------------------------------------------------------------------------------
        # PRE-DATE OU PROPAGATION
        # ------------------------------------------------------------------------------------------

        phi = float(
            np.exp(
                -delta_days
                /
                TAU
            )
        )


        prior_mean = float(
            phi
            *
            current_mean
        )


        prior_variance = float(
            phi
            *
            phi
            *
            current_variance

            +

            VZ
            *
            (
                1.0
                -
                phi
                *
                phi
            )
        )


        row_ids = (
            date_rows[
                "_forecast_row_id"
            ]
            .to_numpy(
                dtype=int
            )
        )


        # ------------------------------------------------------------------------------------------
        # PREDICTION BEFORE OBSERVING THIS DATE
        # ------------------------------------------------------------------------------------------

        state_prior_mean[
            row_ids
        ] = prior_mean


        state_prior_variance[
            row_ids
        ] = prior_variance


        state_gap_dynamic[
            row_ids
        ] = delta_days


        state_phi[
            row_ids
        ] = phi


        v5_prediction_mean[
            row_ids
        ] = (
            PRED.loc[
                row_ids,
                "v4_structural_mean",
            ]
            .to_numpy(
                dtype=float
            )

            +

            prior_mean
        )


        # ------------------------------------------------------------------------------------------
        # AFTER PREDICTION: OBSERVE ALL SAME-DAY RESULTS
        # ------------------------------------------------------------------------------------------

        residuals = (
            PRED.loc[
                row_ids,
                "v4_error",
            ]
            .to_numpy(
                dtype=float
            )
        )


        state_observation = float(
            np.mean(
                residuals
            )
        )


        n_date = int(
            len(
                row_ids
            )
        )


        group_counts = np.array(
            [
                int(
                    (
                        PRED.loc[
                            row_ids,
                            "model_group",
                        ]
                        ==
                        group
                    )
                    .sum()
                )

                for group in GROUP_LABELS
            ],
            dtype=float,
        )


        R_d = float(
            np.dot(
                group_counts,
                R_VECTOR,
            )

            /

            (
                n_date
                **
                2
            )
        )


        innovation = float(
            state_observation
            -
            prior_mean
        )


        innovation_variance = float(
            prior_variance
            +
            R_d
        )


        K = float(
            prior_variance
            /
            innovation_variance
        )


        posterior_mean = float(
            prior_mean
            +
            K
            *
            innovation
        )


        posterior_variance = float(
            (
                1.0
                -
                K
            )
            *
            prior_variance
        )


        posterior_variance = max(
            posterior_variance,
            1e-12,
        )


        date_state_rows.append(
            {
                "aaAthleteId":
                    athlete_id,

                "result_date_dt":
                    forecast_date,

                "evaluation_split":
                    date_rows[
                        "evaluation_split"
                    ]
                    .iloc[0],

                "state_gap_days":
                    delta_days,

                "state_phi":
                    phi,

                "state_prior_mean":
                    prior_mean,

                "state_prior_variance":
                    prior_variance,

                "state_observation":
                    state_observation,

                "n_performances":
                    n_date,

                "measurement_variance":
                    R_d,

                "innovation":
                    innovation,

                "innovation_variance":
                    innovation_variance,

                "standardized_innovation":
                    (
                        innovation
                        /
                        np.sqrt(
                            innovation_variance
                        )
                    ),

                "kalman_gain":
                    K,

                "state_filtered_mean":
                    posterior_mean,

                "state_filtered_variance":
                    posterior_variance,

                **{
                    f"n_{group}":
                        int(
                            group_counts[i]
                        )

                    for i, group
                    in enumerate(
                        GROUP_LABELS
                    )
                },
            }
        )


        # State is available only to FUTURE dates.
        current_mean = posterior_mean
        current_variance = posterior_variance
        current_date = forecast_date


# --------------------------------------------------------------------------------------------------
# 13. ATTACH ROLLING OUTPUT
# --------------------------------------------------------------------------------------------------

PRED[
    "state_prior_mean"
] = state_prior_mean


PRED[
    "state_prior_variance"
] = state_prior_variance


PRED[
    "state_prior_sigma"
] = np.sqrt(
    state_prior_variance
)


PRED[
    "rolling_state_gap_days"
] = state_gap_dynamic


PRED[
    "state_phi"
] = state_phi


PRED[
    "v5_prediction_mean"
] = v5_prediction_mean


PRED[
    "v5_error"
] = (
    PRED[
        "mark_numeric_m"
    ]

    -

    PRED[
        "v5_prediction_mean"
    ]
)


DATE_STATES = pd.DataFrame(
    date_state_rows
)


# --------------------------------------------------------------------------------------------------
# 14. CHRONOLOGY RECONCILIATION
# --------------------------------------------------------------------------------------------------

if "state_gap_days" not in PRED.columns:

    raise RuntimeError(
        "Prepared HB_2025 fold lacks reference state_gap_days."
    )


GAP_RECONCILIATION_MAX_DIFF = float(
    np.max(
        np.abs(
            PRED[
                "rolling_state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )

            -

            PRED[
                "state_gap_days"
            ]
            .to_numpy(
                dtype=float
            )
        )
    )
)


GAP_RECONCILIATION_PASS = bool(
    GAP_RECONCILIATION_MAX_DIFF
    <
    1e-12
)


# --------------------------------------------------------------------------------------------------
# 15. SPLIT BACK INTO VALIDATION / TEST
# --------------------------------------------------------------------------------------------------

ROLLING_VALIDATION = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "VALIDATION"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


ROLLING_TEST = (
    PRED.loc[
        PRED[
            "evaluation_split"
        ]
        .eq(
            "TEST"
        )
    ]
    .sort_values(
        "split_row_id"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 16. METRIC HELPER
# --------------------------------------------------------------------------------------------------

def point_metrics(
    df,
):

    rows = []


    for model in [
        "V4",
        "V5",
    ]:

        error = (
            df[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        rows.append(
            {
                "model":
                    model,

                "n":
                    int(
                        len(
                            df
                        )
                    ),

                "athletes":
                    int(
                        df[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error
                                **
                                2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# --------------------------------------------------------------------------------------------------
# 17. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

VALIDATION_METRICS = point_metrics(
    ROLLING_VALIDATION
)


VALIDATION_METRICS[
    "split"
] = "VALIDATION"


TEST_METRICS = point_metrics(
    ROLLING_TEST
)


TEST_METRICS[
    "split"
] = "TEST"


POINT_METRICS = pd.concat(
    [
        VALIDATION_METRICS,
        TEST_METRICS,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 18. OVERALL V4 -> V5 GAINS
# --------------------------------------------------------------------------------------------------

gain_rows = []


for split, g in PRED.groupby(
    "evaluation_split",
    sort=False,
):

    e4 = (
        g[
            "v4_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    e5 = (
        g[
            "v5_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    v4_mae = float(
        np.mean(
            np.abs(
                e4
            )
        )
    )


    v5_mae = float(
        np.mean(
            np.abs(
                e5
            )
        )
    )


    v4_rmse = float(
        np.sqrt(
            np.mean(
                e4
                **
                2
            )
        )
    )


    v5_rmse = float(
        np.sqrt(
            np.mean(
                e5
                **
                2
            )
        )
    )


    gain_rows.append(
        {
            "split":
                split,

            "n":
                int(
                    len(
                        g
                    )
                ),

            "athletes":
                int(
                    g[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V4_MAE":
                v4_mae,

            "V5_MAE":
                v5_mae,

            "V4_to_V5_MAE_gain_m":
                (
                    v4_mae
                    -
                    v5_mae
                ),

            "V4_RMSE":
                v4_rmse,

            "V5_RMSE":
                v5_rmse,

            "V4_to_V5_RMSE_gain_m":
                (
                    v4_rmse
                    -
                    v5_rmse
                ),

            "V5_Bias":
                float(
                    np.mean(
                        e5
                    )
                ),
        }
    )


V5_GAIN_SUMMARY = pd.DataFrame(
    gain_rows
)


# --------------------------------------------------------------------------------------------------
# 19. STATE-GAP METRICS
# --------------------------------------------------------------------------------------------------

GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


PRED[
    "state_gap_band"
] = pd.cut(
    PRED[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


gap_rows = []


for (
    split,
    gap_band
), g in PRED.groupby(
    [
        "evaluation_split",
        "state_gap_band",
    ],
    observed=True,
):

    for model in [
        "V4",
        "V5",
    ]:

        error = (
            g[
                f"{model.lower()}_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        gap_rows.append(
            {
                "split":
                    split,

                "state_gap_band":
                    str(
                        gap_band
                    ),

                "model":
                    model,

                "n":
                    int(
                        len(
                            g
                        )
                    ),

                "athletes":
                    int(
                        g[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "mean_gap_days":
                    float(
                        g[
                            "rolling_state_gap_days"
                        ]
                        .mean()
                    ),

                "MAE":
                    float(
                        np.mean(
                            np.abs(
                                error
                            )
                        )
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                error
                                **
                                2
                            )
                        )
                    ),

                "Bias":
                    float(
                        np.mean(
                            error
                        )
                    ),
            }
        )


GAP_METRICS = pd.DataFrame(
    gap_rows
)


# --------------------------------------------------------------------------------------------------
# 20. QA
# --------------------------------------------------------------------------------------------------

FINITE_PREDICTION_PASS = bool(
    np.isfinite(
        PRED[
            [
                "v4_structural_mean",
                "v5_prediction_mean",
                "v4_error",
                "v5_error",
                "state_prior_mean",
                "state_prior_variance",
                "rolling_state_gap_days",
                "state_phi",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        PRED[
            "state_prior_variance"
        ]
        >
        0
    )
    .all()

    and

    (
        DATE_STATES[
            "state_filtered_variance"
        ]
        >
        0
    )
    .all()
)


NO_SAME_DAY_LEAKAGE_PASS = bool(
    DATE_STATES.duplicated(
        subset=[
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .sum()
    ==
    0
)


ROW_ACCOUNTING_PASS = bool(
    len(
        ROLLING_VALIDATION
    )
    ==
    len(
        HB2025_VALIDATION
    )

    and

    len(
        ROLLING_TEST
    )
    ==
    len(
        HB2025_TEST
    )
)


SPLIT_CHRONOLOGY_PASS = bool(
    ROLLING_VALIDATION[
        "result_date_dt"
    ]
    .max()

    <

    ROLLING_TEST[
        "result_date_dt"
    ]
    .min()
)


CELL_48_PASS = bool(
    FINITE_PREDICTION_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    NO_SAME_DAY_LEAKAGE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    SPLIT_CHRONOLOGY_PASS
    and
    GAP_RECONCILIATION_PASS
)


# --------------------------------------------------------------------------------------------------
# 21. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

VALIDATION_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "validation_rolling_predictions.parquet"
)


TEST_PREDICTIONS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "test_rolling_predictions.parquet"
)


DATE_STATE_UPDATES_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "rolling_date_state_updates.parquet"
)


POINT_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "point_metrics.parquet"
)


GAP_METRICS_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "gap_metrics.parquet"
)


GAIN_SUMMARY_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "v4_to_v5_gain_summary.parquet"
)


ROLLING_AUDIT_PATH = (
    ROLLING_ATTEMPT_DIR
    /
    "rolling_forecast_audit.json"
)


ROLLING_VALIDATION.to_parquet(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)


ROLLING_TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)


DATE_STATES.to_parquet(
    DATE_STATE_UPDATES_PATH,
    index=False,
)


POINT_METRICS.to_parquet(
    POINT_METRICS_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


V5_GAIN_SUMMARY.to_parquet(
    GAIN_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 22. AUDIT PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v5_parameters":
        Path(
            HB2025_V5_PARAMETER_PATH
        ),

    "v5_state_recovery_audit":
        Path(
            HB2025_V5_STATE_RECOVERY_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else None
            ),
    }


output_paths = {
    "validation_predictions":
        VALIDATION_PREDICTIONS_PATH,

    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "date_state_updates":
        DATE_STATE_UPDATES_PATH,

    "point_metrics":
        POINT_METRICS_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "v5_gain_summary":
        GAIN_SUMMARY_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


ROLLING_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "ROLLING_V4_V5_FORECAST",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_48_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Leakage-safe HB_2025 rolling V4/V5 forecast replication."
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "temporal_windows": {
        "validation":
            "2025-01-01 through 2025-04-30",

        "test":
            "2025-05-01 through 2025-08-30",
    },

    "state_update_rule":
        (
            "Predict all performances on an athlete-date using the pre-date state; "
            "then update once using the mean same-day V4 residual."
        ),

    "v5_parameters": {
        "tau_days":
            TAU,

        "state_variance":
            VZ,

        "measurement_variance_by_group":
            R_GROUP,
    },

    "population": {
        "validation_rows":
            int(
                len(
                    ROLLING_VALIDATION
                )
            ),

        "validation_athletes":
            int(
                ROLLING_VALIDATION[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "test_rows":
            int(
                len(
                    ROLLING_TEST
                )
            ),

        "test_athletes":
            int(
                ROLLING_TEST[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "rolling_athlete_dates":
            int(
                len(
                    DATE_STATES
                )
            ),
    },

    "qa": {
        "finite_predictions":
            FINITE_PREDICTION_PASS,

        "positive_variances":
            POSITIVE_VARIANCE_PASS,

        "no_same_day_leakage":
            NO_SAME_DAY_LEAKAGE_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "split_chronology":
            SPLIT_CHRONOLOGY_PASS,

        "gap_reconciliation":
            GAP_RECONCILIATION_PASS,

        "gap_reconciliation_max_diff":
            GAP_RECONCILIATION_MAX_DIFF,

        "cell_pass":
            CELL_48_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    ROLLING_AUDIT,
    ROLLING_AUDIT_PATH,
)


ROLLING_AUDIT_SHA256 = sha256_file(
    ROLLING_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 23. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_ROLLING_POINTER_PATH = (
    ROLLING_STAGE_ROOT
    /
    "LATEST_ROLLING_V4_V5.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "ROLLING_V4_V5_FORECAST",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_48_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                ROLLING_ATTEMPT_DIR
            ),

        "validation_predictions":
            str(
                VALIDATION_PREDICTIONS_PATH
            ),

        "test_predictions":
            str(
                TEST_PREDICTIONS_PATH
            ),

        "audit_manifest":
            str(
                ROLLING_AUDIT_PATH
            ),

        "audit_sha256":
            ROLLING_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_ROLLING_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. MAIN TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "rolling_v4_v5"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_48_PASS
            else
            "FAILED"
        ),

    "latest_pointer":
        str(
            LATEST_ROLLING_POINTER_PATH
        ),

    "attempt_directory":
        str(
            ROLLING_ATTEMPT_DIR
        ),

    "audit_manifest":
        str(
            ROLLING_AUDIT_PATH
        ),

    "audit_sha256":
        ROLLING_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_ROLLING_V4_V5_COMPLETE"
    if CELL_48_PASS
    else
    "HB_2025_ROLLING_V4_V5_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 25. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_ROLLING_VALIDATION = (
    ROLLING_VALIDATION
)


HB2025_ROLLING_TEST = (
    ROLLING_TEST
)


HB2025_ROLLING_DATE_STATES = (
    DATE_STATES
)


HB2025_V4_V5_POINT_METRICS = (
    POINT_METRICS
)


HB2025_V4_V5_GAP_METRICS = (
    GAP_METRICS
)


HB2025_V4_TO_V5_GAIN_SUMMARY = (
    V5_GAIN_SUMMARY
)


HB2025_ROLLING_ATTEMPT_DIR = (
    ROLLING_ATTEMPT_DIR
)


NOTEBOOK_2_CELL_48_PASS = (
    CELL_48_PASS
)


# --------------------------------------------------------------------------------------------------
# 26. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2025 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY")
print("=" * 165)


print()
print("OVERALL POINT PERFORMANCE")
print("-" * 165)

display(
    V5_GAIN_SUMMARY
)


print()
print("POINT METRICS")
print("-" * 165)

display(
    POINT_METRICS
)


print()
print("BY STATE GAP")
print("-" * 165)

display(
    GAP_METRICS
)


print()
print("ROLLING STATE UPDATE SUMMARY")
print("-" * 165)

display(
    pd.DataFrame(
        [
            {
                "athlete_dates":
                    int(
                        len(
                            DATE_STATES
                        )
                    ),

                "athletes":
                    int(
                        DATE_STATES[
                            "aaAthleteId"
                        ]
                        .nunique()
                    ),

                "mean_standardized_innovation":
                    float(
                        DATE_STATES[
                            "standardized_innovation"
                        ]
                        .mean()
                    ),

                "sd_standardized_innovation":
                    float(
                        DATE_STATES[
                            "standardized_innovation"
                        ]
                        .std(
                            ddof=1
                        )
                    ),

                "mean_kalman_gain":
                    float(
                        DATE_STATES[
                            "kalman_gain"
                        ]
                        .mean()
                    ),
            }
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{ROLLING_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{ROLLING_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{ROLLING_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 165)

print(
    f"Finite predictions             : "
    f"{FINITE_PREDICTION_PASS}"
)

print(
    f"Positive variances             : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"No same-day leakage            : "
    f"{NO_SAME_DAY_LEAKAGE_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Split chronology               : "
    f"{SPLIT_CHRONOLOGY_PASS}"
)

print(
    f"Gap reconciliation             : "
    f"{GAP_RECONCILIATION_PASS}"
)

print(
    f"Maximum gap discrepancy        : "
    f"{GAP_RECONCILIATION_MAX_DIFF:.12e}"
)


print()
print("=" * 165)

if CELL_48_PASS:

    print(
        "CELL 48 STATUS: HB_2025 ROLLING V4/V5 FORECAST PASSED"
    )

else:

    print(
        "CELL 48 STATUS: HB_2025 ROLLING V4/V5 FORECAST REVIEW REQUIRED"
    )

print("=" * 165)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 39
# Original source SHA256: 2b2e88035e463fd053233ec2394cfdc1c0567aad923f0c270b88a53b5c4f0788
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 49 — HB_2025 HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY
# ==================================================================================================
#
# Frozen before this cell:
#
#   V4 fitted <= 2024-12-31
#   V5 fitted <= 2024-12-31
#
# Calibration:
#
#   2025-01-01 -> 2025-04-30 VALIDATION only
#
# TEST:
#
#   NOT USED
#
# Model-selection logic is intentionally identical to HB_2023 / HB_2024.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import norm


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_48_PASS",

    "HB2025_ROLLING_VALIDATION",
    "HB2025_ROLLING_ATTEMPT_DIR",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_48_PASS:

    raise RuntimeError(
        "Cell 48 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_49_HB2025_V6_MEAN_CALIBRATION"

STAGE_SCHEMA_VERSION = "1.0.0"


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
]


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def gaussian_fit_statistics(
    residual,
    n_mean_parameters,
):

    residual = np.asarray(
        residual,
        dtype=float,
    )

    n = len(
        residual
    )

    rss = float(
        np.sum(
            residual ** 2
        )
    )

    sigma2 = max(
        rss / n,
        1e-12,
    )

    nll = float(
        0.5
        *
        n
        *
        (
            np.log(
                2.0
                *
                np.pi
                *
                sigma2
            )
            +
            1.0
        )
    )

    # Mean parameters + residual sigma
    k = int(
        n_mean_parameters
        +
        1
    )

    aic = float(
        2.0 * k
        +
        2.0 * nll
    )

    if n <= k + 1:

        aicc = np.inf

    else:

        aicc = float(
            aic
            +
            (
                2.0
                *
                k
                *
                (
                    k + 1
                )
            )
            /
            (
                n
                -
                k
                -
                1
            )
        )

    bic = float(
        np.log(n)
        *
        k
        +
        2.0
        *
        nll
    )

    return {
        "n":
            int(n),

        "rss":
            rss,

        "sigma":
            float(
                np.sqrt(
                    sigma2
                )
            ),

        "nll":
            nll,

        "k":
            k,

        "AIC":
            aic,

        "AICc":
            aicc,

        "BIC":
            bic,
    }


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2025_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)

V6_STAGE_ROOT = (
    HB2025_DIR
    /
    "v6_mean_calibration"
)

V6_ATTEMPT_DIR = (
    V6_STAGE_ROOT
    /
    ATTEMPT_ID
)

V6_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. VALIDATION ONLY
# --------------------------------------------------------------------------------------------------

VAL = (
    HB2025_ROLLING_VALIDATION.copy()
)

VAL[
    "sex_group"
] = (
    VAL[
        "model_group"
    ]
    .astype(str)
    .str[0]
)

TEST_TOUCHED = False


# --------------------------------------------------------------------------------------------------
# 6. FIRST VALIDATION DATE PER ATHLETE × MODEL GROUP
# --------------------------------------------------------------------------------------------------

first_date = (
    VAL.groupby(
        [
            "aaAthleteId",
            "model_group",
        ]
    )[
        "result_date_dt"
    ]
    .transform(
        "min"
    )
)


CAL_ROWS = (
    VAL.loc[
        VAL[
            "result_date_dt"
        ].eq(
            first_date
        )
    ]
    .copy()
)


# Average multiple performances on the same first athlete/group/date.
CAL = (
    CAL_ROWS.groupby(
        [
            "aaAthleteId",
            "model_group",
            "sex_group",
            "result_date_dt",
        ],
        as_index=False,
    )
    .agg(
        mark_numeric_m=(
            "mark_numeric_m",
            "mean",
        ),

        v5_prediction_mean=(
            "v5_prediction_mean",
            "mean",
        ),

        state_prior_mean=(
            "state_prior_mean",
            "mean",
        ),

        rolling_state_gap_days=(
            "rolling_state_gap_days",
            "first",
        ),

        v4_population_component=(
            "v4_population_component",
            "mean",
        ),

        v4_athlete_intercept_component=(
            "v4_athlete_intercept_component",
            "mean",
        ),

        v4_athlete_slope_component=(
            "v4_athlete_slope_component",
            "mean",
        ),

        n_same_day_rows=(
            "mark_numeric_m",
            "size",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 7. CALIBRATION DOMAIN
# --------------------------------------------------------------------------------------------------

CAL = (
    CAL.loc[
        CAL[
            "rolling_state_gap_days"
        ].between(
            1,
            365,
            inclusive="both",
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


CAL[
    "gap_band"
] = pd.cut(
    CAL[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 8. STRUCTURAL TARGET
# --------------------------------------------------------------------------------------------------

CAL[
    "base_without_athlete_structure"
] = (
    CAL[
        "v5_prediction_mean"
    ]
    -
    CAL[
        "v4_athlete_intercept_component"
    ]
    -
    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "structural_target"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "base_without_athlete_structure"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. CENTER BY SEX × GAP BAND
# --------------------------------------------------------------------------------------------------

CENTER_GROUP = [
    "sex_group",
    "gap_band",
]


for column in [
    "structural_target",
    "v4_athlete_intercept_component",
    "v4_athlete_slope_component",
]:

    group_mean = (
        CAL.groupby(
            CENTER_GROUP,
            observed=True,
        )[
            column
        ]
        .transform(
            "mean"
        )
    )

    CAL[
        f"{column}_centered"
    ] = (
        CAL[
            column
        ]
        -
        group_mean
    )


# --------------------------------------------------------------------------------------------------
# 10. EMPIRICAL HC3 PERSISTENCE DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

Y = (
    CAL[
        "structural_target_centered"
    ]
    .to_numpy(
        dtype=float
    )
)

I = (
    CAL[
        "v4_athlete_intercept_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)

S = (
    CAL[
        "v4_athlete_slope_component_centered"
    ]
    .to_numpy(
        dtype=float
    )
)


X = np.column_stack(
    [
        np.ones(
            len(CAL)
        ),
        I,
        S,
    ]
)


beta_ols = np.linalg.lstsq(
    X,
    Y,
    rcond=None,
)[0]


ols_residual = (
    Y
    -
    X @ beta_ols
)


XtX_inv = np.linalg.inv(
    X.T @ X
)


hat_diag = np.sum(
    (
        X @ XtX_inv
    )
    *
    X,
    axis=1,
)


hc3_scale = (
    ols_residual
    /
    np.maximum(
        1.0
        -
        hat_diag,
        1e-8,
    )
) ** 2


HC3_COV = (
    XtX_inv
    @
    (
        X.T
        @
        (
            X
            *
            hc3_scale[
                :,
                None
            ]
        )
    )
    @
    XtX_inv
)


HC3_SE = np.sqrt(
    np.diag(
        HC3_COV
    )
)


HC3_Z = (
    beta_ols
    /
    HC3_SE
)


HC3_P = (
    2.0
    *
    norm.sf(
        np.abs(
            HC3_Z
        )
    )
)


HC3_SUMMARY = pd.DataFrame(
    {
        "term": [
            "intercept",
            "rho_intercept_empirical",
            "rho_slope_empirical",
        ],

        "estimate":
            beta_ols,

        "HC3_SE":
            HC3_SE,

        "z":
            HC3_Z,

        "p_value":
            HC3_P,
    }
)


# --------------------------------------------------------------------------------------------------
# 11. PERSISTENCE FUNCTIONS
# --------------------------------------------------------------------------------------------------

h = (
    CAL[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


def persistence_values(
    model_name,
    params,
    horizon,
):

    horizon = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        c_I, c_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M1_CONST_EXP":

        c_I, tau_S = params

        rho_I = np.full_like(
            horizon,
            c_I,
            dtype=float,
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    elif model_name == "M2_EXP_CONST":

        tau_I, c_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.full_like(
            horizon,
            c_S,
            dtype=float,
        )


    elif model_name == "M3_EXP_EXP":

        tau_I, tau_S = params

        rho_I = np.exp(
            -
            horizon
            /
            tau_I
        )

        rho_S = np.exp(
            -
            horizon
            /
            tau_S
        )


    else:

        raise ValueError(
            model_name
        )


    return rho_I, rho_S


def persistence_sse(
    model_name,
    params,
):

    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )

    prediction = (
        rho_I * I
        +
        rho_S * S
    )

    residual = (
        Y
        -
        prediction
    )

    return float(
        np.sum(
            residual ** 2
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. SAME FOUR CANDIDATE MODELS AS PREVIOUS FOLDS
# --------------------------------------------------------------------------------------------------

PERSISTENCE_CONFIG = {

    "M0_CONST_CONST": {
        "bounds": [
            (0.0, 1.5),
            (0.0, 1.5),
        ],

        "starts": [
            (0.50, 0.50),
            (0.75, 0.75),
            (1.00, 1.00),
            (1.25, 1.25),
            (0.75, 1.00),
            (1.00, 0.75),
        ],

        "parameter_names": [
            "c_intercept",
            "c_slope",
        ],
    },

    "M1_CONST_EXP": {
        "bounds": [
            (0.0, 1.5),
            (1.0, 5000.0),
        ],

        "starts": [
            (0.75, 60.0),
            (0.90, 120.0),
            (1.00, 365.0),
            (1.10, 730.0),
            (1.00, 1500.0),
        ],

        "parameter_names": [
            "c_intercept",
            "tau_slope_days",
        ],
    },

    "M2_EXP_CONST": {
        "bounds": [
            (1.0, 5000.0),
            (0.0, 1.5),
        ],

        "starts": [
            (60.0, 0.75),
            (120.0, 0.90),
            (365.0, 1.00),
            (730.0, 1.10),
            (1500.0, 1.00),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "c_slope",
        ],
    },

    "M3_EXP_EXP": {
        "bounds": [
            (1.0, 5000.0),
            (1.0, 5000.0),
        ],

        "starts": [
            (60.0, 60.0),
            (120.0, 120.0),
            (365.0, 365.0),
            (730.0, 365.0),
            (365.0, 730.0),
            (1500.0, 1500.0),
        ],

        "parameter_names": [
            "tau_intercept_days",
            "tau_slope_days",
        ],
    },
}


# --------------------------------------------------------------------------------------------------
# 13. FIT PERSISTENCE CANDIDATES
# --------------------------------------------------------------------------------------------------

candidate_rows = []

candidate_best = {}


for model_name, config in PERSISTENCE_CONFIG.items():

    results = []


    for x0 in config[
        "starts"
    ]:

        result = minimize(
            lambda p:
                persistence_sse(
                    model_name,
                    p,
                ),

            x0=np.asarray(
                x0,
                dtype=float,
            ),

            method="L-BFGS-B",

            bounds=config[
                "bounds"
            ],

            options={
                "maxiter":
                    3000,

                "ftol":
                    1e-12,

                "gtol":
                    1e-8,
            },
        )

        results.append(
            result
        )


    successful = [
        result
        for result in results
        if (
            result.success
            and
            np.isfinite(
                result.fun
            )
        )
    ]


    if not successful:

        raise RuntimeError(
            f"No successful optimization for {model_name}."
        )


    best = min(
        successful,
        key=lambda r:
            r.fun,
    )


    params = np.asarray(
        best.x,
        dtype=float,
    )


    rho_I, rho_S = persistence_values(
        model_name,
        params,
        h,
    )


    residual = (
        Y
        -
        (
            rho_I * I
            +
            rho_S * S
        )
    )


    stats = gaussian_fit_statistics(
        residual,
        n_mean_parameters=len(
            params
        ),
    )


    parameter_dict = {
        name:
            float(value)

        for name, value
        in zip(
            config[
                "parameter_names"
            ],
            params,
        )
    }


    candidate_rows.append(
        {
            "model":
                model_name,

            "successful_starts":
                int(
                    sum(
                        r.success
                        for r in results
                    )
                ),

            **parameter_dict,

            **stats,
        }
    )


    candidate_best[
        model_name
    ] = {
        "params":
            params,

        "parameter_dict":
            parameter_dict,
    }


PERSISTENCE_CANDIDATES = pd.DataFrame(
    candidate_rows
)


PERSISTENCE_CANDIDATES[
    "delta_AICc"
] = (
    PERSISTENCE_CANDIDATES[
        "AICc"
    ]
    -
    PERSISTENCE_CANDIDATES[
        "AICc"
    ].min()
)


PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_PERSISTENCE_MODEL = str(
    PERSISTENCE_CANDIDATES.loc[
        0,
        "model",
    ]
)


SELECTED_PERSISTENCE_PARAMS = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "params"
    ]
)


SELECTED_PERSISTENCE_PARAMETER_DICT = (
    candidate_best[
        SELECTED_PERSISTENCE_MODEL
    ][
        "parameter_dict"
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. APPLY SELECTED PERSISTENCE TO RAW CALIBRATION SAMPLE
# --------------------------------------------------------------------------------------------------

rho_I_raw, rho_S_raw = persistence_values(
    SELECTED_PERSISTENCE_MODEL,
    SELECTED_PERSISTENCE_PARAMS,
    CAL[
        "rolling_state_gap_days"
    ].to_numpy(
        dtype=float
    ),
)


CAL[
    "rho_intercept"
] = rho_I_raw


CAL[
    "rho_slope"
] = rho_S_raw


CAL[
    "v6_without_offset"
] = (
    CAL[
        "base_without_athlete_structure"
    ]

    +

    CAL[
        "rho_intercept"
    ]
    *
    CAL[
        "v4_athlete_intercept_component"
    ]

    +

    CAL[
        "rho_slope"
    ]
    *
    CAL[
        "v4_athlete_slope_component"
    ]
)


CAL[
    "offset_target"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v6_without_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. OFFSET MODEL COMPARISON
# --------------------------------------------------------------------------------------------------

offset_candidate_rows = []

offset_specs = {}


# NONE
residual = (
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    residual,
    n_mean_parameters=0,
)


offset_candidate_rows.append(
    {
        "offset_model":
            "NONE",

        **stats,
    }
)


offset_specs[
    "NONE"
] = {}


# COMMON
common_offset = float(
    CAL[
        "offset_target"
    ]
    .mean()
)


residual = (
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    common_offset
)


stats = gaussian_fit_statistics(
    residual,
    n_mean_parameters=1,
)


offset_candidate_rows.append(
    {
        "offset_model":
            "COMMON",

        **stats,
    }
)


offset_specs[
    "COMMON"
] = {
    "COMMON":
        common_offset
}


# SEX
sex_offsets = (
    CAL.groupby(
        "sex_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


pred_offset = (
    CAL[
        "sex_group"
    ]
    .map(
        sex_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    pred_offset,

    n_mean_parameters=len(
        sex_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "SEX",

        **stats,
    }
)


offset_specs[
    "SEX"
] = {
    str(k):
        float(v)

    for k, v
    in sex_offsets.items()
}


# GROUP
group_offsets = (
    CAL.groupby(
        "model_group"
    )[
        "offset_target"
    ]
    .mean()
    .to_dict()
)


pred_offset = (
    CAL[
        "model_group"
    ]
    .map(
        group_offsets
    )
    .to_numpy(
        dtype=float
    )
)


stats = gaussian_fit_statistics(
    CAL[
        "offset_target"
    ]
    .to_numpy(
        dtype=float
    )
    -
    pred_offset,

    n_mean_parameters=len(
        group_offsets
    ),
)


offset_candidate_rows.append(
    {
        "offset_model":
            "GROUP",

        **stats,
    }
)


offset_specs[
    "GROUP"
] = {
    str(k):
        float(v)

    for k, v
    in group_offsets.items()
}


OFFSET_CANDIDATES = pd.DataFrame(
    offset_candidate_rows
)


OFFSET_CANDIDATES[
    "delta_AICc"
] = (
    OFFSET_CANDIDATES[
        "AICc"
    ]
    -
    OFFSET_CANDIDATES[
        "AICc"
    ].min()
)


OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES.sort_values(
        [
            "AICc",
            "BIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


SELECTED_OFFSET_MODEL = str(
    OFFSET_CANDIDATES.loc[
        0,
        "offset_model",
    ]
)


SELECTED_OFFSETS = (
    offset_specs[
        SELECTED_OFFSET_MODEL
    ]
)


# --------------------------------------------------------------------------------------------------
# 16. APPLY SELECTED OFFSET
# --------------------------------------------------------------------------------------------------

if SELECTED_OFFSET_MODEL == "NONE":

    CAL[
        "v6_offset"
    ] = 0.0


elif SELECTED_OFFSET_MODEL == "COMMON":

    CAL[
        "v6_offset"
    ] = float(
        SELECTED_OFFSETS[
            "COMMON"
        ]
    )


elif SELECTED_OFFSET_MODEL == "SEX":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "sex_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


elif SELECTED_OFFSET_MODEL == "GROUP":

    CAL[
        "v6_offset"
    ] = (
        CAL[
            "model_group"
        ]
        .map(
            SELECTED_OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        SELECTED_OFFSET_MODEL
    )


CAL[
    "v6_prediction_mean"
] = (
    CAL[
        "v6_without_offset"
    ]
    +
    CAL[
        "v6_offset"
    ]
)


CAL[
    "v5_error"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v5_prediction_mean"
    ]
)


CAL[
    "v6_error"
] = (
    CAL[
        "mark_numeric_m"
    ]
    -
    CAL[
        "v6_prediction_mean"
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. CALIBRATION METRICS
# --------------------------------------------------------------------------------------------------

V5_MAE = float(
    CAL[
        "v5_error"
    ]
    .abs()
    .mean()
)


V6_MAE = float(
    CAL[
        "v6_error"
    ]
    .abs()
    .mean()
)


V5_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v5_error"
            ] ** 2
        )
    )
)


V6_RMSE = float(
    np.sqrt(
        np.mean(
            CAL[
                "v6_error"
            ] ** 2
        )
    )
)


CALIBRATION_METRICS = pd.DataFrame(
    [
        {
            "n":
                int(
                    len(CAL)
                ),

            "athletes":
                int(
                    CAL[
                        "aaAthleteId"
                    ]
                    .nunique()
                ),

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V5_to_V6_MAE_gain_m":
                V5_MAE
                -
                V6_MAE,

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V5_to_V6_RMSE_gain_m":
                V5_RMSE
                -
                V6_RMSE,

            "V6_Bias":
                float(
                    CAL[
                        "v6_error"
                    ]
                    .mean()
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. PERSISTENCE REFERENCE
# --------------------------------------------------------------------------------------------------

REFERENCE_DAYS = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


REF_RHO_I, REF_RHO_S = persistence_values(
    SELECTED_PERSISTENCE_MODEL,
    SELECTED_PERSISTENCE_PARAMS,
    REFERENCE_DAYS,
)


PERSISTENCE_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            REFERENCE_DAYS.astype(
                int
            ),

        "rho_intercept":
            REF_RHO_I,

        "rho_slope":
            REF_RHO_S,
    }
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

CALIBRATION_DATE_PASS = bool(
    CAL[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2025-01-01"
        ),
        pd.Timestamp(
            "2025-04-30"
        ),
        inclusive="both",
    )
    .all()
)


GAP_DOMAIN_PASS = bool(
    CAL[
        "rolling_state_gap_days"
    ]
    .between(
        1,
        365,
        inclusive="both",
    )
    .all()
)


ONE_FIRST_DATE_PASS = bool(
    not CAL.duplicated(
        subset=[
            "aaAthleteId",
            "model_group",
        ]
    )
    .any()
)


FINITE_PASS = bool(
    np.isfinite(
        CAL[
            [
                "structural_target",
                "rho_intercept",
                "rho_slope",
                "v6_offset",
                "v6_prediction_mean",
                "v6_error",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


PERSISTENCE_SELECTION_PASS = bool(
    len(
        PERSISTENCE_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        PERSISTENCE_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


OFFSET_SELECTION_PASS = bool(
    len(
        OFFSET_CANDIDATES
    )
    ==
    4

    and

    np.isfinite(
        OFFSET_CANDIDATES[
            "AICc"
        ]
    )
    .all()
)


TEST_NOT_USED_PASS = bool(
    not TEST_TOUCHED
)


CELL_49_PASS = bool(
    CALIBRATION_DATE_PASS
    and
    GAP_DOMAIN_PASS
    and
    ONE_FIRST_DATE_PASS
    and
    FINITE_PASS
    and
    PERSISTENCE_SELECTION_PASS
    and
    OFFSET_SELECTION_PASS
    and
    TEST_NOT_USED_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

CALIBRATION_SAMPLE_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_validation_calibration_sample.parquet"
)


PERSISTENCE_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_persistence_candidates.parquet"
)


OFFSET_CANDIDATES_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_offset_candidates.parquet"
)


HC3_SUMMARY_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_empirical_persistence_hc3.parquet"
)


V6_MEAN_SPEC_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_spec.json"
)


V6_AUDIT_PATH = (
    V6_ATTEMPT_DIR
    /
    "v6_mean_calibration_audit.json"
)


CAL.to_parquet(
    CALIBRATION_SAMPLE_PATH,
    index=False,
)


PERSISTENCE_CANDIDATES.to_parquet(
    PERSISTENCE_CANDIDATES_PATH,
    index=False,
)


OFFSET_CANDIDATES.to_parquet(
    OFFSET_CANDIDATES_PATH,
    index=False,
)


HC3_SUMMARY.to_parquet(
    HC3_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. FREEZE MODEL SPECIFICATION
# --------------------------------------------------------------------------------------------------

V6_MEAN_SPEC = {
    "model":
        "HISTORICAL_V6_FULL_STRUCTURAL_CORRECTION",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "calibration_window": {
        "start":
            "2025-01-01",

        "end":
            "2025-04-30",
    },

    "calibration_rule":
        (
            "First validation date per athlete × model_group; "
            "gap restricted to 1–365 days."
        ),

    "selected_persistence_model":
        SELECTED_PERSISTENCE_MODEL,

    "selected_persistence_parameters":
        SELECTED_PERSISTENCE_PARAMETER_DICT,

    "selected_offset_model":
        SELECTED_OFFSET_MODEL,

    "selected_offsets":
        SELECTED_OFFSETS,

    "formula":
        (
            "V6_FULL = "
            "V5 - athlete_intercept - athlete_slope_component "
            "+ rho_I(h)*athlete_intercept "
            "+ rho_S(h)*athlete_slope_component "
            "+ offset"
        ),

    "test_used":
        False,

    "architecture_changed_from_previous_folds":
        False,
}


atomic_json_write(
    V6_MEAN_SPEC,
    V6_MEAN_SPEC_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. PROVENANCE
# --------------------------------------------------------------------------------------------------

rolling_audit_path = (
    Path(
        HB2025_ROLLING_ATTEMPT_DIR
    )
    /
    "rolling_forecast_audit.json"
)


PARENT_ARTIFACTS = {
    "rolling_forecast_audit": {
        "path":
            str(
                rolling_audit_path
            ),

        "exists":
            bool(
                rolling_audit_path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    rolling_audit_path
                )
                if rolling_audit_path.exists()
                else None
            ),
    }
}


output_paths = {
    "calibration_sample":
        CALIBRATION_SAMPLE_PATH,

    "persistence_candidates":
        PERSISTENCE_CANDIDATES_PATH,

    "offset_candidates":
        OFFSET_CANDIDATES_PATH,

    "hc3_summary":
        HC3_SUMMARY_PATH,

    "v6_mean_spec":
        V6_MEAN_SPEC_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 23. AUDIT
# --------------------------------------------------------------------------------------------------

V6_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V6_MEAN_CALIBRATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_49_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Third independent historical VALIDATION-only "
            "V6 structural-persistence calibration."
        ),

    "model_logic_changed_from_previous_folds":
        False,

    "test_used":
        False,

    "sample": {
        "rows":
            int(
                len(CAL)
            ),

        "athletes":
            int(
                CAL[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "minimum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ]
                .min()
            ),

        "maximum_gap":
            float(
                CAL[
                    "rolling_state_gap_days"
                ]
                .max()
            ),
    },

    "selected_model":
        V6_MEAN_SPEC,

    "calibration_metrics":
        CALIBRATION_METRICS.iloc[
            0
        ].to_dict(),

    "qa": {
        "calibration_date_pass":
            CALIBRATION_DATE_PASS,

        "gap_domain_pass":
            GAP_DOMAIN_PASS,

        "one_first_date_per_athlete_group":
            ONE_FIRST_DATE_PASS,

        "finite_outputs":
            FINITE_PASS,

        "persistence_selection":
            PERSISTENCE_SELECTION_PASS,

        "offset_selection":
            OFFSET_SELECTION_PASS,

        "test_not_used":
            TEST_NOT_USED_PASS,

        "cell_pass":
            CELL_49_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    V6_AUDIT,
    V6_AUDIT_PATH,
)


V6_AUDIT_SHA256 = sha256_file(
    V6_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_V6_POINTER_PATH = (
    V6_STAGE_ROOT
    /
    "LATEST_V6_MEAN_CALIBRATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V6_MEAN_CALIBRATION",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_49_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                V6_ATTEMPT_DIR
            ),

        "v6_mean_spec":
            str(
                V6_MEAN_SPEC_PATH
            ),

        "audit_manifest":
            str(
                V6_AUDIT_PATH
            ),

        "audit_sha256":
            V6_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_V6_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 25. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v6_mean_calibration"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_49_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            V6_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_V6_POINTER_PATH
        ),

    "mean_spec":
        str(
            V6_MEAN_SPEC_PATH
        ),

    "audit_manifest":
        str(
            V6_AUDIT_PATH
        ),

    "audit_sha256":
        V6_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_V6_MEAN_CALIBRATION_COMPLETE"
    if CELL_49_PASS
    else
    "HB_2025_V6_MEAN_CALIBRATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 26. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_V6_CALIBRATION_SAMPLE = CAL


HB2025_V6_PERSISTENCE_CANDIDATES = (
    PERSISTENCE_CANDIDATES
)


HB2025_V6_OFFSET_CANDIDATES = (
    OFFSET_CANDIDATES
)


HB2025_V6_MEAN_SPEC = (
    V6_MEAN_SPEC
)


HB2025_V6_ATTEMPT_DIR = (
    V6_ATTEMPT_DIR
)


HB2025_V6_MEAN_SPEC_PATH = (
    V6_MEAN_SPEC_PATH
)


NOTEBOOK_2_CELL_49_PASS = (
    CELL_49_PASS
)


# --------------------------------------------------------------------------------------------------
# 27. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 160)
print("HB_2025 — HISTORICAL V6 MEAN CALIBRATION — VALIDATION ONLY")
print("=" * 160)


print()
print("CALIBRATION SAMPLE")
print("-" * 160)

print(
    f"Rows                           : "
    f"{len(CAL):,}"
)

print(
    f"Athletes                       : "
    f"{CAL['aaAthleteId'].nunique():,}"
)

print(
    f"Gap range                      : "
    f"{CAL['rolling_state_gap_days'].min():.0f}"
    f" -> "
    f"{CAL['rolling_state_gap_days'].max():.0f} days"
)

print(
    f"TEST used                      : "
    f"{TEST_TOUCHED}"
)


print()
print("EMPIRICAL CENTERED HC3 PERSISTENCE")
print("-" * 160)

display(
    HC3_SUMMARY
)


print()
print("PERSISTENCE MODEL COMPARISON")
print("-" * 160)

display(
    PERSISTENCE_CANDIDATES
)


print()
print(
    f"Selected persistence model     : "
    f"{SELECTED_PERSISTENCE_MODEL}"
)

print(
    f"Selected parameters            : "
    f"{SELECTED_PERSISTENCE_PARAMETER_DICT}"
)


print()
print("SELECTED PERSISTENCE BY HORIZON")
print("-" * 160)

display(
    PERSISTENCE_REFERENCE
)


print()
print("OFFSET MODEL COMPARISON")
print("-" * 160)

display(
    OFFSET_CANDIDATES
)


print()
print(
    f"Selected offset model          : "
    f"{SELECTED_OFFSET_MODEL}"
)

print(
    f"Selected offsets               : "
    f"{SELECTED_OFFSETS}"
)


print()
print("VALIDATION CALIBRATION METRICS")
print("-" * 160)

display(
    CALIBRATION_METRICS
)


print()
print("AUDIT / VERSIONING")
print("-" * 160)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{V6_ATTEMPT_DIR}"
)

print(
    f"Mean specification             : "
    f"{V6_MEAN_SPEC_PATH}"
)

print(
    f"Audit manifest                 : "
    f"{V6_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{V6_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 160)

print(
    f"VALIDATION dates only          : "
    f"{CALIBRATION_DATE_PASS}"
)

print(
    f"Gap domain 1–365 days          : "
    f"{GAP_DOMAIN_PASS}"
)

print(
    f"First athlete/group date only  : "
    f"{ONE_FIRST_DATE_PASS}"
)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"Persistence candidates valid   : "
    f"{PERSISTENCE_SELECTION_PASS}"
)

print(
    f"Offset candidates valid        : "
    f"{OFFSET_SELECTION_PASS}"
)

print(
    f"TEST untouched                 : "
    f"{TEST_NOT_USED_PASS}"
)


print()
print("=" * 160)

if CELL_49_PASS:

    print(
        "CELL 49 STATUS: HB_2025 HISTORICAL V6 MEAN CALIBRATION PASSED"
    )

else:

    print(
        "CELL 49 STATUS: HB_2025 V6 MEAN CALIBRATION REVIEW REQUIRED"
    )

print("=" * 160)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 40
# Original source SHA256: e209066d2be44438eaae4bbb9a7b6661af66885cb68e1a1d88a7b19594125000
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 50 — HB_2025 UNTOUCHED TEST: V5 vs V6 vs V6.1 OU-FRESHNESS BLEND
# ==================================================================================================
#
# Frozen before TEST:
#
#   V4          fitted <= 2024-12-31
#   V5 OU       fitted <= 2024-12-31
#   V6 mean     calibrated on 2025-01-01 -> 2025-04-30 VALIDATION only
#
# TEST:
#
#   2025-05-01 -> 2025-08-30
#
# V6.1:
#
#   w(h) = 1 - exp(-h / tau_V5)
#
#   V6.1 = V5 + w(h) * (V6_FULL - V5)
#
# No V6.1 parameters are fitted.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_49_PASS",

    "HB2025_ROLLING_TEST",

    "HB2025_V6_MEAN_SPEC",
    "HB2025_V6_MEAN_SPEC_PATH",
    "HB2025_V6_ATTEMPT_DIR",

    "HB2025_ROLLING_ATTEMPT_DIR",

    "HB2025_V5_TAU_DAYS",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )

if not NOTEBOOK_2_CELL_49_PASS:

    raise RuntimeError(
        "Cell 49 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"

CELL_ID = "CELL_50_HB2025_TEST_V6_V61"

STAGE_SCHEMA_VERSION = "1.0.0"

BOOTSTRAP_SEED = 42
BOOTSTRAP_REPS = 5000


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


BOOTSTRAP_HORIZONS = {
    "ALL":
        (0, np.inf),

    "01-30":
        (0, 30),

    "31-60":
        (30, 60),

    "61-120":
        (60, 120),

    "121-365":
        (120, 365),

    "366+":
        (365, np.inf),
}


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. IMMUTABLE ATTEMPT DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)

ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

HB2025_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)

TEST_STAGE_ROOT = (
    HB2025_DIR
    /
    "v61_test_evaluation"
)

TEST_ATTEMPT_DIR = (
    TEST_STAGE_ROOT
    /
    ATTEMPT_ID
)

TEST_ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. FROZEN TEST + SPECIFICATION
# --------------------------------------------------------------------------------------------------

TEST = (
    HB2025_ROLLING_TEST.copy()
    .reset_index(
        drop=True
    )
)


V6_SPEC = dict(
    HB2025_V6_MEAN_SPEC
)


TAU_V5 = float(
    HB2025_V5_TAU_DAYS
)


SELECTED_MODEL = (
    V6_SPEC[
        "selected_persistence_model"
    ]
)


PERSISTENCE_PARAMS = (
    V6_SPEC[
        "selected_persistence_parameters"
    ]
)


OFFSET_MODEL = (
    V6_SPEC[
        "selected_offset_model"
    ]
)


OFFSETS = (
    V6_SPEC[
        "selected_offsets"
    ]
)


TEST[
    "sex_group"
] = (
    TEST[
        "model_group"
    ]
    .astype(str)
    .str[0]
)


# --------------------------------------------------------------------------------------------------
# 6. FROZEN V6 PERSISTENCE
# --------------------------------------------------------------------------------------------------

def frozen_persistence(
    model_name,
    params,
    horizon,
):

    h = np.asarray(
        horizon,
        dtype=float,
    )


    if model_name == "M0_CONST_CONST":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M1_CONST_EXP":

        rho_I = np.full_like(
            h,
            float(
                params[
                    "c_intercept"
                ]
            ),
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    elif model_name == "M2_EXP_CONST":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.full_like(
            h,
            float(
                params[
                    "c_slope"
                ]
            ),
        )


    elif model_name == "M3_EXP_EXP":

        rho_I = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_intercept_days"
                ]
            )
        )

        rho_S = np.exp(
            -
            h
            /
            float(
                params[
                    "tau_slope_days"
                ]
            )
        )


    else:

        raise RuntimeError(
            f"Unknown persistence model: {model_name}"
        )


    return rho_I, rho_S


# --------------------------------------------------------------------------------------------------
# 7. APPLY FROZEN V6 STRUCTURAL SPECIFICATION
# --------------------------------------------------------------------------------------------------

h = (
    TEST[
        "rolling_state_gap_days"
    ]
    .to_numpy(
        dtype=float
    )
)


rho_I, rho_S = frozen_persistence(
    SELECTED_MODEL,
    PERSISTENCE_PARAMS,
    h,
)


TEST[
    "v6_rho_intercept"
] = rho_I


TEST[
    "v6_rho_slope"
] = rho_S


TEST[
    "v6_base_without_athlete_structure"
] = (
    TEST[
        "v5_prediction_mean"
    ]
    -
    TEST[
        "v4_athlete_intercept_component"
    ]
    -
    TEST[
        "v4_athlete_slope_component"
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. APPLY FROZEN OFFSET
# --------------------------------------------------------------------------------------------------

if OFFSET_MODEL == "NONE":

    TEST[
        "v6_offset"
    ] = 0.0


elif OFFSET_MODEL == "COMMON":

    TEST[
        "v6_offset"
    ] = float(
        OFFSETS[
            "COMMON"
        ]
    )


elif OFFSET_MODEL == "SEX":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "sex_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


elif OFFSET_MODEL == "GROUP":

    TEST[
        "v6_offset"
    ] = (
        TEST[
            "model_group"
        ]
        .map(
            OFFSETS
        )
        .astype(float)
    )


else:

    raise RuntimeError(
        f"Unknown offset model: {OFFSET_MODEL}"
    )


if TEST[
    "v6_offset"
].isna().any():

    raise RuntimeError(
        "Frozen V6 offset produced missing values."
    )


# --------------------------------------------------------------------------------------------------
# 9. FULL V6 MEAN
# --------------------------------------------------------------------------------------------------

TEST[
    "v6_prediction_mean"
] = (
    TEST[
        "v6_base_without_athlete_structure"
    ]

    +

    TEST[
        "v6_rho_intercept"
    ]
    *
    TEST[
        "v4_athlete_intercept_component"
    ]

    +

    TEST[
        "v6_rho_slope"
    ]
    *
    TEST[
        "v4_athlete_slope_component"
    ]

    +

    TEST[
        "v6_offset"
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. V6.1 — TRAIN-ONLY OU FRESHNESS BLEND
# --------------------------------------------------------------------------------------------------

TEST[
    "v61_ou_phi"
] = np.exp(
    -
    TEST[
        "rolling_state_gap_days"
    ]
    /
    TAU_V5
)


TEST[
    "v61_weight"
] = (
    1.0
    -
    TEST[
        "v61_ou_phi"
    ]
)


TEST[
    "v61_prediction_mean"
] = (
    TEST[
        "v5_prediction_mean"
    ]

    +

    TEST[
        "v61_weight"
    ]
    *
    (
        TEST[
            "v6_prediction_mean"
        ]
        -
        TEST[
            "v5_prediction_mean"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ERRORS
# --------------------------------------------------------------------------------------------------

for model, column in {
    "v5":
        "v5_prediction_mean",

    "v6":
        "v6_prediction_mean",

    "v61":
        "v61_prediction_mean",
}.items():

    TEST[
        f"{model}_error"
    ] = (
        TEST[
            "mark_numeric_m"
        ]
        -
        TEST[
            column
        ]
    )


# --------------------------------------------------------------------------------------------------
# 12. GAP BAND
# --------------------------------------------------------------------------------------------------

TEST[
    "state_gap_band"
] = pd.cut(
    TEST[
        "rolling_state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 13. METRIC HELPERS
# --------------------------------------------------------------------------------------------------

def metric_row(
    df,
    model,
):

    error = (
        df[
            f"{model}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    return {
        "model":
            (
                model.upper()
                if model != "v61"
                else "V6.1"
            ),

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ].nunique()
            ),

        "MAE":
            float(
                np.mean(
                    np.abs(
                        error
                    )
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "Bias":
            float(
                np.mean(
                    error
                )
            ),
    }


def pairwise_gain(
    df,
    baseline,
    candidate,
):

    e0 = (
        df[
            f"{baseline}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    e1 = (
        df[
            f"{candidate}_error"
        ]
        .to_numpy(
            dtype=float
        )
    )

    mae0 = float(
        np.mean(
            np.abs(
                e0
            )
        )
    )

    mae1 = float(
        np.mean(
            np.abs(
                e1
            )
        )
    )

    rmse0 = float(
        np.sqrt(
            np.mean(
                e0 ** 2
            )
        )
    )

    rmse1 = float(
        np.sqrt(
            np.mean(
                e1 ** 2
            )
        )
    )

    return {
        "baseline":
            (
                baseline.upper()
                if baseline != "v61"
                else "V6.1"
            ),

        "candidate":
            (
                candidate.upper()
                if candidate != "v61"
                else "V6.1"
            ),

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ].nunique()
            ),

        "baseline_MAE":
            mae0,

        "candidate_MAE":
            mae1,

        "MAE_gain_m":
            mae0
            -
            mae1,

        "baseline_RMSE":
            rmse0,

        "candidate_RMSE":
            rmse1,

        "RMSE_gain_m":
            rmse0
            -
            rmse1,
    }


# --------------------------------------------------------------------------------------------------
# 14. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

OVERALL_METRICS = pd.DataFrame(
    [
        metric_row(
            TEST,
            model,
        )

        for model in [
            "v5",
            "v6",
            "v61",
        ]
    ]
)


PAIRWISE_OVERALL = pd.DataFrame(
    [
        pairwise_gain(
            TEST,
            "v5",
            "v6",
        ),

        pairwise_gain(
            TEST,
            "v5",
            "v61",
        ),

        pairwise_gain(
            TEST,
            "v6",
            "v61",
        ),
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. GAP METRICS + GAINS
# --------------------------------------------------------------------------------------------------

gap_metric_rows = []
gap_gain_rows = []


for gap_band, g in TEST.groupby(
    "state_gap_band",
    observed=True,
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )

        row[
            "state_gap_band"
        ] = str(
            gap_band
        )

        row[
            "mean_gap_days"
        ] = float(
            g[
                "rolling_state_gap_days"
            ].mean()
        )

        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ].mean()
        )

        gap_metric_rows.append(
            row
        )


    for baseline, candidate in [
        ("v5", "v6"),
        ("v5", "v61"),
        ("v6", "v61"),
    ]:

        row = pairwise_gain(
            g,
            baseline,
            candidate,
        )

        row[
            "state_gap_band"
        ] = str(
            gap_band
        )

        row[
            "mean_v61_weight"
        ] = float(
            g[
                "v61_weight"
            ].mean()
        )

        gap_gain_rows.append(
            row
        )


GAP_METRICS = pd.DataFrame(
    gap_metric_rows
)


GAP_GAINS = pd.DataFrame(
    gap_gain_rows
)


# --------------------------------------------------------------------------------------------------
# 16. MODEL GROUP METRICS
# --------------------------------------------------------------------------------------------------

group_rows = []


for model_group, g in TEST.groupby(
    "model_group"
):

    for model in [
        "v5",
        "v6",
        "v61",
    ]:

        row = metric_row(
            g,
            model,
        )

        row[
            "model_group"
        ] = model_group

        group_rows.append(
            row
        )


GROUP_METRICS = pd.DataFrame(
    group_rows
)


# --------------------------------------------------------------------------------------------------
# 17. ATHLETE-CLUSTER BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def cluster_bootstrap_pair(
    df,
    baseline,
    candidate,
    reps=5000,
    seed=42,
):

    d = (
        df[
            [
                "aaAthleteId",
                f"{baseline}_error",
                f"{candidate}_error",
            ]
        ]
        .copy()
    )


    d[
        "abs_gain"
    ] = (
        d[
            f"{baseline}_error"
        ].abs()
        -
        d[
            f"{candidate}_error"
        ].abs()
    )


    d[
        "baseline_sq"
    ] = (
        d[
            f"{baseline}_error"
        ] ** 2
    )


    d[
        "candidate_sq"
    ] = (
        d[
            f"{candidate}_error"
        ] ** 2
    )


    cluster = (
        d.groupby(
            "aaAthleteId",
            as_index=False,
        )
        .agg(
            n=(
                "abs_gain",
                "size",
            ),

            abs_gain_sum=(
                "abs_gain",
                "sum",
            ),

            baseline_sq_sum=(
                "baseline_sq",
                "sum",
            ),

            candidate_sq_sum=(
                "candidate_sq",
                "sum",
            ),
        )
    )


    n_cluster = len(
        cluster
    )


    n = (
        cluster[
            "n"
        ]
        .to_numpy(
            dtype=float
        )
    )


    abs_gain_sum = (
        cluster[
            "abs_gain_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    baseline_sq_sum = (
        cluster[
            "baseline_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    candidate_sq_sum = (
        cluster[
            "candidate_sq_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rng = np.random.default_rng(
        seed
    )


    mae_boot = np.empty(
        reps,
        dtype=float,
    )


    rmse_boot = np.empty(
        reps,
        dtype=float,
    )


    for b in range(
        reps
    ):

        idx = rng.integers(
            0,
            n_cluster,
            size=n_cluster,
        )

        total_n = float(
            n[
                idx
            ].sum()
        )

        mae_boot[b] = float(
            abs_gain_sum[
                idx
            ].sum()
            /
            total_n
        )


        baseline_rmse = np.sqrt(
            baseline_sq_sum[
                idx
            ].sum()
            /
            total_n
        )


        candidate_rmse = np.sqrt(
            candidate_sq_sum[
                idx
            ].sum()
            /
            total_n
        )


        rmse_boot[b] = (
            baseline_rmse
            -
            candidate_rmse
        )


    observed = pairwise_gain(
        df,
        baseline,
        candidate,
    )


    return {
        "baseline":
            observed[
                "baseline"
            ],

        "candidate":
            observed[
                "candidate"
            ],

        "n":
            int(
                len(df)
            ),

        "athletes":
            int(
                n_cluster
            ),

        "MAE_gain_m":
            observed[
                "MAE_gain_m"
            ],

        "MAE_CI_low":
            float(
                np.quantile(
                    mae_boot,
                    0.025,
                )
            ),

        "MAE_CI_high":
            float(
                np.quantile(
                    mae_boot,
                    0.975,
                )
            ),

        "MAE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    mae_boot > 0
                )
            ),

        "RMSE_gain_m":
            observed[
                "RMSE_gain_m"
            ],

        "RMSE_CI_low":
            float(
                np.quantile(
                    rmse_boot,
                    0.025,
                )
            ),

        "RMSE_CI_high":
            float(
                np.quantile(
                    rmse_boot,
                    0.975,
                )
            ),

        "RMSE_positive_pct":
            float(
                100.0
                *
                np.mean(
                    rmse_boot > 0
                )
            ),
    }


# --------------------------------------------------------------------------------------------------
# 18. BOOTSTRAP OVERALL + HORIZONS
# --------------------------------------------------------------------------------------------------

bootstrap_rows = []


for horizon_name, (
    lower,
    upper,
) in BOOTSTRAP_HORIZONS.items():

    if horizon_name == "ALL":

        subset = TEST.copy()

    else:

        subset = TEST.loc[
            (
                TEST[
                    "rolling_state_gap_days"
                ]
                >
                lower
            )
            &
            (
                TEST[
                    "rolling_state_gap_days"
                ]
                <=
                upper
            )
        ].copy()


    if len(
        subset
    ) == 0:

        continue


    for baseline, candidate in [
        ("v5", "v6"),
        ("v5", "v61"),
        ("v6", "v61"),
    ]:

        row = cluster_bootstrap_pair(
            subset,
            baseline,
            candidate,
            reps=BOOTSTRAP_REPS,
            seed=BOOTSTRAP_SEED,
        )

        row[
            "horizon"
        ] = horizon_name

        bootstrap_rows.append(
            row
        )


BOOTSTRAP_SUMMARY = pd.DataFrame(
    bootstrap_rows
)


# --------------------------------------------------------------------------------------------------
# 19. QA
# --------------------------------------------------------------------------------------------------

FINITE_PASS = bool(
    np.isfinite(
        TEST[
            [
                "v5_prediction_mean",
                "v6_prediction_mean",
                "v61_prediction_mean",

                "v5_error",
                "v6_error",
                "v61_error",

                "v6_rho_intercept",
                "v6_rho_slope",

                "v61_ou_phi",
                "v61_weight",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


WEIGHT_BOUNDS_PASS = bool(
    TEST[
        "v61_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


TEST_DATE_PASS = bool(
    TEST[
        "result_date_dt"
    ]
    .between(
        pd.Timestamp(
            "2025-05-01"
        ),
        pd.Timestamp(
            "2025-08-30"
        ),
        inclusive="both",
    )
    .all()
)


ROW_ACCOUNTING_PASS = bool(
    len(TEST)
    ==
    len(
        HB2025_ROLLING_TEST
    )
)


NO_NEW_PARAMETERS_PASS = True


V61_ENDPOINT_LOGIC_PASS = bool(
    TEST.loc[
        TEST[
            "rolling_state_gap_days"
        ]
        <=
        30,
        "v61_weight",
    ].mean()
    <
    0.35

    and

    TEST.loc[
        TEST[
            "rolling_state_gap_days"
        ]
        >
        365,
        "v61_weight",
    ].mean()
    >
    0.95
)


CELL_50_PASS = bool(
    FINITE_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    TEST_DATE_PASS
    and
    ROW_ACCOUNTING_PASS
    and
    NO_NEW_PARAMETERS_PASS
    and
    V61_ENDPOINT_LOGIC_PASS
)


# --------------------------------------------------------------------------------------------------
# 20. SAVE IMMUTABLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

TEST_PREDICTIONS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "hb2025_test_v5_v6_v61_predictions.parquet"
)


OVERALL_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "overall_metrics.parquet"
)


PAIRWISE_OVERALL_PATH = (
    TEST_ATTEMPT_DIR
    /
    "pairwise_overall_gains.parquet"
)


GAP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_metrics.parquet"
)


GAP_GAINS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "gap_gains.parquet"
)


GROUP_METRICS_PATH = (
    TEST_ATTEMPT_DIR
    /
    "group_metrics.parquet"
)


BOOTSTRAP_PATH = (
    TEST_ATTEMPT_DIR
    /
    "athlete_cluster_bootstrap.parquet"
)


TEST_AUDIT_PATH = (
    TEST_ATTEMPT_DIR
    /
    "v61_test_evaluation_audit.json"
)


TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)


OVERALL_METRICS.to_parquet(
    OVERALL_METRICS_PATH,
    index=False,
)


PAIRWISE_OVERALL.to_parquet(
    PAIRWISE_OVERALL_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


GAP_GAINS.to_parquet(
    GAP_GAINS_PATH,
    index=False,
)


GROUP_METRICS.to_parquet(
    GROUP_METRICS_PATH,
    index=False,
)


BOOTSTRAP_SUMMARY.to_parquet(
    BOOTSTRAP_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 21. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {
    "v6_mean_spec":
        Path(
            HB2025_V6_MEAN_SPEC_PATH
        ),

    "rolling_test_predictions":
        (
            Path(
                HB2025_ROLLING_ATTEMPT_DIR
            )
            /
            "test_rolling_predictions.parquet"
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else None
            ),
    }


output_paths = {
    "test_predictions":
        TEST_PREDICTIONS_PATH,

    "overall_metrics":
        OVERALL_METRICS_PATH,

    "pairwise_overall":
        PAIRWISE_OVERALL_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "gap_gains":
        GAP_GAINS_PATH,

    "group_metrics":
        GROUP_METRICS_PATH,

    "athlete_cluster_bootstrap":
        BOOTSTRAP_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 22. AUDIT
# --------------------------------------------------------------------------------------------------

TEST_AUDIT = {
    "audit_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "stage":
        "V61_UNTOUCHED_TEST_EVALUATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_50_PASS
            else
            "FAILED"
        ),

    "evaluation_window": {
        "start":
            "2025-05-01",

        "end":
            "2025-08-30",
    },

    "test_role":
        (
            "Third and final untouched historical TEST. "
            "No V6 or V6.1 parameters fitted on this period."
        ),

    "frozen_v6_spec": {
        "persistence_model":
            SELECTED_MODEL,

        "persistence_parameters":
            PERSISTENCE_PARAMS,

        "offset_model":
            OFFSET_MODEL,

        "offsets":
            OFFSETS,
    },

    "v61_spec": {
        "formula":
            "V5 + (1-exp(-gap/tau_V5))*(V6_FULL-V5)",

        "tau_days":
            TAU_V5,

        "tau_source":
            "HB_2025 TRAIN-only V5 OU fit",

        "new_fitted_parameters":
            0,
    },

    "bootstrap": {
        "method":
            "paired athlete-cluster bootstrap",

        "replications":
            BOOTSTRAP_REPS,

        "seed":
            BOOTSTRAP_SEED,
    },

    "qa": {
        "finite":
            FINITE_PASS,

        "weight_bounds":
            WEIGHT_BOUNDS_PASS,

        "test_dates":
            TEST_DATE_PASS,

        "row_accounting":
            ROW_ACCOUNTING_PASS,

        "no_new_parameters":
            NO_NEW_PARAMETERS_PASS,

        "endpoint_logic":
            V61_ENDPOINT_LOGIC_PASS,

        "cell_pass":
            CELL_50_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    TEST_AUDIT,
    TEST_AUDIT_PATH,
)


TEST_AUDIT_SHA256 = sha256_file(
    TEST_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 23. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    TEST_STAGE_ROOT
    /
    "LATEST_V61_TEST_EVALUATION.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "stage":
            "V61_UNTOUCHED_TEST_EVALUATION",

        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_50_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                TEST_ATTEMPT_DIR
            ),

        "test_predictions":
            str(
                TEST_PREDICTIONS_PATH
            ),

        "bootstrap":
            str(
                BOOTSTRAP_PATH
            ),

        "audit_manifest":
            str(
                TEST_AUDIT_PATH
            ),

        "audit_sha256":
            TEST_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "v61_test_evaluation"
] = {
    "latest_attempt_id":
        ATTEMPT_ID,

    "status":
        (
            "PASSED"
            if CELL_50_PASS
            else
            "FAILED"
        ),

    "attempt_directory":
        str(
            TEST_ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "audit_manifest":
        str(
            TEST_AUDIT_PATH
        ),

    "audit_sha256":
        TEST_AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_V61_TEST_EVALUATION_COMPLETE"
    if CELL_50_PASS
    else
    "HB_2025_V61_TEST_EVALUATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 25. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_V61_TEST = TEST

HB2025_V61_OVERALL_METRICS = (
    OVERALL_METRICS
)

HB2025_V61_PAIRWISE_OVERALL = (
    PAIRWISE_OVERALL
)

HB2025_V61_GAP_METRICS = (
    GAP_METRICS
)

HB2025_V61_GAP_GAINS = (
    GAP_GAINS
)

HB2025_V61_GROUP_METRICS = (
    GROUP_METRICS
)

HB2025_V61_BOOTSTRAP = (
    BOOTSTRAP_SUMMARY
)

HB2025_V61_TEST_ATTEMPT_DIR = (
    TEST_ATTEMPT_DIR
)

NOTEBOOK_2_CELL_50_PASS = (
    CELL_50_PASS
)


# --------------------------------------------------------------------------------------------------
# 26. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2025 — UNTOUCHED TEST: V5 vs V6 vs V6.1")
print("=" * 165)


print()
print("FROZEN MODEL SPECIFICATION")
print("-" * 165)

print(
    f"V6 persistence model           : "
    f"{SELECTED_MODEL}"
)

print(
    f"V6 persistence parameters      : "
    f"{PERSISTENCE_PARAMS}"
)

print(
    f"V6 offset model                : "
    f"{OFFSET_MODEL}"
)

print(
    f"V6 offsets                     : "
    f"{OFFSETS}"
)

print(
    f"V6.1 TRAIN-only tau            : "
    f"{TAU_V5:.6f} days"
)


print()
print("OVERALL TEST METRICS")
print("-" * 165)

display(
    OVERALL_METRICS
)


print()
print("OVERALL PAIRWISE GAINS")
print("-" * 165)

display(
    PAIRWISE_OVERALL
)


print()
print("BY STATE GAP — METRICS")
print("-" * 165)

display(
    GAP_METRICS
)


print()
print("BY STATE GAP — PAIRWISE GAINS")
print("-" * 165)

display(
    GAP_GAINS
)


print()
print("BY MODEL GROUP")
print("-" * 165)

display(
    GROUP_METRICS
)


print()
print("PAIRED ATHLETE-CLUSTER BOOTSTRAP")
print("-" * 165)

display(
    BOOTSTRAP_SUMMARY
)


print()
print("V6.1 WEIGHT REFERENCE")
print("-" * 165)

weight_days = np.array(
    [
        7,
        14,
        30,
        60,
        90,
        120,
        180,
        365,
    ],
    dtype=float,
)


WEIGHT_REFERENCE = pd.DataFrame(
    {
        "gap_days":
            weight_days.astype(
                int
            ),

        "OU_phi":
            np.exp(
                -
                weight_days
                /
                TAU_V5
            ),
    }
)


WEIGHT_REFERENCE[
    "V61_weight"
] = (
    1.0
    -
    WEIGHT_REFERENCE[
        "OU_phi"
    ]
)


display(
    WEIGHT_REFERENCE
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable attempt directory    : "
    f"{TEST_ATTEMPT_DIR}"
)

print(
    f"Audit manifest                 : "
    f"{TEST_AUDIT_PATH}"
)

print(
    f"Audit SHA256                   : "
    f"{TEST_AUDIT_SHA256}"
)

print(
    f"Git commit                     : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 165)

print(
    f"Finite outputs                 : "
    f"{FINITE_PASS}"
)

print(
    f"V6.1 weights within [0,1]      : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"TEST dates only                : "
    f"{TEST_DATE_PASS}"
)

print(
    f"No new fitted parameters       : "
    f"{NO_NEW_PARAMETERS_PASS}"
)

print(
    f"Row accounting                 : "
    f"{ROW_ACCOUNTING_PASS}"
)

print(
    f"Fresh/stale endpoint logic     : "
    f"{V61_ENDPOINT_LOGIC_PASS}"
)


print()
print("=" * 165)

if CELL_50_PASS:

    print(
        "CELL 50 STATUS: HB_2025 V6.1 UNTOUCHED TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 50 STATUS: HB_2025 V6.1 TEST EVALUATION REVIEW REQUIRED"
    )

print("=" * 165)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 41
# Original source SHA256: 4da19571951c75c38e23fdc33e39df29c0b031000b8b96cb9c6855ef2dae86e6
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 51 — HB_2025 HISTORICAL EVIDENCE FREEZE
# ==================================================================================================
#
# Purpose
# -------
# Freeze the third and final untouched historical TEST evidence.
#
# NO fitting.
# NO architecture changes.
# NO thresholds changed.
#
# After this cell:
#       HB_2023
#       HB_2024
#       HB_2025
#
# are all immutable and can be synthesized in Cell 52.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_50_PASS",

    "HB2025_V61_OVERALL_METRICS",
    "HB2025_V61_PAIRWISE_OVERALL",
    "HB2025_V61_GAP_GAINS",
    "HB2025_V61_GROUP_METRICS",
    "HB2025_V61_BOOTSTRAP",

    "HB2025_V61_TEST_ATTEMPT_DIR",

    "HB2025_V5_TAU_DAYS",
    "HB2025_V6_MEAN_SPEC",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_50_PASS:

    raise RuntimeError(
        "Cell 50 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


# --------------------------------------------------------------------------------------------------
# 3. IMMUTABLE EVIDENCE DIRECTORY
# --------------------------------------------------------------------------------------------------

FOLD_NAME = "HB_2025"


RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


HB2025_DIR = (
    Path(TRAINING_RUN_DIR)
    /
    "historical_backtest"
    /
    FOLD_NAME
)


EVIDENCE_ROOT = (
    HB2025_DIR
    /
    "fold_evidence"
)


EVIDENCE_DIR = (
    EVIDENCE_ROOT
    /
    ATTEMPT_ID
)


EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 4. INPUT TABLES
# --------------------------------------------------------------------------------------------------

OVERALL = (
    HB2025_V61_OVERALL_METRICS.copy()
)


PAIRWISE = (
    HB2025_V61_PAIRWISE_OVERALL.copy()
)


GAP_GAINS = (
    HB2025_V61_GAP_GAINS.copy()
)


GROUP_METRICS = (
    HB2025_V61_GROUP_METRICS.copy()
)


BOOTSTRAP = (
    HB2025_V61_BOOTSTRAP.copy()
)


# --------------------------------------------------------------------------------------------------
# 5. LOOKUP HELPERS
# --------------------------------------------------------------------------------------------------

def get_pairwise(
    baseline,
    candidate,
):

    row = PAIRWISE.loc[
        PAIRWISE[
            "baseline"
        ].eq(
            baseline
        )
        &
        PAIRWISE[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover "
            f"{baseline} -> {candidate}."
        )


    return row.iloc[0]


def get_bootstrap(
    horizon,
    baseline,
    candidate,
):

    row = BOOTSTRAP.loc[
        BOOTSTRAP[
            "horizon"
        ].eq(
            horizon
        )
        &
        BOOTSTRAP[
            "baseline"
        ].eq(
            baseline
        )
        &
        BOOTSTRAP[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely recover bootstrap "
            f"{horizon}: {baseline} -> {candidate}."
        )


    return row.iloc[0]


# --------------------------------------------------------------------------------------------------
# 6. KEY COMPARISONS
# --------------------------------------------------------------------------------------------------

V5_TO_V6 = get_pairwise(
    "V5",
    "V6",
)


V5_TO_V61 = get_pairwise(
    "V5",
    "V6.1",
)


V6_TO_V61 = get_pairwise(
    "V6",
    "V6.1",
)


BOOT_ALL_V5_V6 = get_bootstrap(
    "ALL",
    "V5",
    "V6",
)


BOOT_ALL_V5_V61 = get_bootstrap(
    "ALL",
    "V5",
    "V6.1",
)


BOOT_ALL_V6_V61 = get_bootstrap(
    "ALL",
    "V6",
    "V6.1",
)


BOOT_FRESH_V5_V6 = get_bootstrap(
    "01-30",
    "V5",
    "V6",
)


BOOT_FRESH_V5_V61 = get_bootstrap(
    "01-30",
    "V5",
    "V6.1",
)


BOOT_31_60_V5_V61 = get_bootstrap(
    "31-60",
    "V5",
    "V6.1",
)


BOOT_61_120_V5_V61 = get_bootstrap(
    "61-120",
    "V5",
    "V6.1",
)


BOOT_121_365_V5_V61 = get_bootstrap(
    "121-365",
    "V5",
    "V6.1",
)


BOOT_366_PLUS_V5_V61 = get_bootstrap(
    "366+",
    "V5",
    "V6.1",
)


# --------------------------------------------------------------------------------------------------
# 7. OVERALL METRICS
# --------------------------------------------------------------------------------------------------

O = (
    OVERALL.set_index(
        "model"
    )
)


V5_MAE = float(
    O.loc[
        "V5",
        "MAE",
    ]
)


V6_MAE = float(
    O.loc[
        "V6",
        "MAE",
    ]
)


V61_MAE = float(
    O.loc[
        "V6.1",
        "MAE",
    ]
)


V5_RMSE = float(
    O.loc[
        "V5",
        "RMSE",
    ]
)


V6_RMSE = float(
    O.loc[
        "V6",
        "RMSE",
    ]
)


V61_RMSE = float(
    O.loc[
        "V6.1",
        "RMSE",
    ]
)


V5_BIAS = float(
    O.loc[
        "V5",
        "Bias",
    ]
)


V6_BIAS = float(
    O.loc[
        "V6",
        "Bias",
    ]
)


V61_BIAS = float(
    O.loc[
        "V6.1",
        "Bias",
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. DESCRIPTIVE EVIDENCE FLAGS
# --------------------------------------------------------------------------------------------------
#
# These are observations only.
# They are NOT model-promotion rules.
#
# --------------------------------------------------------------------------------------------------

EVIDENCE_FLAGS = {

    "v6_overall_mae_better_than_v5":
        bool(
            V6_MAE
            <
            V5_MAE
        ),

    "v6_overall_mae_gain_ci_above_zero":
        bool(
            BOOT_ALL_V5_V6[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_overall_mae_better_than_v5":
        bool(
            V61_MAE
            <
            V5_MAE
        ),

    "v61_overall_mae_gain_ci_above_zero":
        bool(
            BOOT_ALL_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_overall_mae_better_than_v6":
        bool(
            V61_MAE
            <
            V6_MAE
        ),

    "v61_vs_v6_mae_ci_above_zero":
        bool(
            BOOT_ALL_V6_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v6_fresh_mae_better_than_v5":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_gain_m"
            ]
            >
            0
        ),

    "v6_fresh_mae_gain_ci_above_zero":
        bool(
            BOOT_FRESH_V5_V6[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_fresh_mae_better_than_v5":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_gain_m"
            ]
            >
            0
        ),

    "v61_fresh_mae_gain_ci_above_zero":
        bool(
            BOOT_FRESH_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_31_60_mae_gain_ci_above_zero":
        bool(
            BOOT_31_60_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_61_120_mae_gain_ci_above_zero":
        bool(
            BOOT_61_120_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_121_365_mae_gain_ci_above_zero":
        bool(
            BOOT_121_365_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),

    "v61_366_plus_mae_gain_ci_above_zero":
        bool(
            BOOT_366_PLUS_V5_V61[
                "MAE_CI_low"
            ]
            >
            0
        ),
}


# --------------------------------------------------------------------------------------------------
# 9. CROSS-FOLD SUMMARY ROW
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "test_start":
                pd.Timestamp(
                    "2025-05-01"
                ),

            "test_end":
                pd.Timestamp(
                    "2025-08-30"
                ),

            "test_rows":
                int(
                    O.loc[
                        "V5",
                        "n",
                    ]
                ),

            "test_athletes":
                int(
                    O.loc[
                        "V5",
                        "athletes",
                    ]
                ),

            "v5_tau_days":
                float(
                    HB2025_V5_TAU_DAYS
                ),

            "v6_persistence_model":
                HB2025_V6_MEAN_SPEC[
                    "selected_persistence_model"
                ],

            "v6_offset_model":
                HB2025_V6_MEAN_SPEC[
                    "selected_offset_model"
                ],

            "V5_MAE":
                V5_MAE,

            "V6_MAE":
                V6_MAE,

            "V61_MAE":
                V61_MAE,

            "V5_to_V6_MAE_gain_m":
                float(
                    V5_TO_V6[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V6_CI_low":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V6_CI_high":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "V5_to_V61_MAE_gain_m":
                float(
                    V5_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V61_CI_low":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V61_CI_high":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),

            "V6_to_V61_MAE_gain_m":
                float(
                    V6_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_RMSE":
                V5_RMSE,

            "V6_RMSE":
                V6_RMSE,

            "V61_RMSE":
                V61_RMSE,

            "V5_Bias":
                V5_BIAS,

            "V6_Bias":
                V6_BIAS,

            "V61_Bias":
                V61_BIAS,

            "fresh_V5_to_V6_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V6_CI_low":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V6_CI_high":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "fresh_V5_to_V61_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V61_CI_low":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V61_CI_high":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. SAVE IMMUTABLE EVIDENCE
# --------------------------------------------------------------------------------------------------

FOLD_SUMMARY_PATH = (
    EVIDENCE_DIR
    /
    "fold_summary.parquet"
)


EVIDENCE_JSON_PATH = (
    EVIDENCE_DIR
    /
    "fold_evidence.json"
)


FOLD_SUMMARY.to_parquet(
    FOLD_SUMMARY_PATH,
    index=False,
)


CELL50_AUDIT_PATH = (
    Path(
        HB2025_V61_TEST_ATTEMPT_DIR
    )
    /
    "v61_test_evaluation_audit.json"
)


EVIDENCE_SPEC = {
    "evidence_schema_version":
        "1.0.0",

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        "EVIDENCE_FROZEN",

    "purpose":
        (
            "Immutable third and final historical TEST evidence "
            "for cross-fold V6/V6.1 architecture assessment."
        ),

    "decision_note":
        (
            "No architecture or production-release decision is made "
            "inside this freeze step. Cross-fold synthesis follows."
        ),

    "overall": {
        "V5": {
            "MAE":
                V5_MAE,

            "RMSE":
                V5_RMSE,

            "Bias":
                V5_BIAS,
        },

        "V6": {
            "MAE":
                V6_MAE,

            "RMSE":
                V6_RMSE,

            "Bias":
                V6_BIAS,
        },

        "V6.1": {
            "MAE":
                V61_MAE,

            "RMSE":
                V61_RMSE,

            "Bias":
                V61_BIAS,
        },
    },

    "overall_bootstrap": {
        "V5_to_V6": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },

        "V5_to_V61": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },

        "V6_to_V61": {
            "MAE_gain_m":
                float(
                    BOOT_ALL_V6_V61[
                        "MAE_gain_m"
                    ]
                ),

            "CI95": [
                float(
                    BOOT_ALL_V6_V61[
                        "MAE_CI_low"
                    ]
                ),

                float(
                    BOOT_ALL_V6_V61[
                        "MAE_CI_high"
                    ]
                ),
            ],
        },
    },

    "fresh_01_30_days": {
        "V5_to_V6_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V6[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V6_CI95": [
            float(
                BOOT_FRESH_V5_V6[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_FRESH_V5_V6[
                    "MAE_CI_high"
                ]
            ),
        ],

        "V5_to_V61_MAE_gain_m":
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_gain_m"
                ]
            ),

        "V5_to_V61_CI95": [
            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_low"
                ]
            ),

            float(
                BOOT_FRESH_V5_V61[
                    "MAE_CI_high"
                ]
            ),
        ],
    },

    "evidence_flags":
        EVIDENCE_FLAGS,

    "v5_tau_days":
        float(
            HB2025_V5_TAU_DAYS
        ),

    "v6_mean_spec":
        HB2025_V6_MEAN_SPEC,

    "source_test_evaluation": {
        "directory":
            str(
                HB2025_V61_TEST_ATTEMPT_DIR
            ),

        "audit_path":
            str(
                CELL50_AUDIT_PATH
            ),

        "audit_sha256":
            (
                sha256_file(
                    CELL50_AUDIT_PATH
                )
                if CELL50_AUDIT_PATH.exists()
                else None
            ),
    },

    "artifacts": {
        "fold_summary": {
            "path":
                str(
                    FOLD_SUMMARY_PATH
                ),

            "sha256":
                sha256_file(
                    FOLD_SUMMARY_PATH
                ),
        },
    },
}


atomic_json_write(
    EVIDENCE_SPEC,
    EVIDENCE_JSON_PATH,
)


EVIDENCE_SHA256 = sha256_file(
    EVIDENCE_JSON_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_EVIDENCE_POINTER = (
    EVIDENCE_ROOT
    /
    "LATEST_FOLD_EVIDENCE.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            "EVIDENCE_FROZEN",

        "evidence_directory":
            str(
                EVIDENCE_DIR
            ),

        "evidence_json":
            str(
                EVIDENCE_JSON_PATH
            ),

        "evidence_sha256":
            EVIDENCE_SHA256,

        "fold_summary":
            str(
                FOLD_SUMMARY_PATH
            ),

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_EVIDENCE_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 12. MAIN MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "fold_evidence"
] = {
    "status":
        "EVIDENCE_FROZEN",

    "attempt_id":
        ATTEMPT_ID,

    "evidence_directory":
        str(
            EVIDENCE_DIR
        ),

    "evidence_json":
        str(
            EVIDENCE_JSON_PATH
        ),

    "evidence_sha256":
        EVIDENCE_SHA256,

    "fold_summary":
        str(
            FOLD_SUMMARY_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2025_HISTORICAL_EVIDENCE_FROZEN"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 13. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2025_FOLD_SUMMARY = (
    FOLD_SUMMARY
)


HB2025_FOLD_EVIDENCE = (
    EVIDENCE_SPEC
)


HB2025_FOLD_EVIDENCE_PATH = (
    EVIDENCE_JSON_PATH
)


NOTEBOOK_2_CELL_51_PASS = True


# --------------------------------------------------------------------------------------------------
# 14. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 165)
print("HB_2025 — HISTORICAL EVIDENCE FREEZE")
print("=" * 165)


print()
print("CROSS-FOLD SUMMARY ROW")
print("-" * 165)

display(
    FOLD_SUMMARY
)


print()
print("EVIDENCE FLAGS")
print("-" * 165)

display(
    pd.DataFrame(
        [
            {
                "evidence":
                    key,

                "value":
                    value,
            }

            for key, value
            in EVIDENCE_FLAGS.items()
        ]
    )
)


print()
print("AUDIT / VERSIONING")
print("-" * 165)

print(
    f"Attempt ID                     : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable evidence directory   : "
    f"{EVIDENCE_DIR}"
)

print(
    f"Evidence JSON                  : "
    f"{EVIDENCE_JSON_PATH}"
)

print(
    f"Evidence SHA256                : "
    f"{EVIDENCE_SHA256}"
)

print(
    f"Latest evidence pointer        : "
    f"{LATEST_EVIDENCE_POINTER}"
)


print()
print("=" * 165)
print(
    "CELL 51 STATUS: HB_2025 HISTORICAL EVIDENCE FROZEN"
)
print("=" * 165)



HB_2025 — ROLLING V4/V5 VALIDATION → TEST FORECAST REPLAY

OVERALL POINT PERFORMANCE
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,split,n,athletes,V4_MAE,V5_MAE,V4_to_V5_MAE_gain_m,V4_RMSE,V5_RMSE,V4_to_V5_RMSE_gain_m,V5_Bias
0,VALIDATION,4226,2258,2.747890,2.583669,0.164221,3.700416,3.510531,0.189885,0.050296
1,TEST,11342,4076,2.924033,2.559880,0.364154,3.908156,3.493864,0.414292,0.150106



POINT METRICS
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,model,n,athletes,MAE,RMSE,Bias,split
0,V4,4226,2258,2.747890,3.700416,0.063656,VALIDATION
1,V5,4226,2258,2.583669,3.510531,0.050296,VALIDATION
2,V4,11342,4076,2.924033,3.908156,0.330888,TEST
3,V5,11342,4076,2.559880,3.493864,0.150106,TEST



BY STATE GAP
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


...


AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T141128_576288Z
Immutable attempt directory    : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v61_test_evaluation/20260922T141128_576288Z
Audit manifest                 : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/v61_test_evaluation/20260922T141128_576288Z/v61_test_evaluation_audit.json
Audit SHA256                   : 3f754c51cc35788e8a3c9ff13615f9d8db97d4b40dfe5745809bd65a9e100c99
Git commit                     : a7cb73d264fa6eb35402298c3bf7f921b1c8641e

QA
----------------------------------------------------------------------------------------------------------------------------

,fold,test_start,test_end,test_rows,test_athletes,v5_tau_days,v6_persistence_model,v6_offset_model,V5_MAE,V6_MAE,...,V61_RMSE,V5_Bias,V6_Bias,V61_Bias,fresh_V5_to_V6_MAE_gain_m,fresh_V5_to_V6_CI_low,fresh_V5_to_V6_CI_high,fresh_V5_to_V61_MAE_gain_m,fresh_V5_to_V61_CI_low,fresh_V5_to_V61_CI_high
0,HB_2025,2025-05-01,2025-08-30,11342,4076,95.47821,M0_CONST_CONST,GROUP,2.55988,2.547046,...,3.476288,0.150106,0.212634,0.119142,-0.001373,-0.013748,0.011752,0.004517,0.002655,0.006427



EVIDENCE FLAGS
---------------------------------------------------------------------------------------------------------------------------------------------------------------------


,evidence,value
0,v6_overall_mae_better_than_v5,True
1,v6_overall_mae_gain_ci_above_zero,True
2,v61_overall_mae_better_than_v5,True
3,v61_overall_mae_gain_ci_above_zero,True
4,v61_overall_mae_better_than_v6,False
5,v61_vs_v6_mae_ci_above_zero,False
6,v6_fresh_mae_better_than_v5,False
7,v6_fresh_mae_gain_ci_above_zero,False
8,v61_fresh_mae_better_than_v5,True
9,v61_fresh_mae_gain_ci_above_zero,True



AUDIT / VERSIONING
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                     : 20260922T141131_122368Z
Immutable evidence directory   : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/fold_evidence/20260922T141131_122368Z
Evidence JSON                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/fold_evidence/20260922T141131_122368Z/fold_evidence.json
Evidence SHA256                : cafb08f284a820173b519e7124796d195449df295ffc9177cc73a9540c632c82
Latest evidence pointer        : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/historical_backtest/HB_2025/fold_evidence/LATEST_FOLD_EVIDENCE.json

CELL 51 STATUS: 

## Stage 17 — Cross-fold synthesis and mean architecture decision

Diagnose/repair the historical evidence schema where required, synthesize HB_2023/HB_2024/HB_2025, and freeze the selected V6.1 mean architecture.

**Legacy source cells represented:** 42, 43, 45  
**Execution rule:** run only after the preceding stage has completed successfully.


In [18]:
# ==================================================================================================
# REFACTORED STAGE 17 — Cross-fold synthesis and mean architecture decision
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 42
# Original source SHA256: 8ac1709a43eaca852d4a67766eaa63feccbcb678d5182f3fce41e674047d2cb7
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 52A — DIAGNOSE CROSS-FOLD ARCHITECTURE DECISION GATE
# ==================================================================================================
#
# Purpose
# -------
# Diagnose why Cell 52 raised:
#
#   RuntimeError:
#   Historical evidence does not support the frozen V6.1 architecture decision...
#
# This cell:
#   • fits NOTHING
#   • changes NO architecture
#   • changes NO thresholds
#   • writes NOTHING
#
# It reconstructs the decision evidence directly from the three immutable
# fold-summary artifacts and prints the exact boolean causing the failure.
#
# ==================================================================================================

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "TRAINING_MANIFEST",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


FOLDS = [
    "HB_2023",
    "HB_2024",
    "HB_2025",
]


# --------------------------------------------------------------------------------------------------
# 2. LOAD IMMUTABLE FOLD SUMMARIES DIRECTLY
# --------------------------------------------------------------------------------------------------

frames = []


for fold in FOLDS:

    entry = (
        TRAINING_MANIFEST[
            "historical_backtest"
        ][
            fold
        ][
            "fold_evidence"
        ]
    )


    path = Path(
        entry[
            "fold_summary"
        ]
    )


    if not path.exists():

        raise RuntimeError(
            f"Missing frozen fold summary for {fold}: {path}"
        )


    frame = pd.read_parquet(
        path
    )


    if len(frame) != 1:

        raise RuntimeError(
            f"{fold} fold summary has {len(frame)} rows; expected exactly 1."
        )


    frames.append(
        frame
    )


CF = (
    pd.concat(
        frames,
        ignore_index=True,
    )
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 3. FORCE DECISION COLUMNS TO NUMERIC
# --------------------------------------------------------------------------------------------------

decision_numeric_columns = [
    "V5_to_V6_MAE_gain_m",
    "V5_to_V6_CI_low",
    "V5_to_V6_CI_high",

    "V5_to_V61_MAE_gain_m",
    "V5_to_V61_CI_low",
    "V5_to_V61_CI_high",

    "fresh_V5_to_V6_MAE_gain_m",
    "fresh_V5_to_V6_CI_low",
    "fresh_V5_to_V6_CI_high",

    "fresh_V5_to_V61_MAE_gain_m",
    "fresh_V5_to_V61_CI_low",
    "fresh_V5_to_V61_CI_high",
]


for column in decision_numeric_columns:

    if column not in CF.columns:

        raise RuntimeError(
            f"Required decision column missing: {column}"
        )


    CF[
        column
    ] = pd.to_numeric(
        CF[
            column
        ],
        errors="raise",
    )


# --------------------------------------------------------------------------------------------------
# 4. SHOW EXACT VALUES USED BY THE GATE
# --------------------------------------------------------------------------------------------------

DECISION_INPUT = (
    CF[
        [
            "fold",

            "V5_to_V6_MAE_gain_m",
            "V5_to_V6_CI_low",
            "V5_to_V6_CI_high",

            "V5_to_V61_MAE_gain_m",
            "V5_to_V61_CI_low",
            "V5_to_V61_CI_high",

            "fresh_V5_to_V6_MAE_gain_m",
            "fresh_V5_to_V6_CI_low",
            "fresh_V5_to_V6_CI_high",

            "fresh_V5_to_V61_MAE_gain_m",
            "fresh_V5_to_V61_CI_low",
            "fresh_V5_to_V61_CI_high",
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 5. RECONSTRUCT EACH BOOLEAN EXPLICITLY
# --------------------------------------------------------------------------------------------------

n_folds = len(
    CF
)


v61_overall_positive_count = int(
    (
        CF[
            "V5_to_V61_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


v61_overall_ci_positive_count = int(
    (
        CF[
            "V5_to_V61_CI_low"
        ]
        >
        0
    )
    .sum()
)


v61_fresh_positive_count = int(
    (
        CF[
            "fresh_V5_to_V61_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


v61_fresh_ci_positive_count = int(
    (
        CF[
            "fresh_V5_to_V61_CI_low"
        ]
        >
        0
    )
    .sum()
)


v6_fresh_positive_count = int(
    (
        CF[
            "fresh_V5_to_V6_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


v6_fresh_ci_positive_count = int(
    (
        CF[
            "fresh_V5_to_V6_CI_low"
        ]
        >
        0
    )
    .sum()
)


v6_fresh_significant_harm_count = int(
    (
        CF[
            "fresh_V5_to_V6_CI_high"
        ]
        <
        0
    )
    .sum()
)


DIAGNOSTIC_FLAGS = {

    "v61_overall_gain_positive_all_folds":
        bool(
            v61_overall_positive_count
            ==
            n_folds
        ),

    "v61_overall_ci_positive_at_least_two_folds":
        bool(
            v61_overall_ci_positive_count
            >=
            2
        ),

    "v61_fresh_gain_positive_all_folds":
        bool(
            v61_fresh_positive_count
            ==
            n_folds
        ),

    "v61_fresh_ci_positive_all_folds":
        bool(
            v61_fresh_ci_positive_count
            ==
            n_folds
        ),

    "full_v6_fresh_effect_not_consistently_positive":
        bool(
            v6_fresh_positive_count
            <
            n_folds
        ),

    "full_v6_has_significant_fresh_harm_in_at_least_one_fold":
        bool(
            v6_fresh_significant_harm_count
            >=
            1
        ),

    "v61_adds_no_fitted_blend_parameter":
        True,

    "v61_retains_stale_state_v6_behavior":
        True,
}


DECISION_DIAGNOSTIC = pd.DataFrame(
    [
        {
            "criterion":
                key,

            "value":
                value,
        }

        for key, value
        in DIAGNOSTIC_FLAGS.items()
    ]
)


# --------------------------------------------------------------------------------------------------
# 6. EXPECTED COUNTS
# --------------------------------------------------------------------------------------------------

COUNT_DIAGNOSTIC = pd.DataFrame(
    [
        {
            "quantity":
                "historical_folds",

            "observed":
                n_folds,

            "expected":
                3,
        },

        {
            "quantity":
                "V6.1 overall positive folds",

            "observed":
                v61_overall_positive_count,

            "expected":
                3,
        },

        {
            "quantity":
                "V6.1 overall CI-positive folds",

            "observed":
                v61_overall_ci_positive_count,

            "expected":
                2,
        },

        {
            "quantity":
                "V6.1 fresh positive folds",

            "observed":
                v61_fresh_positive_count,

            "expected":
                3,
        },

        {
            "quantity":
                "V6.1 fresh CI-positive folds",

            "observed":
                v61_fresh_ci_positive_count,

            "expected":
                3,
        },

        {
            "quantity":
                "Full V6 fresh positive folds",

            "observed":
                v6_fresh_positive_count,

            "expected":
                1,
        },

        {
            "quantity":
                "Full V6 fresh CI-positive folds",

            "observed":
                v6_fresh_ci_positive_count,

            "expected":
                1,
        },

        {
            "quantity":
                "Full V6 fresh significantly harmful folds",

            "observed":
                v6_fresh_significant_harm_count,

            "expected":
                1,
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 7. FINAL DIAGNOSTIC RESULT
# --------------------------------------------------------------------------------------------------

ALL_EXPECTED_FLAGS_PASS = bool(
    all(
        DIAGNOSTIC_FLAGS.values()
    )
)


FAILED_FLAGS = [
    key
    for key, value
    in DIAGNOSTIC_FLAGS.items()
    if not value
]


# --------------------------------------------------------------------------------------------------
# 8. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 170)
print("CELL 52A — CROSS-FOLD ARCHITECTURE DECISION DIAGNOSTIC")
print("=" * 170)


print()
print("EXACT FROZEN VALUES USED BY THE DECISION")
print("-" * 170)

display(
    DECISION_INPUT
)


print()
print("REPLICATION COUNTS")
print("-" * 170)

display(
    COUNT_DIAGNOSTIC
)


print()
print("DECISION FLAGS")
print("-" * 170)

display(
    DECISION_DIAGNOSTIC
)


print()
print("FAILED FLAGS")
print("-" * 170)

if FAILED_FLAGS:

    for flag in FAILED_FLAGS:

        print(
            f"  - {flag}"
        )

else:

    print(
        "None."
    )


print()
print("DIAGNOSTIC RESULT")
print("-" * 170)

print(
    f"All expected architecture criteria pass : "
    f"{ALL_EXPECTED_FLAGS_PASS}"
)


print()
print("=" * 170)

if ALL_EXPECTED_FLAGS_PASS:

    print(
        "CELL 52A STATUS: EVIDENCE SUPPORTS THE FROZEN V6.1 DECISION — "
        "CELL 52 GATE IMPLEMENTATION REQUIRES CORRECTION"
    )

else:

    print(
        "CELL 52A STATUS: ONE OR MORE FROZEN SUMMARY VALUES DIFFER FROM "
        "THE EXPECTED HISTORICAL EVIDENCE"
    )

print("=" * 170)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 43
# Original source SHA256: a972eb84214d75393a98c5cf1d8454e009d33a77012ecbf5cf451ec6315e08ba
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 52B — HB_2023 FROZEN EVIDENCE SCHEMA REPAIR
# ==================================================================================================
#
# Purpose
# -------
# HB_2023 was frozen before the later fold-summary schema included all bootstrap CI fields.
#
# The original HB_2023 TEST evaluation is valid and remains immutable.
# This cell:
#
#   1. reads the ORIGINAL frozen HB_2023 TEST artifacts
#   2. reconstructs the complete fold-summary schema
#   3. writes a NEW immutable fold-evidence version
#   4. records provenance back to the superseded evidence package
#   5. updates only the HB_2023 fold-evidence pointer in the manifest
#
# NO model fitting.
# NO recalculation of predictions.
# NO bootstrap rerun.
# NO architecture criteria changed.
# NO original evidence overwritten.
#
# ==================================================================================================

import hashlib
import json
import os

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


FOLD_NAME = "HB_2023"

CELL_ID = "CELL_52B_HB2023_EVIDENCE_SCHEMA_REPAIR"

EVIDENCE_SCHEMA_VERSION = "1.1.0"


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def locate_artifact(
    directory,
    exact_name,
    fallback_patterns,
):

    directory = Path(
        directory
    )

    exact = (
        directory
        /
        exact_name
    )

    if exact.exists():

        return exact


    matches = []


    for pattern in fallback_patterns:

        matches.extend(
            directory.glob(
                pattern
            )
        )


    matches = sorted(
        set(
            matches
        )
    )


    if len(matches) != 1:

        raise RuntimeError(
            f"Could not uniquely locate {exact_name} in {directory}.\n"
            f"Matches: {[str(x) for x in matches]}"
        )


    return matches[0]


def pairwise_row(
    df,
    baseline,
    candidate,
):

    row = df.loc[
        df[
            "baseline"
        ].eq(
            baseline
        )
        &
        df[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely locate pairwise row "
            f"{baseline} -> {candidate}."
        )


    return row.iloc[0]


def bootstrap_row(
    df,
    horizon,
    baseline,
    candidate,
):

    row = df.loc[
        df[
            "horizon"
        ].eq(
            horizon
        )
        &
        df[
            "baseline"
        ].eq(
            baseline
        )
        &
        df[
            "candidate"
        ].eq(
            candidate
        )
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"Could not uniquely locate bootstrap row "
            f"{horizon}: {baseline} -> {candidate}."
        )


    return row.iloc[0]


# --------------------------------------------------------------------------------------------------
# 3. LOCATE ORIGINAL HB_2023 TEST EVALUATION
# --------------------------------------------------------------------------------------------------

HB2023_ROOT = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    FOLD_NAME
)


test_entry = (
    TRAINING_MANIFEST[
        "historical_backtest"
    ][
        FOLD_NAME
    ].get(
        "v61_test_evaluation",
        {},
    )
)


TEST_EVAL_DIR = None


candidate_dir = (
    test_entry.get(
        "attempt_directory"
    )
)


if candidate_dir is not None:

    candidate_dir = Path(
        candidate_dir
    )

    if candidate_dir.exists():

        TEST_EVAL_DIR = (
            candidate_dir
        )


if TEST_EVAL_DIR is None:

    pointer_path = (
        HB2023_ROOT
        /
        "v61_test_evaluation"
        /
        "LATEST_V61_TEST_EVALUATION.json"
    )


    if not pointer_path.exists():

        raise RuntimeError(
            "Could not locate HB_2023 V6.1 TEST evaluation."
        )


    with pointer_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        pointer = json.load(
            f
        )


    TEST_EVAL_DIR = Path(
        pointer[
            "attempt_directory"
        ]
    )


if not TEST_EVAL_DIR.exists():

    raise RuntimeError(
        f"HB_2023 TEST evaluation directory missing: "
        f"{TEST_EVAL_DIR}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOCATE ORIGINAL TEST ARTIFACTS
# --------------------------------------------------------------------------------------------------

OVERALL_PATH = locate_artifact(
    TEST_EVAL_DIR,
    "overall_metrics.parquet",
    [
        "*overall*metrics*.parquet",
    ],
)


PAIRWISE_PATH = locate_artifact(
    TEST_EVAL_DIR,
    "pairwise_overall_gains.parquet",
    [
        "*pairwise*overall*.parquet",
        "*overall*gain*.parquet",
    ],
)


BOOTSTRAP_PATH = locate_artifact(
    TEST_EVAL_DIR,
    "athlete_cluster_bootstrap.parquet",
    [
        "*bootstrap*.parquet",
    ],
)


TEST_AUDIT_PATH = locate_artifact(
    TEST_EVAL_DIR,
    "v61_test_evaluation_audit.json",
    [
        "*test*evaluation*audit*.json",
        "*audit*.json",
    ],
)


# --------------------------------------------------------------------------------------------------
# 5. READ ORIGINAL TEST EVIDENCE
# --------------------------------------------------------------------------------------------------

OVERALL = pd.read_parquet(
    OVERALL_PATH
)


PAIRWISE = pd.read_parquet(
    PAIRWISE_PATH
)


BOOTSTRAP = pd.read_parquet(
    BOOTSTRAP_PATH
)


# --------------------------------------------------------------------------------------------------
# 6. READ SUPERSEDED HB_2023 FOLD EVIDENCE
# --------------------------------------------------------------------------------------------------

OLD_EVIDENCE_ENTRY = (
    TRAINING_MANIFEST[
        "historical_backtest"
    ][
        FOLD_NAME
    ][
        "fold_evidence"
    ]
)


OLD_SUMMARY_PATH = Path(
    OLD_EVIDENCE_ENTRY[
        "fold_summary"
    ]
)


OLD_EVIDENCE_JSON_PATH = Path(
    OLD_EVIDENCE_ENTRY[
        "evidence_json"
    ]
)


if not OLD_SUMMARY_PATH.exists():

    raise RuntimeError(
        f"Old HB_2023 fold summary missing: "
        f"{OLD_SUMMARY_PATH}"
    )


if not OLD_EVIDENCE_JSON_PATH.exists():

    raise RuntimeError(
        f"Old HB_2023 evidence JSON missing: "
        f"{OLD_EVIDENCE_JSON_PATH}"
    )


OLD_SUMMARY = pd.read_parquet(
    OLD_SUMMARY_PATH
)


if len(
    OLD_SUMMARY
) != 1:

    raise RuntimeError(
        "Expected exactly one HB_2023 fold-summary row."
    )


OLD_ROW = (
    OLD_SUMMARY.iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 7. EXTRACT ORIGINAL TEST RESULTS
# --------------------------------------------------------------------------------------------------

O = (
    OVERALL.set_index(
        "model"
    )
)


V5_TO_V6 = pairwise_row(
    PAIRWISE,
    "V5",
    "V6",
)


V5_TO_V61 = pairwise_row(
    PAIRWISE,
    "V5",
    "V6.1",
)


V6_TO_V61 = pairwise_row(
    PAIRWISE,
    "V6",
    "V6.1",
)


BOOT_ALL_V5_V6 = bootstrap_row(
    BOOTSTRAP,
    "ALL",
    "V5",
    "V6",
)


BOOT_ALL_V5_V61 = bootstrap_row(
    BOOTSTRAP,
    "ALL",
    "V5",
    "V6.1",
)


BOOT_ALL_V6_V61 = bootstrap_row(
    BOOTSTRAP,
    "ALL",
    "V6",
    "V6.1",
)


BOOT_FRESH_V5_V6 = bootstrap_row(
    BOOTSTRAP,
    "01-30",
    "V5",
    "V6",
)


BOOT_FRESH_V5_V61 = bootstrap_row(
    BOOTSTRAP,
    "01-30",
    "V5",
    "V6.1",
)


# --------------------------------------------------------------------------------------------------
# 8. RECOVER NON-TEST METADATA FROM ORIGINAL FOLD SUMMARY
# --------------------------------------------------------------------------------------------------

def old_value(
    name,
):

    if name not in OLD_SUMMARY.columns:

        raise RuntimeError(
            f"Old HB_2023 summary missing required metadata: {name}"
        )

    return OLD_ROW[
        name
    ]


# --------------------------------------------------------------------------------------------------
# 9. CONSTRUCT COMPLETE HB_2023 SUMMARY USING SAME SCHEMA AS HB_2024 / HB_2025
# --------------------------------------------------------------------------------------------------

REPAIRED_SUMMARY = pd.DataFrame(
    [
        {
            "fold":
                FOLD_NAME,

            "test_start":
                pd.Timestamp(
                    old_value(
                        "test_start"
                    )
                ),

            "test_end":
                pd.Timestamp(
                    old_value(
                        "test_end"
                    )
                ),

            "test_rows":
                int(
                    O.loc[
                        "V5",
                        "n",
                    ]
                ),

            "test_athletes":
                int(
                    O.loc[
                        "V5",
                        "athletes",
                    ]
                ),

            "v5_tau_days":
                float(
                    old_value(
                        "v5_tau_days"
                    )
                ),

            "v6_persistence_model":
                str(
                    old_value(
                        "v6_persistence_model"
                    )
                ),

            "v6_offset_model":
                str(
                    old_value(
                        "v6_offset_model"
                    )
                ),

            "V5_MAE":
                float(
                    O.loc[
                        "V5",
                        "MAE",
                    ]
                ),

            "V6_MAE":
                float(
                    O.loc[
                        "V6",
                        "MAE",
                    ]
                ),

            "V61_MAE":
                float(
                    O.loc[
                        "V6.1",
                        "MAE",
                    ]
                ),

            "V5_to_V6_MAE_gain_m":
                float(
                    V5_TO_V6[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V6_CI_low":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V6_CI_high":
                float(
                    BOOT_ALL_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "V5_to_V61_MAE_gain_m":
                float(
                    V5_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_to_V61_CI_low":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "V5_to_V61_CI_high":
                float(
                    BOOT_ALL_V5_V61[
                        "MAE_CI_high"
                    ]
                ),

            "V6_to_V61_MAE_gain_m":
                float(
                    V6_TO_V61[
                        "MAE_gain_m"
                    ]
                ),

            "V5_RMSE":
                float(
                    O.loc[
                        "V5",
                        "RMSE",
                    ]
                ),

            "V6_RMSE":
                float(
                    O.loc[
                        "V6",
                        "RMSE",
                    ]
                ),

            "V61_RMSE":
                float(
                    O.loc[
                        "V6.1",
                        "RMSE",
                    ]
                ),

            "V5_Bias":
                float(
                    O.loc[
                        "V5",
                        "Bias",
                    ]
                ),

            "V6_Bias":
                float(
                    O.loc[
                        "V6",
                        "Bias",
                    ]
                ),

            "V61_Bias":
                float(
                    O.loc[
                        "V6.1",
                        "Bias",
                    ]
                ),

            "fresh_V5_to_V6_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V6_CI_low":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V6_CI_high":
                float(
                    BOOT_FRESH_V5_V6[
                        "MAE_CI_high"
                    ]
                ),

            "fresh_V5_to_V61_MAE_gain_m":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_gain_m"
                    ]
                ),

            "fresh_V5_to_V61_CI_low":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_low"
                    ]
                ),

            "fresh_V5_to_V61_CI_high":
                float(
                    BOOT_FRESH_V5_V61[
                        "MAE_CI_high"
                    ]
                ),
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. SCHEMA / VALUE QA BEFORE WRITING
# --------------------------------------------------------------------------------------------------

REQUIRED_DECISION_COLUMNS = [
    "V5_to_V6_MAE_gain_m",
    "V5_to_V6_CI_low",
    "V5_to_V6_CI_high",

    "V5_to_V61_MAE_gain_m",
    "V5_to_V61_CI_low",
    "V5_to_V61_CI_high",

    "fresh_V5_to_V6_MAE_gain_m",
    "fresh_V5_to_V6_CI_low",
    "fresh_V5_to_V6_CI_high",

    "fresh_V5_to_V61_MAE_gain_m",
    "fresh_V5_to_V61_CI_low",
    "fresh_V5_to_V61_CI_high",
]


NO_MISSING_DECISION_VALUES_PASS = bool(
    np.isfinite(
        REPAIRED_SUMMARY[
            REQUIRED_DECISION_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


# Direct consistency with original pairwise/bootstrap artifacts.
PAIRWISE_RECONCILIATION_PASS = bool(

    np.isclose(
        REPAIRED_SUMMARY.loc[
            0,
            "V5_to_V6_MAE_gain_m",
        ],
        BOOT_ALL_V5_V6[
            "MAE_gain_m"
        ],
    )

    and

    np.isclose(
        REPAIRED_SUMMARY.loc[
            0,
            "V5_to_V61_MAE_gain_m",
        ],
        BOOT_ALL_V5_V61[
            "MAE_gain_m"
        ],
    )

    and

    np.isclose(
        REPAIRED_SUMMARY.loc[
            0,
            "fresh_V5_to_V6_MAE_gain_m",
        ],
        BOOT_FRESH_V5_V6[
            "MAE_gain_m"
        ],
    )

    and

    np.isclose(
        REPAIRED_SUMMARY.loc[
            0,
            "fresh_V5_to_V61_MAE_gain_m",
        ],
        BOOT_FRESH_V5_V61[
            "MAE_gain_m"
        ],
    )
)


EXPECTED_HB2023_EVIDENCE_PASS = bool(

    # V6.1 overall improvement CI > 0
    REPAIRED_SUMMARY.loc[
        0,
        "V5_to_V61_CI_low",
    ]
    >
    0

    and

    # Full V6 was significantly harmful at 1–30 days
    REPAIRED_SUMMARY.loc[
        0,
        "fresh_V5_to_V6_CI_high",
    ]
    <
    0

    and

    # V6.1 protected the fresh horizon
    REPAIRED_SUMMARY.loc[
        0,
        "fresh_V5_to_V61_CI_low",
    ]
    >
    0
)


REPAIR_PASS = bool(
    NO_MISSING_DECISION_VALUES_PASS
    and
    PAIRWISE_RECONCILIATION_PASS
    and
    EXPECTED_HB2023_EVIDENCE_PASS
)


if not REPAIR_PASS:

    raise RuntimeError(
        "HB_2023 source evidence does not reconcile. "
        "No repaired evidence package was written."
    )


# --------------------------------------------------------------------------------------------------
# 11. CREATE NEW IMMUTABLE EVIDENCE VERSION
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


EVIDENCE_ROOT = (
    HB2023_ROOT
    /
    "fold_evidence"
)


REPAIR_DIR = (
    EVIDENCE_ROOT
    /
    ATTEMPT_ID
)


REPAIR_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


REPAIRED_SUMMARY_PATH = (
    REPAIR_DIR
    /
    "fold_summary.parquet"
)


REPAIRED_EVIDENCE_PATH = (
    REPAIR_DIR
    /
    "fold_evidence.json"
)


REPAIRED_SUMMARY.to_parquet(
    REPAIRED_SUMMARY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 12. NEW EVIDENCE MANIFEST
# --------------------------------------------------------------------------------------------------

REPAIRED_EVIDENCE = {
    "evidence_schema_version":
        EVIDENCE_SCHEMA_VERSION,

    "fold":
        FOLD_NAME,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        "EVIDENCE_FROZEN",

    "evidence_type":
        "SCHEMA_REPAIR",

    "model_recomputed":
        False,

    "predictions_recomputed":
        False,

    "bootstrap_recomputed":
        False,

    "architecture_changed":
        False,

    "decision_thresholds_changed":
        False,

    "repair_reason":
        (
            "The original HB_2023 fold evidence was created under an "
            "earlier summary schema that omitted several bootstrap confidence-"
            "interval fields required by the later three-fold architecture "
            "synthesis. Values are copied directly from the original immutable "
            "HB_2023 untouched-TEST pairwise/bootstrap artifacts."
        ),

    "superseded_evidence": {
        "summary_path":
            str(
                OLD_SUMMARY_PATH
            ),

        "summary_sha256":
            sha256_file(
                OLD_SUMMARY_PATH
            ),

        "evidence_json_path":
            str(
                OLD_EVIDENCE_JSON_PATH
            ),

        "evidence_json_sha256":
            sha256_file(
                OLD_EVIDENCE_JSON_PATH
            ),

        "attempt_id":
            OLD_EVIDENCE_ENTRY.get(
                "attempt_id"
            ),
    },

    "source_test_evaluation": {
        "attempt_directory":
            str(
                TEST_EVAL_DIR
            ),

        "overall_metrics": {
            "path":
                str(
                    OVERALL_PATH
                ),

            "sha256":
                sha256_file(
                    OVERALL_PATH
                ),
        },

        "pairwise_overall": {
            "path":
                str(
                    PAIRWISE_PATH
                ),

            "sha256":
                sha256_file(
                    PAIRWISE_PATH
                ),
        },

        "athlete_cluster_bootstrap": {
            "path":
                str(
                    BOOTSTRAP_PATH
                ),

            "sha256":
                sha256_file(
                    BOOTSTRAP_PATH
                ),
        },

        "test_evaluation_audit": {
            "path":
                str(
                    TEST_AUDIT_PATH
                ),

            "sha256":
                sha256_file(
                    TEST_AUDIT_PATH
                ),
        },
    },

    "repaired_decision_fields": {
        column:
            float(
                REPAIRED_SUMMARY.loc[
                    0,
                    column,
                ]
            )

        for column
        in REQUIRED_DECISION_COLUMNS
    },

    "qa": {
        "no_missing_decision_values":
            NO_MISSING_DECISION_VALUES_PASS,

        "pairwise_bootstrap_reconciliation":
            PAIRWISE_RECONCILIATION_PASS,

        "expected_HB2023_evidence":
            EXPECTED_HB2023_EVIDENCE_PASS,

        "repair_pass":
            REPAIR_PASS,
    },

    "artifacts": {
        "fold_summary": {
            "path":
                str(
                    REPAIRED_SUMMARY_PATH
                ),

            "sha256":
                sha256_file(
                    REPAIRED_SUMMARY_PATH
                ),
        },
    },
}


atomic_json_write(
    REPAIRED_EVIDENCE,
    REPAIRED_EVIDENCE_PATH,
)


REPAIRED_EVIDENCE_SHA256 = sha256_file(
    REPAIRED_EVIDENCE_PATH
)


# --------------------------------------------------------------------------------------------------
# 13. UPDATE LATEST EVIDENCE POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    EVIDENCE_ROOT
    /
    "LATEST_FOLD_EVIDENCE.json"
)


atomic_json_write(
    {
        "fold":
            FOLD_NAME,

        "attempt_id":
            ATTEMPT_ID,

        "status":
            "EVIDENCE_FROZEN",

        "evidence_schema_version":
            EVIDENCE_SCHEMA_VERSION,

        "evidence_type":
            "SCHEMA_REPAIR",

        "evidence_directory":
            str(
                REPAIR_DIR
            ),

        "evidence_json":
            str(
                REPAIRED_EVIDENCE_PATH
            ),

        "evidence_sha256":
            REPAIRED_EVIDENCE_SHA256,

        "fold_summary":
            str(
                REPAIRED_SUMMARY_PATH
            ),

        "supersedes":
            str(
                OLD_EVIDENCE_JSON_PATH
            ),

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 14. UPDATE TRAINING MANIFEST — POINTER ONLY
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    FOLD_NAME
][
    "fold_evidence"
] = {
    "status":
        "EVIDENCE_FROZEN",

    "attempt_id":
        ATTEMPT_ID,

    "evidence_schema_version":
        EVIDENCE_SCHEMA_VERSION,

    "evidence_type":
        "SCHEMA_REPAIR",

    "evidence_directory":
        str(
            REPAIR_DIR
        ),

    "evidence_json":
        str(
            REPAIRED_EVIDENCE_PATH
        ),

    "evidence_sha256":
        REPAIRED_EVIDENCE_SHA256,

    "fold_summary":
        str(
            REPAIRED_SUMMARY_PATH
        ),

    "supersedes_evidence_json":
        str(
            OLD_EVIDENCE_JSON_PATH
        ),

    "supersedes_evidence_sha256":
        sha256_file(
            OLD_EVIDENCE_JSON_PATH
        ),
}


TRAINING_MANIFEST[
    "status"
] = (
    "HB_2023_EVIDENCE_SCHEMA_REPAIRED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 15. GLOBALS
# --------------------------------------------------------------------------------------------------

HB2023_FOLD_SUMMARY = (
    REPAIRED_SUMMARY
)


HB2023_FOLD_EVIDENCE = (
    REPAIRED_EVIDENCE
)


HB2023_FOLD_EVIDENCE_PATH = (
    REPAIRED_EVIDENCE_PATH
)


NOTEBOOK_2_CELL_52B_PASS = True


# --------------------------------------------------------------------------------------------------
# 16. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 175)
print("HB_2023 — IMMUTABLE FOLD-EVIDENCE SCHEMA REPAIR")
print("=" * 175)


print()
print("REPAIRED HB_2023 DECISION VALUES")
print("-" * 175)

display(
    REPAIRED_SUMMARY[
        [
            "fold",

            "V5_to_V6_MAE_gain_m",
            "V5_to_V6_CI_low",
            "V5_to_V6_CI_high",

            "V5_to_V61_MAE_gain_m",
            "V5_to_V61_CI_low",
            "V5_to_V61_CI_high",

            "fresh_V5_to_V6_MAE_gain_m",
            "fresh_V5_to_V6_CI_low",
            "fresh_V5_to_V6_CI_high",

            "fresh_V5_to_V61_MAE_gain_m",
            "fresh_V5_to_V61_CI_low",
            "fresh_V5_to_V61_CI_high",
        ]
    ]
)


print()
print("SCHEMA REPAIR QA")
print("-" * 175)

print(
    f"No missing decision values       : "
    f"{NO_MISSING_DECISION_VALUES_PASS}"
)

print(
    f"Original artifacts reconcile     : "
    f"{PAIRWISE_RECONCILIATION_PASS}"
)

print(
    f"Expected HB_2023 evidence signs  : "
    f"{EXPECTED_HB2023_EVIDENCE_PASS}"
)


print()
print("VERSIONING")
print("-" * 175)

print(
    f"Superseded summary               : "
    f"{OLD_SUMMARY_PATH}"
)

print(
    f"Superseded evidence JSON         : "
    f"{OLD_EVIDENCE_JSON_PATH}"
)

print(
    f"New immutable evidence directory : "
    f"{REPAIR_DIR}"
)

print(
    f"New fold summary                 : "
    f"{REPAIRED_SUMMARY_PATH}"
)

print(
    f"New evidence JSON                : "
    f"{REPAIRED_EVIDENCE_PATH}"
)

print(
    f"New evidence SHA256              : "
    f"{REPAIRED_EVIDENCE_SHA256}"
)


print()
print("=" * 175)
print(
    "CELL 52B STATUS: HB_2023 EVIDENCE SCHEMA REPAIRED — "
    "ORIGINAL EVIDENCE PRESERVED"
)
print("=" * 175)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 45
# Original source SHA256: 5f53a62f9afe3f9e78f278d9fb680258b9a5bc09075ea64845501fc997f34658
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 52 — FINAL CROSS-FOLD HISTORICAL SYNTHESIS + MEAN ARCHITECTURE DECISION
# ==================================================================================================
#
# Purpose
# -------
# Synthesize the THREE frozen historical evidence packages:
#
#       HB_2023
#       HB_2024
#       HB_2025
#
# and freeze the mean-architecture decision.
#
# IMPORTANT
# ---------
# • NO fitting occurs here.
# • NO predictions are recomputed.
# • NO bootstrap is rerun.
# • NO architecture thresholds are changed.
# • 2026 TEST is excluded from architecture selection because V6.1 was conceived
#   after inspecting the 2026 TEST behavior.
#
# HB_2023 note
# ------------
# The active HB_2023 fold evidence now points to schema version 1.1.0,
# reconstructed immutably from the ORIGINAL untouched-TEST artifacts.
# The superseded evidence remains preserved.
#
# Decision scope
# --------------
# Select the MEAN architecture that proceeds to:
#
#       FINAL UNCERTAINTY CALIBRATION
#
# This cell does NOT promote ACTIVE_MODEL.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. FIXED DECISION DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = (
    "CELL_52_FINAL_CROSS_FOLD_ARCHITECTURE_DECISION"
)

STAGE_SCHEMA_VERSION = "1.1.0"


FOLDS = [
    "HB_2023",
    "HB_2024",
    "HB_2025",
]


SELECTED_ARCHITECTURE = (
    "V6.1_OU_FRESHNESS_BLEND"
)


SELECTED_FORMULA = (
    "V6.1 = V5 + "
    "(1 - exp(-state_gap_days / tau_V5)) "
    "* (V6_FULL - V5)"
)


DECISION_STATUS = (
    "SELECTED_FOR_FINAL_UNCERTAINTY_CALIBRATION"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. CREATE NEW IMMUTABLE SYNTHESIS ATTEMPT
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


SYNTHESIS_ROOT = (
    Path(
        TRAINING_RUN_DIR
    )
    /
    "historical_backtest"
    /
    "cross_fold_synthesis"
)


SYNTHESIS_DIR = (
    SYNTHESIS_ROOT
    /
    ATTEMPT_ID
)


SYNTHESIS_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 5. LOAD ACTIVE FROZEN EVIDENCE FOR ALL THREE FOLDS
# --------------------------------------------------------------------------------------------------

fold_frames = []

PARENT_EVIDENCE = {}


for fold in FOLDS:

    try:

        evidence_entry = (
            TRAINING_MANIFEST[
                "historical_backtest"
            ][
                fold
            ][
                "fold_evidence"
            ]
        )

    except KeyError as exc:

        raise RuntimeError(
            f"Frozen fold evidence missing for {fold}."
        ) from exc


    if evidence_entry.get(
        "status"
    ) != "EVIDENCE_FROZEN":

        raise RuntimeError(
            f"{fold} evidence status is not EVIDENCE_FROZEN."
        )


    summary_path = Path(
        evidence_entry[
            "fold_summary"
        ]
    )


    evidence_json_path = Path(
        evidence_entry[
            "evidence_json"
        ]
    )


    if not summary_path.exists():

        raise RuntimeError(
            f"Missing frozen summary for {fold}: "
            f"{summary_path}"
        )


    if not evidence_json_path.exists():

        raise RuntimeError(
            f"Missing frozen evidence JSON for {fold}: "
            f"{evidence_json_path}"
        )


    summary_sha = sha256_file(
        summary_path
    )


    evidence_sha = sha256_file(
        evidence_json_path
    )


    manifest_evidence_sha = (
        evidence_entry.get(
            "evidence_sha256"
        )
    )


    if (
        manifest_evidence_sha is not None
        and
        manifest_evidence_sha != evidence_sha
    ):

        raise RuntimeError(
            f"Evidence SHA mismatch for {fold}."
        )


    frame = pd.read_parquet(
        summary_path
    )


    if len(frame) != 1:

        raise RuntimeError(
            f"{fold} summary contains "
            f"{len(frame)} rows; expected 1."
        )


    if str(
        frame.iloc[
            0
        ][
            "fold"
        ]
    ) != fold:

        raise RuntimeError(
            f"Fold identity mismatch for {fold}."
        )


    fold_frames.append(
        frame
    )


    PARENT_EVIDENCE[
        fold
    ] = {
        "status":
            evidence_entry.get(
                "status"
            ),

        "attempt_id":
            evidence_entry.get(
                "attempt_id"
            ),

        "evidence_schema_version":
            evidence_entry.get(
                "evidence_schema_version"
            ),

        "evidence_type":
            evidence_entry.get(
                "evidence_type"
            ),

        "summary_path":
            str(
                summary_path
            ),

        "summary_sha256":
            summary_sha,

        "evidence_json_path":
            str(
                evidence_json_path
            ),

        "evidence_json_sha256":
            evidence_sha,

        "supersedes_evidence_json":
            evidence_entry.get(
                "supersedes_evidence_json"
            ),

        "supersedes_evidence_sha256":
            evidence_entry.get(
                "supersedes_evidence_sha256"
            ),
    }


CROSS_FOLD = (
    pd.concat(
        fold_frames,
        ignore_index=True,
    )
    .sort_values(
        "fold"
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 6. REQUIRED SCHEMA
# --------------------------------------------------------------------------------------------------

required_columns = [
    "fold",

    "test_start",
    "test_end",

    "test_rows",
    "test_athletes",

    "v5_tau_days",

    "v6_persistence_model",
    "v6_offset_model",

    "V5_MAE",
    "V6_MAE",
    "V61_MAE",

    "V5_to_V6_MAE_gain_m",
    "V5_to_V6_CI_low",
    "V5_to_V6_CI_high",

    "V5_to_V61_MAE_gain_m",
    "V5_to_V61_CI_low",
    "V5_to_V61_CI_high",

    "V6_to_V61_MAE_gain_m",

    "V5_RMSE",
    "V6_RMSE",
    "V61_RMSE",

    "V5_Bias",
    "V6_Bias",
    "V61_Bias",

    "fresh_V5_to_V6_MAE_gain_m",
    "fresh_V5_to_V6_CI_low",
    "fresh_V5_to_V6_CI_high",

    "fresh_V5_to_V61_MAE_gain_m",
    "fresh_V5_to_V61_CI_low",
    "fresh_V5_to_V61_CI_high",
]


missing_columns = [
    column
    for column in required_columns
    if column not in CROSS_FOLD.columns
]


if missing_columns:

    raise RuntimeError(
        "Cross-fold evidence is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. NUMERIC TYPE / MISSING-VALUE VALIDATION
# --------------------------------------------------------------------------------------------------

numeric_columns = [
    "test_rows",
    "test_athletes",

    "v5_tau_days",

    "V5_MAE",
    "V6_MAE",
    "V61_MAE",

    "V5_to_V6_MAE_gain_m",
    "V5_to_V6_CI_low",
    "V5_to_V6_CI_high",

    "V5_to_V61_MAE_gain_m",
    "V5_to_V61_CI_low",
    "V5_to_V61_CI_high",

    "V6_to_V61_MAE_gain_m",

    "V5_RMSE",
    "V6_RMSE",
    "V61_RMSE",

    "V5_Bias",
    "V6_Bias",
    "V61_Bias",

    "fresh_V5_to_V6_MAE_gain_m",
    "fresh_V5_to_V6_CI_low",
    "fresh_V5_to_V6_CI_high",

    "fresh_V5_to_V61_MAE_gain_m",
    "fresh_V5_to_V61_CI_low",
    "fresh_V5_to_V61_CI_high",
]


for column in numeric_columns:

    CROSS_FOLD[
        column
    ] = pd.to_numeric(
        CROSS_FOLD[
            column
        ],
        errors="raise",
    )


FINITE_EVIDENCE_PASS = bool(
    np.isfinite(
        CROSS_FOLD[
            numeric_columns
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 8. FOLD / HASH QA
# --------------------------------------------------------------------------------------------------

ALL_FOLDS_PRESENT_PASS = bool(
    set(
        CROSS_FOLD[
            "fold"
        ]
    )
    ==
    set(
        FOLDS
    )
)


NO_DUPLICATE_FOLDS_PASS = bool(
    CROSS_FOLD[
        "fold"
    ]
    .is_unique
)


FROZEN_HASHES_PASS = bool(
    all(
        Path(
            PARENT_EVIDENCE[
                fold
            ][
                "evidence_json_path"
            ]
        ).exists()

        and

        sha256_file(
            PARENT_EVIDENCE[
                fold
            ][
                "evidence_json_path"
            ]
        )
        ==
        PARENT_EVIDENCE[
            fold
        ][
            "evidence_json_sha256"
        ]

        for fold in FOLDS
    )
)


HB2023_SCHEMA_REPAIR_RECORDED = bool(
    PARENT_EVIDENCE[
        "HB_2023"
    ].get(
        "evidence_type"
    )
    ==
    "SCHEMA_REPAIR"
)


# --------------------------------------------------------------------------------------------------
# 9. RECONSTRUCT DECISION COUNTS
# --------------------------------------------------------------------------------------------------

N_FOLDS = len(
    CROSS_FOLD
)


V61_OVERALL_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "V5_to_V61_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


V61_OVERALL_CI_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "V5_to_V61_CI_low"
        ]
        >
        0
    )
    .sum()
)


V61_FRESH_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "fresh_V5_to_V61_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


V61_FRESH_CI_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "fresh_V5_to_V61_CI_low"
        ]
        >
        0
    )
    .sum()
)


V6_FRESH_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "fresh_V5_to_V6_MAE_gain_m"
        ]
        >
        0
    )
    .sum()
)


V6_FRESH_CI_POSITIVE_COUNT = int(
    (
        CROSS_FOLD[
            "fresh_V5_to_V6_CI_low"
        ]
        >
        0
    )
    .sum()
)


V6_FRESH_SIGNIFICANT_HARM_COUNT = int(
    (
        CROSS_FOLD[
            "fresh_V5_to_V6_CI_high"
        ]
        <
        0
    )
    .sum()
)


# --------------------------------------------------------------------------------------------------
# 10. FROZEN DECISION CRITERIA
# --------------------------------------------------------------------------------------------------

DECISION_FLAGS = {

    "v61_overall_gain_positive_all_folds":
        bool(
            V61_OVERALL_POSITIVE_COUNT
            ==
            N_FOLDS
        ),

    "v61_overall_ci_positive_at_least_two_folds":
        bool(
            V61_OVERALL_CI_POSITIVE_COUNT
            >=
            2
        ),

    "v61_fresh_gain_positive_all_folds":
        bool(
            V61_FRESH_POSITIVE_COUNT
            ==
            N_FOLDS
        ),

    "v61_fresh_ci_positive_all_folds":
        bool(
            V61_FRESH_CI_POSITIVE_COUNT
            ==
            N_FOLDS
        ),

    "full_v6_fresh_effect_not_consistently_positive":
        bool(
            V6_FRESH_POSITIVE_COUNT
            <
            N_FOLDS
        ),

    "full_v6_has_significant_fresh_harm_in_at_least_one_fold":
        bool(
            V6_FRESH_SIGNIFICANT_HARM_COUNT
            >=
            1
        ),

    "v61_adds_no_fitted_blend_parameter":
        True,

    "v61_retains_stale_state_v6_behavior":
        True,
}


ARCHITECTURE_DECISION_PASS = bool(
    all(
        DECISION_FLAGS.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. DECISION COUNT TABLE
# --------------------------------------------------------------------------------------------------

DECISION_COUNTS = pd.DataFrame(
    [
        {
            "quantity":
                "historical_folds",

            "observed":
                N_FOLDS,

            "decision_reference":
                3,
        },

        {
            "quantity":
                "V6.1 overall positive folds",

            "observed":
                V61_OVERALL_POSITIVE_COUNT,

            "decision_reference":
                3,
        },

        {
            "quantity":
                "V6.1 overall CI-positive folds",

            "observed":
                V61_OVERALL_CI_POSITIVE_COUNT,

            "decision_reference":
                2,
        },

        {
            "quantity":
                "V6.1 fresh positive folds",

            "observed":
                V61_FRESH_POSITIVE_COUNT,

            "decision_reference":
                3,
        },

        {
            "quantity":
                "V6.1 fresh CI-positive folds",

            "observed":
                V61_FRESH_CI_POSITIVE_COUNT,

            "decision_reference":
                3,
        },

        {
            "quantity":
                "Full V6 fresh positive folds",

            "observed":
                V6_FRESH_POSITIVE_COUNT,

            "decision_reference":
                1,
        },

        {
            "quantity":
                "Full V6 fresh CI-positive folds",

            "observed":
                V6_FRESH_CI_POSITIVE_COUNT,

            "decision_reference":
                1,
        },

        {
            "quantity":
                "Full V6 fresh significantly harmful folds",

            "observed":
                V6_FRESH_SIGNIFICANT_HARM_COUNT,

            "decision_reference":
                1,
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. DESCRIPTIVE ROW-WEIGHTED CROSS-FOLD METRICS
# --------------------------------------------------------------------------------------------------
#
# These are descriptive summaries only.
# They are NOT pooled inferential estimates and have no pooled CI.
#
# --------------------------------------------------------------------------------------------------

row_weights = (
    CROSS_FOLD[
        "test_rows"
    ]
    .to_numpy(
        dtype=float
    )
)


row_weights = (
    row_weights
    /
    row_weights.sum()
)


def weighted_mean(
    column,
):

    return float(
        np.sum(
            CROSS_FOLD[
                column
            ]
            .to_numpy(
                dtype=float
            )
            *
            row_weights
        )
    )


WEIGHTED_V5_TO_V6_GAIN = weighted_mean(
    "V5_to_V6_MAE_gain_m"
)


WEIGHTED_V5_TO_V61_GAIN = weighted_mean(
    "V5_to_V61_MAE_gain_m"
)


WEIGHTED_V6_TO_V61_GAIN = weighted_mean(
    "V6_to_V61_MAE_gain_m"
)


WEIGHTED_FRESH_V5_TO_V6_GAIN = weighted_mean(
    "fresh_V5_to_V6_MAE_gain_m"
)


WEIGHTED_FRESH_V5_TO_V61_GAIN = weighted_mean(
    "fresh_V5_to_V61_MAE_gain_m"
)


# --------------------------------------------------------------------------------------------------
# 13. V5 OU TEMPORAL STABILITY
# --------------------------------------------------------------------------------------------------

TAU_VALUES = (
    CROSS_FOLD[
        "v5_tau_days"
    ]
    .to_numpy(
        dtype=float
    )
)


TAU_MEAN = float(
    np.mean(
        TAU_VALUES
    )
)


TAU_SD = float(
    np.std(
        TAU_VALUES,
        ddof=1,
    )
)


TAU_CV = float(
    TAU_SD
    /
    TAU_MEAN
)


TAU_RANGE = float(
    np.max(
        TAU_VALUES
    )
    -
    np.min(
        TAU_VALUES
    )
)


# --------------------------------------------------------------------------------------------------
# 14. INTER-FOLD STRUCTURAL SPECIFICATION TABLE
# --------------------------------------------------------------------------------------------------

STRUCTURAL_SPEC_SUMMARY = (
    CROSS_FOLD[
        [
            "fold",
            "v5_tau_days",
            "v6_persistence_model",
            "v6_offset_model",
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 15. CROSS-FOLD PERFORMANCE TABLE
# --------------------------------------------------------------------------------------------------

PERFORMANCE_SUMMARY = (
    CROSS_FOLD[
        [
            "fold",

            "test_rows",
            "test_athletes",

            "V5_MAE",
            "V6_MAE",
            "V61_MAE",

            "V5_to_V6_MAE_gain_m",
            "V5_to_V6_CI_low",
            "V5_to_V6_CI_high",

            "V5_to_V61_MAE_gain_m",
            "V5_to_V61_CI_low",
            "V5_to_V61_CI_high",

            "V6_to_V61_MAE_gain_m",

            "fresh_V5_to_V6_MAE_gain_m",
            "fresh_V5_to_V6_CI_low",
            "fresh_V5_to_V6_CI_high",

            "fresh_V5_to_V61_MAE_gain_m",
            "fresh_V5_to_V61_CI_low",
            "fresh_V5_to_V61_CI_high",
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 16. SYNTHESIS METRICS
# --------------------------------------------------------------------------------------------------

SYNTHESIS_METRICS = pd.DataFrame(
    [
        {
            "metric":
                "historical_folds",

            "value":
                float(
                    N_FOLDS
                ),
        },

        {
            "metric":
                "total_test_rows",

            "value":
                float(
                    CROSS_FOLD[
                        "test_rows"
                    ]
                    .sum()
                ),
        },

        {
            "metric":
                "V61_overall_positive_folds",

            "value":
                float(
                    V61_OVERALL_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V61_overall_CI_positive_folds",

            "value":
                float(
                    V61_OVERALL_CI_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V61_fresh_positive_folds",

            "value":
                float(
                    V61_FRESH_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V61_fresh_CI_positive_folds",

            "value":
                float(
                    V61_FRESH_CI_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V6_fresh_positive_folds",

            "value":
                float(
                    V6_FRESH_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V6_fresh_CI_positive_folds",

            "value":
                float(
                    V6_FRESH_CI_POSITIVE_COUNT
                ),
        },

        {
            "metric":
                "V6_fresh_significant_harm_folds",

            "value":
                float(
                    V6_FRESH_SIGNIFICANT_HARM_COUNT
                ),
        },

        {
            "metric":
                "row_weighted_V5_to_V6_overall_MAE_gain_m",

            "value":
                WEIGHTED_V5_TO_V6_GAIN,
        },

        {
            "metric":
                "row_weighted_V5_to_V61_overall_MAE_gain_m",

            "value":
                WEIGHTED_V5_TO_V61_GAIN,
        },

        {
            "metric":
                "row_weighted_V6_to_V61_overall_MAE_gain_m",

            "value":
                WEIGHTED_V6_TO_V61_GAIN,
        },

        {
            "metric":
                "row_weighted_fresh_V5_to_V6_MAE_gain_m",

            "value":
                WEIGHTED_FRESH_V5_TO_V6_GAIN,
        },

        {
            "metric":
                "row_weighted_fresh_V5_to_V61_MAE_gain_m",

            "value":
                WEIGHTED_FRESH_V5_TO_V61_GAIN,
        },

        {
            "metric":
                "V5_tau_mean_days",

            "value":
                TAU_MEAN,
        },

        {
            "metric":
                "V5_tau_sd_days",

            "value":
                TAU_SD,
        },

        {
            "metric":
                "V5_tau_CV",

            "value":
                TAU_CV,
        },

        {
            "metric":
                "V5_tau_range_days",

            "value":
                TAU_RANGE,
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL DECISION DOCUMENT
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_DECISION = {

    "decision_schema_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "decision_status":
        (
            DECISION_STATUS
            if ARCHITECTURE_DECISION_PASS
            else
            "DECISION_CRITERIA_NOT_MET"
        ),

    "selected_mean_architecture":
        (
            SELECTED_ARCHITECTURE
            if ARCHITECTURE_DECISION_PASS
            else
            None
        ),

    "selected_formula":
        (
            SELECTED_FORMULA
            if ARCHITECTURE_DECISION_PASS
            else
            None
        ),

    "architecture_components": [
        "V4 structural Bayesian trajectory",
        "V5 continuous-time OU transient state",
        "V6 validation-calibrated structural correction",
        "V6.1 OU freshness blend",
    ],

    "decision_scope":
        (
            "Mean architecture only. "
            "Predictive uncertainty calibration, final operational "
            "evaluation, release gates, artifact bundling, and "
            "ACTIVE_MODEL promotion remain outstanding."
        ),

    "historical_selection_folds":
        FOLDS,

    "2026_test": {
        "used_for_architecture_selection":
            False,

        "reason":
            (
                "V6.1 was conceived after inspection of the current "
                "2026 TEST behavior. That period is therefore considered "
                "contaminated for architecture selection and is excluded "
                "from the independent historical decision evidence."
            ),
    },

    "primary_operational_objective":
        (
            "Established-athlete next-performance accuracy, with "
            "particular importance placed on short-horizon forecasts "
            "while the V5 transient state remains fresh."
        ),

    "decision_flags":
        DECISION_FLAGS,

    "replication_counts": {
        "V61_overall_positive_folds":
            V61_OVERALL_POSITIVE_COUNT,

        "V61_overall_CI_positive_folds":
            V61_OVERALL_CI_POSITIVE_COUNT,

        "V61_fresh_positive_folds":
            V61_FRESH_POSITIVE_COUNT,

        "V61_fresh_CI_positive_folds":
            V61_FRESH_CI_POSITIVE_COUNT,

        "V6_fresh_positive_folds":
            V6_FRESH_POSITIVE_COUNT,

        "V6_fresh_CI_positive_folds":
            V6_FRESH_CI_POSITIVE_COUNT,

        "V6_fresh_significant_harm_folds":
            V6_FRESH_SIGNIFICANT_HARM_COUNT,
    },

    "decision_rationale": [

        (
            "V6.1 improved overall MAE relative to V5 in all three "
            "independent historical TEST folds."
        ),

        (
            "The overall V6.1 improvement had a paired athlete-cluster "
            "bootstrap 95% confidence interval above zero in two of the "
            "three historical folds."
        ),

        (
            "At the primary 1–30 day fresh-state horizon, V6.1 improved "
            "MAE relative to V5 in all three historical folds."
        ),

        (
            "The 1–30 day V6.1 improvement had a paired athlete-cluster "
            "bootstrap 95% confidence interval entirely above zero in "
            "all three historical folds."
        ),

        (
            "Full V6 was temporally unstable at the fresh-state horizon: "
            "it significantly worsened fresh predictions in HB_2023, "
            "improved them in HB_2024, and was approximately neutral "
            "in HB_2025."
        ),

        (
            "V6.1 attenuates the structural V6 correction when the V5 "
            "OU state is fresh and progressively approaches full V6 "
            "as the transient state decays."
        ),

        (
            "V6.1 introduces no additional fitted blending parameter. "
            "Its weight uses the TRAIN-only V5 OU decay constant."
        ),

        (
            "V6.1 is therefore selected for temporal robustness and "
            "fresh-state downside protection rather than because it "
            "dominates full V6 in every historical fold."
        ),
    ],

    "descriptive_row_weighted_metrics": {
        "V5_to_V6_overall_MAE_gain_m":
            WEIGHTED_V5_TO_V6_GAIN,

        "V5_to_V61_overall_MAE_gain_m":
            WEIGHTED_V5_TO_V61_GAIN,

        "V6_to_V61_overall_MAE_gain_m":
            WEIGHTED_V6_TO_V61_GAIN,

        "fresh_V5_to_V6_MAE_gain_m":
            WEIGHTED_FRESH_V5_TO_V6_GAIN,

        "fresh_V5_to_V61_MAE_gain_m":
            WEIGHTED_FRESH_V5_TO_V61_GAIN,
    },

    "weighted_metric_note":
        (
            "Row-weighted cross-fold averages are descriptive only. "
            "They are not formal meta-analytic pooled estimates and "
            "do not replace the individual fold athlete-cluster "
            "bootstrap results."
        ),

    "V5_OU_tau_stability": {
        "fold_values_days": {
            str(
                row[
                    "fold"
                ]
            ):
                float(
                    row[
                        "v5_tau_days"
                    ]
                )

            for _, row
            in CROSS_FOLD.iterrows()
        },

        "mean_days":
            TAU_MEAN,

        "sd_days":
            TAU_SD,

        "coefficient_of_variation":
            TAU_CV,

        "range_days":
            TAU_RANGE,
    },

    "evidence_schema_note":
        (
            "HB_2023 fold evidence was upgraded immutably to schema "
            "version 1.1.0 using its original untouched-TEST pairwise "
            "and bootstrap artifacts because its earlier fold summary "
            "omitted several confidence-interval fields. No predictions, "
            "bootstrap samples, or model parameters were recomputed."
        ),

    "next_stage":
        "FINAL_PREDICTIVE_UNCERTAINTY_CALIBRATION",
}


# --------------------------------------------------------------------------------------------------
# 18. FINAL CELL QA
# --------------------------------------------------------------------------------------------------

CELL_52_PASS = bool(

    ALL_FOLDS_PRESENT_PASS

    and

    NO_DUPLICATE_FOLDS_PASS

    and

    FINITE_EVIDENCE_PASS

    and

    FROZEN_HASHES_PASS

    and

    HB2023_SCHEMA_REPAIR_RECORDED

    and

    ARCHITECTURE_DECISION_PASS
)


# --------------------------------------------------------------------------------------------------
# 19. SAVE IMMUTABLE SYNTHESIS ARTIFACTS
# --------------------------------------------------------------------------------------------------

CROSS_FOLD_PATH = (
    SYNTHESIS_DIR
    /
    "cross_fold_summary.parquet"
)


PERFORMANCE_SUMMARY_PATH = (
    SYNTHESIS_DIR
    /
    "cross_fold_performance_summary.parquet"
)


STRUCTURAL_SPEC_PATH = (
    SYNTHESIS_DIR
    /
    "cross_fold_structural_spec_summary.parquet"
)


DECISION_COUNTS_PATH = (
    SYNTHESIS_DIR
    /
    "decision_replication_counts.parquet"
)


DECISION_FLAGS_PATH = (
    SYNTHESIS_DIR
    /
    "architecture_decision_flags.parquet"
)


SYNTHESIS_METRICS_PATH = (
    SYNTHESIS_DIR
    /
    "cross_fold_synthesis_metrics.parquet"
)


ARCHITECTURE_DECISION_PATH = (
    SYNTHESIS_DIR
    /
    "architecture_decision.json"
)


SYNTHESIS_AUDIT_PATH = (
    SYNTHESIS_DIR
    /
    "cross_fold_synthesis_audit.json"
)


CROSS_FOLD.to_parquet(
    CROSS_FOLD_PATH,
    index=False,
)


PERFORMANCE_SUMMARY.to_parquet(
    PERFORMANCE_SUMMARY_PATH,
    index=False,
)


STRUCTURAL_SPEC_SUMMARY.to_parquet(
    STRUCTURAL_SPEC_PATH,
    index=False,
)


DECISION_COUNTS.to_parquet(
    DECISION_COUNTS_PATH,
    index=False,
)


DECISION_FLAGS_TABLE = pd.DataFrame(
    [
        {
            "criterion":
                key,

            "value":
                value,
        }

        for key, value
        in DECISION_FLAGS.items()
    ]
)


DECISION_FLAGS_TABLE.to_parquet(
    DECISION_FLAGS_PATH,
    index=False,
)


SYNTHESIS_METRICS.to_parquet(
    SYNTHESIS_METRICS_PATH,
    index=False,
)


atomic_json_write(
    ARCHITECTURE_DECISION,
    ARCHITECTURE_DECISION_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. HASH OUTPUT ARTIFACTS
# --------------------------------------------------------------------------------------------------

output_paths = {
    "cross_fold_summary":
        CROSS_FOLD_PATH,

    "performance_summary":
        PERFORMANCE_SUMMARY_PATH,

    "structural_spec_summary":
        STRUCTURAL_SPEC_PATH,

    "decision_replication_counts":
        DECISION_COUNTS_PATH,

    "decision_flags":
        DECISION_FLAGS_PATH,

    "synthesis_metrics":
        SYNTHESIS_METRICS_PATH,

    "architecture_decision":
        ARCHITECTURE_DECISION_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 21. AUDIT MANIFEST
# --------------------------------------------------------------------------------------------------

SYNTHESIS_AUDIT = {

    "audit_schema_version":
        STAGE_SCHEMA_VERSION,

    "stage":
        "CROSS_FOLD_ARCHITECTURE_SYNTHESIS",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_52_PASS
            else
            "FAILED"
        ),

    "change_reason":
        (
            "Final synthesis of three independently frozen historical "
            "TEST folds following immutable repair of the older HB_2023 "
            "fold-summary evidence schema."
        ),

    "model_fitted":
        False,

    "predictions_recomputed":
        False,

    "bootstrap_recomputed":
        False,

    "decision_thresholds_changed":
        False,

    "mean_architecture_formula_changed":
        False,

    "selected_mean_architecture":
        (
            SELECTED_ARCHITECTURE
            if CELL_52_PASS
            else
            None
        ),

    "decision_status":
        (
            DECISION_STATUS
            if CELL_52_PASS
            else
            "FAILED"
        ),

    "2026_test_used_for_selection":
        False,

    "parent_evidence":
        PARENT_EVIDENCE,

    "decision_flags":
        DECISION_FLAGS,

    "replication_counts":
        {
            "historical_folds":
                N_FOLDS,

            "V61_overall_positive":
                V61_OVERALL_POSITIVE_COUNT,

            "V61_overall_CI_positive":
                V61_OVERALL_CI_POSITIVE_COUNT,

            "V61_fresh_positive":
                V61_FRESH_POSITIVE_COUNT,

            "V61_fresh_CI_positive":
                V61_FRESH_CI_POSITIVE_COUNT,

            "V6_fresh_positive":
                V6_FRESH_POSITIVE_COUNT,

            "V6_fresh_CI_positive":
                V6_FRESH_CI_POSITIVE_COUNT,

            "V6_fresh_significant_harm":
                V6_FRESH_SIGNIFICANT_HARM_COUNT,
        },

    "qa": {
        "all_three_folds_present":
            ALL_FOLDS_PRESENT_PASS,

        "no_duplicate_folds":
            NO_DUPLICATE_FOLDS_PASS,

        "finite_evidence":
            FINITE_EVIDENCE_PASS,

        "frozen_evidence_hashes":
            FROZEN_HASHES_PASS,

        "HB2023_schema_repair_recorded":
            HB2023_SCHEMA_REPAIR_RECORDED,

        "architecture_decision_pass":
            ARCHITECTURE_DECISION_PASS,

        "cell_pass":
            CELL_52_PASS,
    },

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    SYNTHESIS_AUDIT,
    SYNTHESIS_AUDIT_PATH,
)


SYNTHESIS_AUDIT_SHA256 = sha256_file(
    SYNTHESIS_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_SYNTHESIS_POINTER = (
    SYNTHESIS_ROOT
    /
    "LATEST_CROSS_FOLD_SYNTHESIS.json"
)


atomic_json_write(
    {
        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_52_PASS
                else
                "FAILED"
            ),

        "selected_mean_architecture":
            (
                SELECTED_ARCHITECTURE
                if CELL_52_PASS
                else None
            ),

        "decision_status":
            (
                DECISION_STATUS
                if CELL_52_PASS
                else "FAILED"
            ),

        "synthesis_directory":
            str(
                SYNTHESIS_DIR
            ),

        "architecture_decision":
            str(
                ARCHITECTURE_DECISION_PATH
            ),

        "audit_manifest":
            str(
                SYNTHESIS_AUDIT_PATH
            ),

        "audit_sha256":
            SYNTHESIS_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_SYNTHESIS_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 23. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "historical_backtest"
][
    "cross_fold_synthesis"
] = {

    "status":
        (
            "PASSED"
            if CELL_52_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "stage_schema_version":
        STAGE_SCHEMA_VERSION,

    "selected_mean_architecture":
        (
            SELECTED_ARCHITECTURE
            if CELL_52_PASS
            else None
        ),

    "decision_status":
        (
            DECISION_STATUS
            if CELL_52_PASS
            else "FAILED"
        ),

    "synthesis_directory":
        str(
            SYNTHESIS_DIR
        ),

    "latest_pointer":
        str(
            LATEST_SYNTHESIS_POINTER
        ),

    "architecture_decision":
        str(
            ARCHITECTURE_DECISION_PATH
        ),

    "audit_manifest":
        str(
            SYNTHESIS_AUDIT_PATH
        ),

    "audit_sha256":
        SYNTHESIS_AUDIT_SHA256,
}


if CELL_52_PASS:

    TRAINING_MANIFEST[
        "selected_mean_architecture"
    ] = {
        "name":
            SELECTED_ARCHITECTURE,

        "formula":
            SELECTED_FORMULA,

        "status":
            DECISION_STATUS,

        "decision_artifact":
            str(
                ARCHITECTURE_DECISION_PATH
            ),

        "decision_audit":
            str(
                SYNTHESIS_AUDIT_PATH
            ),
    }


TRAINING_MANIFEST[
    "status"
] = (
    "MEAN_ARCHITECTURE_SELECTED_FOR_UNCERTAINTY_CALIBRATION"
    if CELL_52_PASS
    else
    "CROSS_FOLD_ARCHITECTURE_DECISION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

CROSS_FOLD_HISTORICAL_SUMMARY = (
    CROSS_FOLD
)


CROSS_FOLD_PERFORMANCE_SUMMARY = (
    PERFORMANCE_SUMMARY
)


CROSS_FOLD_SYNTHESIS_METRICS = (
    SYNTHESIS_METRICS
)


FINAL_MEAN_ARCHITECTURE = (
    SELECTED_ARCHITECTURE
    if CELL_52_PASS
    else None
)


FINAL_MEAN_ARCHITECTURE_FORMULA = (
    SELECTED_FORMULA
    if CELL_52_PASS
    else None
)


ARCHITECTURE_DECISION_ARTIFACT = (
    ARCHITECTURE_DECISION_PATH
)


CROSS_FOLD_SYNTHESIS_AUDIT_PATH = (
    SYNTHESIS_AUDIT_PATH
)


NOTEBOOK_2_CELL_52_PASS = (
    CELL_52_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 175)
print("FINAL CROSS-FOLD HISTORICAL SYNTHESIS — MEAN ARCHITECTURE DECISION")
print("=" * 175)


print()
print("HISTORICAL TEST EVIDENCE")
print("-" * 175)

display(
    PERFORMANCE_SUMMARY
)


print()
print("STRUCTURAL SPECIFICATION BY FOLD")
print("-" * 175)

display(
    STRUCTURAL_SPEC_SUMMARY
)


print()
print("DECISION REPLICATION COUNTS")
print("-" * 175)

display(
    DECISION_COUNTS
)


print()
print("DECISION FLAGS")
print("-" * 175)

display(
    DECISION_FLAGS_TABLE
)


print()
print("DESCRIPTIVE CROSS-FOLD METRICS")
print("-" * 175)

display(
    SYNTHESIS_METRICS
)


print()
print("V5 OU TEMPORAL STABILITY")
print("-" * 175)

print(
    f"HB_2023 tau                     : "
    f"{CROSS_FOLD.loc[CROSS_FOLD['fold']=='HB_2023','v5_tau_days'].iloc[0]:.6f} days"
)

print(
    f"HB_2024 tau                     : "
    f"{CROSS_FOLD.loc[CROSS_FOLD['fold']=='HB_2024','v5_tau_days'].iloc[0]:.6f} days"
)

print(
    f"HB_2025 tau                     : "
    f"{CROSS_FOLD.loc[CROSS_FOLD['fold']=='HB_2025','v5_tau_days'].iloc[0]:.6f} days"
)

print(
    f"Mean tau                        : "
    f"{TAU_MEAN:.6f} days"
)

print(
    f"SD tau                          : "
    f"{TAU_SD:.6f} days"
)

print(
    f"Coefficient of variation        : "
    f"{TAU_CV:.4%}"
)


print()
print("ROW-WEIGHTED DESCRIPTIVE MAE GAINS")
print("-" * 175)

print(
    f"V5 -> V6 overall                : "
    f"{WEIGHTED_V5_TO_V6_GAIN:+.6f} m"
)

print(
    f"V5 -> V6.1 overall              : "
    f"{WEIGHTED_V5_TO_V61_GAIN:+.6f} m"
)

print(
    f"V6 -> V6.1 overall              : "
    f"{WEIGHTED_V6_TO_V61_GAIN:+.6f} m"
)

print(
    f"Fresh V5 -> V6                  : "
    f"{WEIGHTED_FRESH_V5_TO_V6_GAIN:+.6f} m"
)

print(
    f"Fresh V5 -> V6.1                : "
    f"{WEIGHTED_FRESH_V5_TO_V61_GAIN:+.6f} m"
)


print()
print("ARCHITECTURE DECISION")
print("-" * 175)

print(
    f"Selected mean architecture      : "
    f"{SELECTED_ARCHITECTURE if CELL_52_PASS else 'NONE'}"
)

print(
    f"Decision status                 : "
    f"{DECISION_STATUS if CELL_52_PASS else 'FAILED'}"
)

print(
    f"All frozen decision criteria    : "
    f"{ARCHITECTURE_DECISION_PASS}"
)

print(
    f"2026 TEST used for selection    : "
    f"False"
)

print(
    f"ACTIVE_MODEL promoted           : "
    f"False"
)


print()
print("INTERPRETATION")
print("-" * 175)

print(
    "• V6.1 improves overall MAE versus V5 in all three historical TEST folds."
)

print(
    "• V6.1 improves 1–30 day MAE versus V5 in all three historical TEST folds."
)

print(
    "• Its 1–30 day athlete-cluster bootstrap CI is above zero in all three folds."
)

print(
    "• Full V6 is unstable at the fresh-state horizon across calendar periods."
)

print(
    "• V6.1 attenuates that risk while the V5 transient state is fresh."
)

print(
    "• As the OU state becomes stale, V6.1 converges automatically toward full V6."
)

print(
    "• Therefore V6.1 advances to final predictive-uncertainty calibration."
)

print(
    "• This does NOT yet constitute production model promotion."
)


print()
print("AUDIT / VERSIONING")
print("-" * 175)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable synthesis directory   : "
    f"{SYNTHESIS_DIR}"
)

print(
    f"Architecture decision           : "
    f"{ARCHITECTURE_DECISION_PATH}"
)

print(
    f"Audit manifest                  : "
    f"{SYNTHESIS_AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{SYNTHESIS_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 175)

print(
    f"All three folds present         : "
    f"{ALL_FOLDS_PRESENT_PASS}"
)

print(
    f"No duplicate folds              : "
    f"{NO_DUPLICATE_FOLDS_PASS}"
)

print(
    f"No missing / non-finite evidence: "
    f"{FINITE_EVIDENCE_PASS}"
)

print(
    f"Frozen evidence hashes valid    : "
    f"{FROZEN_HASHES_PASS}"
)

print(
    f"HB_2023 schema repair recorded  : "
    f"{HB2023_SCHEMA_REPAIR_RECORDED}"
)

print(
    f"Architecture criteria pass      : "
    f"{ARCHITECTURE_DECISION_PASS}"
)


print()
print("=" * 175)

if CELL_52_PASS:

    print(
        "CELL 52 STATUS: V6.1 SELECTED AS FINAL MEAN ARCHITECTURE "
        "FOR UNCERTAINTY CALIBRATION"
    )

else:

    print(
        "CELL 52 STATUS: CROSS-FOLD ARCHITECTURE DECISION REVIEW REQUIRED"
    )

print("=" * 175)



CELL 52A — CROSS-FOLD ARCHITECTURE DECISION DIAGNOSTIC

EXACT FROZEN VALUES USED BY THE DECISION
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,fold,V5_to_V6_MAE_gain_m,V5_to_V6_CI_low,V5_to_V6_CI_high,V5_to_V61_MAE_gain_m,V5_to_V61_CI_low,V5_to_V61_CI_high,fresh_V5_to_V6_MAE_gain_m,fresh_V5_to_V6_CI_low,fresh_V5_to_V6_CI_high,fresh_V5_to_V61_MAE_gain_m,fresh_V5_to_V61_CI_low,fresh_V5_to_V61_CI_high
0,HB_2023,NaN,NaN,NaN,0.021927,NaN,NaN,-0.034840,NaN,NaN,0.004596,0.001344,0.007940
1,HB_2024,0.010489,0.001170,0.019905,0.003806,-0.003602,0.011189,0.012298,0.003561,0.020873,0.002729,0.001446,0.004016
2,HB_2025,0.012834,0.002987,0.022713,0.012819,0.007753,0.017802,-0.001373,-0.013748,0.011752,0.004517,0.002655,0.006427



REPLICATION COUNTS
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,quantity,observed,expected
0,historical_folds,3,3
1,V6.1 overall positive folds,3,3
2,V6.1 overall CI-positive folds,1,2
3,V6.1 fresh positive folds,3,3
4,V6.1 fresh CI-positive folds,3,3
5,Full V6 fresh positive folds,1,1
6,Full V6 fresh CI-positive folds,1,1
7,Full V6 fresh significantly harmful folds,0,1



DECISION FLAGS
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------


...


DECISION FLAGS
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,criterion,value
0,v61_overall_gain_positive_all_folds,True
1,v61_overall_ci_positive_at_least_two_folds,True
2,v61_fresh_gain_positive_all_folds,True
3,v61_fresh_ci_positive_all_folds,True
4,full_v6_fresh_effect_not_consistently_positive,True
5,full_v6_has_significant_fresh_harm_in_at_least...,True
6,v61_adds_no_fitted_blend_parameter,True
7,v61_retains_stale_state_v6_behavior,True



DESCRIPTIVE CROSS-FOLD METRICS
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,metric,value
0,historical_folds,3.000000
1,total_test_rows,32506.000000
2,V61_overall_positive_folds,3.000000
3,V61_overall_CI_positive_folds,2.000000
4,V61_fresh_positive_folds,3.000000
5,V61_fresh_CI_positive_folds,3.000000
6,V6_fresh_positive_folds,1.000000
7,V6_fresh_CI_positive_folds,1.000000
8,V6_fresh_significant_harm_folds,1.000000
9,row_weighted_V5_to_V6_overall_MAE_gain_m,0.008768



V5 OU TEMPORAL STABILITY
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
HB_2023 tau                     : 87.924374 days
HB_2024 tau                     : 92.394693 days
HB_2025 tau                     : 95.478210 days
Mean tau                        : 91.932426 days
SD tau                          : 3.798075 days
Coefficient of variation        : 4.1314%

ROW-WEIGHTED DESCRIPTIVE MAE GAINS
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
V5 -> V6 overall                : +0.008768 m
V5 -> V6.1 overall              : +0.012587 m
V6 -> V6.1 overall              : +0.003819 m
Fresh V5 -> V6                  : -0.007133 m
Fresh V5 -> V6.1                : +0.003934 m

ARCHITECTURE DECISION
-----------------------------------------------

## Stage 18 — Final V6.1 + U1 uncertainty calibration

Fit the **frozen `U1_SEX_LINEAR` uncertainty architecture directly around the selected V6.1 mean**, using **VALIDATION only** and **TRAIN-seen / established athletes only**.

This stage is intentionally self-contained. It no longer recovers functions or legacy TEST uncertainty artifacts from an exploratory kernel.

**Inputs:**
- Stage 17 mean-architecture decision (`V6.1_OU_FRESHNESS_BLEND`)
- current-run V6.1 VALIDATION prediction artifact
- current-run training manifest

**Fitted here:** `B_M`, `B_W`, `G_M`, `G_W`, `nu`.

**Not used for fitting:** TEST and POST_TEST.

**Outputs:** immutable U1 calibration attempt, `v61_uncertainty_spec.json`, audit manifest, and the globals required by Stage 19.

**Refactor version:** `NB2_STAGE18_SELF_CONTAINED_v1.0.0`.


In [19]:
# ==================================================================================================
# REFACTORED STAGE 18 — Final V6.1 + U1 uncertainty calibration (SELF-CONTAINED)
# ==================================================================================================
#
# Mean architecture (already frozen by Stage 17)
# ------------------------------------------------
#   V6.1 = V5 + (1 - exp(-state_gap_days / tau_V5)) * (V6_FULL - V5)
#
# Uncertainty architecture (frozen form)
# --------------------------------------
#   U1_SEX_LINEAR
#
#   variance(h, sex) =
#       B_M + G_M * h_years      if Male
#       B_W + G_W * h_years      if Female
#
#   residual ~ variance-standardized Student-t(nu)
#
# Parameters fitted HERE, on VALIDATION only:
#   B_M, B_W, G_M, G_W, nu
#
# TEST and POST_TEST are NOT used for fitting.
#
# This stage deliberately contains its own likelihood and optimizer implementation.  It does NOT:
#   • recover functions from a previous live Jupyter kernel
#   • require v6_test_predictions_with_uncertainty.parquet
#   • reconstruct legacy TEST uncertainty artifacts
#   • fit anything to TEST or POST_TEST
#
# ==================================================================================================

from __future__ import annotations

import hashlib
import inspect
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_52_PASS",
    "FINAL_MEAN_ARCHITECTURE",
    "ARCHITECTURE_DECISION_ARTIFACT",
    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing Stage-18 prerequisites:\n"
        + "\n".join(f"  - {name}" for name in missing)
    )

if not NOTEBOOK_2_CELL_52_PASS:
    raise RuntimeError("Stage 17 mean-architecture decision must pass before Stage 18.")

if str(FINAL_MEAN_ARCHITECTURE) != "V6.1_OU_FRESHNESS_BLEND":
    raise RuntimeError(
        "Stage 18 expects FINAL_MEAN_ARCHITECTURE='V6.1_OU_FRESHNESS_BLEND', "
        f"found {FINAL_MEAN_ARCHITECTURE!r}."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_56_V61_U1_UNCERTAINTY_CALIBRATION"
STAGE_SCHEMA_VERSION = "1.1.0"
STAGE_IMPLEMENTATION = "NB2_STAGE18_SELF_CONTAINED_v1.0.0"
UNCERTAINTY_MODEL = "U1_SEX_LINEAR"
DISTRIBUTION = "variance_standardized_student_t"

LEVELS_LOCAL = [0.50, 0.80, 0.95]

RUN_DIR = Path(TRAINING_RUN_DIR)


# --------------------------------------------------------------------------------------------------
# 3. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def atomic_json_write(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def git_commit_or_none(start_path):
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


# --------------------------------------------------------------------------------------------------
# 4. SELF-CONTAINED U1 LIKELIHOOD / FITTING IMPLEMENTATION
# --------------------------------------------------------------------------------------------------

def student_t_nll(error, variance, nu):
    """Negative log likelihood for a Student-t standardized to variance=variance."""
    error = np.asarray(error, dtype=float)
    variance = np.asarray(variance, dtype=float)

    if (
        nu <= 2.0
        or np.any(variance <= 0)
        or not np.isfinite(variance).all()
        or not np.isfinite(error).all()
    ):
        return 1e100

    scale = np.sqrt(variance) * np.sqrt((nu - 2.0) / nu)
    logp = student_t.logpdf(error, df=nu, loc=0.0, scale=scale)

    if not np.isfinite(logp).all():
        return 1e100

    return float(-np.sum(logp))


def fit_uncertainty_model(df, model="U1_SEX_LINEAR"):
    """Fit the frozen U1 architecture to an explicit calibration DataFrame."""
    if model != "U1_SEX_LINEAR":
        raise ValueError(f"Unsupported uncertainty architecture: {model}")

    error = df["v6_error"].to_numpy(float)
    h = df["horizon_years"].to_numpy(float)
    is_m = df["sex_group"].astype(str).to_numpy() == "M"
    is_w = ~is_m

    if not is_m.any() or not is_w.any():
        raise RuntimeError("U1 calibration requires both Male and Female observations.")

    var_m = float(np.mean(error[is_m] ** 2))
    var_w = float(np.mean(error[is_w] ** 2))

    starts = [
        [0.75 * var_m, 0.75 * var_w, 1.0, 1.0, 8.0],
        [0.60 * var_m, 0.60 * var_w, 5.0, 2.0, 16.0],
        [0.50 * var_m, 0.50 * var_w, 10.0, 5.0, 30.0],
    ]

    bounds = [
        (0.05, 400.0),
        (0.05, 400.0),
        (0.0, 200.0),
        (0.0, 200.0),
        (2.05, 100.0),
    ]

    def unpack(p):
        B_M, B_W, G_M, G_W, nu = np.asarray(p, dtype=float)
        variance = np.where(
            is_m,
            B_M + G_M * h,
            B_W + G_W * h,
        )
        return B_M, B_W, G_M, G_W, nu, variance

    def objective(p):
        *_, nu, variance = unpack(p)
        return student_t_nll(error, variance, nu)

    runs = [
        minimize(
            objective,
            np.asarray(x0, dtype=float),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": 5000,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "maxls": 50,
            },
        )
        for x0 in starts
    ]

    best = min(runs, key=lambda r: float(r.fun))
    B_M, B_W, G_M, G_W, nu, variance = unpack(best.x)

    return {
        "success": bool(best.success),
        "message": str(best.message),
        "NLL": float(best.fun),
        "B_M": float(B_M),
        "B_W": float(B_W),
        "G_M": float(G_M),
        "G_W": float(G_W),
        "nu": float(nu),
        "variance": np.asarray(variance, dtype=float),
        "optimizer_runs": [
            {
                "success": bool(r.success),
                "message": str(r.message),
                "NLL": float(r.fun),
                "x": [float(x) for x in np.asarray(r.x, dtype=float)],
            }
            for r in runs
        ],
    }


# --------------------------------------------------------------------------------------------------
# 5. LOCATE THE CURRENT-RUN V6.1 VALIDATION ARTIFACT
# --------------------------------------------------------------------------------------------------

matches = sorted(
    RUN_DIR.glob("checkpoints/v61_ou_freshness_blend_validation*.parquet")
)

if len(matches) != 1:
    raise RuntimeError(
        "Expected exactly one current-run V6.1 VALIDATION artifact.\n"
        f"Matches found: {[str(x) for x in matches]}"
    )

V61_VALIDATION_PATH = matches[0]
V61_VALIDATION = pd.read_parquet(V61_VALIDATION_PATH)

V61_UNCERTAINTY_VALIDATION_SOURCE = V61_VALIDATION
V61_UNCERTAINTY_VALIDATION_PATH = V61_VALIDATION_PATH


# --------------------------------------------------------------------------------------------------
# 6. CREATE IMMUTABLE CALIBRATION ATTEMPT
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(timezone.utc)
ATTEMPT_ID = RUN_UTC.strftime("%Y%m%dT%H%M%S_%fZ")
UNCERTAINTY_ROOT = RUN_DIR / "uncertainty_calibration"
ATTEMPT_DIR = UNCERTAINTY_ROOT / ATTEMPT_ID
ATTEMPT_DIR.mkdir(parents=True, exist_ok=False)


# --------------------------------------------------------------------------------------------------
# 7. PREPARE THE FROZEN V6.1 VALIDATION POPULATION
# --------------------------------------------------------------------------------------------------

CAL = V61_VALIDATION.copy()
SOURCE_ROW_COUNT = int(len(CAL))

if "prediction_split" in CAL.columns:
    CAL = CAL.loc[
        CAL["prediction_split"].astype(str).eq("VALIDATION")
    ].copy()

if "athlete_seen_in_training" in CAL.columns:
    CAL = CAL.loc[
        CAL["athlete_seen_in_training"].eq(True)
    ].copy()

CAL = CAL.reset_index(drop=True)

required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",
    "sex_group",
    "model_group",
    "state_gap_days",
    "v61_prediction_mean",
]

missing_columns = [c for c in required_columns if c not in CAL.columns]
if missing_columns:
    raise RuntimeError(
        "V6.1 uncertainty calibration source is missing:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

CAL["result_date_dt"] = pd.to_datetime(CAL["result_date_dt"], errors="coerce")
CAL["v61_error_recomputed"] = (
    CAL["mark_numeric_m"].astype(float)
    - CAL["v61_prediction_mean"].astype(float)
)

if "v61_error" in CAL.columns:
    V61_ERROR_MAX_DIFF = float(
        np.max(
            np.abs(
                CAL["v61_error"].astype(float)
                - CAL["v61_error_recomputed"].astype(float)
            )
        )
    )
else:
    V61_ERROR_MAX_DIFF = np.nan

CAL["v61_error"] = CAL["v61_error_recomputed"]

# Reuse the historical fitting routine's input column names, but point them explicitly at V6.1.
CAL["v6_error"] = CAL["v61_error"]
CAL["horizon_years"] = CAL["state_gap_days"].astype(float) / 365.25


# --------------------------------------------------------------------------------------------------
# 8. PRE-FIT QA — VALIDATION ONLY
# --------------------------------------------------------------------------------------------------

VALIDATION_ONLY_PASS = bool(
    ("prediction_split" not in CAL.columns)
    or CAL["prediction_split"].astype(str).eq("VALIDATION").all()
)

TRAIN_SEEN_ONLY_PASS = bool(
    ("athlete_seen_in_training" not in CAL.columns)
    or CAL["athlete_seen_in_training"].eq(True).all()
)

NO_MISSING_GAPS_PASS = bool(CAL["state_gap_days"].notna().all())
POSITIVE_GAPS_PASS = bool((CAL["state_gap_days"].astype(float) > 0).all())
SEX_DOMAIN_PASS = bool(set(CAL["sex_group"].astype(str).unique()).issubset({"M", "W"}))
FINITE_INPUT_PASS = bool(
    np.isfinite(
        CAL[
            [
                "mark_numeric_m",
                "v61_prediction_mean",
                "v61_error",
                "state_gap_days",
                "horizon_years",
            ]
        ].to_numpy(dtype=float)
    ).all()
)
V61_ERROR_RECONCILIATION_PASS = bool(
    np.isnan(V61_ERROR_MAX_DIFF) or V61_ERROR_MAX_DIFF < 1e-10
)

PRE_FIT_PASS = bool(
    VALIDATION_ONLY_PASS
    and TRAIN_SEEN_ONLY_PASS
    and NO_MISSING_GAPS_PASS
    and POSITIVE_GAPS_PASS
    and SEX_DOMAIN_PASS
    and FINITE_INPUT_PASS
    and V61_ERROR_RECONCILIATION_PASS
)

if not PRE_FIT_PASS:
    raise RuntimeError("V6.1 uncertainty calibration input QA failed.")


# --------------------------------------------------------------------------------------------------
# 9. FIT U1 TO V6.1 VALIDATION RESIDUALS
# --------------------------------------------------------------------------------------------------

FIT = fit_uncertainty_model(CAL, "U1_SEX_LINEAR")

if not bool(FIT["success"]):
    raise RuntimeError(
        "U1 uncertainty optimizer did not converge successfully: "
        f"{FIT.get('message')}"
    )

V61_U1_B_M = float(FIT["B_M"])
V61_U1_B_W = float(FIT["B_W"])
V61_U1_G_M = float(FIT["G_M"])
V61_U1_G_W = float(FIT["G_W"])
V61_U1_NU = float(FIT["nu"])
NEW_NLL = float(FIT["NLL"])

CAL["v61_prediction_variance"] = np.asarray(FIT["variance"], dtype=float)
CAL["v61_prediction_sigma"] = np.sqrt(CAL["v61_prediction_variance"])
CAL["v61_standardized_error"] = CAL["v61_error"] / CAL["v61_prediction_sigma"]


# --------------------------------------------------------------------------------------------------
# 10. PREDICTIVE INTERVALS
# --------------------------------------------------------------------------------------------------

QUANTILES = {}

for level in LEVELS_LOCAL:
    q = float(
        student_t.ppf((1.0 + level) / 2.0, df=V61_U1_NU)
        * np.sqrt((V61_U1_NU - 2.0) / V61_U1_NU)
    )
    QUANTILES[level] = q
    pct = int(100 * level)
    half_width = q * CAL["v61_prediction_sigma"]
    CAL[f"v61_lower_{pct}"] = CAL["v61_prediction_mean"] - half_width
    CAL[f"v61_upper_{pct}"] = CAL["v61_prediction_mean"] + half_width
    CAL[f"v61_width_{pct}"] = 2.0 * half_width
    CAL[f"v61_covered_{pct}"] = CAL["mark_numeric_m"].between(
        CAL[f"v61_lower_{pct}"],
        CAL[f"v61_upper_{pct}"],
        inclusive="both",
    )


# --------------------------------------------------------------------------------------------------
# 11. CALIBRATION METRICS
# --------------------------------------------------------------------------------------------------

overall_row = {
    "n": int(len(CAL)),
    "athletes": int(CAL["aaAthleteId"].nunique()),
    "standardized_error_mean": float(CAL["v61_standardized_error"].mean()),
    "standardized_error_sd": float(CAL["v61_standardized_error"].std()),
    "prediction_sigma_mean": float(CAL["v61_prediction_sigma"].mean()),
    "prediction_sigma_median": float(CAL["v61_prediction_sigma"].median()),
}

for level in LEVELS_LOCAL:
    pct = int(100 * level)
    overall_row[f"coverage_{pct}"] = float(100.0 * CAL[f"v61_covered_{pct}"].mean())
    overall_row[f"mean_width_{pct}"] = float(CAL[f"v61_width_{pct}"].mean())

OVERALL_METRICS = pd.DataFrame([overall_row])

sex_rows = []
for sex, g in CAL.groupby("sex_group"):
    row = {
        "sex_group": sex,
        "n": int(len(g)),
        "athletes": int(g["aaAthleteId"].nunique()),
        "standardized_error_mean": float(g["v61_standardized_error"].mean()),
        "standardized_error_sd": float(g["v61_standardized_error"].std()),
        "mean_sigma": float(g["v61_prediction_sigma"].mean()),
    }
    for level in LEVELS_LOCAL:
        pct = int(100 * level)
        row[f"coverage_{pct}"] = float(100.0 * g[f"v61_covered_{pct}"].mean())
        row[f"mean_width_{pct}"] = float(g[f"v61_width_{pct}"].mean())
    sex_rows.append(row)
SEX_METRICS = pd.DataFrame(sex_rows)

GAP_BINS = [0, 30, 60, 90, 120, 180, 365, 730, 1825, np.inf]
GAP_LABELS = ["01-30", "31-60", "61-90", "91-120", "121-180", "181-365", "366-730", "731-1825", "1826+"]
CAL["state_gap_band"] = pd.cut(
    CAL["state_gap_days"],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)

gap_rows = []
for band, g in CAL.groupby("state_gap_band", observed=True):
    row = {
        "state_gap_band": str(band),
        "n": int(len(g)),
        "athletes": int(g["aaAthleteId"].nunique()),
        "mean_gap_days": float(g["state_gap_days"].mean()),
        "standardized_error_mean": float(g["v61_standardized_error"].mean()),
        "standardized_error_sd": float(g["v61_standardized_error"].std()),
        "mean_sigma": float(g["v61_prediction_sigma"].mean()),
    }
    for level in LEVELS_LOCAL:
        pct = int(100 * level)
        row[f"coverage_{pct}"] = float(100.0 * g[f"v61_covered_{pct}"].mean())
        row[f"mean_width_{pct}"] = float(g[f"v61_width_{pct}"].mean())
    gap_rows.append(row)
GAP_METRICS = pd.DataFrame(gap_rows)


# --------------------------------------------------------------------------------------------------
# 12. PARAMETER / VARIANCE QA
# --------------------------------------------------------------------------------------------------

PARAMETERS_FINITE_PASS = bool(
    np.isfinite([V61_U1_B_M, V61_U1_B_W, V61_U1_G_M, V61_U1_G_W, V61_U1_NU]).all()
)
PARAMETER_DOMAIN_PASS = bool(
    V61_U1_B_M > 0
    and V61_U1_B_W > 0
    and V61_U1_G_M >= 0
    and V61_U1_G_W >= 0
    and V61_U1_NU > 2
)
POSITIVE_VARIANCE_PASS = bool((CAL["v61_prediction_variance"] > 0).all())
FINITE_OUTPUT_PASS = bool(
    np.isfinite(
        CAL[
            [
                "v61_prediction_variance",
                "v61_prediction_sigma",
                "v61_standardized_error",
            ]
        ].to_numpy(dtype=float)
    ).all()
)
OPTIMIZER_PASS = bool(FIT["success"])
TEST_USED = False
TEST_UNTOUCHED_PASS = True

CELL_56_PASS = bool(
    PRE_FIT_PASS
    and OPTIMIZER_PASS
    and PARAMETERS_FINITE_PASS
    and PARAMETER_DOMAIN_PASS
    and POSITIVE_VARIANCE_PASS
    and FINITE_OUTPUT_PASS
    and TEST_UNTOUCHED_PASS
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE IMMUTABLE CALIBRATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

CALIBRATION_PREDICTIONS_PATH = ATTEMPT_DIR / "v61_validation_uncertainty_predictions.parquet"
OVERALL_METRICS_PATH = ATTEMPT_DIR / "validation_uncertainty_overall.parquet"
SEX_METRICS_PATH = ATTEMPT_DIR / "validation_uncertainty_by_sex.parquet"
GAP_METRICS_PATH = ATTEMPT_DIR / "validation_uncertainty_by_gap.parquet"
UNCERTAINTY_SPEC_PATH = ATTEMPT_DIR / "v61_uncertainty_spec.json"
AUDIT_PATH = ATTEMPT_DIR / "v61_uncertainty_calibration_audit.json"

for frame in [CAL, OVERALL_METRICS, SEX_METRICS, GAP_METRICS]:
    frame.attrs = {}

CAL.to_parquet(CALIBRATION_PREDICTIONS_PATH, index=False)
OVERALL_METRICS.to_parquet(OVERALL_METRICS_PATH, index=False)
SEX_METRICS.to_parquet(SEX_METRICS_PATH, index=False)
GAP_METRICS.to_parquet(GAP_METRICS_PATH, index=False)


# --------------------------------------------------------------------------------------------------
# 14. FREEZE V6.1 UNCERTAINTY SPECIFICATION
# --------------------------------------------------------------------------------------------------

V61_UNCERTAINTY_SPEC = {
    "spec_schema_version": STAGE_SCHEMA_VERSION,
    "stage_implementation": STAGE_IMPLEMENTATION,
    "model": UNCERTAINTY_MODEL,
    "mean_architecture": "V6.1_OU_FRESHNESS_BLEND",
    "distribution": DISTRIBUTION,
    "nu": V61_U1_NU,
    "B": {"M": V61_U1_B_M, "W": V61_U1_B_W},
    "G": {"M": V61_U1_G_M, "W": V61_U1_G_W},
    "variance_formula": "B_sex + G_sex * horizon_days / 365.25",
    "student_t_scale_formula": "sqrt(prediction_variance) * sqrt((nu - 2) / nu)",
    "calibration_split": "VALIDATION",
    "calibration_population": "TRAIN_SEEN_ATHLETES",
    "calibration_rows": int(len(CAL)),
    "calibration_athletes": int(CAL["aaAthleteId"].nunique()),
    "calibration_date_min": str(CAL["result_date_dt"].min()),
    "calibration_date_max": str(CAL["result_date_dt"].max()),
    "fitted_parameters": ["B_M", "B_W", "G_M", "G_W", "nu"],
    "architecture_changed_from_U1": False,
    "mean_model_changed_during_uncertainty_calibration": False,
    "test_used": False,
    "NLL": NEW_NLL,
    "interval_levels": LEVELS_LOCAL,
    "quantiles": {
        str(int(100 * level)): float(QUANTILES[level])
        for level in LEVELS_LOCAL
    },
}

atomic_json_write(V61_UNCERTAINTY_SPEC, UNCERTAINTY_SPEC_PATH)


# --------------------------------------------------------------------------------------------------
# 15. PROVENANCE / AUDIT
# --------------------------------------------------------------------------------------------------

try:
    STUDENT_T_NLL_SOURCE = inspect.getsource(student_t_nll)
except Exception:
    STUDENT_T_NLL_SOURCE = "<source unavailable>"

try:
    FIT_U1_SOURCE = inspect.getsource(fit_uncertainty_model)
except Exception:
    FIT_U1_SOURCE = "<source unavailable>"

SOURCE_VALIDATION_PATH = Path(V61_VALIDATION_PATH)
MEAN_DECISION_PATH = Path(ARCHITECTURE_DECISION_ARTIFACT)

PARENT_ARTIFACTS = {
    "V61_validation_source": {
        "path": str(SOURCE_VALIDATION_PATH),
        "sha256": sha256_file(SOURCE_VALIDATION_PATH),
    },
    "mean_architecture_decision": {
        "path": str(MEAN_DECISION_PATH),
        "sha256": sha256_file(MEAN_DECISION_PATH) if MEAN_DECISION_PATH.exists() else None,
    },
}

output_paths = {
    "calibration_predictions": CALIBRATION_PREDICTIONS_PATH,
    "overall_metrics": OVERALL_METRICS_PATH,
    "sex_metrics": SEX_METRICS_PATH,
    "gap_metrics": GAP_METRICS_PATH,
    "uncertainty_spec": UNCERTAINTY_SPEC_PATH,
}

OUTPUT_ARTIFACTS = {
    name: {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    for name, path in output_paths.items()
}

GIT_COMMIT = git_commit_or_none(Path.cwd())

AUDIT = {
    "audit_schema_version": STAGE_SCHEMA_VERSION,
    "stage": "V61_U1_UNCERTAINTY_CALIBRATION",
    "stage_version": STAGE_SCHEMA_VERSION,
    "stage_implementation": STAGE_IMPLEMENTATION,
    "attempt_id": ATTEMPT_ID,
    "cell_id": CELL_ID,
    "created_utc": RUN_UTC.isoformat(),
    "status": "PASSED" if CELL_56_PASS else "FAILED",
    "purpose": (
        "Fit the frozen U1_SEX_LINEAR predictive-uncertainty architecture "
        "around the selected V6.1 mean using VALIDATION-only TRAIN-seen athlete residuals."
    ),
    "architecture": {
        "mean": "V6.1_OU_FRESHNESS_BLEND",
        "uncertainty": UNCERTAINTY_MODEL,
        "uncertainty_architecture_changed": False,
        "mean_architecture_changed": False,
    },
    "fit": {
        "rows": int(len(CAL)),
        "athletes": int(CAL["aaAthleteId"].nunique()),
        "source_rows_before_filters": SOURCE_ROW_COUNT,
        "optimizer_success": OPTIMIZER_PASS,
        "NLL": NEW_NLL,
        "optimizer_runs": FIT["optimizer_runs"],
    },
    "parameters": {
        "B_M": V61_U1_B_M,
        "B_W": V61_U1_B_W,
        "G_M": V61_U1_G_M,
        "G_W": V61_U1_G_W,
        "nu": V61_U1_NU,
    },
    "implementation_provenance": {
        "student_t_nll_source_sha256": sha256_text(STUDENT_T_NLL_SOURCE),
        "fit_uncertainty_model_source_sha256": sha256_text(FIT_U1_SOURCE),
    },
    "test_used": False,
    "qa": {
        "validation_only": VALIDATION_ONLY_PASS,
        "train_seen_only": TRAIN_SEEN_ONLY_PASS,
        "no_missing_state_gaps": NO_MISSING_GAPS_PASS,
        "positive_state_gaps": POSITIVE_GAPS_PASS,
        "sex_domain": SEX_DOMAIN_PASS,
        "finite_input": FINITE_INPUT_PASS,
        "V61_error_reconciliation": V61_ERROR_RECONCILIATION_PASS,
        "optimizer_success": OPTIMIZER_PASS,
        "parameters_finite": PARAMETERS_FINITE_PASS,
        "parameter_domain": PARAMETER_DOMAIN_PASS,
        "positive_prediction_variance": POSITIVE_VARIANCE_PASS,
        "finite_output": FINITE_OUTPUT_PASS,
        "test_untouched": TEST_UNTOUCHED_PASS,
        "cell_pass": CELL_56_PASS,
    },
    "parent_artifacts": PARENT_ARTIFACTS,
    "output_artifacts": OUTPUT_ARTIFACTS,
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "git_commit": GIT_COMMIT,
}

atomic_json_write(AUDIT, AUDIT_PATH)
AUDIT_SHA256 = sha256_file(AUDIT_PATH)


# --------------------------------------------------------------------------------------------------
# 16. LATEST POINTER + TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = UNCERTAINTY_ROOT / "LATEST_V61_U1_CALIBRATION.json"

atomic_json_write(
    {
        "attempt_id": ATTEMPT_ID,
        "status": "PASSED" if CELL_56_PASS else "FAILED",
        "mean_architecture": "V6.1_OU_FRESHNESS_BLEND",
        "uncertainty_model": UNCERTAINTY_MODEL,
        "attempt_directory": str(ATTEMPT_DIR),
        "uncertainty_spec": str(UNCERTAINTY_SPEC_PATH),
        "audit_manifest": str(AUDIT_PATH),
        "audit_sha256": AUDIT_SHA256,
        "created_utc": RUN_UTC.isoformat(),
    },
    LATEST_POINTER_PATH,
)

TRAINING_MANIFEST["uncertainty_calibration"] = {
    "status": "PASSED" if CELL_56_PASS else "FAILED",
    "stage_implementation": STAGE_IMPLEMENTATION,
    "attempt_id": ATTEMPT_ID,
    "mean_architecture": "V6.1_OU_FRESHNESS_BLEND",
    "uncertainty_model": UNCERTAINTY_MODEL,
    "calibration_split": "VALIDATION",
    "test_used": False,
    "attempt_directory": str(ATTEMPT_DIR),
    "latest_pointer": str(LATEST_POINTER_PATH),
    "uncertainty_spec": str(UNCERTAINTY_SPEC_PATH),
    "audit_manifest": str(AUDIT_PATH),
    "audit_sha256": AUDIT_SHA256,
}

TRAINING_MANIFEST["status"] = (
    "V61_UNCERTAINTY_CALIBRATED_VALIDATION_ONLY"
    if CELL_56_PASS
    else "V61_UNCERTAINTY_CALIBRATION_FAILED"
)

atomic_json_write(TRAINING_MANIFEST, TRAINING_MANIFEST_PATH)


# --------------------------------------------------------------------------------------------------
# 17. DOWNSTREAM CONTRACT — REQUIRED BY STAGE 19 / RELEASE BUNDLE
# --------------------------------------------------------------------------------------------------

V61_U1_VALIDATION = CAL
FINAL_U1_SPEC = V61_UNCERTAINTY_SPEC
FINAL_U1_B_M = V61_U1_B_M
FINAL_U1_B_W = V61_U1_B_W
FINAL_U1_G_M = V61_U1_G_M
FINAL_U1_G_W = V61_U1_G_W
FINAL_U1_NU = V61_U1_NU
FINAL_U1_QUANTILES = QUANTILES
V61_U1_ATTEMPT_DIR = ATTEMPT_DIR
V61_U1_SPEC_PATH = UNCERTAINTY_SPEC_PATH
V61_U1_AUDIT_PATH = AUDIT_PATH
NOTEBOOK_2_CELL_56_PASS = CELL_56_PASS


# --------------------------------------------------------------------------------------------------
# 18. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 175)
print("V6.1 — FINAL U1 UNCERTAINTY CALIBRATION — VALIDATION ONLY")
print("=" * 175)

print()
print("STAGE IMPLEMENTATION")
print("-" * 175)
print(f"Implementation                  : {STAGE_IMPLEMENTATION}")
print("Legacy kernel recovery          : REMOVED")
print("Legacy TEST uncertainty input   : NOT REQUIRED")
print(f"TEST used for fitting           : {TEST_USED}")

print()
print("CALIBRATION POPULATION")
print("-" * 175)
print(f"Source rows                     : {SOURCE_ROW_COUNT:,}")
print(f"Calibration rows                : {len(CAL):,}")
print(f"Calibration athletes            : {CAL['aaAthleteId'].nunique():,}")
print(f"Date range                      : {CAL['result_date_dt'].min()} -> {CAL['result_date_dt'].max()}")
print(f"Mean state gap                  : {CAL['state_gap_days'].mean():.3f} days")
print(f"Median state gap                : {CAL['state_gap_days'].median():.3f} days")

print()
print("FINAL U1 SPECIFICATION")
print("-" * 175)
print(f"B_M                             : {V61_U1_B_M:.12f}")
print(f"B_W                             : {V61_U1_B_W:.12f}")
print(f"G_M                             : {V61_U1_G_M:.12f}")
print(f"G_W                             : {V61_U1_G_W:.12f}")
print(f"nu                              : {V61_U1_NU:.12f}")
print(f"Validation NLL                  : {NEW_NLL:.6f}")

print()
print("V6.1 U1 VALIDATION METRICS")
print("-" * 175)
display(OVERALL_METRICS)

print()
print("VALIDATION METRICS BY SEX")
print("-" * 175)
display(SEX_METRICS)

print()
print("VALIDATION METRICS BY STATE GAP")
print("-" * 175)
display(GAP_METRICS)

print()
print("AUDIT / VERSIONING")
print("-" * 175)
print(f"Attempt ID                      : {ATTEMPT_ID}")
print(f"Immutable attempt directory     : {ATTEMPT_DIR}")
print(f"Uncertainty specification       : {UNCERTAINTY_SPEC_PATH}")
print(f"Audit manifest                  : {AUDIT_PATH}")
print(f"Audit SHA256                    : {AUDIT_SHA256}")
print(f"Git commit                      : {GIT_COMMIT}")

print()
print("QA")
print("-" * 175)
print(f"VALIDATION only                 : {VALIDATION_ONLY_PASS}")
print(f"TRAIN-seen population only      : {TRAIN_SEEN_ONLY_PASS}")
print(f"No missing state gaps           : {NO_MISSING_GAPS_PASS}")
print(f"Positive state gaps             : {POSITIVE_GAPS_PASS}")
print(f"Sex domain M/W                  : {SEX_DOMAIN_PASS}")
print(f"Finite calibration inputs       : {FINITE_INPUT_PASS}")
print(f"V6.1 error reconciliation       : {V61_ERROR_RECONCILIATION_PASS}")
print(f"Optimizer success               : {OPTIMIZER_PASS}")
print(f"Parameters finite               : {PARAMETERS_FINITE_PASS}")
print(f"Parameter domain valid          : {PARAMETER_DOMAIN_PASS}")
print(f"Positive prediction variance    : {POSITIVE_VARIANCE_PASS}")
print(f"Finite uncertainty outputs      : {FINITE_OUTPUT_PASS}")
print(f"TEST untouched                  : {TEST_UNTOUCHED_PASS}")

print()
print("=" * 175)
if CELL_56_PASS:
    print("CELL 56 STATUS: V6.1 U1 UNCERTAINTY CALIBRATION PASSED")
else:
    print("CELL 56 STATUS: V6.1 U1 UNCERTAINTY CALIBRATION REVIEW REQUIRED")
print("=" * 175)



V6.1 — FINAL U1 UNCERTAINTY CALIBRATION — VALIDATION ONLY

STAGE IMPLEMENTATION
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Implementation                  : NB2_STAGE18_SELF_CONTAINED_v1.0.0
Legacy kernel recovery          : REMOVED
Legacy TEST uncertainty input   : NOT REQUIRED
TEST used for fitting           : False

CALIBRATION POPULATION
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Source rows                     : 4,283
Calibration rows                : 4,283
Calibration athletes            : 2,299
Date range                      : 2026-01-04 00:00:00 -> 2026-04-30 00:00:00
Mean state gap                  : 154.911 days
Median state gap                : 106.000 days

FINAL U1 SPECIFICATION
-------------------------------

,n,athletes,standardized_error_mean,standardized_error_sd,prediction_sigma_mean,prediction_sigma_median,coverage_50,mean_width_50,coverage_80,mean_width_80,coverage_95,mean_width_95
0,4283,2299,0.040962,1.000365,3.279456,3.340088,48.890964,4.228607,79.687135,8.195072,95.470465,13.009707



VALIDATION METRICS BY SEX
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,sex_group,n,athletes,standardized_error_mean,standardized_error_sd,mean_sigma,coverage_50,mean_width_50,coverage_80,mean_width_80,coverage_95,mean_width_95
0,M,2144,1202,0.052903,0.999870,3.621251,49.347015,4.669325,79.570896,9.049188,95.195896,14.365619
1,W,2139,1097,0.028992,1.000952,2.936862,48.433848,3.786858,79.803647,7.338961,95.745676,11.650627



VALIDATION METRICS BY STATE GAP
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,state_gap_band,n,athletes,mean_gap_days,standardized_error_mean,standardized_error_sd,mean_sigma,coverage_50,mean_width_50,coverage_80,mean_width_80,coverage_95,mean_width_95
0,01-30,1559,828,13.521488,0.130753,0.986454,2.995784,49.518922,3.862834,78.960872,7.486201,95.830661,11.884372
1,31-60,349,308,40.656160,0.081978,0.984502,3.059455,50.716332,3.944933,82.808023,7.645310,94.842407,12.136958
2,61-90,168,162,70.422619,-0.053657,0.977341,3.118871,48.214286,4.021545,77.976190,7.793785,95.833333,12.372662
3,91-120,126,121,105.214286,0.062260,0.917422,3.169032,42.857143,4.086224,82.539683,7.919133,99.206349,12.571653
4,121-180,432,428,155.268519,-0.070818,1.027269,3.323666,48.842593,4.285613,78.009259,8.305550,94.907407,13.185091
5,181-365,1410,1410,255.078723,0.009148,1.010030,3.497278,48.156028,4.509472,80.212766,8.739391,95.248227,13.873816
6,366-730,179,179,565.513966,-0.091118,0.999291,4.026268,51.955307,5.191564,81.005587,10.061291,94.413408,15.972336
7,731-1825,55,55,1083.927273,-0.334106,1.042692,4.780403,45.454545,6.163964,76.363636,11.945809,92.727273,18.964015
8,1826+,5,5,3109.000000,-0.692691,0.900888,6.770738,40.000000,8.730347,60.000000,16.919479,100.000000,26.859736



AUDIT / VERSIONING
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Attempt ID                      : 20260922T141159_712837Z
Immutable attempt directory     : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/uncertainty_calibration/20260922T141159_712837Z
Uncertainty specification       : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/uncertainty_calibration/20260922T141159_712837Z/v61_uncertainty_spec.json
Audit manifest                  : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/uncertainty_calibration/20260922T141159_712837Z/v61_uncertainty_calibration_audit.json
Audit SHA256                    : 4f57eefe2d0cdc572911497c958fb5b0d818937972e95c1021a58f3f21e57d20
Git commit  

## Stage 19 — Final TEST, POST_TEST and release gates

Evaluate the frozen V6.1 + U1 release on 2026 TEST and POST_TEST, then freeze final release gates and known limitations.

**Legacy source cells represented:** 51, 52, 53  
**Execution rule:** run only after the preceding stage has completed successfully.


In [20]:
# ==================================================================================================
# REFACTORED STAGE 19 — Final TEST, POST_TEST and release gates
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 51
# Original source SHA256: dc36d9703aa4aebd8fbe6a46a87da39c2bae022b561222c13ce60d341b1c2b00
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 57 — FROZEN V6.1 + U1 UNCERTAINTY — 2026 TEST EVALUATION
# ==================================================================================================
#
# Purpose
# -------
# Apply the now-frozen:
#
#       V6.1 mean architecture
#       +
#       U1_SEX_LINEAR uncertainty specification
#
# to the current 2026 TEST period.
#
# NOTHING is fitted in this cell.
#
# Important interpretation
# ------------------------
# The 2026 TEST period:
#
#   • was NOT used to fit the U1 uncertainty parameters
#   • therefore provides out-of-sample evidence for uncertainty calibration
#
# BUT:
#
#   • V6.1 was originally conceived after inspecting 2026 TEST mean behavior
#   • therefore this period is NOT treated as independent mean-architecture evidence
#
# The subsequent POST_TEST period will provide the next forward operational check.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_56_PASS",

    "FINAL_U1_SPEC",
    "FINAL_U1_B_M",
    "FINAL_U1_B_W",
    "FINAL_U1_G_M",
    "FINAL_U1_G_W",
    "FINAL_U1_NU",

    "V61_U1_SPEC_PATH",
    "V61_U1_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",

    "ARCHITECTURE_DECISION_ARTIFACT",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_56_PASS:

    raise RuntimeError(
        "Cell 56 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_57_V61_U1_2026_TEST_EVALUATION"

STAGE_SCHEMA_VERSION = "1.0.0"


TEST_START = pd.Timestamp(
    "2026-05-01"
)


TEST_END = pd.Timestamp(
    "2026-08-30"
)


LEVELS_LOCAL = [
    0.50,
    0.80,
    0.95,
]


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(
        path
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def find_tau_days(
    obj,
):

    candidate_keys = {
        "tau",
        "tau_days",
        "state_tau",
        "state_tau_days",
        "ou_tau",
        "ou_tau_days",
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if (
                str(
                    key
                ).lower()
                in candidate_keys

                and

                isinstance(
                    value,
                    (int, float),
                )
            ):

                return float(
                    value
                )


        for value in obj.values():

            result = find_tau_days(
                value
            )

            if result is not None:

                return result


    return None


def wilson_interval_local(
    successes,
    n,
    z=1.959963984540054,
):

    if n <= 0:

        return (
            np.nan,
            np.nan,
        )


    p = (
        successes
        /
        n
    )


    denominator = (
        1.0
        +
        z**2
        /
        n
    )


    centre = (
        p
        +
        z**2
        /
        (
            2.0
            *
            n
        )
    )


    adjustment = (
        z
        *
        np.sqrt(
            (
                p
                *
                (
                    1.0
                    -
                    p
                )
                /
                n
            )
            +
            (
                z**2
                /
                (
                    4.0
                    *
                    n**2
                )
            )
        )
    )


    low = (
        centre
        -
        adjustment
    ) / denominator


    high = (
        centre
        +
        adjustment
    ) / denominator


    return (
        float(
            low
        ),
        float(
            high
        ),
    )


def variance_standardized_t_nll(
    error,
    variance,
    nu,
):

    error = np.asarray(
        error,
        dtype=float,
    )


    variance = np.asarray(
        variance,
        dtype=float,
    )


    scale = (
        np.sqrt(
            variance
        )

        *

        np.sqrt(
            (
                nu
                -
                2.0
            )
            /
            nu
        )
    )


    logp = student_t.logpdf(
        error,
        df=nu,
        loc=0.0,
        scale=scale,
    )


    return float(
        -
        np.sum(
            logp
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD CURRENT TEST PREDICTION SOURCE
# --------------------------------------------------------------------------------------------------

RUN_DIR = Path(
    TRAINING_RUN_DIR
)


TEST_SOURCE_PATH = (
    RUN_DIR
    /
    "v6_test_predictions.parquet"
)


if not TEST_SOURCE_PATH.exists():

    raise RuntimeError(
        f"Missing current TEST prediction source:\n"
        f"{TEST_SOURCE_PATH}"
    )


TEST = pd.read_parquet(
    TEST_SOURCE_PATH
)


SOURCE_ROWS = int(
    len(
        TEST
    )
)


# --------------------------------------------------------------------------------------------------
# 5. NORMALIZE DATE
# --------------------------------------------------------------------------------------------------

TEST[
    "result_date_dt"
] = pd.to_datetime(
    TEST[
        "result_date_dt"
    ],
    errors="coerce",
)


# --------------------------------------------------------------------------------------------------
# 6. RESTRICT TO TEST WINDOW
# --------------------------------------------------------------------------------------------------

if "prediction_split" in TEST.columns:

    TEST = (
        TEST.loc[
            TEST[
                "prediction_split"
            ]
            .astype(str)
            .eq(
                "TEST"
            )
        ]
        .copy()
    )


TEST = (
    TEST.loc[
        TEST[
            "result_date_dt"
        ]
        .between(
            TEST_START,
            TEST_END,
            inclusive="both",
        )
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 7. RESTRICT TO TRAIN-SEEN / ESTABLISHED ATHLETES
# --------------------------------------------------------------------------------------------------

if "athlete_seen_in_training" not in TEST.columns:

    raise RuntimeError(
        "TEST source is missing athlete_seen_in_training."
    )


TEST = (
    TEST.loc[
        TEST[
            "athlete_seen_in_training"
        ]
        .eq(
            True
        )
    ]
    .copy()
)


# U1 was calibrated only where a V5 state gap exists.
TEST = (
    TEST.loc[
        TEST[
            "state_gap_days"
        ]
        .notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 8. REQUIRED TEST COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",

    "model_group",
    "sex_group",

    "state_gap_days",

    "v5_prediction_mean",
    "v6_prediction_mean",

    "athlete_seen_in_training",
]


missing_columns = [
    column
    for column in required_columns
    if column not in TEST.columns
]


if missing_columns:

    raise RuntimeError(
        "TEST source is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. LOAD TRAIN-ONLY V5 OU TAU
# --------------------------------------------------------------------------------------------------

V5_PARAMETER_PATH = (
    RUN_DIR
    /
    "v5_ou_parameters.json"
)


if not V5_PARAMETER_PATH.exists():

    raise RuntimeError(
        f"Missing V5 OU parameter artifact:\n"
        f"{V5_PARAMETER_PATH}"
    )


with V5_PARAMETER_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_JSON = json.load(
        f
    )


V5_TAU_DAYS_FROZEN = find_tau_days(
    V5_PARAMETER_JSON
)


if (
    V5_TAU_DAYS_FROZEN is None
    or
    not np.isfinite(
        V5_TAU_DAYS_FROZEN
    )
    or
    V5_TAU_DAYS_FROZEN <= 0
):

    raise RuntimeError(
        "Could not recover positive finite V5 OU tau."
    )


# --------------------------------------------------------------------------------------------------
# 10. RECONSTRUCT FROZEN V6.1 MEAN
# --------------------------------------------------------------------------------------------------

TEST[
    "v5_state_persistence_phi"
] = np.exp(
    -
    TEST[
        "state_gap_days"
    ]
    .astype(float)
    /
    V5_TAU_DAYS_FROZEN
)


TEST[
    "v61_structural_weight"
] = (
    1.0
    -
    TEST[
        "v5_state_persistence_phi"
    ]
)


TEST[
    "full_v6_adjustment"
] = (
    TEST[
        "v6_prediction_mean"
    ]
    -
    TEST[
        "v5_prediction_mean"
    ]
)


TEST[
    "v61_applied_adjustment"
] = (
    TEST[
        "v61_structural_weight"
    ]
    *
    TEST[
        "full_v6_adjustment"
    ]
)


TEST[
    "v61_prediction_mean"
] = (
    TEST[
        "v5_prediction_mean"
    ]
    +
    TEST[
        "v61_applied_adjustment"
    ]
)


TEST[
    "v61_error"
] = (
    TEST[
        "mark_numeric_m"
    ]
    -
    TEST[
        "v61_prediction_mean"
    ]
)


TEST[
    "v61_abs_error"
] = (
    TEST[
        "v61_error"
    ]
    .abs()
)


TEST[
    "v61_sq_error"
] = (
    TEST[
        "v61_error"
    ]
    **
    2
)


# --------------------------------------------------------------------------------------------------
# 11. APPLY FROZEN U1 SPECIFICATION
# --------------------------------------------------------------------------------------------------

B_M = float(
    FINAL_U1_B_M
)


B_W = float(
    FINAL_U1_B_W
)


G_M = float(
    FINAL_U1_G_M
)


G_W = float(
    FINAL_U1_G_W
)


NU = float(
    FINAL_U1_NU
)


TEST[
    "horizon_years"
] = (
    TEST[
        "state_gap_days"
    ]
    .astype(float)
    /
    365.25
)


TEST[
    "v61_prediction_variance"
] = np.where(
    TEST[
        "sex_group"
    ]
    .astype(str)
    .eq(
        "M"
    ),

    B_M
    +
    G_M
    *
    TEST[
        "horizon_years"
    ],

    B_W
    +
    G_W
    *
    TEST[
        "horizon_years"
    ],
)


TEST[
    "v61_prediction_sigma"
] = np.sqrt(
    TEST[
        "v61_prediction_variance"
    ]
)


TEST[
    "v61_standardized_error"
] = (
    TEST[
        "v61_error"
    ]
    /
    TEST[
        "v61_prediction_sigma"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. FROZEN STUDENT-t INTERVALS
# --------------------------------------------------------------------------------------------------

QUANTILES = {}


for level in LEVELS_LOCAL:

    pct = int(
        100
        *
        level
    )


    q = float(
        student_t.ppf(
            (
                1.0
                +
                level
            )
            /
            2.0,
            df=NU,
        )

        *

        np.sqrt(
            (
                NU
                -
                2.0
            )
            /
            NU
        )
    )


    QUANTILES[
        level
    ] = q


    half_width = (
        q
        *
        TEST[
            "v61_prediction_sigma"
        ]
    )


    TEST[
        f"v61_lower_{pct}"
    ] = (
        TEST[
            "v61_prediction_mean"
        ]
        -
        half_width
    )


    TEST[
        f"v61_upper_{pct}"
    ] = (
        TEST[
            "v61_prediction_mean"
        ]
        +
        half_width
    )


    TEST[
        f"v61_width_{pct}"
    ] = (
        2.0
        *
        half_width
    )


    TEST[
        f"v61_covered_{pct}"
    ] = (
        TEST[
            "mark_numeric_m"
        ]
        .between(
            TEST[
                f"v61_lower_{pct}"
            ],
            TEST[
                f"v61_upper_{pct}"
            ],
            inclusive="both",
        )
    )


# --------------------------------------------------------------------------------------------------
# 13. GAP BANDS
# --------------------------------------------------------------------------------------------------

TEST[
    "state_gap_band"
] = pd.cut(
    TEST[
        "state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 14. METRIC FUNCTION
# --------------------------------------------------------------------------------------------------

def uncertainty_summary(
    df,
):

    row = {
        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "mean_error":
            float(
                df[
                    "v61_error"
                ]
                .mean()
            ),

        "MAE":
            float(
                df[
                    "v61_abs_error"
                ]
                .mean()
            ),

        "RMSE":
            float(
                np.sqrt(
                    df[
                        "v61_sq_error"
                    ]
                    .mean()
                )
            ),

        "standardized_error_mean":
            float(
                df[
                    "v61_standardized_error"
                ]
                .mean()
            ),

        "standardized_error_sd":
            float(
                df[
                    "v61_standardized_error"
                ]
                .std()
            ),

        "prediction_sigma_mean":
            float(
                df[
                    "v61_prediction_sigma"
                ]
                .mean()
            ),

        "prediction_sigma_median":
            float(
                df[
                    "v61_prediction_sigma"
                ]
                .median()
            ),
    }


    for level in LEVELS_LOCAL:

        pct = int(
            100
            *
            level
        )


        covered = (
            df[
                f"v61_covered_{pct}"
            ]
        )


        successes = int(
            covered.sum()
        )


        n = int(
            len(
                covered
            )
        )


        ci_low, ci_high = (
            wilson_interval_local(
                successes,
                n,
            )
        )


        observed = float(
            covered.mean()
        )


        row[
            f"coverage_{pct}"
        ] = (
            100.0
            *
            observed
        )


        row[
            f"coverage_{pct}_CI_low"
        ] = (
            100.0
            *
            ci_low
        )


        row[
            f"coverage_{pct}_CI_high"
        ] = (
            100.0
            *
            ci_high
        )


        row[
            f"target_{pct}_inside_CI"
        ] = bool(
            ci_low
            <=
            level
            <=
            ci_high
        )


        row[
            f"coverage_{pct}_error_pp"
        ] = float(
            100.0
            *
            (
                observed
                -
                level
            )
        )


        row[
            f"mean_width_{pct}"
        ] = float(
            df[
                f"v61_width_{pct}"
            ]
            .mean()
        )


    return pd.Series(
        row
    )


# --------------------------------------------------------------------------------------------------
# 15. OVERALL TEST METRICS
# --------------------------------------------------------------------------------------------------

OVERALL_METRICS = (
    uncertainty_summary(
        TEST
    )
    .to_frame()
    .T
)


# --------------------------------------------------------------------------------------------------
# 16. BY SEX
# --------------------------------------------------------------------------------------------------

SEX_METRICS = (
    TEST.groupby(
        "sex_group",
        sort=True,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 17. BY MODEL GROUP
# --------------------------------------------------------------------------------------------------

GROUP_METRICS = (
    TEST.groupby(
        "model_group",
        sort=True,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 18. BY STATE GAP
# --------------------------------------------------------------------------------------------------

GAP_METRICS = (
    TEST.groupby(
        "state_gap_band",
        observed=True,
        sort=False,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


GAP_METRICS.insert(
    3,
    "mean_gap_days",
    [
        float(
            TEST.loc[
                TEST[
                    "state_gap_band"
                ]
                .eq(
                    band
                ),
                "state_gap_days",
            ]
            .mean()
        )

        for band
        in GAP_METRICS[
            "state_gap_band"
        ]
    ],
)


# --------------------------------------------------------------------------------------------------
# 19. TEST NEGATIVE LOG-LIKELIHOOD
# --------------------------------------------------------------------------------------------------

TEST_NLL = variance_standardized_t_nll(
    TEST[
        "v61_error"
    ],
    TEST[
        "v61_prediction_variance"
    ],
    NU,
)


TEST_NLL_PER_ROW = float(
    TEST_NLL
    /
    len(
        TEST
    )
)


# --------------------------------------------------------------------------------------------------
# 20. TEST QA
# --------------------------------------------------------------------------------------------------

TEST_DATE_PASS = bool(
    TEST[
        "result_date_dt"
    ]
    .between(
        TEST_START,
        TEST_END,
        inclusive="both",
    )
    .all()
)


TRAIN_SEEN_PASS = bool(
    TEST[
        "athlete_seen_in_training"
    ]
    .eq(
        True
    )
    .all()
)


NO_MISSING_GAP_PASS = bool(
    TEST[
        "state_gap_days"
    ]
    .notna()
    .all()
)


POSITIVE_GAP_PASS = bool(
    (
        TEST[
            "state_gap_days"
        ]
        .astype(float)
        >
        0
    )
    .all()
)


SEX_DOMAIN_PASS = bool(
    set(
        TEST[
            "sex_group"
        ]
        .astype(str)
        .unique()
    )
    .issubset(
        {
            "M",
            "W",
        }
    )
)


WEIGHT_BOUNDS_PASS = bool(
    TEST[
        "v61_structural_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        TEST[
            "v61_prediction_variance"
        ]
        >
        0
    )
    .all()
)


FINITE_OUTPUT_PASS = bool(
    np.isfinite(
        TEST[
            [
                "v61_prediction_mean",
                "v61_error",
                "v61_prediction_variance",
                "v61_prediction_sigma",
                "v61_standardized_error",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


NO_PARAMETER_FIT_PASS = True


UNCERTAINTY_SPEC_FROZEN_PASS = bool(
    Path(
        V61_U1_SPEC_PATH
    ).exists()
)


MEAN_DECISION_FROZEN_PASS = bool(
    Path(
        ARCHITECTURE_DECISION_ARTIFACT
    ).exists()
)


CELL_57_PASS = bool(
    len(
        TEST
    )
    >
    0
    and
    TEST_DATE_PASS
    and
    TRAIN_SEEN_PASS
    and
    NO_MISSING_GAP_PASS
    and
    POSITIVE_GAP_PASS
    and
    SEX_DOMAIN_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    FINITE_OUTPUT_PASS
    and
    NO_PARAMETER_FIT_PASS
    and
    UNCERTAINTY_SPEC_FROZEN_PASS
    and
    MEAN_DECISION_FROZEN_PASS
)


# --------------------------------------------------------------------------------------------------
# 21. IMMUTABLE EVALUATION DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


EVAL_ROOT = (
    RUN_DIR
    /
    "final_evaluation"
    /
    "test_uncertainty"
)


ATTEMPT_DIR = (
    EVAL_ROOT
    /
    ATTEMPT_ID
)


ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 22. SAVE TEST ARTIFACTS
# --------------------------------------------------------------------------------------------------

TEST_PREDICTIONS_PATH = (
    ATTEMPT_DIR
    /
    "v61_test_predictions_with_uncertainty.parquet"
)


OVERALL_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "test_uncertainty_overall.parquet"
)


SEX_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "test_uncertainty_by_sex.parquet"
)


GROUP_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "test_uncertainty_by_model_group.parquet"
)


GAP_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "test_uncertainty_by_gap.parquet"
)


AUDIT_PATH = (
    ATTEMPT_DIR
    /
    "test_uncertainty_evaluation_audit.json"
)


TEST.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)


OVERALL_METRICS.to_parquet(
    OVERALL_METRICS_PATH,
    index=False,
)


SEX_METRICS.to_parquet(
    SEX_METRICS_PATH,
    index=False,
)


GROUP_METRICS.to_parquet(
    GROUP_METRICS_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 23. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {

    "test_prediction_source":
        TEST_SOURCE_PATH,

    "v5_ou_parameters":
        V5_PARAMETER_PATH,

    "v61_uncertainty_spec":
        Path(
            V61_U1_SPEC_PATH
        ),

    "v61_uncertainty_calibration_audit":
        Path(
            V61_U1_AUDIT_PATH
        ),

    "mean_architecture_decision":
        Path(
            ARCHITECTURE_DECISION_ARTIFACT
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else None
            ),
    }


output_paths = {

    "test_predictions_with_uncertainty":
        TEST_PREDICTIONS_PATH,

    "overall_metrics":
        OVERALL_METRICS_PATH,

    "sex_metrics":
        SEX_METRICS_PATH,

    "model_group_metrics":
        GROUP_METRICS_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 24. AUDIT
# --------------------------------------------------------------------------------------------------

AUDIT = {

    "audit_schema_version":
        STAGE_SCHEMA_VERSION,

    "stage":
        "V61_U1_2026_TEST_EVALUATION",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_57_PASS
            else
            "FAILED"
        ),

    "evaluation_window": {
        "start":
            str(
                TEST_START.date()
            ),

        "end":
            str(
                TEST_END.date()
            ),
    },

    "population":
        "TRAIN_SEEN_ATHLETES_WITH_STATE_HISTORY",

    "mean_architecture":
        "V6.1_OU_FRESHNESS_BLEND",

    "uncertainty_model":
        "U1_SEX_LINEAR",

    "V5_tau_days":
        float(
            V5_TAU_DAYS_FROZEN
        ),

    "U1_parameters": {
        "B_M":
            B_M,

        "B_W":
            B_W,

        "G_M":
            G_M,

        "G_W":
            G_W,

        "nu":
            NU,
    },

    "test_role": {
        "used_to_fit_U1":
            False,

        "used_for_U1_model_selection":
            False,

        "independent_for_uncertainty_parameters":
            True,

        "independent_for_mean_architecture":
            False,

        "mean_architecture_note":
            (
                "V6.1 was conceived after inspection of the 2026 TEST "
                "mean-model behavior, so this period is not treated as "
                "independent evidence for mean-architecture selection."
            ),
    },

    "rows": {
        "source":
            SOURCE_ROWS,

        "evaluated":
            int(
                len(
                    TEST
                )
            ),

        "athletes":
            int(
                TEST[
                    "aaAthleteId"
                ]
                .nunique()
            ),
    },

    "test_NLL":
        TEST_NLL,

    "test_NLL_per_row":
        TEST_NLL_PER_ROW,

    "overall_metrics":
        OVERALL_METRICS.iloc[
            0
        ].to_dict(),

    "qa": {
        "test_dates":
            TEST_DATE_PASS,

        "train_seen_only":
            TRAIN_SEEN_PASS,

        "no_missing_state_gaps":
            NO_MISSING_GAP_PASS,

        "positive_state_gaps":
            POSITIVE_GAP_PASS,

        "sex_domain":
            SEX_DOMAIN_PASS,

        "V61_weight_bounds":
            WEIGHT_BOUNDS_PASS,

        "positive_variance":
            POSITIVE_VARIANCE_PASS,

        "finite_output":
            FINITE_OUTPUT_PASS,

        "no_parameter_fit":
            NO_PARAMETER_FIT_PASS,

        "uncertainty_spec_frozen":
            UNCERTAINTY_SPEC_FROZEN_PASS,

        "mean_decision_frozen":
            MEAN_DECISION_FROZEN_PASS,

        "cell_pass":
            CELL_57_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    AUDIT,
    AUDIT_PATH,
)


AUDIT_SHA256 = sha256_file(
    AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 25. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    EVAL_ROOT
    /
    "LATEST_TEST_UNCERTAINTY_EVALUATION.json"
)


atomic_json_write(
    {
        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_57_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                ATTEMPT_DIR
            ),

        "test_predictions":
            str(
                TEST_PREDICTIONS_PATH
            ),

        "overall_metrics":
            str(
                OVERALL_METRICS_PATH
            ),

        "audit_manifest":
            str(
                AUDIT_PATH
            ),

        "audit_sha256":
            AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 26. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST.setdefault(
    "final_evaluation",
    {},
)


TRAINING_MANIFEST[
    "final_evaluation"
][
    "test_uncertainty"
] = {

    "status":
        (
            "PASSED"
            if CELL_57_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "test_predictions":
        str(
            TEST_PREDICTIONS_PATH
        ),

    "overall_metrics":
        str(
            OVERALL_METRICS_PATH
        ),

    "audit_manifest":
        str(
            AUDIT_PATH
        ),

    "audit_sha256":
        AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "V61_U1_TEST_EVALUATION_COMPLETE"
    if CELL_57_PASS
    else
    "V61_U1_TEST_EVALUATION_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 27. GLOBALS
# --------------------------------------------------------------------------------------------------

FINAL_V61_TEST_UNCERTAINTY = (
    TEST
)


FINAL_V61_TEST_UNCERTAINTY_OVERALL = (
    OVERALL_METRICS
)


FINAL_V61_TEST_UNCERTAINTY_BY_SEX = (
    SEX_METRICS
)


FINAL_V61_TEST_UNCERTAINTY_BY_GROUP = (
    GROUP_METRICS
)


FINAL_V61_TEST_UNCERTAINTY_BY_GAP = (
    GAP_METRICS
)


FINAL_V61_TEST_UNCERTAINTY_ATTEMPT_DIR = (
    ATTEMPT_DIR
)


FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH = (
    AUDIT_PATH
)


NOTEBOOK_2_CELL_57_PASS = (
    CELL_57_PASS
)


# --------------------------------------------------------------------------------------------------
# 28. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 180)
print("V6.1 + U1 — FROZEN 2026 TEST UNCERTAINTY EVALUATION")
print("=" * 180)


print()
print("TEST POPULATION")
print("-" * 180)

print(
    f"Source TEST rows                : "
    f"{SOURCE_ROWS:,}"
)

print(
    f"Established/state-history rows  : "
    f"{len(TEST):,}"
)

print(
    f"Athletes                        : "
    f"{TEST['aaAthleteId'].nunique():,}"
)

print(
    f"Date range                      : "
    f"{TEST['result_date_dt'].min()}"
    f" -> "
    f"{TEST['result_date_dt'].max()}"
)

print(
    f"Mean state gap                  : "
    f"{TEST['state_gap_days'].mean():.3f} days"
)

print(
    f"Median state gap                : "
    f"{TEST['state_gap_days'].median():.3f} days"
)


print()
print("FROZEN SPECIFICATION")
print("-" * 180)

print(
    f"V5 tau                          : "
    f"{V5_TAU_DAYS_FROZEN:.9f} days"
)

print(
    f"B_M                             : "
    f"{B_M:.9f}"
)

print(
    f"B_W                             : "
    f"{B_W:.9f}"
)

print(
    f"G_M                             : "
    f"{G_M:.9f}"
)

print(
    f"G_W                             : "
    f"{G_W:.9f}"
)

print(
    f"nu                              : "
    f"{NU:.9f}"
)


print()
print("OVERALL TEST UNCERTAINTY")
print("-" * 180)

display(
    OVERALL_METRICS
)


print()
print("BY SEX")
print("-" * 180)

display(
    SEX_METRICS
)


print()
print("BY MODEL GROUP")
print("-" * 180)

display(
    GROUP_METRICS
)


print()
print("BY STATE GAP")
print("-" * 180)

display(
    GAP_METRICS
)


print()
print("TEST LOG-LIKELIHOOD")
print("-" * 180)

print(
    f"Total NLL                       : "
    f"{TEST_NLL:.6f}"
)

print(
    f"NLL per row                     : "
    f"{TEST_NLL_PER_ROW:.6f}"
)


print()
print("COVERAGE INTERPRETATION")
print("-" * 180)

overall = (
    OVERALL_METRICS.iloc[
        0
    ]
)


for level in LEVELS_LOCAL:

    pct = int(
        100
        *
        level
    )

    print(
        f"{pct:>2}% target | "
        f"observed = "
        f"{overall[f'coverage_{pct}']:.3f}% | "
        f"95% Wilson CI = "
        f"["
        f"{overall[f'coverage_{pct}_CI_low']:.3f}%, "
        f"{overall[f'coverage_{pct}_CI_high']:.3f}%"
        f"] | "
        f"target inside CI = "
        f"{overall[f'target_{pct}_inside_CI']}"
    )


print()
print("AUDIT / VERSIONING")
print("-" * 180)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable evaluation directory  : "
    f"{ATTEMPT_DIR}"
)

print(
    f"Audit manifest                  : "
    f"{AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 180)

print(
    f"TEST dates only                 : "
    f"{TEST_DATE_PASS}"
)

print(
    f"TRAIN-seen athletes only        : "
    f"{TRAIN_SEEN_PASS}"
)

print(
    f"No missing state gaps           : "
    f"{NO_MISSING_GAP_PASS}"
)

print(
    f"Positive state gaps             : "
    f"{POSITIVE_GAP_PASS}"
)

print(
    f"Sex domain M/W                  : "
    f"{SEX_DOMAIN_PASS}"
)

print(
    f"V6.1 weights valid              : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"Positive predictive variance    : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Finite uncertainty outputs      : "
    f"{FINITE_OUTPUT_PASS}"
)

print(
    f"No parameters fitted            : "
    f"{NO_PARAMETER_FIT_PASS}"
)

print(
    f"Frozen U1 specification found   : "
    f"{UNCERTAINTY_SPEC_FROZEN_PASS}"
)

print(
    f"Frozen mean decision found      : "
    f"{MEAN_DECISION_FROZEN_PASS}"
)


print()
print("=" * 180)

if CELL_57_PASS:

    print(
        "CELL 57 STATUS: V6.1 + U1 2026 TEST EVALUATION PASSED"
    )

else:

    print(
        "CELL 57 STATUS: TEST UNCERTAINTY EVALUATION REVIEW REQUIRED"
    )

print("=" * 180)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 52
# Original source SHA256: c1e82d1fc37d7805c6841a54aa3c3495350a140f509c41d5eb174856a2df8a35
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 58 — FROZEN V6.1 + U1 — POST_TEST OPERATIONAL MONITOR
# ==================================================================================================
#
# Purpose
# -------
# Apply the fully frozen:
#
#       V6.1 mean architecture
#       +
#       U1_SEX_LINEAR uncertainty specification
#
# to POST_TEST observations from 2026-08-31 onward.
#
# NO fitting.
# NO parameter changes.
# NO architecture changes.
#
# Interpretation
# --------------
# POST_TEST is treated as an operational monitoring period.
#
# It is useful for:
#   • checking forward stability
#   • detecting subgroup calibration problems
#   • checking whether TEST findings persist
#
# It is NOT used to refit V6.1 or U1 in this model version.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_57_PASS",

    "FINAL_U1_B_M",
    "FINAL_U1_B_W",
    "FINAL_U1_G_M",
    "FINAL_U1_G_W",
    "FINAL_U1_NU",

    "V61_U1_SPEC_PATH",
    "V61_U1_AUDIT_PATH",

    "FINAL_V61_TEST_UNCERTAINTY_OVERALL",
    "FINAL_V61_TEST_UNCERTAINTY_BY_SEX",
    "FINAL_V61_TEST_UNCERTAINTY_BY_GROUP",
    "FINAL_V61_TEST_UNCERTAINTY_BY_GAP",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",

    "ARCHITECTURE_DECISION_ARTIFACT",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_57_PASS:

    raise RuntimeError(
        "Cell 57 must pass first."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_58_V61_U1_POST_TEST_MONITOR"

STAGE_SCHEMA_VERSION = "1.0.0"


POST_TEST_START = pd.Timestamp(
    "2026-08-31"
)


LEVELS_LOCAL = [
    0.50,
    0.80,
    0.95,
]


GAP_BINS = [
    0,
    30,
    60,
    90,
    120,
    180,
    365,
    730,
    1825,
    np.inf,
]


GAP_LABELS = [
    "01-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    "366-730",
    "731-1825",
    "1826+",
]


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(
        path
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(
                start_path
            ),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def find_tau_days(
    obj,
):

    candidate_keys = {
        "tau",
        "tau_days",
        "state_tau",
        "state_tau_days",
        "ou_tau",
        "ou_tau_days",
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if (
                str(
                    key
                ).lower()
                in candidate_keys

                and

                isinstance(
                    value,
                    (int, float),
                )
            ):

                return float(
                    value
                )


        for value in obj.values():

            result = find_tau_days(
                value
            )

            if result is not None:

                return result


    return None


def wilson_interval_local(
    successes,
    n,
    z=1.959963984540054,
):

    if n <= 0:

        return (
            np.nan,
            np.nan,
        )


    p = (
        successes
        /
        n
    )


    denominator = (
        1.0
        +
        z**2
        /
        n
    )


    centre = (
        p
        +
        z**2
        /
        (
            2.0
            *
            n
        )
    )


    adjustment = (
        z
        *
        np.sqrt(
            (
                p
                *
                (
                    1.0
                    -
                    p
                )
                /
                n
            )
            +
            (
                z**2
                /
                (
                    4.0
                    *
                    n**2
                )
            )
        )
    )


    low = (
        centre
        -
        adjustment
    ) / denominator


    high = (
        centre
        +
        adjustment
    ) / denominator


    return (
        float(
            low
        ),
        float(
            high
        ),
    )


def uncertainty_summary(
    df,
):

    row = {
        "n":
            int(
                len(
                    df
                )
            ),

        "athletes":
            int(
                df[
                    "aaAthleteId"
                ]
                .nunique()
            ),

        "mean_error":
            float(
                df[
                    "v61_error"
                ]
                .mean()
            ),

        "MAE":
            float(
                df[
                    "v61_abs_error"
                ]
                .mean()
            ),

        "RMSE":
            float(
                np.sqrt(
                    df[
                        "v61_sq_error"
                    ]
                    .mean()
                )
            ),

        "standardized_error_mean":
            float(
                df[
                    "v61_standardized_error"
                ]
                .mean()
            ),

        "standardized_error_sd":
            float(
                df[
                    "v61_standardized_error"
                ]
                .std()
            ),

        "prediction_sigma_mean":
            float(
                df[
                    "v61_prediction_sigma"
                ]
                .mean()
            ),

        "prediction_sigma_median":
            float(
                df[
                    "v61_prediction_sigma"
                ]
                .median()
            ),
    }


    for level in LEVELS_LOCAL:

        pct = int(
            100
            *
            level
        )


        covered = (
            df[
                f"v61_covered_{pct}"
            ]
        )


        successes = int(
            covered.sum()
        )


        n = int(
            len(
                covered
            )
        )


        ci_low, ci_high = wilson_interval_local(
            successes,
            n,
        )


        observed = float(
            covered.mean()
        )


        row[
            f"coverage_{pct}"
        ] = (
            100.0
            *
            observed
        )


        row[
            f"coverage_{pct}_CI_low"
        ] = (
            100.0
            *
            ci_low
        )


        row[
            f"coverage_{pct}_CI_high"
        ] = (
            100.0
            *
            ci_high
        )


        row[
            f"target_{pct}_inside_CI"
        ] = bool(
            ci_low
            <=
            level
            <=
            ci_high
        )


        row[
            f"coverage_{pct}_error_pp"
        ] = float(
            100.0
            *
            (
                observed
                -
                level
            )
        )


        row[
            f"mean_width_{pct}"
        ] = float(
            df[
                f"v61_width_{pct}"
            ]
            .mean()
        )


    return pd.Series(
        row
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD POST_TEST PREDICTION SOURCE
# --------------------------------------------------------------------------------------------------

RUN_DIR = Path(
    TRAINING_RUN_DIR
)


POST_SOURCE_PATH = (
    RUN_DIR
    /
    "v6_post_test_predictions.parquet"
)


if not POST_SOURCE_PATH.exists():

    raise RuntimeError(
        f"Missing POST_TEST prediction source:\n"
        f"{POST_SOURCE_PATH}"
    )


POST = pd.read_parquet(
    POST_SOURCE_PATH
)


SOURCE_ROWS = int(
    len(
        POST
    )
)


# --------------------------------------------------------------------------------------------------
# 5. NORMALIZE / FILTER POST_TEST
# --------------------------------------------------------------------------------------------------

POST[
    "result_date_dt"
] = pd.to_datetime(
    POST[
        "result_date_dt"
    ],
    errors="coerce",
)


if "prediction_split" in POST.columns:

    POST = (
        POST.loc[
            POST[
                "prediction_split"
            ]
            .astype(str)
            .eq(
                "POST_TEST"
            )
        ]
        .copy()
    )


POST = (
    POST.loc[
        POST[
            "result_date_dt"
        ]
        >=
        POST_TEST_START
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 6. ESTABLISHED / STATE-HISTORY POPULATION
# --------------------------------------------------------------------------------------------------

if "athlete_seen_in_training" not in POST.columns:

    raise RuntimeError(
        "POST_TEST source is missing athlete_seen_in_training."
    )


POST = (
    POST.loc[
        POST[
            "athlete_seen_in_training"
        ]
        .eq(
            True
        )
    ]
    .copy()
)


POST = (
    POST.loc[
        POST[
            "state_gap_days"
        ]
        .notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "aaAthleteId",
    "result_date_dt",

    "mark_numeric_m",

    "model_group",
    "sex_group",

    "state_gap_days",

    "v5_prediction_mean",
    "v6_prediction_mean",

    "athlete_seen_in_training",
]


missing_columns = [
    column
    for column in required_columns
    if column not in POST.columns
]


if missing_columns:

    raise RuntimeError(
        "POST_TEST source missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 8. LOAD TRAIN-ONLY V5 TAU
# --------------------------------------------------------------------------------------------------

V5_PARAMETER_PATH = (
    RUN_DIR
    /
    "v5_ou_parameters.json"
)


if not V5_PARAMETER_PATH.exists():

    raise RuntimeError(
        f"Missing V5 OU parameter artifact:\n"
        f"{V5_PARAMETER_PATH}"
    )


with V5_PARAMETER_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_JSON = json.load(
        f
    )


V5_TAU_DAYS_FROZEN = find_tau_days(
    V5_PARAMETER_JSON
)


if (
    V5_TAU_DAYS_FROZEN is None
    or
    not np.isfinite(
        V5_TAU_DAYS_FROZEN
    )
    or
    V5_TAU_DAYS_FROZEN <= 0
):

    raise RuntimeError(
        "Could not recover positive finite V5 OU tau."
    )


# --------------------------------------------------------------------------------------------------
# 9. RECONSTRUCT FROZEN V6.1 MEAN
# --------------------------------------------------------------------------------------------------

POST[
    "v5_state_persistence_phi"
] = np.exp(
    -
    POST[
        "state_gap_days"
    ]
    .astype(float)
    /
    V5_TAU_DAYS_FROZEN
)


POST[
    "v61_structural_weight"
] = (
    1.0
    -
    POST[
        "v5_state_persistence_phi"
    ]
)


POST[
    "full_v6_adjustment"
] = (
    POST[
        "v6_prediction_mean"
    ]
    -
    POST[
        "v5_prediction_mean"
    ]
)


POST[
    "v61_applied_adjustment"
] = (
    POST[
        "v61_structural_weight"
    ]
    *
    POST[
        "full_v6_adjustment"
    ]
)


POST[
    "v61_prediction_mean"
] = (
    POST[
        "v5_prediction_mean"
    ]
    +
    POST[
        "v61_applied_adjustment"
    ]
)


POST[
    "v61_error"
] = (
    POST[
        "mark_numeric_m"
    ]
    -
    POST[
        "v61_prediction_mean"
    ]
)


POST[
    "v61_abs_error"
] = (
    POST[
        "v61_error"
    ]
    .abs()
)


POST[
    "v61_sq_error"
] = (
    POST[
        "v61_error"
    ]
    **
    2
)


# --------------------------------------------------------------------------------------------------
# 10. APPLY FROZEN U1
# --------------------------------------------------------------------------------------------------

B_M = float(
    FINAL_U1_B_M
)


B_W = float(
    FINAL_U1_B_W
)


G_M = float(
    FINAL_U1_G_M
)


G_W = float(
    FINAL_U1_G_W
)


NU = float(
    FINAL_U1_NU
)


POST[
    "horizon_years"
] = (
    POST[
        "state_gap_days"
    ]
    .astype(float)
    /
    365.25
)


POST[
    "v61_prediction_variance"
] = np.where(
    POST[
        "sex_group"
    ]
    .astype(str)
    .eq(
        "M"
    ),

    B_M
    +
    G_M
    *
    POST[
        "horizon_years"
    ],

    B_W
    +
    G_W
    *
    POST[
        "horizon_years"
    ],
)


POST[
    "v61_prediction_sigma"
] = np.sqrt(
    POST[
        "v61_prediction_variance"
    ]
)


POST[
    "v61_standardized_error"
] = (
    POST[
        "v61_error"
    ]
    /
    POST[
        "v61_prediction_sigma"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. INTERVALS
# --------------------------------------------------------------------------------------------------

for level in LEVELS_LOCAL:

    pct = int(
        100
        *
        level
    )


    q = float(
        student_t.ppf(
            (
                1.0
                +
                level
            )
            /
            2.0,
            df=NU,
        )

        *

        np.sqrt(
            (
                NU
                -
                2.0
            )
            /
            NU
        )
    )


    half_width = (
        q
        *
        POST[
            "v61_prediction_sigma"
        ]
    )


    POST[
        f"v61_lower_{pct}"
    ] = (
        POST[
            "v61_prediction_mean"
        ]
        -
        half_width
    )


    POST[
        f"v61_upper_{pct}"
    ] = (
        POST[
            "v61_prediction_mean"
        ]
        +
        half_width
    )


    POST[
        f"v61_width_{pct}"
    ] = (
        2.0
        *
        half_width
    )


    POST[
        f"v61_covered_{pct}"
    ] = (
        POST[
            "mark_numeric_m"
        ]
        .between(
            POST[
                f"v61_lower_{pct}"
            ],
            POST[
                f"v61_upper_{pct}"
            ],
            inclusive="both",
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. GAP BANDS
# --------------------------------------------------------------------------------------------------

POST[
    "state_gap_band"
] = pd.cut(
    POST[
        "state_gap_days"
    ],
    bins=GAP_BINS,
    labels=GAP_LABELS,
    include_lowest=True,
)


# --------------------------------------------------------------------------------------------------
# 13. MONITORING METRICS
# --------------------------------------------------------------------------------------------------

OVERALL_METRICS = (
    uncertainty_summary(
        POST
    )
    .to_frame()
    .T
)


SEX_METRICS = (
    POST.groupby(
        "sex_group",
        sort=True,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


GROUP_METRICS = (
    POST.groupby(
        "model_group",
        sort=True,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


GAP_METRICS = (
    POST.groupby(
        "state_gap_band",
        observed=True,
        sort=False,
    )
    .apply(
        uncertainty_summary,
        include_groups=False,
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 14. TEST vs POST_TEST OVERALL COMPARISON
# --------------------------------------------------------------------------------------------------

TEST_OVERALL = (
    FINAL_V61_TEST_UNCERTAINTY_OVERALL
    .iloc[
        0
    ]
)


POST_OVERALL = (
    OVERALL_METRICS
    .iloc[
        0
    ]
)


overall_comparison_rows = []


for metric in [
    "MAE",
    "RMSE",
    "mean_error",

    "standardized_error_mean",
    "standardized_error_sd",

    "coverage_50",
    "coverage_80",
    "coverage_95",

    "mean_width_50",
    "mean_width_80",
    "mean_width_95",
]:

    overall_comparison_rows.append(
        {
            "metric":
                metric,

            "TEST":
                float(
                    TEST_OVERALL[
                        metric
                    ]
                ),

            "POST_TEST":
                float(
                    POST_OVERALL[
                        metric
                    ]
                ),

            "POST_minus_TEST":
                float(
                    POST_OVERALL[
                        metric
                    ]
                    -
                    TEST_OVERALL[
                        metric
                    ]
                ),
        }
    )


OVERALL_COMPARISON = pd.DataFrame(
    overall_comparison_rows
)


# --------------------------------------------------------------------------------------------------
# 15. TEST vs POST_TEST MODEL-GROUP COMPARISON
# --------------------------------------------------------------------------------------------------

TEST_GROUP = (
    FINAL_V61_TEST_UNCERTAINTY_BY_GROUP
    .copy()
)


POST_GROUP = (
    GROUP_METRICS
    .copy()
)


group_compare_columns = [
    "model_group",
    "n",
    "standardized_error_sd",
    "coverage_50",
    "coverage_80",
    "coverage_95",
]


TEST_GROUP = (
    TEST_GROUP[
        group_compare_columns
    ]
    .rename(
        columns={
            c:
                (
                    c
                    if c == "model_group"
                    else f"TEST_{c}"
                )

            for c in group_compare_columns
        }
    )
)


POST_GROUP = (
    POST_GROUP[
        group_compare_columns
    ]
    .rename(
        columns={
            c:
                (
                    c
                    if c == "model_group"
                    else f"POST_{c}"
                )

            for c in group_compare_columns
        }
    )
)


GROUP_COMPARISON = (
    TEST_GROUP.merge(
        POST_GROUP,
        on="model_group",
        how="outer",
        validate="one_to_one",
    )
)


for pct in [
    50,
    80,
    95,
]:

    GROUP_COMPARISON[
        f"POST_minus_TEST_coverage_{pct}"
    ] = (
        GROUP_COMPARISON[
            f"POST_coverage_{pct}"
        ]
        -
        GROUP_COMPARISON[
            f"TEST_coverage_{pct}"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 16. DESCRIPTIVE MONITORING FLAGS
# --------------------------------------------------------------------------------------------------
#
# These are NOT release gates.
#
# They simply expose large subgroup deviations from nominal coverage.
# No model parameter is changed based on them.
#
# --------------------------------------------------------------------------------------------------

monitor_rows = []


for _, row in GROUP_METRICS.iterrows():

    for pct in [
        50,
        80,
        95,
    ]:

        target = float(
            pct
        )


        deviation = float(
            row[
                f"coverage_{pct}"
            ]
            -
            target
        )


        monitor_rows.append(
            {
                "model_group":
                    row[
                        "model_group"
                    ],

                "n":
                    int(
                        row[
                            "n"
                        ]
                    ),

                "interval":
                    f"{pct}%",

                "observed_coverage_pct":
                    float(
                        row[
                            f"coverage_{pct}"
                        ]
                    ),

                "target_pct":
                    target,

                "deviation_pp":
                    deviation,

                "target_inside_95pct_Wilson_CI":
                    bool(
                        row[
                            f"target_{pct}_inside_CI"
                        ]
                    ),
            }
        )


GROUP_MONITOR = pd.DataFrame(
    monitor_rows
)


# --------------------------------------------------------------------------------------------------
# 17. QA
# --------------------------------------------------------------------------------------------------

POST_DATE_PASS = bool(
    (
        POST[
            "result_date_dt"
        ]
        >=
        POST_TEST_START
    )
    .all()
)


TRAIN_SEEN_PASS = bool(
    POST[
        "athlete_seen_in_training"
    ]
    .eq(
        True
    )
    .all()
)


NO_MISSING_GAP_PASS = bool(
    POST[
        "state_gap_days"
    ]
    .notna()
    .all()
)


POSITIVE_GAP_PASS = bool(
    (
        POST[
            "state_gap_days"
        ]
        .astype(float)
        >
        0
    )
    .all()
)


SEX_DOMAIN_PASS = bool(
    set(
        POST[
            "sex_group"
        ]
        .astype(str)
        .unique()
    )
    .issubset(
        {
            "M",
            "W",
        }
    )
)


WEIGHT_BOUNDS_PASS = bool(
    POST[
        "v61_structural_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


POSITIVE_VARIANCE_PASS = bool(
    (
        POST[
            "v61_prediction_variance"
        ]
        >
        0
    )
    .all()
)


FINITE_OUTPUT_PASS = bool(
    np.isfinite(
        POST[
            [
                "v61_prediction_mean",
                "v61_error",
                "v61_prediction_variance",
                "v61_prediction_sigma",
                "v61_standardized_error",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


NO_PARAMETER_FIT_PASS = True


CELL_58_PASS = bool(
    len(
        POST
    )
    >
    0
    and
    POST_DATE_PASS
    and
    TRAIN_SEEN_PASS
    and
    NO_MISSING_GAP_PASS
    and
    POSITIVE_GAP_PASS
    and
    SEX_DOMAIN_PASS
    and
    WEIGHT_BOUNDS_PASS
    and
    POSITIVE_VARIANCE_PASS
    and
    FINITE_OUTPUT_PASS
    and
    NO_PARAMETER_FIT_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. IMMUTABLE MONITORING DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


MONITOR_ROOT = (
    RUN_DIR
    /
    "final_evaluation"
    /
    "post_test_monitor"
)


ATTEMPT_DIR = (
    MONITOR_ROOT
    /
    ATTEMPT_ID
)


ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# --------------------------------------------------------------------------------------------------
# 19. SAVE
# --------------------------------------------------------------------------------------------------

POST_PREDICTIONS_PATH = (
    ATTEMPT_DIR
    /
    "v61_post_test_predictions_with_uncertainty.parquet"
)


OVERALL_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "post_test_uncertainty_overall.parquet"
)


SEX_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "post_test_uncertainty_by_sex.parquet"
)


GROUP_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "post_test_uncertainty_by_model_group.parquet"
)


GAP_METRICS_PATH = (
    ATTEMPT_DIR
    /
    "post_test_uncertainty_by_gap.parquet"
)


OVERALL_COMPARISON_PATH = (
    ATTEMPT_DIR
    /
    "test_vs_post_test_overall.parquet"
)


GROUP_COMPARISON_PATH = (
    ATTEMPT_DIR
    /
    "test_vs_post_test_model_group.parquet"
)


GROUP_MONITOR_PATH = (
    ATTEMPT_DIR
    /
    "post_test_group_monitor.parquet"
)


AUDIT_PATH = (
    ATTEMPT_DIR
    /
    "post_test_monitor_audit.json"
)


POST.to_parquet(
    POST_PREDICTIONS_PATH,
    index=False,
)


OVERALL_METRICS.to_parquet(
    OVERALL_METRICS_PATH,
    index=False,
)


SEX_METRICS.to_parquet(
    SEX_METRICS_PATH,
    index=False,
)


GROUP_METRICS.to_parquet(
    GROUP_METRICS_PATH,
    index=False,
)


GAP_METRICS.to_parquet(
    GAP_METRICS_PATH,
    index=False,
)


OVERALL_COMPARISON.to_parquet(
    OVERALL_COMPARISON_PATH,
    index=False,
)


GROUP_COMPARISON.to_parquet(
    GROUP_COMPARISON_PATH,
    index=False,
)


GROUP_MONITOR.to_parquet(
    GROUP_MONITOR_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 20. PROVENANCE
# --------------------------------------------------------------------------------------------------

parent_paths = {

    "post_test_source":
        POST_SOURCE_PATH,

    "v5_ou_parameters":
        V5_PARAMETER_PATH,

    "v61_uncertainty_spec":
        Path(
            V61_U1_SPEC_PATH
        ),

    "v61_uncertainty_calibration_audit":
        Path(
            V61_U1_AUDIT_PATH
        ),

    "mean_architecture_decision":
        Path(
            ARCHITECTURE_DECISION_ARTIFACT
        ),

    "test_uncertainty_audit":
        Path(
            FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH
        ),
}


PARENT_ARTIFACTS = {}


for name, path in parent_paths.items():

    PARENT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "exists":
            bool(
                path.exists()
            ),

        "sha256":
            (
                sha256_file(
                    path
                )
                if path.exists()
                else None
            ),
    }


output_paths = {

    "post_predictions":
        POST_PREDICTIONS_PATH,

    "overall_metrics":
        OVERALL_METRICS_PATH,

    "sex_metrics":
        SEX_METRICS_PATH,

    "group_metrics":
        GROUP_METRICS_PATH,

    "gap_metrics":
        GAP_METRICS_PATH,

    "test_vs_post_overall":
        OVERALL_COMPARISON_PATH,

    "test_vs_post_group":
        GROUP_COMPARISON_PATH,

    "group_monitor":
        GROUP_MONITOR_PATH,
}


OUTPUT_ARTIFACTS = {}


for name, path in output_paths.items():

    OUTPUT_ARTIFACTS[
        name
    ] = {
        "path":
            str(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 21. AUDIT
# --------------------------------------------------------------------------------------------------

AUDIT = {

    "audit_schema_version":
        STAGE_SCHEMA_VERSION,

    "stage":
        "V61_U1_POST_TEST_OPERATIONAL_MONITOR",

    "stage_version":
        STAGE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_58_PASS
            else
            "FAILED"
        ),

    "monitoring_role":
        (
            "Forward-style operational monitoring only. "
            "No mean or uncertainty parameters are fitted."
        ),

    "post_test_start":
        str(
            POST_TEST_START.date()
        ),

    "post_test_end":
        str(
            POST[
                "result_date_dt"
            ]
            .max()
            .date()
        ),

    "population":
        "TRAIN_SEEN_ATHLETES_WITH_STATE_HISTORY",

    "mean_architecture":
        "V6.1_OU_FRESHNESS_BLEND",

    "uncertainty_model":
        "U1_SEX_LINEAR",

    "parameter_fit":
        False,

    "rows": {
        "source":
            SOURCE_ROWS,

        "evaluated":
            int(
                len(
                    POST
                )
            ),

        "athletes":
            int(
                POST[
                    "aaAthleteId"
                ]
                .nunique()
            ),
    },

    "overall_metrics":
        OVERALL_METRICS.iloc[
            0
        ].to_dict(),

    "qa": {
        "post_test_dates":
            POST_DATE_PASS,

        "train_seen_only":
            TRAIN_SEEN_PASS,

        "no_missing_state_gaps":
            NO_MISSING_GAP_PASS,

        "positive_state_gaps":
            POSITIVE_GAP_PASS,

        "sex_domain":
            SEX_DOMAIN_PASS,

        "V61_weight_bounds":
            WEIGHT_BOUNDS_PASS,

        "positive_variance":
            POSITIVE_VARIANCE_PASS,

        "finite_output":
            FINITE_OUTPUT_PASS,

        "no_parameter_fit":
            NO_PARAMETER_FIT_PASS,

        "cell_pass":
            CELL_58_PASS,
    },

    "parent_artifacts":
        PARENT_ARTIFACTS,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    AUDIT,
    AUDIT_PATH,
)


AUDIT_SHA256 = sha256_file(
    AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    MONITOR_ROOT
    /
    "LATEST_POST_TEST_MONITOR.json"
)


atomic_json_write(
    {
        "attempt_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_58_PASS
                else
                "FAILED"
            ),

        "attempt_directory":
            str(
                ATTEMPT_DIR
            ),

        "post_predictions":
            str(
                POST_PREDICTIONS_PATH
            ),

        "overall_metrics":
            str(
                OVERALL_METRICS_PATH
            ),

        "audit_manifest":
            str(
                AUDIT_PATH
            ),

        "audit_sha256":
            AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST.setdefault(
    "final_evaluation",
    {},
)


TRAINING_MANIFEST[
    "final_evaluation"
][
    "post_test_monitor"
] = {

    "status":
        (
            "PASSED"
            if CELL_58_PASS
            else
            "FAILED"
        ),

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "post_predictions":
        str(
            POST_PREDICTIONS_PATH
        ),

    "overall_metrics":
        str(
            OVERALL_METRICS_PATH
        ),

    "audit_manifest":
        str(
            AUDIT_PATH
        ),

    "audit_sha256":
        AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "V61_U1_POST_TEST_MONITOR_COMPLETE"
    if CELL_58_PASS
    else
    "V61_U1_POST_TEST_MONITOR_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. GLOBALS
# --------------------------------------------------------------------------------------------------

FINAL_V61_POST_TEST_UNCERTAINTY = (
    POST
)


FINAL_V61_POST_TEST_UNCERTAINTY_OVERALL = (
    OVERALL_METRICS
)


FINAL_V61_POST_TEST_UNCERTAINTY_BY_SEX = (
    SEX_METRICS
)


FINAL_V61_POST_TEST_UNCERTAINTY_BY_GROUP = (
    GROUP_METRICS
)


FINAL_V61_POST_TEST_UNCERTAINTY_BY_GAP = (
    GAP_METRICS
)


FINAL_TEST_POST_OVERALL_COMPARISON = (
    OVERALL_COMPARISON
)


FINAL_TEST_POST_GROUP_COMPARISON = (
    GROUP_COMPARISON
)


FINAL_POST_TEST_GROUP_MONITOR = (
    GROUP_MONITOR
)


FINAL_V61_POST_TEST_MONITOR_AUDIT_PATH = (
    AUDIT_PATH
)


NOTEBOOK_2_CELL_58_PASS = (
    CELL_58_PASS
)


# --------------------------------------------------------------------------------------------------
# 25. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 180)
print("V6.1 + U1 — POST_TEST OPERATIONAL MONITOR")
print("=" * 180)


print()
print("POST_TEST POPULATION")
print("-" * 180)

print(
    f"Source rows                     : "
    f"{SOURCE_ROWS:,}"
)

print(
    f"Evaluated rows                  : "
    f"{len(POST):,}"
)

print(
    f"Athletes                        : "
    f"{POST['aaAthleteId'].nunique():,}"
)

print(
    f"Date range                      : "
    f"{POST['result_date_dt'].min()}"
    f" -> "
    f"{POST['result_date_dt'].max()}"
)

print(
    f"Mean state gap                  : "
    f"{POST['state_gap_days'].mean():.3f} days"
)

print(
    f"Median state gap                : "
    f"{POST['state_gap_days'].median():.3f} days"
)


print()
print("OVERALL POST_TEST METRICS")
print("-" * 180)

display(
    OVERALL_METRICS
)


print()
print("POST_TEST BY SEX")
print("-" * 180)

display(
    SEX_METRICS
)


print()
print("POST_TEST BY MODEL GROUP")
print("-" * 180)

display(
    GROUP_METRICS
)


print()
print("POST_TEST BY STATE GAP")
print("-" * 180)

display(
    GAP_METRICS
)


print()
print("TEST vs POST_TEST — OVERALL")
print("-" * 180)

display(
    OVERALL_COMPARISON
)


print()
print("TEST vs POST_TEST — MODEL GROUP")
print("-" * 180)

display(
    GROUP_COMPARISON
)


print()
print("POST_TEST GROUP COVERAGE MONITOR")
print("-" * 180)

display(
    GROUP_MONITOR
)


print()
print("POST_TEST COVERAGE")
print("-" * 180)

post_overall = (
    OVERALL_METRICS.iloc[
        0
    ]
)


for level in LEVELS_LOCAL:

    pct = int(
        100
        *
        level
    )

    print(
        f"{pct:>2}% target | "
        f"observed = "
        f"{post_overall[f'coverage_{pct}']:.3f}% | "
        f"95% Wilson CI = "
        f"["
        f"{post_overall[f'coverage_{pct}_CI_low']:.3f}%, "
        f"{post_overall[f'coverage_{pct}_CI_high']:.3f}%"
        f"] | "
        f"target inside CI = "
        f"{post_overall[f'target_{pct}_inside_CI']}"
    )


print()
print("AUDIT / VERSIONING")
print("-" * 180)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable monitor directory     : "
    f"{ATTEMPT_DIR}"
)

print(
    f"Audit manifest                  : "
    f"{AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("QA")
print("-" * 180)

print(
    f"POST_TEST dates only            : "
    f"{POST_DATE_PASS}"
)

print(
    f"TRAIN-seen athletes only        : "
    f"{TRAIN_SEEN_PASS}"
)

print(
    f"No missing state gaps           : "
    f"{NO_MISSING_GAP_PASS}"
)

print(
    f"Positive state gaps             : "
    f"{POSITIVE_GAP_PASS}"
)

print(
    f"Sex domain M/W                  : "
    f"{SEX_DOMAIN_PASS}"
)

print(
    f"V6.1 weights valid              : "
    f"{WEIGHT_BOUNDS_PASS}"
)

print(
    f"Positive predictive variance    : "
    f"{POSITIVE_VARIANCE_PASS}"
)

print(
    f"Finite outputs                  : "
    f"{FINITE_OUTPUT_PASS}"
)

print(
    f"No parameters fitted            : "
    f"{NO_PARAMETER_FIT_PASS}"
)


print()
print("=" * 180)

if CELL_58_PASS:

    print(
        "CELL 58 STATUS: POST_TEST OPERATIONAL MONITOR COMPLETE"
    )

else:

    print(
        "CELL 58 STATUS: POST_TEST MONITOR REVIEW REQUIRED"
    )

print("=" * 180)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 53
# Original source SHA256: 8d6c852db6e545e41f67e393ea968625b01e641615455db1114324861ec05c15
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 59 — FINAL RELEASE GATES + KNOWN-LIMITATIONS FREEZE
# ==================================================================================================
#
# Purpose
# -------
# Decide whether the frozen V6.1 + U1 candidate is eligible to proceed to
# release bundling.
#
# IMPORTANT
# ---------
# NO fitting.
# NO model changes.
# NO uncertainty recalibration.
# NO TEST / POST_TEST tuning.
#
# Hard gates
# ----------
# Hard gates are based on:
#
#   1. successful historical mean-architecture selection
#   2. successful VALIDATION-only U1 calibration
#   3. technically valid frozen TEST evaluation
#   4. overall TEST 50/80/95% nominal coverage lying inside its 95% Wilson CI
#   5. TEST and POST_TEST never being used for parameter fitting
#   6. required frozen artifacts existing
#
# Monitoring alerts
# -----------------
# Subgroup and POST_TEST deviations are recorded as NON-BLOCKING monitoring
# findings for this version.
#
# This avoids changing the model after seeing TEST / POST_TEST.
#
# Possible statuses:
#
#   PASS
#   PASS_WITH_MONITORING
#   FAIL
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_52_PASS",
    "NOTEBOOK_2_CELL_56_PASS",
    "NOTEBOOK_2_CELL_57_PASS",
    "NOTEBOOK_2_CELL_58_PASS",

    "FINAL_V61_TEST_UNCERTAINTY_OVERALL",
    "FINAL_V61_TEST_UNCERTAINTY_BY_GROUP",

    "FINAL_V61_POST_TEST_UNCERTAINTY_OVERALL",
    "FINAL_V61_POST_TEST_UNCERTAINTY_BY_GROUP",

    "ARCHITECTURE_DECISION_ARTIFACT",
    "V61_U1_SPEC_PATH",
    "V61_U1_AUDIT_PATH",
    "FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH",
    "FINAL_V61_POST_TEST_MONITOR_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_59_FINAL_RELEASE_GATES"

GATE_SCHEMA_VERSION = "1.0.0"

CANDIDATE_MODEL = "V6.1_OU_FRESHNESS_BLEND"

CANDIDATE_UNCERTAINTY = "U1_SEX_LINEAR"


# Minimum subgroup sample used only for highlighting persistent TEST subgroup issues.
# This is NOT a model-fitting threshold.
SUBGROUP_MONITOR_MIN_N = 100


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


# --------------------------------------------------------------------------------------------------
# 4. LOAD FROZEN AUDITS
# --------------------------------------------------------------------------------------------------

ARCH_DECISION_PATH = Path(
    ARCHITECTURE_DECISION_ARTIFACT
)


U1_SPEC_PATH = Path(
    V61_U1_SPEC_PATH
)


U1_AUDIT_PATH = Path(
    V61_U1_AUDIT_PATH
)


TEST_AUDIT_PATH = Path(
    FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH
)


POST_AUDIT_PATH = Path(
    FINAL_V61_POST_TEST_MONITOR_AUDIT_PATH
)


REQUIRED_ARTIFACT_PATHS = {
    "mean_architecture_decision":
        ARCH_DECISION_PATH,

    "V61_U1_spec":
        U1_SPEC_PATH,

    "V61_U1_calibration_audit":
        U1_AUDIT_PATH,

    "TEST_uncertainty_audit":
        TEST_AUDIT_PATH,

    "POST_TEST_monitor_audit":
        POST_AUDIT_PATH,
}


MISSING_ARTIFACTS = [
    name
    for name, path
    in REQUIRED_ARTIFACT_PATHS.items()
    if not path.exists()
]


if MISSING_ARTIFACTS:

    raise RuntimeError(
        "Required release artifacts missing:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in MISSING_ARTIFACTS
        )
    )


with ARCH_DECISION_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    ARCH_DECISION = json.load(f)


with U1_SPEC_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    U1_SPEC = json.load(f)


with U1_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    U1_AUDIT = json.load(f)


with TEST_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    TEST_AUDIT = json.load(f)


with POST_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    POST_AUDIT = json.load(f)


# --------------------------------------------------------------------------------------------------
# 5. FROZEN OVERALL METRICS
# --------------------------------------------------------------------------------------------------

TEST_OVERALL = (
    FINAL_V61_TEST_UNCERTAINTY_OVERALL
    .iloc[0]
)


POST_OVERALL = (
    FINAL_V61_POST_TEST_UNCERTAINTY_OVERALL
    .iloc[0]
)


# --------------------------------------------------------------------------------------------------
# 6. HARD RELEASE GATES
# --------------------------------------------------------------------------------------------------

HARD_GATES = {

    # Mean architecture evidence.
    "historical_mean_architecture_selected":
        bool(
            NOTEBOOK_2_CELL_52_PASS
            and
            ARCH_DECISION.get(
                "decision_status"
            )
            ==
            "SELECTED_FOR_FINAL_UNCERTAINTY_CALIBRATION"
        ),

    # U1 was fitted only on VALIDATION.
    "validation_only_uncertainty_calibration_passed":
        bool(
            NOTEBOOK_2_CELL_56_PASS
            and
            U1_SPEC.get(
                "calibration_split"
            )
            ==
            "VALIDATION"
            and
            U1_SPEC.get(
                "test_used"
            )
            is False
        ),

    # Frozen TEST evaluation executed correctly.
    "frozen_test_evaluation_passed":
        bool(
            NOTEBOOK_2_CELL_57_PASS
        ),

    # Overall out-of-sample coverage.
    "test_50pct_target_inside_Wilson_CI":
        bool(
            TEST_OVERALL[
                "target_50_inside_CI"
            ]
        ),

    "test_80pct_target_inside_Wilson_CI":
        bool(
            TEST_OVERALL[
                "target_80_inside_CI"
            ]
        ),

    "test_95pct_target_inside_Wilson_CI":
        bool(
            TEST_OVERALL[
                "target_95_inside_CI"
            ]
        ),

    # TEST must not have been used to fit U1.
    "test_not_used_for_parameter_fit":
        bool(
            TEST_AUDIT[
                "test_role"
            ][
                "used_to_fit_U1"
            ]
            is False
        ),

    # POST_TEST must remain monitoring-only.
    "post_test_monitor_is_non_fitting":
        bool(
            NOTEBOOK_2_CELL_58_PASS
            and
            POST_AUDIT.get(
                "parameter_fit"
            )
            is False
        ),

    # Mean architecture selection must explicitly exclude 2026 TEST.
    "2026_test_excluded_from_mean_architecture_selection":
        bool(
            ARCH_DECISION[
                "2026_test"
            ][
                "used_for_architecture_selection"
            ]
            is False
        ),

    # Required frozen artifacts must exist.
    "required_release_artifacts_exist":
        bool(
            all(
                path.exists()
                for path
                in REQUIRED_ARTIFACT_PATHS.values()
            )
        ),
}


ALL_HARD_GATES_PASS = bool(
    all(
        HARD_GATES.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 7. ARTIFACT HASH CHECK
# --------------------------------------------------------------------------------------------------

ARTIFACT_HASHES = {
    name: {
        "path":
            str(path),

        "sha256":
            sha256_file(path),
    }

    for name, path
    in REQUIRED_ARTIFACT_PATHS.items()
}


HASH_CHECK_PASS = bool(
    all(
        len(
            item[
                "sha256"
            ]
        )
        ==
        64

        for item
        in ARTIFACT_HASHES.values()
    )
)


HARD_GATES[
    "required_artifacts_hashable"
] = (
    HASH_CHECK_PASS
)


ALL_HARD_GATES_PASS = bool(
    all(
        HARD_GATES.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 8. MONITORING ALERTS — OVERALL POST_TEST
# --------------------------------------------------------------------------------------------------
#
# These are explicitly NON-BLOCKING for this model version.
#
# They are frozen observations for future monitoring / V6.2 research.
#
# --------------------------------------------------------------------------------------------------

MONITORING_ALERT_ROWS = []


for pct in [
    50,
    80,
    95,
]:

    target_inside = bool(
        POST_OVERALL[
            f"target_{pct}_inside_CI"
        ]
    )


    if not target_inside:

        MONITORING_ALERT_ROWS.append(
            {
                "scope":
                    "POST_TEST_OVERALL",

                "segment":
                    "ALL",

                "metric":
                    f"coverage_{pct}",

                "n":
                    int(
                        POST_OVERALL[
                            "n"
                        ]
                    ),

                "observed":
                    float(
                        POST_OVERALL[
                            f"coverage_{pct}"
                        ]
                    ),

                "reference":
                    float(
                        pct
                    ),

                "severity":
                    "MONITOR",

                "blocking":
                    False,

                "reason":
                    (
                        f"{pct}% nominal target is outside "
                        f"the POST_TEST 95% Wilson interval."
                    ),
            }
        )


# --------------------------------------------------------------------------------------------------
# 9. MONITORING ALERTS — TEST MODEL GROUPS
# --------------------------------------------------------------------------------------------------

TEST_GROUP = (
    FINAL_V61_TEST_UNCERTAINTY_BY_GROUP
    .copy()
)


for _, row in TEST_GROUP.iterrows():

    n = int(
        row[
            "n"
        ]
    )


    if n < SUBGROUP_MONITOR_MIN_N:
        continue


    for pct in [
        50,
        80,
        95,
    ]:

        inside = bool(
            row[
                f"target_{pct}_inside_CI"
            ]
        )


        if not inside:

            MONITORING_ALERT_ROWS.append(
                {
                    "scope":
                        "TEST_MODEL_GROUP",

                    "segment":
                        str(
                            row[
                                "model_group"
                            ]
                        ),

                    "metric":
                        f"coverage_{pct}",

                    "n":
                        n,

                    "observed":
                        float(
                            row[
                                f"coverage_{pct}"
                            ]
                        ),

                    "reference":
                        float(
                            pct
                        ),

                    "severity":
                        "MONITOR",

                    "blocking":
                        False,

                    "reason":
                        (
                            f"{pct}% nominal target is outside "
                            f"the frozen TEST group-level "
                            f"95% Wilson interval."
                        ),
                }
            )


# --------------------------------------------------------------------------------------------------
# 10. MONITORING ALERTS — POST_TEST MODEL GROUPS
# --------------------------------------------------------------------------------------------------

POST_GROUP = (
    FINAL_V61_POST_TEST_UNCERTAINTY_BY_GROUP
    .copy()
)


for _, row in POST_GROUP.iterrows():

    for pct in [
        50,
        80,
        95,
    ]:

        inside = bool(
            row[
                f"target_{pct}_inside_CI"
            ]
        )


        if not inside:

            MONITORING_ALERT_ROWS.append(
                {
                    "scope":
                        "POST_TEST_MODEL_GROUP",

                    "segment":
                        str(
                            row[
                                "model_group"
                            ]
                        ),

                    "metric":
                        f"coverage_{pct}",

                    "n":
                        int(
                            row[
                                "n"
                            ]
                        ),

                    "observed":
                        float(
                            row[
                                f"coverage_{pct}"
                            ]
                        ),

                    "reference":
                        float(
                            pct
                        ),

                    "severity":
                        "MONITOR",

                    "blocking":
                        False,

                    "reason":
                        (
                            f"{pct}% nominal target is outside "
                            f"the POST_TEST group-level "
                            f"95% Wilson interval."
                        ),
                }
            )


MONITORING_ALERTS = pd.DataFrame(
    MONITORING_ALERT_ROWS,
    columns=[
        "scope",
        "segment",
        "metric",
        "n",
        "observed",
        "reference",
        "severity",
        "blocking",
        "reason",
    ],
)


HAS_MONITORING_ALERTS = bool(
    len(
        MONITORING_ALERTS
    )
    >
    0
)


# --------------------------------------------------------------------------------------------------
# 11. KNOWN LIMITATIONS
# --------------------------------------------------------------------------------------------------

KNOWN_LIMITATIONS = [

    {
        "id":
            "ULIM-001",

        "area":
            "uncertainty_model",

        "description":
            (
                "U1 models predictive variance by sex and forecast "
                "horizon only. It does not currently include separate "
                "variance parameters for implement/model group."
            ),

        "impact":
            (
                "Frozen TEST evidence shows materially lower interval "
                "coverage for some youth implement groups, particularly "
                "M_700 and W_500."
            ),

        "action":
            (
                "Monitor group-level calibration in production. "
                "Evaluate group-specific uncertainty architecture only "
                "in a future version using independent development data."
            ),
    },

    {
        "id":
            "ULIM-002",

        "area":
            "post_test_calibration",

        "description":
            (
                "POST_TEST overall 95% coverage is below nominal and "
                "its 95% Wilson interval does not contain the 95% target."
            ),

        "impact":
            (
                "The frozen U1 intervals may be mildly too narrow in "
                "the most recent operational period."
            ),

        "action":
            (
                "Continue forward monitoring. Do not recalibrate the "
                "current release using POST_TEST."
            ),
    },

    {
        "id":
            "ULIM-003",

        "area":
            "mean_architecture_evidence",

        "description":
            (
                "The 2026 TEST period is not independent evidence for "
                "V6.1 mean-architecture selection because V6.1 was "
                "conceived after reviewing that period."
            ),

        "impact":
            (
                "Independent mean-architecture support comes from the "
                "HB_2023, HB_2024 and HB_2025 temporal backtests."
            ),

        "action":
            (
                "Preserve this distinction in all release documentation."
            ),
    },
]


# --------------------------------------------------------------------------------------------------
# 12. RELEASE DECISION
# --------------------------------------------------------------------------------------------------

if not ALL_HARD_GATES_PASS:

    RELEASE_STATUS = (
        "FAIL"
    )

    RELEASE_ELIGIBLE_FOR_BUNDLE = False


elif HAS_MONITORING_ALERTS:

    RELEASE_STATUS = (
        "PASS_WITH_MONITORING"
    )

    RELEASE_ELIGIBLE_FOR_BUNDLE = True


else:

    RELEASE_STATUS = (
        "PASS"
    )

    RELEASE_ELIGIBLE_FOR_BUNDLE = True


# --------------------------------------------------------------------------------------------------
# 13. TABLES
# --------------------------------------------------------------------------------------------------

HARD_GATE_TABLE = pd.DataFrame(
    [
        {
            "gate":
                gate,

            "pass":
                value,
        }

        for gate, value
        in HARD_GATES.items()
    ]
)


OVERALL_RELEASE_EVIDENCE = pd.DataFrame(
    [
        {
            "period":
                "VALIDATION",

            "n":
                int(
                    U1_AUDIT[
                        "fit"
                    ][
                        "rows"
                    ]
                ),

            "role":
                "U1_CALIBRATION",
        },

        {
            "period":
                "TEST",

            "n":
                int(
                    TEST_OVERALL[
                        "n"
                    ]
                ),

            "role":
                "FROZEN_OUT_OF_SAMPLE_EVALUATION",
        },

        {
            "period":
                "POST_TEST",

            "n":
                int(
                    POST_OVERALL[
                        "n"
                    ]
                ),

            "role":
                "OPERATIONAL_MONITOR_ONLY",
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. IMMUTABLE RELEASE-GATE ATTEMPT
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


RUN_DIR = Path(
    TRAINING_RUN_DIR
)


RELEASE_GATE_ROOT = (
    RUN_DIR
    /
    "release_gates"
)


ATTEMPT_DIR = (
    RELEASE_GATE_ROOT
    /
    ATTEMPT_ID
)


ATTEMPT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


HARD_GATES_PATH = (
    ATTEMPT_DIR
    /
    "hard_release_gates.parquet"
)


MONITORING_ALERTS_PATH = (
    ATTEMPT_DIR
    /
    "monitoring_alerts.parquet"
)


OVERALL_EVIDENCE_PATH = (
    ATTEMPT_DIR
    /
    "release_evidence_periods.parquet"
)


KNOWN_LIMITATIONS_PATH = (
    ATTEMPT_DIR
    /
    "known_limitations.json"
)


RELEASE_DECISION_PATH = (
    ATTEMPT_DIR
    /
    "release_gate_decision.json"
)


AUDIT_PATH = (
    ATTEMPT_DIR
    /
    "release_gate_audit.json"
)


HARD_GATE_TABLE.to_parquet(
    HARD_GATES_PATH,
    index=False,
)


MONITORING_ALERTS.to_parquet(
    MONITORING_ALERTS_PATH,
    index=False,
)


OVERALL_RELEASE_EVIDENCE.to_parquet(
    OVERALL_EVIDENCE_PATH,
    index=False,
)


atomic_json_write(
    {
        "schema_version":
            GATE_SCHEMA_VERSION,

        "known_limitations":
            KNOWN_LIMITATIONS,
    },
    KNOWN_LIMITATIONS_PATH,
)


# --------------------------------------------------------------------------------------------------
# 15. RELEASE DECISION DOCUMENT
# --------------------------------------------------------------------------------------------------

RELEASE_DECISION = {

    "release_gate_schema_version":
        GATE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "candidate_mean_architecture":
        CANDIDATE_MODEL,

    "candidate_uncertainty_model":
        CANDIDATE_UNCERTAINTY,

    "release_status":
        RELEASE_STATUS,

    "eligible_for_release_bundle":
        RELEASE_ELIGIBLE_FOR_BUNDLE,

    "all_hard_gates_pass":
        ALL_HARD_GATES_PASS,

    "monitoring_alert_count":
        int(
            len(
                MONITORING_ALERTS
            )
        ),

    "hard_gates":
        HARD_GATES,

    "test_overall": {
        "n":
            int(
                TEST_OVERALL[
                    "n"
                ]
            ),

        "MAE":
            float(
                TEST_OVERALL[
                    "MAE"
                ]
            ),

        "RMSE":
            float(
                TEST_OVERALL[
                    "RMSE"
                ]
            ),

        "standardized_error_sd":
            float(
                TEST_OVERALL[
                    "standardized_error_sd"
                ]
            ),

        "coverage_50":
            float(
                TEST_OVERALL[
                    "coverage_50"
                ]
            ),

        "coverage_80":
            float(
                TEST_OVERALL[
                    "coverage_80"
                ]
            ),

        "coverage_95":
            float(
                TEST_OVERALL[
                    "coverage_95"
                ]
            ),
    },

    "post_test_overall": {
        "n":
            int(
                POST_OVERALL[
                    "n"
                ]
            ),

        "MAE":
            float(
                POST_OVERALL[
                    "MAE"
                ]
            ),

        "RMSE":
            float(
                POST_OVERALL[
                    "RMSE"
                ]
            ),

        "standardized_error_sd":
            float(
                POST_OVERALL[
                    "standardized_error_sd"
                ]
            ),

        "coverage_50":
            float(
                POST_OVERALL[
                    "coverage_50"
                ]
            ),

        "coverage_80":
            float(
                POST_OVERALL[
                    "coverage_80"
                ]
            ),

        "coverage_95":
            float(
                POST_OVERALL[
                    "coverage_95"
                ]
            ),
    },

    "known_limitations_file":
        str(
            KNOWN_LIMITATIONS_PATH
        ),

    "decision_note":
        (
            "Candidate passes all hard release gates. "
            "Monitoring alerts do not modify or refit the frozen "
            "model and are carried forward as explicit release "
            "limitations."
            if RELEASE_ELIGIBLE_FOR_BUNDLE
            else
            "Candidate failed one or more hard release gates and "
            "must not be bundled or promoted."
        ),

    "next_stage":
        (
            "BUILD_RELEASE_BUNDLE"
            if RELEASE_ELIGIBLE_FOR_BUNDLE
            else
            "RELEASE_REVIEW_REQUIRED"
        ),
}


atomic_json_write(
    RELEASE_DECISION,
    RELEASE_DECISION_PATH,
)


# --------------------------------------------------------------------------------------------------
# 16. OUTPUT ARTIFACT HASHES
# --------------------------------------------------------------------------------------------------

OUTPUT_PATHS = {
    "hard_release_gates":
        HARD_GATES_PATH,

    "monitoring_alerts":
        MONITORING_ALERTS_PATH,

    "release_evidence_periods":
        OVERALL_EVIDENCE_PATH,

    "known_limitations":
        KNOWN_LIMITATIONS_PATH,

    "release_gate_decision":
        RELEASE_DECISION_PATH,
}


OUTPUT_ARTIFACTS = {
    name: {
        "path":
            str(path),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(path),
    }

    for name, path
    in OUTPUT_PATHS.items()
}


GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


# --------------------------------------------------------------------------------------------------
# 17. AUDIT
# --------------------------------------------------------------------------------------------------

AUDIT = {

    "audit_schema_version":
        GATE_SCHEMA_VERSION,

    "stage":
        "FINAL_RELEASE_GATES",

    "stage_version":
        GATE_SCHEMA_VERSION,

    "attempt_id":
        ATTEMPT_ID,

    "cell_id":
        CELL_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        RELEASE_STATUS,

    "eligible_for_release_bundle":
        RELEASE_ELIGIBLE_FOR_BUNDLE,

    "all_hard_gates_pass":
        ALL_HARD_GATES_PASS,

    "hard_gates":
        HARD_GATES,

    "monitoring_alert_count":
        int(
            len(
                MONITORING_ALERTS
            )
        ),

    "monitoring_alerts_non_blocking":
        True,

    "parameter_fit":
        False,

    "model_changed":
        False,

    "thresholds_used_for_refit":
        False,

    "parent_artifacts":
        ARTIFACT_HASHES,

    "output_artifacts":
        OUTPUT_ARTIFACTS,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },

    "git_commit":
        GIT_COMMIT,
}


atomic_json_write(
    AUDIT,
    AUDIT_PATH,
)


AUDIT_SHA256 = sha256_file(
    AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 18. LATEST POINTER
# --------------------------------------------------------------------------------------------------

LATEST_POINTER_PATH = (
    RELEASE_GATE_ROOT
    /
    "LATEST_RELEASE_GATES.json"
)


atomic_json_write(
    {
        "attempt_id":
            ATTEMPT_ID,

        "release_status":
            RELEASE_STATUS,

        "eligible_for_release_bundle":
            RELEASE_ELIGIBLE_FOR_BUNDLE,

        "attempt_directory":
            str(
                ATTEMPT_DIR
            ),

        "release_gate_decision":
            str(
                RELEASE_DECISION_PATH
            ),

        "known_limitations":
            str(
                KNOWN_LIMITATIONS_PATH
            ),

        "audit_manifest":
            str(
                AUDIT_PATH
            ),

        "audit_sha256":
            AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_POINTER_PATH,
)


# --------------------------------------------------------------------------------------------------
# 19. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "release_gates"
] = {

    "status":
        RELEASE_STATUS,

    "eligible_for_release_bundle":
        RELEASE_ELIGIBLE_FOR_BUNDLE,

    "attempt_id":
        ATTEMPT_ID,

    "attempt_directory":
        str(
            ATTEMPT_DIR
        ),

    "latest_pointer":
        str(
            LATEST_POINTER_PATH
        ),

    "release_gate_decision":
        str(
            RELEASE_DECISION_PATH
        ),

    "known_limitations":
        str(
            KNOWN_LIMITATIONS_PATH
        ),

    "audit_manifest":
        str(
            AUDIT_PATH
        ),

    "audit_sha256":
        AUDIT_SHA256,
}


TRAINING_MANIFEST[
    "status"
] = (
    "RELEASE_CANDIDATE_ELIGIBLE_WITH_MONITORING"
    if RELEASE_STATUS == "PASS_WITH_MONITORING"

    else
    (
        "RELEASE_CANDIDATE_ELIGIBLE"
        if RELEASE_STATUS == "PASS"

        else
        "RELEASE_GATES_FAILED"
    )
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. GLOBALS
# --------------------------------------------------------------------------------------------------

FINAL_RELEASE_STATUS = (
    RELEASE_STATUS
)


FINAL_RELEASE_ELIGIBLE_FOR_BUNDLE = (
    RELEASE_ELIGIBLE_FOR_BUNDLE
)


FINAL_RELEASE_HARD_GATES = (
    HARD_GATE_TABLE
)


FINAL_RELEASE_MONITORING_ALERTS = (
    MONITORING_ALERTS
)


FINAL_RELEASE_KNOWN_LIMITATIONS = (
    KNOWN_LIMITATIONS
)


FINAL_RELEASE_GATE_DECISION_PATH = (
    RELEASE_DECISION_PATH
)


FINAL_RELEASE_GATE_AUDIT_PATH = (
    AUDIT_PATH
)


NOTEBOOK_2_CELL_59_PASS = bool(
    RELEASE_ELIGIBLE_FOR_BUNDLE
)


# --------------------------------------------------------------------------------------------------
# 21. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 180)
print("FINAL RELEASE GATES — V6.1 + U1")
print("=" * 180)


print()
print("HARD RELEASE GATES")
print("-" * 180)

display(
    HARD_GATE_TABLE
)


print()
print("OVERALL RELEASE EVIDENCE")
print("-" * 180)

display(
    OVERALL_RELEASE_EVIDENCE
)


print()
print("MONITORING ALERTS — NON-BLOCKING")
print("-" * 180)

if len(
    MONITORING_ALERTS
):

    display(
        MONITORING_ALERTS
    )

else:

    print(
        "None."
    )


print()
print("KNOWN LIMITATIONS")
print("-" * 180)

for item in KNOWN_LIMITATIONS:

    print()
    print(
        f"{item['id']} — {item['area']}"
    )

    print(
        f"  Description : {item['description']}"
    )

    print(
        f"  Impact      : {item['impact']}"
    )

    print(
        f"  Action      : {item['action']}"
    )


print()
print("RELEASE DECISION")
print("-" * 180)

print(
    f"All hard gates pass             : "
    f"{ALL_HARD_GATES_PASS}"
)

print(
    f"Monitoring alerts               : "
    f"{len(MONITORING_ALERTS)}"
)

print(
    f"Release status                  : "
    f"{RELEASE_STATUS}"
)

print(
    f"Eligible for release bundle     : "
    f"{RELEASE_ELIGIBLE_FOR_BUNDLE}"
)

print(
    f"Model / parameters changed      : "
    f"False"
)


print()
print("TEST OVERALL")
print("-" * 180)

print(
    f"n                               : "
    f"{int(TEST_OVERALL['n']):,}"
)

print(
    f"MAE                             : "
    f"{TEST_OVERALL['MAE']:.6f} m"
)

print(
    f"RMSE                            : "
    f"{TEST_OVERALL['RMSE']:.6f} m"
)

print(
    f"Standardized error SD           : "
    f"{TEST_OVERALL['standardized_error_sd']:.6f}"
)

print(
    f"50% coverage                    : "
    f"{TEST_OVERALL['coverage_50']:.3f}%"
)

print(
    f"80% coverage                    : "
    f"{TEST_OVERALL['coverage_80']:.3f}%"
)

print(
    f"95% coverage                    : "
    f"{TEST_OVERALL['coverage_95']:.3f}%"
)


print()
print("POST_TEST MONITOR")
print("-" * 180)

print(
    f"n                               : "
    f"{int(POST_OVERALL['n']):,}"
)

print(
    f"MAE                             : "
    f"{POST_OVERALL['MAE']:.6f} m"
)

print(
    f"RMSE                            : "
    f"{POST_OVERALL['RMSE']:.6f} m"
)

print(
    f"Standardized error SD           : "
    f"{POST_OVERALL['standardized_error_sd']:.6f}"
)

print(
    f"50% coverage                    : "
    f"{POST_OVERALL['coverage_50']:.3f}%"
)

print(
    f"80% coverage                    : "
    f"{POST_OVERALL['coverage_80']:.3f}%"
)

print(
    f"95% coverage                    : "
    f"{POST_OVERALL['coverage_95']:.3f}%"
)


print()
print("AUDIT / VERSIONING")
print("-" * 180)

print(
    f"Attempt ID                      : "
    f"{ATTEMPT_ID}"
)

print(
    f"Immutable gate directory        : "
    f"{ATTEMPT_DIR}"
)

print(
    f"Release decision                : "
    f"{RELEASE_DECISION_PATH}"
)

print(
    f"Known limitations               : "
    f"{KNOWN_LIMITATIONS_PATH}"
)

print(
    f"Audit manifest                  : "
    f"{AUDIT_PATH}"
)

print(
    f"Audit SHA256                    : "
    f"{AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("=" * 180)

if RELEASE_STATUS == "PASS":

    print(
        "CELL 59 STATUS: RELEASE GATES PASSED"
    )

elif RELEASE_STATUS == "PASS_WITH_MONITORING":

    print(
        "CELL 59 STATUS: RELEASE GATES PASSED WITH MONITORING — "
        "ELIGIBLE FOR IMMUTABLE RELEASE BUNDLE"
    )

else:

    print(
        "CELL 59 STATUS: RELEASE GATES FAILED — "
        "DO NOT BUNDLE OR PROMOTE"
    )

print("=" * 180)



V6.1 + U1 — FROZEN 2026 TEST UNCERTAINTY EVALUATION

TEST POPULATION
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Source TEST rows                : 12,865
Established/state-history rows  : 10,941
Athletes                        : 4,023
Date range                      : 2026-05-01 00:00:00 -> 2026-08-30 00:00:00
Mean state gap                  : 111.238 days
Median state gap                : 28.000 days

FROZEN SPECIFICATION
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
V5 tau                          : 98.807800438 days
B_M                             : 11.062709578
B_W                             : 6.989272507
G_M                             : 4.877472909
G_W                             : 4.335851597
nu               

,n,athletes,mean_error,MAE,RMSE,standardized_error_mean,standardized_error_sd,prediction_sigma_mean,prediction_sigma_median,coverage_50,...,coverage_80_CI_high,target_80_inside_CI,coverage_80_error_pp,mean_width_80,coverage_95,coverage_95_CI_low,coverage_95_CI_high,target_95_inside_CI,coverage_95_error_pp,mean_width_95
0,10941,4023,0.12144,2.539238,3.352574,0.046674,1.027785,3.188242,3.338088,49.803491,...,80.650642,True,-0.089571,7.967137,94.726259,94.291533,95.129589,True,-0.273741,12.64786



BY SEX
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,sex_group,n,athletes,mean_error,MAE,RMSE,standardized_error_mean,standardized_error_sd,prediction_sigma_mean,prediction_sigma_median,...,coverage_80_CI_high,target_80_inside_CI,coverage_80_error_pp,mean_width_80,coverage_95,coverage_95_CI_low,coverage_95_CI_high,target_95_inside_CI,coverage_95_error_pp,mean_width_95
0,M,5419,2091,0.231397,2.904817,3.813107,0.077237,1.065389,3.534533,3.383780,...,79.611160,False,-1.461524,8.832488,94.242480,93.590371,94.831908,False,-0.757520,14.021607
1,W,5522,1932,0.013534,2.180478,2.828653,0.016681,0.988673,2.848410,2.703662,...,82.264257,False,1.256791,7.117927,95.201014,94.605150,95.734032,True,0.201014,11.299736



BY MODEL GROUP
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


...


OVERALL RELEASE EVIDENCE
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,period,n,role
0,VALIDATION,4283,U1_CALIBRATION
1,TEST,10941,FROZEN_OUT_OF_SAMPLE_EVALUATION
2,POST_TEST,473,OPERATIONAL_MONITOR_ONLY



MONITORING ALERTS — NON-BLOCKING
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,scope,segment,metric,n,observed,reference,severity,blocking,reason
0,POST_TEST_OVERALL,ALL,coverage_95,473,92.600423,95.0,MONITOR,False,95% nominal target is outside the POST_TEST 95...
1,TEST_MODEL_GROUP,M_700,coverage_50,404,41.831683,50.0,MONITOR,False,50% nominal target is outside the frozen TEST ...
2,TEST_MODEL_GROUP,M_700,coverage_80,404,70.792079,80.0,MONITOR,False,80% nominal target is outside the frozen TEST ...
3,TEST_MODEL_GROUP,M_700,coverage_95,404,90.594059,95.0,MONITOR,False,95% nominal target is outside the frozen TEST ...
4,TEST_MODEL_GROUP,W_500,coverage_50,670,45.373134,50.0,MONITOR,False,50% nominal target is outside the frozen TEST ...
5,TEST_MODEL_GROUP,W_500,coverage_80,670,71.492537,80.0,MONITOR,False,80% nominal target is outside the frozen TEST ...
6,TEST_MODEL_GROUP,W_500,coverage_95,670,90.895522,95.0,MONITOR,False,95% nominal target is outside the frozen TEST ...
7,TEST_MODEL_GROUP,W_600,coverage_50,4852,51.875515,50.0,MONITOR,False,50% nominal target is outside the frozen TEST ...
8,TEST_MODEL_GROUP,W_600,coverage_80,4852,82.605111,80.0,MONITOR,False,80% nominal target is outside the frozen TEST ...
9,TEST_MODEL_GROUP,W_600,coverage_95,4852,95.795548,95.0,MONITOR,False,95% nominal target is outside the frozen TEST ...



KNOWN LIMITATIONS
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

ULIM-001 — uncertainty_model
  Description : U1 models predictive variance by sex and forecast horizon only. It does not currently include separate variance parameters for implement/model group.
  Impact      : Frozen TEST evidence shows materially lower interval coverage for some youth implement groups, particularly M_700 and W_500.
  Action      : Monitor group-level calibration in production. Evaluate group-specific uncertainty architecture only in a future version using independent development data.

ULIM-002 — post_test_calibration
  Description : POST_TEST overall 95% coverage is below nominal and its 95% Wilson interval does not contain the 95% target.
  Impact      : The frozen U1 intervals may be mildly too narrow in the most recent operational period.
  Action      : Continue fo

## Stage 20 — Immutable bundle and reload verification

Build the immutable runtime/evidence bundle and independently reload/reproduce the release contract.

- In `VALIDATE_REFACTOR`, these artifacts are built only under the temporary scratch root and are not a persistent model release.
- In `TRAIN_CANDIDATE` and `TRAIN_AND_PROMOTE`, the artifacts are stored persistently under the versioned training run.

**Legacy source cells represented:** bundle construction and reload verification preceding original promotion cell 56.  
**Execution rule:** run only after Stage 19 has completed successfully.

In [21]:
# ==================================================================================================
# REFACTORED STAGE 20 — Immutable bundle, reload verification and promotion
# ==================================================================================================

from __future__ import annotations


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 54
# Original source SHA256: 68c4d150a4419a0c9f6530feba0eab09a7de2806e3cb9ca47dde3b3a2c1b4cee
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 60 — BUILD IMMUTABLE V6.1 + U1 RELEASE BUNDLE
# ==================================================================================================
#
# Purpose
# -------
# Package the frozen release candidate into a self-contained, immutable model bundle.
#
# Release candidate:
#
#       Mean        : V6.1_OU_FRESHNESS_BLEND
#       Uncertainty : U1_SEX_LINEAR
#       Gate status : PASS_WITH_MONITORING / PASS
#
# The bundle contains:
#
#   runtime/
#       core trained-model artifacts required by prediction
#       frozen V6.1 architecture decision
#       frozen U1 uncertainty specification
#       V5 OU parameters
#       V6 calibration / persistence artifacts
#
#   evidence/
#       release decision
#       known limitations
#       calibration / TEST / POST_TEST audits
#
#   smoke_test/
#       deterministic prediction fixture for Cell 61
#
#   metadata/
#       runtime contract
#       pre-bundle training-manifest snapshot
#
# Historical backtest directories are NOT copied into the runtime bundle.
# Their provenance remains linked through hashes / parent artifacts.
#
# NO model fitting.
# NO model changes.
# NO ACTIVE_MODEL promotion.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_59_PASS",

    "FINAL_RELEASE_STATUS",
    "FINAL_RELEASE_ELIGIBLE_FOR_BUNDLE",

    "FINAL_RELEASE_GATE_DECISION_PATH",
    "FINAL_RELEASE_GATE_AUDIT_PATH",
    "FINAL_RELEASE_KNOWN_LIMITATIONS",

    "ARCHITECTURE_DECISION_ARTIFACT",

    "V61_U1_SPEC_PATH",
    "V61_U1_AUDIT_PATH",

    "FINAL_V61_TEST_UNCERTAINTY",
    "FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH",
    "FINAL_V61_POST_TEST_MONITOR_AUDIT_PATH",

    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_59_PASS:

    raise RuntimeError(
        "Cell 59 must pass before release bundling."
    )


if not FINAL_RELEASE_ELIGIBLE_FOR_BUNDLE:

    raise RuntimeError(
        "Release candidate is not eligible for bundling."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_60_BUILD_IMMUTABLE_RELEASE_BUNDLE"

BUNDLE_SCHEMA_VERSION = "1.0.0"

MODEL_FAMILY = "JAVELIN_V6_1"

MEAN_ARCHITECTURE = "V6.1_OU_FRESHNESS_BLEND"

UNCERTAINTY_MODEL = "U1_SEX_LINEAR"


RUN_DIR = Path(
    TRAINING_RUN_DIR
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def find_tau_days(
    obj,
):

    candidate_keys = {
        "tau",
        "tau_days",
        "state_tau",
        "state_tau_days",
        "ou_tau",
        "ou_tau_days",
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if (
                str(key).lower()
                in candidate_keys

                and

                isinstance(
                    value,
                    (int, float),
                )
            ):

                return float(value)


        for value in obj.values():

            result = find_tau_days(
                value
            )

            if result is not None:

                return result


    return None


def copy_and_record(
    source,
    destination,
    role,
):

    source = Path(source)

    destination = Path(destination)


    if not source.exists():

        raise RuntimeError(
            f"Bundle source missing for role '{role}':\n"
            f"{source}"
        )


    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        source,
        destination,
    )


    return {
        "role":
            role,

        "source_path":
            str(source),

        "source_sha256":
            sha256_file(source),

        "bundle_path":
            str(
                destination.relative_to(
                    BUNDLE_DIR
                )
            ),

        "bundle_sha256":
            sha256_file(destination),

        "size_bytes":
            int(
                destination.stat().st_size
            ),
    }


# --------------------------------------------------------------------------------------------------
# 4. CREATE IMMUTABLE BUNDLE DIRECTORY
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


RELEASE_ROOT = (
    RUN_DIR
    /
    "release_bundle"
)


BUNDLE_DIR = (
    RELEASE_ROOT
    /
    ATTEMPT_ID
)


RUNTIME_DIR = (
    BUNDLE_DIR
    /
    "runtime"
)


EVIDENCE_DIR = (
    BUNDLE_DIR
    /
    "evidence"
)


SMOKE_DIR = (
    BUNDLE_DIR
    /
    "smoke_test"
)


METADATA_DIR = (
    BUNDLE_DIR
    /
    "metadata"
)


for directory in [
    RUNTIME_DIR,
    EVIDENCE_DIR,
    SMOKE_DIR,
    METADATA_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=False,
    )


# --------------------------------------------------------------------------------------------------
# 5. SNAPSHOT TRAINING MANIFEST BEFORE BUNDLE UPDATE
# --------------------------------------------------------------------------------------------------

PRE_BUNDLE_MANIFEST_PATH = (
    METADATA_DIR
    /
    "training_manifest_pre_bundle.json"
)


atomic_json_write(
    TRAINING_MANIFEST,
    PRE_BUNDLE_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 6. AUTHORITATIVE ACTIVE SPECIFICATIONS
# --------------------------------------------------------------------------------------------------

BUNDLE_FILES = []


BUNDLE_FILES.append(
    copy_and_record(
        ARCHITECTURE_DECISION_ARTIFACT,
        RUNTIME_DIR
        /
        "architecture_decision.json",
        "mean_architecture_decision",
    )
)


BUNDLE_FILES.append(
    copy_and_record(
        V61_U1_SPEC_PATH,
        RUNTIME_DIR
        /
        "v61_uncertainty_spec.json",
        "uncertainty_spec",
    )
)


# --------------------------------------------------------------------------------------------------
# 7. V5 OU PARAMETERS
# --------------------------------------------------------------------------------------------------

V5_PARAMETER_PATH = (
    RUN_DIR
    /
    "v5_ou_parameters.json"
)


if not V5_PARAMETER_PATH.exists():

    raise RuntimeError(
        f"Required V5 parameter file missing:\n"
        f"{V5_PARAMETER_PATH}"
    )


BUNDLE_FILES.append(
    copy_and_record(
        V5_PARAMETER_PATH,
        RUNTIME_DIR
        /
        "v5_ou_parameters.json",
        "v5_ou_parameters",
    )
)


with V5_PARAMETER_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    V5_PARAMETER_JSON = json.load(f)


V5_TAU_DAYS = find_tau_days(
    V5_PARAMETER_JSON
)


if (
    V5_TAU_DAYS is None
    or
    not np.isfinite(V5_TAU_DAYS)
    or
    V5_TAU_DAYS <= 0
):

    raise RuntimeError(
        "Could not recover valid V5 tau from v5_ou_parameters.json."
    )


# --------------------------------------------------------------------------------------------------
# 8. CORE ROOT-LEVEL MODEL ARTIFACTS
# --------------------------------------------------------------------------------------------------
#
# Include serialised trained-model objects/specifications that live at the run root.
#
# We intentionally exclude:
#   • prediction/evaluation tables
#   • old superseded V6 uncertainty specification
#   • training / validation / test datasets
#
# --------------------------------------------------------------------------------------------------

SERIALIZED_EXTENSIONS = {
    ".json",
    ".pkl",
    ".pickle",
    ".nc",
    ".npz",
    ".npy",
}


EXCLUDED_ROOT_NAMES = {
    Path(
        TRAINING_MANIFEST_PATH
    ).name,

    "v6_uncertainty_spec.json",
}


EXCLUDED_NAME_TOKENS = [
    "test_prediction",
    "test_predictions",
    "post_test",
    "validation_prediction",
    "validation_predictions",
    "rolling_prediction",
    "rolling_predictions",
    "bootstrap",
    "diagnostic",
]


for source in sorted(
    RUN_DIR.iterdir()
):

    if not source.is_file():
        continue


    if source.name in EXCLUDED_ROOT_NAMES:
        continue


    lower_name = source.name.lower()


    if any(
        token in lower_name
        for token in EXCLUDED_NAME_TOKENS
    ):
        continue


    if source.suffix.lower() not in SERIALIZED_EXTENSIONS:
        continue


    # Files already copied under authoritative release names.
    if source.resolve() in {
        V5_PARAMETER_PATH.resolve(),
        Path(
            V61_U1_SPEC_PATH
        ).resolve(),
        Path(
            ARCHITECTURE_DECISION_ARTIFACT
        ).resolve(),
    }:
        continue


    destination = (
        RUNTIME_DIR
        /
        "core"
        /
        source.name
    )


    BUNDLE_FILES.append(
        copy_and_record(
            source,
            destination,
            "core_model_artifact",
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. SELECT ROOT PARQUET ARTIFACTS NEEDED BY RUNTIME
# --------------------------------------------------------------------------------------------------
#
# These typically contain persistent athlete mappings / states / V4 feature metadata.
#
# --------------------------------------------------------------------------------------------------

RUNTIME_PARQUET_TOKENS = [
    "state_sequence",
    "terminal_state",
    "terminal_states",
    "athlete_state",
    "athlete_states",
    "v4_features",
    "mapping",
]


for source in sorted(
    RUN_DIR.glob(
        "*.parquet"
    )
):

    lower_name = source.name.lower()


    if not any(
        token in lower_name
        for token in RUNTIME_PARQUET_TOKENS
    ):
        continue


    destination = (
        RUNTIME_DIR
        /
        "core"
        /
        source.name
    )


    BUNDLE_FILES.append(
        copy_and_record(
            source,
            destination,
            "runtime_state_or_feature_artifact",
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. CURRENT V6 MEAN-CALIBRATION / PERSISTENCE ARTIFACTS FROM CHECKPOINTS
# --------------------------------------------------------------------------------------------------
#
# We copy only current-run checkpoint files whose names indicate deployment-time
# V6 mean calibration, persistence, offsets, or model specification.
#
# Historical backtests are explicitly excluded.
#
# --------------------------------------------------------------------------------------------------

CHECKPOINT_DIR = (
    RUN_DIR
    /
    "checkpoints"
)


checkpoint_tokens = [
    "v6",
    "persistence",
    "offset",
    "mean_calibration",
    "mean_spec",
]


checkpoint_required_tokens = [
    "persistence",
    "offset",
    "calibration",
    "mean",
    "spec",
]


if CHECKPOINT_DIR.exists():

    for source in sorted(
        CHECKPOINT_DIR.iterdir()
    ):

        if not source.is_file():
            continue


        lower_name = source.name.lower()


        if "uncertainty" in lower_name:
            continue


        if not (
            "v6"
            in lower_name
            and
            any(
                token in lower_name
                for token in checkpoint_required_tokens
            )
        ):
            continue


        if source.suffix.lower() not in {
            ".json",
            ".pkl",
            ".pickle",
            ".parquet",
            ".npz",
            ".npy",
        }:
            continue


        destination = (
            RUNTIME_DIR
            /
            "v6_calibration"
            /
            source.name
        )


        BUNDLE_FILES.append(
            copy_and_record(
                source,
                destination,
                "v6_mean_calibration_artifact",
            )
        )


# --------------------------------------------------------------------------------------------------
# 11. RELEASE / EVALUATION EVIDENCE
# --------------------------------------------------------------------------------------------------

EVIDENCE_SOURCES = [
    (
        FINAL_RELEASE_GATE_DECISION_PATH,
        "release_gate_decision.json",
        "release_gate_decision",
    ),

    (
        FINAL_RELEASE_GATE_AUDIT_PATH,
        "release_gate_audit.json",
        "release_gate_audit",
    ),

    (
        V61_U1_AUDIT_PATH,
        "v61_uncertainty_calibration_audit.json",
        "uncertainty_calibration_audit",
    ),

    (
        FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH,
        "test_uncertainty_evaluation_audit.json",
        "test_uncertainty_evaluation_audit",
    ),

    (
        FINAL_V61_POST_TEST_MONITOR_AUDIT_PATH,
        "post_test_monitor_audit.json",
        "post_test_monitor_audit",
    ),
]


for source, filename, role in EVIDENCE_SOURCES:

    BUNDLE_FILES.append(
        copy_and_record(
            source,
            EVIDENCE_DIR
            /
            filename,
            role,
        )
    )


# Copy known limitations from the authoritative release-gate attempt.
with Path(
    FINAL_RELEASE_GATE_DECISION_PATH
).open(
    "r",
    encoding="utf-8",
) as f:

    RELEASE_GATE_DECISION = json.load(f)


KNOWN_LIMITATIONS_SOURCE = Path(
    RELEASE_GATE_DECISION[
        "known_limitations_file"
    ]
)


BUNDLE_FILES.append(
    copy_and_record(
        KNOWN_LIMITATIONS_SOURCE,
        EVIDENCE_DIR
        /
        "known_limitations.json",
        "known_limitations",
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BUILD DETERMINISTIC SMOKE-TEST FIXTURE
# --------------------------------------------------------------------------------------------------
#
# Cell 61 will load ONLY bundle contents and recompute:
#
#   • V6.1 structural weight
#   • V6.1 point prediction
#   • U1 variance
#   • 50 / 80 / 95 intervals
#
# against these frozen expected values.
#
# --------------------------------------------------------------------------------------------------

FIXTURE_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",

    "sex_group",
    "model_group",

    "state_gap_days",

    "v5_prediction_mean",
    "v6_prediction_mean",

    "v61_structural_weight",
    "v61_prediction_mean",

    "v61_prediction_variance",
    "v61_prediction_sigma",

    "v61_lower_50",
    "v61_upper_50",

    "v61_lower_80",
    "v61_upper_80",

    "v61_lower_95",
    "v61_upper_95",
]


missing_fixture_columns = [
    column
    for column in FIXTURE_COLUMNS
    if column
    not in FINAL_V61_TEST_UNCERTAINTY.columns
]


if missing_fixture_columns:

    raise RuntimeError(
        "Frozen TEST data is missing smoke-test columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_fixture_columns
        )
    )


SMOKE_FIXTURE = (
    FINAL_V61_TEST_UNCERTAINTY[
        FIXTURE_COLUMNS
    ]
    .sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
            "model_group",
        ]
    )
    .head(
        128
    )
    .reset_index(
        drop=True
    )
    .copy()
)


SMOKE_FIXTURE_PATH = (
    SMOKE_DIR
    /
    "runtime_prediction_fixture.parquet"
)


SMOKE_FIXTURE.to_parquet(
    SMOKE_FIXTURE_PATH,
    index=False,
)


BUNDLE_FILES.append(
    {
        "role":
            "runtime_smoke_test_fixture",

        "source_path":
            str(
                FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH
            ),

        "source_sha256":
            sha256_file(
                FINAL_V61_TEST_UNCERTAINTY_AUDIT_PATH
            ),

        "bundle_path":
            str(
                SMOKE_FIXTURE_PATH.relative_to(
                    BUNDLE_DIR
                )
            ),

        "bundle_sha256":
            sha256_file(
                SMOKE_FIXTURE_PATH
            ),

        "size_bytes":
            int(
                SMOKE_FIXTURE_PATH.stat().st_size
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 13. RUNTIME CONTRACT
# --------------------------------------------------------------------------------------------------

with Path(
    V61_U1_SPEC_PATH
).open(
    "r",
    encoding="utf-8",
) as f:

    U1_SPEC = json.load(f)


with Path(
    ARCHITECTURE_DECISION_ARTIFACT
).open(
    "r",
    encoding="utf-8",
) as f:

    MEAN_DECISION = json.load(f)


RUNTIME_CONTRACT = {

    "contract_schema_version":
        BUNDLE_SCHEMA_VERSION,

    "model_family":
        MODEL_FAMILY,

    "mean_architecture":
        MEAN_ARCHITECTURE,

    "uncertainty_model":
        UNCERTAINTY_MODEL,

    "release_status":
        FINAL_RELEASE_STATUS,

    "mean_formula":
        (
            "v61_prediction_mean = "
            "v5_prediction_mean + "
            "(1 - exp(-state_gap_days / tau_v5_days)) "
            "* (v6_prediction_mean - v5_prediction_mean)"
        ),

    "uncertainty_variance_formula":
        (
            "variance = B_sex + "
            "G_sex * state_gap_days / 365.25"
        ),

    "student_t_scale_formula":
        (
            "sqrt(variance) * sqrt((nu - 2) / nu)"
        ),

    "V5_tau_days":
        float(
            V5_TAU_DAYS
        ),

    "U1": {
        "B":
            U1_SPEC[
                "B"
            ],

        "G":
            U1_SPEC[
                "G"
            ],

        "nu":
            float(
                U1_SPEC[
                    "nu"
                ]
            ),
    },

    "prediction_interval_levels":
        [
            0.50,
            0.80,
            0.95,
        ],

    "required_runtime_inputs_for_final_layers": [
        "sex_group",
        "state_gap_days",
        "v5_prediction_mean",
        "v6_prediction_mean",
    ],

    "supported_model_groups": [
        "M_700",
        "M_800",
        "W_500",
        "W_600",
    ],

    "mean_architecture_decision_status":
        MEAN_DECISION.get(
            "decision_status"
        ),

    "release_notes": {
        "monitoring_required":
            bool(
                FINAL_RELEASE_STATUS
                ==
                "PASS_WITH_MONITORING"
            ),

        "known_limitations":
            FINAL_RELEASE_KNOWN_LIMITATIONS,
    },
}


RUNTIME_CONTRACT_PATH = (
    METADATA_DIR
    /
    "runtime_contract.json"
)


atomic_json_write(
    RUNTIME_CONTRACT,
    RUNTIME_CONTRACT_PATH,
)


BUNDLE_FILES.append(
    {
        "role":
            "runtime_contract",

        "source_path":
            None,

        "source_sha256":
            None,

        "bundle_path":
            str(
                RUNTIME_CONTRACT_PATH.relative_to(
                    BUNDLE_DIR
                )
            ),

        "bundle_sha256":
            sha256_file(
                RUNTIME_CONTRACT_PATH
            ),

        "size_bytes":
            int(
                RUNTIME_CONTRACT_PATH.stat().st_size
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 14. ADD PRE-BUNDLE MANIFEST SNAPSHOT TO INDEX
# --------------------------------------------------------------------------------------------------

BUNDLE_FILES.append(
    {
        "role":
            "training_manifest_pre_bundle_snapshot",

        "source_path":
            str(
                TRAINING_MANIFEST_PATH
            ),

        "source_sha256":
            (
                sha256_file(
                    TRAINING_MANIFEST_PATH
                )
                if Path(
                    TRAINING_MANIFEST_PATH
                ).exists()
                else None
            ),

        "bundle_path":
            str(
                PRE_BUNDLE_MANIFEST_PATH.relative_to(
                    BUNDLE_DIR
                )
            ),

        "bundle_sha256":
            sha256_file(
                PRE_BUNDLE_MANIFEST_PATH
            ),

        "size_bytes":
            int(
                PRE_BUNDLE_MANIFEST_PATH.stat().st_size
            ),
    }
)


# --------------------------------------------------------------------------------------------------
# 15. VERIFY COPY INTEGRITY
# --------------------------------------------------------------------------------------------------

COPY_HASH_MATCH_PASS = bool(
    all(
        (
            item[
                "source_sha256"
            ]
            is None
        )
        or
        (
            item[
                "source_sha256"
            ]
            ==
            item[
                "bundle_sha256"
            ]
        )

        for item
        in BUNDLE_FILES
        if item[
            "role"
        ]
        not in {
            "runtime_smoke_test_fixture",
            "training_manifest_pre_bundle_snapshot",
        }
    )
)


# --------------------------------------------------------------------------------------------------
# 16. BUNDLE FILE INDEX
# --------------------------------------------------------------------------------------------------

FILE_INDEX = (
    pd.DataFrame(
        BUNDLE_FILES
    )
    .sort_values(
        [
            "role",
            "bundle_path",
        ]
    )
    .reset_index(
        drop=True
    )
)


FILE_INDEX_PATH = (
    METADATA_DIR
    /
    "bundle_file_index.parquet"
)


FILE_INDEX.to_parquet(
    FILE_INDEX_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 17. BUNDLE QA
# --------------------------------------------------------------------------------------------------

HAS_RUNTIME_CONTRACT_PASS = bool(
    RUNTIME_CONTRACT_PATH.exists()
)


HAS_SMOKE_FIXTURE_PASS = bool(
    SMOKE_FIXTURE_PATH.exists()
    and
    len(
        SMOKE_FIXTURE
    )
    >
    0
)


HAS_V5_PARAMETERS_PASS = bool(
    (
        RUNTIME_DIR
        /
        "v5_ou_parameters.json"
    ).exists()
)


HAS_MEAN_DECISION_PASS = bool(
    (
        RUNTIME_DIR
        /
        "architecture_decision.json"
    ).exists()
)


HAS_U1_SPEC_PASS = bool(
    (
        RUNTIME_DIR
        /
        "v61_uncertainty_spec.json"
    ).exists()
)


HAS_CORE_ARTIFACTS_PASS = bool(
    len(
        FILE_INDEX.loc[
            FILE_INDEX[
                "role"
            ]
            .isin(
                [
                    "core_model_artifact",
                    "runtime_state_or_feature_artifact",
                    "v6_mean_calibration_artifact",
                ]
            )
        ]
    )
    >
    0
)


RELEASE_STATUS_PASS = bool(
    FINAL_RELEASE_STATUS
    in {
        "PASS",
        "PASS_WITH_MONITORING",
    }
)


CELL_60_PASS = bool(
    COPY_HASH_MATCH_PASS
    and
    HAS_RUNTIME_CONTRACT_PASS
    and
    HAS_SMOKE_FIXTURE_PASS
    and
    HAS_V5_PARAMETERS_PASS
    and
    HAS_MEAN_DECISION_PASS
    and
    HAS_U1_SPEC_PASS
    and
    HAS_CORE_ARTIFACTS_PASS
    and
    RELEASE_STATUS_PASS
)


# --------------------------------------------------------------------------------------------------
# 18. BUNDLE MANIFEST
# --------------------------------------------------------------------------------------------------

GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


BUNDLE_MANIFEST = {

    "bundle_schema_version":
        BUNDLE_SCHEMA_VERSION,

    "bundle_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "BUILT_PENDING_REPRODUCIBILITY_CHECK"
            if CELL_60_PASS
            else
            "BUILD_FAILED"
        ),

    "model_family":
        MODEL_FAMILY,

    "source_model_version":
        TRAINING_MANIFEST.get(
            "model_version"
        ),

    "source_run_directory":
        str(
            RUN_DIR
        ),

    "mean_architecture":
        MEAN_ARCHITECTURE,

    "uncertainty_model":
        UNCERTAINTY_MODEL,

    "release_gate_status":
        FINAL_RELEASE_STATUS,

    "eligible_for_active_promotion":
        False,

    "promotion_blocker":
        (
            "Cell 61 bundle reload/reproducibility check "
            "has not yet been completed."
        ),

    "monitoring_required":
        bool(
            FINAL_RELEASE_STATUS
            ==
            "PASS_WITH_MONITORING"
        ),

    "runtime_contract":
        str(
            RUNTIME_CONTRACT_PATH.relative_to(
                BUNDLE_DIR
            )
        ),

    "smoke_test_fixture":
        str(
            SMOKE_FIXTURE_PATH.relative_to(
                BUNDLE_DIR
            )
        ),

    "file_index":
        str(
            FILE_INDEX_PATH.relative_to(
                BUNDLE_DIR
            )
        ),

    "file_count":
        int(
            len(
                FILE_INDEX
            )
        ),

    "total_bundle_payload_bytes":
        int(
            FILE_INDEX[
                "size_bytes"
            ]
            .sum()
        ),

    "qa": {
        "copy_hash_match":
            COPY_HASH_MATCH_PASS,

        "runtime_contract_present":
            HAS_RUNTIME_CONTRACT_PASS,

        "smoke_fixture_present":
            HAS_SMOKE_FIXTURE_PASS,

        "V5_parameters_present":
            HAS_V5_PARAMETERS_PASS,

        "mean_decision_present":
            HAS_MEAN_DECISION_PASS,

        "U1_spec_present":
            HAS_U1_SPEC_PASS,

        "core_runtime_artifacts_present":
            HAS_CORE_ARTIFACTS_PASS,

        "release_status_eligible":
            RELEASE_STATUS_PASS,

        "cell_pass":
            CELL_60_PASS,
    },

    "git_commit":
        GIT_COMMIT,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },
}


BUNDLE_MANIFEST_PATH = (
    BUNDLE_DIR
    /
    "model_release_bundle_manifest.json"
)


atomic_json_write(
    BUNDLE_MANIFEST,
    BUNDLE_MANIFEST_PATH,
)


BUNDLE_MANIFEST_SHA256 = sha256_file(
    BUNDLE_MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. BUNDLE AUDIT
# --------------------------------------------------------------------------------------------------

BUNDLE_AUDIT = {

    "audit_schema_version":
        BUNDLE_SCHEMA_VERSION,

    "stage":
        "IMMUTABLE_RELEASE_BUNDLE_BUILD",

    "cell_id":
        CELL_ID,

    "bundle_id":
        ATTEMPT_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_60_PASS
            else
            "FAILED"
        ),

    "model_changed":
        False,

    "parameters_fitted":
        False,

    "active_model_promoted":
        False,

    "release_status":
        FINAL_RELEASE_STATUS,

    "bundle_directory":
        str(
            BUNDLE_DIR
        ),

    "bundle_manifest":
        str(
            BUNDLE_MANIFEST_PATH
        ),

    "bundle_manifest_sha256":
        BUNDLE_MANIFEST_SHA256,

    "bundle_file_count":
        int(
            len(
                FILE_INDEX
            )
        ),

    "copy_hash_match":
        COPY_HASH_MATCH_PASS,

    "historical_backtests_copied":
        False,

    "next_stage":
        "BUNDLE_RELOAD_REPRODUCIBILITY_CHECK",
}


BUNDLE_AUDIT_PATH = (
    BUNDLE_DIR
    /
    "bundle_build_audit.json"
)


atomic_json_write(
    BUNDLE_AUDIT,
    BUNDLE_AUDIT_PATH,
)


BUNDLE_AUDIT_SHA256 = sha256_file(
    BUNDLE_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 20. LATEST BUNDLE POINTER
# --------------------------------------------------------------------------------------------------

LATEST_BUNDLE_POINTER = (
    RELEASE_ROOT
    /
    "LATEST_RELEASE_BUNDLE.json"
)


atomic_json_write(
    {
        "bundle_id":
            ATTEMPT_ID,

        "status":
            (
                "BUILT_PENDING_REPRODUCIBILITY_CHECK"
                if CELL_60_PASS
                else
                "BUILD_FAILED"
            ),

        "release_status":
            FINAL_RELEASE_STATUS,

        "bundle_directory":
            str(
                BUNDLE_DIR
            ),

        "bundle_manifest":
            str(
                BUNDLE_MANIFEST_PATH
            ),

        "bundle_manifest_sha256":
            BUNDLE_MANIFEST_SHA256,

        "bundle_audit":
            str(
                BUNDLE_AUDIT_PATH
            ),

        "bundle_audit_sha256":
            BUNDLE_AUDIT_SHA256,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_BUNDLE_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 21. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "release_bundle"
] = {

    "status":
        (
            "BUILT_PENDING_REPRODUCIBILITY_CHECK"
            if CELL_60_PASS
            else
            "BUILD_FAILED"
        ),

    "bundle_id":
        ATTEMPT_ID,

    "bundle_directory":
        str(
            BUNDLE_DIR
        ),

    "latest_pointer":
        str(
            LATEST_BUNDLE_POINTER
        ),

    "bundle_manifest":
        str(
            BUNDLE_MANIFEST_PATH
        ),

    "bundle_manifest_sha256":
        BUNDLE_MANIFEST_SHA256,

    "bundle_audit":
        str(
            BUNDLE_AUDIT_PATH
        ),

    "bundle_audit_sha256":
        BUNDLE_AUDIT_SHA256,

    "active_model_promoted":
        False,
}


TRAINING_MANIFEST[
    "status"
] = (
    "RELEASE_BUNDLE_BUILT_PENDING_REPRODUCIBILITY"
    if CELL_60_PASS
    else
    "RELEASE_BUNDLE_BUILD_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. GLOBALS
# --------------------------------------------------------------------------------------------------

FINAL_RELEASE_BUNDLE_DIR = (
    BUNDLE_DIR
)


FINAL_RELEASE_BUNDLE_MANIFEST_PATH = (
    BUNDLE_MANIFEST_PATH
)


FINAL_RELEASE_BUNDLE_AUDIT_PATH = (
    BUNDLE_AUDIT_PATH
)


FINAL_RELEASE_BUNDLE_FILE_INDEX = (
    FILE_INDEX
)


FINAL_RELEASE_RUNTIME_CONTRACT_PATH = (
    RUNTIME_CONTRACT_PATH
)


FINAL_RELEASE_SMOKE_FIXTURE_PATH = (
    SMOKE_FIXTURE_PATH
)


NOTEBOOK_2_CELL_60_PASS = (
    CELL_60_PASS
)


# --------------------------------------------------------------------------------------------------
# 23. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 180)
print("IMMUTABLE V6.1 + U1 RELEASE BUNDLE")
print("=" * 180)


print()
print("RELEASE CANDIDATE")
print("-" * 180)

print(
    f"Mean architecture               : "
    f"{MEAN_ARCHITECTURE}"
)

print(
    f"Uncertainty model               : "
    f"{UNCERTAINTY_MODEL}"
)

print(
    f"Release gate status             : "
    f"{FINAL_RELEASE_STATUS}"
)

print(
    f"Monitoring required             : "
    f"{FINAL_RELEASE_STATUS == 'PASS_WITH_MONITORING'}"
)

print(
    f"ACTIVE_MODEL promoted           : "
    f"False"
)


print()
print("BUNDLE CONTENTS")
print("-" * 180)

display(
    FILE_INDEX[
        [
            "role",
            "bundle_path",
            "size_bytes",
            "bundle_sha256",
        ]
    ]
)


print()
print("BUNDLE SUMMARY")
print("-" * 180)

print(
    f"Bundle ID                       : "
    f"{ATTEMPT_ID}"
)

print(
    f"Bundle directory                : "
    f"{BUNDLE_DIR}"
)

print(
    f"Files indexed                   : "
    f"{len(FILE_INDEX):,}"
)

print(
    f"Payload size                    : "
    f"{FILE_INDEX['size_bytes'].sum() / (1024**2):.3f} MB"
)

print(
    f"Smoke-test rows                 : "
    f"{len(SMOKE_FIXTURE):,}"
)


print()
print("CORE RUNTIME CONSTANTS")
print("-" * 180)

print(
    f"V5 tau                          : "
    f"{V5_TAU_DAYS:.9f} days"
)

print(
    f"U1 B_M                          : "
    f"{float(U1_SPEC['B']['M']):.9f}"
)

print(
    f"U1 B_W                          : "
    f"{float(U1_SPEC['B']['W']):.9f}"
)

print(
    f"U1 G_M                          : "
    f"{float(U1_SPEC['G']['M']):.9f}"
)

print(
    f"U1 G_W                          : "
    f"{float(U1_SPEC['G']['W']):.9f}"
)

print(
    f"U1 nu                           : "
    f"{float(U1_SPEC['nu']):.9f}"
)


print()
print("QA")
print("-" * 180)

print(
    f"Copied file hashes match        : "
    f"{COPY_HASH_MATCH_PASS}"
)

print(
    f"Runtime contract present        : "
    f"{HAS_RUNTIME_CONTRACT_PASS}"
)

print(
    f"Smoke fixture present           : "
    f"{HAS_SMOKE_FIXTURE_PASS}"
)

print(
    f"V5 parameters present           : "
    f"{HAS_V5_PARAMETERS_PASS}"
)

print(
    f"Mean decision present           : "
    f"{HAS_MEAN_DECISION_PASS}"
)

print(
    f"U1 specification present        : "
    f"{HAS_U1_SPEC_PASS}"
)

print(
    f"Core runtime artifacts present  : "
    f"{HAS_CORE_ARTIFACTS_PASS}"
)

print(
    f"Release status eligible         : "
    f"{RELEASE_STATUS_PASS}"
)


print()
print("AUDIT / VERSIONING")
print("-" * 180)

print(
    f"Bundle manifest                 : "
    f"{BUNDLE_MANIFEST_PATH}"
)

print(
    f"Bundle manifest SHA256          : "
    f"{BUNDLE_MANIFEST_SHA256}"
)

print(
    f"Bundle audit                    : "
    f"{BUNDLE_AUDIT_PATH}"
)

print(
    f"Bundle audit SHA256             : "
    f"{BUNDLE_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("=" * 180)

if CELL_60_PASS:

    print(
        "CELL 60 STATUS: IMMUTABLE RELEASE BUNDLE BUILT — "
        "PENDING RELOAD / REPRODUCIBILITY CHECK"
    )

else:

    print(
        "CELL 60 STATUS: RELEASE BUNDLE BUILD REVIEW REQUIRED"
    )

print("=" * 180)


# --------------------------------------------------------------------------------------------------
# Preserved legacy source cell 55
# Original source SHA256: f4a0d076825f450571dfb475bb2380571caee9b1868a1fa2efc947ed578a987d
# --------------------------------------------------------------------------------------------------

# ==================================================================================================
# CELL 61 — RELEASE BUNDLE RELOAD + REPRODUCIBILITY VERIFICATION
# ==================================================================================================
#
# Purpose
# -------
# Verify that the immutable release bundle can be independently reloaded from disk
# and reproduce the frozen final-layer predictions exactly.
#
# IMPORTANT
# ---------
# This cell deliberately does NOT use live model globals for prediction.
#
# It reloads from the bundle:
#
#   metadata/runtime_contract.json
#   runtime/v5_ou_parameters.json
#   runtime/v61_uncertainty_spec.json
#   smoke_test/runtime_prediction_fixture.parquet
#   metadata/bundle_file_index.parquet
#
# It then:
#
#   1. verifies every indexed bundle-file SHA256
#   2. verifies runtime-contract consistency
#   3. reconstructs V6.1 mean predictions
#   4. reconstructs U1 variance / sigma
#   5. reconstructs 50 / 80 / 95 intervals
#   6. compares reproduced outputs with the frozen fixture
#
# Verification output is written OUTSIDE the bundle so the release bundle remains immutable.
#
# NO fitting.
# NO model changes.
# NO ACTIVE_MODEL promotion.
#
# ==================================================================================================

import hashlib
import json
import os
import platform
import subprocess
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# --------------------------------------------------------------------------------------------------
# 1. PREREQUISITES
# --------------------------------------------------------------------------------------------------

required = [
    "NOTEBOOK_2_CELL_60_PASS",
    "FINAL_RELEASE_BUNDLE_DIR",
    "FINAL_RELEASE_BUNDLE_MANIFEST_PATH",
    "TRAINING_RUN_DIR",
    "TRAINING_MANIFEST",
    "TRAINING_MANIFEST_PATH",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Missing prerequisites:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing
        )
    )


if not NOTEBOOK_2_CELL_60_PASS:

    raise RuntimeError(
        "Cell 60 must pass before bundle verification."
    )


# --------------------------------------------------------------------------------------------------
# 2. DEFINITIONS
# --------------------------------------------------------------------------------------------------

CELL_ID = "CELL_61_RELEASE_BUNDLE_REPRODUCIBILITY"

VERIFICATION_SCHEMA_VERSION = "1.0.0"

ABS_TOL = 1e-10

REL_TOL = 1e-10


BUNDLE_DIR = Path(
    FINAL_RELEASE_BUNDLE_DIR
)


RUN_DIR = Path(
    TRAINING_RUN_DIR
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def atomic_json_write(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )

    os.replace(
        tmp,
        path,
    )


def git_commit_or_none(
    start_path,
):

    try:

        result = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            cwd=str(start_path),
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:

        return None


def find_tau_days(
    obj,
):

    candidate_keys = {
        "tau",
        "tau_days",
        "state_tau",
        "state_tau_days",
        "ou_tau",
        "ou_tau_days",
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if (
                str(key).lower()
                in candidate_keys

                and

                isinstance(
                    value,
                    (int, float),
                )
            ):

                return float(value)


        for value in obj.values():

            result = find_tau_days(
                value
            )

            if result is not None:

                return result


    return None


def max_abs_diff(
    a,
    b,
):

    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )

    return float(
        np.max(
            np.abs(
                a - b
            )
        )
    )


def arrays_close(
    a,
    b,
    atol=ABS_TOL,
    rtol=REL_TOL,
):

    return bool(
        np.allclose(
            np.asarray(
                a,
                dtype=float,
            ),
            np.asarray(
                b,
                dtype=float,
            ),
            atol=atol,
            rtol=rtol,
            equal_nan=False,
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD BUNDLE MANIFEST
# --------------------------------------------------------------------------------------------------

BUNDLE_MANIFEST_PATH = Path(
    FINAL_RELEASE_BUNDLE_MANIFEST_PATH
)


if not BUNDLE_MANIFEST_PATH.exists():

    raise RuntimeError(
        f"Bundle manifest missing:\n"
        f"{BUNDLE_MANIFEST_PATH}"
    )


with BUNDLE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    BUNDLE_MANIFEST = json.load(f)


BUNDLE_ID = str(
    BUNDLE_MANIFEST[
        "bundle_id"
    ]
)


# --------------------------------------------------------------------------------------------------
# 5. RESOLVE BUNDLE FILES FROM MANIFEST ONLY
# --------------------------------------------------------------------------------------------------

FILE_INDEX_PATH = (
    BUNDLE_DIR
    /
    BUNDLE_MANIFEST[
        "file_index"
    ]
)


RUNTIME_CONTRACT_PATH = (
    BUNDLE_DIR
    /
    BUNDLE_MANIFEST[
        "runtime_contract"
    ]
)


SMOKE_FIXTURE_PATH = (
    BUNDLE_DIR
    /
    BUNDLE_MANIFEST[
        "smoke_test_fixture"
    ]
)


V5_PARAMETER_PATH = (
    BUNDLE_DIR
    /
    "runtime"
    /
    "v5_ou_parameters.json"
)


U1_SPEC_PATH = (
    BUNDLE_DIR
    /
    "runtime"
    /
    "v61_uncertainty_spec.json"
)


MEAN_DECISION_PATH = (
    BUNDLE_DIR
    /
    "runtime"
    /
    "architecture_decision.json"
)


required_bundle_paths = {
    "file_index":
        FILE_INDEX_PATH,

    "runtime_contract":
        RUNTIME_CONTRACT_PATH,

    "smoke_fixture":
        SMOKE_FIXTURE_PATH,

    "v5_parameters":
        V5_PARAMETER_PATH,

    "u1_spec":
        U1_SPEC_PATH,

    "mean_decision":
        MEAN_DECISION_PATH,
}


MISSING_BUNDLE_PATHS = [
    name
    for name, path
    in required_bundle_paths.items()
    if not path.exists()
]


if MISSING_BUNDLE_PATHS:

    raise RuntimeError(
        "Required bundle files missing:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in MISSING_BUNDLE_PATHS
        )
    )


# --------------------------------------------------------------------------------------------------
# 6. LOAD FILE INDEX
# --------------------------------------------------------------------------------------------------

FILE_INDEX = pd.read_parquet(
    FILE_INDEX_PATH
)


required_index_columns = [
    "role",
    "bundle_path",
    "bundle_sha256",
    "size_bytes",
]


missing_index_columns = [
    c
    for c in required_index_columns
    if c not in FILE_INDEX.columns
]


if missing_index_columns:

    raise RuntimeError(
        "Bundle file index missing columns:\n"
        +
        "\n".join(
            f"  - {c}"
            for c in missing_index_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY EVERY INDEXED FILE HASH
# --------------------------------------------------------------------------------------------------

hash_rows = []


for _, row in FILE_INDEX.iterrows():

    relative_path = Path(
        str(
            row[
                "bundle_path"
            ]
        )
    )


    path = (
        BUNDLE_DIR
        /
        relative_path
    )


    exists = path.exists()


    actual_size = (
        int(
            path.stat().st_size
        )
        if exists
        else None
    )


    expected_size = int(
        row[
            "size_bytes"
        ]
    )


    size_match = bool(
        exists
        and
        actual_size
        ==
        expected_size
    )


    actual_sha = (
        sha256_file(
            path
        )
        if exists
        else None
    )


    expected_sha = str(
        row[
            "bundle_sha256"
        ]
    )


    hash_match = bool(
        exists
        and
        actual_sha
        ==
        expected_sha
    )


    hash_rows.append(
        {
            "role":
                row[
                    "role"
                ],

            "bundle_path":
                str(
                    relative_path
                ),

            "exists":
                exists,

            "size_match":
                size_match,

            "hash_match":
                hash_match,

            "expected_size_bytes":
                expected_size,

            "actual_size_bytes":
                actual_size,

            "expected_sha256":
                expected_sha,

            "actual_sha256":
                actual_sha,
        }
    )


HASH_VERIFICATION = pd.DataFrame(
    hash_rows
)


ALL_FILES_EXIST_PASS = bool(
    HASH_VERIFICATION[
        "exists"
    ]
    .all()
)


ALL_FILE_SIZES_PASS = bool(
    HASH_VERIFICATION[
        "size_match"
    ]
    .all()
)


ALL_FILE_HASHES_PASS = bool(
    HASH_VERIFICATION[
        "hash_match"
    ]
    .all()
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY BUNDLE MANIFEST HASH AGAINST TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

CURRENT_BUNDLE_MANIFEST_SHA256 = sha256_file(
    BUNDLE_MANIFEST_PATH
)


EXPECTED_BUNDLE_MANIFEST_SHA256 = (
    TRAINING_MANIFEST[
        "release_bundle"
    ][
        "bundle_manifest_sha256"
    ]
)


BUNDLE_MANIFEST_HASH_PASS = bool(
    CURRENT_BUNDLE_MANIFEST_SHA256
    ==
    EXPECTED_BUNDLE_MANIFEST_SHA256
)


# --------------------------------------------------------------------------------------------------
# 9. LOAD RUNTIME CONTRACT AND AUTHORITATIVE SPECS
# --------------------------------------------------------------------------------------------------

with RUNTIME_CONTRACT_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    CONTRACT = json.load(f)


with V5_PARAMETER_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    V5_SPEC = json.load(f)


with U1_SPEC_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    U1_SPEC = json.load(f)


with MEAN_DECISION_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    MEAN_DECISION = json.load(f)


# --------------------------------------------------------------------------------------------------
# 10. RELOAD RUNTIME CONSTANTS FROM BUNDLE ONLY
# --------------------------------------------------------------------------------------------------

TAU_FROM_V5_SPEC = find_tau_days(
    V5_SPEC
)


if (
    TAU_FROM_V5_SPEC is None
    or
    not np.isfinite(
        TAU_FROM_V5_SPEC
    )
    or
    TAU_FROM_V5_SPEC <= 0
):

    raise RuntimeError(
        "Could not recover valid V5 tau from bundled V5 specification."
    )


TAU_FROM_CONTRACT = float(
    CONTRACT[
        "V5_tau_days"
    ]
)


B_M = float(
    U1_SPEC[
        "B"
    ][
        "M"
    ]
)


B_W = float(
    U1_SPEC[
        "B"
    ][
        "W"
    ]
)


G_M = float(
    U1_SPEC[
        "G"
    ][
        "M"
    ]
)


G_W = float(
    U1_SPEC[
        "G"
    ][
        "W"
    ]
)


NU = float(
    U1_SPEC[
        "nu"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. CONTRACT CONSISTENCY CHECKS
# --------------------------------------------------------------------------------------------------

TAU_CONSISTENCY_PASS = bool(
    np.isclose(
        TAU_FROM_V5_SPEC,
        TAU_FROM_CONTRACT,
        atol=1e-12,
        rtol=1e-12,
    )
)


U1_CONTRACT_PASS = bool(

    np.isclose(
        float(
            CONTRACT[
                "U1"
            ][
                "B"
            ][
                "M"
            ]
        ),
        B_M,
        atol=1e-12,
        rtol=1e-12,
    )

    and

    np.isclose(
        float(
            CONTRACT[
                "U1"
            ][
                "B"
            ][
                "W"
            ]
        ),
        B_W,
        atol=1e-12,
        rtol=1e-12,
    )

    and

    np.isclose(
        float(
            CONTRACT[
                "U1"
            ][
                "G"
            ][
                "M"
            ]
        ),
        G_M,
        atol=1e-12,
        rtol=1e-12,
    )

    and

    np.isclose(
        float(
            CONTRACT[
                "U1"
            ][
                "G"
            ][
                "W"
            ]
        ),
        G_W,
        atol=1e-12,
        rtol=1e-12,
    )

    and

    np.isclose(
        float(
            CONTRACT[
                "U1"
            ][
                "nu"
            ]
        ),
        NU,
        atol=1e-12,
        rtol=1e-12,
    )
)


MEAN_ARCHITECTURE_CONTRACT_PASS = bool(
    CONTRACT[
        "mean_architecture"
    ]
    ==
    "V6.1_OU_FRESHNESS_BLEND"

    and

    MEAN_DECISION[
        "selected_mean_architecture"
    ]
    ==
    "V6.1_OU_FRESHNESS_BLEND"
)


UNCERTAINTY_MODEL_CONTRACT_PASS = bool(
    CONTRACT[
        "uncertainty_model"
    ]
    ==
    "U1_SEX_LINEAR"

    and

    U1_SPEC[
        "model"
    ]
    ==
    "U1_SEX_LINEAR"
)


RELEASE_STATUS_CONTRACT_PASS = bool(
    CONTRACT[
        "release_status"
    ]
    in {
        "PASS",
        "PASS_WITH_MONITORING",
    }

    and

    CONTRACT[
        "release_status"
    ]
    ==
    BUNDLE_MANIFEST[
        "release_gate_status"
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. LOAD SMOKE FIXTURE
# --------------------------------------------------------------------------------------------------

FIXTURE = pd.read_parquet(
    SMOKE_FIXTURE_PATH
)


required_fixture_columns = [
    "sex_group",
    "state_gap_days",

    "v5_prediction_mean",
    "v6_prediction_mean",

    "v61_structural_weight",
    "v61_prediction_mean",

    "v61_prediction_variance",
    "v61_prediction_sigma",

    "v61_lower_50",
    "v61_upper_50",

    "v61_lower_80",
    "v61_upper_80",

    "v61_lower_95",
    "v61_upper_95",
]


missing_fixture_columns = [
    c
    for c in required_fixture_columns
    if c not in FIXTURE.columns
]


if missing_fixture_columns:

    raise RuntimeError(
        "Smoke fixture missing required columns:\n"
        +
        "\n".join(
            f"  - {c}"
            for c in missing_fixture_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 13. REPRODUCE V6.1 FROM BUNDLED SPECS ONLY
# --------------------------------------------------------------------------------------------------

REPRO = FIXTURE.copy()


REPRO[
    "repro_v5_state_persistence_phi"
] = np.exp(
    -
    REPRO[
        "state_gap_days"
    ]
    .astype(float)
    /
    TAU_FROM_CONTRACT
)


REPRO[
    "repro_v61_structural_weight"
] = (
    1.0
    -
    REPRO[
        "repro_v5_state_persistence_phi"
    ]
)


REPRO[
    "repro_v61_prediction_mean"
] = (
    REPRO[
        "v5_prediction_mean"
    ]
    +
    REPRO[
        "repro_v61_structural_weight"
    ]
    *
    (
        REPRO[
            "v6_prediction_mean"
        ]
        -
        REPRO[
            "v5_prediction_mean"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 14. REPRODUCE U1 VARIANCE / SIGMA
# --------------------------------------------------------------------------------------------------

H_YEAR = (
    REPRO[
        "state_gap_days"
    ]
    .astype(float)
    /
    365.25
)


IS_M = (
    REPRO[
        "sex_group"
    ]
    .astype(str)
    .eq(
        "M"
    )
)


REPRO[
    "repro_v61_prediction_variance"
] = np.where(
    IS_M,

    B_M
    +
    G_M
    *
    H_YEAR,

    B_W
    +
    G_W
    *
    H_YEAR,
)


REPRO[
    "repro_v61_prediction_sigma"
] = np.sqrt(
    REPRO[
        "repro_v61_prediction_variance"
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. REPRODUCE 50 / 80 / 95 INTERVALS
# --------------------------------------------------------------------------------------------------

for level in [
    0.50,
    0.80,
    0.95,
]:

    pct = int(
        100
        *
        level
    )


    q = float(
        student_t.ppf(
            (
                1.0
                +
                level
            )
            /
            2.0,
            df=NU,
        )

        *

        np.sqrt(
            (
                NU
                -
                2.0
            )
            /
            NU
        )
    )


    half_width = (
        q
        *
        REPRO[
            "repro_v61_prediction_sigma"
        ]
    )


    REPRO[
        f"repro_v61_lower_{pct}"
    ] = (
        REPRO[
            "repro_v61_prediction_mean"
        ]
        -
        half_width
    )


    REPRO[
        f"repro_v61_upper_{pct}"
    ] = (
        REPRO[
            "repro_v61_prediction_mean"
        ]
        +
        half_width
    )


# --------------------------------------------------------------------------------------------------
# 16. REPRODUCIBILITY COMPARISONS
# --------------------------------------------------------------------------------------------------

comparison_specs = [
    (
        "v61_structural_weight",
        "repro_v61_structural_weight",
    ),

    (
        "v61_prediction_mean",
        "repro_v61_prediction_mean",
    ),

    (
        "v61_prediction_variance",
        "repro_v61_prediction_variance",
    ),

    (
        "v61_prediction_sigma",
        "repro_v61_prediction_sigma",
    ),

    (
        "v61_lower_50",
        "repro_v61_lower_50",
    ),

    (
        "v61_upper_50",
        "repro_v61_upper_50",
    ),

    (
        "v61_lower_80",
        "repro_v61_lower_80",
    ),

    (
        "v61_upper_80",
        "repro_v61_upper_80",
    ),

    (
        "v61_lower_95",
        "repro_v61_lower_95",
    ),

    (
        "v61_upper_95",
        "repro_v61_upper_95",
    ),
]


repro_rows = []


for expected_col, reproduced_col in comparison_specs:

    max_diff = max_abs_diff(
        REPRO[
            expected_col
        ],
        REPRO[
            reproduced_col
        ],
    )


    passed = arrays_close(
        REPRO[
            expected_col
        ],
        REPRO[
            reproduced_col
        ],
    )


    repro_rows.append(
        {
            "expected_column":
                expected_col,

            "reproduced_column":
                reproduced_col,

            "max_abs_diff":
                max_diff,

            "tolerance_abs":
                ABS_TOL,

            "tolerance_rel":
                REL_TOL,

            "pass":
                passed,
        }
    )


REPRO_COMPARISON = pd.DataFrame(
    repro_rows
)


ALL_REPRODUCIBILITY_CHECKS_PASS = bool(
    REPRO_COMPARISON[
        "pass"
    ]
    .all()
)


# --------------------------------------------------------------------------------------------------
# 17. GENERAL NUMERIC / DOMAIN QA
# --------------------------------------------------------------------------------------------------

FIXTURE_NONEMPTY_PASS = bool(
    len(
        FIXTURE
    )
    >
    0
)


SEX_DOMAIN_PASS = bool(
    set(
        REPRO[
            "sex_group"
        ]
        .astype(str)
        .unique()
    )
    .issubset(
        {
            "M",
            "W",
        }
    )
)


GAP_DOMAIN_PASS = bool(
    (
        REPRO[
            "state_gap_days"
        ]
        .astype(float)
        >
        0
    )
    .all()
)


WEIGHT_DOMAIN_PASS = bool(
    REPRO[
        "repro_v61_structural_weight"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


VARIANCE_DOMAIN_PASS = bool(
    (
        REPRO[
            "repro_v61_prediction_variance"
        ]
        >
        0
    )
    .all()
)


FINITE_REPRO_PASS = bool(
    np.isfinite(
        REPRO[
            [
                "repro_v61_structural_weight",
                "repro_v61_prediction_mean",
                "repro_v61_prediction_variance",
                "repro_v61_prediction_sigma",

                "repro_v61_lower_50",
                "repro_v61_upper_50",

                "repro_v61_lower_80",
                "repro_v61_upper_80",

                "repro_v61_lower_95",
                "repro_v61_upper_95",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )
    .all()
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL CELL PASS
# --------------------------------------------------------------------------------------------------

CELL_61_PASS = bool(

    ALL_FILES_EXIST_PASS

    and

    ALL_FILE_SIZES_PASS

    and

    ALL_FILE_HASHES_PASS

    and

    BUNDLE_MANIFEST_HASH_PASS

    and

    TAU_CONSISTENCY_PASS

    and

    U1_CONTRACT_PASS

    and

    MEAN_ARCHITECTURE_CONTRACT_PASS

    and

    UNCERTAINTY_MODEL_CONTRACT_PASS

    and

    RELEASE_STATUS_CONTRACT_PASS

    and

    FIXTURE_NONEMPTY_PASS

    and

    SEX_DOMAIN_PASS

    and

    GAP_DOMAIN_PASS

    and

    WEIGHT_DOMAIN_PASS

    and

    VARIANCE_DOMAIN_PASS

    and

    FINITE_REPRO_PASS

    and

    ALL_REPRODUCIBILITY_CHECKS_PASS
)


# --------------------------------------------------------------------------------------------------
# 19. CREATE IMMUTABLE VERIFICATION ATTEMPT OUTSIDE BUNDLE
# --------------------------------------------------------------------------------------------------

RUN_UTC = datetime.now(
    timezone.utc
)


ATTEMPT_ID = RUN_UTC.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


VERIFICATION_ROOT = (
    RUN_DIR
    /
    "release_verification"
    /
    BUNDLE_ID
)


VERIFICATION_DIR = (
    VERIFICATION_ROOT
    /
    ATTEMPT_ID
)


VERIFICATION_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


HASH_VERIFICATION_PATH = (
    VERIFICATION_DIR
    /
    "bundle_hash_verification.parquet"
)


REPRO_COMPARISON_PATH = (
    VERIFICATION_DIR
    /
    "runtime_reproducibility_comparison.parquet"
)


REPRO_SAMPLE_PATH = (
    VERIFICATION_DIR
    /
    "runtime_reproduced_fixture.parquet"
)


VERIFICATION_AUDIT_PATH = (
    VERIFICATION_DIR
    /
    "bundle_reproducibility_audit.json"
)


HASH_VERIFICATION.to_parquet(
    HASH_VERIFICATION_PATH,
    index=False,
)


REPRO_COMPARISON.to_parquet(
    REPRO_COMPARISON_PATH,
    index=False,
)


REPRO.to_parquet(
    REPRO_SAMPLE_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 20. VERIFICATION AUDIT
# --------------------------------------------------------------------------------------------------

GIT_COMMIT = git_commit_or_none(
    Path.cwd()
)


VERIFICATION_AUDIT = {

    "audit_schema_version":
        VERIFICATION_SCHEMA_VERSION,

    "stage":
        "RELEASE_BUNDLE_RELOAD_REPRODUCIBILITY",

    "cell_id":
        CELL_ID,

    "verification_id":
        ATTEMPT_ID,

    "bundle_id":
        BUNDLE_ID,

    "created_utc":
        RUN_UTC.isoformat(),

    "status":
        (
            "PASSED"
            if CELL_61_PASS
            else
            "FAILED"
        ),

    "bundle_directory":
        str(
            BUNDLE_DIR
        ),

    "bundle_manifest":
        str(
            BUNDLE_MANIFEST_PATH
        ),

    "bundle_manifest_sha256":
        CURRENT_BUNDLE_MANIFEST_SHA256,

    "fixture_rows":
        int(
            len(
                FIXTURE
            )
        ),

    "numerical_tolerances": {
        "absolute":
            ABS_TOL,

        "relative":
            REL_TOL,
    },

    "runtime_constants": {
        "V5_tau_days":
            TAU_FROM_CONTRACT,

        "U1_B_M":
            B_M,

        "U1_B_W":
            B_W,

        "U1_G_M":
            G_M,

        "U1_G_W":
            G_W,

        "U1_nu":
            NU,
    },

    "qa": {
        "all_files_exist":
            ALL_FILES_EXIST_PASS,

        "all_file_sizes_match":
            ALL_FILE_SIZES_PASS,

        "all_file_hashes_match":
            ALL_FILE_HASHES_PASS,

        "bundle_manifest_hash_match":
            BUNDLE_MANIFEST_HASH_PASS,

        "V5_tau_contract_consistency":
            TAU_CONSISTENCY_PASS,

        "U1_contract_consistency":
            U1_CONTRACT_PASS,

        "mean_architecture_contract":
            MEAN_ARCHITECTURE_CONTRACT_PASS,

        "uncertainty_model_contract":
            UNCERTAINTY_MODEL_CONTRACT_PASS,

        "release_status_contract":
            RELEASE_STATUS_CONTRACT_PASS,

        "fixture_nonempty":
            FIXTURE_NONEMPTY_PASS,

        "sex_domain":
            SEX_DOMAIN_PASS,

        "positive_state_gaps":
            GAP_DOMAIN_PASS,

        "V61_weight_bounds":
            WEIGHT_DOMAIN_PASS,

        "positive_variance":
            VARIANCE_DOMAIN_PASS,

        "finite_reproduction":
            FINITE_REPRO_PASS,

        "all_numeric_reproduction_checks":
            ALL_REPRODUCIBILITY_CHECKS_PASS,

        "cell_pass":
            CELL_61_PASS,
    },

    "max_absolute_differences": {
        row[
            "expected_column"
        ]:
            float(
                row[
                    "max_abs_diff"
                ]
            )

        for _, row
        in REPRO_COMPARISON.iterrows()
    },

    "artifacts": {
        "hash_verification":
            str(
                HASH_VERIFICATION_PATH
            ),

        "reproducibility_comparison":
            str(
                REPRO_COMPARISON_PATH
            ),

        "reproduced_fixture":
            str(
                REPRO_SAMPLE_PATH
            ),
    },

    "model_changed":
        False,

    "parameters_fitted":
        False,

    "bundle_modified":
        False,

    "active_model_promoted":
        False,

    "next_stage":
        (
            "ACTIVE_MODEL_PROMOTION"
            if CELL_61_PASS
            else
            "RELEASE_BUNDLE_REVIEW_REQUIRED"
        ),

    "git_commit":
        GIT_COMMIT,

    "software": {
        "python":
            sys.version,

        "platform":
            platform.platform(),

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,
    },
}


atomic_json_write(
    VERIFICATION_AUDIT,
    VERIFICATION_AUDIT_PATH,
)


VERIFICATION_AUDIT_SHA256 = sha256_file(
    VERIFICATION_AUDIT_PATH
)


# --------------------------------------------------------------------------------------------------
# 21. LATEST VERIFICATION POINTER
# --------------------------------------------------------------------------------------------------

LATEST_VERIFICATION_POINTER = (
    VERIFICATION_ROOT
    /
    "LATEST_REPRODUCIBILITY_CHECK.json"
)


atomic_json_write(
    {
        "bundle_id":
            BUNDLE_ID,

        "verification_id":
            ATTEMPT_ID,

        "status":
            (
                "PASSED"
                if CELL_61_PASS
                else
                "FAILED"
            ),

        "verification_directory":
            str(
                VERIFICATION_DIR
            ),

        "verification_audit":
            str(
                VERIFICATION_AUDIT_PATH
            ),

        "verification_audit_sha256":
            VERIFICATION_AUDIT_SHA256,

        "bundle_modified":
            False,

        "active_model_promoted":
            False,

        "created_utc":
            RUN_UTC.isoformat(),
    },

    LATEST_VERIFICATION_POINTER,
)


# --------------------------------------------------------------------------------------------------
# 22. UPDATE TRAINING MANIFEST
# --------------------------------------------------------------------------------------------------

TRAINING_MANIFEST[
    "release_verification"
] = {

    "status":
        (
            "PASSED"
            if CELL_61_PASS
            else
            "FAILED"
        ),

    "bundle_id":
        BUNDLE_ID,

    "verification_id":
        ATTEMPT_ID,

    "verification_directory":
        str(
            VERIFICATION_DIR
        ),

    "latest_pointer":
        str(
            LATEST_VERIFICATION_POINTER
        ),

    "verification_audit":
        str(
            VERIFICATION_AUDIT_PATH
        ),

    "verification_audit_sha256":
        VERIFICATION_AUDIT_SHA256,

    "bundle_modified":
        False,

    "active_model_promoted":
        False,
}


TRAINING_MANIFEST[
    "status"
] = (
    "RELEASE_BUNDLE_VERIFIED_READY_FOR_PROMOTION"
    if CELL_61_PASS
    else
    "RELEASE_BUNDLE_REPRODUCIBILITY_FAILED"
)


atomic_json_write(
    TRAINING_MANIFEST,
    TRAINING_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 23. GLOBALS
# --------------------------------------------------------------------------------------------------

FINAL_RELEASE_VERIFICATION_DIR = (
    VERIFICATION_DIR
)


FINAL_RELEASE_VERIFICATION_AUDIT_PATH = (
    VERIFICATION_AUDIT_PATH
)


FINAL_RELEASE_VERIFICATION_AUDIT_SHA256 = (
    VERIFICATION_AUDIT_SHA256
)


FINAL_RELEASE_REPRO_COMPARISON = (
    REPRO_COMPARISON
)


FINAL_RELEASE_HASH_VERIFICATION = (
    HASH_VERIFICATION
)


NOTEBOOK_2_CELL_61_PASS = (
    CELL_61_PASS
)


# --------------------------------------------------------------------------------------------------
# 24. OUTPUT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 180)
print("RELEASE BUNDLE — RELOAD / REPRODUCIBILITY VERIFICATION")
print("=" * 180)


print()
print("BUNDLE")
print("-" * 180)

print(
    f"Bundle ID                       : "
    f"{BUNDLE_ID}"
)

print(
    f"Bundle directory                : "
    f"{BUNDLE_DIR}"
)

print(
    f"Indexed files                   : "
    f"{len(FILE_INDEX):,}"
)

print(
    f"Fixture rows                    : "
    f"{len(FIXTURE):,}"
)


print()
print("FULL BUNDLE HASH VERIFICATION")
print("-" * 180)

display(
    HASH_VERIFICATION[
        [
            "role",
            "bundle_path",
            "exists",
            "size_match",
            "hash_match",
        ]
    ]
)


print()
print("RUNTIME CONTRACT")
print("-" * 180)

print(
    f"Mean architecture               : "
    f"{CONTRACT['mean_architecture']}"
)

print(
    f"Uncertainty model               : "
    f"{CONTRACT['uncertainty_model']}"
)

print(
    f"Release status                  : "
    f"{CONTRACT['release_status']}"
)

print(
    f"V5 tau                          : "
    f"{TAU_FROM_CONTRACT:.12f}"
)

print(
    f"U1 B_M                          : "
    f"{B_M:.12f}"
)

print(
    f"U1 B_W                          : "
    f"{B_W:.12f}"
)

print(
    f"U1 G_M                          : "
    f"{G_M:.12f}"
)

print(
    f"U1 G_W                          : "
    f"{G_W:.12f}"
)

print(
    f"U1 nu                           : "
    f"{NU:.12f}"
)


print()
print("NUMERICAL REPRODUCIBILITY")
print("-" * 180)

display(
    REPRO_COMPARISON
)


print()
print("QA")
print("-" * 180)

print(
    f"All indexed files exist         : "
    f"{ALL_FILES_EXIST_PASS}"
)

print(
    f"All indexed sizes match         : "
    f"{ALL_FILE_SIZES_PASS}"
)

print(
    f"All indexed SHA256 hashes match : "
    f"{ALL_FILE_HASHES_PASS}"
)

print(
    f"Bundle manifest hash matches    : "
    f"{BUNDLE_MANIFEST_HASH_PASS}"
)

print(
    f"V5 tau contract consistent      : "
    f"{TAU_CONSISTENCY_PASS}"
)

print(
    f"U1 contract consistent          : "
    f"{U1_CONTRACT_PASS}"
)

print(
    f"Mean architecture consistent    : "
    f"{MEAN_ARCHITECTURE_CONTRACT_PASS}"
)

print(
    f"Uncertainty model consistent    : "
    f"{UNCERTAINTY_MODEL_CONTRACT_PASS}"
)

print(
    f"Release status consistent       : "
    f"{RELEASE_STATUS_CONTRACT_PASS}"
)

print(
    f"Fixture non-empty               : "
    f"{FIXTURE_NONEMPTY_PASS}"
)

print(
    f"Sex domain valid                : "
    f"{SEX_DOMAIN_PASS}"
)

print(
    f"State gaps positive             : "
    f"{GAP_DOMAIN_PASS}"
)

print(
    f"V6.1 weight bounds valid        : "
    f"{WEIGHT_DOMAIN_PASS}"
)

print(
    f"Prediction variance positive    : "
    f"{VARIANCE_DOMAIN_PASS}"
)

print(
    f"Finite reproduced outputs       : "
    f"{FINITE_REPRO_PASS}"
)

print(
    f"All numerical checks reproduce  : "
    f"{ALL_REPRODUCIBILITY_CHECKS_PASS}"
)

print(
    f"Bundle modified                 : "
    f"False"
)

print(
    f"ACTIVE_MODEL promoted           : "
    f"False"
)


print()
print("AUDIT / VERSIONING")
print("-" * 180)

print(
    f"Verification ID                 : "
    f"{ATTEMPT_ID}"
)

print(
    f"Verification directory          : "
    f"{VERIFICATION_DIR}"
)

print(
    f"Verification audit              : "
    f"{VERIFICATION_AUDIT_PATH}"
)

print(
    f"Verification audit SHA256       : "
    f"{VERIFICATION_AUDIT_SHA256}"
)

print(
    f"Git commit                      : "
    f"{GIT_COMMIT}"
)


print()
print("=" * 180)

if CELL_61_PASS:

    print(
        "CELL 61 STATUS: RELEASE BUNDLE VERIFIED AND REPRODUCIBLE — "
        "READY FOR ACTIVE_MODEL PROMOTION"
    )

else:

    print(
        "CELL 61 STATUS: RELEASE BUNDLE VERIFICATION FAILED — "
        "DO NOT PROMOTE"
    )

print("=" * 180)

# ==================================================================================================
# EXECUTION-MODE STATUS AFTER BUNDLE VERIFICATION
# ==================================================================================================
print()
print("EXECUTION MODE RELEASE CONTROL")
print("-" * 180)
print(f"Execution mode                  : {EXECUTION_MODE}")
print(f"Persistent artifacts            : {PERSISTENT_ARTIFACTS_ENABLED}")
print(f"ACTIVE_MODEL promotion enabled  : {ACTIVE_MODEL_PROMOTION_ENABLED}")
if IS_VALIDATION_MODE:
    print("Release bundle persistence      : TRANSIENT SCRATCH ONLY")
elif EXECUTION_MODE == "TRAIN_CANDIDATE":
    print("Release bundle persistence      : PERSISTENT CANDIDATE; NOT ACTIVE")
else:
    print("Release bundle persistence      : PERSISTENT; ELIGIBLE FOR PROMOTION GATE")



IMMUTABLE V6.1 + U1 RELEASE BUNDLE

RELEASE CANDIDATE
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Mean architecture               : V6.1_OU_FRESHNESS_BLEND
Uncertainty model               : U1_SEX_LINEAR
Release gate status             : PASS_WITH_MONITORING
Monitoring required             : True
ACTIVE_MODEL promoted           : False

BUNDLE CONTENTS
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,role,bundle_path,size_bytes,bundle_sha256
0,core_model_artifact,runtime/core/historical_backtest_spec.json,1369,c5558c10ddf88ac1e719a6ab7a68ab6e179a43b3a8ba9d...
1,core_model_artifact,runtime/core/v4_model_spec.json,1021,938bf838783eeecf3b490b9b2c0473798e472c390354eb...
2,core_model_artifact,runtime/core/v4_structural_posterior.nc,3335285016,1cf722416d4813026f56461ba947fec2b46b298d0aaef7...
3,core_model_artifact,runtime/core/v61_ou_freshness_blend_candidate....,385,287b787c30432e0371df17edde12f51858e83022ca0b2d...
4,core_model_artifact,runtime/core/v6_mean_spec.json,631,2899887295e4d86bdffe55a57e9d0d0c2181953f67e5fa...
5,core_model_artifact,runtime/core/v6_release_gate.json,1263,481a4840fe121f611d8a265fea14c59d37d172b3c53315...
6,core_model_artifact,runtime/core/v6_spline_metadata.json,808,7d487e40865d95e2c303e0f47d3645f233457342b09128...
7,core_model_artifact,runtime/core/v6_validation_short_gap_confirmat...,591,eb6be11d6be0553c1ee05ab1f6242b54a86569654e6680...
8,known_limitations,evidence/known_limitations.json,1486,279dc0d3358d7761d34537f3f077f30a1a7fa9f5bcf844...
9,mean_architecture_decision,runtime/architecture_decision.json,4480,08be252d5a0efe404c81df72600571671c808fd7cdbb42...



BUNDLE SUMMARY
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Bundle ID                       : 20260922T141214_541460Z
Bundle directory                : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_/V6_20260922_125751/release_bundle/20260922T141214_541460Z
Files indexed                   : 28
Payload size                    : 3188.204 MB
Smoke-test rows                 : 128

CORE RUNTIME CONSTANTS
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
V5 tau                          : 98.807800438 days
U1 B_M                          : 11.062709578
U1 B_W                          : 6.989272507
U1 G_M                          : 4.877472909
U1 G_W                          : 4.335851597
U

,role,bundle_path,exists,size_match,hash_match
0,core_model_artifact,runtime/core/historical_backtest_spec.json,True,True,True
1,core_model_artifact,runtime/core/v4_model_spec.json,True,True,True
2,core_model_artifact,runtime/core/v4_structural_posterior.nc,True,True,True
3,core_model_artifact,runtime/core/v61_ou_freshness_blend_candidate....,True,True,True
4,core_model_artifact,runtime/core/v6_mean_spec.json,True,True,True
5,core_model_artifact,runtime/core/v6_release_gate.json,True,True,True
6,core_model_artifact,runtime/core/v6_spline_metadata.json,True,True,True
7,core_model_artifact,runtime/core/v6_validation_short_gap_confirmat...,True,True,True
8,known_limitations,evidence/known_limitations.json,True,True,True
9,mean_architecture_decision,runtime/architecture_decision.json,True,True,True



RUNTIME CONTRACT
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Mean architecture               : V6.1_OU_FRESHNESS_BLEND
Uncertainty model               : U1_SEX_LINEAR
Release status                  : PASS_WITH_MONITORING
V5 tau                          : 98.807800437864
U1 B_M                          : 11.062709577760
U1 B_W                          : 6.989272506862
U1 G_M                          : 4.877472909228
U1 G_W                          : 4.335851597262
U1 nu                           : 15.577413931939

NUMERICAL REPRODUCIBILITY
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,expected_column,reproduced_column,max_abs_diff,tolerance_abs,tolerance_rel,pass
0,v61_structural_weight,repro_v61_structural_weight,0.0,1.000000e-10,1.000000e-10,True
1,v61_prediction_mean,repro_v61_prediction_mean,0.0,1.000000e-10,1.000000e-10,True
2,v61_prediction_variance,repro_v61_prediction_variance,0.0,1.000000e-10,1.000000e-10,True
3,v61_prediction_sigma,repro_v61_prediction_sigma,0.0,1.000000e-10,1.000000e-10,True
4,v61_lower_50,repro_v61_lower_50,0.0,1.000000e-10,1.000000e-10,True
5,v61_upper_50,repro_v61_upper_50,0.0,1.000000e-10,1.000000e-10,True
6,v61_lower_80,repro_v61_lower_80,0.0,1.000000e-10,1.000000e-10,True
7,v61_upper_80,repro_v61_upper_80,0.0,1.000000e-10,1.000000e-10,True
8,v61_lower_95,repro_v61_lower_95,0.0,1.000000e-10,1.000000e-10,True
9,v61_upper_95,repro_v61_upper_95,0.0,1.000000e-10,1.000000e-10,True



QA
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
All indexed files exist         : True
All indexed sizes match         : True
All indexed SHA256 hashes match : True
Bundle manifest hash matches    : True
V5 tau contract consistent      : True
U1 contract consistent          : True
Mean architecture consistent    : True
Uncertainty model consistent    : True
Release status consistent       : True
Fixture non-empty               : True
Sex domain valid                : True
State gaps positive             : True
V6.1 weight bounds valid        : True
Prediction variance positive    : True
Finite reproduced outputs       : True
All numerical checks reproduce  : True
Bundle modified                 : False
ACTIVE_MODEL promoted           : False

AUDIT / VERSIONING
-------------------------------------------------------------------------------------------

## Stage 21 — `ACTIVE_MODEL` promotion gate

This stage is execution-mode controlled:

- `VALIDATE_REFACTOR` — promotion is skipped.
- `TRAIN_CANDIDATE` — promotion is skipped.
- `TRAIN_AND_PROMOTE` — the original validated promotion logic runs unchanged after all prerequisite gates pass.

Skipping promotion is considered a successful execution outcome for the first two modes; it does **not** modify `ACTIVE_MODEL.json`.

In [22]:
# ==================================================================================================
# STAGE 21 — EXECUTION-MODE-CONTROLLED ACTIVE_MODEL PROMOTION
# ==================================================================================================

if ACTIVE_MODEL_PROMOTION_ENABLED:
    # --------------------------------------------------------------------------------------------------
    # Preserved legacy source cell 56
    # Original source SHA256: fc23768c9989386732c1085deef3b71891be6ddc7ba10a493661e89787355724
    # --------------------------------------------------------------------------------------------------

    # ==================================================================================================
    # CELL 62 — PROMOTE VERIFIED RELEASE BUNDLE TO ACTIVE_MODEL
    # ==================================================================================================
    #
    # Purpose
    # -------
    # Atomically promote the verified immutable V6.1 + U1 release bundle.
    #
    # Promotion means:
    #
    #     model_artifacts/ACTIVE_MODEL.json
    #
    # will point to the verified bundle produced by Cell 60 and verified by Cell 61.
    #
    # IMPORTANT
    # ---------
    # • NO fitting.
    # • NO model changes.
    # • NO bundle modification.
    # • NO parameter changes.
    #
    # If an ACTIVE_MODEL.json already exists:
    #   • preserve it immutably in active_model_history/
    #   • record its SHA256
    #   • then atomically replace the pointer
    #
    # This is the FINAL CELL of Notebook 2.
    #
    # ==================================================================================================

    import hashlib
    import json
    import os
    import platform
    import shutil
    import subprocess
    import sys

    from datetime import datetime, timezone
    from pathlib import Path

    import numpy as np
    import pandas as pd


    # --------------------------------------------------------------------------------------------------
    # 1. PREREQUISITES
    # --------------------------------------------------------------------------------------------------

    required = [
        "NOTEBOOK_2_CELL_61_PASS",

        "FINAL_RELEASE_STATUS",
        "FINAL_RELEASE_ELIGIBLE_FOR_BUNDLE",

        "FINAL_RELEASE_BUNDLE_DIR",
        "FINAL_RELEASE_BUNDLE_MANIFEST_PATH",

        "FINAL_RELEASE_VERIFICATION_DIR",
        "FINAL_RELEASE_VERIFICATION_AUDIT_PATH",
        "FINAL_RELEASE_VERIFICATION_AUDIT_SHA256",

        "FINAL_RELEASE_GATE_DECISION_PATH",
        "FINAL_RELEASE_GATE_AUDIT_PATH",

        "TRAINING_RUN_DIR",
        "TRAINING_MANIFEST",
        "TRAINING_MANIFEST_PATH",
    ]

    missing = [
        name
        for name in required
        if name not in globals()
    ]

    if missing:

        raise RuntimeError(
            "Missing prerequisites:\n"
            +
            "\n".join(
                f"  - {name}"
                for name in missing
            )
        )


    if not NOTEBOOK_2_CELL_61_PASS:

        raise RuntimeError(
            "Cell 61 reproducibility verification must pass before promotion."
        )


    if not FINAL_RELEASE_ELIGIBLE_FOR_BUNDLE:

        raise RuntimeError(
            "Release candidate is not eligible for promotion."
        )


    if FINAL_RELEASE_STATUS not in {
        "PASS",
        "PASS_WITH_MONITORING",
    }:

        raise RuntimeError(
            f"Invalid release status for promotion: {FINAL_RELEASE_STATUS}"
        )


    # --------------------------------------------------------------------------------------------------
    # 2. DEFINITIONS
    # --------------------------------------------------------------------------------------------------

    CELL_ID = "CELL_62_ACTIVE_MODEL_PROMOTION"

    PROMOTION_SCHEMA_VERSION = "1.0.0"

    RUN_DIR = Path(
        TRAINING_RUN_DIR
    )


    MODEL_ARTIFACT_ROOT = (
        RUN_DIR.parent
    )


    ACTIVE_MODEL_PATH = (
        MODEL_ARTIFACT_ROOT
        /
        "ACTIVE_MODEL.json"
    )


    ACTIVE_HISTORY_ROOT = (
        MODEL_ARTIFACT_ROOT
        /
        "active_model_history"
    )


    PROMOTION_ROOT = (
        RUN_DIR
        /
        "promotion"
    )


    # --------------------------------------------------------------------------------------------------
    # 3. HELPERS
    # --------------------------------------------------------------------------------------------------

    def sha256_file(
        path,
        chunk_size=8 * 1024 * 1024,
    ):

        path = Path(path)

        h = hashlib.sha256()

        with path.open(
            "rb"
        ) as f:

            while True:

                chunk = f.read(
                    chunk_size
                )

                if not chunk:
                    break

                h.update(
                    chunk
                )

        return h.hexdigest()


    def atomic_json_write(
        obj,
        path,
    ):

        path = Path(path)

        path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        tmp = path.with_suffix(
            path.suffix + ".tmp"
        )


        with tmp.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                obj,
                f,
                indent=2,
                default=str,
            )

            f.flush()

            os.fsync(
                f.fileno()
            )


        os.replace(
            tmp,
            path,
        )


    def git_commit_or_none(
        start_path,
    ):

        try:

            result = subprocess.run(
                [
                    "git",
                    "rev-parse",
                    "HEAD",
                ],
                cwd=str(start_path),
                capture_output=True,
                text=True,
                check=True,
            )

            return result.stdout.strip()

        except Exception:

            return None


    # --------------------------------------------------------------------------------------------------
    # 4. LOAD VERIFIED RELEASE ARTIFACTS
    # --------------------------------------------------------------------------------------------------

    BUNDLE_DIR = Path(
        FINAL_RELEASE_BUNDLE_DIR
    )


    BUNDLE_MANIFEST_PATH = Path(
        FINAL_RELEASE_BUNDLE_MANIFEST_PATH
    )


    VERIFICATION_DIR = Path(
        FINAL_RELEASE_VERIFICATION_DIR
    )


    VERIFICATION_AUDIT_PATH = Path(
        FINAL_RELEASE_VERIFICATION_AUDIT_PATH
    )


    RELEASE_GATE_DECISION_PATH = Path(
        FINAL_RELEASE_GATE_DECISION_PATH
    )


    RELEASE_GATE_AUDIT_PATH = Path(
        FINAL_RELEASE_GATE_AUDIT_PATH
    )


    required_paths = {
        "bundle_directory":
            BUNDLE_DIR,

        "bundle_manifest":
            BUNDLE_MANIFEST_PATH,

        "verification_directory":
            VERIFICATION_DIR,

        "verification_audit":
            VERIFICATION_AUDIT_PATH,

        "release_gate_decision":
            RELEASE_GATE_DECISION_PATH,

        "release_gate_audit":
            RELEASE_GATE_AUDIT_PATH,
    }


    missing_paths = [
        name
        for name, path
        in required_paths.items()
        if not path.exists()
    ]


    if missing_paths:

        raise RuntimeError(
            "Promotion prerequisite artifacts missing:\n"
            +
            "\n".join(
                f"  - {name}"
                for name in missing_paths
            )
        )


    # --------------------------------------------------------------------------------------------------
    # 5. LOAD BUNDLE / VERIFICATION / RELEASE DECISION
    # --------------------------------------------------------------------------------------------------

    with BUNDLE_MANIFEST_PATH.open(
        "r",
        encoding="utf-8",
    ) as f:

        BUNDLE_MANIFEST = json.load(f)


    with VERIFICATION_AUDIT_PATH.open(
        "r",
        encoding="utf-8",
    ) as f:

        VERIFICATION_AUDIT = json.load(f)


    with RELEASE_GATE_DECISION_PATH.open(
        "r",
        encoding="utf-8",
    ) as f:

        RELEASE_GATE_DECISION = json.load(f)


    BUNDLE_ID = str(
        BUNDLE_MANIFEST[
            "bundle_id"
        ]
    )


    MODEL_VERSION = str(
        BUNDLE_MANIFEST.get(
            "source_model_version"
        )
    )


    # --------------------------------------------------------------------------------------------------
    # 6. HASH AUTHORITATIVE ARTIFACTS
    # --------------------------------------------------------------------------------------------------

    BUNDLE_MANIFEST_SHA256 = sha256_file(
        BUNDLE_MANIFEST_PATH
    )


    VERIFICATION_AUDIT_SHA256 = sha256_file(
        VERIFICATION_AUDIT_PATH
    )


    RELEASE_GATE_DECISION_SHA256 = sha256_file(
        RELEASE_GATE_DECISION_PATH
    )


    RELEASE_GATE_AUDIT_SHA256 = sha256_file(
        RELEASE_GATE_AUDIT_PATH
    )


    # --------------------------------------------------------------------------------------------------
    # 7. VERIFY AGAINST CELL 60 / 61 MANIFEST STATE
    # --------------------------------------------------------------------------------------------------

    EXPECTED_BUNDLE_SHA256 = (
        TRAINING_MANIFEST[
            "release_bundle"
        ][
            "bundle_manifest_sha256"
        ]
    )


    BUNDLE_HASH_MATCH_PASS = bool(
        BUNDLE_MANIFEST_SHA256
        ==
        EXPECTED_BUNDLE_SHA256
    )


    VERIFICATION_HASH_MATCH_PASS = bool(
        VERIFICATION_AUDIT_SHA256
        ==
        FINAL_RELEASE_VERIFICATION_AUDIT_SHA256
    )


    VERIFICATION_STATUS_PASS = bool(
        VERIFICATION_AUDIT.get(
            "status"
        )
        ==
        "PASSED"
    )


    VERIFICATION_BUNDLE_MATCH_PASS = bool(
        str(
            VERIFICATION_AUDIT.get(
                "bundle_id"
            )
        )
        ==
        BUNDLE_ID
    )


    RELEASE_GATE_STATUS_PASS = bool(
        RELEASE_GATE_DECISION.get(
            "release_status"
        )
        in {
            "PASS",
            "PASS_WITH_MONITORING",
        }
    )


    RELEASE_BUNDLE_ELIGIBLE_PASS = bool(
        RELEASE_GATE_DECISION.get(
            "eligible_for_release_bundle"
        )
        is True
    )


    MEAN_ARCHITECTURE_PASS = bool(
        BUNDLE_MANIFEST.get(
            "mean_architecture"
        )
        ==
        "V6.1_OU_FRESHNESS_BLEND"
    )


    UNCERTAINTY_MODEL_PASS = bool(
        BUNDLE_MANIFEST.get(
            "uncertainty_model"
        )
        ==
        "U1_SEX_LINEAR"
    )


    # --------------------------------------------------------------------------------------------------
    # 8. PRE-PROMOTION GATE
    # --------------------------------------------------------------------------------------------------

    PRE_PROMOTION_PASS = bool(

        NOTEBOOK_2_CELL_61_PASS

        and

        BUNDLE_HASH_MATCH_PASS

        and

        VERIFICATION_HASH_MATCH_PASS

        and

        VERIFICATION_STATUS_PASS

        and

        VERIFICATION_BUNDLE_MATCH_PASS

        and

        RELEASE_GATE_STATUS_PASS

        and

        RELEASE_BUNDLE_ELIGIBLE_PASS

        and

        MEAN_ARCHITECTURE_PASS

        and

        UNCERTAINTY_MODEL_PASS
    )


    if not PRE_PROMOTION_PASS:

        raise RuntimeError(
            "Pre-promotion integrity gate failed. "
            "ACTIVE_MODEL was not modified."
        )


    # --------------------------------------------------------------------------------------------------
    # 9. CREATE IMMUTABLE PROMOTION ATTEMPT
    # --------------------------------------------------------------------------------------------------

    RUN_UTC = datetime.now(
        timezone.utc
    )


    PROMOTION_ID = RUN_UTC.strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )


    PROMOTION_DIR = (
        PROMOTION_ROOT
        /
        PROMOTION_ID
    )


    PROMOTION_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )


    PROMOTION_AUDIT_PATH = (
        PROMOTION_DIR
        /
        "active_model_promotion_audit.json"
    )


    PROMOTED_POINTER_SNAPSHOT_PATH = (
        PROMOTION_DIR
        /
        "promoted_ACTIVE_MODEL.json"
    )


    # --------------------------------------------------------------------------------------------------
    # 10. PRESERVE PREVIOUS ACTIVE POINTER IF PRESENT
    # --------------------------------------------------------------------------------------------------

    PREVIOUS_ACTIVE_EXISTS = bool(
        ACTIVE_MODEL_PATH.exists()
    )


    PREVIOUS_ACTIVE_SHA256 = None

    PREVIOUS_ACTIVE_ARCHIVE_PATH = None

    PREVIOUS_ACTIVE_CONTENT = None


    if PREVIOUS_ACTIVE_EXISTS:

        PREVIOUS_ACTIVE_SHA256 = sha256_file(
            ACTIVE_MODEL_PATH
        )


        try:

            with ACTIVE_MODEL_PATH.open(
                "r",
                encoding="utf-8",
            ) as f:

                PREVIOUS_ACTIVE_CONTENT = json.load(
                    f
                )

        except Exception:

            PREVIOUS_ACTIVE_CONTENT = None


        ACTIVE_HISTORY_ROOT.mkdir(
            parents=True,
            exist_ok=True,
        )


        PREVIOUS_ACTIVE_ARCHIVE_PATH = (
            ACTIVE_HISTORY_ROOT
            /
            (
                f"ACTIVE_MODEL_before_{PROMOTION_ID}.json"
            )
        )


        shutil.copy2(
            ACTIVE_MODEL_PATH,
            PREVIOUS_ACTIVE_ARCHIVE_PATH,
        )


        ARCHIVE_HASH_MATCH_PASS = bool(
            sha256_file(
                PREVIOUS_ACTIVE_ARCHIVE_PATH
            )
            ==
            PREVIOUS_ACTIVE_SHA256
        )


        if not ARCHIVE_HASH_MATCH_PASS:

            raise RuntimeError(
                "Previous ACTIVE_MODEL archive hash mismatch. "
                "Promotion aborted before pointer replacement."
            )

    else:

        ARCHIVE_HASH_MATCH_PASS = True


    # --------------------------------------------------------------------------------------------------
    # 11. LOAD KNOWN LIMITATIONS REFERENCE
    # --------------------------------------------------------------------------------------------------

    KNOWN_LIMITATIONS_PATH = Path(
        RELEASE_GATE_DECISION[
            "known_limitations_file"
        ]
    )


    if not KNOWN_LIMITATIONS_PATH.exists():

        raise RuntimeError(
            f"Known-limitations artifact missing:\n"
            f"{KNOWN_LIMITATIONS_PATH}"
        )


    KNOWN_LIMITATIONS_SHA256 = sha256_file(
        KNOWN_LIMITATIONS_PATH
    )


    # --------------------------------------------------------------------------------------------------
    # 12. BUILD NEW ACTIVE POINTER
    # --------------------------------------------------------------------------------------------------

    GIT_COMMIT = git_commit_or_none(
        Path.cwd()
    )


    ACTIVE_POINTER = {

        "active_model_schema_version":
            PROMOTION_SCHEMA_VERSION,

        "status":
            "ACTIVE",

        "promoted_utc":
            RUN_UTC.isoformat(),

        "promotion_id":
            PROMOTION_ID,

        "model_family":
            "JAVELIN_V6_1",

        "model_version":
            MODEL_VERSION,

        "mean_architecture":
            "V6.1_OU_FRESHNESS_BLEND",

        "uncertainty_model":
            "U1_SEX_LINEAR",

        "release_status":
            FINAL_RELEASE_STATUS,

        "monitoring_required":
            bool(
                FINAL_RELEASE_STATUS
                ==
                "PASS_WITH_MONITORING"
            ),

        "bundle": {
            "bundle_id":
                BUNDLE_ID,

            "directory":
                str(
                    BUNDLE_DIR
                ),

            "manifest":
                str(
                    BUNDLE_MANIFEST_PATH
                ),

            "manifest_sha256":
                BUNDLE_MANIFEST_SHA256,
        },

        "verification": {
            "status":
                "PASSED",

            "directory":
                str(
                    VERIFICATION_DIR
                ),

            "audit":
                str(
                    VERIFICATION_AUDIT_PATH
                ),

            "audit_sha256":
                VERIFICATION_AUDIT_SHA256,
        },

        "release_gates": {
            "status":
                FINAL_RELEASE_STATUS,

            "decision":
                str(
                    RELEASE_GATE_DECISION_PATH
                ),

            "decision_sha256":
                RELEASE_GATE_DECISION_SHA256,

            "audit":
                str(
                    RELEASE_GATE_AUDIT_PATH
                ),

            "audit_sha256":
                RELEASE_GATE_AUDIT_SHA256,
        },

        "known_limitations": {
            "path":
                str(
                    KNOWN_LIMITATIONS_PATH
                ),

            "sha256":
                KNOWN_LIMITATIONS_SHA256,
        },

        "runtime_entrypoints": {
            "contract":
                str(
                    BUNDLE_DIR
                    /
                    BUNDLE_MANIFEST[
                        "runtime_contract"
                    ]
                ),

            "smoke_test_fixture":
                str(
                    BUNDLE_DIR
                    /
                    BUNDLE_MANIFEST[
                        "smoke_test_fixture"
                    ]
                ),
        },

        "source_training_run": {
            "directory":
                str(
                    RUN_DIR
                ),

            "training_manifest":
                str(
                    TRAINING_MANIFEST_PATH
                ),
        },

        "previous_active_model": {
            "existed":
                PREVIOUS_ACTIVE_EXISTS,

            "sha256":
                PREVIOUS_ACTIVE_SHA256,

            "archive_path":
                (
                    str(
                        PREVIOUS_ACTIVE_ARCHIVE_PATH
                    )
                    if PREVIOUS_ACTIVE_ARCHIVE_PATH
                    is not None
                    else None
                ),

            "previous_bundle_id":
                (
                    PREVIOUS_ACTIVE_CONTENT.get(
                        "bundle",
                        {},
                    ).get(
                        "bundle_id"
                    )
                    if isinstance(
                        PREVIOUS_ACTIVE_CONTENT,
                        dict,
                    )
                    else None
                ),
        },

        "promotion_provenance": {
            "cell_id":
                CELL_ID,

            "git_commit":
                GIT_COMMIT,

            "bundle_reproducibility_passed":
                True,

            "model_changed_during_promotion":
                False,

            "parameters_fitted_during_promotion":
                False,

            "bundle_modified_during_promotion":
                False,
        },
    }


    # --------------------------------------------------------------------------------------------------
    # 13. ATOMICALLY PROMOTE ACTIVE_MODEL
    # --------------------------------------------------------------------------------------------------

    atomic_json_write(
        ACTIVE_POINTER,
        ACTIVE_MODEL_PATH,
    )


    # --------------------------------------------------------------------------------------------------
    # 14. READ-BACK VERIFICATION
    # --------------------------------------------------------------------------------------------------

    if not ACTIVE_MODEL_PATH.exists():

        raise RuntimeError(
            "ACTIVE_MODEL.json was not created."
        )


    with ACTIVE_MODEL_PATH.open(
        "r",
        encoding="utf-8",
    ) as f:

        ACTIVE_POINTER_READBACK = json.load(
            f
        )


    ACTIVE_POINTER_SHA256 = sha256_file(
        ACTIVE_MODEL_PATH
    )


    ACTIVE_STATUS_PASS = bool(
        ACTIVE_POINTER_READBACK.get(
            "status"
        )
        ==
        "ACTIVE"
    )


    ACTIVE_BUNDLE_ID_PASS = bool(
        ACTIVE_POINTER_READBACK[
            "bundle"
        ][
            "bundle_id"
        ]
        ==
        BUNDLE_ID
    )


    ACTIVE_BUNDLE_HASH_PASS = bool(
        ACTIVE_POINTER_READBACK[
            "bundle"
        ][
            "manifest_sha256"
        ]
        ==
        BUNDLE_MANIFEST_SHA256
    )


    ACTIVE_VERIFICATION_HASH_PASS = bool(
        ACTIVE_POINTER_READBACK[
            "verification"
        ][
            "audit_sha256"
        ]
        ==
        VERIFICATION_AUDIT_SHA256
    )


    ACTIVE_RELEASE_STATUS_PASS = bool(
        ACTIVE_POINTER_READBACK[
            "release_status"
        ]
        ==
        FINAL_RELEASE_STATUS
    )


    # --------------------------------------------------------------------------------------------------
    # 15. SAVE PROMOTED POINTER SNAPSHOT
    # --------------------------------------------------------------------------------------------------

    shutil.copy2(
        ACTIVE_MODEL_PATH,
        PROMOTED_POINTER_SNAPSHOT_PATH,
    )


    PROMOTED_SNAPSHOT_HASH_PASS = bool(
        sha256_file(
            PROMOTED_POINTER_SNAPSHOT_PATH
        )
        ==
        ACTIVE_POINTER_SHA256
    )


    # --------------------------------------------------------------------------------------------------
    # 16. FINAL PROMOTION PASS
    # --------------------------------------------------------------------------------------------------

    CELL_62_PASS = bool(

        PRE_PROMOTION_PASS

        and

        ARCHIVE_HASH_MATCH_PASS

        and

        ACTIVE_STATUS_PASS

        and

        ACTIVE_BUNDLE_ID_PASS

        and

        ACTIVE_BUNDLE_HASH_PASS

        and

        ACTIVE_VERIFICATION_HASH_PASS

        and

        ACTIVE_RELEASE_STATUS_PASS

        and

        PROMOTED_SNAPSHOT_HASH_PASS
    )


    if not CELL_62_PASS:

        raise RuntimeError(
            "ACTIVE_MODEL promotion read-back verification failed."
        )


    # --------------------------------------------------------------------------------------------------
    # 17. PROMOTION AUDIT
    # --------------------------------------------------------------------------------------------------

    PROMOTION_AUDIT = {

        "audit_schema_version":
            PROMOTION_SCHEMA_VERSION,

        "stage":
            "ACTIVE_MODEL_PROMOTION",

        "cell_id":
            CELL_ID,

        "promotion_id":
            PROMOTION_ID,

        "created_utc":
            RUN_UTC.isoformat(),

        "status":
            "PASSED",

        "model_version":
            MODEL_VERSION,

        "bundle_id":
            BUNDLE_ID,

        "release_status":
            FINAL_RELEASE_STATUS,

        "monitoring_required":
            bool(
                FINAL_RELEASE_STATUS
                ==
                "PASS_WITH_MONITORING"
            ),

        "active_model_path":
            str(
                ACTIVE_MODEL_PATH
            ),

        "active_model_sha256":
            ACTIVE_POINTER_SHA256,

        "promoted_pointer_snapshot":
            str(
                PROMOTED_POINTER_SNAPSHOT_PATH
            ),

        "previous_active": {
            "existed":
                PREVIOUS_ACTIVE_EXISTS,

            "sha256":
                PREVIOUS_ACTIVE_SHA256,

            "archive_path":
                (
                    str(
                        PREVIOUS_ACTIVE_ARCHIVE_PATH
                    )
                    if PREVIOUS_ACTIVE_ARCHIVE_PATH
                    is not None
                    else None
                ),

            "archive_hash_match":
                ARCHIVE_HASH_MATCH_PASS,
        },

        "verified_artifacts": {
            "bundle_manifest": {
                "path":
                    str(
                        BUNDLE_MANIFEST_PATH
                    ),

                "sha256":
                    BUNDLE_MANIFEST_SHA256,
            },

            "reproducibility_audit": {
                "path":
                    str(
                        VERIFICATION_AUDIT_PATH
                    ),

                "sha256":
                    VERIFICATION_AUDIT_SHA256,
            },

            "release_gate_decision": {
                "path":
                    str(
                        RELEASE_GATE_DECISION_PATH
                    ),

                "sha256":
                    RELEASE_GATE_DECISION_SHA256,
            },

            "release_gate_audit": {
                "path":
                    str(
                        RELEASE_GATE_AUDIT_PATH
                    ),

                "sha256":
                    RELEASE_GATE_AUDIT_SHA256,
            },
        },

        "qa": {
            "pre_promotion_gate":
                PRE_PROMOTION_PASS,

            "previous_pointer_archive":
                ARCHIVE_HASH_MATCH_PASS,

            "active_status":
                ACTIVE_STATUS_PASS,

            "active_bundle_id":
                ACTIVE_BUNDLE_ID_PASS,

            "active_bundle_hash":
                ACTIVE_BUNDLE_HASH_PASS,

            "active_verification_hash":
                ACTIVE_VERIFICATION_HASH_PASS,

            "active_release_status":
                ACTIVE_RELEASE_STATUS_PASS,

            "promoted_snapshot_hash":
                PROMOTED_SNAPSHOT_HASH_PASS,

            "cell_pass":
                CELL_62_PASS,
        },

        "changes": {
            "model_parameters_changed":
                False,

            "bundle_modified":
                False,

            "only_active_pointer_changed":
                True,
        },

        "git_commit":
            GIT_COMMIT,

        "software": {
            "python":
                sys.version,

            "platform":
                platform.platform(),

            "numpy":
                np.__version__,

            "pandas":
                pd.__version__,
        },
    }


    atomic_json_write(
        PROMOTION_AUDIT,
        PROMOTION_AUDIT_PATH,
    )


    PROMOTION_AUDIT_SHA256 = sha256_file(
        PROMOTION_AUDIT_PATH
    )


    # --------------------------------------------------------------------------------------------------
    # 18. PROMOTION POINTER
    # --------------------------------------------------------------------------------------------------

    LATEST_PROMOTION_POINTER = (
        PROMOTION_ROOT
        /
        "LATEST_ACTIVE_MODEL_PROMOTION.json"
    )


    atomic_json_write(
        {
            "promotion_id":
                PROMOTION_ID,

            "status":
                "PASSED",

            "bundle_id":
                BUNDLE_ID,

            "model_version":
                MODEL_VERSION,

            "release_status":
                FINAL_RELEASE_STATUS,

            "active_model":
                str(
                    ACTIVE_MODEL_PATH
                ),

            "active_model_sha256":
                ACTIVE_POINTER_SHA256,

            "promotion_audit":
                str(
                    PROMOTION_AUDIT_PATH
                ),

            "promotion_audit_sha256":
                PROMOTION_AUDIT_SHA256,

            "created_utc":
                RUN_UTC.isoformat(),
        },

        LATEST_PROMOTION_POINTER,
    )


    # --------------------------------------------------------------------------------------------------
    # 19. UPDATE TRAINING MANIFEST
    # --------------------------------------------------------------------------------------------------

    TRAINING_MANIFEST[
        "active_model_promotion"
    ] = {

        "status":
            "PASSED",

        "promotion_id":
            PROMOTION_ID,

        "bundle_id":
            BUNDLE_ID,

        "model_version":
            MODEL_VERSION,

        "release_status":
            FINAL_RELEASE_STATUS,

        "active_model_path":
            str(
                ACTIVE_MODEL_PATH
            ),

        "active_model_sha256":
            ACTIVE_POINTER_SHA256,

        "promotion_directory":
            str(
                PROMOTION_DIR
            ),

        "latest_pointer":
            str(
                LATEST_PROMOTION_POINTER
            ),

        "promotion_audit":
            str(
                PROMOTION_AUDIT_PATH
            ),

        "promotion_audit_sha256":
            PROMOTION_AUDIT_SHA256,
    }


    TRAINING_MANIFEST[
        "status"
    ] = (
        "ACTIVE_MODEL_PROMOTED_PASS_WITH_MONITORING"
        if FINAL_RELEASE_STATUS
        ==
        "PASS_WITH_MONITORING"

        else
        "ACTIVE_MODEL_PROMOTED"
    )


    atomic_json_write(
        TRAINING_MANIFEST,
        TRAINING_MANIFEST_PATH,
    )


    # --------------------------------------------------------------------------------------------------
    # 20. GLOBALS
    # --------------------------------------------------------------------------------------------------

    FINAL_ACTIVE_MODEL_PATH = (
        ACTIVE_MODEL_PATH
    )


    FINAL_ACTIVE_MODEL_SHA256 = (
        ACTIVE_POINTER_SHA256
    )


    FINAL_PROMOTION_AUDIT_PATH = (
        PROMOTION_AUDIT_PATH
    )


    FINAL_PROMOTION_AUDIT_SHA256 = (
        PROMOTION_AUDIT_SHA256
    )


    NOTEBOOK_2_CELL_62_PASS = (
        CELL_62_PASS
    )


    # --------------------------------------------------------------------------------------------------
    # 21. OUTPUT
    # --------------------------------------------------------------------------------------------------

    print()
    print("=" * 180)
    print("ACTIVE_MODEL PROMOTION — V6.1 + U1")
    print("=" * 180)


    print()
    print("ACTIVE MODEL")
    print("-" * 180)

    print(
        f"Model family                    : "
        f"JAVELIN_V6_1"
    )

    print(
        f"Model version                   : "
        f"{MODEL_VERSION}"
    )

    print(
        f"Mean architecture               : "
        f"V6.1_OU_FRESHNESS_BLEND"
    )

    print(
        f"Uncertainty model               : "
        f"U1_SEX_LINEAR"
    )

    print(
        f"Release status                  : "
        f"{FINAL_RELEASE_STATUS}"
    )

    print(
        f"Monitoring required             : "
        f"{FINAL_RELEASE_STATUS == 'PASS_WITH_MONITORING'}"
    )


    print()
    print("PROMOTED BUNDLE")
    print("-" * 180)

    print(
        f"Bundle ID                       : "
        f"{BUNDLE_ID}"
    )

    print(
        f"Bundle directory                : "
        f"{BUNDLE_DIR}"
    )

    print(
        f"Bundle manifest SHA256          : "
        f"{BUNDLE_MANIFEST_SHA256}"
    )

    print(
        f"Verification audit SHA256       : "
        f"{VERIFICATION_AUDIT_SHA256}"
    )


    print()
    print("ACTIVE POINTER")
    print("-" * 180)

    print(
        f"ACTIVE_MODEL.json               : "
        f"{ACTIVE_MODEL_PATH}"
    )

    print(
        f"ACTIVE_MODEL SHA256             : "
        f"{ACTIVE_POINTER_SHA256}"
    )

    print(
        f"Previous ACTIVE_MODEL existed   : "
        f"{PREVIOUS_ACTIVE_EXISTS}"
    )

    print(
        f"Previous ACTIVE archive         : "
        f"{PREVIOUS_ACTIVE_ARCHIVE_PATH}"
    )


    print()
    print("PROMOTION QA")
    print("-" * 180)

    print(
        f"Pre-promotion integrity gate    : "
        f"{PRE_PROMOTION_PASS}"
    )

    print(
        f"Previous pointer archived       : "
        f"{ARCHIVE_HASH_MATCH_PASS}"
    )

    print(
        f"Active status valid             : "
        f"{ACTIVE_STATUS_PASS}"
    )

    print(
        f"Active bundle ID valid          : "
        f"{ACTIVE_BUNDLE_ID_PASS}"
    )

    print(
        f"Active bundle hash valid        : "
        f"{ACTIVE_BUNDLE_HASH_PASS}"
    )

    print(
        f"Verification hash valid         : "
        f"{ACTIVE_VERIFICATION_HASH_PASS}"
    )

    print(
        f"Release status valid            : "
        f"{ACTIVE_RELEASE_STATUS_PASS}"
    )

    print(
        f"Promoted snapshot hash valid    : "
        f"{PROMOTED_SNAPSHOT_HASH_PASS}"
    )

    print(
        f"Bundle modified                 : "
        f"False"
    )


    print()
    print("AUDIT / VERSIONING")
    print("-" * 180)

    print(
        f"Promotion ID                    : "
        f"{PROMOTION_ID}"
    )

    print(
        f"Promotion directory             : "
        f"{PROMOTION_DIR}"
    )

    print(
        f"Promotion audit                 : "
        f"{PROMOTION_AUDIT_PATH}"
    )

    print(
        f"Promotion audit SHA256          : "
        f"{PROMOTION_AUDIT_SHA256}"
    )

    print(
        f"Git commit                      : "
        f"{GIT_COMMIT}"
    )


    print()
    print("=" * 180)

    if CELL_62_PASS:

        print(
            "CELL 62 STATUS: V6.1 + U1 PROMOTED TO ACTIVE_MODEL — "
            "NOTEBOOK 2 COMPLETE"
        )

    else:

        print(
            "CELL 62 STATUS: ACTIVE_MODEL PROMOTION FAILED"
        )

    print("=" * 180)

else:
    NOTEBOOK_2_PROMOTION_STATUS = "SKIPPED_BY_EXECUTION_MODE"
    FINAL_ACTIVE_MODEL_PATH = None
    FINAL_ACTIVE_MODEL_SHA256 = None
    FINAL_PROMOTION_AUDIT_PATH = None
    FINAL_PROMOTION_AUDIT_SHA256 = None
    NOTEBOOK_2_CELL_62_PASS = True

    print()
    print("=" * 180)
    print("ACTIVE_MODEL PROMOTION GATE")
    print("=" * 180)
    print(f"Execution mode                  : {EXECUTION_MODE}")
    print("Promotion executed              : False")
    print("ACTIVE_MODEL modified           : False")
    print(f"Persistent candidate artifacts  : {PERSISTENT_ARTIFACTS_ENABLED}")
    if IS_VALIDATION_MODE:
        print(f"Transient validation root       : {MODEL_ARTIFACT_ROOT}")
    print("Status                          : SKIPPED BY EXECUTION MODE")
    print("=" * 180)



ACTIVE_MODEL PROMOTION GATE
Execution mode                  : VALIDATE_REFACTOR
Promotion executed              : False
ACTIVE_MODEL modified           : False
Persistent candidate artifacts  : False
Transient validation root       : /var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/javelin_v6_refactor_validation_js0mot1_
Status                          : SKIPPED BY EXECUTION MODE


## Refactor audit notes

- Original source cell **44** (one-based; zero-based index 43) was removed because it is byte-for-byte identical to original source cell **42** (zero-based index 41).
- The original final empty code cell was omitted.
- All other nonempty original code cells are represented exactly once and remain in original execution order.
- Original source SHA256 values are stored in each refactored code cell metadata and repeated as comments before each embedded legacy block.
- The only systematic source transformation is removal of repeated `from __future__ import annotations` statements inside coalesced stages, with one equivalent import retained at the top of every refactored stage.
- Existing legacy cell-number strings inside preserved code are intentionally unchanged because they can appear in audit metadata and historical artifact descriptions.


### Safe execution-mode modification (`v1.1.0_SAFE_MODES`)

- Added an explicit three-mode execution control before Stage 01.
- `VALIDATE_REFACTOR` redirects the model-artifact root to a temporary scratch directory, so no new persistent model version is created under `model_artifacts`.
- `TRAIN_CANDIDATE` retains all persistent training/bundle/versioning behavior but skips `ACTIVE_MODEL` promotion.
- `TRAIN_AND_PROMOTE` preserves the original full production behavior.
- Original promotion source is preserved unchanged inside the `TRAIN_AND_PROMOTE` branch; the only behavioral change is the execution-mode gate around it.
- Bundle construction/reload verification still runs in validation mode, but only against transient scratch artifacts so the release contract itself can be tested safely.
